# 06 — sakai224_full IIC, over-clustering e spectral clustering

Questo notebook implementa la fase di classificazione non supervisionata
principale della pipeline proposta da Sakai et al.

Il punto di partenza è il VAE `sakai224_full` già addestrato per 100 epoche.

Il decoder non viene più utilizzato.

L'encoder viene congelato e produce:

$$
z=\mu_\phi(x)
\in
\mathbb{R}^{512}.
$$

Per ogni evento vengono costruite due viste:

$$
x
$$

originale/non perturbata e

$$
x'=\xi(x)
$$

perturbata esclusivamente nella direzione temporale.

Le due viste vengono elaborate dallo stesso encoder VAE con pesi condivisi:

$$
z=\mu_\phi(x),
$$

$$
z'=\mu_\phi(x').
$$

L'IIC massimizza:

$$
I\left(
\Phi(z),
\Phi(z')
\right).
$$

La configurazione pubblicata da Sakai è:

$$
C=36,
\qquad
W=250,
\qquad
K=5,
$$

$$
\eta=5\times10^{-4},
\qquad
\mathrm{batch}=128.
$$

Il classificatore principale segue la Figura 8:

$$
512
\rightarrow
FC(512)
\rightarrow
BN
\rightarrow
ReLU
\rightarrow
FC(C)
\rightarrow
SoftMax.
$$

Un ramo ausiliario realizza l'over-clustering con:

$$
W=250.
$$

Successivamente i risultati dei

$$
K=5
$$

classificatori vengono combinati e sottoposti a spectral clustering.

Le Gravity Spy label non vengono utilizzate durante:

- training IIC;
- over-clustering;
- selezione degli output;
- costruzione dell'affinità;
- spectral clustering.

La valutazione con le label viene rinviata fino al congelamento della
partizione finale.

## STEP 06-A0 — Audit di fedeltà e gerarchia delle fonti

Prima di implementare l'IIC vengono separati rigorosamente i dettagli
pubblicati dalle scelte introdotte nel progetto.

### Fonti operative autorizzate

Per il ramo `sakai224_full` si usano:

1. lo snapshot HTML allegato della version of record di *Scientific Reports*;
2. il Supplemental Material allegato;
3. la scaletta aggiornata della tesi;
4. l'addendum vincolante;
5. gli artefatti e i sorgenti già verificati del progetto.

Non vengono importati automaticamente dettagli da altre implementazioni.
Finché una fonte aggiuntiva non viene esplicitamente approvata, ogni dettaglio
non pubblicato nello snapshot HTML o nel supplemento è classificato
`PROJECT`.

### PAPER — elementi esplicitamente pubblicati

Le due viste dello stesso evento sono elaborate da encoder VAE pre-addestrati
con pesi condivisi e producono:

$$
z=\mu_\phi(x),
\qquad
z'=\mu_\phi(x').
$$

L'IIC massimizza:

$$
I\!\left(\Phi(z),\Phi(z')\right),
$$

con uscita SoftMax. La configurazione pubblicata adottata nel progetto è:

$$
C=36,
\qquad
W=250,
\qquad
K=5,
$$

$$
\eta=5\times10^{-4},
\qquad
B=128,
$$

e Adam con:

$$
\beta_1=0.9,
\qquad
\beta_2=0.999,
\qquad
\epsilon_{\mathrm{Adam}}=10^{-8}.
$$

L'Eq. (4) usa una distribuzione congiunta $P_{ij}$ e i marginali
$P_i,P_j$:

$$
I
=
\sum_{i=1}^{C}
\sum_{j=1}^{C}
P_{ij}
\ln
\left(
\frac{P_{ij}}{P_iP_j}
\right).
$$

L'over-clustering utilizza lo stesso obiettivo con:

$$
\Phi(\cdot)\in\mathbb{R}^{W},
\qquad
W>C.
$$

Lo snapshot HTML non pubblica però in modo operativo lo stimatore empirico
minibatch di $P_{ij}$.

### PROJECT — dettagli da congelare

Sono quindi `PROJECT`:

- lo stimatore empirico minibatch della distribuzione congiunta;
- la stabilizzazione numerica della loss;
- l'eventuale simmetrizzazione o rinormalizzazione di $P$;
- il congelamento dell'encoder in modalità EVAL e senza gradienti;
- la relazione tra head principale e over-clustering;
- inizializzazione, seed e semantica numerica della BatchNorm;
- numero di epoche, shuffle, tail batch e checkpoint/resume;
- split IIC e dettagli numerici dello spectral clustering.

Le Gravity Spy label restano escluse fino al congelamento della partizione
finale.


## STEP 06-A1 — Contratto della popolazione IIC e delle due viste

Il paper specifica esplicitamente un training size rate pari a

$$
r_{\mathrm{VAE}}=0.8
$$

per il training del VAE.

Non specifica invece in modo univoco un ulteriore split train/validation
utilizzato durante il training dell'IIC.

La classificazione finale viene presentata sull'intero dataset e le label
Gravity Spy vengono utilizzate soltanto per valutare la partizione ottenuta.

### Decisione PROJECT primaria

Il training IIC di `sakai224_full` viene quindi definito come transduttivo
sull'intero corpus pubblico ricostruito:

$$
D_{\mathrm{IIC}}
=
8583.
$$

Lo split VAE

$$
6866+1717=8583
$$

rimane conservato come informazione di provenienza e potrà essere usato
successivamente per diagnostiche separate, ma nessuno dei 1.717 eventi viene
escluso dal training IIC principale.

Questa è una scelta PROJECT motivata dalla struttura del paper, non un
dettaglio esplicitamente pubblicato dagli autori.

### Coppia di viste

Per ogni evento viene costruita la coppia:

$$
x_i
=
\text{crop centrale},
$$

$$
x'_i
=
\xi(x_i),
$$

dove la perturbazione agisce esclusivamente nella direzione temporale.

La vista centrale usa:

$$
\mathrm{crop\_left}=24.
$$

La vista perturbata primaria segue la modalità PROJECT già congelata
`symmetric_figure`:

$$
\mathrm{crop\_left}\in\{0,\ldots,48\},
$$

equivalentemente:

$$
s
=
\mathrm{crop\_left}-24
\in
\{-24,\ldots,+24\}.
$$

Lo stesso crop viene applicato alle quattro durate dell'evento.

La modalità `legacy_text`, limitata a `crop_left = 0,...,24`, resta
un'ablation successiva.

### Encoder

Entrambe le viste vengono elaborate dallo stesso encoder `FINAL` dopo
100 epoche:

$$
z_i=\mu_\phi(x_i),
$$

$$
z'_i=\mu_\phi(x'_i).
$$

L'encoder viene mantenuto:

- congelato;
- in modalità evaluation;
- senza gradienti;
- senza aggiornamenti delle statistiche BatchNorm.

Il decoder non viene utilizzato.

Le label Gravity Spy restano escluse dal contratto IIC.

In [ ]:
# ============================================================
# NOTEBOOK 06 — IIC DATA POPULATION CONTRACT
# STEP 06-A1
#
# READ ONLY.
#
# Freezes:
#   - full transductive population: 8583 events
#   - central / perturbed view policy
#   - frozen FINAL encoder policy
#
# NO image extraction.
# NO model loading.
# NO labels.
# NO IIC training.
# NO Drive writes.
# ============================================================

from pathlib import Path

import csv
import hashlib
import json
import math
import os
import tempfile


# ============================================================
# PORTABLE ENVIRONMENT ROOTS
# ============================================================

try:
    from google.colab import drive
except ImportError:
    drive = None


_in_colab = drive is not None


def _resolve_absolute_root(
    value,
    *,
    variable_name,
):

    root = Path(
        value
    ).expanduser()

    if not root.is_absolute():
        raise ValueError(
            f"{variable_name} must be an absolute path: {value!r}"
        )

    return root.resolve(
        strict=False
    )


_local_runtime_override = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)

if _local_runtime_override is not None:

    LOCAL_RUNTIME_ROOT = _resolve_absolute_root(
        _local_runtime_override,
        variable_name=(
            "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
        ),
    )

elif _in_colab:

    LOCAL_RUNTIME_ROOT = Path(
        "/content"
    )

else:

    LOCAL_RUNTIME_ROOT = _resolve_absolute_root(
        tempfile.gettempdir(),
        variable_name=(
            "system temporary directory"
        ),
    )


if _in_colab:

    _colab_drive_mount = (
        LOCAL_RUNTIME_ROOT
        / "drive"
    )

    drive.mount(
        str(
            _colab_drive_mount
        ),
        force_remount=False,
    )


_project_root_override = os.environ.get(
    "GRAVITYSPY_PROJECT_ROOT"
)

if _project_root_override is not None:

    PROJECT_ROOT = _resolve_absolute_root(
        _project_root_override,
        variable_name=(
            "GRAVITYSPY_PROJECT_ROOT"
        ),
    )

elif _in_colab:

    PROJECT_ROOT = (
        _colab_drive_mount
        / "MyDrive"
        / "TESI_SAKAI2022"
    ).resolve(
        strict=False
    )

else:

    raise RuntimeError(
        "GRAVITYSPY_PROJECT_ROOT must be set outside Colab."
    )


# ============================================================
# DERIVED PERSISTENT PATHS
# ============================================================

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

TRANSFER_V1 = (
    SAKAI224_ROOT
    / "transfer_v1"
)

OFFICIAL_RUN_DIR = (
    SAKAI224_ROOT
    / "runs"
    / "sakai224_full_v1"
)

FINAL_CHECKPOINT_PATH = (
    OFFICIAL_RUN_DIR
    / "final.pt"
)

EMBEDDING_DIR = (
    SAKAI224_ROOT
    / "embeddings"
    / "sakai224_full_mu_epoch100_v1"
)


OUT272_TAR_PATH = (
    TRANSFER_V1
    / "sakai224_full_out272_gray.tar"
)

EVENT_MANIFEST_PATH = (
    TRANSFER_V1
    / "manifest_events_272.csv"
)

TRAIN_SPLIT_PATH = (
    TRANSFER_V1
    / "split_train_indices_seed123.csv"
)

VALIDATION_SPLIT_PATH = (
    TRANSFER_V1
    / "split_val_indices_seed123.csv"
)

TRAIN_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_train_f32.bin"
)

VALIDATION_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_validation_f32.bin"
)

TRAIN_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_train.csv"
)

VALIDATION_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_validation.csv"
)


# ============================================================
# FROZEN SOURCE IDENTITIES
# ============================================================

EXPECTED_HASHES = {

    FINAL_CHECKPOINT_PATH:
        (
            "2bb6fd2ce2f73f8f9e479ed6969a52c1"
            "cdd41c1ae6e2cfc33da687ebba798f71"
        ),

    OUT272_TAR_PATH:
        (
            "70c766c3652f586c8da4c8cadf134a8c"
            "e83cfd07c8cd10545367f380017ddab2"
        ),

    EVENT_MANIFEST_PATH:
        (
            "620960e6b35da31bc4cf1a8a6da86790"
            "27d378e1d4b2ac73c5267c1c5311ae82"
        ),

    TRAIN_SPLIT_PATH:
        (
            "8c7a513b98dbe996261bc80965500ab15"
            "ab77f498ec9440858003152e42f926f"
        ),

    VALIDATION_SPLIT_PATH:
        (
            "e63109c88df7b3ee97317f94791f26df"
            "3fcaf8b937921cfb27967d01c2546467"
        ),

    TRAIN_MU_PATH:
        (
            "b620fd9711719a7bd40293b6536ea6fe"
            "b235cbadb1d1e37dca0e46651332ad5e"
        ),

    VALIDATION_MU_PATH:
        (
            "c1692b3e7a4c86a7f4d6d7172117e110"
            "83d2bd2394947838a24d1e2b81f27cf9"
        ),

    TRAIN_ROW_MAP_PATH:
        (
            "4b25c7f41a945fc8195e93c11e60da3b"
            "2e5b8be8298f6919008fc57dc39cc9e8"
        ),

    VALIDATION_ROW_MAP_PATH:
        (
            "f5a0e48c79db4d466f834cbc54bdbf7e"
            "8c34d3fa45f545f58b37c71627c97b48"
        ),
}


# ============================================================
# SHA-256
# ============================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


source_presence_gate = all(
    path.is_file()
    for path in EXPECTED_HASHES
)

assert source_presence_gate


calculated_hashes = {
    path:
        sha256_file(path)

    for path in EXPECTED_HASHES
}


source_hash_gate = all(
    calculated_hashes[path]
    == expected_hash

    for path, expected_hash
    in EXPECTED_HASHES.items()
)

assert source_hash_gate


# ============================================================
# LABEL-FREE ROW MAPS
# ============================================================

EXPECTED_ROW_MAP_FIELDS = [
    "split",
    "split_pos",
    "manifest_index",
    "event_index",
    "event_id",
]


def read_row_map(path):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.DictReader(handle)

        assert list(
            reader.fieldnames
        ) == EXPECTED_ROW_MAP_FIELDS

        return list(reader)


train_rows = read_row_map(
    TRAIN_ROW_MAP_PATH
)

validation_rows = read_row_map(
    VALIDATION_ROW_MAP_PATH
)


assert len(train_rows) == 6866
assert len(validation_rows) == 1717


# ============================================================
# RECONSTRUCT COMPLETE POPULATION IN MANIFEST ORDER
# ============================================================

full_iic_rows = [
    None
] * 8583


def insert_rows(
    rows,
    expected_split,
):

    for expected_split_pos, row in enumerate(rows):

        manifest_index = int(
            row["manifest_index"]
        )

        event_index = int(
            row["event_index"]
        )

        split_pos = int(
            row["split_pos"]
        )

        event_id = row[
            "event_id"
        ]


        assert row[
            "split"
        ] == expected_split

        assert split_pos == expected_split_pos

        assert event_index == manifest_index + 1

        assert 0 <= manifest_index < 8583

        assert full_iic_rows[
            manifest_index
        ] is None


        full_iic_rows[
            manifest_index
        ] = {

            "manifest_index":
                manifest_index,

            "event_index":
                event_index,

            "event_id":
                event_id,

            "source_vae_split":
                expected_split,

            "source_vae_split_pos":
                split_pos,
        }


insert_rows(
    train_rows,
    "train",
)

insert_rows(
    validation_rows,
    "validation",
)


population_gate = (
    all(
        row is not None
        for row in full_iic_rows
    )

    and [
        row["manifest_index"]
        for row in full_iic_rows
    ]
    == list(
        range(8583)
    )

    and len({
        row["event_id"]
        for row in full_iic_rows
    })
    == 8583

    and sum(
        row["source_vae_split"]
        == "train"

        for row in full_iic_rows
    )
    == 6866

    and sum(
        row["source_vae_split"]
        == "validation"

        for row in full_iic_rows
    )
    == 1717
)

assert population_gate


# ============================================================
# CANONICAL POPULATION IDENTITY
# ============================================================

population_identity_text = "\n".join(

    (
        f"{row['manifest_index']},"
        f"{row['event_index']},"
        f"{row['event_id']},"
        f"{row['source_vae_split']},"
        f"{row['source_vae_split_pos']}"
    )

    for row in full_iic_rows
) + "\n"


FULL_IIC_POPULATION_SHA256 = hashlib.sha256(
    population_identity_text.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# FROZEN DATA CONTRACT
# ============================================================

IIC_DATA_CONTRACT = {

    "contract_version":
        1,

    "main_scientific_goal":
        "sakai224_full_vs_Sakai",

    "population": {

        "primary_mode":
            "full_8583_transductive",

        "event_count":
            8583,

        "manifest_order":
            True,

        "identity_sha256":
            FULL_IIC_POPULATION_SHA256,

        "paper_explicit_iic_split":
            False,

        "classification":
            "PROJECT",

        "vae_split_role":
            (
                "provenance_and_future_splitwise_"
                "diagnostics_only"
            ),

        "vae_validation_excluded_from_iic_fit":
            False,
    },

    "views": {

        "original": {

            "name":
                "central",

            "crop_left":
                24,

            "time_shift":
                0,
        },

        "perturbed": {

            "mode":
                "symmetric_figure",

            "crop_left_min":
                0,

            "crop_left_max":
                48,

            "signed_shift_min":
                -24,

            "signed_shift_max":
                24,

            "same_crop_for_all_four_channels":
                True,

            "stateless_key":
                [
                    "augmentation_seed",
                    "epoch",
                    "manifest_index",
                ],

            "augmentation_seed":
                789,

            "classification":
                "PROJECT",
        },

        "legacy_text_retained_as_ablation":
            True,
    },

    "encoder": {

        "checkpoint":
            "final.pt",

        "completed_epochs":
            100,

        "feature":
            "mu",

        "latent_dim":
            512,

        "shared_between_views":
            True,

        "execution":
            "both_views_on_the_fly",

        "frozen":
            True,

        "eval_mode":
            True,

        "gradients":
            False,

        "batchnorm_updates":
            False,

        "decoder_used":
            False,
    },

    "paper_parameters_already_fixed": {

        "C":
            36,

        "W":
            250,

        "K":
            5,

        "batch_size":
            128,

        "learning_rate":
            5.0e-4,
    },

    "labels": {

        "used_for_iic":
            False,

        "used_for_spectral":
            False,

        "evaluation_deferred":
            True,
    },

    "still_unresolved": [

        "epoch_count_and_stopping",

        "classifier_initialization_and_seeds",

        "shuffle_policy",

        "tail_batch_policy",

        "exact_IIC_loss_numerics",

        "main_overcluster_loss_combination",

        "checkpoint_resume_policy",
    ],
}


IIC_DATA_CONTRACT_JSON = json.dumps(
    IIC_DATA_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_DATA_CONTRACT_SHA256 = hashlib.sha256(
    IIC_DATA_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# CONTRACT GATES
# ============================================================

population_policy_gate = (
    IIC_DATA_CONTRACT[
        "population"
    ][
        "primary_mode"
    ]
    == "full_8583_transductive"

    and IIC_DATA_CONTRACT[
        "population"
    ][
        "event_count"
    ]
    == 8583

    and IIC_DATA_CONTRACT[
        "population"
    ][
        "paper_explicit_iic_split"
    ]
    is False

    and IIC_DATA_CONTRACT[
        "population"
    ][
        "vae_validation_excluded_from_iic_fit"
    ]
    is False
)


view_gate = (
    IIC_DATA_CONTRACT[
        "views"
    ][
        "original"
    ][
        "crop_left"
    ]
    == 24

    and IIC_DATA_CONTRACT[
        "views"
    ][
        "perturbed"
    ][
        "signed_shift_min"
    ]
    == -24

    and IIC_DATA_CONTRACT[
        "views"
    ][
        "perturbed"
    ][
        "signed_shift_max"
    ]
    == 24
)


encoder_gate = (
    IIC_DATA_CONTRACT[
        "encoder"
    ][
        "checkpoint"
    ]
    == "final.pt"

    and IIC_DATA_CONTRACT[
        "encoder"
    ][
        "feature"
    ]
    == "mu"

    and IIC_DATA_CONTRACT[
        "encoder"
    ][
        "latent_dim"
    ]
    == 512

    and IIC_DATA_CONTRACT[
        "encoder"
    ][
        "frozen"
    ]
    is True

    and IIC_DATA_CONTRACT[
        "encoder"
    ][
        "decoder_used"
    ]
    is False
)


label_gate = (
    IIC_DATA_CONTRACT[
        "labels"
    ][
        "used_for_iic"
    ]
    is False

    and IIC_DATA_CONTRACT[
        "labels"
    ][
        "used_for_spectral"
    ]
    is False
)


assert population_policy_gate
assert view_gate
assert encoder_gate
assert label_gate


# ============================================================
# BATCH GEOMETRY — DESCRIPTIVE ONLY
#
# Tail handling is frozen later.
# ============================================================

IIC_BATCH_SIZE = 128

batches_if_drop_last_false = math.ceil(
    8583 / IIC_BATCH_SIZE
)

tail_if_drop_last_false = (
    8583
    % IIC_BATCH_SIZE
)


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 IIC DATA POPULATION CONTRACT ==="
)

print()

print(
    "PRIMARY_IIC_POPULATION = full_8583_transductive"
)

print(
    "IIC_EVENT_COUNT =",
    len(
        full_iic_rows
    ),
)

print(
    "VAE_TRAIN_MEMBERS =",
    sum(
        row["source_vae_split"] == "train"
        for row in full_iic_rows
    ),
)

print(
    "VAE_VALIDATION_MEMBERS =",
    sum(
        row["source_vae_split"] == "validation"
        for row in full_iic_rows
    ),
)

print(
    "FULL_IIC_POPULATION_SHA256 =",
    FULL_IIC_POPULATION_SHA256,
)

print()

print(
    "ORIGINAL_VIEW_CROP_LEFT = 24"
)

print(
    "PERTURBED_VIEW_MODE = symmetric_figure"
)

print(
    "PERTURBED_CROP_LEFT_RANGE = [0, 48]"
)

print(
    "PERTURBED_SIGNED_SHIFT_RANGE = [-24, 24]"
)

print(
    "AUGMENTATION_SEED = 789"
)

print()

print(
    "ENCODER_CHECKPOINT = final.pt"
)

print(
    "ENCODER_COMPLETED_EPOCHS = 100"
)

print(
    "ENCODER_FEATURE = mu"
)

print(
    "ENCODER_LATENT_DIM = 512"
)

print(
    "ENCODER_FROZEN = True"
)

print(
    "DECODER_USED = False"
)

print()

print(
    "BATCH_SIZE_PAPER =",
    IIC_BATCH_SIZE,
)

print(
    "BATCHES_IF_DROP_LAST_FALSE =",
    batches_if_drop_last_false,
)

print(
    "TAIL_IF_DROP_LAST_FALSE =",
    tail_if_drop_last_false,
)

print(
    "TAIL_POLICY_FROZEN =",
    False,
)

print()

print(
    "IIC_DATA_CONTRACT_SHA256 =",
    IIC_DATA_CONTRACT_SHA256,
)

print()

print(
    "SOURCE_PRESENCE_GATE =",
    source_presence_gate,
)

print(
    "SOURCE_HASH_GATE =",
    source_hash_gate,
)

print(
    "POPULATION_GATE =",
    population_gate,
)

print(
    "POPULATION_POLICY_GATE =",
    population_policy_gate,
)

print(
    "VIEW_GATE =",
    view_gate,
)

print(
    "ENCODER_GATE =",
    encoder_gate,
)

print(
    "LABEL_GATE =",
    label_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "DATA_EXTRACTED =",
    False,
)

print(
    "MODEL_LOADED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A1_DATA_CONTRACT_PASS =",
    (
        source_presence_gate
        and source_hash_gate
        and population_gate
        and population_policy_gate
        and view_gate
        and encoder_gate
        and label_gate
    ),
)

Mounted at /content/drive

=== NOTEBOOK 06 IIC DATA POPULATION CONTRACT ===

PRIMARY_IIC_POPULATION = full_8583_transductive
IIC_EVENT_COUNT = 8583
VAE_TRAIN_MEMBERS = 6866
VAE_VALIDATION_MEMBERS = 1717
FULL_IIC_POPULATION_SHA256 = bd95286ec6c70cb2b5c674a8a46f3fd450048367d91dc17fd3625977c538b8f4

ORIGINAL_VIEW_CROP_LEFT = 24
PERTURBED_VIEW_MODE = symmetric_figure
PERTURBED_CROP_LEFT_RANGE = [0, 48]
PERTURBED_SIGNED_SHIFT_RANGE = [-24, 24]
AUGMENTATION_SEED = 789

ENCODER_CHECKPOINT = final.pt
ENCODER_COMPLETED_EPOCHS = 100
ENCODER_FEATURE = mu
ENCODER_LATENT_DIM = 512
ENCODER_FROZEN = True
DECODER_USED = False

BATCH_SIZE_PAPER = 128
BATCHES_IF_DROP_LAST_FALSE = 68
TAIL_IF_DROP_LAST_FALSE = 7
TAIL_POLICY_FROZEN = False

IIC_DATA_CONTRACT_SHA256 = 03bfab28cd2f513dd3616aca9f3946333d12d85508e95bb860f6682941002e17

SOURCE_PRESENCE_GATE = True
SOURCE_HASH_GATE = True
POPULATION_GATE = True
POPULATION_POLICY_GATE = True
VIEW_GATE = True
ENCODER_GATE = True
LABEL_GATE = True

LABELS_READ = Fa

## STEP 06-A2R — Eq. (4) PAPER e stimatore minibatch PROJECT

Questo step sostituisce il precedente STEP 06-A2.

### Obiettivo PAPER

Sakai massimizza la mutual information tra le predizioni SoftMax delle due
viste:

$$
I
=
\sum_{i,j}
P_{ij}
\ln
\left(
\frac{P_{ij}}
{P_iP_j}
\right),
$$

con:

$$
P_i=\sum_jP_{ij},
\qquad
P_j=\sum_iP_{ij}.
$$

Lo stesso obiettivo viene applicato al ramo principale con:

$$
C=36,
$$

e all'over-clustering con:

$$
W=250>C.
$$

### Stimatore empirico PROJECT

Lo snapshot HTML non specifica in modo operativo come stimare $P_{ij}$ da
un minibatch. Il progetto adotta quindi, per $B$ coppie allineate:

$$
p_b=\Phi(z_b),
\qquad
q_b=\Phi(z'_b),
$$

lo stimatore:

$$
\widehat P_{ij}
=
\frac{1}{B}
\sum_{b=1}^{B}
p_{b,i}q_{b,j}.
$$

Poiché $p_b$ e $q_b$ sono distribuzioni SoftMax:

$$
\sum_{i,j}\widehat P_{ij}=1
$$

per costruzione.

Sono scelte `PROJECT`:

- nessuna simmetrizzazione di $\widehat P$;
- nessuna rinormalizzazione aggiuntiva;
- uso di `float32` durante il training;
- convenzione $0\log0=0$, realizzata valutando soltanto le celle
  $\widehat P_{ij}>0$;
- nessun epsilon arbitrario aggiunto alla formula.

La loss minimizzata è:

$$
\mathcal L_{\mathrm{IIC}}=-I.
$$

I known-answer test Python/NumPy di questo step costituiscono il riferimento
numerico per la futura implementazione C11. La relazione tra loss principale
e loss di over-clustering non viene decisa qui.

Non vengono creati modelli, optimizer o checkpoint e non vengono lette
label.


### Ruolo rispetto alle ablation mini C11

La presente implementazione Python della loss costituisce anche il riferimento
numerico per la successiva implementazione IIC procedurale in C11.

I casi sintetici e i known-answer test definiti in questo step verranno
riutilizzati come golden vector per verificare che la futura loss C11 dei
profili:

$$
\mathrm{mini75},
\qquad
\mathrm{mini112},
\qquad
\mathrm{mini224\_1000}
$$

replichi la stessa formulazione matematica prima di qualsiasi training reale.

Questo non rende il backend C11 parte della replica full primaria: serve a
garantire che full Python e ablation mini condividano la stessa definizione
dell'obiettivo IIC.

In [ ]:
# ============================================================
# NOTEBOOK 06 — SAKAI Eq. (4) IIC LOSS AUDIT
# STEP 06-A2R
#
# Supersedes previous A2.
#
# Source:
#   Sakai Eq. (4) only
#   + explicit PROJECT floating-point handling.
#
# NO external IIC implementation assumptions.
# NO model.
# NO labels.
# NO training.
# NO Drive writes.
# ============================================================

import hashlib
import json
import math

import numpy as np
import torch


# ============================================================
# PARENT DATA CONTRACT
# ============================================================

EXPECTED_IIC_DATA_CONTRACT_SHA256 = (
    "03bfab28cd2f513dd3616aca9f394633"
    "3d12d85508e95bb860f6682941002e17"
)

assert (
    IIC_DATA_CONTRACT_SHA256
    == EXPECTED_IIC_DATA_CONTRACT_SHA256
)


# ============================================================
# PROBABILITY VALIDATION
# ============================================================

def validate_probability_matrix(
    probabilities,
    *,
    name,
):

    if probabilities.ndim != 2:
        raise ValueError(
            f"{name} must be rank 2"
        )

    if not torch.is_floating_point(
        probabilities
    ):
        raise TypeError(
            f"{name} must be floating point"
        )

    if not torch.isfinite(
        probabilities
    ).all():
        raise ValueError(
            f"{name} contains NaN/Inf"
        )

    if torch.any(
        probabilities < 0.0
    ):
        raise ValueError(
            f"{name} contains negative values"
        )

    row_sums = probabilities.sum(
        dim=1
    )

    if not torch.allclose(
        row_sums,
        torch.ones_like(
            row_sums
        ),
        rtol=1.0e-6,
        atol=1.0e-7,
    ):
        raise ValueError(
            f"{name} rows must sum to 1"
        )


# ============================================================
# SAKAI Eq. (4) JOINT
#
# P_ij = (1/N) sum_n p_ni q_nj
#
# NO symmetrisation.
# NO extra renormalisation.
# ============================================================

def sakai_joint_probability(
    probabilities_a,
    probabilities_b,
):

    validate_probability_matrix(
        probabilities_a,
        name="probabilities_a",
    )

    validate_probability_matrix(
        probabilities_b,
        name="probabilities_b",
    )

    if (
        probabilities_a.shape
        != probabilities_b.shape
    ):
        raise ValueError(
            "paired probability matrices "
            "must have identical shape"
        )

    batch_size = (
        probabilities_a.shape[0]
    )

    if batch_size <= 0:
        raise ValueError(
            "batch must not be empty"
        )

    joint = (
        probabilities_a.transpose(
            0,
            1,
        )
        @ probabilities_b
    ) / float(
        batch_size
    )

    return joint


# ============================================================
# SAKAI Eq. (4) MUTUAL INFORMATION
#
# 0 * log(0) is handled by excluding P_ij == 0 terms.
# No arbitrary epsilon is introduced.
# ============================================================

def sakai_iic_negative_mutual_information(
    probabilities_a,
    probabilities_b,
):

    joint = sakai_joint_probability(
        probabilities_a,
        probabilities_b,
    )

    marginal_i = joint.sum(
        dim=1
    )

    marginal_j = joint.sum(
        dim=0
    )

    positive = (
        joint > 0.0
    )

    positive_indices = positive.nonzero(
        as_tuple=True
    )

    joint_positive = joint[
        positive_indices
    ]

    marginal_i_positive = marginal_i[
        positive_indices[0]
    ]

    marginal_j_positive = marginal_j[
        positive_indices[1]
    ]

    if torch.any(
        marginal_i_positive <= 0.0
    ):
        raise RuntimeError(
            "positive joint cell has "
            "non-positive row marginal"
        )

    if torch.any(
        marginal_j_positive <= 0.0
    ):
        raise RuntimeError(
            "positive joint cell has "
            "non-positive column marginal"
        )

    mutual_information = torch.sum(
        joint_positive
        * (
            torch.log(
                joint_positive
            )
            - torch.log(
                marginal_i_positive
            )
            - torch.log(
                marginal_j_positive
            )
        )
    )

    return {
        "loss":
            -mutual_information,

        "mutual_information":
            mutual_information,

        "joint":
            joint,

        "marginal_i":
            marginal_i,

        "marginal_j":
            marginal_j,
    }


# ============================================================
# INDEPENDENT NUMPY IMPLEMENTATION OF THE SAME PAPER EQUATION
# ============================================================

def numpy_sakai_iic_reference(
    probabilities_a,
    probabilities_b,
):

    probabilities_a = np.asarray(
        probabilities_a,
        dtype=np.float64,
    )

    probabilities_b = np.asarray(
        probabilities_b,
        dtype=np.float64,
    )

    n = probabilities_a.shape[0]

    joint = (
        probabilities_a.T
        @ probabilities_b
    ) / float(n)

    marginal_i = joint.sum(
        axis=1
    )

    marginal_j = joint.sum(
        axis=0
    )

    positive = (
        joint > 0.0
    )

    rows, cols = np.nonzero(
        positive
    )

    jp = joint[
        rows,
        cols,
    ]

    mi = np.sum(
        jp
        * (
            np.log(jp)
            - np.log(
                marginal_i[rows]
            )
            - np.log(
                marginal_j[cols]
            )
        )
    )

    return {
        "loss":
            float(-mi),

        "mutual_information":
            float(mi),

        "joint":
            joint,
    }


# ============================================================
# TEST 1 — FIXED NUMPY / TORCH REFERENCE
# ============================================================

fixed_a = np.asarray([
    [0.60, 0.30, 0.10],
    [0.20, 0.50, 0.30],
    [0.10, 0.20, 0.70],
    [0.55, 0.15, 0.30],
], dtype=np.float64)


fixed_b = np.asarray([
    [0.50, 0.35, 0.15],
    [0.25, 0.45, 0.30],
    [0.05, 0.25, 0.70],
    [0.40, 0.20, 0.40],
], dtype=np.float64)


torch_fixed_a = torch.tensor(
    fixed_a,
    dtype=torch.float64,
)

torch_fixed_b = torch.tensor(
    fixed_b,
    dtype=torch.float64,
)


torch_reference = (
    sakai_iic_negative_mutual_information(
        torch_fixed_a,
        torch_fixed_b,
    )
)


numpy_reference = (
    numpy_sakai_iic_reference(
        fixed_a,
        fixed_b,
    )
)


numpy_reference_gate = (
    np.allclose(
        torch_reference[
            "joint"
        ].detach().cpu().numpy(),
        numpy_reference[
            "joint"
        ],
        rtol=0.0,
        atol=1.0e-14,
    )

    and math.isclose(
        float(
            torch_reference[
                "loss"
            ].item()
        ),
        numpy_reference[
            "loss"
        ],
        rel_tol=0.0,
        abs_tol=1.0e-14,
    )
)

assert numpy_reference_gate


# ============================================================
# TEST 2 — BALANCED PERFECT AGREEMENT
#
# MI = log(4)
# ============================================================

balanced = torch.eye(
    4,
    dtype=torch.float64,
).repeat_interleave(
    2,
    dim=0,
)


balanced_result = (
    sakai_iic_negative_mutual_information(
        balanced,
        balanced,
    )
)


balanced_expected_mi = math.log(
    4.0
)


balanced_gate = math.isclose(
    float(
        balanced_result[
            "mutual_information"
        ].item()
    ),
    balanced_expected_mi,
    rel_tol=0.0,
    abs_tol=1.0e-12,
)

assert balanced_gate


# ============================================================
# TEST 3 — UNIFORM OUTPUT
#
# MI = 0
# ============================================================

uniform = torch.full(
    (
        8,
        4,
    ),
    0.25,
    dtype=torch.float64,
)


uniform_result = (
    sakai_iic_negative_mutual_information(
        uniform,
        uniform,
    )
)


uniform_gate = (
    abs(
        float(
            uniform_result[
                "mutual_information"
            ].item()
        )
    )
    < 1.0e-12
)

assert uniform_gate


# ============================================================
# TEST 4 — COMPLETE COLLAPSE
#
# MI = 0
# ============================================================

collapsed = torch.zeros(
    (
        8,
        4,
    ),
    dtype=torch.float64,
)

collapsed[
    :,
    0,
] = 1.0


collapsed_result = (
    sakai_iic_negative_mutual_information(
        collapsed,
        collapsed,
    )
)


collapsed_gate = (
    abs(
        float(
            collapsed_result[
                "mutual_information"
            ].item()
        )
    )
    < 1.0e-12
)

assert collapsed_gate


# ============================================================
# TEST 5 — NO HIDDEN SYMMETRISATION
#
# Construct an asymmetric empirical joint.
# ============================================================

asym_a = torch.tensor([
    [1.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
], dtype=torch.float64)


asym_b = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 1.0],
], dtype=torch.float64)


asym_result = (
    sakai_iic_negative_mutual_information(
        asym_a,
        asym_b,
    )
)


no_symmetrisation_gate = (
    not torch.equal(
        asym_result[
            "joint"
        ],
        asym_result[
            "joint"
        ].T,
    )
)

assert no_symmetrisation_gate


# ============================================================
# TEST 6 — VIEW SWAP
#
# Swapping views transposes P.
# Mutual information itself remains identical.
# ============================================================

swap_ab = (
    sakai_iic_negative_mutual_information(
        torch_fixed_a,
        torch_fixed_b,
    )
)


swap_ba = (
    sakai_iic_negative_mutual_information(
        torch_fixed_b,
        torch_fixed_a,
    )
)


view_swap_gate = (
    torch.allclose(
        swap_ab[
            "joint"
        ],
        swap_ba[
            "joint"
        ].T,
        rtol=0.0,
        atol=1.0e-15,
    )

    and math.isclose(
        float(
            swap_ab[
                "loss"
            ].item()
        ),
        float(
            swap_ba[
                "loss"
            ].item()
        ),
        rel_tol=0.0,
        abs_tol=1.0e-15,
    )
)

assert view_swap_gate


# ============================================================
# TEST 7 — PAIRED BATCH ORDER INVARIANCE
# ============================================================

permutation = torch.tensor(
    [3, 1, 0, 2],
    dtype=torch.int64,
)


permuted_result = (
    sakai_iic_negative_mutual_information(
        torch_fixed_a[
            permutation
        ],
        torch_fixed_b[
            permutation
        ],
    )
)


batch_order_gate = (
    torch.allclose(
        torch_reference[
            "joint"
        ],
        permuted_result[
            "joint"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )

    and math.isclose(
        float(
            torch_reference[
                "loss"
            ].item()
        ),
        float(
            permuted_result[
                "loss"
            ].item()
        ),
        rel_tol=0.0,
        abs_tol=1.0e-15,
    )
)

assert batch_order_gate


# ============================================================
# TEST 8 — JOINT NORMALIZATION FOLLOWS FROM Eq. (4)
# ============================================================

joint_sum_gate = torch.allclose(
    torch_reference[
        "joint"
    ].sum(),
    torch.tensor(
        1.0,
        dtype=torch.float64,
    ),
    rtol=0.0,
    atol=1.0e-15,
)

assert joint_sum_gate


# ============================================================
# TEST 9 — AUTOGRAD
# ============================================================

with torch.random.fork_rng():

    torch.manual_seed(
        123456
    )

    logits_a = torch.randn(
        (
            17,
            7,
        ),
        dtype=torch.float32,
        requires_grad=True,
    )

    logits_b = torch.randn(
        (
            17,
            7,
        ),
        dtype=torch.float32,
        requires_grad=True,
    )


probabilities_a = torch.softmax(
    logits_a,
    dim=1,
)

probabilities_b = torch.softmax(
    logits_b,
    dim=1,
)


autograd_result = (
    sakai_iic_negative_mutual_information(
        probabilities_a,
        probabilities_b,
    )
)


autograd_result[
    "loss"
].backward()


autograd_gate = (
    logits_a.grad
    is not None

    and logits_b.grad
    is not None

    and torch.isfinite(
        logits_a.grad
    ).all()

    and torch.isfinite(
        logits_b.grad
    ).all()

    and float(
        torch.linalg.vector_norm(
            logits_a.grad
        ).item()
    ) > 0.0

    and float(
        torch.linalg.vector_norm(
            logits_b.grad
        ).item()
    ) > 0.0
)

assert autograd_gate


# ============================================================
# TEST 10 — PAPER DIMENSIONS
# ============================================================

with torch.random.fork_rng():

    torch.manual_seed(
        136
    )

    c36_a = torch.softmax(
        torch.randn(
            13,
            36,
        ),
        dim=1,
    )

    c36_b = torch.softmax(
        torch.randn(
            13,
            36,
        ),
        dim=1,
    )

    w250_a = torch.softmax(
        torch.randn(
            13,
            250,
        ),
        dim=1,
    )

    w250_b = torch.softmax(
        torch.randn(
            13,
            250,
        ),
        dim=1,
    )


c36_result = (
    sakai_iic_negative_mutual_information(
        c36_a,
        c36_b,
    )
)

w250_result = (
    sakai_iic_negative_mutual_information(
        w250_a,
        w250_b,
    )
)


paper_dimension_gate = (
    c36_result[
        "joint"
    ].shape
    == (
        36,
        36,
    )

    and w250_result[
        "joint"
    ].shape
    == (
        250,
        250,
    )

    and torch.isfinite(
        c36_result[
            "loss"
        ]
    )

    and torch.isfinite(
        w250_result[
            "loss"
        ]
    )
)

assert paper_dimension_gate


# ============================================================
# CORRECTED CONTRACT
# ============================================================

SUPERSEDED_IIC_LOSS_CONTRACT_SHA256 = (
    "5b76d50075a068c4db581da872ef3941"
    "356c9c2fd83d06488e3cd0615e9cf4c0"
)


IIC_LOSS_CONTRACT = {

    "contract_version":
        2,

    "supersedes_sha256":
        SUPERSEDED_IIC_LOSS_CONTRACT_SHA256,

    "source":
        "Sakai_Eq4",

    "paper": {

        "joint":
            "Pij=(1/N)*sum_n Phi_i(z_n)*Phi_j(zprime_n)",

        "marginals":
            "Pi=sum_j_Pij;Pj=sum_i_Pij",

        "objective":
            "maximize_mutual_information",

        "main_output_dim_C":
            36,

        "overcluster_output_dim_W":
            250,

        "overcluster_same_equation":
            True,
    },

    "project_numerics": {

        "symmetrize_joint":
            False,

        "extra_joint_renormalization":
            False,

        "epsilon_added":
            False,

        "zero_term_policy":
            "evaluate_only_Pij_gt_0",

        "training_dtype":
            "float32",

        "known_answer_dtype":
            "float64",
    },

    "still_unresolved": [

        "main_overcluster_training_relationship",

        "epoch_count",

        "initialization",

        "classifier_seeds",

        "shuffle",

        "tail_batch",

        "checkpoint_resume",
    ],

    "labels_used":
        False,

    "parent_data_contract_sha256":
        IIC_DATA_CONTRACT_SHA256,
}


IIC_LOSS_CONTRACT_JSON = json.dumps(
    IIC_LOSS_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_LOSS_CONTRACT_SHA256 = hashlib.sha256(
    IIC_LOSS_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# FINAL GATE
# ============================================================

loss_audit_gate = all([
    numpy_reference_gate,
    balanced_gate,
    uniform_gate,
    collapsed_gate,
    no_symmetrisation_gate,
    view_swap_gate,
    batch_order_gate,
    joint_sum_gate,
    autograd_gate,
    paper_dimension_gate,
])


assert loss_audit_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 SAKAI Eq.4 LOSS AUDIT — CORRECTED ==="
)

print()

print(
    "SOURCE = Sakai_Eq4"
)

print(
    "JOINT_SYMMETRIZED =",
    False,
)

print(
    "EXTRA_JOINT_RENORMALIZATION =",
    False,
)

print(
    "ARBITRARY_EPSILON_ADDED =",
    False,
)

print(
    "ZERO_TERM_POLICY = Pij_gt_0_only"
)

print()

print(
    "NUMPY_REFERENCE_LOSS =",
    numpy_reference[
        "loss"
    ],
)

print(
    "TORCH_REFERENCE_LOSS =",
    float(
        torch_reference[
            "loss"
        ].item()
    ),
)

print()

print(
    "BALANCED_MI =",
    float(
        balanced_result[
            "mutual_information"
        ].item()
    ),
)

print(
    "UNIFORM_MI =",
    float(
        uniform_result[
            "mutual_information"
        ].item()
    ),
)

print(
    "COLLAPSED_MI =",
    float(
        collapsed_result[
            "mutual_information"
        ].item()
    ),
)

print()

print(
    "NUMPY_REFERENCE_GATE =",
    numpy_reference_gate,
)

print(
    "BALANCED_GATE =",
    balanced_gate,
)

print(
    "UNIFORM_GATE =",
    uniform_gate,
)

print(
    "COLLAPSED_GATE =",
    collapsed_gate,
)

print(
    "NO_SYMMETRISATION_GATE =",
    no_symmetrisation_gate,
)

print(
    "VIEW_SWAP_GATE =",
    view_swap_gate,
)

print(
    "BATCH_ORDER_GATE =",
    batch_order_gate,
)

print(
    "JOINT_SUM_GATE =",
    joint_sum_gate,
)

print(
    "AUTOGRAD_GATE =",
    autograd_gate,
)

print(
    "PAPER_DIMENSION_GATE =",
    paper_dimension_gate,
)

print()

print(
    "SUPERSEDED_IIC_LOSS_CONTRACT_SHA256 =",
    SUPERSEDED_IIC_LOSS_CONTRACT_SHA256,
)

print(
    "CORRECTED_IIC_LOSS_CONTRACT_SHA256 =",
    IIC_LOSS_CONTRACT_SHA256,
)

print()

print(
    "MAIN_OVERCLUSTER_RELATIONSHIP_FROZEN =",
    False,
)

print(
    "LABELS_READ =",
    False,
)

print(
    "MODEL_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A2R_SAKAI_EQ4_LOSS_PASS =",
    loss_audit_gate,
)


=== NOTEBOOK 06 SAKAI Eq.4 LOSS AUDIT — CORRECTED ===

SOURCE = Sakai_Eq4
JOINT_SYMMETRIZED = False
EXTRA_JOINT_RENORMALIZATION = False
ARBITRARY_EPSILON_ADDED = False
ZERO_TERM_POLICY = Pij_gt_0_only

NUMPY_REFERENCE_LOSS = -0.02352875972182373
TORCH_REFERENCE_LOSS = -0.023528759721823736

BALANCED_MI = 1.3862943611198906
UNIFORM_MI = 0.0
COLLAPSED_MI = 0.0

NUMPY_REFERENCE_GATE = True
BALANCED_GATE = True
UNIFORM_GATE = True
COLLAPSED_GATE = True
NO_SYMMETRISATION_GATE = True
VIEW_SWAP_GATE = True
BATCH_ORDER_GATE = True
JOINT_SUM_GATE = True
AUTOGRAD_GATE = True
PAPER_DIMENSION_GATE = tensor(True)

SUPERSEDED_IIC_LOSS_CONTRACT_SHA256 = 5b76d50075a068c4db581da872ef3941356c9c2fd83d06488e3cd0615e9cf4c0
CORRECTED_IIC_LOSS_CONTRACT_SHA256 = b0b108727286c8202b4ebfcae987d6776a80eb3a8f993c992a5a328f785179f1

MAIN_OVERCLUSTER_RELATIONSHIP_FROZEN = False
LABELS_READ = False
MODEL_CREATED = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_06_A2R_SAKAI_EQ4

## STEP 06-A3A — Contratto dell'architettura IIC

La Figura 8 di Sakai specifica, per il classificatore IIC principale:

$$
\mu_\phi(x)\in\mathbb{R}^{512}
$$

seguito da:

$$
512
\rightarrow
FC(512)
\rightarrow
BN
\rightarrow
ReLU
\rightarrow
FC(C)
\rightarrow
SoftMax.
$$

Per il ramo principale:

$$
C=36.
$$

L'over-clustering utilizza la stessa logica con:

$$
W=250.
$$

### Separazione dei due rami

Nel progetto vengono mantenuti due rami trainabili distinti:

$$
\Phi_C
$$

e

$$
\Phi_W.
$$

Essi condividono esclusivamente l'encoder VAE congelato.

Non condividono i parametri dei rispettivi layer fully-connected o BatchNorm.

Questa è una scelta PROJECT compatibile con il ruolo di head principale e
head ausiliaria dell'IIC e rende esplicita un'informazione non completamente
specificata da Sakai.

### Batch Normalization

La notazione $B$ della Figura 8 stabilisce la presenza della Batch
Normalization, ma non ne pubblica i dettagli numerici.

Il progetto congela quindi:

$$
\epsilon_{\mathrm{BN}}=10^{-5},
$$

$$
\mathrm{momentum}=0.1,
$$

con parametri affine apprendibili e running statistics, secondo la semantica
PyTorch già utilizzata nel VAE full.

Questi valori sono PROJECT.

### Bias

Il primo fully-connected è seguito immediatamente da BatchNorm. Il progetto
usa quindi:

- `bias=False` nel layer $512\rightarrow512$;
- `bias=True` nel layer finale $512\rightarrow C$ o $512\rightarrow W$.

Questa è una scelta PROJECT e non viene attribuita a Sakai.

### Feature di ingresso

Il classificatore riceve direttamente:

$$
\mu_\phi(x)
$$

e

$$
\mu_\phi(x').
$$

Non vengono applicati:

- PCA;
- z-score;
- normalizzazione L2;
- whitening;
- altre trasformazioni dello spazio latente.

Questo preserva il passaggio diretto encoder-IIC mostrato nella Figura 8.

### Cinque classificatori

La configurazione finale utilizza:

$$
K=5
$$

classificatori indipendenti.

Ogni classificatore possiede quindi una propria coppia:

$$
(\Phi_C^{(k)},\Phi_W^{(k)}),
\qquad
k=1,\ldots,5.
$$

L'encoder VAE è invece unico, congelato e condiviso.

L'inizializzazione, i seed, la combinazione delle loss e la policy temporale
del training non vengono ancora congelati in questo step.

In [ ]:
# ============================================================
# NOTEBOOK 06 — IIC ARCHITECTURE CONTRACT
# STEP 06-A3A
#
# CONTRACT ONLY.
# NO model creation.
# NO labels.
# NO training.
# NO Drive writes.
# ============================================================

import hashlib
import json


# ============================================================
# PARENT CONTRACTS
# ============================================================

EXPECTED_IIC_DATA_CONTRACT_SHA256 = (
    "03bfab28cd2f513dd3616aca9f394633"
    "3d12d85508e95bb860f6682941002e17"
)

EXPECTED_IIC_LOSS_CONTRACT_SHA256 = (
    "b0b108727286c8202b4ebfcae987d677"
    "6a80eb3a8f993c992a5a328f785179f1"
)


assert (
    IIC_DATA_CONTRACT_SHA256
    == EXPECTED_IIC_DATA_CONTRACT_SHA256
)

assert (
    IIC_LOSS_CONTRACT_SHA256
    == EXPECTED_IIC_LOSS_CONTRACT_SHA256
)


# ============================================================
# FROZEN ARCHITECTURE
# ============================================================

IIC_ARCHITECTURE_CONTRACT = {

    "contract_version": 1,

    "input": {
        "feature": "mu",
        "dimension": 512,
        "pca": False,
        "zscore": False,
        "l2_normalization": False,
        "whitening": False,
    },

    "encoder": {
        "shared": True,
        "frozen": True,
        "eval_mode": True,
        "trainable": False,
    },

    "main_head": {
        "hidden_linear": [512, 512],
        "hidden_bias": False,
        "batch_norm": True,
        "batch_norm_eps": 1.0e-5,
        "batch_norm_momentum": 0.1,
        "batch_norm_affine": True,
        "batch_norm_running_stats": True,
        "activation": "relu",
        "output_linear": [512, 36],
        "output_bias": True,
        "output_activation": "softmax_dim1",
        "output_dim_C": 36,
    },

    "overcluster_head": {
        "hidden_linear": [512, 512],
        "hidden_bias": False,
        "batch_norm": True,
        "batch_norm_eps": 1.0e-5,
        "batch_norm_momentum": 0.1,
        "batch_norm_affine": True,
        "batch_norm_running_stats": True,
        "activation": "relu",
        "output_linear": [512, 250],
        "output_bias": True,
        "output_activation": "softmax_dim1",
        "output_dim_W": 250,
    },

    "branch_parameter_sharing": False,

    "classifier_count_K": 5,

    "independent_classifier_pairs": True,

    "paper_project_classification": {
        "input_mu512": "PAPER",
        "hidden_width_512": "PAPER",
        "batch_norm_presence": "PAPER",
        "relu": "PAPER",
        "C_36": "PAPER",
        "W_250": "PAPER",
        "K_5": "PAPER",

        "bn_eps": "PROJECT",
        "bn_momentum": "PROJECT",
        "bn_semantics": "PROJECT",
        "hidden_bias_false": "PROJECT",
        "output_bias_true": "PROJECT",
        "main_overcluster_parameter_sharing_false": "PROJECT",
    },

    "still_unresolved": [
        "parameter_initialization",
        "five_classifier_seeds",
        "main_overcluster_loss_combination",
        "epoch_count_and_stopping",
        "shuffle_policy",
        "tail_batch_policy",
        "checkpoint_resume_policy",
    ],

    "parent_data_contract_sha256":
        IIC_DATA_CONTRACT_SHA256,

    "parent_loss_contract_sha256":
        IIC_LOSS_CONTRACT_SHA256,
}


# ============================================================
# STATIC PARAMETER COUNTS
# ============================================================

MAIN_HEAD_PARAMETERS = (
    512 * 512
    + 2 * 512
    + 512 * 36
    + 36
)

OVERCLUSTER_HEAD_PARAMETERS = (
    512 * 512
    + 2 * 512
    + 512 * 250
    + 250
)

PARAMETERS_PER_CLASSIFIER = (
    MAIN_HEAD_PARAMETERS
    + OVERCLUSTER_HEAD_PARAMETERS
)

PARAMETERS_ALL_K5 = (
    5
    * PARAMETERS_PER_CLASSIFIER
)


# ============================================================
# CONTRACT SHA
# ============================================================

IIC_ARCHITECTURE_CONTRACT_JSON = json.dumps(
    IIC_ARCHITECTURE_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

IIC_ARCHITECTURE_CONTRACT_SHA256 = hashlib.sha256(
    IIC_ARCHITECTURE_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# GATES
# ============================================================

paper_shape_gate = (
    IIC_ARCHITECTURE_CONTRACT[
        "input"
    ]["dimension"] == 512

    and IIC_ARCHITECTURE_CONTRACT[
        "main_head"
    ]["hidden_linear"] == [512, 512]

    and IIC_ARCHITECTURE_CONTRACT[
        "main_head"
    ]["output_dim_C"] == 36

    and IIC_ARCHITECTURE_CONTRACT[
        "overcluster_head"
    ]["output_dim_W"] == 250

    and IIC_ARCHITECTURE_CONTRACT[
        "classifier_count_K"
    ] == 5
)


raw_mu_gate = (
    IIC_ARCHITECTURE_CONTRACT[
        "input"
    ]["feature"] == "mu"

    and IIC_ARCHITECTURE_CONTRACT[
        "input"
    ]["pca"] is False

    and IIC_ARCHITECTURE_CONTRACT[
        "input"
    ]["zscore"] is False

    and IIC_ARCHITECTURE_CONTRACT[
        "input"
    ]["l2_normalization"] is False
)


encoder_gate = (
    IIC_ARCHITECTURE_CONTRACT[
        "encoder"
    ]["frozen"] is True

    and IIC_ARCHITECTURE_CONTRACT[
        "encoder"
    ]["eval_mode"] is True

    and IIC_ARCHITECTURE_CONTRACT[
        "encoder"
    ]["trainable"] is False
)


branch_gate = (
    IIC_ARCHITECTURE_CONTRACT[
        "branch_parameter_sharing"
    ] is False

    and IIC_ARCHITECTURE_CONTRACT[
        "independent_classifier_pairs"
    ] is True
)


parameter_count_gate = (
    MAIN_HEAD_PARAMETERS == 281636

    and OVERCLUSTER_HEAD_PARAMETERS == 391418

    and PARAMETERS_PER_CLASSIFIER == 673054

    and PARAMETERS_ALL_K5 == 3365270
)


assert paper_shape_gate
assert raw_mu_gate
assert encoder_gate
assert branch_gate
assert parameter_count_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 IIC ARCHITECTURE CONTRACT ==="
)

print()

print(
    "INPUT_FEATURE = raw_mu"
)

print(
    "INPUT_DIM =",
    512,
)

print()

print(
    "MAIN_HEAD = 512 -> 512 -> BN -> ReLU -> 36 -> SoftMax"
)

print(
    "OVERCLUSTER_HEAD = 512 -> 512 -> BN -> ReLU -> 250 -> SoftMax"
)

print(
    "MAIN_OVERCLUSTER_PARAMETER_SHARING =",
    False,
)

print()

print(
    "BN_EPS_PROJECT =",
    1.0e-5,
)

print(
    "BN_MOMENTUM_PROJECT =",
    0.1,
)

print()

print(
    "CLASSIFIER_COUNT_K =",
    5,
)

print()

print(
    "MAIN_HEAD_PARAMETERS =",
    MAIN_HEAD_PARAMETERS,
)

print(
    "OVERCLUSTER_HEAD_PARAMETERS =",
    OVERCLUSTER_HEAD_PARAMETERS,
)

print(
    "PARAMETERS_PER_CLASSIFIER =",
    PARAMETERS_PER_CLASSIFIER,
)

print(
    "PARAMETERS_ALL_K5 =",
    PARAMETERS_ALL_K5,
)

print()

print(
    "PAPER_SHAPE_GATE =",
    paper_shape_gate,
)

print(
    "RAW_MU_GATE =",
    raw_mu_gate,
)

print(
    "ENCODER_GATE =",
    encoder_gate,
)

print(
    "BRANCH_GATE =",
    branch_gate,
)

print(
    "PARAMETER_COUNT_GATE =",
    parameter_count_gate,
)

print()

print(
    "IIC_ARCHITECTURE_CONTRACT_SHA256 =",
    IIC_ARCHITECTURE_CONTRACT_SHA256,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "MODEL_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A3A_ARCHITECTURE_CONTRACT_PASS =",
    (
        paper_shape_gate
        and raw_mu_gate
        and encoder_gate
        and branch_gate
        and parameter_count_gate
    ),
)


=== NOTEBOOK 06 IIC ARCHITECTURE CONTRACT ===

INPUT_FEATURE = raw_mu
INPUT_DIM = 512

MAIN_HEAD = 512 -> 512 -> BN -> ReLU -> 36 -> SoftMax
OVERCLUSTER_HEAD = 512 -> 512 -> BN -> ReLU -> 250 -> SoftMax
MAIN_OVERCLUSTER_PARAMETER_SHARING = False

BN_EPS_PROJECT = 1e-05
BN_MOMENTUM_PROJECT = 0.1

CLASSIFIER_COUNT_K = 5

MAIN_HEAD_PARAMETERS = 281636
OVERCLUSTER_HEAD_PARAMETERS = 391418
PARAMETERS_PER_CLASSIFIER = 673054
PARAMETERS_ALL_K5 = 3365270

PAPER_SHAPE_GATE = True
RAW_MU_GATE = True
ENCODER_GATE = True
BRANCH_GATE = True
PARAMETER_COUNT_GATE = True

IIC_ARCHITECTURE_CONTRACT_SHA256 = 78c3cb17c031c7fc9e21949a4be1dbad379dd1f4854a71b4a09dcefbe3d6df78

LABELS_READ = False
MODEL_CREATED = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_06_A3A_ARCHITECTURE_CONTRACT_PASS = True


## STEP 06-A3B — Probe strutturale di un classificatore IIC

Viene creato un singolo classificatore di prova composto da:

$$
\Phi_C:
512
\rightarrow
512
\rightarrow
BN
\rightarrow
ReLU
\rightarrow
36
\rightarrow
SoftMax,
$$

e:

$$
\Phi_W:
512
\rightarrow
512
\rightarrow
BN
\rightarrow
ReLU
\rightarrow
250
\rightarrow
SoftMax.
$$

Questo modello è esclusivamente diagnostico.

L'inizializzazione utilizzata in questo step è temporanea e non costituisce
l'inizializzazione ufficiale del training.

Vengono verificati:

- numero esatto dei parametri;
- shape degli output;
- normalizzazione SoftMax;
- indipendenza dei parametri dei due rami;
- aggiornamento delle running statistics BatchNorm in TRAIN;
- invariance delle running statistics in EVAL;
- determinismo del forward in EVAL.

Non vengono utilizzati dati Gravity Spy, label, encoder VAE o optimizer.

In [ ]:
# ============================================================
# NOTEBOOK 06 — IIC SINGLE-CLASSIFIER STRUCTURAL PROBE
# STEP 06-A3B
#
# Synthetic data only.
# Temporary initialization only.
#
# NO VAE loading.
# NO Gravity Spy data.
# NO labels.
# NO optimizer.
# NO training.
# NO Drive writes.
# ============================================================

import torch
import torch.nn as nn


EXPECTED_ARCHITECTURE_SHA256 = (
    "78c3cb17c031c7fc9e21949a4be1dbad"
    "379dd1f4854a71b4a09dcefbe3d6df78"
)

assert (
    IIC_ARCHITECTURE_CONTRACT_SHA256
    == EXPECTED_ARCHITECTURE_SHA256
)


# ============================================================
# IIC HEAD
# ============================================================

class IICHead(nn.Module):

    def __init__(
        self,
        output_dim,
    ):
        super().__init__()

        self.hidden = nn.Linear(
            512,
            512,
            bias=False,
        )

        self.bn = nn.BatchNorm1d(
            512,
            eps=1.0e-5,
            momentum=0.1,
            affine=True,
            track_running_stats=True,
        )

        self.relu = nn.ReLU(
            inplace=False
        )

        self.output = nn.Linear(
            512,
            output_dim,
            bias=True,
        )


    def forward(
        self,
        x,
    ):

        x = self.hidden(x)
        x = self.bn(x)
        x = self.relu(x)

        logits = self.output(x)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        return probabilities


# ============================================================
# ONE CLASSIFIER = MAIN + OVERCLUSTER
# ============================================================

class IICClassifierPair(nn.Module):

    def __init__(self):
        super().__init__()

        self.main_head = IICHead(
            36
        )

        self.overcluster_head = IICHead(
            250
        )


    def forward(
        self,
        x,
    ):

        return {
            "main":
                self.main_head(x),

            "overcluster":
                self.overcluster_head(x),
        }


# ============================================================
# TEMPORARY PROBE INITIALIZATION
#
# This seed is NOT an official training seed.
# fork_rng avoids changing the global RNG state permanently.
# ============================================================

with torch.random.fork_rng():

    torch.manual_seed(
        314159
    )

    probe_classifier = (
        IICClassifierPair()
    )


# ============================================================
# PARAMETER COUNTS
# ============================================================

main_parameters_observed = sum(
    p.numel()
    for p
    in probe_classifier.main_head.parameters()
)


overcluster_parameters_observed = sum(
    p.numel()
    for p
    in probe_classifier.overcluster_head.parameters()
)


classifier_parameters_observed = sum(
    p.numel()
    for p
    in probe_classifier.parameters()
)


parameter_count_gate = (
    main_parameters_observed
    == 281636

    and overcluster_parameters_observed
    == 391418

    and classifier_parameters_observed
    == 673054
)


assert parameter_count_gate


# ============================================================
# PARAMETER / BUFFER INDEPENDENCE
# ============================================================

main_parameter_ptrs = {
    p.data_ptr()
    for p
    in probe_classifier.main_head.parameters()
}


overcluster_parameter_ptrs = {
    p.data_ptr()
    for p
    in probe_classifier.overcluster_head.parameters()
}


parameter_independence_gate = (
    main_parameter_ptrs.isdisjoint(
        overcluster_parameter_ptrs
    )
)


main_buffer_ptrs = {
    b.data_ptr()
    for b
    in probe_classifier.main_head.buffers()
    if b.numel() > 0
}


overcluster_buffer_ptrs = {
    b.data_ptr()
    for b
    in probe_classifier.overcluster_head.buffers()
    if b.numel() > 0
}


buffer_independence_gate = (
    main_buffer_ptrs.isdisjoint(
        overcluster_buffer_ptrs
    )
)


assert parameter_independence_gate
assert buffer_independence_gate


# ============================================================
# SYNTHETIC INPUT
# ============================================================

probe_generator = torch.Generator(
    device="cpu"
)

probe_generator.manual_seed(
    271828
)


probe_input_train = torch.randn(
    (
        128,
        512,
    ),
    generator=probe_generator,
    dtype=torch.float32,
)


probe_input_eval = torch.randn(
    (
        17,
        512,
    ),
    generator=probe_generator,
    dtype=torch.float32,
)


# ============================================================
# TRAIN-MODE STRUCTURAL FORWARD
#
# no_grad means this is NOT training.
# BatchNorm running statistics still update in train mode.
# ============================================================

probe_classifier.train()


main_running_mean_before = (
    probe_classifier
    .main_head
    .bn
    .running_mean
    .clone()
)


over_running_mean_before = (
    probe_classifier
    .overcluster_head
    .bn
    .running_mean
    .clone()
)


with torch.no_grad():

    train_outputs = probe_classifier(
        probe_input_train
    )


main_train = train_outputs[
    "main"
]

over_train = train_outputs[
    "overcluster"
]


shape_gate = (
    main_train.shape
    == (
        128,
        36,
    )

    and over_train.shape
    == (
        128,
        250,
    )
)


softmax_gate = (
    torch.isfinite(
        main_train
    ).all()

    and torch.isfinite(
        over_train
    ).all()

    and torch.all(
        main_train >= 0.0
    )

    and torch.all(
        over_train >= 0.0
    )

    and torch.allclose(
        main_train.sum(
            dim=1
        ),
        torch.ones(
            128
        ),
        rtol=1.0e-6,
        atol=1.0e-6,
    )

    and torch.allclose(
        over_train.sum(
            dim=1
        ),
        torch.ones(
            128
        ),
        rtol=1.0e-6,
        atol=1.0e-6,
    )
)


bn_train_update_gate = (
    not torch.equal(
        main_running_mean_before,
        probe_classifier
        .main_head
        .bn
        .running_mean,
    )

    and not torch.equal(
        over_running_mean_before,
        probe_classifier
        .overcluster_head
        .bn
        .running_mean,
    )
)


assert shape_gate
assert softmax_gate
assert bn_train_update_gate


# ============================================================
# EVAL-MODE CONTRACT
# ============================================================

probe_classifier.eval()


main_eval_mean_before = (
    probe_classifier
    .main_head
    .bn
    .running_mean
    .clone()
)


over_eval_mean_before = (
    probe_classifier
    .overcluster_head
    .bn
    .running_mean
    .clone()
)


with torch.no_grad():

    eval_outputs_1 = probe_classifier(
        probe_input_eval
    )

    eval_outputs_2 = probe_classifier(
        probe_input_eval
    )


bn_eval_immutability_gate = (
    torch.equal(
        main_eval_mean_before,
        probe_classifier
        .main_head
        .bn
        .running_mean,
    )

    and torch.equal(
        over_eval_mean_before,
        probe_classifier
        .overcluster_head
        .bn
        .running_mean,
    )
)


eval_determinism_gate = (
    torch.equal(
        eval_outputs_1[
            "main"
        ],
        eval_outputs_2[
            "main"
        ],
    )

    and torch.equal(
        eval_outputs_1[
            "overcluster"
        ],
        eval_outputs_2[
            "overcluster"
        ],
    )
)


eval_shape_gate = (
    eval_outputs_1[
        "main"
    ].shape
    == (
        17,
        36,
    )

    and eval_outputs_1[
        "overcluster"
    ].shape
    == (
        17,
        250,
    )
)


assert bn_eval_immutability_gate
assert eval_determinism_gate
assert eval_shape_gate


# ============================================================
# FINAL GATE
# ============================================================

probe_gate = all([
    parameter_count_gate,
    parameter_independence_gate,
    buffer_independence_gate,
    shape_gate,
    softmax_gate,
    bn_train_update_gate,
    bn_eval_immutability_gate,
    eval_determinism_gate,
    eval_shape_gate,
])


assert probe_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 IIC CLASSIFIER STRUCTURAL PROBE ==="
)

print()

print(
    "MAIN_PARAMETERS =",
    main_parameters_observed,
)

print(
    "OVERCLUSTER_PARAMETERS =",
    overcluster_parameters_observed,
)

print(
    "CLASSIFIER_PAIR_PARAMETERS =",
    classifier_parameters_observed,
)

print()

print(
    "TRAIN_MAIN_SHAPE =",
    tuple(
        main_train.shape
    ),
)

print(
    "TRAIN_OVERCLUSTER_SHAPE =",
    tuple(
        over_train.shape
    ),
)

print()

print(
    "MAIN_SOFTMAX_MAX_ROW_ERROR =",
    float(
        torch.max(
            torch.abs(
                main_train.sum(dim=1)
                - 1.0
            )
        ).item()
    ),
)

print(
    "OVERCLUSTER_SOFTMAX_MAX_ROW_ERROR =",
    float(
        torch.max(
            torch.abs(
                over_train.sum(dim=1)
                - 1.0
            )
        ).item()
    ),
)

print()

print(
    "PARAMETER_COUNT_GATE =",
    parameter_count_gate,
)

print(
    "PARAMETER_INDEPENDENCE_GATE =",
    parameter_independence_gate,
)

print(
    "BUFFER_INDEPENDENCE_GATE =",
    buffer_independence_gate,
)

print(
    "SHAPE_GATE =",
    shape_gate,
)

print(
    "SOFTMAX_GATE =",
    softmax_gate,
)

print(
    "BN_TRAIN_UPDATE_GATE =",
    bn_train_update_gate,
)

print(
    "BN_EVAL_IMMUTABILITY_GATE =",
    bn_eval_immutability_gate,
)

print(
    "EVAL_DETERMINISM_GATE =",
    eval_determinism_gate,
)

print()

print(
    "PROBE_INITIALIZATION = temporary_pytorch_default"
)

print(
    "OFFICIAL_INITIALIZATION_FROZEN =",
    False,
)

print(
    "OFFICIAL_CLASSIFIER_SEEDS_FROZEN =",
    False,
)

print(
    "MAIN_OVERCLUSTER_LOSS_COMBINATION_FROZEN =",
    False,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "GRAVITY_SPY_DATA_READ =",
    False,
)

print(
    "VAE_ENCODER_LOADED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A3B_STRUCTURAL_PROBE_PASS =",
    probe_gate,
)


=== NOTEBOOK 06 IIC CLASSIFIER STRUCTURAL PROBE ===

MAIN_PARAMETERS = 281636
OVERCLUSTER_PARAMETERS = 391418
CLASSIFIER_PAIR_PARAMETERS = 673054

TRAIN_MAIN_SHAPE = (128, 36)
TRAIN_OVERCLUSTER_SHAPE = (128, 250)

MAIN_SOFTMAX_MAX_ROW_ERROR = 1.1920928955078125e-07
OVERCLUSTER_SOFTMAX_MAX_ROW_ERROR = 1.7881393432617188e-07

PARAMETER_COUNT_GATE = True
PARAMETER_INDEPENDENCE_GATE = True
BUFFER_INDEPENDENCE_GATE = True
SHAPE_GATE = True
SOFTMAX_GATE = True
BN_TRAIN_UPDATE_GATE = True
BN_EVAL_IMMUTABILITY_GATE = True
EVAL_DETERMINISM_GATE = True

PROBE_INITIALIZATION = temporary_pytorch_default
OFFICIAL_INITIALIZATION_FROZEN = False
OFFICIAL_CLASSIFIER_SEEDS_FROZEN = False
MAIN_OVERCLUSTER_LOSS_COMBINATION_FROZEN = False

LABELS_READ = False
GRAVITY_SPY_DATA_READ = False
VAE_ENCODER_LOADED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_06_A3B_STRUCTURAL_PROBE_PASS = True


## STEP 06-A3C — Inizializzazione, seed e relazione main/over-clustering

Sakai specifica:

$$
C=36,
\qquad
W=250,
$$

e afferma che l'auxiliary over-clustering viene utilizzato nel calcolo della
mutual information con la stessa formulazione dell'Eq. (4).

Il paper non specifica però:

- se le due loss siano alternate;
- se siano ottimizzate con optimizer distinti;
- il loro peso relativo;
- uno scheduling specifico.

### Decisione PROJECT

Per evitare di introdurre uno scheduling non pubblicato, il progetto utilizza
entrambe le componenti nello stesso train step:

$$
\mathcal L_{\mathrm{total}}
=
\mathcal L_C
+
\mathcal L_W.
$$

I due termini hanno quindi peso unitario:

$$
\lambda_C=1,
\qquad
\lambda_W=1.
$$

Questa scelta è PROJECT e non viene attribuita a Sakai.

Le due head rimangono indipendenti nei parametri ma vengono aggiornate dallo
stesso optimizer Adam.

### Inizializzazione e cinque classificatori

Sakai dichiara che il clustering dipende dai valori iniziali casuali delle
reti e utilizza:

$$
K=5
$$

classificatori.

La distribuzione esatta dell'inizializzazione e i seed non sono pubblicati.

Il progetto congela quindi:

- inizializzazione nativa PyTorch dei layer `Linear`;
- inizializzazione standard PyTorch della BatchNorm;
- cinque seed indipendenti:

$$
123,\ 124,\ 125,\ 126,\ 127.
$$

Questi sono parametri PROJECT di riproducibilità.

Il numero di epoche, shuffle, tail batch e checkpoint vengono congelati nello
step successivo.

### Limite di fedeltà del rapporto main/over-clustering

Il paper conferma l'utilizzo dell'auxiliary over-clustering e specifica che
esso utilizza la stessa formulazione di mutual information dell'Eq. (4) con
$W>C$.

Non viene tuttavia pubblicata la modalità esatta con cui i due obiettivi
vengono combinati durante l'ottimizzazione.

La scelta

$$
\mathcal L_{\mathrm{total}}
=
\mathcal L_C+\mathcal L_W
$$

con pesi unitari e aggiornamento simultaneo è pertanto una scelta PROJECT.

È stata congelata prima del training e senza consultare le label, come
completamento operativo minimale della parte non specificata dal paper.

In [ ]:
# ============================================================
# NOTEBOOK 06 — IIC INITIALIZATION / LOSS-COMBINATION CONTRACT
# STEP 06-A3C
#
# CONTRACT ONLY.
# NO Gravity Spy data.
# NO labels.
# NO optimizer step.
# NO training.
# NO Drive writes.
# ============================================================

import hashlib
import json
import torch


# ============================================================
# PARENT CONTRACTS
# ============================================================

EXPECTED_LOSS_SHA256 = (
    "b0b108727286c8202b4ebfcae987d677"
    "6a80eb3a8f993c992a5a328f785179f1"
)

EXPECTED_ARCHITECTURE_SHA256 = (
    "78c3cb17c031c7fc9e21949a4be1dbad"
    "379dd1f4854a71b4a09dcefbe3d6df78"
)

assert (
    IIC_LOSS_CONTRACT_SHA256
    == EXPECTED_LOSS_SHA256
)

assert (
    IIC_ARCHITECTURE_CONTRACT_SHA256
    == EXPECTED_ARCHITECTURE_SHA256
)


# ============================================================
# PROJECT SEEDS
# ============================================================

IIC_CLASSIFIER_SEEDS = [
    123,
    124,
    125,
    126,
    127,
]

assert len(IIC_CLASSIFIER_SEEDS) == 5
assert len(set(IIC_CLASSIFIER_SEEDS)) == 5


# ============================================================
# LOSS COMBINATION
# ============================================================

IIC_MAIN_WEIGHT = 1.0
IIC_OVERCLUSTER_WEIGHT = 1.0


def combine_iic_losses(
    main_loss,
    overcluster_loss,
):

    return (
        IIC_MAIN_WEIGHT
        * main_loss

        + IIC_OVERCLUSTER_WEIGHT
        * overcluster_loss
    )


# Synthetic known-answer only.
synthetic_main_loss = torch.tensor(
    -0.25,
    dtype=torch.float64,
)

synthetic_over_loss = torch.tensor(
    -0.40,
    dtype=torch.float64,
)

synthetic_total_loss = combine_iic_losses(
    synthetic_main_loss,
    synthetic_over_loss,
)


loss_combination_gate = (
    float(
        synthetic_total_loss.item()
    )
    == -0.65
)

assert loss_combination_gate


# ============================================================
# REPRODUCIBLE INITIALIZATION
# ============================================================

def build_iic_pair_from_seed(seed):

    with torch.random.fork_rng():

        torch.manual_seed(
            seed
        )

        model = IICClassifierPair()

    return model


def module_state_sha256(module):

    digest = hashlib.sha256()

    for name, tensor in sorted(
        module.state_dict().items()
    ):

        tensor = (
            tensor
            .detach()
            .cpu()
            .contiguous()
        )

        digest.update(
            name.encode("utf-8")
        )

        digest.update(
            str(
                tensor.dtype
            ).encode("ascii")
        )

        digest.update(
            str(
                tuple(
                    tensor.shape
                )
            ).encode("ascii")
        )

        digest.update(
            tensor.numpy().tobytes(
                order="C"
            )
        )

    return digest.hexdigest()


initial_state_hashes = []


for seed in IIC_CLASSIFIER_SEEDS:

    model_a = build_iic_pair_from_seed(
        seed
    )

    model_b = build_iic_pair_from_seed(
        seed
    )

    sha_a = module_state_sha256(
        model_a
    )

    sha_b = module_state_sha256(
        model_b
    )

    assert sha_a == sha_b

    initial_state_hashes.append(
        sha_a
    )


seed_reproducibility_gate = (
    len(
        initial_state_hashes
    ) == 5
)


seed_independence_gate = (
    len(
        set(
            initial_state_hashes
        )
    ) == 5
)


assert seed_reproducibility_gate
assert seed_independence_gate


# ============================================================
# BATCHNORM INITIALIZATION
# ============================================================

reference_pair = build_iic_pair_from_seed(
    IIC_CLASSIFIER_SEEDS[0]
)


def bn_initial_state_ok(bn):

    return (
        torch.equal(
            bn.weight,
            torch.ones_like(
                bn.weight
            )
        )

        and torch.equal(
            bn.bias,
            torch.zeros_like(
                bn.bias
            )
        )

        and torch.equal(
            bn.running_mean,
            torch.zeros_like(
                bn.running_mean
            )
        )

        and torch.equal(
            bn.running_var,
            torch.ones_like(
                bn.running_var
            )
        )

        and int(
            bn.num_batches_tracked.item()
        ) == 0
    )


bn_initialization_gate = (
    bn_initial_state_ok(
        reference_pair.main_head.bn
    )

    and bn_initial_state_ok(
        reference_pair.overcluster_head.bn
    )
)


assert bn_initialization_gate


# ============================================================
# PARAMETER COUNT
# ============================================================

parameter_count_gate = all(

    sum(
        p.numel()
        for p
        in build_iic_pair_from_seed(
            seed
        ).parameters()
    )
    == 673054

    for seed
    in IIC_CLASSIFIER_SEEDS
)


assert parameter_count_gate


# ============================================================
# FROZEN CONTRACT
# ============================================================

IIC_TRAINING_CORE_CONTRACT = {

    "contract_version":
        1,

    "main_overcluster_relationship": {

        "mode":
            "simultaneous_equal_weight_sum",

        "total_loss":
            "L_main + L_overcluster",

        "main_weight":
            1.0,

        "overcluster_weight":
            1.0,

        "alternating_schedule":
            False,

        "classification":
            "PROJECT",
    },

    "initialization": {

        "linear":
            "native_PyTorch_reset_parameters",

        "batchnorm":
            "native_PyTorch",

        "classification":
            "PROJECT",
    },

    "classifier_seeds": {

        "values":
            IIC_CLASSIFIER_SEEDS,

        "count":
            5,

        "classification":
            "PROJECT",
    },

    "paper_fixed": {

        "C":
            36,

        "W":
            250,

        "K":
            5,
    },

    "still_unresolved": [

        "epoch_count",

        "shuffle_policy",

        "tail_batch_policy",

        "Adam_construction",

        "checkpoint_resume_policy",
    ],

    "parent_loss_sha256":
        IIC_LOSS_CONTRACT_SHA256,

    "parent_architecture_sha256":
        IIC_ARCHITECTURE_CONTRACT_SHA256,
}


IIC_TRAINING_CORE_JSON = json.dumps(
    IIC_TRAINING_CORE_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_TRAINING_CORE_SHA256 = hashlib.sha256(
    IIC_TRAINING_CORE_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# FINAL GATE
# ============================================================

a3c_gate = all([
    loss_combination_gate,
    seed_reproducibility_gate,
    seed_independence_gate,
    bn_initialization_gate,
    parameter_count_gate,
])


assert a3c_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 IIC TRAINING CORE CONTRACT ==="
)

print()

print(
    "MAIN_OVERCLUSTER_MODE = simultaneous_equal_weight_sum"
)

print(
    "MAIN_WEIGHT =",
    IIC_MAIN_WEIGHT,
)

print(
    "OVERCLUSTER_WEIGHT =",
    IIC_OVERCLUSTER_WEIGHT,
)

print(
    "ALTERNATING_SCHEDULE =",
    False,
)

print()

print(
    "CLASSIFIER_SEEDS =",
    IIC_CLASSIFIER_SEEDS,
)

print()

for index, digest in enumerate(
    initial_state_hashes
):

    print(
        f"CLASSIFIER_{index}_INITIAL_STATE_SHA256 =",
        digest,
    )


print()

print(
    "LOSS_COMBINATION_GATE =",
    loss_combination_gate,
)

print(
    "SEED_REPRODUCIBILITY_GATE =",
    seed_reproducibility_gate,
)

print(
    "SEED_INDEPENDENCE_GATE =",
    seed_independence_gate,
)

print(
    "BN_INITIALIZATION_GATE =",
    bn_initialization_gate,
)

print(
    "PARAMETER_COUNT_GATE =",
    parameter_count_gate,
)

print()

print(
    "IIC_TRAINING_CORE_SHA256 =",
    IIC_TRAINING_CORE_SHA256,
)

print()

print(
    "EPOCH_COUNT_FROZEN =",
    False,
)

print(
    "SHUFFLE_POLICY_FROZEN =",
    False,
)

print(
    "TAIL_BATCH_POLICY_FROZEN =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "LABELS_READ =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A3C_TRAINING_CORE_PASS =",
    a3c_gate,
)


=== NOTEBOOK 06 IIC TRAINING CORE CONTRACT ===

MAIN_OVERCLUSTER_MODE = simultaneous_equal_weight_sum
MAIN_WEIGHT = 1.0
OVERCLUSTER_WEIGHT = 1.0
ALTERNATING_SCHEDULE = False

CLASSIFIER_SEEDS = [123, 124, 125, 126, 127]

CLASSIFIER_0_INITIAL_STATE_SHA256 = f32c9bf96502452e9fc8d1b4f568b1f73267484afc7d200c11a42ba254505e44
CLASSIFIER_1_INITIAL_STATE_SHA256 = 4612177ef9739e79a342cadf192bdd6293f90a055b3a6740f28d469f124d677e
CLASSIFIER_2_INITIAL_STATE_SHA256 = d594b6df5360d75bc3f3befb4bdee28ab7d13b1df56a2a1b5208ee296666c1fe
CLASSIFIER_3_INITIAL_STATE_SHA256 = e170fbed391f079cbf5951684bc3f6f1ff370eaa0fef6199dc2874bbc3cefa7c
CLASSIFIER_4_INITIAL_STATE_SHA256 = 82434e04c0678153d52346c30ce03c0220e7011be476622e85363e8c11835cd7

LOSS_COMBINATION_GATE = True
SEED_REPRODUCIBILITY_GATE = True
SEED_INDEPENDENCE_GATE = True
BN_INITIALIZATION_GATE = True
PARAMETER_COUNT_GATE = True

IIC_TRAINING_CORE_SHA256 = 6f8cdcd5518ac41a59dca08ae33e7024f318575e28a190f2d49a0269642929c0

EPOCH_COUNT_FROZEN = False
S

## STEP 06-A3C-R — Riallineamento finale di provenienza e handoff C11

L'audit del notebook completo conferma la validità numerica degli step
precedenti, ma precisa la provenienza di alcuni dettagli non pubblicati.

Sono classificati `PAPER`:

$$
I
=
\sum_{i,j}
P_{ij}
\ln
\left(
\frac{P_{ij}}{P_iP_j}
\right),
$$

l'uso dello stesso obiettivo per l'over-clustering con:

$$
W>C,
$$

l'encoder VAE pre-addestrato, i pesi condivisi fra le due viste e i valori:

$$
C=36,
\qquad
W=250,
\qquad
K=5.
$$

È invece classificato `PROJECT` lo stimatore empirico minibatch:

$$
\widehat P_{ij}
=
\frac{1}{B}
\sum_{b=1}^{B}
p_{b,i}q_{b,j}.
$$

Sono inoltre `PROJECT`:

- assenza di simmetrizzazione e rinormalizzazione aggiuntiva;
- gestione numerica dei termini nulli;
- encoder congelato in EVAL e senza gradienti;
- head principale e over-clustering separate;
- somma simultanea delle due loss con pesi unitari;
- inizializzazione PyTorch e seed.

La formulazione Python resta il riferimento matematico e numerico per le
future ablation C11. La corrispondenza fra Python e C11 verrà verificata con
golden vector; non sarà richiesta identità bitwise fra i rispettivi generatori
casuali.

In [ ]:
# ============================================================
# STEP 06-A3C-R — FINAL PROVENANCE / C11 REALIGNMENT
#
# Run immediately after 06-A3C.
# No data, labels, optimizer step, training, or Drive writes.
# ============================================================

import copy
import hashlib
import json


OLD_LOSS_SHA = (
    "b0b108727286c8202b4ebfcae987d677"
    "6a80eb3a8f993c992a5a328f785179f1"
)

OLD_ARCH_SHA = (
    "78c3cb17c031c7fc9e21949a4be1dbad"
    "379dd1f4854a71b4a09dcefbe3d6df78"
)

OLD_CORE_SHA = (
    "6f8cdcd5518ac41a59dca08ae33e7024"
    "f318575e28a190f2d49a0269642929c0"
)


assert IIC_LOSS_CONTRACT_SHA256 == OLD_LOSS_SHA
assert IIC_ARCHITECTURE_CONTRACT_SHA256 == OLD_ARCH_SHA
assert IIC_TRAINING_CORE_SHA256 == OLD_CORE_SHA

assert loss_audit_gate
assert probe_gate
assert a3c_gate


# Future cells use names that do not attribute the empirical
# minibatch estimator directly to Sakai.
project_empirical_joint_probability = (
    sakai_joint_probability
)

project_iic_negative_mutual_information = (
    sakai_iic_negative_mutual_information
)

numpy_project_iic_reference = (
    numpy_sakai_iic_reference
)

paper_dimension_gate = bool(
    paper_dimension_gate
)


# ============================================================
# LOSS CONTRACT V3
# ============================================================

IIC_LOSS_CONTRACT = {

    "contract_version":
        3,

    "supersedes_sha256":
        OLD_LOSS_SHA,

    "source_hierarchy": [
        "PAPER:Sakai_Eq4_mutual_information_objective",
        (
            "PAPER:overclustering_uses_same_objective_"
            "with_W_greater_than_C"
        ),
        "PROJECT:paired_minibatch_empirical_joint_estimator",
        "PROJECT:floating_point_zero_handling",
    ],

    "paper": {

        "objective":
            "maximize_mutual_information",

        "mutual_information":
            "sum_ij Pij*ln(Pij/(Pi*Pj))",

        "main_output_dim_C":
            36,

        "overcluster_output_dim_W":
            250,

        "overcluster_same_objective":
            True,

        "empirical_minibatch_estimator_explicitly_published":
            False,
    },

    "project_estimator": {

        "joint":
            "P_hat=(1/B)*sum_b p_b*q_b^T",

        "paired_rows":
            True,

        "symmetrize_joint":
            False,

        "extra_joint_renormalization":
            False,

        "epsilon_added":
            False,

        "zero_term_policy":
            "evaluate_only_P_hat_ij_gt_0",

        "training_dtype":
            "float32",

        "known_answer_dtype":
            "float64",
    },

    "future_C11_reference": {

        "same_equation_required":
            True,

        "same_project_estimator_required":
            True,

        "golden_vector_validation_required":
            True,

        "cross_backend_bitwise_identity_required":
            False,
    },

    "still_unresolved": [
        "main_overcluster_training_relationship",
        "epoch_count",
        "initialization",
        "classifier_seeds",
        "shuffle",
        "tail_batch",
        "checkpoint_resume",
    ],

    "labels_used":
        False,

    "parent_data_contract_sha256":
        IIC_DATA_CONTRACT_SHA256,
}


IIC_LOSS_CONTRACT_JSON = json.dumps(
    IIC_LOSS_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

IIC_LOSS_CONTRACT_SHA256 = hashlib.sha256(
    IIC_LOSS_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


assert IIC_LOSS_CONTRACT_SHA256 == (
    "2bf9cba7d23de553cbad1d876b1f1b14"
    "3ede63154ac20bbb29ca092223c5e7ac"
)


# ============================================================
# ARCHITECTURE CONTRACT V2
# ============================================================

IIC_ARCHITECTURE_CONTRACT = copy.deepcopy(
    IIC_ARCHITECTURE_CONTRACT
)

IIC_ARCHITECTURE_CONTRACT[
    "contract_version"
] = 2

IIC_ARCHITECTURE_CONTRACT[
    "supersedes_sha256"
] = OLD_ARCH_SHA


IIC_ARCHITECTURE_CONTRACT[
    "encoder"
].update({

    "pretrained_encoder_classification":
        "PAPER",

    "shared_weights_between_views_classification":
        "PAPER",

    "frozen_eval_no_grad_classification":
        "PROJECT",
})


IIC_ARCHITECTURE_CONTRACT[
    "paper_project_classification"
].update({

    "pretrained_encoder":
        "PAPER",

    "shared_encoder_weights_between_views":
        "PAPER",

    "encoder_frozen_eval_no_grad":
        "PROJECT",

    "overcluster_head_mirrors_main_except_output_dim":
        "PROJECT",
})


IIC_ARCHITECTURE_CONTRACT[
    "parent_loss_contract_sha256"
] = IIC_LOSS_CONTRACT_SHA256


IIC_ARCHITECTURE_CONTRACT_JSON = json.dumps(
    IIC_ARCHITECTURE_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

IIC_ARCHITECTURE_CONTRACT_SHA256 = hashlib.sha256(
    IIC_ARCHITECTURE_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


assert IIC_ARCHITECTURE_CONTRACT_SHA256 == (
    "12fd00c3dddc6474249a86899b14797c"
    "c891e815126cfb6989eb96e489d5e758"
)


# ============================================================
# TRAINING-CORE CONTRACT V2 / C11 HANDOFF
# ============================================================

IIC_TRAINING_CORE_CONTRACT = copy.deepcopy(
    IIC_TRAINING_CORE_CONTRACT
)

IIC_TRAINING_CORE_CONTRACT[
    "contract_version"
] = 2

IIC_TRAINING_CORE_CONTRACT[
    "supersedes_sha256"
] = OLD_CORE_SHA


IIC_TRAINING_CORE_CONTRACT[
    "initialization"
] = {

    "python_realization":
        "native_PyTorch_reset_parameters",

    "batchnorm_gamma":
        1.0,

    "batchnorm_beta":
        0.0,

    "classification":
        "PROJECT",

    "future_C11_validation": {

        "same_seed_identifiers":
            True,

        "loss_golden_vectors_required":
            True,

        "head_forward_golden_vectors_required":
            True,

        "cross_backend_bitwise_RNG_identity_required":
            False,

        "C11_initialization_contract_to_be_frozen_before_mini_training":
            True,
    },
}


IIC_TRAINING_CORE_CONTRACT[
    "parent_loss_sha256"
] = IIC_LOSS_CONTRACT_SHA256

IIC_TRAINING_CORE_CONTRACT[
    "parent_architecture_sha256"
] = IIC_ARCHITECTURE_CONTRACT_SHA256


IIC_TRAINING_CORE_JSON = json.dumps(
    IIC_TRAINING_CORE_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

IIC_TRAINING_CORE_SHA256 = hashlib.sha256(
    IIC_TRAINING_CORE_JSON.encode(
        "utf-8"
    )
).hexdigest()


assert IIC_TRAINING_CORE_SHA256 == (
    "f1df0a78746739c7552668e0ed0cac29"
    "5b8d61d772088cb7893a85d6ae6bfe0a"
)


# ============================================================
# FINAL GATES
# ============================================================

provenance_gate = (

    IIC_LOSS_CONTRACT[
        "paper"
    ][
        "empirical_minibatch_estimator_explicitly_published"
    ]
    is False

    and IIC_LOSS_CONTRACT[
        "project_estimator"
    ][
        "joint"
    ]
    == "P_hat=(1/B)*sum_b p_b*q_b^T"

    and IIC_ARCHITECTURE_CONTRACT[
        "encoder"
    ][
        "frozen_eval_no_grad_classification"
    ]
    == "PROJECT"
)


c11_handoff_gate = (

    IIC_LOSS_CONTRACT[
        "future_C11_reference"
    ][
        "golden_vector_validation_required"
    ]
    is True

    and IIC_TRAINING_CORE_CONTRACT[
        "initialization"
    ][
        "future_C11_validation"
    ][
        "head_forward_golden_vectors_required"
    ]
    is True

    and IIC_TRAINING_CORE_CONTRACT[
        "initialization"
    ][
        "future_C11_validation"
    ][
        "cross_backend_bitwise_RNG_identity_required"
    ]
    is False
)


final_realignment_gate = (

    provenance_gate
    and c11_handoff_gate
    and paper_dimension_gate
    and loss_audit_gate
    and probe_gate
    and a3c_gate
)


assert final_realignment_gate


print()
print(
    "=== NOTEBOOK 06 FINAL PROVENANCE / C11 REALIGNMENT ==="
)
print()

print(
    "EMPIRICAL_MINIBATCH_JOINT_CLASSIFICATION = PROJECT"
)

print(
    "ENCODER_FROZEN_EVAL_NO_GRAD_CLASSIFICATION = PROJECT"
)

print(
    "EQ4_MI_OBJECTIVE_CLASSIFICATION = PAPER"
)

print(
    "OVERCLUSTER_SAME_OBJECTIVE_W_GT_C_CLASSIFICATION = PAPER"
)

print()

print(
    "FINAL_IIC_LOSS_CONTRACT_SHA256 =",
    IIC_LOSS_CONTRACT_SHA256,
)

print(
    "FINAL_IIC_ARCHITECTURE_CONTRACT_SHA256 =",
    IIC_ARCHITECTURE_CONTRACT_SHA256,
)

print(
    "FINAL_IIC_TRAINING_CORE_SHA256 =",
    IIC_TRAINING_CORE_SHA256,
)

print()

print(
    "PROVENANCE_GATE =",
    provenance_gate,
)

print(
    "C11_HANDOFF_GATE =",
    c11_handoff_gate,
)

print(
    "PAPER_DIMENSION_GATE =",
    paper_dimension_gate,
)

print()

print(
    "NUMERICAL_TESTS_CHANGED =",
    False,
)

print(
    "MODEL_STATE_CHANGED =",
    False,
)

print(
    "LABELS_READ =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A3C_R_FINAL_REALIGNMENT_PASS =",
    final_realignment_gate,
)


=== NOTEBOOK 06 FINAL PROVENANCE / C11 REALIGNMENT ===

EMPIRICAL_MINIBATCH_JOINT_CLASSIFICATION = PROJECT
ENCODER_FROZEN_EVAL_NO_GRAD_CLASSIFICATION = PROJECT
EQ4_MI_OBJECTIVE_CLASSIFICATION = PAPER
OVERCLUSTER_SAME_OBJECTIVE_W_GT_C_CLASSIFICATION = PAPER

FINAL_IIC_LOSS_CONTRACT_SHA256 = 2bf9cba7d23de553cbad1d876b1f1b143ede63154ac20bbb29ca092223c5e7ac
FINAL_IIC_ARCHITECTURE_CONTRACT_SHA256 = 12fd00c3dddc6474249a86899b14797cc891e815126cfb6989eb96e489d5e758
FINAL_IIC_TRAINING_CORE_SHA256 = f1df0a78746739c7552668e0ed0cac295b8d61d772088cb7893a85d6ae6bfe0a

PROVENANCE_GATE = True
C11_HANDOFF_GATE = True
PAPER_DIMENSION_GATE = True

NUMERICAL_TESTS_CHANGED = False
MODEL_STATE_CHANGED = False
LABELS_READ = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_06_A3C_R_FINAL_REALIGNMENT_PASS = True


## STEP 06-A3D — Contratto finale del training IIC

Il paper pubblica per l'IIC:

$$
B=128,
$$

$$
\eta=5\times10^{-4},
$$

e Adam con:

$$
\beta_1=0.9,
\qquad
\beta_2=0.999,
\qquad
\epsilon_{\mathrm{Adam}}=10^{-8}.
$$

Il paper riporta inoltre un tempo di training IIC di circa:

$$
0.3\ \mathrm{h}/100\ \mathrm{epochs}
$$

sull'hardware utilizzato dagli autori.

Non viene tuttavia pubblicato un criterio di arresto o di selezione del
checkpoint IIC.

### Budget temporale PROJECT

Il progetto congela quindi:

$$
E=100
$$

epoche complete per ciascuno dei:

$$
K=5
$$

classificatori.

Non viene utilizzato early stopping.

Il risultato scientifico ufficiale di ciascun classificatore sarà lo stato
dopo la centesima epoca, non un eventuale minimo diagnostico osservato prima.

### Adam

Ogni coppia main/over-clustering utilizza un proprio optimizer Adam con:

$$
\eta=5\times10^{-4},
$$

$$
(\beta_1,\beta_2)=(0.9,0.999),
$$

$$
\epsilon=10^{-8}.
$$

Non vengono utilizzati:

- weight decay;
- AMSGrad;
- scheduler del learning rate;
- gradient clipping.

Questi dettagli non specificati sono classificati PROJECT.

### Shuffle

L'ordine degli eventi viene permutato a ogni epoca.

La policy PROJECT utilizza lo stesso shuffle deterministico già adottato nel
progetto, con seed base:

$$
s_{\mathrm{shuffle}}=101112.
$$

I cinque classificatori utilizzano lo stesso ordine dei dati e le stesse
perturbazioni.

In questo modo la principale sorgente controllata di differenza fra i cinque
classificatori rimane l'inizializzazione della rete:

$$
123,\ 124,\ 125,\ 126,\ 127.
$$

### Ultimo minibatch

Con:

$$
D=8583
$$

e:

$$
B=128,
$$

si ottengono:

$$
68
$$

minibatch per epoca, con ultimo minibatch di:

$$
7
$$

eventi.

La policy PROJECT è:

`drop_last = False`.

Tutti gli 8.583 eventi partecipano quindi a ogni epoca.

BatchNorm rimane matematicamente definita anche nell'ultimo batch perché:

$$
7>1.
$$

### Numero di update

Per ciascun classificatore:

$$
68\times100=6800
$$

optimizer step massimi.

Per i cinque classificatori:

$$
5\times6800=34000
$$

step complessivi.

### Checkpoint e resume

Per ogni classificatore vengono mantenuti:

- `latest.pt`, aggiornato a ogni epoca completata;
- `final.pt`, creato dopo 100 epoche;
- `metrics.csv`;
- `run_config.json`.

Non viene definito un checkpoint `BEST` scientifico perché non esiste un
criterio PAPER di selezione IIC e non verranno utilizzate le label per
scegliere l'epoca.

Il resume riparte esclusivamente dal confine dell'ultima epoca completata.

### Relazione con le future ablation C11

Il protocollo costituisce il target comune anche per:

$$
\mathrm{mini75},
\quad
\mathrm{mini112},
\quad
\mathrm{mini224\_1000}.
$$

Per i mini verranno mantenuti, ove tecnicamente possibile:

- 100 epoche;
- Adam identico;
- batch 128;
- stessa policy main/over-clustering;
- stessi seed logici;
- stessa policy shuffle;
- stessa gestione del tail batch.

Qualunque deviazione tecnica necessaria nel backend C11 verrà dichiarata
esplicitamente prima della relativa run.

In [ ]:
# ============================================================
# NOTEBOOK 06 — FINAL IIC TRAINING POLICY
# STEP 06-A3D
#
# CONTRACT ONLY.
# NO real data.
# NO labels.
# NO optimizer step.
# NO training.
# NO Drive writes.
# ============================================================

import hashlib
import json
import math


# ============================================================
# CANONICAL PARENT CONTRACTS
# ============================================================

EXPECTED_LOSS_SHA256 = (
    "2bf9cba7d23de553cbad1d876b1f1b14"
    "3ede63154ac20bbb29ca092223c5e7ac"
)

EXPECTED_ARCHITECTURE_SHA256 = (
    "12fd00c3dddc6474249a86899b14797c"
    "c891e815126cfb6989eb96e489d5e758"
)

EXPECTED_TRAINING_CORE_SHA256 = (
    "f1df0a78746739c7552668e0ed0cac29"
    "5b8d61d772088cb7893a85d6ae6bfe0a"
)


assert (
    IIC_LOSS_CONTRACT_SHA256
    == EXPECTED_LOSS_SHA256
)

assert (
    IIC_ARCHITECTURE_CONTRACT_SHA256
    == EXPECTED_ARCHITECTURE_SHA256
)

assert (
    IIC_TRAINING_CORE_SHA256
    == EXPECTED_TRAINING_CORE_SHA256
)


# ============================================================
# PAPER PARAMETERS
# ============================================================

IIC_BATCH_SIZE = 128

IIC_LEARNING_RATE = 5.0e-4

IIC_ADAM_BETA1 = 0.9
IIC_ADAM_BETA2 = 0.999
IIC_ADAM_EPS = 1.0e-8


# ============================================================
# PROJECT TRAINING POLICY
# ============================================================

IIC_EPOCHS = 100

IIC_EARLY_STOPPING = False

IIC_LR_SCHEDULER = False

IIC_WEIGHT_DECAY = 0.0

IIC_AMSGRAD = False

IIC_GRADIENT_CLIPPING = False


IIC_SHUFFLE = True
IIC_SHUFFLE_SEED = 101112

IIC_AUGMENTATION_SEED = 789

IIC_DROP_LAST = False


IIC_CLASSIFIER_COUNT = 5

IIC_CLASSIFIER_SEEDS = [
    123,
    124,
    125,
    126,
    127,
]


# ============================================================
# BATCH GEOMETRY
# ============================================================

IIC_EVENT_COUNT = 8583


IIC_BATCHES_PER_EPOCH = math.ceil(
    IIC_EVENT_COUNT
    / IIC_BATCH_SIZE
)


IIC_TAIL_BATCH_SIZE = (
    IIC_EVENT_COUNT
    % IIC_BATCH_SIZE
)


IIC_STEPS_PER_CLASSIFIER = (
    IIC_BATCHES_PER_EPOCH
    * IIC_EPOCHS
)


IIC_TOTAL_K5_STEPS = (
    IIC_STEPS_PER_CLASSIFIER
    * IIC_CLASSIFIER_COUNT
)


assert IIC_BATCHES_PER_EPOCH == 68
assert IIC_TAIL_BATCH_SIZE == 7
assert IIC_STEPS_PER_CLASSIFIER == 6800
assert IIC_TOTAL_K5_STEPS == 34000


# ============================================================
# CHECKPOINT POLICY
# ============================================================

IIC_CHECKPOINT_POLICY = {

    "latest":
        "after_each_completed_epoch",

    "final":
        "after_epoch_99",

    "best":
        False,

    "resume":
        "next_epoch_boundary",

    "metrics":
        "append_after_completed_epoch",

    "scientific_checkpoint":
        "final_epoch_100",
}


# ============================================================
# FINAL TRAINING CONTRACT
# ============================================================

IIC_FINAL_TRAINING_CONTRACT = {

    "contract_version":
        1,

    "paper": {

        "batch_size":
            IIC_BATCH_SIZE,

        "learning_rate":
            IIC_LEARNING_RATE,

        "adam_beta1":
            IIC_ADAM_BETA1,

        "adam_beta2":
            IIC_ADAM_BETA2,

        "adam_epsilon":
            IIC_ADAM_EPS,

        "reported_runtime_basis":
            "approximately_0.3_h_per_100_epochs",

        "explicit_final_epoch_count_published":
            False,
    },


    "project_epochs": {

        "epochs":
            IIC_EPOCHS,

        "epoch_indices":
            [0, 99],

        "early_stopping":
            IIC_EARLY_STOPPING,

        "scientific_selection":
            "final_after_100_epochs",

        "classification":
            "PROJECT_high_fidelity_choice",
    },


    "optimizer": {

        "type":
            "Adam",

        "one_optimizer_per_classifier_pair":
            True,

        "learning_rate":
            IIC_LEARNING_RATE,

        "betas": [
            IIC_ADAM_BETA1,
            IIC_ADAM_BETA2,
        ],

        "epsilon":
            IIC_ADAM_EPS,

        "weight_decay":
            IIC_WEIGHT_DECAY,

        "amsgrad":
            IIC_AMSGRAD,

        "lr_scheduler":
            IIC_LR_SCHEDULER,

        "gradient_clipping":
            IIC_GRADIENT_CLIPPING,
    },


    "shuffle": {

        "enabled":
            IIC_SHUFFLE,

        "base_seed":
            IIC_SHUFFLE_SEED,

        "same_order_across_K":
            True,

        "policy":
            "PROJECT_deterministic_epoch_shuffle",
    },


    "augmentation": {

        "base_seed":
            IIC_AUGMENTATION_SEED,

        "same_views_across_K":
            True,

        "key":
            [
                "augmentation_seed",
                "epoch",
                "manifest_index",
            ],
    },


    "batching": {

        "event_count":
            IIC_EVENT_COUNT,

        "batch_size":
            IIC_BATCH_SIZE,

        "drop_last":
            IIC_DROP_LAST,

        "batches_per_epoch":
            IIC_BATCHES_PER_EPOCH,

        "tail_batch_size":
            IIC_TAIL_BATCH_SIZE,

        "all_events_each_epoch":
            True,
    },


    "classifier_ensemble": {

        "K":
            IIC_CLASSIFIER_COUNT,

        "initialization_seeds":
            IIC_CLASSIFIER_SEEDS,

        "same_shuffle_across_K":
            True,

        "same_augmentation_across_K":
            True,

        "training_mode":
            "independent_classifier_pairs",
    },


    "update_budget": {

        "steps_per_classifier":
            IIC_STEPS_PER_CLASSIFIER,

        "steps_all_K5":
            IIC_TOTAL_K5_STEPS,
    },


    "checkpoint_policy":
        IIC_CHECKPOINT_POLICY,


    "future_C11_target": {

        "epochs":
            100,

        "batch_size_target":
            128,

        "same_Adam_parameters":
            True,

        "same_loss_definition":
            True,

        "same_main_overcluster_policy":
            True,

        "same_logical_seed_values":
            True,

        "same_tail_policy_target":
            True,

        "deviations_require_pre_run_documentation":
            True,
    },


    "labels_used":
        False,


    "parent_loss_sha256":
        IIC_LOSS_CONTRACT_SHA256,

    "parent_architecture_sha256":
        IIC_ARCHITECTURE_CONTRACT_SHA256,

    "parent_training_core_sha256":
        IIC_TRAINING_CORE_SHA256,
}


# ============================================================
# HASH
# ============================================================

IIC_FINAL_TRAINING_JSON = json.dumps(
    IIC_FINAL_TRAINING_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_FINAL_TRAINING_SHA256 = hashlib.sha256(
    IIC_FINAL_TRAINING_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# GATES
# ============================================================

paper_optimizer_gate = (
    IIC_BATCH_SIZE == 128

    and IIC_LEARNING_RATE == 5.0e-4

    and IIC_ADAM_BETA1 == 0.9

    and IIC_ADAM_BETA2 == 0.999

    and IIC_ADAM_EPS == 1.0e-8
)


epoch_policy_gate = (
    IIC_EPOCHS == 100

    and IIC_EARLY_STOPPING is False

    and IIC_CHECKPOINT_POLICY[
        "scientific_checkpoint"
    ] == "final_epoch_100"
)


batch_policy_gate = (
    IIC_BATCHES_PER_EPOCH == 68

    and IIC_TAIL_BATCH_SIZE == 7

    and IIC_DROP_LAST is False
)


ensemble_control_gate = (
    IIC_CLASSIFIER_SEEDS
    == [
        123,
        124,
        125,
        126,
        127,
    ]

    and IIC_SHUFFLE_SEED == 101112

    and IIC_AUGMENTATION_SEED == 789
)


c11_target_gate = (
    IIC_FINAL_TRAINING_CONTRACT[
        "future_C11_target"
    ][
        "same_Adam_parameters"
    ]
    is True

    and IIC_FINAL_TRAINING_CONTRACT[
        "future_C11_target"
    ][
        "deviations_require_pre_run_documentation"
    ]
    is True
)


a3d_gate = all([
    paper_optimizer_gate,
    epoch_policy_gate,
    batch_policy_gate,
    ensemble_control_gate,
    c11_target_gate,
])


assert a3d_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 FINAL IIC TRAINING POLICY ==="
)

print()

print(
    "EPOCHS_PROJECT =",
    IIC_EPOCHS,
)

print(
    "EARLY_STOPPING =",
    IIC_EARLY_STOPPING,
)

print(
    "SCIENTIFIC_CHECKPOINT = final_epoch_100"
)

print()

print(
    "BATCH_SIZE_PAPER =",
    IIC_BATCH_SIZE,
)

print(
    "BATCHES_PER_EPOCH =",
    IIC_BATCHES_PER_EPOCH,
)

print(
    "TAIL_BATCH_SIZE =",
    IIC_TAIL_BATCH_SIZE,
)

print(
    "DROP_LAST =",
    IIC_DROP_LAST,
)

print()

print(
    "ADAM_LR =",
    IIC_LEARNING_RATE,
)

print(
    "ADAM_BETAS =",
    (
        IIC_ADAM_BETA1,
        IIC_ADAM_BETA2,
    ),
)

print(
    "ADAM_EPS =",
    IIC_ADAM_EPS,
)

print()

print(
    "SHUFFLE_SEED_PROJECT =",
    IIC_SHUFFLE_SEED,
)

print(
    "AUGMENTATION_SEED_PROJECT =",
    IIC_AUGMENTATION_SEED,
)

print(
    "CLASSIFIER_SEEDS_PROJECT =",
    IIC_CLASSIFIER_SEEDS,
)

print()

print(
    "STEPS_PER_CLASSIFIER =",
    IIC_STEPS_PER_CLASSIFIER,
)

print(
    "TOTAL_K5_STEPS =",
    IIC_TOTAL_K5_STEPS,
)

print()

print(
    "PAPER_OPTIMIZER_GATE =",
    paper_optimizer_gate,
)

print(
    "EPOCH_POLICY_GATE =",
    epoch_policy_gate,
)

print(
    "BATCH_POLICY_GATE =",
    batch_policy_gate,
)

print(
    "ENSEMBLE_CONTROL_GATE =",
    ensemble_control_gate,
)

print(
    "C11_TARGET_GATE =",
    c11_target_gate,
)

print()

print(
    "IIC_FINAL_TRAINING_SHA256 =",
    IIC_FINAL_TRAINING_SHA256,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "REAL_DATA_READ_IN_A3D =",
    False,
)

print(
    "OPTIMIZER_CREATED_IN_A3D =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A3D_FINAL_TRAINING_POLICY_PASS =",
    a3d_gate,
)


=== NOTEBOOK 06 FINAL IIC TRAINING POLICY ===

EPOCHS_PROJECT = 100
EARLY_STOPPING = False
SCIENTIFIC_CHECKPOINT = final_epoch_100

BATCH_SIZE_PAPER = 128
BATCHES_PER_EPOCH = 68
TAIL_BATCH_SIZE = 7
DROP_LAST = False

ADAM_LR = 0.0005
ADAM_BETAS = (0.9, 0.999)
ADAM_EPS = 1e-08

SHUFFLE_SEED_PROJECT = 101112
AUGMENTATION_SEED_PROJECT = 789
CLASSIFIER_SEEDS_PROJECT = [123, 124, 125, 126, 127]

STEPS_PER_CLASSIFIER = 6800
TOTAL_K5_STEPS = 34000

PAPER_OPTIMIZER_GATE = True
EPOCH_POLICY_GATE = True
BATCH_POLICY_GATE = True
ENSEMBLE_CONTROL_GATE = True
C11_TARGET_GATE = True

IIC_FINAL_TRAINING_SHA256 = 4f27a9bbf4290dff54767655d2025928c2eac751607d217925b0d5bba2b2f537

LABELS_READ = False
REAL_DATA_READ_IN_A3D = False
OPTIMIZER_CREATED_IN_A3D = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

NOTEBOOK_06_A3D_FINAL_TRAINING_POLICY_PASS = True


## STEP 06-A4A — Runtime e sorgenti reali per l'IIC

Il protocollo IIC è stato completamente congelato prima dell'accesso ai dati
reali.

Per il training IIC servono due rappresentazioni dello stesso evento:

$$
z_i=\mu_\phi(x_i),
$$

$$
z'_i=\mu_\phi(x'_i).
$$

La feature centrale $z_i$ è già disponibile nell'artefatto canonico
epoch-100 pubblicato nel Notebook 04.

Poiché l'encoder è congelato, riutilizzare tale feature è matematicamente
equivalente a ricalcolare la stessa vista centrale ad ogni epoca.

La vista perturbata $x'_i$ deve invece essere generata dinamicamente dalle
quattro PNG `272 x 224`, perché il crop dipende da:

$$
(\mathrm{seed},\mathrm{epoch},\mathrm{manifest\_index}).
$$

Per questo viene ripristinata nella root runtime locale soltanto la sorgente:

`out272_gray`.

Non è necessario ripristinare l'intero dataset centrale `.bin`.

Dal manifest eventi vengono conservati esclusivamente:

- `event_index`;
- `event_id`;
- path delle quattro viste temporali.

Il campo Gravity Spy `label` non viene conservato nella struttura dati IIC.

Questo step non carica il VAE e non esegue training.

In [3]:
# ============================================================
# NOTEBOOK 06 — REAL IIC SOURCE PREFLIGHT
# STEP 06-A4A
#
# Restores only out272_gray if necessary.
# Builds canonical full-mu matrix in manifest order.
#
# NO VAE load.
# NO IIC model creation.
# NO optimizer.
# NO training.
# NO Drive writes.
# ============================================================

from pathlib import Path

import csv
import hashlib
import os
import platform
import shutil
import subprocess

import numpy as np
import torch


# ============================================================
# PARENT TRAINING CONTRACT
# ============================================================

EXPECTED_FINAL_TRAINING_SHA256 = (
    "4f27a9bbf4290dff54767655d2025928"
    "c2eac751607d217925b0d5bba2b2f537"
)

assert (
    IIC_FINAL_TRAINING_SHA256
    == EXPECTED_FINAL_TRAINING_SHA256
)


# ============================================================
# RUNTIME
# ============================================================

runtime_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

gpu_name = (
    torch.cuda.get_device_name(0)
    if runtime_device.type == "cuda"
    else "NONE"
)


# ============================================================
# LOCAL TEMPORARY PATHS
# ============================================================

LOCAL_TRANSFER_DIR = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_transfer_v1"
)

LOCAL_DATA_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

LOCAL_OUT272_DIR = (
    LOCAL_DATA_ROOT
    / "out272_gray"
)

LOCAL_OUT272_TAR = (
    LOCAL_TRANSFER_DIR
    / "sakai224_full_out272_gray.tar"
)


EXPECTED_OUT272_SHA256 = (
    "70c766c3652f586c8da4c8cadf134a8c"
    "e83cfd07c8cd10545367f380017ddab2"
)

EXPECTED_OUT272_FILES = 34_332
EXPECTED_OUT272_BYTES = 1_144_867_540


# ============================================================
# STREAMING SHA
# ============================================================

def sha256_stream(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


# ============================================================
# RESTORE LOCAL TAR IF NECESSARY
# ============================================================

LOCAL_TRANSFER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if not LOCAL_OUT272_TAR.exists():

    print(
        "Copying out272 TAR from Drive..."
    )

    shutil.copy2(
        OUT272_TAR_PATH,
        LOCAL_OUT272_TAR,
    )


local_tar_hash_gate = (
    sha256_stream(
        LOCAL_OUT272_TAR
    )
    == EXPECTED_OUT272_SHA256
)

assert local_tar_hash_gate


# ============================================================
# AUDIT OR EXTRACT OUT272
# ============================================================

def audit_out272_directory():

    if not LOCAL_OUT272_DIR.is_dir():
        return (
            False,
            0,
            0,
        )

    files = [
        path
        for path
        in LOCAL_OUT272_DIR.rglob("*")
        if path.is_file()
    ]

    total_bytes = sum(
        path.stat().st_size
        for path
        in files
    )

    valid = (
        len(files)
        == EXPECTED_OUT272_FILES

        and total_bytes
        == EXPECTED_OUT272_BYTES
    )

    return (
        valid,
        len(files),
        total_bytes,
    )


(
    out272_ready,
    out272_file_count,
    out272_total_bytes,
) = audit_out272_directory()


if not out272_ready:

    if LOCAL_OUT272_DIR.exists():

        raise RuntimeError(
            "STOP: existing out272 directory "
            "does not match the frozen payload."
        )


    LOCAL_DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "Extracting verified out272 TAR..."
    )


    subprocess.run(
        [
            "tar",
            "-xf",
            str(
                LOCAL_OUT272_TAR
            ),
            "-C",
            str(
                LOCAL_DATA_ROOT
            ),
        ],
        check=True,
    )


    (
        out272_ready,
        out272_file_count,
        out272_total_bytes,
    ) = audit_out272_directory()


assert out272_ready


# ============================================================
# LABEL-EXCLUDED EVENT PATH MAP
#
# manifest_events_272.csv contains a label column.
# It is deliberately NOT stored in iic_event_rows.
# ============================================================

REQUIRED_EVENT_COLUMNS = [
    "event_index",
    "event_id",
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


iic_event_rows = []


with EVENT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle:

    reader = csv.DictReader(
        handle
    )


    assert reader.fieldnames is not None

    assert all(
        column in reader.fieldnames
        for column
        in REQUIRED_EVENT_COLUMNS
    )


    for manifest_index, source_row in enumerate(
        reader
    ):

        iic_event_rows.append({

            "manifest_index":
                manifest_index,

            "event_index":
                int(
                    source_row[
                        "event_index"
                    ]
                ),

            "event_id":
                source_row[
                    "event_id"
                ],

            "img_0_5_272":
                source_row[
                    "img_0_5_272"
                ],

            "img_1_0_272":
                source_row[
                    "img_1_0_272"
                ],

            "img_2_0_272":
                source_row[
                    "img_2_0_272"
                ],

            "img_4_0_272":
                source_row[
                    "img_4_0_272"
                ],
        })


event_map_gate = (
    len(
        iic_event_rows
    )
    == 8583

    and all(
        row[
            "manifest_index"
        ]
        == index

        for index, row
        in enumerate(
            iic_event_rows
        )
    )

    and all(
        row[
            "event_index"
        ]
        == index + 1

        for index, row
        in enumerate(
            iic_event_rows
        )
    )

    and len({
        row[
            "event_id"
        ]
        for row
        in iic_event_rows
    })
    == 8583

    and all(
        "label" not in row
        for row
        in iic_event_rows
    )
)

assert event_map_gate


# ============================================================
# VERIFY ALL 34,332 PATH REFERENCES
# ============================================================

VIEW_FIELDS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


all_png_paths_exist = True


for row in iic_event_rows:

    for field in VIEW_FIELDS:

        path = (
            LOCAL_DATA_ROOT
            / row[field]
        )

        if not path.is_file():

            all_png_paths_exist = False
            break


    if not all_png_paths_exist:
        break


assert all_png_paths_exist


# ============================================================
# LOAD CANONICAL CENTRAL MU
#
# Source matrices are in frozen split-row order.
# Reconstruct one matrix indexed directly by manifest_index.
# ============================================================

train_mu = np.memmap(
    TRAIN_MU_PATH,
    dtype="<f4",
    mode="r",
    shape=(
        6866,
        512,
    ),
)


validation_mu = np.memmap(
    VALIDATION_MU_PATH,
    dtype="<f4",
    mode="r",
    shape=(
        1717,
        512,
    ),
)


full_central_mu = np.empty(
    (
        8583,
        512,
    ),
    dtype=np.float32,
)


filled = np.zeros(
    8583,
    dtype=bool,
)


for split_pos, row in enumerate(
    train_rows
):

    manifest_index = int(
        row[
            "manifest_index"
        ]
    )

    full_central_mu[
        manifest_index
    ] = train_mu[
        split_pos
    ]

    filled[
        manifest_index
    ] = True


for split_pos, row in enumerate(
    validation_rows
):

    manifest_index = int(
        row[
            "manifest_index"
        ]
    )

    assert not filled[
        manifest_index
    ]

    full_central_mu[
        manifest_index
    ] = validation_mu[
        split_pos
    ]

    filled[
        manifest_index
    ] = True


central_mu_gate = (
    filled.all()

    and full_central_mu.shape
        == (
            8583,
            512,
        )

    and full_central_mu.dtype
        == np.float32

    and np.isfinite(
        full_central_mu
    ).all()
)


assert central_mu_gate


FULL_CENTRAL_MU_SHA256 = hashlib.sha256(
    full_central_mu.astype(
        "<f4",
        copy=False,
    ).tobytes(
        order="C"
    )
).hexdigest()


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 REAL IIC SOURCE PREFLIGHT ==="
)

print()

print(
    "PYTHON_VERSION =",
    platform.python_version(),
)

print(
    "PYTORCH_VERSION =",
    torch.__version__,
)

print(
    "RUNTIME_DEVICE =",
    runtime_device,
)

print(
    "GPU_NAME =",
    gpu_name,
)

print()

print(
    "OUT272_DIRECTORY =",
    LOCAL_OUT272_DIR,
)

print(
    "OUT272_FILE_COUNT =",
    out272_file_count,
)

print(
    "OUT272_TOTAL_BYTES =",
    out272_total_bytes,
)

print(
    "OUT272_TAR_HASH_GATE =",
    local_tar_hash_gate,
)

print(
    "OUT272_PAYLOAD_GATE =",
    out272_ready,
)

print(
    "ALL_PNG_PATHS_EXIST =",
    all_png_paths_exist,
)

print()

print(
    "IIC_EVENT_ROWS =",
    len(
        iic_event_rows
    ),
)

print(
    "LABEL_FIELD_STORED =",
    False,
)

print(
    "EVENT_MAP_GATE =",
    event_map_gate,
)

print()

print(
    "FULL_CENTRAL_MU_SHAPE =",
    full_central_mu.shape,
)

print(
    "FULL_CENTRAL_MU_SHA256 =",
    FULL_CENTRAL_MU_SHA256,
)

print(
    "CENTRAL_MU_GATE =",
    central_mu_gate,
)

print()

print(
    "VAE_MODEL_LOADED =",
    False,
)

print(
    "IIC_MODEL_CREATED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A4A_REAL_SOURCE_PREFLIGHT_PASS =",
    (
        local_tar_hash_gate
        and out272_ready
        and all_png_paths_exist
        and event_map_gate
        and central_mu_gate
    ),
)

Copying out272 TAR from Drive...
Extracting verified out272 TAR...

=== NOTEBOOK 06 REAL IIC SOURCE PREFLIGHT ===

PYTHON_VERSION = 3.13.15
PYTORCH_VERSION = 2.11.0+cu128
RUNTIME_DEVICE = cuda
GPU_NAME = NVIDIA A100-SXM4-40GB

OUT272_DIRECTORY = /content/sakai224_full_data/out272_gray
OUT272_FILE_COUNT = 34332
OUT272_TOTAL_BYTES = 1144867540
OUT272_TAR_HASH_GATE = True
OUT272_PAYLOAD_GATE = True
ALL_PNG_PATHS_EXIST = True

IIC_EVENT_ROWS = 8583
LABEL_FIELD_STORED = False
EVENT_MAP_GATE = True

FULL_CENTRAL_MU_SHAPE = (8583, 512)
FULL_CENTRAL_MU_SHA256 = fd55be588ef05df8afdf7ee048aa535e12eda37bd13ef8d59e5968fcfc9467f8
CENTRAL_MU_GATE = True

VAE_MODEL_LOADED = False
IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A4A_REAL_SOURCE_PREFLIGHT_PASS = True


## STEP 06-A4B — Restore dell'encoder FINAL e verifica contro gli embedding canonici

Viene ricostruito esclusivamente l'encoder VAE già validato nel Notebook 02
e caricato lo stato proveniente dal checkpoint ufficiale:

`final.pt`

corrispondente a 100 epoche completate.

Prima di estrarre i soli pesi encoder viene verificata l'identità SHA-256
dell'intero `model_state_dict` contenuto nel checkpoint.

L'encoder viene poi mantenuto:

- in modalità `EVAL`;
- con parametri congelati;
- senza gradienti;
- senza aggiornamenti BatchNorm.

Questa policy di congelamento è PROJECT.

### Known-answer sui dati reali

Per un insieme fisso e label-blind di eventi viene ricostruita la vista
centrale direttamente dalle PNG:

$$
\mathrm{crop\_left}=24.
$$

Si calcola quindi:

$$
\mu_\phi(x)
$$

con l'encoder appena ripristinato e si confronta il risultato con gli
embedding canonici già pubblicati dal Notebook 04.

Il test verifica quindi simultaneamente:

- checkpoint corretto;
- architettura encoder corretta;
- ordine dei quattro canali;
- crop centrale;
- normalizzazione;
- forward EVAL;
- identità delle feature utilizzate dall'IIC.

Non vengono ancora generate viste perturbate e non viene creato alcun
classificatore IIC.

In [4]:
# ============================================================
# NOTEBOOK 06 — RESTORE FINAL ENCODER + REAL MU KNOWN ANSWER
# STEP 06-A4B
#
# Uses the exact encoder already validated in Notebook 02/04.
#
# NO labels.
# NO perturbed view yet.
# NO IIC model.
# NO optimizer.
# NO training.
# NO Drive writes.
# ============================================================

from dataclasses import dataclass
from PIL import Image

import gc
import hashlib
import struct

import numpy as np
import torch
import torch.nn as nn


# ============================================================
# FROZEN IDENTITIES
# ============================================================

EXPECTED_FINAL_CHECKPOINT_SHA256 = (
    "2bb6fd2ce2f73f8f9e479ed6969a52c1"
    "cdd41c1ae6e2cfc33da687ebba798f71"
)

EXPECTED_FINAL_MODEL_STATE_SHA256 = (
    "3ac5a1cecdee2aa204419290b5cec55f"
    "3e6a4a846beebaa3f6b4e495f6fe6096"
)

EXPECTED_FULL_CENTRAL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)


assert (
    sha256_stream(
        FINAL_CHECKPOINT_PATH
    )
    == EXPECTED_FINAL_CHECKPOINT_SHA256
)

assert (
    FULL_CENTRAL_MU_SHA256
    == EXPECTED_FULL_CENTRAL_MU_SHA256
)


# ============================================================
# DETERMINISTIC CUDA EXECUTION POLICY
# ============================================================

assert runtime_device.type == "cuda"

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ============================================================
# CANONICAL TREE SHA-256
#
# Same rule used by the official checkpoint audit.
# ============================================================

def canonical_tree_sha256(obj):

    digest = hashlib.sha256()

    def update(value):

        if torch.is_tensor(value):

            tensor = (
                value
                .detach()
                .cpu()
                .contiguous()
            )

            digest.update(b"TENSOR\0")

            digest.update(
                str(
                    tensor.dtype
                ).encode("utf-8")
            )

            digest.update(b"\0")

            digest.update(
                repr(
                    tuple(
                        tensor.shape
                    )
                ).encode("utf-8")
            )

            digest.update(b"\0")

            digest.update(
                tensor.numpy().tobytes(
                    order="C"
                )
            )

            return


        if isinstance(value, dict):

            digest.update(b"DICT\0")

            for key in sorted(
                value.keys(),
                key=lambda item: (
                    type(item).__name__,
                    repr(item),
                ),
            ):

                update(key)
                update(value[key])

            return


        if isinstance(
            value,
            (list, tuple),
        ):

            digest.update(
                b"LIST\0"
                if isinstance(
                    value,
                    list,
                )
                else b"TUPLE\0"
            )

            for item in value:
                update(item)

            return


        if value is None:

            digest.update(b"NONE\0")
            return


        if isinstance(value, bool):

            digest.update(
                b"BOOL1\0"
                if value
                else b"BOOL0\0"
            )

            return


        if isinstance(value, int):

            digest.update(b"INT\0")

            digest.update(
                str(value).encode(
                    "ascii"
                )
            )

            digest.update(b"\0")

            return


        if isinstance(value, float):

            digest.update(b"FLOAT64\0")

            digest.update(
                struct.pack(
                    "<d",
                    value,
                )
            )

            return


        if isinstance(value, str):

            encoded = value.encode(
                "utf-8"
            )

            digest.update(b"STR\0")

            digest.update(
                len(encoded).to_bytes(
                    8,
                    "little",
                )
            )

            digest.update(encoded)

            return


        raise TypeError(
            f"Unsupported hash type: {type(value)}"
        )


    update(obj)

    return digest.hexdigest()


# ============================================================
# EXACT FROZEN FULL SPEC
# ============================================================

@dataclass(frozen=True)
class Sakai224FullSpec:

    input_channels: int = 4
    input_side: int = 224

    channels: tuple = (
        64,
        128,
        256,
        512,
    )

    latent_dim: int = 512
    bottleneck_side: int = 7

    bn_eps: float = 1.0e-5
    bn_momentum: float = 0.1


SPEC_IIC_ENCODER = Sakai224FullSpec()


# ============================================================
# EXACT ENCODER FROM NOTEBOOK 02 / NOTEBOOK 04
# ============================================================

class Sakai224FullEncoder(nn.Module):

    def __init__(
        self,
        spec,
    ):

        super().__init__()

        eps = spec.bn_eps
        momentum = spec.bn_momentum

        self.latent_dim = (
            spec.latent_dim
        )


        self.conv1 = nn.Conv2d(
            4,
            64,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            64,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )

        self.relu1 = nn.ReLU(
            inplace=False
        )


        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
            padding=0,
        )


        self.conv2 = nn.Conv2d(
            64,
            128,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            128,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )

        self.relu2 = nn.ReLU(
            inplace=False
        )


        self.conv3 = nn.Conv2d(
            128,
            256,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            256,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )

        self.relu3 = nn.ReLU(
            inplace=False
        )


        self.conv4 = nn.Conv2d(
            256,
            512,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )

        self.bn4 = nn.BatchNorm2d(
            512,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )

        self.relu4 = nn.ReLU(
            inplace=False
        )


        self.fc = nn.Linear(
            512,
            2 * spec.latent_dim,
            bias=True,
        )


    def forward(
        self,
        x,
    ):

        x = self.relu1(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)


        x = self.relu2(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.relu3(
            self.bn3(
                self.conv3(x)
            )
        )

        x = self.relu4(
            self.bn4(
                self.conv4(x)
            )
        )


        x = x.mean(
            dim=(2, 3)
        )


        packed = self.fc(x)


        mu = packed[
            :,
            :self.latent_dim
        ]

        logvar = packed[
            :,
            self.latent_dim:
        ]


        return (
            mu,
            logvar,
        )


# ============================================================
# CHECKPOINT PAYLOAD — CPU
# ============================================================

print(
    "Loading FINAL checkpoint on CPU..."
)


final_payload = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)


checkpoint_identity_gate = (

    final_payload[
        "kind"
    ] == "FINAL"

    and final_payload[
        "run_id"
    ] == "sakai224_full_v1"

    and final_payload[
        "last_completed_epoch"
    ] == 99

    and final_payload[
        "optimizer_step"
    ] == 5400

    and final_payload[
        "model_state_sha256"
    ]
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)


assert checkpoint_identity_gate


calculated_full_state_sha256 = (
    canonical_tree_sha256(
        final_payload[
            "model_state_dict"
        ]
    )
)


full_state_hash_gate = (
    calculated_full_state_sha256
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)


assert full_state_hash_gate


# ============================================================
# EXTRACT ONLY THE ENCODER STATE
# ============================================================

encoder_state_dict = {

    key[
        len("encoder.") :
    ]:
        value

    for key, value
    in final_payload[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}


encoder_state_structure_gate = (
    len(
        encoder_state_dict
    )
    == 26
)


assert encoder_state_structure_gate


# ============================================================
# BUILD + STRICT LOAD ENCODER
# ============================================================

official_iic_encoder = (
    Sakai224FullEncoder(
        SPEC_IIC_ENCODER
    )
)


encoder_parameter_count = sum(
    parameter.numel()
    for parameter
    in official_iic_encoder.parameters()
)


encoder_parameter_tensor_count = len(
    list(
        official_iic_encoder.parameters()
    )
)


encoder_structure_gate = (

    encoder_parameter_count
    == 2_077_824

    and encoder_parameter_tensor_count
    == 14

    and len(
        official_iic_encoder.state_dict()
    )
    == 26
)


assert encoder_structure_gate


load_result = (
    official_iic_encoder.load_state_dict(
        encoder_state_dict,
        strict=True,
    )
)


strict_load_gate = (
    load_result.missing_keys == []
    and load_result.unexpected_keys == []
)


assert strict_load_gate


ENCODER_FINAL_STATE_SHA256 = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


# Full payload no longer required.
del encoder_state_dict
del final_payload

gc.collect()


# ============================================================
# FREEZE + MOVE TO GPU
# ============================================================

official_iic_encoder.requires_grad_(
    False
)

official_iic_encoder.eval()


official_iic_encoder = (
    official_iic_encoder.to(
        device=runtime_device,
        dtype=torch.float32,
    )
)


encoder_freeze_gate = (

    official_iic_encoder.training
    is False

    and all(
        parameter.requires_grad
        is False

        for parameter
        in official_iic_encoder.parameters()
    )
)


assert encoder_freeze_gate


state_before_forward = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


assert (
    state_before_forward
    == ENCODER_FINAL_STATE_SHA256
)


# ============================================================
# EXACT CENTRAL PNG LOADER
# ============================================================

def load_png_event_crop_224(
    manifest_index,
    crop_left,
):

    row = iic_event_rows[
        manifest_index
    ]

    channels = []


    for field in VIEW_FIELDS:

        path = (
            LOCAL_DATA_ROOT
            / row[field]
        )


        with Image.open(
            path
        ) as image:

            if image.mode != "L":

                raise RuntimeError(
                    f"Non-grayscale PNG: {path}"
                )


            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()


        if array.shape != (
            224,
            272,
        ):

            raise RuntimeError(
                f"Unexpected PNG shape: {array.shape}"
            )


        crop = array[
            :,
            crop_left:
            crop_left + 224
        ]


        if crop.shape != (
            224,
            224,
        ):

            raise RuntimeError(
                "Invalid temporal crop"
            )


        channels.append(
            crop
        )


    tensor_u8 = np.stack(
        channels,
        axis=0,
    )


    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(
            255.0
        )
    )


    return tensor


# ============================================================
# FIXED LABEL-BLIND KNOWN-ANSWER SET
# ============================================================

CENTRAL_AUDIT_MANIFEST_INDICES = [
    0,
    2145,
    4291,
    6864,
    7663,
    8582,
]


central_batch_cpu = torch.stack(
    [
        load_png_event_crop_224(
            manifest_index,
            crop_left=24,
        )

        for manifest_index
        in CENTRAL_AUDIT_MANIFEST_INDICES
    ],
    dim=0,
)


central_input_gate = (

    tuple(
        central_batch_cpu.shape
    )
    == (
        6,
        4,
        224,
        224,
    )

    and central_batch_cpu.dtype
        == torch.float32

    and torch.isfinite(
        central_batch_cpu
    ).all().item()

    and float(
        central_batch_cpu.min()
    ) >= 0.0

    and float(
        central_batch_cpu.max()
    ) <= 1.0
)


assert central_input_gate


# ============================================================
# REAL ENCODER FORWARD
# ============================================================

central_batch_gpu = (
    central_batch_cpu.to(
        device=runtime_device,
        dtype=torch.float32,
    )
)


with torch.inference_mode():

    (
        observed_mu_gpu,
        observed_logvar_gpu,
    ) = official_iic_encoder(
        central_batch_gpu
    )


torch.cuda.synchronize()


observed_mu = (
    observed_mu_gpu
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.float32,
        copy=False,
    )
)


observed_logvar = (
    observed_logvar_gpu
    .detach()
    .cpu()
    .numpy()
)


expected_mu = np.asarray(
    full_central_mu[
        CENTRAL_AUDIT_MANIFEST_INDICES
    ],
    dtype=np.float32,
)


# ============================================================
# KNOWN-ANSWER COMPARISON
# ============================================================

mu_difference = (
    observed_mu
    - expected_mu
)


mu_max_abs_difference = float(
    np.max(
        np.abs(
            mu_difference
        )
    )
)


mu_mean_abs_difference = float(
    np.mean(
        np.abs(
            mu_difference
        )
    )
)


mu_bitwise_gate = np.array_equal(
    observed_mu,
    expected_mu,
)


mu_numeric_gate = np.allclose(
    observed_mu,
    expected_mu,
    rtol=1.0e-6,
    atol=1.0e-5,
)


logvar_gate = (
    observed_logvar.shape
    == (
        6,
        512,
    )

    and np.isfinite(
        observed_logvar
    ).all()
)


assert mu_numeric_gate
assert logvar_gate


# ============================================================
# EVAL MUST NOT MODIFY ENCODER STATE
# ============================================================

state_after_forward = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_state_preserved_gate = (
    state_before_forward
    == state_after_forward
    == ENCODER_FINAL_STATE_SHA256
)


assert encoder_state_preserved_gate


# ============================================================
# FINAL GATE
# ============================================================

a4b_gate = all([
    checkpoint_identity_gate,
    full_state_hash_gate,
    encoder_state_structure_gate,
    encoder_structure_gate,
    strict_load_gate,
    encoder_freeze_gate,
    central_input_gate,
    bool(
        mu_numeric_gate
    ),
    logvar_gate,
    encoder_state_preserved_gate,
])


assert a4b_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 FINAL ENCODER REAL-DATA AUDIT ==="
)

print()

print(
    "CHECKPOINT_KIND = FINAL"
)

print(
    "CHECKPOINT_LAST_COMPLETED_EPOCH = 99"
)

print(
    "CHECKPOINT_OPTIMIZER_STEP = 5400"
)

print(
    "FULL_MODEL_STATE_SHA256 =",
    calculated_full_state_sha256,
)

print()

print(
    "ENCODER_PARAMETERS =",
    encoder_parameter_count,
)

print(
    "ENCODER_PARAMETER_TENSORS =",
    encoder_parameter_tensor_count,
)

print(
    "ENCODER_STATE_TENSORS =",
    len(
        official_iic_encoder.state_dict()
    ),
)

print(
    "ENCODER_FINAL_STATE_SHA256 =",
    ENCODER_FINAL_STATE_SHA256,
)

print()

print(
    "CENTRAL_AUDIT_MANIFEST_INDICES =",
    CENTRAL_AUDIT_MANIFEST_INDICES,
)

print(
    "OBSERVED_MU_SHAPE =",
    observed_mu.shape,
)

print()

print(
    "MU_BITWISE_EQUAL =",
    mu_bitwise_gate,
)

print(
    "MU_MAX_ABS_DIFFERENCE =",
    mu_max_abs_difference,
)

print(
    "MU_MEAN_ABS_DIFFERENCE =",
    mu_mean_abs_difference,
)

print(
    "MU_NUMERIC_GATE =",
    bool(
        mu_numeric_gate
    ),
)

print()

print(
    "CHECKPOINT_IDENTITY_GATE =",
    checkpoint_identity_gate,
)

print(
    "FULL_STATE_HASH_GATE =",
    full_state_hash_gate,
)

print(
    "ENCODER_STRUCTURE_GATE =",
    encoder_structure_gate,
)

print(
    "STRICT_LOAD_GATE =",
    strict_load_gate,
)

print(
    "ENCODER_FREEZE_GATE =",
    encoder_freeze_gate,
)

print(
    "CENTRAL_INPUT_GATE =",
    central_input_gate,
)

print(
    "LOGVAR_GATE =",
    logvar_gate,
)

print(
    "ENCODER_STATE_PRESERVED_GATE =",
    encoder_state_preserved_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "PERTURBED_VIEWS_GENERATED =",
    False,
)

print(
    "IIC_MODEL_CREATED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A4B_FINAL_ENCODER_REAL_DATA_PASS =",
    a4b_gate,
)

Loading FINAL checkpoint on CPU...

=== NOTEBOOK 06 FINAL ENCODER REAL-DATA AUDIT ===

CHECKPOINT_KIND = FINAL
CHECKPOINT_LAST_COMPLETED_EPOCH = 99
CHECKPOINT_OPTIMIZER_STEP = 5400
FULL_MODEL_STATE_SHA256 = 3ac5a1cecdee2aa204419290b5cec55f3e6a4a846beebaa3f6b4e495f6fe6096

ENCODER_PARAMETERS = 2077824
ENCODER_PARAMETER_TENSORS = 14
ENCODER_STATE_TENSORS = 26
ENCODER_FINAL_STATE_SHA256 = c9c1fe2c2ae0d27d7f5b8c8bef81b727f0da957744027f150e2af10d3a315dfd

CENTRAL_AUDIT_MANIFEST_INDICES = [0, 2145, 4291, 6864, 7663, 8582]
OBSERVED_MU_SHAPE = (6, 512)

MU_BITWISE_EQUAL = False
MU_MAX_ABS_DIFFERENCE = 1.1920928955078125e-06
MU_MEAN_ABS_DIFFERENCE = 1.5737835212803475e-07
MU_NUMERIC_GATE = True

CHECKPOINT_IDENTITY_GATE = True
FULL_STATE_HASH_GATE = True
ENCODER_STRUCTURE_GATE = True
STRICT_LOAD_GATE = True
ENCODER_FREEZE_GATE = True
CENTRAL_INPUT_GATE = True
LOGVAR_GATE = True
ENCODER_STATE_PRESERVED_GATE = True

LABELS_READ = False
PERTURBED_VIEWS_GENERATED = False
IIC_MODEL_CREATED = False
O

## STEP 06-A4C — Audit delle due interpretazioni della perturbazione temporale

L'IIC utilizza due viste dello stesso evento:

$$
x
$$

e

$$
x'=\xi(x).
$$

La vista originale $x$ è il crop centrale senza time shift ed è già stata
verificata nello STEP 06-A4B.

Lo snapshot HTML pubblicato descrive invece la vista perturbata mediante uno
shift temporale casuale nel range:

$$
0\text{--}24\ \mathrm{px}.
$$

Le informazioni pubblicate non determinano però in modo sufficiente la
traduzione esatta di tale intervallo nella coordinata operativa
`crop_left` del frame largo 272 px.

Per evitare di nascondere questa ambiguità vengono quindi mantenute e
verificate due interpretazioni PROJECT, entrambe congelate prima delle label.

### Interpretazione A — `legacy_text`

Traduzione più letterale del testo:

$$
\mathrm{crop\_left}\in\{0,\ldots,24\}.
$$

Poiché il crop centrale è:

$$
\mathrm{crop\_left}=24,
$$

nella convenzione relativa del progetto ciò corrisponde a:

$$
s=\mathrm{crop\_left}-24
\in[-24,0].
$$

Questa variante viene mantenuta come ramo di fedeltà al testo HTML.

### Interpretazione B — `symmetric_figure`

Interpretazione già utilizzata nel training PROJECT del VAE full:

$$
\mathrm{crop\_left}\in\{0,\ldots,48\},
$$

ossia:

$$
s=\mathrm{crop\_left}-24
\in[-24,+24].
$$

Questa variante viene mantenuta come ramo di continuità con il VAE full già
addestrato.

### Regole comuni

Entrambe le modalità utilizzano:

$$
s_{\mathrm{augmentation}}=789
$$

e un mapping stateless:

$$
(\mathrm{seed},\mathrm{epoch},\mathrm{manifest\_index})
\longrightarrow
\mathrm{crop\_left}.
$$

Lo stesso crop viene applicato alle quattro durate dello stesso evento.

Entrambe le viste perturbate vengono elaborate dallo stesso encoder `FINAL`
congelato:

$$
z'=\mu_\phi(x').
$$

Lo step verifica:

- i golden vector C dell'augmentation stateless;
- il range completo dei crop per entrambe le interpretazioni;
- la variazione tra epoche;
- known-answer di crop su eventi reali;
- shape e normalizzazione degli input;
- finitezza delle feature perturbate;
- risposta dell'encoder al cambiamento del crop;
- coerenza quando il crop perturbato coincide con quello centrale;
- differenza fra le due interpretazioni;
- immutabilità dell'encoder.

Nessuna delle due modalità viene scelta sulla base delle label.

La decisione successiva conserverà entrambe come protocolli pre-registrati
prima di qualsiasi training IIC ufficiale.

In [ ]:
# ============================================================
# NOTEBOOK 06 — DUAL TEMPORAL-PERTURBATION AUDIT
# STEP 06-A4C
#
# Audits BOTH:
#   1. legacy_text       crop_left 0..24
#   2. symmetric_figure  crop_left 0..48
#
# NO labels.
# NO IIC classifier.
# NO optimizer.
# NO training.
# NO Drive writes.
# ============================================================

import hashlib

import numpy as np
import torch


# ============================================================
# PARENT STATE
# ============================================================

assert a4b_gate

EXPECTED_ENCODER_FINAL_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b727"
    "f0da957744027f150e2af10d3a315dfd"
)

assert (
    ENCODER_FINAL_STATE_SHA256
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)


# ============================================================
# EXACT STATELESS AUGMENTATION ALREADY VALIDATED IN NOTEBOOK 01
# ============================================================

MASK64 = (
    (1 << 64) - 1
)

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = (
    0x4753415547563121
)


def c_u64(value):

    return value & MASK64


def gs_augment_stateless_mix64(value):

    value = c_u64(
        value
        + 0x9E3779B97F4A7C15
    )

    value = c_u64(
        (
            value
            ^ (
                value >> 30
            )
        )
        * 0xBF58476D1CE4E5B9
    )

    value = c_u64(
        (
            value
            ^ (
                value >> 27
            )
        )
        * 0x94D049BB133111EB
    )

    return c_u64(
        value
        ^ (
            value >> 31
        )
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):

    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(
            base_seed
        )
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(
            epoch
        )
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(
            manifest_index
        )
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(
            mode
        )
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(
            stream_index
        )
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):

    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:

        min_crop = 0
        max_crop = 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:

        min_crop = 0
        max_crop = 48

    else:

        raise ValueError(
            "invalid augmentation mode"
        )


    crop_range = (
        max_crop
        - min_crop
        + 1
    )


    threshold = (
        c_u64(
            0 - crop_range
        )
        % crop_range
    )


    stream_index = 0


    while True:

        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )


        if value >= threshold:

            return (
                min_crop
                + value % crop_range
            )


        stream_index += 1


# ============================================================
# FIVE REAL C GOLDEN VECTORS FROM NOTEBOOK 01
# ============================================================

GOLDEN_VECTORS = [

    (
        0,
        0,
        0,
        GS_AUGMENT_MODE_LEGACY_TEXT,
        7,
    ),

    (
        123,
        0,
        7663,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        36,
    ),

    (
        789,
        79,
        4224,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        25,
    ),

    (
        456,
        3,
        42,
        GS_AUGMENT_MODE_LEGACY_TEXT,
        0,
    ),

    (
        MASK64,
        MASK64,
        MASK64,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        40,
    ),
]


golden_observed = []


for (
    seed,
    epoch,
    manifest_index,
    mode,
    expected,
) in GOLDEN_VECTORS:

    actual = (
        gs_augment_crop_left_for_event_mode(
            seed,
            epoch,
            manifest_index,
            mode,
        )
    )

    golden_observed.append(
        actual
    )

    assert actual == expected


augmentation_golden_gate = True


# ============================================================
# FULL POPULATION CROP MAPS
# ============================================================

AUGMENTATION_SEED = 789


def build_full_crop_map(
    epoch,
    mode,
):

    return np.asarray(
        [
            gs_augment_crop_left_for_event_mode(
                AUGMENTATION_SEED,
                epoch,
                manifest_index,
                mode,
            )

            for manifest_index
            in range(
                8583
            )
        ],
        dtype=np.int16,
    )


legacy_crop_epoch0 = build_full_crop_map(
    epoch=0,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)

legacy_crop_epoch1 = build_full_crop_map(
    epoch=1,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


symmetric_crop_epoch0 = build_full_crop_map(
    epoch=0,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)

symmetric_crop_epoch1 = build_full_crop_map(
    epoch=1,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


# ============================================================
# FULL-POPULATION KNOWN ANSWERS
# ============================================================

legacy_full_gate = (

    legacy_crop_epoch0.shape
    == (
        8583,
    )

    and int(
        legacy_crop_epoch0.min()
    ) == 0

    and int(
        legacy_crop_epoch0.max()
    ) == 24

    and len(
        np.unique(
            legacy_crop_epoch0
        )
    ) == 25

    and int(
        np.sum(
            legacy_crop_epoch0 == 24
        )
    ) == 365

    and int(
        np.sum(
            legacy_crop_epoch0
            != legacy_crop_epoch1
        )
    ) == 8236
)


symmetric_full_gate = (

    symmetric_crop_epoch0.shape
    == (
        8583,
    )

    and int(
        symmetric_crop_epoch0.min()
    ) == 0

    and int(
        symmetric_crop_epoch0.max()
    ) == 48

    and len(
        np.unique(
            symmetric_crop_epoch0
        )
    ) == 49

    and int(
        np.sum(
            symmetric_crop_epoch0 == 24
        )
    ) == 180

    and int(
        np.sum(
            symmetric_crop_epoch0
            != symmetric_crop_epoch1
        )
    ) == 8394
)


assert legacy_full_gate
assert symmetric_full_gate


dual_crop_distinct_gate = (
    not np.array_equal(
        legacy_crop_epoch0,
        symmetric_crop_epoch0,
    )
)


assert dual_crop_distinct_gate


# ============================================================
# CROP MAP HASHES
# ============================================================

def int16_sha256(array):

    return hashlib.sha256(
        np.asarray(
            array,
            dtype="<i2",
        ).tobytes(
            order="C"
        )
    ).hexdigest()


LEGACY_CROP_EPOCH0_SHA256 = (
    int16_sha256(
        legacy_crop_epoch0
    )
)

LEGACY_CROP_EPOCH1_SHA256 = (
    int16_sha256(
        legacy_crop_epoch1
    )
)

SYMMETRIC_CROP_EPOCH0_SHA256 = (
    int16_sha256(
        symmetric_crop_epoch0
    )
)

SYMMETRIC_CROP_EPOCH1_SHA256 = (
    int16_sha256(
        symmetric_crop_epoch1
    )
)


# ============================================================
# FIXED LABEL-BLIND REAL-DATA AUDIT SET
# ============================================================

AUDIT_MANIFEST_INDICES = [
    0,
    2145,
    4291,
    6864,
    7663,
    8582,
]


EXPECTED_LEGACY_EPOCH0 = [
    7,
    21,
    1,
    20,
    6,
    21,
]

EXPECTED_LEGACY_EPOCH1 = [
    24,
    7,
    15,
    3,
    10,
    12,
]


EXPECTED_SYMMETRIC_EPOCH0 = [
    29,
    39,
    22,
    2,
    36,
    21,
]

EXPECTED_SYMMETRIC_EPOCH1 = [
    26,
    26,
    42,
    21,
    48,
    7,
]


def selected_crops(
    full_map,
):

    return [
        int(
            full_map[
                manifest_index
            ]
        )

        for manifest_index
        in AUDIT_MANIFEST_INDICES
    ]


legacy_selected_epoch0 = selected_crops(
    legacy_crop_epoch0
)

legacy_selected_epoch1 = selected_crops(
    legacy_crop_epoch1
)

symmetric_selected_epoch0 = selected_crops(
    symmetric_crop_epoch0
)

symmetric_selected_epoch1 = selected_crops(
    symmetric_crop_epoch1
)


fixed_crop_known_answer_gate = (

    legacy_selected_epoch0
    == EXPECTED_LEGACY_EPOCH0

    and legacy_selected_epoch1
    == EXPECTED_LEGACY_EPOCH1

    and symmetric_selected_epoch0
    == EXPECTED_SYMMETRIC_EPOCH0

    and symmetric_selected_epoch1
    == EXPECTED_SYMMETRIC_EPOCH1
)


assert fixed_crop_known_answer_gate


# ============================================================
# REAL IMAGE BATCH BUILDER
# ============================================================

def make_real_perturbed_batch(
    manifest_indices,
    *,
    epoch,
    mode,
):

    tensors = []
    crops = []


    for manifest_index in manifest_indices:

        crop_left = (
            gs_augment_crop_left_for_event_mode(
                AUGMENTATION_SEED,
                epoch,
                manifest_index,
                mode,
            )
        )


        tensor = load_png_event_crop_224(
            manifest_index,
            crop_left,
        )


        tensors.append(
            tensor
        )

        crops.append(
            int(
                crop_left
            )
        )


    return (
        torch.stack(
            tensors,
            dim=0,
        ),
        crops,
    )


# ============================================================
# BUILD FOUR REAL BATCHES
# ============================================================

(
    legacy_batch_epoch0_cpu,
    legacy_batch_crops_epoch0,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=0,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


(
    legacy_batch_epoch1_cpu,
    legacy_batch_crops_epoch1,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=1,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


(
    symmetric_batch_epoch0_cpu,
    symmetric_batch_crops_epoch0,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=0,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


(
    symmetric_batch_epoch1_cpu,
    symmetric_batch_crops_epoch1,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=1,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


# ============================================================
# INPUT CONTRACT
# ============================================================

def real_batch_valid(
    batch,
):

    return (

        tuple(
            batch.shape
        )
        == (
            6,
            4,
            224,
            224,
        )

        and batch.dtype
            == torch.float32

        and torch.isfinite(
            batch
        ).all().item()

        and float(
            batch.min()
        ) >= 0.0

        and float(
            batch.max()
        ) <= 1.0
    )


perturbed_input_gate = all([

    real_batch_valid(
        legacy_batch_epoch0_cpu
    ),

    real_batch_valid(
        legacy_batch_epoch1_cpu
    ),

    real_batch_valid(
        symmetric_batch_epoch0_cpu
    ),

    real_batch_valid(
        symmetric_batch_epoch1_cpu
    ),
])


assert perturbed_input_gate


# ============================================================
# SAME FROZEN ENCODER FOR ALL FOUR CASES
# ============================================================

encoder_state_before_dual_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


def frozen_encoder_forward(
    batch_cpu,
):

    with torch.inference_mode():

        (
            mu_gpu,
            logvar_gpu,
        ) = official_iic_encoder(
            batch_cpu.to(
                runtime_device,
                dtype=torch.float32,
            )
        )


    mu = (
        mu_gpu
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )


    logvar = (
        logvar_gpu
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )


    return (
        mu,
        logvar,
    )


(
    legacy_mu_epoch0,
    legacy_logvar_epoch0,
) = frozen_encoder_forward(
    legacy_batch_epoch0_cpu
)


(
    legacy_mu_epoch1,
    legacy_logvar_epoch1,
) = frozen_encoder_forward(
    legacy_batch_epoch1_cpu
)


(
    symmetric_mu_epoch0,
    symmetric_logvar_epoch0,
) = frozen_encoder_forward(
    symmetric_batch_epoch0_cpu
)


(
    symmetric_mu_epoch1,
    symmetric_logvar_epoch1,
) = frozen_encoder_forward(
    symmetric_batch_epoch1_cpu
)


torch.cuda.synchronize()


# ============================================================
# FEATURE CONTRACT
# ============================================================

def feature_matrix_valid(
    mu,
    logvar,
):

    return (

        mu.shape
        == (
            6,
            512,
        )

        and logvar.shape
        == (
            6,
            512,
        )

        and np.isfinite(
            mu
        ).all()

        and np.isfinite(
            logvar
        ).all()
    )


perturbed_feature_gate = all([

    feature_matrix_valid(
        legacy_mu_epoch0,
        legacy_logvar_epoch0,
    ),

    feature_matrix_valid(
        legacy_mu_epoch1,
        legacy_logvar_epoch1,
    ),

    feature_matrix_valid(
        symmetric_mu_epoch0,
        symmetric_logvar_epoch0,
    ),

    feature_matrix_valid(
        symmetric_mu_epoch1,
        symmetric_logvar_epoch1,
    ),
])


assert perturbed_feature_gate


# ============================================================
# CENTRAL REFERENCE
# ============================================================

central_mu_audit = np.asarray(
    full_central_mu[
        AUDIT_MANIFEST_INDICES
    ],
    dtype=np.float32,
)


# ============================================================
# RESPONSE TO CROP
#
# crop_left == 24:
#   should reproduce central mu numerically.
#
# crop_left != 24:
#   feature must respond to changed image.
# ============================================================

def central_response_gate(
    mu,
    crops,
):

    gates = []


    for row_index, crop_left in enumerate(
        crops
    ):

        if crop_left == 24:

            ok = np.allclose(
                mu[
                    row_index
                ],
                central_mu_audit[
                    row_index
                ],
                rtol=1.0e-6,
                atol=1.0e-5,
            )

        else:

            distance = float(
                np.linalg.norm(
                    mu[
                        row_index
                    ]
                    - central_mu_audit[
                        row_index
                    ]
                )
            )

            ok = (
                distance > 0.0
            )


        gates.append(
            bool(
                ok
            )
        )


    return all(
        gates
    )


legacy_response_epoch0_gate = (
    central_response_gate(
        legacy_mu_epoch0,
        legacy_batch_crops_epoch0,
    )
)


legacy_response_epoch1_gate = (
    central_response_gate(
        legacy_mu_epoch1,
        legacy_batch_crops_epoch1,
    )
)


symmetric_response_epoch0_gate = (
    central_response_gate(
        symmetric_mu_epoch0,
        symmetric_batch_crops_epoch0,
    )
)


symmetric_response_epoch1_gate = (
    central_response_gate(
        symmetric_mu_epoch1,
        symmetric_batch_crops_epoch1,
    )
)


feature_response_gate = all([

    legacy_response_epoch0_gate,
    legacy_response_epoch1_gate,

    symmetric_response_epoch0_gate,
    symmetric_response_epoch1_gate,
])


assert feature_response_gate


# ============================================================
# EPOCH-TO-EPOCH RESPONSE
# ============================================================

legacy_epoch_l2 = np.linalg.norm(
    legacy_mu_epoch0
    - legacy_mu_epoch1,
    axis=1,
)


symmetric_epoch_l2 = np.linalg.norm(
    symmetric_mu_epoch0
    - symmetric_mu_epoch1,
    axis=1,
)


epoch_feature_change_gate = (

    np.all(
        legacy_epoch_l2 > 0.0
    )

    and np.all(
        symmetric_epoch_l2 > 0.0
    )
)


assert epoch_feature_change_gate


# ============================================================
# LEGACY vs SYMMETRIC RESPONSE
#
# If the two modes choose the same crop for a sample, the
# corresponding features should agree numerically.
#
# If they choose different crops, they should respond.
# ============================================================

legacy_vs_symmetric_l2_epoch0 = np.linalg.norm(
    legacy_mu_epoch0
    - symmetric_mu_epoch0,
    axis=1,
)


dual_mode_feature_gates = []


for row_index in range(
    len(
        AUDIT_MANIFEST_INDICES
    )
):

    legacy_crop = (
        legacy_batch_crops_epoch0[
            row_index
        ]
    )

    symmetric_crop = (
        symmetric_batch_crops_epoch0[
            row_index
        ]
    )


    if legacy_crop == symmetric_crop:

        ok = np.allclose(
            legacy_mu_epoch0[
                row_index
            ],
            symmetric_mu_epoch0[
                row_index
            ],
            rtol=1.0e-6,
            atol=1.0e-5,
        )

    else:

        ok = (
            legacy_vs_symmetric_l2_epoch0[
                row_index
            ]
            > 0.0
        )


    dual_mode_feature_gates.append(
        bool(
            ok
        )
    )


dual_mode_feature_gate = all(
    dual_mode_feature_gates
)


assert dual_mode_feature_gate


# ============================================================
# FEATURE HASHES — AUDIT SET ONLY
# ============================================================

def float32_sha256(
    array,
):

    return hashlib.sha256(
        np.asarray(
            array,
            dtype="<f4",
        ).tobytes(
            order="C"
        )
    ).hexdigest()


LEGACY_MU_EPOCH0_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch0
    )
)

LEGACY_MU_EPOCH1_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch1
    )
)

SYMMETRIC_MU_EPOCH0_AUDIT_SHA256 = (
    float32_sha256(
        symmetric_mu_epoch0
    )
)

SYMMETRIC_MU_EPOCH1_AUDIT_SHA256 = (
    float32_sha256(
        symmetric_mu_epoch1
    )
)


# ============================================================
# DESCRIPTIVE L2 DISTANCES
# ============================================================

legacy_central_l2_epoch0 = np.linalg.norm(
    legacy_mu_epoch0
    - central_mu_audit,
    axis=1,
)


legacy_central_l2_epoch1 = np.linalg.norm(
    legacy_mu_epoch1
    - central_mu_audit,
    axis=1,
)


symmetric_central_l2_epoch0 = np.linalg.norm(
    symmetric_mu_epoch0
    - central_mu_audit,
    axis=1,
)


symmetric_central_l2_epoch1 = np.linalg.norm(
    symmetric_mu_epoch1
    - central_mu_audit,
    axis=1,
)


# ============================================================
# ENCODER MUST REMAIN IMMUTABLE
# ============================================================

encoder_state_after_dual_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_state_dual_gate = (

    encoder_state_before_dual_audit
    == encoder_state_after_dual_audit
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)


assert encoder_state_dual_gate


# ============================================================
# DUAL-POLICY AUDIT CONTRACT
#
# IMPORTANT:
# This does NOT select the official training branch yet.
# ============================================================

IIC_DUAL_PERTURBATION_AUDIT = {

    "augmentation_seed":
        789,

    "central_crop_left":
        24,

    "legacy_text": {

        "mode_id":
            GS_AUGMENT_MODE_LEGACY_TEXT,

        "crop_left_range":
            [
                0,
                24,
            ],

        "relative_shift_range":
            [
                -24,
                0,
            ],

        "role":
            "HTML_text_fidelity_interpretation",
    },

    "symmetric_figure": {

        "mode_id":
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,

        "crop_left_range":
            [
                0,
                48,
            ],

        "relative_shift_range":
            [
                -24,
                24,
            ],

        "role":
            "PROJECT_VAE_continuity_interpretation",
    },

    "both_retained":
        True,

    "training_branch_selected":
        False,

    "selection_based_on_labels":
        False,
}


# ============================================================
# FINAL GATE
# ============================================================

a4c_dual_gate = all([

    augmentation_golden_gate,

    legacy_full_gate,

    symmetric_full_gate,

    dual_crop_distinct_gate,

    fixed_crop_known_answer_gate,

    perturbed_input_gate,

    perturbed_feature_gate,

    feature_response_gate,

    epoch_feature_change_gate,

    dual_mode_feature_gate,

    encoder_state_dual_gate,

    IIC_DUAL_PERTURBATION_AUDIT[
        "both_retained"
    ],

    not IIC_DUAL_PERTURBATION_AUDIT[
        "training_branch_selected"
    ],
])


assert a4c_dual_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 DUAL TEMPORAL-PERTURBATION AUDIT ==="
)

print()

print(
    "AUGMENTATION_GOLDEN_GATE =",
    augmentation_golden_gate,
)

print()

print(
    "--- LEGACY_TEXT ---"
)

print(
    "LEGACY_EPOCH0_RANGE =",
    (
        int(
            legacy_crop_epoch0.min()
        ),
        int(
            legacy_crop_epoch0.max()
        ),
    ),
)

print(
    "LEGACY_EPOCH0_DISTINCT_CROPS =",
    len(
        np.unique(
            legacy_crop_epoch0
        )
    ),
)

print(
    "LEGACY_EPOCH0_CENTRAL_EVENTS =",
    int(
        np.sum(
            legacy_crop_epoch0 == 24
        )
    ),
)

print(
    "LEGACY_EVENTS_CHANGED_EPOCH0_TO_1 =",
    int(
        np.sum(
            legacy_crop_epoch0
            != legacy_crop_epoch1
        )
    ),
)

print(
    "LEGACY_AUDIT_CROPS_EPOCH0 =",
    legacy_selected_epoch0,
)

print(
    "LEGACY_AUDIT_CROPS_EPOCH1 =",
    legacy_selected_epoch1,
)

print(
    "LEGACY_CROP_EPOCH0_SHA256 =",
    LEGACY_CROP_EPOCH0_SHA256,
)

print(
    "LEGACY_CROP_EPOCH1_SHA256 =",
    LEGACY_CROP_EPOCH1_SHA256,
)

print()

print(
    "--- SYMMETRIC_FIGURE ---"
)

print(
    "SYMMETRIC_EPOCH0_RANGE =",
    (
        int(
            symmetric_crop_epoch0.min()
        ),
        int(
            symmetric_crop_epoch0.max()
        ),
    ),
)

print(
    "SYMMETRIC_EPOCH0_DISTINCT_CROPS =",
    len(
        np.unique(
            symmetric_crop_epoch0
        )
    ),
)

print(
    "SYMMETRIC_EPOCH0_CENTRAL_EVENTS =",
    int(
        np.sum(
            symmetric_crop_epoch0 == 24
        )
    ),
)

print(
    "SYMMETRIC_EVENTS_CHANGED_EPOCH0_TO_1 =",
    int(
        np.sum(
            symmetric_crop_epoch0
            != symmetric_crop_epoch1
        )
    ),
)

print(
    "SYMMETRIC_AUDIT_CROPS_EPOCH0 =",
    symmetric_selected_epoch0,
)

print(
    "SYMMETRIC_AUDIT_CROPS_EPOCH1 =",
    symmetric_selected_epoch1,
)

print(
    "SYMMETRIC_CROP_EPOCH0_SHA256 =",
    SYMMETRIC_CROP_EPOCH0_SHA256,
)

print(
    "SYMMETRIC_CROP_EPOCH1_SHA256 =",
    SYMMETRIC_CROP_EPOCH1_SHA256,
)

print()

print(
    "--- FEATURE RESPONSE ---"
)

print(
    "LEGACY_CENTRAL_L2_EPOCH0 =",
    legacy_central_l2_epoch0.tolist(),
)

print(
    "LEGACY_CENTRAL_L2_EPOCH1 =",
    legacy_central_l2_epoch1.tolist(),
)

print(
    "SYMMETRIC_CENTRAL_L2_EPOCH0 =",
    symmetric_central_l2_epoch0.tolist(),
)

print(
    "SYMMETRIC_CENTRAL_L2_EPOCH1 =",
    symmetric_central_l2_epoch1.tolist(),
)

print(
    "LEGACY_VS_SYMMETRIC_L2_EPOCH0 =",
    legacy_vs_symmetric_l2_epoch0.tolist(),
)

print()

print(
    "LEGACY_MU_EPOCH0_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH0_AUDIT_SHA256,
)

print(
    "LEGACY_MU_EPOCH1_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH1_AUDIT_SHA256,
)

print(
    "SYMMETRIC_MU_EPOCH0_AUDIT_SHA256 =",
    SYMMETRIC_MU_EPOCH0_AUDIT_SHA256,
)

print(
    "SYMMETRIC_MU_EPOCH1_AUDIT_SHA256 =",
    SYMMETRIC_MU_EPOCH1_AUDIT_SHA256,
)

print()

print(
    "LEGACY_FULL_GATE =",
    legacy_full_gate,
)

print(
    "SYMMETRIC_FULL_GATE =",
    symmetric_full_gate,
)

print(
    "DUAL_CROP_DISTINCT_GATE =",
    dual_crop_distinct_gate,
)

print(
    "FIXED_CROP_KNOWN_ANSWER_GATE =",
    fixed_crop_known_answer_gate,
)

print(
    "PERTURBED_INPUT_GATE =",
    perturbed_input_gate,
)

print(
    "PERTURBED_FEATURE_GATE =",
    perturbed_feature_gate,
)

print(
    "FEATURE_RESPONSE_GATE =",
    feature_response_gate,
)

print(
    "EPOCH_FEATURE_CHANGE_GATE =",
    epoch_feature_change_gate,
)

print(
    "DUAL_MODE_FEATURE_GATE =",
    dual_mode_feature_gate,
)

print(
    "ENCODER_STATE_DUAL_GATE =",
    encoder_state_dual_gate,
)

print()

print(
    "BOTH_INTERPRETATIONS_RETAINED =",
    True,
)

print(
    "TRAINING_BRANCH_SELECTED =",
    False,
)

print(
    "SELECTION_BASED_ON_LABELS =",
    False,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "IIC_MODEL_CREATED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A4C_DUAL_PERTURBATION_PASS =",
    a4c_dual_gate,
)


=== NOTEBOOK 06 DUAL TEMPORAL-PERTURBATION AUDIT ===

AUGMENTATION_GOLDEN_GATE = True

--- LEGACY_TEXT ---
LEGACY_EPOCH0_RANGE = (0, 24)
LEGACY_EPOCH0_DISTINCT_CROPS = 25
LEGACY_EPOCH0_CENTRAL_EVENTS = 365
LEGACY_EVENTS_CHANGED_EPOCH0_TO_1 = 8236
LEGACY_AUDIT_CROPS_EPOCH0 = [7, 21, 1, 20, 6, 21]
LEGACY_AUDIT_CROPS_EPOCH1 = [24, 7, 15, 3, 10, 12]
LEGACY_CROP_EPOCH0_SHA256 = 30dbe7e291812b3515999e729d57e6a4f5e12717762d3021f6601d8def952b30
LEGACY_CROP_EPOCH1_SHA256 = 40c67f6e1ec6f4dbb5706efb7fe14687ccfb2bd47143d2fd21cfb17667c5d5de

--- SYMMETRIC_FIGURE ---
SYMMETRIC_EPOCH0_RANGE = (0, 48)
SYMMETRIC_EPOCH0_DISTINCT_CROPS = 49
SYMMETRIC_EPOCH0_CENTRAL_EVENTS = 180
SYMMETRIC_EVENTS_CHANGED_EPOCH0_TO_1 = 8394
SYMMETRIC_AUDIT_CROPS_EPOCH0 = [29, 39, 22, 2, 36, 21]
SYMMETRIC_AUDIT_CROPS_EPOCH1 = [26, 26, 42, 21, 48, 7]
SYMMETRIC_CROP_EPOCH0_SHA256 = bf9da0f12354ebb6843ce21e15f4831df11849001741cbb40a7c3567bae59159
SYMMETRIC_CROP_EPOCH1_SHA256 = a118687a1ccb3b1062b57e697a05c2e45eef121790fe0a3aa9

## STEP 06-A4D — Pre-registrazione delle due varianti IIC

L'audit precedente ha confermato che entrambe le interpretazioni operative
della perturbazione temporale sono numericamente valide.

L'ambiguità non viene risolta utilizzando le label.

Vengono quindi pre-registrate due run IIC complete.

### Ramo A — `symmetric_figure`

$$
\mathrm{crop\_left}\in[0,48].
$$

Questo è il ramo principale PROJECT del full perché mantiene continuità con
la perturbazione già utilizzata durante il training del VAE `sakai224_full`.

Percorso:

$$
\mathrm{VAE}_{\mathrm{symmetric}}
\rightarrow
\mathrm{IIC}_{\mathrm{symmetric}}
\rightarrow
K=5
\rightarrow
\mathrm{spectral}.
$$

### Ramo B — `legacy_text`

$$
\mathrm{crop\_left}\in[0,24].
$$

Questo ramo costituisce una fidelity sensitivity pre-registrata rispetto alla
formulazione testuale `0--24 px` dello snapshot HTML.

Il VAE sottostante resta lo stesso VAE full già congelato; pertanto questa
variante testa specificamente la sensibilità della fase IIC
all'interpretazione della perturbazione e non costituisce un secondo
training VAE.

### Parametri identici

Le due varianti condividono:

$$
C=36,
\qquad
W=250,
\qquad
K=5,
$$

$$
B=128,
\qquad
E=100,
$$

$$
\eta=5\times10^{-4},
$$

Adam con:

$$
\beta_1=0.9,
\qquad
\beta_2=0.999,
\qquad
\epsilon=10^{-8},
$$

gli stessi seed dei classificatori, lo stesso shuffle, lo stesso encoder
`FINAL` congelato e la stessa definizione della loss.

L'unica differenza intenzionale fra le due run è la modalità di perturbazione
temporale.

### Policy di valutazione

Entrambe le pipeline saranno completate fino alla partizione spectral finale.

Solo dopo il congelamento di entrambe verranno associate le Gravity Spy label.

Non verrà selezionata a posteriori la variante con la migliore metrica.

Il ramo `symmetric_figure` rimane il risultato principale del progetto;
`legacy_text` viene riportato separatamente come sensitivity di fedeltà al
testo pubblicato.

In [ ]:
# ============================================================
# NOTEBOOK 06 — DUAL IIC EXPERIMENT PRE-REGISTRATION
# STEP 06-A4D
#
# CONTRACT ONLY.
# NO models.
# NO optimizers.
# NO training.
# NO labels.
# NO Drive writes.
# ============================================================

import hashlib
import json


# ============================================================
# REQUIRED COMPLETED GATES
# ============================================================

assert a4b_gate
assert a4c_dual_gate

assert (
    IIC_FINAL_TRAINING_SHA256
    == "4f27a9bbf4290dff54767655d2025928c2eac751607d217925b0d5bba2b2f537"
)


# ============================================================
# IDENTIFY THE EXACT A4C AUDIT
# ============================================================

IIC_DUAL_PERTURBATION_AUDIT_IDENTITY = {

    "augmentation_seed":
        789,

    "legacy": {
        "epoch0_crop_sha256":
            LEGACY_CROP_EPOCH0_SHA256,

        "epoch1_crop_sha256":
            LEGACY_CROP_EPOCH1_SHA256,

        "epoch0_mu_audit_sha256":
            LEGACY_MU_EPOCH0_AUDIT_SHA256,

        "epoch1_mu_audit_sha256":
            LEGACY_MU_EPOCH1_AUDIT_SHA256,
    },

    "symmetric": {
        "epoch0_crop_sha256":
            SYMMETRIC_CROP_EPOCH0_SHA256,

        "epoch1_crop_sha256":
            SYMMETRIC_CROP_EPOCH1_SHA256,

        "epoch0_mu_audit_sha256":
            SYMMETRIC_MU_EPOCH0_AUDIT_SHA256,

        "epoch1_mu_audit_sha256":
            SYMMETRIC_MU_EPOCH1_AUDIT_SHA256,
    },
}


IIC_DUAL_PERTURBATION_AUDIT_JSON = json.dumps(
    IIC_DUAL_PERTURBATION_AUDIT_IDENTITY,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_DUAL_PERTURBATION_AUDIT_SHA256 = hashlib.sha256(
    IIC_DUAL_PERTURBATION_AUDIT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# TWO PRE-REGISTERED EXPERIMENTS
# ============================================================

IIC_EXPERIMENTS = {

    "symmetric_figure": {

        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "role":
            "PRIMARY_PROJECT_FULL_BRANCH",

        "augmentation_mode":
            "symmetric_figure",

        "augmentation_mode_id":
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,

        "crop_left_range":
            [0, 48],

        "relative_shift_range":
            [-24, 24],

        "vae_training_augmentation_matches":
            True,

        "full_pipeline_role":
            (
                "main end-to-end branch consistent "
                "with frozen VAE training"
            ),
    },


    "legacy_text": {

        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "role":
            "HTML_TEXT_FIDELITY_SENSITIVITY",

        "augmentation_mode":
            "legacy_text",

        "augmentation_mode_id":
            GS_AUGMENT_MODE_LEGACY_TEXT,

        "crop_left_range":
            [0, 24],

        "relative_shift_range":
            [-24, 0],

        "vae_training_augmentation_matches":
            False,

        "full_pipeline_role":
            (
                "IIC-only sensitivity to literal "
                "0-24 px text interpretation"
            ),
    },
}


# ============================================================
# SHARED FROZEN PROTOCOL
# ============================================================

IIC_DUAL_EXPERIMENT_CONTRACT = {

    "contract_version":
        1,

    "main_scientific_goal":
        "sakai224_full_vs_Sakai",

    "experiments":
        IIC_EXPERIMENTS,

    "shared_protocol": {

        "event_count":
            8583,

        "population_mode":
            "full_transductive",

        "central_feature":
            "canonical_mu_epoch100",

        "central_mu_sha256":
            FULL_CENTRAL_MU_SHA256,

        "encoder_checkpoint":
            "FINAL_epoch100",

        "encoder_state_sha256":
            ENCODER_FINAL_STATE_SHA256,

        "C":
            36,

        "W":
            250,

        "K":
            5,

        "batch_size":
            128,

        "epochs":
            100,

        "learning_rate":
            5.0e-4,

        "adam_betas":
            [0.9, 0.999],

        "adam_epsilon":
            1.0e-8,

        "classifier_seeds":
            [123, 124, 125, 126, 127],

        "shuffle_seed":
            101112,

        "augmentation_seed":
            789,

        "drop_last":
            False,

        "main_overcluster_policy":
            "simultaneous_equal_weight_sum",

        "loss_contract_sha256":
            IIC_LOSS_CONTRACT_SHA256,

        "architecture_contract_sha256":
            IIC_ARCHITECTURE_CONTRACT_SHA256,

        "training_core_sha256":
            IIC_TRAINING_CORE_SHA256,

        "final_training_sha256":
            IIC_FINAL_TRAINING_SHA256,

        "dual_perturbation_audit_sha256":
            IIC_DUAL_PERTURBATION_AUDIT_SHA256,
    },


    "evaluation_policy": {

        "both_runs_must_reach_final_partition":
            True,

        "spectral_required_for_both":
            True,

        "labels_before_both_spectral_frozen":
            False,

        "select_best_mode_using_labels":
            False,

        "primary_report_branch":
            "symmetric_figure",

        "fidelity_sensitivity_branch":
            "legacy_text",

        "report_both":
            True,
    },


    "future_mini_C11": {

        "mandatory_common_primary_mode":
            "symmetric_figure",

        "reason":
            (
                "matched continuity with mini VAE "
                "training and primary full branch"
            ),

        "legacy_text":
            (
                "full fidelity sensitivity; not silently "
                "promoted to mandatory mini protocol"
            ),

        "same_IIC_equations_required":
            True,

        "golden_vector_validation_required":
            True,
    },


    "labels_used":
        False,
}


# ============================================================
# HASH
# ============================================================

IIC_DUAL_EXPERIMENT_JSON = json.dumps(
    IIC_DUAL_EXPERIMENT_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_DUAL_EXPERIMENT_SHA256 = hashlib.sha256(
    IIC_DUAL_EXPERIMENT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# GATES
# ============================================================

two_run_gate = (
    set(
        IIC_EXPERIMENTS.keys()
    )
    == {
        "symmetric_figure",
        "legacy_text",
    }
)


single_difference_gate = (

    IIC_EXPERIMENTS[
        "symmetric_figure"
    ][
        "augmentation_mode"
    ]
    !=
    IIC_EXPERIMENTS[
        "legacy_text"
    ][
        "augmentation_mode"
    ]

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "shared_protocol"
    ][
        "C"
    ] == 36

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "shared_protocol"
    ][
        "W"
    ] == 250

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "shared_protocol"
    ][
        "K"
    ] == 5
)


label_blinding_gate = (

    IIC_DUAL_EXPERIMENT_CONTRACT[
        "evaluation_policy"
    ][
        "labels_before_both_spectral_frozen"
    ]
    is False

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "evaluation_policy"
    ][
        "select_best_mode_using_labels"
    ]
    is False

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "labels_used"
    ]
    is False
)


both_final_gate = (

    IIC_DUAL_EXPERIMENT_CONTRACT[
        "evaluation_policy"
    ][
        "both_runs_must_reach_final_partition"
    ]
    is True

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "evaluation_policy"
    ][
        "spectral_required_for_both"
    ]
    is True
)


c11_alignment_gate = (

    IIC_DUAL_EXPERIMENT_CONTRACT[
        "future_mini_C11"
    ][
        "mandatory_common_primary_mode"
    ]
    == "symmetric_figure"

    and IIC_DUAL_EXPERIMENT_CONTRACT[
        "future_mini_C11"
    ][
        "same_IIC_equations_required"
    ]
    is True
)


a4d_gate = all([
    two_run_gate,
    single_difference_gate,
    label_blinding_gate,
    both_final_gate,
    c11_alignment_gate,
])


assert a4d_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 DUAL IIC PRE-REGISTRATION ==="
)

print()

print(
    "PRIMARY_RUN_ID =",
    IIC_EXPERIMENTS[
        "symmetric_figure"
    ][
        "run_id"
    ],
)

print(
    "PRIMARY_MODE = symmetric_figure"
)

print(
    "PRIMARY_CROP_RANGE = [0, 48]"
)

print()

print(
    "FIDELITY_RUN_ID =",
    IIC_EXPERIMENTS[
        "legacy_text"
    ][
        "run_id"
    ],
)

print(
    "FIDELITY_MODE = legacy_text"
)

print(
    "FIDELITY_CROP_RANGE = [0, 24]"
)

print()

print(
    "BOTH_RUNS_C =",
    36,
)

print(
    "BOTH_RUNS_W =",
    250,
)

print(
    "BOTH_RUNS_K =",
    5,
)

print(
    "BOTH_RUNS_EPOCHS =",
    100,
)

print(
    "BOTH_RUNS_BATCH =",
    128,
)

print()

print(
    "DUAL_PERTURBATION_AUDIT_SHA256 =",
    IIC_DUAL_PERTURBATION_AUDIT_SHA256,
)

print(
    "IIC_DUAL_EXPERIMENT_SHA256 =",
    IIC_DUAL_EXPERIMENT_SHA256,
)

print()

print(
    "TWO_RUN_GATE =",
    two_run_gate,
)

print(
    "SINGLE_DIFFERENCE_GATE =",
    single_difference_gate,
)

print(
    "LABEL_BLINDING_GATE =",
    label_blinding_gate,
)

print(
    "BOTH_FINAL_GATE =",
    both_final_gate,
)

print(
    "C11_ALIGNMENT_GATE =",
    c11_alignment_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "IIC_MODEL_CREATED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A4D_DUAL_IIC_PREREGISTRATION_PASS =",
    a4d_gate,
)


=== NOTEBOOK 06 DUAL IIC PRE-REGISTRATION ===

PRIMARY_RUN_ID = sakai224_full_iic_symmetric_v1
PRIMARY_MODE = symmetric_figure
PRIMARY_CROP_RANGE = [0, 48]

FIDELITY_RUN_ID = sakai224_full_iic_legacy_text_v1
FIDELITY_MODE = legacy_text
FIDELITY_CROP_RANGE = [0, 24]

BOTH_RUNS_C = 36
BOTH_RUNS_W = 250
BOTH_RUNS_K = 5
BOTH_RUNS_EPOCHS = 100
BOTH_RUNS_BATCH = 128

DUAL_PERTURBATION_AUDIT_SHA256 = bc9754805f1188fed220fc857377b168b0b4fc290789933d38602925bcdbf94c
IIC_DUAL_EXPERIMENT_SHA256 = 4fbae488ccbe6226c4c589c8bfcd712ed20e0ddf65ec95e4dfb80120d8432fbc

TWO_RUN_GATE = True
SINGLE_DIFFERENCE_GATE = True
LABEL_BLINDING_GATE = True
BOTH_FINAL_GATE = True
C11_ALIGNMENT_GATE = True

LABELS_READ = False
IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A4D_DUAL_IIC_PREREGISTRATION_PASS = True


## STEP 06-A5A — Smoke end-to-end IIC su dati reali

Prima delle run ufficiali viene eseguito un singolo optimizer step reale per
ciascuna delle due perturbazioni pre-registrate.

Entrambi gli smoke utilizzano:

- gli stessi 128 eventi;
- lo stesso ordine PROJECT;
- le stesse feature centrali;
- lo stesso encoder `FINAL`;
- lo stesso seed di inizializzazione del classificatore;
- lo stesso optimizer Adam;
- la stessa loss.

L'unica differenza è:

$$
\mathrm{legacy\_text}
\quad\text{vs}\quad
\mathrm{symmetric\_figure}.
$$

### Pairing

Per ciascun evento:

$$
z=\mu_\phi(x),
\qquad
z'=\mu_\phi(x').
$$

La stessa head IIC viene applicata a entrambe le feature.

### BatchNorm — scelta PROJECT

Nel singolo train step viene eseguito:

1. forward della vista centrale $z$;
2. forward della vista perturbata $z'$;
3. calcolo delle due mutual information;
4. backward della somma;
5. un unico `optimizer.step()`.

Le statistiche BatchNorm vengono quindi aggiornate durante entrambi i forward.

Questa semantica non è specificata dal paper ed è classificata PROJECT.

### Optimizer PyTorch

Per rendere esplicita la realizzazione numerica di Adam vengono disabilitati
i percorsi `foreach` e `fused`.

Questo facilita la riproducibilità e la futura validazione della versione C11,
senza attribuire tale dettaglio a Sakai.

Lo smoke non produce artefatti ufficiali e non modifica il VAE.

In [ ]:
# ============================================================
# NOTEBOOK 06 — REAL END-TO-END IIC ONE-STEP SMOKE
# STEP 06-A5A
#
# 1 real Adam step for each pre-registered perturbation.
#
# NO labels.
# NO official runs.
# NO Drive writes.
# ============================================================

import hashlib

import numpy as np
import torch


# ============================================================
# PARENT GATES
# ============================================================

assert a4d_gate
assert runtime_device.type == "cuda"


# ============================================================
# EXACT PROJECT SHUFFLE FROM NOTEBOOK 01 / C
# ============================================================

MASK64 = (1 << 64) - 1

VAE_C_RNG_DOMAIN_SHUFFLE = 2

SPLITMIX_INCREMENT = 0x9E3779B97F4A7C15
SPLITMIX_MUL1 = 0xBF58476D1CE4E5B9
SPLITMIX_MUL2 = 0x94D049BB133111EB

DERIVE_SEED_TAG = 0xD1B54A32D192ED03


def u64(value):
    return value & MASK64


def vae_c_rng_mix64_py(value):

    value = u64(
        value + SPLITMIX_INCREMENT
    )

    value = u64(
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    )

    value = u64(
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    )

    return u64(
        value ^ (value >> 31)
    )


def vae_c_rng_derive_seed_py(
    base_seed,
    domain,
    epoch,
    manifest_index,
    stream_index,
):

    h = vae_c_rng_mix64_py(
        u64(base_seed)
        ^ DERIVE_SEED_TAG
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(domain)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(epoch)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(manifest_index)
    )

    h = vae_c_rng_mix64_py(
        h ^ u64(stream_index)
    )

    return h


def vae_c_rng_next_u64_py(state):

    state = u64(
        state + SPLITMIX_INCREMENT
    )

    value = state

    value = u64(
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    )

    value = u64(
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    )

    value = u64(
        value ^ (value >> 31)
    )

    return (
        state,
        value,
    )


def vae_c_shuffle_bounded_u64_py(
    state,
    upper_bound_exclusive,
):

    threshold = (
        u64(
            0
            - upper_bound_exclusive
        )
        % upper_bound_exclusive
    )

    while True:

        (
            state,
            random_value,
        ) = vae_c_rng_next_u64_py(
            state
        )

        if random_value >= threshold:

            return (
                state,
                random_value
                % upper_bound_exclusive,
            )


def vae_c_shuffle_make_permutation_py(
    count,
    base_seed,
    epoch,
):

    derived_seed = (
        vae_c_rng_derive_seed_py(
            base_seed,
            VAE_C_RNG_DOMAIN_SHUFFLE,
            epoch,
            0,
            0,
        )
    )

    state = derived_seed

    permutation = list(
        range(count)
    )


    for index in range(
        count - 1,
        0,
        -1,
    ):

        (
            state,
            selected,
        ) = vae_c_shuffle_bounded_u64_py(
            state,
            index + 1,
        )

        (
            permutation[index],
            permutation[selected],
        ) = (
            permutation[selected],
            permutation[index],
        )


    return permutation


# ============================================================
# ORIGINAL C GOLDEN VECTORS
# ============================================================

shuffle_golden_gate = (

    vae_c_shuffle_make_permutation_py(
        8,
        101112,
        0,
    )
    == [
        0, 5, 4, 7,
        1, 3, 6, 2,
    ]

    and vae_c_shuffle_make_permutation_py(
        8,
        101112,
        1,
    )
    == [
        2, 3, 1, 4,
        0, 5, 6, 7,
    ]
)


assert shuffle_golden_gate


# ============================================================
# FULL 8583 SHUFFLE — EPOCH 0
#
# Population is already manifest ordered, so permutation
# positions are directly manifest_index values.
# ============================================================

full_permutation_epoch0 = (
    vae_c_shuffle_make_permutation_py(
        8583,
        101112,
        0,
    )
)


EXPECTED_FIRST_16 = [
    6926,
    5642,
    4410,
    8192,
    953,
    7325,
    8221,
    8226,
    5494,
    962,
    1495,
    8456,
    3759,
    7567,
    4863,
    7695,
]


full_shuffle_gate = (

    len(
        full_permutation_epoch0
    ) == 8583

    and len(
        set(
            full_permutation_epoch0
        )
    ) == 8583

    and full_permutation_epoch0[
        :16
    ] == EXPECTED_FIRST_16
)


assert full_shuffle_gate


SMOKE_MANIFEST_INDICES = (
    full_permutation_epoch0[
        :128
    ]
)


assert len(
    SMOKE_MANIFEST_INDICES
) == 128


# ============================================================
# CENTRAL FEATURES
# ============================================================

central_mu_smoke_cpu = torch.from_numpy(
    np.asarray(
        full_central_mu[
            SMOKE_MANIFEST_INDICES
        ],
        dtype=np.float32,
    ).copy()
)


assert central_mu_smoke_cpu.shape == (
    128,
    512,
)


# ============================================================
# BUILD + ENCODE PERTURBED VIEWS
# ============================================================

def build_perturbed_mu_smoke(
    mode,
):

    (
        image_batch_cpu,
        crops,
    ) = make_real_perturbed_batch(
        SMOKE_MANIFEST_INDICES,
        epoch=0,
        mode=mode,
    )


    assert image_batch_cpu.shape == (
        128,
        4,
        224,
        224,
    )


    with torch.inference_mode():

        (
            mu_gpu,
            _,
        ) = official_iic_encoder(
            image_batch_cpu.to(
                runtime_device,
                dtype=torch.float32,
            )
        )


    torch.cuda.synchronize()


    mu_cpu = (
        mu_gpu
        .detach()
        .cpu()
    )


    return (
        mu_cpu,
        crops,
    )


print(
    "Encoding legacy_text smoke batch..."
)

(
    legacy_mu_smoke_cpu,
    legacy_smoke_crops,
) = build_perturbed_mu_smoke(
    GS_AUGMENT_MODE_LEGACY_TEXT
)


print(
    "Encoding symmetric_figure smoke batch..."
)

(
    symmetric_mu_smoke_cpu,
    symmetric_smoke_crops,
) = build_perturbed_mu_smoke(
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE
)


perturbed_mu_gate = (

    legacy_mu_smoke_cpu.shape
        == (
            128,
            512,
        )

    and symmetric_mu_smoke_cpu.shape
        == (
            128,
            512,
        )

    and torch.isfinite(
        legacy_mu_smoke_cpu
    ).all().item()

    and torch.isfinite(
        symmetric_mu_smoke_cpu
    ).all().item()
)


assert perturbed_mu_gate


# ============================================================
# PARAMETER-ONLY HASH
# ============================================================

def parameter_sha256(model):

    digest = hashlib.sha256()

    for name, parameter in sorted(
        model.named_parameters()
    ):

        tensor = (
            parameter
            .detach()
            .cpu()
            .contiguous()
        )

        digest.update(
            name.encode("utf-8")
        )

        digest.update(
            tensor.numpy().tobytes(
                order="C"
            )
        )

    return digest.hexdigest()


# ============================================================
# ONE REAL TRAIN STEP
# ============================================================

def run_one_step_smoke(
    *,
    mode_name,
    perturbed_mu_cpu,
):

    # Same initial classifier for both modes.
    model = build_iic_pair_from_seed(
        123
    ).to(
        runtime_device
    )


    initial_state_sha = (
        module_state_sha256(
            model
        )
    )


    assert initial_state_sha == (
        "f32c9bf96502452e9fc8d1b4f568b1f732"
        "67484afc7d200c11a42ba254505e44"
    )


    parameter_sha_before = (
        parameter_sha256(
            model
        )
    )


    model.train()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=5.0e-4,
        betas=(
            0.9,
            0.999,
        ),
        eps=1.0e-8,
        weight_decay=0.0,
        amsgrad=False,
        foreach=False,
        maximize=False,
        capturable=False,
        differentiable=False,
        fused=False,
    )


    central_gpu = (
        central_mu_smoke_cpu.to(
            runtime_device
        )
    )

    perturbed_gpu = (
        perturbed_mu_cpu.to(
            runtime_device
        )
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    # --------------------------------------------------------
    # SAME CLASSIFIER HEADS, TWO VIEWS
    #
    # PROJECT BN policy:
    # central forward first, perturbed forward second.
    # --------------------------------------------------------

    central_outputs = model(
        central_gpu
    )

    perturbed_outputs = model(
        perturbed_gpu
    )


    main_result = (
        project_iic_negative_mutual_information(
            central_outputs[
                "main"
            ],
            perturbed_outputs[
                "main"
            ],
        )
    )


    over_result = (
        project_iic_negative_mutual_information(
            central_outputs[
                "overcluster"
            ],
            perturbed_outputs[
                "overcluster"
            ],
        )
    )


    total_loss = combine_iic_losses(
        main_result[
            "loss"
        ],
        over_result[
            "loss"
        ],
    )


    loss_gate = bool(
        torch.isfinite(
            total_loss
        ).item()

        and torch.isfinite(
            main_result[
                "loss"
            ]
        ).item()

        and torch.isfinite(
            over_result[
                "loss"
            ]
        ).item()
    )


    assert loss_gate


    total_loss.backward()


    # --------------------------------------------------------
    # GRADIENT AUDIT
    # --------------------------------------------------------

    main_grad_norm_sq = 0.0
    over_grad_norm_sq = 0.0

    gradients_finite = True


    for parameter in (
        model.main_head.parameters()
    ):

        if parameter.grad is None:

            gradients_finite = False
            continue

        if not torch.isfinite(
            parameter.grad
        ).all():

            gradients_finite = False

        main_grad_norm_sq += float(
            torch.sum(
                parameter.grad
                * parameter.grad
            ).item()
        )


    for parameter in (
        model.overcluster_head.parameters()
    ):

        if parameter.grad is None:

            gradients_finite = False
            continue

        if not torch.isfinite(
            parameter.grad
        ).all():

            gradients_finite = False

        over_grad_norm_sq += float(
            torch.sum(
                parameter.grad
                * parameter.grad
            ).item()
        )


    main_grad_norm = (
        main_grad_norm_sq
        ** 0.5
    )

    over_grad_norm = (
        over_grad_norm_sq
        ** 0.5
    )


    gradient_gate = (
        gradients_finite
        and main_grad_norm > 0.0
        and over_grad_norm > 0.0
    )


    assert gradient_gate


    # Encoder must never receive gradients.
    encoder_gradient_gate = all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )


    assert encoder_gradient_gate


    # --------------------------------------------------------
    # ADAM STEP
    # --------------------------------------------------------

    optimizer.step()

    torch.cuda.synchronize()


    parameter_sha_after = (
        parameter_sha256(
            model
        )
    )


    parameter_update_gate = (
        parameter_sha_before
        != parameter_sha_after
    )


    optimizer_state_gate = (
        len(
            optimizer.state
        ) == 10
    )


    # Two train-mode forwards per head:
    # central + perturbed.
    bn_tracking_gate = (

        int(
            model
            .main_head
            .bn
            .num_batches_tracked
            .item()
        ) == 2

        and int(
            model
            .overcluster_head
            .bn
            .num_batches_tracked
            .item()
        ) == 2
    )


    assert parameter_update_gate
    assert optimizer_state_gate
    assert bn_tracking_gate


    return {

        "mode":
            mode_name,

        "main_loss":
            float(
                main_result[
                    "loss"
                ].detach().cpu().item()
            ),

        "overcluster_loss":
            float(
                over_result[
                    "loss"
                ].detach().cpu().item()
            ),

        "total_loss":
            float(
                total_loss
                .detach()
                .cpu()
                .item()
            ),

        "main_mi":
            float(
                main_result[
                    "mutual_information"
                ]
                .detach()
                .cpu()
                .item()
            ),

        "overcluster_mi":
            float(
                over_result[
                    "mutual_information"
                ]
                .detach()
                .cpu()
                .item()
            ),

        "main_grad_norm":
            main_grad_norm,

        "over_grad_norm":
            over_grad_norm,

        "initial_state_sha256":
            initial_state_sha,

        "parameter_sha_before":
            parameter_sha_before,

        "parameter_sha_after":
            parameter_sha_after,

        "loss_gate":
            loss_gate,

        "gradient_gate":
            gradient_gate,

        "encoder_gradient_gate":
            encoder_gradient_gate,

        "parameter_update_gate":
            parameter_update_gate,

        "optimizer_state_gate":
            optimizer_state_gate,

        "bn_tracking_gate":
            bn_tracking_gate,
    }


# ============================================================
# ENCODER STATE MUST REMAIN FROZEN
# ============================================================

encoder_state_before_smoke = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


print(
    "Running legacy_text one-step smoke..."
)

legacy_smoke = run_one_step_smoke(
    mode_name="legacy_text",
    perturbed_mu_cpu=legacy_mu_smoke_cpu,
)


print(
    "Running symmetric_figure one-step smoke..."
)

symmetric_smoke = run_one_step_smoke(
    mode_name="symmetric_figure",
    perturbed_mu_cpu=symmetric_mu_smoke_cpu,
)


encoder_state_after_smoke = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_immutability_gate = (

    encoder_state_before_smoke
    == encoder_state_after_smoke
    == ENCODER_FINAL_STATE_SHA256
)


assert encoder_immutability_gate


# ============================================================
# FAIR DUAL-MODE SMOKE
# ============================================================

same_initialization_gate = (

    legacy_smoke[
        "initial_state_sha256"
    ]
    ==
    symmetric_smoke[
        "initial_state_sha256"
    ]
)


same_central_batch_gate = (
    SMOKE_MANIFEST_INDICES[
        :16
    ]
    == EXPECTED_FIRST_16
)


dual_smoke_gate = (

    same_initialization_gate
    and same_central_batch_gate
    and encoder_immutability_gate

    and all(
        legacy_smoke[key]

        for key in [
            "loss_gate",
            "gradient_gate",
            "encoder_gradient_gate",
            "parameter_update_gate",
            "optimizer_state_gate",
            "bn_tracking_gate",
        ]
    )

    and all(
        symmetric_smoke[key]

        for key in [
            "loss_gate",
            "gradient_gate",
            "encoder_gradient_gate",
            "parameter_update_gate",
            "optimizer_state_gate",
            "bn_tracking_gate",
        ]
    )
)


assert dual_smoke_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 DUAL REAL IIC ONE-STEP SMOKE ==="
)

print()

print(
    "FIRST_16_MANIFEST_INDICES =",
    SMOKE_MANIFEST_INDICES[
        :16
    ],
)

print(
    "SHUFFLE_GOLDEN_GATE =",
    shuffle_golden_gate,
)

print(
    "FULL_SHUFFLE_GATE =",
    full_shuffle_gate,
)

print()

for result in [
    legacy_smoke,
    symmetric_smoke,
]:

    print(
        "---",
        result["mode"],
        "---",
    )

    print(
        "MAIN_MI =",
        result["main_mi"],
    )

    print(
        "OVERCLUSTER_MI =",
        result["overcluster_mi"],
    )

    print(
        "MAIN_LOSS =",
        result["main_loss"],
    )

    print(
        "OVERCLUSTER_LOSS =",
        result["overcluster_loss"],
    )

    print(
        "TOTAL_LOSS =",
        result["total_loss"],
    )

    print(
        "MAIN_GRAD_NORM =",
        result["main_grad_norm"],
    )

    print(
        "OVERCLUSTER_GRAD_NORM =",
        result["over_grad_norm"],
    )

    print(
        "PARAMETERS_CHANGED =",
        result[
            "parameter_update_gate"
        ],
    )

    print(
        "ADAM_STATE_TENSORS =",
        10,
    )

    print(
        "BN_BATCHES_TRACKED_PER_HEAD =",
        2,
    )

    print()


print(
    "SAME_INITIALIZATION_GATE =",
    same_initialization_gate,
)

print(
    "SAME_CENTRAL_BATCH_GATE =",
    same_central_batch_gate,
)

print(
    "PERTURBED_MU_GATE =",
    perturbed_mu_gate,
)

print(
    "ENCODER_IMMUTABILITY_GATE =",
    encoder_immutability_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "OFFICIAL_RUN_CREATED =",
    False,
)

print(
    "OFFICIAL_CHECKPOINT_WRITTEN =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5A_DUAL_REAL_ONE_STEP_SMOKE_PASS =",
    dual_smoke_gate,
)

Encoding legacy_text smoke batch...
Encoding symmetric_figure smoke batch...
Running legacy_text one-step smoke...
Running symmetric_figure one-step smoke...

=== NOTEBOOK 06 DUAL REAL IIC ONE-STEP SMOKE ===

FIRST_16_MANIFEST_INDICES = [6926, 5642, 4410, 8192, 953, 7325, 8221, 8226, 5494, 962, 1495, 8456, 3759, 7567, 4863, 7695]
SHUFFLE_GOLDEN_GATE = True
FULL_SHUFFLE_GATE = True

--- legacy_text ---
MAIN_MI = 0.0006363919237628579
OVERCLUSTER_MI = 0.00037637626519426703
MAIN_LOSS = -0.0006363919237628579
OVERCLUSTER_LOSS = -0.00037637626519426703
TOTAL_LOSS = -0.001012768130749464
MAIN_GRAD_NORM = 0.022840456708540516
OVERCLUSTER_GRAD_NORM = 0.0035355902585864555
PARAMETERS_CHANGED = True
ADAM_STATE_TENSORS = 10
BN_BATCHES_TRACKED_PER_HEAD = 2

--- symmetric_figure ---
MAIN_MI = 0.0005501513369381428
OVERCLUSTER_MI = 0.0003521889739204198
MAIN_LOSS = -0.0005501513369381428
OVERCLUSTER_LOSS = -0.0003521889739204198
TOTAL_LOSS = -0.0009023402817547321
MAIN_GRAD_NORM = 0.014050878735474

## STEP 06-A5B — Benchmark fair delle due pipeline IIC

Prima delle run ufficiali viene misurato il costo reale del training IIC sulla
GPU corrente.

Per ciascuna modalità:

$$
\mathrm{legacy\_text}
$$

e

$$
\mathrm{symmetric\_figure},
$$

vengono utilizzati:

- gli stessi eventi;
- lo stesso ordine;
- batch da 128;
- gli stessi cinque seed;
- cinque modelli e cinque optimizer indipendenti;
- la stessa policy BatchNorm;
- la stessa loss.

### Riutilizzo delle feature fra i cinque classificatori

Per ogni minibatch vengono calcolate una sola volta:

$$
z=\mu_\phi(x)
$$

e

$$
z'=\mu_\phi(x').
$$

Le stesse feature vengono poi fornite separatamente ai:

$$
K=5
$$

classificatori indipendenti.

Ogni classificatore mantiene:

- parametri propri;
- BatchNorm propria;
- optimizer Adam proprio;
- stato optimizer proprio.

Non viene mediata o condivisa alcuna loss fra classificatori.

Questa organizzazione è una ottimizzazione PROJECT dell'esecuzione: evita di
ricalcolare cinque volte lo stesso encoder congelato senza modificare
l'algoritmo dei singoli classificatori.

Il benchmark misura separatamente:

1. caricamento e crop delle PNG;
2. trasferimento GPU e forward dell'encoder congelato;
3. training delle cinque coppie IIC;
4. tempo end-to-end;
5. memoria GPU.

Nessun risultato del benchmark viene utilizzato per modificare i parametri
scientifici già congelati.

In [ ]:
# ============================================================
# NOTEBOOK 06 — FAIR K5 IIC BENCHMARK
# STEP 06-A5B — CORRECTED
#
# Temporary models only.
# Same batches for both perturbation modes.
#
# NO labels.
# NO official checkpoints.
# NO Drive writes.
# ============================================================

import gc
import statistics
import time

import numpy as np
import torch


assert dual_smoke_gate
assert a4d_gate
assert runtime_device.type == "cuda"


# ============================================================
# TEMPORARY OPTIMIZER FACTORY
# ============================================================

def make_iic_optimizer(model):

    return torch.optim.Adam(
        model.parameters(),
        lr=5.0e-4,
        betas=(0.9, 0.999),
        eps=1.0e-8,
        weight_decay=0.0,
        amsgrad=False,
        foreach=False,
        maximize=False,
        capturable=False,
        differentiable=False,
        fused=False,
    )


# ============================================================
# ONE CLASSIFIER STEP
# ============================================================

def train_one_iic_pair(
    model,
    optimizer,
    central_gpu,
    perturbed_gpu,
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    central_outputs = model(
        central_gpu
    )

    perturbed_outputs = model(
        perturbed_gpu
    )

    main_result = (
        project_iic_negative_mutual_information(
            central_outputs["main"],
            perturbed_outputs["main"],
        )
    )

    over_result = (
        project_iic_negative_mutual_information(
            central_outputs["overcluster"],
            perturbed_outputs["overcluster"],
        )
    )

    total_loss = combine_iic_losses(
        main_result["loss"],
        over_result["loss"],
    )

    assert torch.isfinite(
        total_loss
    ).item()

    total_loss.backward()

    optimizer.step()

    return float(
        total_loss.detach().item()
    )


# ============================================================
# STATE COMPARISON HELPER
# ============================================================

def module_state_allclose(
    model_a,
    model_b,
    *,
    rtol=1.0e-7,
    atol=1.0e-8,
):

    state_a = model_a.state_dict()
    state_b = model_b.state_dict()

    if set(state_a.keys()) != set(
        state_b.keys()
    ):
        return False

    for key in state_a:

        tensor_a = (
            state_a[key]
            .detach()
            .cpu()
        )

        tensor_b = (
            state_b[key]
            .detach()
            .cpu()
        )

        if tensor_a.dtype in (
            torch.int8,
            torch.int16,
            torch.int32,
            torch.int64,
            torch.uint8,
            torch.bool,
        ):

            if not torch.equal(
                tensor_a,
                tensor_b,
            ):
                return False

        else:

            if not torch.allclose(
                tensor_a,
                tensor_b,
                rtol=rtol,
                atol=atol,
            ):
                return False

    return True


# ============================================================
# PROVE THAT REUSING THE SAME FROZEN FEATURES IS SAFE
# ============================================================

equivalence_indices = (
    vae_c_shuffle_make_permutation_py(
        8583,
        101112,
        0,
    )[:128]
)


equivalence_central_gpu = torch.from_numpy(
    np.asarray(
        full_central_mu[
            equivalence_indices
        ],
        dtype=np.float32,
    ).copy()
).to(
    runtime_device
)


(
    equivalence_images_cpu,
    _,
) = make_real_perturbed_batch(
    equivalence_indices,
    epoch=0,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


with torch.no_grad():

    (
        equivalence_perturbed_gpu,
        _,
    ) = official_iic_encoder(
        equivalence_images_cpu.to(
            runtime_device,
            dtype=torch.float32,
        )
    )


equivalence_perturbed_gpu = (
    equivalence_perturbed_gpu
    .detach()
    .clone()
    .contiguous()
)


standalone_model = (
    build_iic_pair_from_seed(
        123
    ).to(
        runtime_device
    )
)

standalone_optimizer = (
    make_iic_optimizer(
        standalone_model
    )
)


ensemble_model_0 = (
    build_iic_pair_from_seed(
        123
    ).to(
        runtime_device
    )
)

ensemble_optimizer_0 = (
    make_iic_optimizer(
        ensemble_model_0
    )
)


initial_equivalence_gate = (
    module_state_sha256(
        standalone_model
    )
    == module_state_sha256(
        ensemble_model_0
    )
)


assert initial_equivalence_gate


standalone_loss = train_one_iic_pair(
    standalone_model,
    standalone_optimizer,
    equivalence_central_gpu,
    equivalence_perturbed_gpu,
)


ensemble0_loss = train_one_iic_pair(
    ensemble_model_0,
    ensemble_optimizer_0,
    equivalence_central_gpu,
    equivalence_perturbed_gpu,
)


torch.cuda.synchronize()


shared_feature_equivalence_gate = (

    np.isclose(
        standalone_loss,
        ensemble0_loss,
        rtol=1.0e-7,
        atol=1.0e-9,
    )

    and module_state_allclose(
        standalone_model,
        ensemble_model_0,
    )
)


assert shared_feature_equivalence_gate


del standalone_model
del standalone_optimizer
del ensemble_model_0
del ensemble_optimizer_0
del equivalence_central_gpu
del equivalence_perturbed_gpu
del equivalence_images_cpu

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# BENCHMARK BATCHES
# ============================================================

BENCHMARK_WARMUP_BATCHES = 1
BENCHMARK_MEASURED_BATCHES = 8

BENCHMARK_TOTAL_BATCHES = (
    BENCHMARK_WARMUP_BATCHES
    + BENCHMARK_MEASURED_BATCHES
)


benchmark_permutation = (
    vae_c_shuffle_make_permutation_py(
        8583,
        101112,
        0,
    )
)


benchmark_batches = []


for batch_index in range(
    BENCHMARK_TOTAL_BATCHES
):

    start = (
        batch_index
        * 128
    )

    stop = (
        start
        + 128
    )

    indices = benchmark_permutation[
        start:stop
    ]

    assert len(indices) == 128

    benchmark_batches.append(
        indices
    )


# ============================================================
# FAIR CPU FILE-CACHE PREWARM
# ============================================================

print(
    "Prewarming PNG file cache..."
)


for indices in benchmark_batches:

    (
        cache_batch,
        _,
    ) = make_real_perturbed_batch(
        indices,
        epoch=0,
        mode=GS_AUGMENT_MODE_LEGACY_TEXT,
    )

    del cache_batch


gc.collect()


# ============================================================
# ENCODER IMMUTABILITY BASELINE
# ============================================================

encoder_state_before_benchmark = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


assert (
    encoder_state_before_benchmark
    == ENCODER_FINAL_STATE_SHA256
)


# ============================================================
# FAIR MODE BENCHMARK
# ============================================================

def benchmark_iic_mode(
    *,
    mode_name,
    mode_id,
):

    gc.collect()
    torch.cuda.empty_cache()


    models = [
        build_iic_pair_from_seed(
            seed
        ).to(
            runtime_device
        )

        for seed
        in IIC_CLASSIFIER_SEEDS
    ]


    optimizers = [
        make_iic_optimizer(
            model
        )

        for model
        in models
    ]


    # ========================================================
    # PARAMETER / OPTIMIZER INDEPENDENCE
    # ========================================================

    model_parameter_ptr_sets = [
        {
            parameter.data_ptr()
            for parameter
            in model.parameters()
        }

        for model
        in models
    ]


    ensemble_parameter_independence_gate = all(

        model_parameter_ptr_sets[i].isdisjoint(
            model_parameter_ptr_sets[j]
        )

        for i
        in range(
            len(
                model_parameter_ptr_sets
            )
        )

        for j
        in range(
            i + 1,
            len(
                model_parameter_ptr_sets
            )
        )
    )


    optimizer_ownership_gate = all(

        {
            parameter.data_ptr()

            for group
            in optimizer.param_groups

            for parameter
            in group["params"]
        }
        == model_parameter_ptr_sets[index]

        for index, optimizer
        in enumerate(
            optimizers
        )
    )


    assert ensemble_parameter_independence_gate
    assert optimizer_ownership_gate


    loader_times = []
    encoder_times = []
    k5_train_times = []
    total_times = []


    # ========================================================
    # PROCESS ONE BATCH
    # ========================================================

    def process_batch(
        manifest_indices,
        *,
        record,
    ):

        total_start = time.perf_counter()


        # ====================================================
        # CPU DATA
        # ====================================================

        loader_start = time.perf_counter()


        central_cpu = torch.from_numpy(
            np.asarray(
                full_central_mu[
                    manifest_indices
                ],
                dtype=np.float32,
            ).copy()
        )


        (
            perturbed_images_cpu,
            _,
        ) = make_real_perturbed_batch(
            manifest_indices,
            epoch=0,
            mode=mode_id,
        )


        loader_seconds = (
            time.perf_counter()
            - loader_start
        )


        # ====================================================
        # GPU TRANSFER + FROZEN ENCODER
        # ====================================================

        encoder_start = time.perf_counter()


        central_gpu = central_cpu.to(
            runtime_device
        )


        perturbed_images_gpu = (
            perturbed_images_cpu.to(
                runtime_device
            )
        )


        with torch.no_grad():

            (
                perturbed_mu_gpu,
                _,
            ) = official_iic_encoder(
                perturbed_images_gpu
            )


        perturbed_mu_gpu = (
            perturbed_mu_gpu
            .detach()
            .clone()
            .contiguous()
        )


        torch.cuda.synchronize()


        encoder_seconds = (
            time.perf_counter()
            - encoder_start
        )


        # ====================================================
        # FIVE INDEPENDENT CLASSIFIERS
        # ====================================================

        train_start = time.perf_counter()


        batch_losses = []


        for model, optimizer in zip(
            models,
            optimizers,
        ):

            batch_losses.append(
                train_one_iic_pair(
                    model,
                    optimizer,
                    central_gpu,
                    perturbed_mu_gpu,
                )
            )


        torch.cuda.synchronize()


        k5_train_seconds = (
            time.perf_counter()
            - train_start
        )


        total_seconds = (
            time.perf_counter()
            - total_start
        )


        assert np.isfinite(
            batch_losses
        ).all()


        if record:

            loader_times.append(
                loader_seconds
            )

            encoder_times.append(
                encoder_seconds
            )

            k5_train_times.append(
                k5_train_seconds
            )

            total_times.append(
                total_seconds
            )


    # ========================================================
    # WARM-UP
    # ========================================================

    print(
        f"Warm-up {mode_name}..."
    )


    process_batch(
        benchmark_batches[0],
        record=False,
    )


    torch.cuda.synchronize()


    optimizer_state_gate = all(
        len(
            optimizer.state
        ) == 10

        for optimizer
        in optimizers
    )


    assert optimizer_state_gate


    baseline_allocated = (
        torch.cuda.memory_allocated()
    )

    baseline_reserved = (
        torch.cuda.memory_reserved()
    )


    torch.cuda.reset_peak_memory_stats()


    # ========================================================
    # MEASURE
    # ========================================================

    print(
        f"Benchmarking {mode_name}..."
    )


    for batch_indices in benchmark_batches[
        1:
    ]:

        process_batch(
            batch_indices,
            record=True,
        )


    torch.cuda.synchronize()


    peak_allocated = (
        torch.cuda.max_memory_allocated()
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved()
    )


    result = {

        "mode":
            mode_name,

        "measured_batches":
            len(
                total_times
            ),

        "samples":
            len(
                total_times
            ) * 128,

        "loader_median":
            statistics.median(
                loader_times
            ),

        "encoder_median":
            statistics.median(
                encoder_times
            ),

        "k5_train_median":
            statistics.median(
                k5_train_times
            ),

        "total_median":
            statistics.median(
                total_times
            ),

        "total_mean":
            statistics.mean(
                total_times
            ),

        "throughput_median":
            (
                128.0
                / statistics.median(
                    total_times
                )
            ),

        "baseline_allocated":
            int(
                baseline_allocated
            ),

        "baseline_reserved":
            int(
                baseline_reserved
            ),

        "peak_allocated":
            int(
                peak_allocated
            ),

        "peak_reserved":
            int(
                peak_reserved
            ),

        "peak_extra_allocated":
            int(
                max(
                    0,
                    peak_allocated
                    - baseline_allocated,
                )
            ),

        "ensemble_parameter_independence_gate":
            ensemble_parameter_independence_gate,

        "optimizer_ownership_gate":
            optimizer_ownership_gate,

        "optimizer_state_gate":
            optimizer_state_gate,
    }


    # Conservative projection:
    # 68 full-batch equivalents although the real tail is 7.
    result[
        "projected_epoch_seconds"
    ] = (
        result[
            "total_median"
        ]
        * 68
    )


    result[
        "projected_100_epochs_minutes"
    ] = (
        result[
            "projected_epoch_seconds"
        ]
        * 100
        / 60.0
    )


    del models
    del optimizers

    gc.collect()
    torch.cuda.empty_cache()


    return result


# ============================================================
# RUN BOTH MODES
# ============================================================

legacy_benchmark = benchmark_iic_mode(
    mode_name="legacy_text",
    mode_id=GS_AUGMENT_MODE_LEGACY_TEXT,
)


symmetric_benchmark = benchmark_iic_mode(
    mode_name="symmetric_figure",
    mode_id=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


# ============================================================
# ENCODER MUST STILL BE FROZEN
# ============================================================

encoder_state_after_benchmark = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_benchmark_immutability_gate = (

    encoder_state_before_benchmark
    == encoder_state_after_benchmark
    == ENCODER_FINAL_STATE_SHA256
)


encoder_gradient_gate = all(
    parameter.grad is None

    for parameter
    in official_iic_encoder.parameters()
)


assert encoder_benchmark_immutability_gate
assert encoder_gradient_gate


# ============================================================
# FAIRNESS / NUMERIC GATES
# ============================================================

def benchmark_valid(result):

    return (

        result[
            "measured_batches"
        ] == 8

        and result[
            "samples"
        ] == 1024

        and result[
            "loader_median"
        ] > 0.0

        and result[
            "encoder_median"
        ] > 0.0

        and result[
            "k5_train_median"
        ] > 0.0

        and result[
            "total_median"
        ] > 0.0

        and result[
            "throughput_median"
        ] > 0.0

        and result[
            "peak_allocated"
        ] > 0

        and result[
            "ensemble_parameter_independence_gate"
        ]

        and result[
            "optimizer_ownership_gate"
        ]

        and result[
            "optimizer_state_gate"
        ]
    )


legacy_benchmark_gate = benchmark_valid(
    legacy_benchmark
)

symmetric_benchmark_gate = benchmark_valid(
    symmetric_benchmark
)


benchmark_gate = (

    initial_equivalence_gate
    and shared_feature_equivalence_gate
    and legacy_benchmark_gate
    and symmetric_benchmark_gate
    and encoder_benchmark_immutability_gate
    and encoder_gradient_gate
)


assert benchmark_gate


# ============================================================
# COMBINED PROJECTION
# ============================================================

projected_both_modes_100_epochs_minutes = (

    legacy_benchmark[
        "projected_100_epochs_minutes"
    ]

    + symmetric_benchmark[
        "projected_100_epochs_minutes"
    ]
)


# ============================================================
# REPORT
# ============================================================

def gibibytes(value):

    return (
        value
        / (
            1024.0
            ** 3
        )
    )


print()
print(
    "=== NOTEBOOK 06 FAIR K5 IIC BENCHMARK ==="
)

print()

print(
    "INITIAL_EQUIVALENCE_GATE =",
    initial_equivalence_gate,
)

print(
    "SHARED_FEATURE_EXECUTION_EQUIVALENCE_GATE =",
    shared_feature_equivalence_gate,
)

print(
    "ENCODER_BENCHMARK_IMMUTABILITY_GATE =",
    encoder_benchmark_immutability_gate,
)

print(
    "ENCODER_GRADIENT_GATE =",
    encoder_gradient_gate,
)

print()


for result in [
    legacy_benchmark,
    symmetric_benchmark,
]:

    print(
        "---",
        result["mode"],
        "---",
    )

    print(
        "MEASURED_BATCHES =",
        result[
            "measured_batches"
        ],
    )

    print(
        "MEASURED_SAMPLES =",
        result[
            "samples"
        ],
    )

    print(
        "LOADER_MEDIAN_S =",
        result[
            "loader_median"
        ],
    )

    print(
        "ENCODER_MEDIAN_S =",
        result[
            "encoder_median"
        ],
    )

    print(
        "K5_TRAIN_MEDIAN_S =",
        result[
            "k5_train_median"
        ],
    )

    print(
        "TOTAL_MEDIAN_S =",
        result[
            "total_median"
        ],
    )

    print(
        "THROUGHPUT_MEDIAN_SAMPLE_S =",
        result[
            "throughput_median"
        ],
    )

    print(
        "BASELINE_ALLOCATED_GIB =",
        gibibytes(
            result[
                "baseline_allocated"
            ]
        ),
    )

    print(
        "BASELINE_RESERVED_GIB =",
        gibibytes(
            result[
                "baseline_reserved"
            ]
        ),
    )

    print(
        "PEAK_ALLOCATED_GIB =",
        gibibytes(
            result[
                "peak_allocated"
            ]
        ),
    )

    print(
        "PEAK_RESERVED_GIB =",
        gibibytes(
            result[
                "peak_reserved"
            ]
        ),
    )

    print(
        "PEAK_EXTRA_ALLOCATED_GIB =",
        gibibytes(
            result[
                "peak_extra_allocated"
            ]
        ),
    )

    print(
        "PROJECTED_EPOCH_SECONDS =",
        result[
            "projected_epoch_seconds"
        ],
    )

    print(
        "PROJECTED_100_EPOCHS_MINUTES =",
        result[
            "projected_100_epochs_minutes"
        ],
    )

    print(
        "ENSEMBLE_PARAMETER_INDEPENDENCE_GATE =",
        result[
            "ensemble_parameter_independence_gate"
        ],
    )

    print(
        "OPTIMIZER_OWNERSHIP_GATE =",
        result[
            "optimizer_ownership_gate"
        ],
    )

    print(
        "OPTIMIZER_STATE_GATE =",
        result[
            "optimizer_state_gate"
        ],
    )

    print()


print(
    "PROJECTED_BOTH_MODES_100_EPOCHS_MINUTES =",
    projected_both_modes_100_epochs_minutes,
)

print()

print(
    "LEGACY_BENCHMARK_GATE =",
    legacy_benchmark_gate,
)

print(
    "SYMMETRIC_BENCHMARK_GATE =",
    symmetric_benchmark_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "OFFICIAL_RUN_CREATED =",
    False,
)

print(
    "OFFICIAL_CHECKPOINT_WRITTEN =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5B_FAIR_K5_BENCHMARK_PASS =",
    benchmark_gate,
)

Prewarming PNG file cache...
Warm-up legacy_text...
Benchmarking legacy_text...
Warm-up symmetric_figure...
Benchmarking symmetric_figure...

=== NOTEBOOK 06 FAIR K5 IIC BENCHMARK ===

INITIAL_EQUIVALENCE_GATE = True
SHARED_FEATURE_EXECUTION_EQUIVALENCE_GATE = True
ENCODER_BENCHMARK_IMMUTABILITY_GATE = True
ENCODER_GRADIENT_GATE = True

--- legacy_text ---
MEASURED_BATCHES = 8
MEASURED_SAMPLES = 1024
LOADER_MEDIAN_S = 0.5902575530001286
ENCODER_MEDIAN_S = 0.030343202999802088
K5_TRAIN_MEDIAN_S = 0.042449433999991015
TOTAL_MEDIAN_S = 0.663436268499936
THROUGHPUT_MEDIAN_SAMPLE_S = 192.93488474697756
BASELINE_ALLOCATED_GIB = 0.08367109298706055
BASELINE_RESERVED_GIB = 0.962890625
PEAK_ALLOCATED_GIB = 0.9452433586120605
PEAK_RESERVED_GIB = 0.96484375
PEAK_EXTRA_ALLOCATED_GIB = 0.861572265625
PROJECTED_EPOCH_SECONDS = 45.113666257995646
PROJECTED_100_EPOCHS_MINUTES = 75.18944376332608
ENSEMBLE_PARAMETER_INDEPENDENCE_GATE = True
OPTIMIZER_OWNERSHIP_GATE = True
OPTIMIZER_STATE_GATE = True

--

## STEP 06-A5C — Contratto delle run ufficiali IIC e checkpoint/resume

Le due run IIC pre-registrate sono:

- `sakai224_full_iic_symmetric_v1`, ramo principale;
- `sakai224_full_iic_legacy_text_v1`, fidelity sensitivity.

Per ogni modalità vengono addestrati simultaneamente, ma in modo indipendente:

$$
K=5
$$

classificatori.

Le feature centrali e perturbate dello stesso minibatch possono essere
condivise fra i cinque classificatori perché l'encoder VAE è congelato.

Ogni classificatore mantiene tuttavia:

- propri parametri;
- proprie statistiche BatchNorm;
- proprio optimizer Adam;
- proprio stato optimizer.

### Checkpoint

Ogni run mantiene:

- `latest.pt`;
- `final.pt`;
- `metrics.csv`;
- `run_config.json`;
- `runtime_sessions.jsonl`.

Non viene utilizzato un checkpoint `BEST`.

`latest.pt` viene pubblicato soltanto dopo il completamento integrale di una
epoca.

Il resume avviene quindi esclusivamente da:

$$
E_{\mathrm{next}}
=
E_{\mathrm{last\ completed}}+1.
$$

`final.pt` viene creato soltanto dopo il completamento della centesima epoca.

### Contenuto del checkpoint

Il checkpoint contiene:

- i cinque `state_dict` dei classificatori;
- i cinque stati Adam;
- ultimo epoch completato;
- optimizer step per classificatore;
- modalità di perturbazione;
- seed;
- identità del dataset e del VAE;
- hash dei contratti Notebook 06;
- versione runtime.

### Metriche durante il training

Le metriche registrate sono esclusivamente unsupervised:

- mutual information principale;
- mutual information over-clustering;
- loss totale;
- tempo dell'epoca.

Non vengono utilizzate per scegliere il checkpoint scientifico.

Le label Gravity Spy rimangono escluse.

In [ ]:
# ============================================================
# NOTEBOOK 06 — OFFICIAL IIC RUN / CHECKPOINT CONTRACT
# STEP 06-A5C
#
# CONTRACT ONLY.
# NO training.
# NO official directories created.
# NO Drive writes.
# ============================================================

import hashlib
import json


assert benchmark_gate
assert a4d_gate


# ============================================================
# OFFICIAL RUN IDS
# ============================================================

IIC_OFFICIAL_RUNS = {

    "symmetric_figure": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "role":
            "PRIMARY_PROJECT_FULL_BRANCH",
    },

    "legacy_text": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "role":
            "HTML_TEXT_FIDELITY_SENSITIVITY",
    },
}


# ============================================================
# CHECKPOINT / METRICS CONTRACT
# ============================================================

IIC_OFFICIAL_RUN_CONTRACT = {

    "contract_version": 1,

    "runs":
        IIC_OFFICIAL_RUNS,

    "classifier_count_K":
        5,

    "training": {

        "epochs":
            100,

        "batches_per_epoch":
            68,

        "optimizer_steps_per_classifier":
            6800,

        "scientific_checkpoint":
            "FINAL_epoch100",

        "best_checkpoint":
            False,
    },

    "checkpoint": {

        "latest":
            "after_each_fully_completed_epoch",

        "final":
            "after_epoch_99",

        "resume":
            "last_completed_epoch_plus_1",

        "atomic_publish":
            True,

        "contains": [
            "five_classifier_state_dicts",
            "five_optimizer_state_dicts",
            "last_completed_epoch",
            "optimizer_steps_per_classifier",
            "augmentation_mode",
            "classifier_seeds",
            "shuffle_seed",
            "augmentation_seed",
            "contract_hashes",
            "runtime_identity",
        ],
    },

    "metrics": {

        "labels_used":
            False,

        "fields_per_classifier": [
            "mean_batch_main_mi",
            "mean_batch_overcluster_mi",
            "mean_batch_total_loss",
        ],

        "epoch_fields": [
            "epoch",
            "elapsed_seconds",
            "optimizer_steps_per_classifier",
        ],

        "used_for_checkpoint_selection":
            False,
    },

    "persistent_files": [
        "latest.pt",
        "final.pt",
        "metrics.csv",
        "run_config.json",
        "runtime_sessions.jsonl",
    ],

    "source_identity": {

        "central_mu_sha256":
            FULL_CENTRAL_MU_SHA256,

        "encoder_state_sha256":
            ENCODER_FINAL_STATE_SHA256,

        "loss_contract_sha256":
            IIC_LOSS_CONTRACT_SHA256,

        "architecture_contract_sha256":
            IIC_ARCHITECTURE_CONTRACT_SHA256,

        "training_core_sha256":
            IIC_TRAINING_CORE_SHA256,

        "final_training_sha256":
            IIC_FINAL_TRAINING_SHA256,

        "dual_experiment_sha256":
            IIC_DUAL_EXPERIMENT_SHA256,
    },

    "future_C11": {

        "same_metric_meanings":
            True,

        "same_checkpoint_epoch_semantics":
            True,

        "same_K":
            5,

        "golden_vector_validation_before_training":
            True,
    },

    "labels_used":
        False,
}


IIC_OFFICIAL_RUN_JSON = json.dumps(
    IIC_OFFICIAL_RUN_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_OFFICIAL_RUN_SHA256 = hashlib.sha256(
    IIC_OFFICIAL_RUN_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# GATES
# ============================================================

checkpoint_gate = (

    IIC_OFFICIAL_RUN_CONTRACT[
        "checkpoint"
    ][
        "resume"
    ]
    == "last_completed_epoch_plus_1"

    and IIC_OFFICIAL_RUN_CONTRACT[
        "checkpoint"
    ][
        "atomic_publish"
    ]
    is True

    and IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "best_checkpoint"
    ]
    is False
)


final_selection_gate = (

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "scientific_checkpoint"
    ]
    == "FINAL_epoch100"

    and IIC_OFFICIAL_RUN_CONTRACT[
        "metrics"
    ][
        "used_for_checkpoint_selection"
    ]
    is False
)


dual_run_identity_gate = (
    set(
        IIC_OFFICIAL_RUNS.keys()
    )
    == {
        "symmetric_figure",
        "legacy_text",
    }
)


c11_checkpoint_alignment_gate = (

    IIC_OFFICIAL_RUN_CONTRACT[
        "future_C11"
    ][
        "same_checkpoint_epoch_semantics"
    ]
    is True

    and IIC_OFFICIAL_RUN_CONTRACT[
        "future_C11"
    ][
        "golden_vector_validation_before_training"
    ]
    is True
)


a5c_gate = all([
    checkpoint_gate,
    final_selection_gate,
    dual_run_identity_gate,
    c11_checkpoint_alignment_gate,
])


assert a5c_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 OFFICIAL IIC RUN CONTRACT ==="
)

print()

print(
    "PRIMARY_RUN_ID =",
    IIC_OFFICIAL_RUNS[
        "symmetric_figure"
    ][
        "run_id"
    ],
)

print(
    "FIDELITY_RUN_ID =",
    IIC_OFFICIAL_RUNS[
        "legacy_text"
    ][
        "run_id"
    ],
)

print()

print(
    "CLASSIFIER_COUNT_K = 5"
)

print(
    "EPOCHS = 100"
)

print(
    "STEPS_PER_CLASSIFIER = 6800"
)

print(
    "SCIENTIFIC_CHECKPOINT = FINAL_epoch100"
)

print(
    "BEST_CHECKPOINT_USED = False"
)

print()

print(
    "LATEST_POLICY = after_each_fully_completed_epoch"
)

print(
    "RESUME_POLICY = last_completed_epoch_plus_1"
)

print(
    "ATOMIC_PUBLISH = True"
)

print()

print(
    "CHECKPOINT_GATE =",
    checkpoint_gate,
)

print(
    "FINAL_SELECTION_GATE =",
    final_selection_gate,
)

print(
    "DUAL_RUN_IDENTITY_GATE =",
    dual_run_identity_gate,
)

print(
    "C11_CHECKPOINT_ALIGNMENT_GATE =",
    c11_checkpoint_alignment_gate,
)

print()

print(
    "IIC_OFFICIAL_RUN_SHA256 =",
    IIC_OFFICIAL_RUN_SHA256,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "OFFICIAL_RUN_DIRECTORY_CREATED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5C_OFFICIAL_RUN_CONTRACT_PASS =",
    a5c_gate,
)


=== NOTEBOOK 06 OFFICIAL IIC RUN CONTRACT ===

PRIMARY_RUN_ID = sakai224_full_iic_symmetric_v1
FIDELITY_RUN_ID = sakai224_full_iic_legacy_text_v1

CLASSIFIER_COUNT_K = 5
EPOCHS = 100
STEPS_PER_CLASSIFIER = 6800
SCIENTIFIC_CHECKPOINT = FINAL_epoch100
BEST_CHECKPOINT_USED = False

LATEST_POLICY = after_each_fully_completed_epoch
RESUME_POLICY = last_completed_epoch_plus_1
ATOMIC_PUBLISH = True

CHECKPOINT_GATE = True
FINAL_SELECTION_GATE = True
DUAL_RUN_IDENTITY_GATE = True
C11_CHECKPOINT_ALIGNMENT_GATE = True

IIC_OFFICIAL_RUN_SHA256 = 010aeaa0aedc657c1b1214604e67992aff4fffb9bf5bc0450a63758868ff783a

LABELS_READ = False
OFFICIAL_RUN_DIRECTORY_CREATED = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5C_OFFICIAL_RUN_CONTRACT_PASS = True


## STEP 06-A5D — Checkpoint round-trip e resume deterministico

Prima delle run ufficiali viene verificata end-to-end la semantica di
checkpoint e resume su dati reali.

Il test utilizza entrambe le modalità pre-registrate:

$$
\mathrm{symmetric\_figure}
$$

e

$$
\mathrm{legacy\_text}.
$$

Per ciascuna vengono confrontati due percorsi:

### Percorso continuo

$$
E_0
\rightarrow
E_1.
$$

### Percorso con resume

$$
E_0
\rightarrow
\mathrm{checkpoint}
\rightarrow
\mathrm{restore}
\rightarrow
E_1.
$$

Alla fine devono coincidere esattamente:

- i cinque stati dei classificatori;
- i cinque stati Adam;
- i contatori degli optimizer step;
- le statistiche BatchNorm.

Il checkpoint temporaneo viene scritto esclusivamente nella root runtime locale
tramite pubblicazione atomica `.partial -> final`.

### Transazione di fine epoca

Per le run ufficiali viene congelato l'ordine PROJECT:

1. completamento dell'intera epoca;
2. append della riga in `metrics.csv`;
3. pubblicazione atomica di `latest.pt`.

In caso di interruzione tra i punti 2 e 3, al resume eventuali righe metrics
successive a `last_completed_epoch` vengono eliminate.

In questo modo il checkpoint `latest.pt` rimane l'autorità sullo stato
neurale realmente completato.

Viene inoltre testato esplicitamente l'ultimo minibatch reale:

$$
8583\bmod128=7,
$$

con BatchNorm attiva.

Nessun artefatto ufficiale viene creato.

In [ ]:
# ============================================================
# NOTEBOOK 06 — CHECKPOINT ROUND-TRIP / RESUME TEST
# STEP 06-A5D
#
# Temporary /content artifacts only.
#
# NO labels.
# NO official run.
# NO Drive writes.
# ============================================================

from pathlib import Path

import csv
import gc
import hashlib
import os
import shutil

import numpy as np
import torch


assert a5c_gate
assert benchmark_gate
assert runtime_device.type == "cuda"

assert IIC_OFFICIAL_RUN_SHA256 == (
    "010aeaa0aedc657c1b1214604e67992a"
    "ff4fffb9bf5bc0450a63758868ff783a"
)


# ============================================================
# TEMPORARY TEST DIRECTORY
# ============================================================

A5D_TEMP_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "notebook06_a5d_checkpoint_resume_test"
)


if A5D_TEMP_ROOT.exists():

    shutil.rmtree(
        A5D_TEMP_ROOT
    )


A5D_TEMP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# TREE -> CPU
# ============================================================

def tree_to_cpu(value):

    if torch.is_tensor(value):

        return (
            value
            .detach()
            .cpu()
            .clone()
        )


    if isinstance(value, dict):

        return {
            key:
                tree_to_cpu(item)

            for key, item
            in value.items()
        }


    if isinstance(value, list):

        return [
            tree_to_cpu(item)
            for item
            in value
        ]


    if isinstance(value, tuple):

        return tuple(
            tree_to_cpu(item)
            for item
            in value
        )


    return value


# ============================================================
# ENSEMBLE FACTORY
# ============================================================

def build_k5_ensemble():

    models = [
        build_iic_pair_from_seed(
            seed
        ).to(
            runtime_device
        )

        for seed
        in IIC_CLASSIFIER_SEEDS
    ]


    optimizers = [
        make_iic_optimizer(
            model
        )

        for model
        in models
    ]


    return (
        models,
        optimizers,
    )


# ============================================================
# HASHES
# ============================================================

def ensemble_model_hashes(models):

    return [
        canonical_tree_sha256(
            model.state_dict()
        )

        for model
        in models
    ]


def ensemble_optimizer_hashes(
    optimizers,
):

    return [
        canonical_tree_sha256(
            tree_to_cpu(
                optimizer.state_dict()
            )
        )

        for optimizer
        in optimizers
    ]


# ============================================================
# TEMPORARY EPOCH HARNESS
#
# Uses the real PROJECT epoch shuffle and augmentation,
# but only the first two full batches to keep this a test.
# ============================================================

A5D_BATCHES_PER_TEMP_EPOCH = 2


def run_a5d_temp_epoch(
    *,
    models,
    optimizers,
    optimizer_steps,
    epoch,
    mode_id,
):

    permutation = (
        vae_c_shuffle_make_permutation_py(
            8583,
            101112,
            epoch,
        )
    )


    epoch_losses = []


    for batch_index in range(
        A5D_BATCHES_PER_TEMP_EPOCH
    ):

        start = (
            batch_index
            * 128
        )

        stop = (
            start
            + 128
        )


        manifest_indices = permutation[
            start:stop
        ]


        assert len(
            manifest_indices
        ) == 128


        # ----------------------------------------------------
        # CENTRAL mu
        # ----------------------------------------------------

        central_gpu = torch.from_numpy(
            np.asarray(
                full_central_mu[
                    manifest_indices
                ],
                dtype=np.float32,
            ).copy()
        ).to(
            runtime_device
        )


        # ----------------------------------------------------
        # PERTURBED PNG -> frozen encoder -> mu'
        # ----------------------------------------------------

        (
            images_cpu,
            _,
        ) = make_real_perturbed_batch(
            manifest_indices,
            epoch=epoch,
            mode=mode_id,
        )


        with torch.no_grad():

            (
                perturbed_gpu,
                _,
            ) = official_iic_encoder(
                images_cpu.to(
                    runtime_device,
                    dtype=torch.float32,
                )
            )


        perturbed_gpu = (
            perturbed_gpu
            .detach()
            .clone()
            .contiguous()
        )


        # ----------------------------------------------------
        # K=5 independent updates
        # ----------------------------------------------------

        batch_losses = []


        for classifier_index, (
            model,
            optimizer,
        ) in enumerate(
            zip(
                models,
                optimizers,
            )
        ):

            loss = train_one_iic_pair(
                model,
                optimizer,
                central_gpu,
                perturbed_gpu,
            )


            assert np.isfinite(
                loss
            )


            optimizer_steps[
                classifier_index
            ] += 1


            batch_losses.append(
                loss
            )


        epoch_losses.append(
            batch_losses
        )


    torch.cuda.synchronize()


    return np.asarray(
        epoch_losses,
        dtype=np.float64,
    )


# ============================================================
# ATOMIC CHECKPOINT
# ============================================================

def atomic_torch_save(
    payload,
    final_path,
):

    final_path = Path(
        final_path
    )

    partial_path = Path(
        str(
            final_path
        )
        + ".partial"
    )


    if partial_path.exists():

        partial_path.unlink()


    torch.save(
        payload,
        partial_path,
    )


    assert partial_path.is_file()
    assert partial_path.stat().st_size > 0


    os.replace(
        partial_path,
        final_path,
    )


    assert final_path.is_file()
    assert not partial_path.exists()


# ============================================================
# BUILD CHECKPOINT PAYLOAD
# ============================================================

def make_a5d_checkpoint(
    *,
    mode_name,
    mode_id,
    last_completed_epoch,
    models,
    optimizers,
    optimizer_steps,
):

    classifier_states = [
        tree_to_cpu(
            model.state_dict()
        )

        for model
        in models
    ]


    optimizer_states = [
        tree_to_cpu(
            optimizer.state_dict()
        )

        for optimizer
        in optimizers
    ]


    payload = {

        "format_version":
            1,

        "kind":
            "TEMP_LATEST",

        "run_id":
            (
                "a5d_temp_"
                + mode_name
            ),

        "augmentation_mode":
            mode_name,

        "augmentation_mode_id":
            int(
                mode_id
            ),

        "last_completed_epoch":
            int(
                last_completed_epoch
            ),

        "next_epoch":
            int(
                last_completed_epoch
                + 1
            ),

        "classifier_seeds":
            list(
                IIC_CLASSIFIER_SEEDS
            ),

        "optimizer_steps_per_classifier":
            list(
                optimizer_steps
            ),

        "classifier_state_dicts":
            classifier_states,

        "optimizer_state_dicts":
            optimizer_states,

        "contract_hashes": {

            "loss":
                IIC_LOSS_CONTRACT_SHA256,

            "architecture":
                IIC_ARCHITECTURE_CONTRACT_SHA256,

            "training_core":
                IIC_TRAINING_CORE_SHA256,

            "final_training":
                IIC_FINAL_TRAINING_SHA256,

            "dual_experiment":
                IIC_DUAL_EXPERIMENT_SHA256,

            "official_run":
                IIC_OFFICIAL_RUN_SHA256,
        },

        "labels_used":
            False,
    }


    payload[
        "classifier_states_sha256"
    ] = canonical_tree_sha256(
        classifier_states
    )


    payload[
        "optimizer_states_sha256"
    ] = canonical_tree_sha256(
        optimizer_states
    )


    return payload


# ============================================================
# RESTORE
# ============================================================

def move_optimizer_state_to_device(
    optimizer,
):

    for state in optimizer.state.values():

        for key, value in list(
            state.items()
        ):

            if torch.is_tensor(
                value
            ):

                state[key] = value.to(
                    runtime_device
                )


def restore_a5d_checkpoint(
    checkpoint_path,
    *,
    expected_mode_name,
):

    payload = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )


    identity_gate = (

        payload[
            "kind"
        ] == "TEMP_LATEST"

        and payload[
            "augmentation_mode"
        ] == expected_mode_name

        and payload[
            "classifier_seeds"
        ] == IIC_CLASSIFIER_SEEDS

        and payload[
            "labels_used"
        ] is False

        and payload[
            "contract_hashes"
        ][
            "official_run"
        ] == IIC_OFFICIAL_RUN_SHA256
    )


    assert identity_gate


    checkpoint_hash_gate = (

        canonical_tree_sha256(
            payload[
                "classifier_state_dicts"
            ]
        )
        == payload[
            "classifier_states_sha256"
        ]

        and canonical_tree_sha256(
            payload[
                "optimizer_state_dicts"
            ]
        )
        == payload[
            "optimizer_states_sha256"
        ]
    )


    assert checkpoint_hash_gate


    (
        models,
        optimizers,
    ) = build_k5_ensemble()


    for index in range(5):

        load_result = models[
            index
        ].load_state_dict(
            payload[
                "classifier_state_dicts"
            ][
                index
            ],
            strict=True,
        )


        assert (
            load_result.missing_keys == []
            and load_result.unexpected_keys == []
        )


        optimizers[
            index
        ].load_state_dict(
            payload[
                "optimizer_state_dicts"
            ][
                index
            ]
        )


        move_optimizer_state_to_device(
            optimizers[
                index
            ]
        )


    optimizer_restore_gate = all(

        len(
            optimizer.state
        ) == 10

        for optimizer
        in optimizers
    )


    assert optimizer_restore_gate


    restored_model_hash_gate = (

        ensemble_model_hashes(
            models
        )
        == [
            canonical_tree_sha256(
                state
            )

            for state
            in payload[
                "classifier_state_dicts"
            ]
        ]
    )


    restored_optimizer_hash_gate = (

        ensemble_optimizer_hashes(
            optimizers
        )
        == [
            canonical_tree_sha256(
                state
            )

            for state
            in payload[
                "optimizer_state_dicts"
            ]
        ]
    )


    assert restored_model_hash_gate
    assert restored_optimizer_hash_gate


    return (
        payload,
        models,
        optimizers,
        identity_gate,
        checkpoint_hash_gate,
        optimizer_restore_gate,
    )


# ============================================================
# METRICS RECONCILIATION TEST
# ============================================================

def reconcile_metrics_to_checkpoint(
    path,
    last_completed_epoch,
):

    path = Path(
        path
    )


    with path.open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        fieldnames = list(
            reader.fieldnames
        )

        rows = list(
            reader
        )


    kept_rows = [
        row
        for row
        in rows
        if int(
            row[
                "epoch"
            ]
        )
        <= last_completed_epoch
    ]


    temp_path = Path(
        str(
            path
        )
        + ".partial"
    )


    with temp_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(
            kept_rows
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )


    os.replace(
        temp_path,
        path,
    )


    return kept_rows


metrics_test_path = (
    A5D_TEMP_ROOT
    / "metrics_reconciliation.csv"
)


with metrics_test_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:

    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "epoch",
            "dummy_loss",
        ],
    )

    writer.writeheader()

    writer.writerow({
        "epoch":
            0,
        "dummy_loss":
            -0.10,
    })

    # Simulates a metrics row written just before a crash,
    # while latest.pt still says epoch 0.
    writer.writerow({
        "epoch":
            1,
        "dummy_loss":
            -0.20,
    })


reconciled_rows = (
    reconcile_metrics_to_checkpoint(
        metrics_test_path,
        last_completed_epoch=0,
    )
)


metrics_reconciliation_gate = (

    len(
        reconciled_rows
    ) == 1

    and int(
        reconciled_rows[
            0
        ][
            "epoch"
        ]
    ) == 0

    and not Path(
        str(
            metrics_test_path
        )
        + ".partial"
    ).exists()
)


assert metrics_reconciliation_gate


# ============================================================
# TAIL-BATCH TEST — REAL SIZE 7
# ============================================================

tail_permutation = (
    vae_c_shuffle_make_permutation_py(
        8583,
        101112,
        0,
    )
)


tail_indices = tail_permutation[
    -7:
]


assert len(
    tail_indices
) == 7


tail_central_gpu = torch.from_numpy(
    np.asarray(
        full_central_mu[
            tail_indices
        ],
        dtype=np.float32,
    ).copy()
).to(
    runtime_device
)


(
    tail_images_cpu,
    _,
) = make_real_perturbed_batch(
    tail_indices,
    epoch=0,
    mode=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


with torch.no_grad():

    (
        tail_perturbed_gpu,
        _,
    ) = official_iic_encoder(
        tail_images_cpu.to(
            runtime_device
        )
    )


tail_perturbed_gpu = (
    tail_perturbed_gpu
    .detach()
    .clone()
    .contiguous()
)


tail_model = (
    build_iic_pair_from_seed(
        123
    ).to(
        runtime_device
    )
)

tail_optimizer = (
    make_iic_optimizer(
        tail_model
    )
)


tail_state_before = (
    module_state_sha256(
        tail_model
    )
)


tail_loss = train_one_iic_pair(
    tail_model,
    tail_optimizer,
    tail_central_gpu,
    tail_perturbed_gpu,
)


tail_state_after = (
    module_state_sha256(
        tail_model
    )
)


tail_batch_gate = (

    np.isfinite(
        tail_loss
    )

    and tail_state_before
        != tail_state_after

    and int(
        tail_model
        .main_head
        .bn
        .num_batches_tracked
        .item()
    ) == 2

    and int(
        tail_model
        .overcluster_head
        .bn
        .num_batches_tracked
        .item()
    ) == 2

    and len(
        tail_optimizer.state
    ) == 10
)


assert tail_batch_gate


del tail_model
del tail_optimizer
del tail_central_gpu
del tail_perturbed_gpu
del tail_images_cpu

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# CHECKPOINT / RESUME TEST FOR ONE MODE
# ============================================================

def test_checkpoint_resume_mode(
    *,
    mode_name,
    mode_id,
):

    mode_dir = (
        A5D_TEMP_ROOT
        / mode_name
    )


    mode_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    checkpoint_path = (
        mode_dir
        / "latest.pt"
    )


    # ========================================================
    # A) UNINTERRUPTED: epoch 0 -> epoch 1
    # ========================================================

    (
        continuous_models,
        continuous_optimizers,
    ) = build_k5_ensemble()


    continuous_steps = [
        0,
        0,
        0,
        0,
        0,
    ]


    continuous_epoch0_losses = (
        run_a5d_temp_epoch(
            models=continuous_models,
            optimizers=continuous_optimizers,
            optimizer_steps=continuous_steps,
            epoch=0,
            mode_id=mode_id,
        )
    )


    continuous_epoch1_losses = (
        run_a5d_temp_epoch(
            models=continuous_models,
            optimizers=continuous_optimizers,
            optimizer_steps=continuous_steps,
            epoch=1,
            mode_id=mode_id,
        )
    )


    continuous_model_hashes = (
        ensemble_model_hashes(
            continuous_models
        )
    )


    continuous_optimizer_hashes = (
        ensemble_optimizer_hashes(
            continuous_optimizers
        )
    )


    assert continuous_steps == [
        4,
        4,
        4,
        4,
        4,
    ]


    # ========================================================
    # B) RESUME PATH: epoch0 -> save -> load -> epoch1
    # ========================================================

    (
        pre_models,
        pre_optimizers,
    ) = build_k5_ensemble()


    pre_steps = [
        0,
        0,
        0,
        0,
        0,
    ]


    resumed_epoch0_losses = (
        run_a5d_temp_epoch(
            models=pre_models,
            optimizers=pre_optimizers,
            optimizer_steps=pre_steps,
            epoch=0,
            mode_id=mode_id,
        )
    )


    epoch0_loss_gate = np.array_equal(
        continuous_epoch0_losses,
        resumed_epoch0_losses,
    )


    assert epoch0_loss_gate
    assert pre_steps == [
        2,
        2,
        2,
        2,
        2,
    ]


    checkpoint_payload = (
        make_a5d_checkpoint(
            mode_name=mode_name,
            mode_id=mode_id,
            last_completed_epoch=0,
            models=pre_models,
            optimizers=pre_optimizers,
            optimizer_steps=pre_steps,
        )
    )


    atomic_torch_save(
        checkpoint_payload,
        checkpoint_path,
    )


    checkpoint_file_gate = (

        checkpoint_path.is_file()

        and checkpoint_path.stat().st_size
            > 0

        and not Path(
            str(
                checkpoint_path
            )
            + ".partial"
        ).exists()
    )


    assert checkpoint_file_gate


    checkpoint_file_sha256 = (
        sha256_stream(
            checkpoint_path
        )
    )


    # Destroy pre-resume state.
    del pre_models
    del pre_optimizers
    del checkpoint_payload

    gc.collect()
    torch.cuda.empty_cache()


    (
        restored_payload,
        resumed_models,
        resumed_optimizers,
        identity_gate,
        checkpoint_hash_gate,
        optimizer_restore_gate,
    ) = restore_a5d_checkpoint(
        checkpoint_path,
        expected_mode_name=mode_name,
    )


    resume_boundary_gate = (

        restored_payload[
            "last_completed_epoch"
        ] == 0

        and restored_payload[
            "next_epoch"
        ] == 1

        and restored_payload[
            "optimizer_steps_per_classifier"
        ] == [
            2,
            2,
            2,
            2,
            2,
        ]
    )


    assert resume_boundary_gate


    resumed_steps = list(
        restored_payload[
            "optimizer_steps_per_classifier"
        ]
    )


    resumed_epoch1_losses = (
        run_a5d_temp_epoch(
            models=resumed_models,
            optimizers=resumed_optimizers,
            optimizer_steps=resumed_steps,
            epoch=1,
            mode_id=mode_id,
        )
    )


    resumed_model_hashes = (
        ensemble_model_hashes(
            resumed_models
        )
    )


    resumed_optimizer_hashes = (
        ensemble_optimizer_hashes(
            resumed_optimizers
        )
    )


    # ========================================================
    # EXACT CONTINUOUS vs RESUMED COMPARISON
    # ========================================================

    epoch1_loss_gate = np.array_equal(
        continuous_epoch1_losses,
        resumed_epoch1_losses,
    )


    model_resume_exact_gate = (
        continuous_model_hashes
        == resumed_model_hashes
    )


    optimizer_resume_exact_gate = (
        continuous_optimizer_hashes
        == resumed_optimizer_hashes
    )


    optimizer_step_resume_gate = (

        continuous_steps
        == resumed_steps
        == [
            4,
            4,
            4,
            4,
            4,
        ]
    )


    # Two batches / epoch, two BN forwards / batch,
    # two temporary epochs -> 8 updates per BN.
    bn_resume_gate = all(

        int(
            model
            .main_head
            .bn
            .num_batches_tracked
            .item()
        ) == 8

        and int(
            model
            .overcluster_head
            .bn
            .num_batches_tracked
            .item()
        ) == 8

        for model
        in resumed_models
    )


    mode_gate = all([
        identity_gate,
        checkpoint_hash_gate,
        optimizer_restore_gate,
        resume_boundary_gate,
        checkpoint_file_gate,
        epoch0_loss_gate,
        epoch1_loss_gate,
        model_resume_exact_gate,
        optimizer_resume_exact_gate,
        optimizer_step_resume_gate,
        bn_resume_gate,
    ])


    assert mode_gate


    result = {

        "mode":
            mode_name,

        "checkpoint_file_sha256":
            checkpoint_file_sha256,

        "checkpoint_bytes":
            checkpoint_path.stat().st_size,

        "epoch0_loss_gate":
            epoch0_loss_gate,

        "epoch1_loss_gate":
            epoch1_loss_gate,

        "model_resume_exact_gate":
            model_resume_exact_gate,

        "optimizer_resume_exact_gate":
            optimizer_resume_exact_gate,

        "optimizer_step_resume_gate":
            optimizer_step_resume_gate,

        "bn_resume_gate":
            bn_resume_gate,

        "checkpoint_hash_gate":
            checkpoint_hash_gate,

        "mode_gate":
            mode_gate,
    }


    del continuous_models
    del continuous_optimizers
    del resumed_models
    del resumed_optimizers
    del restored_payload

    gc.collect()
    torch.cuda.empty_cache()


    return result


# ============================================================
# RUN BOTH PRE-REGISTERED MODES
# ============================================================

print(
    "Testing symmetric_figure checkpoint/resume..."
)

a5d_symmetric = (
    test_checkpoint_resume_mode(
        mode_name="symmetric_figure",
        mode_id=GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )
)


print(
    "Testing legacy_text checkpoint/resume..."
)

a5d_legacy = (
    test_checkpoint_resume_mode(
        mode_name="legacy_text",
        mode_id=GS_AUGMENT_MODE_LEGACY_TEXT,
    )
)


# ============================================================
# ENCODER MUST STILL BE IMMUTABLE
# ============================================================

encoder_after_a5d = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_a5d_gate = (
    encoder_after_a5d
    == ENCODER_FINAL_STATE_SHA256
)


encoder_gradient_a5d_gate = all(
    parameter.grad is None

    for parameter
    in official_iic_encoder.parameters()
)


assert encoder_a5d_gate
assert encoder_gradient_a5d_gate


# ============================================================
# FREEZE CHECKPOINT / RESUME CONTRACT
# ============================================================

IIC_CHECKPOINT_RESUME_CONTRACT = {

    "contract_version":
        1,

    "epoch_transaction_order": [
        "complete_epoch",
        "append_metrics_row",
        "atomic_publish_latest",
    ],

    "resume": {

        "authority":
            "latest.pt",

        "start_epoch":
            "last_completed_epoch_plus_1",

        "metrics_reconciliation":
            "truncate_rows_after_last_completed_epoch",
    },

    "checkpoint": {

        "atomic_partial_then_replace":
            True,

        "five_classifier_states":
            True,

        "five_optimizer_states":
            True,

        "optimizer_steps_per_classifier":
            True,

        "contract_hashes":
            True,
    },

    "validated_modes": [
        "symmetric_figure",
        "legacy_text",
    ],

    "tail_batch_size_validated":
        7,

    "exact_resume_required":
        True,

    "labels_used":
        False,

    "parent_official_run_sha256":
        IIC_OFFICIAL_RUN_SHA256,
}


IIC_CHECKPOINT_RESUME_JSON = json.dumps(
    IIC_CHECKPOINT_RESUME_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_CHECKPOINT_RESUME_SHA256 = (
    hashlib.sha256(
        IIC_CHECKPOINT_RESUME_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# FINAL GATE
# ============================================================

a5d_gate = all([

    metrics_reconciliation_gate,

    tail_batch_gate,

    a5d_symmetric[
        "mode_gate"
    ],

    a5d_legacy[
        "mode_gate"
    ],

    encoder_a5d_gate,

    encoder_gradient_a5d_gate,
])


assert a5d_gate


# ============================================================
# CLEANUP TEMPORARY ARTIFACTS
# ============================================================

shutil.rmtree(
    A5D_TEMP_ROOT
)


temporary_cleanup_gate = (
    not A5D_TEMP_ROOT.exists()
)


assert temporary_cleanup_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 CHECKPOINT / RESUME AUDIT ==="
)

print()

print(
    "METRICS_RECONCILIATION_GATE =",
    metrics_reconciliation_gate,
)

print(
    "TAIL_BATCH_SIZE =",
    len(
        tail_indices
    ),
)

print(
    "TAIL_BATCH_GATE =",
    tail_batch_gate,
)

print()


for result in [
    a5d_symmetric,
    a5d_legacy,
]:

    print(
        "---",
        result[
            "mode"
        ],
        "---",
    )

    print(
        "CHECKPOINT_BYTES =",
        result[
            "checkpoint_bytes"
        ],
    )

    print(
        "CHECKPOINT_FILE_SHA256 =",
        result[
            "checkpoint_file_sha256"
        ],
    )

    print(
        "EPOCH0_LOSS_EXACT_GATE =",
        result[
            "epoch0_loss_gate"
        ],
    )

    print(
        "EPOCH1_LOSS_EXACT_GATE =",
        result[
            "epoch1_loss_gate"
        ],
    )

    print(
        "MODEL_RESUME_EXACT_GATE =",
        result[
            "model_resume_exact_gate"
        ],
    )

    print(
        "OPTIMIZER_RESUME_EXACT_GATE =",
        result[
            "optimizer_resume_exact_gate"
        ],
    )

    print(
        "OPTIMIZER_STEP_RESUME_GATE =",
        result[
            "optimizer_step_resume_gate"
        ],
    )

    print(
        "BN_RESUME_GATE =",
        result[
            "bn_resume_gate"
        ],
    )

    print(
        "CHECKPOINT_HASH_GATE =",
        result[
            "checkpoint_hash_gate"
        ],
    )

    print()


print(
    "ENCODER_A5D_GATE =",
    encoder_a5d_gate,
)

print(
    "ENCODER_GRADIENT_A5D_GATE =",
    encoder_gradient_a5d_gate,
)

print(
    "TEMPORARY_CLEANUP_GATE =",
    temporary_cleanup_gate,
)

print()

print(
    "IIC_CHECKPOINT_RESUME_SHA256 =",
    IIC_CHECKPOINT_RESUME_SHA256,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "OFFICIAL_RUN_CREATED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5D_CHECKPOINT_RESUME_PASS =",
    a5d_gate,
)

Testing symmetric_figure checkpoint/resume...
Testing legacy_text checkpoint/resume...

=== NOTEBOOK 06 CHECKPOINT / RESUME AUDIT ===

METRICS_RECONCILIATION_GATE = True
TAIL_BATCH_SIZE = 7
TAIL_BATCH_GATE = True

--- symmetric_figure ---
CHECKPOINT_BYTES = 40491019
CHECKPOINT_FILE_SHA256 = 6983198bca943bcbf6bc131095892bc8db8622a9d948ca346285069f0cc20bc0
EPOCH0_LOSS_EXACT_GATE = True
EPOCH1_LOSS_EXACT_GATE = True
MODEL_RESUME_EXACT_GATE = True
OPTIMIZER_RESUME_EXACT_GATE = True
OPTIMIZER_STEP_RESUME_GATE = True
BN_RESUME_GATE = True
CHECKPOINT_HASH_GATE = True

--- legacy_text ---
CHECKPOINT_BYTES = 40490955
CHECKPOINT_FILE_SHA256 = 4c610cbfaa5a7c3b09f677fbb39c07bd66e610eb148528cb3dd8a610cf61a8f3
EPOCH0_LOSS_EXACT_GATE = True
EPOCH1_LOSS_EXACT_GATE = True
MODEL_RESUME_EXACT_GATE = True
OPTIMIZER_RESUME_EXACT_GATE = True
OPTIMIZER_STEP_RESUME_GATE = True
BN_RESUME_GATE = True
CHECKPOINT_HASH_GATE = True

ENCODER_A5D_GATE = True
ENCODER_GRADIENT_A5D_GATE = True
TEMPORARY_CLEANUP_GATE = T

## STEP 06-A5E — Preflight finale della run IIC primaria

La run primaria pre-registrata è:

`sakai224_full_iic_symmetric_v1`.

Essa utilizza:

$$
D=8583,
\qquad
C=36,
\qquad
W=250,
\qquad
K=5,
$$

$$
B=128,
\qquad
E=100,
$$

con Adam:

$$
\eta=5\times10^{-4},
$$

$$
(\beta_1,\beta_2)=(0.9,0.999),
$$

$$
\epsilon=10^{-8}.
$$

La perturbazione primaria è:

`symmetric_figure`

con:

$$
\mathrm{crop\_left}\in[0,48].
$$

La fidelity sensitivity `legacy_text` resta congelata e verrà eseguita
successivamente senza modificare il protocollo sulla base dei risultati
della run primaria.

Il preflight verifica:

- identità di tutti i contratti Notebook 06;
- identità del VAE `FINAL`;
- identità delle feature centrali;
- disponibilità delle PNG perturbabili;
- geometria batch completa;
- cinque seed indipendenti;
- checkpoint/resume già validato;
- assenza preventiva della directory ufficiale;
- stima temporale coerente con il benchmark.

Nessuna label viene letta e nessun artefatto ufficiale viene creato.

In [ ]:
# ============================================================
# NOTEBOOK 06 — FINAL PRIMARY IIC RUN PREFLIGHT
# STEP 06-A5E
#
# READ ONLY.
# NO official directories.
# NO training.
# NO labels.
# ============================================================

from pathlib import Path

import hashlib
import json
import math

import numpy as np
import torch


# ============================================================
# CANONICAL CONTRACT HASHES
# ============================================================

EXPECTED_HASHES = {

    "loss":
        (
            "2bf9cba7d23de553cbad1d876b1f1b14"
            "3ede63154ac20bbb29ca092223c5e7ac"
        ),

    "architecture":
        (
            "12fd00c3dddc6474249a86899b14797c"
            "c891e815126cfb6989eb96e489d5e758"
        ),

    "training_core":
        (
            "f1df0a78746739c7552668e0ed0cac29"
            "5b8d61d772088cb7893a85d6ae6bfe0a"
        ),

    "final_training":
        (
            "4f27a9bbf4290dff54767655d2025928"
            "c2eac751607d217925b0d5bba2b2f537"
        ),

    "dual_experiment":
        (
            "4fbae488ccbe6226c4c589c8bfcd712e"
            "d20e0ddf65ec95e4dfb80120d8432fbc"
        ),

    "official_run":
        (
            "010aeaa0aedc657c1b1214604e67992a"
            "ff4fffb9bf5bc0450a63758868ff783a"
        ),

    "checkpoint_resume":
        (
            "a2dc5e9135b69486494f6045b40cb55"
            "d96843cf6819cbce20738a098b66f7a39"
        ),
}


contract_hash_gate = (

    IIC_LOSS_CONTRACT_SHA256
        == EXPECTED_HASHES["loss"]

    and IIC_ARCHITECTURE_CONTRACT_SHA256
        == EXPECTED_HASHES["architecture"]

    and IIC_TRAINING_CORE_SHA256
        == EXPECTED_HASHES["training_core"]

    and IIC_FINAL_TRAINING_SHA256
        == EXPECTED_HASHES["final_training"]

    and IIC_DUAL_EXPERIMENT_SHA256
        == EXPECTED_HASHES["dual_experiment"]

    and IIC_OFFICIAL_RUN_SHA256
        == EXPECTED_HASHES["official_run"]

    and IIC_CHECKPOINT_RESUME_SHA256
        == EXPECTED_HASHES["checkpoint_resume"]
)


assert contract_hash_gate


# ============================================================
# PRIMARY RUN IDENTITY
# ============================================================

PRIMARY_MODE_NAME = (
    "symmetric_figure"
)

PRIMARY_MODE_ID = (
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE
)

PRIMARY_RUN_ID = (
    "sakai224_full_iic_symmetric_v1"
)


PRIMARY_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / PRIMARY_RUN_ID
)


primary_identity_gate = (

    IIC_EXPERIMENTS[
        PRIMARY_MODE_NAME
    ][
        "run_id"
    ]
    == PRIMARY_RUN_ID

    and IIC_EXPERIMENTS[
        PRIMARY_MODE_NAME
    ][
        "role"
    ]
    == "PRIMARY_PROJECT_FULL_BRANCH"

    and PRIMARY_MODE_ID
    == GS_AUGMENT_MODE_SYMMETRIC_FIGURE
)


assert primary_identity_gate


# ============================================================
# REFUSE ACCIDENTAL OVERWRITE
# ============================================================

official_directory_absent_gate = (
    not PRIMARY_RUN_DIR.exists()
)


assert official_directory_absent_gate, (
    f"STOP: official run directory already exists: "
    f"{PRIMARY_RUN_DIR}"
)


# ============================================================
# DATA / ENCODER IDENTITIES
# ============================================================

EXPECTED_FULL_CENTRAL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)

EXPECTED_ENCODER_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b727"
    "f0da957744027f150e2af10d3a315dfd"
)

EXPECTED_FINAL_CHECKPOINT_SHA256 = (
    "2bb6fd2ce2f73f8f9e479ed6969a52c1"
    "cdd41c1ae6e2cfc33da687ebba798f71"
)


data_identity_gate = (

    FULL_CENTRAL_MU_SHA256
    == EXPECTED_FULL_CENTRAL_MU_SHA256

    and full_central_mu.shape
    == (
        8583,
        512,
    )

    and np.isfinite(
        full_central_mu
    ).all()

    and out272_ready

    and all_png_paths_exist
)


encoder_identity_gate = (

    ENCODER_FINAL_STATE_SHA256
    == EXPECTED_ENCODER_STATE_SHA256

    and canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == EXPECTED_ENCODER_STATE_SHA256

    and sha256_stream(
        FINAL_CHECKPOINT_PATH
    )
    == EXPECTED_FINAL_CHECKPOINT_SHA256

    and official_iic_encoder.training
    is False

    and all(
        parameter.requires_grad
        is False
        for parameter
        in official_iic_encoder.parameters()
    )
)


assert data_identity_gate
assert encoder_identity_gate


# ============================================================
# FULL BATCH GEOMETRY
# ============================================================

EVENT_COUNT = 8583
BATCH_SIZE = 128
EPOCHS = 100
K = 5

BATCHES_PER_EPOCH = math.ceil(
    EVENT_COUNT / BATCH_SIZE
)

TAIL_BATCH_SIZE = (
    EVENT_COUNT % BATCH_SIZE
)

STEPS_PER_CLASSIFIER = (
    BATCHES_PER_EPOCH
    * EPOCHS
)


batch_geometry_gate = (

    BATCHES_PER_EPOCH == 68

    and TAIL_BATCH_SIZE == 7

    and STEPS_PER_CLASSIFIER == 6800
)


assert batch_geometry_gate


# ============================================================
# SEEDS / FIRST + LAST EPOCH SHUFFLES
# ============================================================

EXPECTED_CLASSIFIER_SEEDS = [
    123,
    124,
    125,
    126,
    127,
]


epoch0_permutation = (
    vae_c_shuffle_make_permutation_py(
        EVENT_COUNT,
        101112,
        0,
    )
)


epoch99_permutation = (
    vae_c_shuffle_make_permutation_py(
        EVENT_COUNT,
        101112,
        99,
    )
)


shuffle_epoch_gate = (

    IIC_CLASSIFIER_SEEDS
    == EXPECTED_CLASSIFIER_SEEDS

    and len(
        epoch0_permutation
    ) == EVENT_COUNT

    and len(
        epoch99_permutation
    ) == EVENT_COUNT

    and len(
        set(
            epoch0_permutation
        )
    ) == EVENT_COUNT

    and len(
        set(
            epoch99_permutation
        )
    ) == EVENT_COUNT

    and epoch0_permutation
        != epoch99_permutation
)


assert shuffle_epoch_gate


# ============================================================
# AUGMENTATION FIRST + LAST EPOCH
# ============================================================

epoch0_crop_map = np.asarray(
    [
        gs_augment_crop_left_for_event_mode(
            789,
            0,
            manifest_index,
            PRIMARY_MODE_ID,
        )

        for manifest_index
        in range(
            EVENT_COUNT
        )
    ],
    dtype=np.int16,
)


epoch99_crop_map = np.asarray(
    [
        gs_augment_crop_left_for_event_mode(
            789,
            99,
            manifest_index,
            PRIMARY_MODE_ID,
        )

        for manifest_index
        in range(
            EVENT_COUNT
        )
    ],
    dtype=np.int16,
)


augmentation_epoch_gate = (

    int(
        epoch0_crop_map.min()
    ) == 0

    and int(
        epoch0_crop_map.max()
    ) == 48

    and int(
        epoch99_crop_map.min()
    ) == 0

    and int(
        epoch99_crop_map.max()
    ) == 48

    and not np.array_equal(
        epoch0_crop_map,
        epoch99_crop_map,
    )
)


assert augmentation_epoch_gate


# ============================================================
# BUILD EXACT OFFICIAL RUN CONFIG
# ============================================================

PRIMARY_RUN_CONFIG = {

    "format_version":
        1,

    "run_id":
        PRIMARY_RUN_ID,

    "role":
        "PRIMARY_PROJECT_FULL_BRANCH",

    "augmentation_mode":
        PRIMARY_MODE_NAME,

    "augmentation_mode_id":
        int(
            PRIMARY_MODE_ID
        ),

    "population": {

        "event_count":
            EVENT_COUNT,

        "mode":
            "full_8583_transductive",
    },

    "encoder": {

        "checkpoint":
            "sakai224_full_v1/final.pt",

        "epoch":
            100,

        "feature":
            "mu",

        "latent_dim":
            512,

        "state_sha256":
            ENCODER_FINAL_STATE_SHA256,

        "frozen":
            True,
    },

    "iic": {

        "C":
            36,

        "W":
            250,

        "K":
            K,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "drop_last":
            False,

        "loss_combination":
            "simultaneous_equal_weight_sum",

        "main_weight":
            1.0,

        "overcluster_weight":
            1.0,
    },

    "optimizer": {

        "type":
            "Adam",

        "learning_rate":
            5.0e-4,

        "betas":
            [
                0.9,
                0.999,
            ],

        "epsilon":
            1.0e-8,

        "weight_decay":
            0.0,

        "amsgrad":
            False,

        "foreach":
            False,

        "fused":
            False,
    },

    "rng": {

        "classifier_seeds":
            EXPECTED_CLASSIFIER_SEEDS,

        "shuffle_seed":
            101112,

        "augmentation_seed":
            789,
    },

    "checkpoint": {

        "latest_after_each_completed_epoch":
            True,

        "final_after_epoch_99":
            True,

        "best":
            False,

        "resume":
            "last_completed_epoch_plus_1",
    },

    "contracts": {

        "loss":
            IIC_LOSS_CONTRACT_SHA256,

        "architecture":
            IIC_ARCHITECTURE_CONTRACT_SHA256,

        "training_core":
            IIC_TRAINING_CORE_SHA256,

        "final_training":
            IIC_FINAL_TRAINING_SHA256,

        "dual_experiment":
            IIC_DUAL_EXPERIMENT_SHA256,

        "official_run":
            IIC_OFFICIAL_RUN_SHA256,

        "checkpoint_resume":
            IIC_CHECKPOINT_RESUME_SHA256,
    },

    "source_identity": {

        "central_mu_sha256":
            FULL_CENTRAL_MU_SHA256,

        "vae_final_checkpoint_sha256":
            EXPECTED_FINAL_CHECKPOINT_SHA256,
    },

    "labels_used":
        False,
}


PRIMARY_RUN_CONFIG_JSON = json.dumps(
    PRIMARY_RUN_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


PRIMARY_RUN_CONFIG_SHA256 = hashlib.sha256(
    PRIMARY_RUN_CONFIG_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# BENCHMARK PROJECTION
# ============================================================

projected_primary_minutes = float(
    symmetric_benchmark[
        "projected_100_epochs_minutes"
    ]
)


benchmark_projection_gate = (

    symmetric_benchmark_gate

    and projected_primary_minutes
        > 0.0

    and projected_primary_minutes
        < 180.0
)


assert benchmark_projection_gate


# ============================================================
# CHECKPOINT TEST MUST HAVE PASSED FOR THIS MODE
# ============================================================

checkpoint_resume_gate = (

    a5d_symmetric[
        "mode_gate"
    ]

    and IIC_CHECKPOINT_RESUME_SHA256
        == EXPECTED_HASHES[
            "checkpoint_resume"
        ]
)


assert checkpoint_resume_gate


# ============================================================
# FINAL GATE
# ============================================================

a5e_gate = all([

    contract_hash_gate,
    primary_identity_gate,
    official_directory_absent_gate,
    data_identity_gate,
    encoder_identity_gate,
    batch_geometry_gate,
    shuffle_epoch_gate,
    augmentation_epoch_gate,
    benchmark_projection_gate,
    checkpoint_resume_gate,
])


assert a5e_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 PRIMARY IIC FINAL PREFLIGHT ==="
)

print()

print(
    "RUN_ID =",
    PRIMARY_RUN_ID,
)

print(
    "MODE =",
    PRIMARY_MODE_NAME,
)

print(
    "RUN_DIRECTORY =",
    PRIMARY_RUN_DIR,
)

print()

print(
    "EVENT_COUNT =",
    EVENT_COUNT,
)

print(
    "BATCH_SIZE =",
    BATCH_SIZE,
)

print(
    "BATCHES_PER_EPOCH =",
    BATCHES_PER_EPOCH,
)

print(
    "TAIL_BATCH_SIZE =",
    TAIL_BATCH_SIZE,
)

print(
    "EPOCHS =",
    EPOCHS,
)

print(
    "STEPS_PER_CLASSIFIER =",
    STEPS_PER_CLASSIFIER,
)

print()

print(
    "CLASSIFIER_SEEDS =",
    EXPECTED_CLASSIFIER_SEEDS,
)

print(
    "SHUFFLE_SEED =",
    101112,
)

print(
    "AUGMENTATION_SEED =",
    789,
)

print()

print(
    "PRIMARY_RUN_CONFIG_SHA256 =",
    PRIMARY_RUN_CONFIG_SHA256,
)

print(
    "PROJECTED_PRIMARY_100_EPOCHS_MINUTES =",
    projected_primary_minutes,
)

print()

print(
    "CONTRACT_HASH_GATE =",
    contract_hash_gate,
)

print(
    "PRIMARY_IDENTITY_GATE =",
    primary_identity_gate,
)

print(
    "OFFICIAL_DIRECTORY_ABSENT_GATE =",
    official_directory_absent_gate,
)

print(
    "DATA_IDENTITY_GATE =",
    data_identity_gate,
)

print(
    "ENCODER_IDENTITY_GATE =",
    encoder_identity_gate,
)

print(
    "BATCH_GEOMETRY_GATE =",
    batch_geometry_gate,
)

print(
    "SHUFFLE_EPOCH_GATE =",
    shuffle_epoch_gate,
)

print(
    "AUGMENTATION_EPOCH_GATE =",
    augmentation_epoch_gate,
)

print(
    "BENCHMARK_PROJECTION_GATE =",
    benchmark_projection_gate,
)

print(
    "CHECKPOINT_RESUME_GATE =",
    checkpoint_resume_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "OFFICIAL_RUN_DIRECTORY_CREATED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5E_PRIMARY_FINAL_PREFLIGHT_PASS =",
    a5e_gate,
)


=== NOTEBOOK 06 PRIMARY IIC FINAL PREFLIGHT ===

RUN_ID = sakai224_full_iic_symmetric_v1
MODE = symmetric_figure
RUN_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_symmetric_v1

EVENT_COUNT = 8583
BATCH_SIZE = 128
BATCHES_PER_EPOCH = 68
TAIL_BATCH_SIZE = 7
EPOCHS = 100
STEPS_PER_CLASSIFIER = 6800

CLASSIFIER_SEEDS = [123, 124, 125, 126, 127]
SHUFFLE_SEED = 101112
AUGMENTATION_SEED = 789

PRIMARY_RUN_CONFIG_SHA256 = 6a34e279824042a12be112ba04ce0fb4c30f500c901921a8d2aee5ab7b817aca
PROJECTED_PRIMARY_100_EPOCHS_MINUTES = 76.33784124998631

CONTRACT_HASH_GATE = True
PRIMARY_IDENTITY_GATE = True
OFFICIAL_DIRECTORY_ABSENT_GATE = True
DATA_IDENTITY_GATE = True
ENCODER_IDENTITY_GATE = True
BATCH_GEOMETRY_GATE = True
SHUFFLE_EPOCH_GATE = True
AUGMENTATION_EPOCH_GATE = True
BENCHMARK_PROJECTION_GATE = True
CHECKPOINT_RESUME_GATE = True

LABELS_READ = False
OFFICIAL_RUN_DIRECTORY_CREATED = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEB

## STEP 06-A5F — Definizione del runner ufficiale IIC

Viene definito il runner utilizzato dalle due run IIC pre-registrate.

Il runner implementa il protocollo già congelato:

$$
D=8583,\quad
B=128,\quad
E=100,\quad
K=5,
$$

con encoder VAE `FINAL` congelato.

Per ogni minibatch:

1. vengono recuperate le feature centrali canoniche;
2. viene costruita la vista temporale perturbata;
3. il frozen encoder calcola una sola volta $\mu_\phi(x')$;
4. le stesse coppie $(z,z')$ vengono fornite ai cinque classificatori
   indipendenti;
5. ciascun classificatore esegue il proprio backward e il proprio step Adam.

La transazione ufficiale di fine epoca è:

$$
\text{epoch completata}
\rightarrow
\text{metrics.csv}
\rightarrow
\text{latest.pt}.
$$

`latest.pt` è l'autorità per il resume.

Alla centesima epoca viene inoltre creato:

`final.pt`.

Questa cella definisce soltanto il runner e non avvia alcuna run.

In [ ]:
# ============================================================
# NOTEBOOK 06 — OFFICIAL IIC RUNNER DEFINITION
# STEP 06-A5F
#
# DEFINITIONS ONLY.
#
# NO official directory creation.
# NO Drive writes.
# NO training.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path

import csv
import gc
import hashlib
import json
import os
import platform
import time

import numpy as np
import torch


assert a5e_gate
assert a5d_gate
assert benchmark_gate


# ============================================================
# METRICS SCHEMA
# ============================================================

OFFICIAL_METRICS_FIELDS = [
    "epoch",
    "elapsed_seconds",
    "batches",
    "samples",

    "optimizer_step_c0",
    "optimizer_step_c1",
    "optimizer_step_c2",
    "optimizer_step_c3",
    "optimizer_step_c4",

    "mean_batch_main_mi_c0",
    "mean_batch_main_mi_c1",
    "mean_batch_main_mi_c2",
    "mean_batch_main_mi_c3",
    "mean_batch_main_mi_c4",

    "mean_batch_overcluster_mi_c0",
    "mean_batch_overcluster_mi_c1",
    "mean_batch_overcluster_mi_c2",
    "mean_batch_overcluster_mi_c3",
    "mean_batch_overcluster_mi_c4",

    "mean_batch_total_loss_c0",
    "mean_batch_total_loss_c1",
    "mean_batch_total_loss_c2",
    "mean_batch_total_loss_c3",
    "mean_batch_total_loss_c4",
]


# ============================================================
# SMALL ATOMIC TEXT WRITER
# ============================================================

def atomic_write_text(
    path,
    text,
):

    path = Path(path)

    partial_path = Path(
        str(path) + ".partial"
    )


    with partial_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:

        handle.write(
            text
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


    os.replace(
        partial_path,
        path,
    )


# ============================================================
# RUNTIME SESSION LOG
# ============================================================

def append_runtime_session(
    path,
    record,
):

    path = Path(path)


    with path.open(
        "a",
        encoding="utf-8",
    ) as handle:

        handle.write(
            json.dumps(
                record,
                sort_keys=True,
                ensure_ascii=True,
            )
            + "\n"
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


# ============================================================
# METRICS INITIALIZATION
# ============================================================

def initialize_metrics_file(
    path,
):

    path = Path(path)


    with path.open(
        "x",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=OFFICIAL_METRICS_FIELDS,
        )

        writer.writeheader()

        handle.flush()

        os.fsync(
            handle.fileno()
        )


# ============================================================
# METRICS APPEND
# ============================================================

def append_metrics_row(
    path,
    row,
):

    path = Path(path)


    with path.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=OFFICIAL_METRICS_FIELDS,
        )

        writer.writerow(
            row
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


# ============================================================
# OFFICIAL CHECKPOINT PAYLOAD
# ============================================================

def make_official_iic_checkpoint(
    *,
    kind,
    run_id,
    mode_name,
    mode_id,
    last_completed_epoch,
    models,
    optimizers,
    optimizer_steps,
    run_config_sha256,
):

    classifier_states = [
        tree_to_cpu(
            model.state_dict()
        )

        for model
        in models
    ]


    optimizer_states = [
        tree_to_cpu(
            optimizer.state_dict()
        )

        for optimizer
        in optimizers
    ]


    payload = {

        "format_version":
            1,

        "kind":
            kind,

        "run_id":
            run_id,

        "augmentation_mode":
            mode_name,

        "augmentation_mode_id":
            int(
                mode_id
            ),

        "last_completed_epoch":
            int(
                last_completed_epoch
            ),

        "next_epoch":
            int(
                last_completed_epoch + 1
            ),

        "classifier_seeds":
            list(
                IIC_CLASSIFIER_SEEDS
            ),

        "optimizer_steps_per_classifier":
            list(
                optimizer_steps
            ),

        "classifier_state_dicts":
            classifier_states,

        "optimizer_state_dicts":
            optimizer_states,

        "classifier_states_sha256":
            canonical_tree_sha256(
                classifier_states
            ),

        "optimizer_states_sha256":
            canonical_tree_sha256(
                optimizer_states
            ),

        "run_config_sha256":
            run_config_sha256,

        "contracts": {

            "loss":
                IIC_LOSS_CONTRACT_SHA256,

            "architecture":
                IIC_ARCHITECTURE_CONTRACT_SHA256,

            "training_core":
                IIC_TRAINING_CORE_SHA256,

            "final_training":
                IIC_FINAL_TRAINING_SHA256,

            "dual_experiment":
                IIC_DUAL_EXPERIMENT_SHA256,

            "official_run":
                IIC_OFFICIAL_RUN_SHA256,

            "checkpoint_resume":
                IIC_CHECKPOINT_RESUME_SHA256,
        },

        "source_identity": {

            "central_mu_sha256":
                FULL_CENTRAL_MU_SHA256,

            "encoder_state_sha256":
                ENCODER_FINAL_STATE_SHA256,
        },

        "runtime": {

            "python":
                platform.python_version(),

            "torch":
                torch.__version__,

            "cuda":
                torch.version.cuda,

            "gpu":
                torch.cuda.get_device_name(
                    0
                ),
        },

        "labels_used":
            False,
    }


    return payload


# ============================================================
# RESTORE OFFICIAL CHECKPOINT
# ============================================================

def restore_official_iic_checkpoint(
    checkpoint_path,
    *,
    expected_run_id,
    expected_mode_name,
    expected_run_config_sha256,
):

    payload = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )


    identity_gate = (

        payload[
            "kind"
        ] in {
            "LATEST",
            "FINAL",
        }

        and payload[
            "run_id"
        ] == expected_run_id

        and payload[
            "augmentation_mode"
        ] == expected_mode_name

        and payload[
            "classifier_seeds"
        ] == IIC_CLASSIFIER_SEEDS

        and payload[
            "run_config_sha256"
        ] == expected_run_config_sha256

        and payload[
            "labels_used"
        ] is False

        and payload[
            "contracts"
        ][
            "checkpoint_resume"
        ] == IIC_CHECKPOINT_RESUME_SHA256
    )


    assert identity_gate


    classifier_hash_gate = (

        canonical_tree_sha256(
            payload[
                "classifier_state_dicts"
            ]
        )
        == payload[
            "classifier_states_sha256"
        ]
    )


    optimizer_hash_gate = (

        canonical_tree_sha256(
            payload[
                "optimizer_state_dicts"
            ]
        )
        == payload[
            "optimizer_states_sha256"
        ]
    )


    assert classifier_hash_gate
    assert optimizer_hash_gate


    (
        models,
        optimizers,
    ) = build_k5_ensemble()


    for index in range(
        5
    ):

        load_result = models[
            index
        ].load_state_dict(
            payload[
                "classifier_state_dicts"
            ][
                index
            ],
            strict=True,
        )


        assert (
            load_result.missing_keys == []
            and load_result.unexpected_keys == []
        )


        optimizers[
            index
        ].load_state_dict(
            payload[
                "optimizer_state_dicts"
            ][
                index
            ]
        )


        move_optimizer_state_to_device(
            optimizers[
                index
            ]
        )


    restore_gate = (

        ensemble_model_hashes(
            models
        )
        == [
            canonical_tree_sha256(
                state
            )

            for state
            in payload[
                "classifier_state_dicts"
            ]
        ]

        and ensemble_optimizer_hashes(
            optimizers
        )
        == [
            canonical_tree_sha256(
                state
            )

            for state
            in payload[
                "optimizer_state_dicts"
            ]
        ]

        and all(
            len(
                optimizer.state
            ) == 10

            for optimizer
            in optimizers
        )
    )


    assert restore_gate


    return (
        payload,
        models,
        optimizers,
    )


# ============================================================
# ONE OFFICIAL CLASSIFIER STEP WITH METRICS
# ============================================================

def train_one_iic_pair_with_metrics(
    *,
    model,
    optimizer,
    central_gpu,
    perturbed_gpu,
):

    model.train()


    optimizer.zero_grad(
        set_to_none=True
    )


    central_outputs = model(
        central_gpu
    )


    perturbed_outputs = model(
        perturbed_gpu
    )


    main_result = (
        project_iic_negative_mutual_information(
            central_outputs[
                "main"
            ],
            perturbed_outputs[
                "main"
            ],
        )
    )


    over_result = (
        project_iic_negative_mutual_information(
            central_outputs[
                "overcluster"
            ],
            perturbed_outputs[
                "overcluster"
            ],
        )
    )


    total_loss = combine_iic_losses(
        main_result[
            "loss"
        ],
        over_result[
            "loss"
        ],
    )


    numeric_gate = (

        torch.isfinite(
            main_result[
                "mutual_information"
            ]
        ).item()

        and torch.isfinite(
            over_result[
                "mutual_information"
            ]
        ).item()

        and torch.isfinite(
            total_loss
        ).item()
    )


    assert numeric_gate


    total_loss.backward()


    gradient_gate = all(

        parameter.grad is not None
        and torch.isfinite(
            parameter.grad
        ).all().item()

        for parameter
        in model.parameters()
    )


    assert gradient_gate


    optimizer.step()


    return {

        "main_mi":
            float(
                main_result[
                    "mutual_information"
                ].detach().item()
            ),

        "overcluster_mi":
            float(
                over_result[
                    "mutual_information"
                ].detach().item()
            ),

        "total_loss":
            float(
                total_loss
                .detach()
                .item()
            ),
    }


# ============================================================
# ONE COMPLETE OFFICIAL EPOCH
# ============================================================

def run_official_iic_epoch(
    *,
    models,
    optimizers,
    optimizer_steps,
    epoch,
    mode_id,
):

    epoch_start = time.perf_counter()


    permutation = (
        vae_c_shuffle_make_permutation_py(
            8583,
            101112,
            epoch,
        )
    )


    main_mi_sums = np.zeros(
        5,
        dtype=np.float64,
    )

    over_mi_sums = np.zeros(
        5,
        dtype=np.float64,
    )

    total_loss_sums = np.zeros(
        5,
        dtype=np.float64,
    )


    batch_count = 0
    sample_count = 0


    for batch_start in range(
        0,
        8583,
        128,
    ):

        manifest_indices = permutation[
            batch_start:
            batch_start + 128
        ]


        batch_size_actual = len(
            manifest_indices
        )


        assert (
            1
            < batch_size_actual
            <= 128
        )


        # ----------------------------------------------------
        # CENTRAL FEATURES
        # ----------------------------------------------------

        central_gpu = torch.from_numpy(
            np.asarray(
                full_central_mu[
                    manifest_indices
                ],
                dtype=np.float32,
            ).copy()
        ).to(
            runtime_device
        )


        # ----------------------------------------------------
        # PERTURBED IMAGES
        # ----------------------------------------------------

        (
            images_cpu,
            _,
        ) = make_real_perturbed_batch(
            manifest_indices,
            epoch=epoch,
            mode=mode_id,
        )


        # ----------------------------------------------------
        # FROZEN ENCODER
        # ----------------------------------------------------

        with torch.no_grad():

            (
                perturbed_gpu,
                _,
            ) = official_iic_encoder(
                images_cpu.to(
                    runtime_device,
                    dtype=torch.float32,
                )
            )


        perturbed_gpu = (
            perturbed_gpu
            .detach()
            .clone()
            .contiguous()
        )


        # ----------------------------------------------------
        # K=5 INDEPENDENT CLASSIFIERS
        # ----------------------------------------------------

        for classifier_index, (
            model,
            optimizer,
        ) in enumerate(
            zip(
                models,
                optimizers,
            )
        ):

            metrics = (
                train_one_iic_pair_with_metrics(
                    model=model,
                    optimizer=optimizer,
                    central_gpu=central_gpu,
                    perturbed_gpu=perturbed_gpu,
                )
            )


            optimizer_steps[
                classifier_index
            ] += 1


            main_mi_sums[
                classifier_index
            ] += metrics[
                "main_mi"
            ]


            over_mi_sums[
                classifier_index
            ] += metrics[
                "overcluster_mi"
            ]


            total_loss_sums[
                classifier_index
            ] += metrics[
                "total_loss"
            ]


        batch_count += 1

        sample_count += (
            batch_size_actual
        )


    torch.cuda.synchronize()


    elapsed_seconds = (
        time.perf_counter()
        - epoch_start
    )


    assert batch_count == 68
    assert sample_count == 8583


    return {

        "epoch":
            int(
                epoch
            ),

        "elapsed_seconds":
            float(
                elapsed_seconds
            ),

        "batches":
            batch_count,

        "samples":
            sample_count,

        "mean_batch_main_mi":
            (
                main_mi_sums
                / batch_count
            ),

        "mean_batch_overcluster_mi":
            (
                over_mi_sums
                / batch_count
            ),

        "mean_batch_total_loss":
            (
                total_loss_sums
                / batch_count
            ),
    }


# ============================================================
# CSV ROW BUILDER
# ============================================================

def make_official_metrics_row(
    *,
    epoch_result,
    optimizer_steps,
):

    row = {

        "epoch":
            epoch_result[
                "epoch"
            ],

        "elapsed_seconds":
            epoch_result[
                "elapsed_seconds"
            ],

        "batches":
            epoch_result[
                "batches"
            ],

        "samples":
            epoch_result[
                "samples"
            ],
    }


    for index in range(
        5
    ):

        row[
            f"optimizer_step_c{index}"
        ] = optimizer_steps[
            index
        ]


        row[
            f"mean_batch_main_mi_c{index}"
        ] = float(
            epoch_result[
                "mean_batch_main_mi"
            ][
                index
            ]
        )


        row[
            f"mean_batch_overcluster_mi_c{index}"
        ] = float(
            epoch_result[
                "mean_batch_overcluster_mi"
            ][
                index
            ]
        )


        row[
            f"mean_batch_total_loss_c{index}"
        ] = float(
            epoch_result[
                "mean_batch_total_loss"
            ][
                index
            ]
        )


    return row


# ============================================================
# OFFICIAL RUNNER
# ============================================================

def run_official_iic(
    *,
    run_id,
    mode_name,
    mode_id,
    run_config,
    run_config_json,
    run_config_sha256,
):

    run_dir = (
        SAKAI224_ROOT
        / "iic"
        / run_id
    )


    latest_path = (
        run_dir
        / "latest.pt"
    )


    final_path = (
        run_dir
        / "final.pt"
    )


    metrics_path = (
        run_dir
        / "metrics.csv"
    )


    config_path = (
        run_dir
        / "run_config.json"
    )


    sessions_path = (
        run_dir
        / "runtime_sessions.jsonl"
    )


    # ========================================================
    # COMPLETED RUN REFUSAL
    # ========================================================

    if final_path.exists():

        raise RuntimeError(
            "STOP: FINAL checkpoint already exists."
        )


    # ========================================================
    # NEW RUN
    # ========================================================

    if not run_dir.exists():

        run_dir.mkdir(
            parents=True,
            exist_ok=False,
        )


        atomic_write_text(
            config_path,
            run_config_json,
        )


        assert (
            sha256_stream(
                config_path
            )
            == run_config_sha256
        )


        initialize_metrics_file(
            metrics_path
        )


        (
            models,
            optimizers,
        ) = build_k5_ensemble()


        optimizer_steps = [
            0,
            0,
            0,
            0,
            0,
        ]


        start_epoch = 0


        session_mode = (
            "NEW"
        )


    # ========================================================
    # RESUME
    # ========================================================

    else:

        if not latest_path.is_file():

            raise RuntimeError(
                "STOP: run directory exists but "
                "latest.pt is missing."
            )


        if not config_path.is_file():

            raise RuntimeError(
                "STOP: run_config.json missing."
            )


        if (
            sha256_stream(
                config_path
            )
            != run_config_sha256
        ):

            raise RuntimeError(
                "STOP: run_config hash mismatch."
            )


        (
            payload,
            models,
            optimizers,
        ) = restore_official_iic_checkpoint(
            latest_path,
            expected_run_id=run_id,
            expected_mode_name=mode_name,
            expected_run_config_sha256=run_config_sha256,
        )


        last_completed_epoch = int(
            payload[
                "last_completed_epoch"
            ]
        )


        reconcile_metrics_to_checkpoint(
            metrics_path,
            last_completed_epoch,
        )


        optimizer_steps = list(
            payload[
                "optimizer_steps_per_classifier"
            ]
        )


        start_epoch = (
            last_completed_epoch + 1
        )


        session_mode = (
            "RESUME"
        )


        del payload


    # ========================================================
    # RUNTIME SESSION START
    # ========================================================

    append_runtime_session(
        sessions_path,
        {

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "event":
                "session_start",

            "mode":
                session_mode,

            "start_epoch":
                start_epoch,

            "device":
                str(
                    runtime_device
                ),

            "gpu":
                torch.cuda.get_device_name(
                    0
                ),

            "python":
                platform.python_version(),

            "torch":
                torch.__version__,
        },
    )


    # ========================================================
    # PRE-SESSION ENCODER IDENTITY
    # ========================================================

    encoder_before_session = (
        canonical_tree_sha256(
            official_iic_encoder.state_dict()
        )
    )


    assert (
        encoder_before_session
        == ENCODER_FINAL_STATE_SHA256
    )


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        start_epoch,
        100,
    ):

        epoch_result = (
            run_official_iic_epoch(
                models=models,
                optimizers=optimizers,
                optimizer_steps=optimizer_steps,
                epoch=epoch,
                mode_id=mode_id,
            )
        )


        # ====================================================
        # 1. METRICS COMMIT
        # ====================================================

        metrics_row = (
            make_official_metrics_row(
                epoch_result=epoch_result,
                optimizer_steps=optimizer_steps,
            )
        )


        append_metrics_row(
            metrics_path,
            metrics_row,
        )


        # ====================================================
        # 2. LATEST CHECKPOINT COMMIT
        # ====================================================

        latest_payload = (
            make_official_iic_checkpoint(
                kind="LATEST",
                run_id=run_id,
                mode_name=mode_name,
                mode_id=mode_id,
                last_completed_epoch=epoch,
                models=models,
                optimizers=optimizers,
                optimizer_steps=optimizer_steps,
                run_config_sha256=run_config_sha256,
            )
        )


        atomic_torch_save(
            latest_payload,
            latest_path,
        )


        # ====================================================
        # EPOCH REPORT
        # ====================================================

        mean_main = float(
            np.mean(
                epoch_result[
                    "mean_batch_main_mi"
                ]
            )
        )


        mean_over = float(
            np.mean(
                epoch_result[
                    "mean_batch_overcluster_mi"
                ]
            )
        )


        mean_total_loss = float(
            np.mean(
                epoch_result[
                    "mean_batch_total_loss"
                ]
            )
        )


        print(
            f"epoch {epoch + 1:03d}/100 | "
            f"{epoch_result['elapsed_seconds']:.2f}s | "
            f"main_MI={mean_main:.8f} | "
            f"over_MI={mean_over:.8f} | "
            f"loss={mean_total_loss:.8f} | "
            f"step={optimizer_steps[0]}"
        )


        # ====================================================
        # FINAL
        # ====================================================

        if epoch == 99:

            final_payload = (
                make_official_iic_checkpoint(
                    kind="FINAL",
                    run_id=run_id,
                    mode_name=mode_name,
                    mode_id=mode_id,
                    last_completed_epoch=epoch,
                    models=models,
                    optimizers=optimizers,
                    optimizer_steps=optimizer_steps,
                    run_config_sha256=run_config_sha256,
                )
            )


            atomic_torch_save(
                final_payload,
                final_path,
            )


    # ========================================================
    # POST-SESSION ENCODER IDENTITY
    # ========================================================

    encoder_after_session = (
        canonical_tree_sha256(
            official_iic_encoder.state_dict()
        )
    )


    assert (
        encoder_before_session
        == encoder_after_session
        == ENCODER_FINAL_STATE_SHA256
    )


    assert all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )


    # ========================================================
    # SESSION COMPLETE
    # ========================================================

    append_runtime_session(
        sessions_path,
        {

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "event":
                "session_complete",

            "last_completed_epoch":
                99,

            "optimizer_steps_per_classifier":
                optimizer_steps,
        },
    )


    print()
    print(
        "OFFICIAL_IIC_RUN_COMPLETE =",
        final_path.is_file(),
    )

    print(
        "FINAL_PATH =",
        final_path,
    )


    return final_path


# ============================================================
# DEFINITION-ONLY GATE
# ============================================================

official_runner_definition_gate = (

    callable(
        run_official_iic
    )

    and callable(
        run_official_iic_epoch
    )

    and callable(
        restore_official_iic_checkpoint
    )

    and len(
        OFFICIAL_METRICS_FIELDS
    ) == 24

    and not PRIMARY_RUN_DIR.exists()
)


assert official_runner_definition_gate


print()
print(
    "=== NOTEBOOK 06 OFFICIAL IIC RUNNER DEFINITION ==="
)

print()

print(
    "RUNNER_DEFINED =",
    True,
)

print(
    "METRICS_FIELDS =",
    len(
        OFFICIAL_METRICS_FIELDS
    ),
)

print(
    "OFFICIAL_RUN_DIRECTORY_EXISTS =",
    PRIMARY_RUN_DIR.exists(),
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_A5F_OFFICIAL_RUNNER_DEFINITION_PASS =",
    official_runner_definition_gate,
)


=== NOTEBOOK 06 OFFICIAL IIC RUNNER DEFINITION ===

RUNNER_DEFINED = True
METRICS_FIELDS = 24
OFFICIAL_RUN_DIRECTORY_EXISTS = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5F_OFFICIAL_RUNNER_DEFINITION_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 06 — ROBUST OFFICIAL IIC RESUME PATCH
# STEP 06-A5F-R
#
# Fixes:
# 1. interrupted NEW run before first latest.pt
# 2. interrupted FINAL publication after epoch 99 latest.pt
#
# DEFINITIONS ONLY.
# NO training.
# NO Drive writes.
# ============================================================

def run_official_iic(
    *,
    run_id,
    mode_name,
    mode_id,
    run_config,
    run_config_json,
    run_config_sha256,
):

    run_dir = (
        SAKAI224_ROOT
        / "iic"
        / run_id
    )

    latest_path = (
        run_dir
        / "latest.pt"
    )

    final_path = (
        run_dir
        / "final.pt"
    )

    metrics_path = (
        run_dir
        / "metrics.csv"
    )

    config_path = (
        run_dir
        / "run_config.json"
    )

    sessions_path = (
        run_dir
        / "runtime_sessions.jsonl"
    )


    # ========================================================
    # ALREADY COMPLETED
    # ========================================================

    if final_path.exists():

        raise RuntimeError(
            "STOP: FINAL checkpoint already exists."
        )


    # ========================================================
    # CASE A — BRAND NEW RUN
    # ========================================================

    if not run_dir.exists():

        run_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

        atomic_write_text(
            config_path,
            run_config_json,
        )

        assert (
            sha256_stream(
                config_path
            )
            == run_config_sha256
        )

        initialize_metrics_file(
            metrics_path
        )

        (
            models,
            optimizers,
        ) = build_k5_ensemble()

        optimizer_steps = [
            0,
            0,
            0,
            0,
            0,
        ]

        start_epoch = 0

        session_mode = "NEW"


    # ========================================================
    # CASE B — DIRECTORY EXISTS
    # ========================================================

    else:

        # ----------------------------------------------------
        # CONFIG MUST EXIST AND MATCH
        # ----------------------------------------------------

        if not config_path.is_file():

            raise RuntimeError(
                "STOP: run directory exists but "
                "run_config.json is missing."
            )

        if (
            sha256_stream(
                config_path
            )
            != run_config_sha256
        ):

            raise RuntimeError(
                "STOP: run_config hash mismatch."
            )


        # ----------------------------------------------------
        # B1 — CRASH BEFORE FIRST COMPLETED EPOCH
        #
        # Directory/config/metrics may exist,
        # but latest.pt does not exist yet.
        #
        # Restart cleanly from epoch 0.
        # ----------------------------------------------------

        if not latest_path.is_file():

            if not metrics_path.is_file():

                raise RuntimeError(
                    "STOP: incomplete run has no "
                    "metrics.csv."
                )


            # There must be no committed epoch without latest.
            # Reconcile to "nothing completed".
            reconcile_metrics_to_checkpoint(
                metrics_path,
                -1,
            )


            (
                models,
                optimizers,
            ) = build_k5_ensemble()


            optimizer_steps = [
                0,
                0,
                0,
                0,
                0,
            ]


            start_epoch = 0

            session_mode = (
                "RESTART_BEFORE_FIRST_CHECKPOINT"
            )


        # ----------------------------------------------------
        # B2 — NORMAL RESUME FROM LATEST
        # ----------------------------------------------------

        else:

            (
                payload,
                models,
                optimizers,
            ) = restore_official_iic_checkpoint(
                latest_path,
                expected_run_id=run_id,
                expected_mode_name=mode_name,
                expected_run_config_sha256=run_config_sha256,
            )


            last_completed_epoch = int(
                payload[
                    "last_completed_epoch"
                ]
            )


            assert (
                0
                <= last_completed_epoch
                <= 99
            )


            reconcile_metrics_to_checkpoint(
                metrics_path,
                last_completed_epoch,
            )


            optimizer_steps = list(
                payload[
                    "optimizer_steps_per_classifier"
                ]
            )


            start_epoch = (
                last_completed_epoch
                + 1
            )


            session_mode = "RESUME"


            del payload


    # ========================================================
    # SESSION START
    # ========================================================

    append_runtime_session(
        sessions_path,
        {
            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "event":
                "session_start",

            "mode":
                session_mode,

            "start_epoch":
                start_epoch,

            "device":
                str(
                    runtime_device
                ),

            "gpu":
                torch.cuda.get_device_name(
                    0
                ),

            "python":
                platform.python_version(),

            "torch":
                torch.__version__,
        },
    )


    # ========================================================
    # ENCODER IDENTITY
    # ========================================================

    encoder_before_session = (
        canonical_tree_sha256(
            official_iic_encoder.state_dict()
        )
    )

    assert (
        encoder_before_session
        == ENCODER_FINAL_STATE_SHA256
    )


    # ========================================================
    # TRAIN REMAINING EPOCHS
    # ========================================================

    for epoch in range(
        start_epoch,
        100,
    ):

        epoch_result = (
            run_official_iic_epoch(
                models=models,
                optimizers=optimizers,
                optimizer_steps=optimizer_steps,
                epoch=epoch,
                mode_id=mode_id,
            )
        )


        # ----------------------------------------------------
        # 1. METRICS
        # ----------------------------------------------------

        metrics_row = (
            make_official_metrics_row(
                epoch_result=epoch_result,
                optimizer_steps=optimizer_steps,
            )
        )

        append_metrics_row(
            metrics_path,
            metrics_row,
        )


        # ----------------------------------------------------
        # 2. LATEST
        # ----------------------------------------------------

        latest_payload = (
            make_official_iic_checkpoint(
                kind="LATEST",
                run_id=run_id,
                mode_name=mode_name,
                mode_id=mode_id,
                last_completed_epoch=epoch,
                models=models,
                optimizers=optimizers,
                optimizer_steps=optimizer_steps,
                run_config_sha256=run_config_sha256,
            )
        )

        atomic_torch_save(
            latest_payload,
            latest_path,
        )


        mean_main = float(
            np.mean(
                epoch_result[
                    "mean_batch_main_mi"
                ]
            )
        )

        mean_over = float(
            np.mean(
                epoch_result[
                    "mean_batch_overcluster_mi"
                ]
            )
        )

        mean_total_loss = float(
            np.mean(
                epoch_result[
                    "mean_batch_total_loss"
                ]
            )
        )


        print(
            f"epoch {epoch + 1:03d}/100 | "
            f"{epoch_result['elapsed_seconds']:.2f}s | "
            f"main_MI={mean_main:.8f} | "
            f"over_MI={mean_over:.8f} | "
            f"loss={mean_total_loss:.8f} | "
            f"step={optimizer_steps[0]}"
        )


    # ========================================================
    # FINALIZATION
    #
    # IMPORTANT:
    # This runs ALSO when we resume with latest.pt already
    # at epoch 99 and final.pt is missing.
    # ========================================================

    if not final_path.exists():

        if not latest_path.is_file():

            raise RuntimeError(
                "STOP: cannot finalize without latest.pt."
            )


        latest_for_final = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=True,
        )


        finalization_gate = (

            latest_for_final[
                "kind"
            ] == "LATEST"

            and latest_for_final[
                "last_completed_epoch"
            ] == 99

            and latest_for_final[
                "next_epoch"
            ] == 100

            and latest_for_final[
                "optimizer_steps_per_classifier"
            ] == [
                6800,
                6800,
                6800,
                6800,
                6800,
            ]

            and latest_for_final[
                "run_config_sha256"
            ] == run_config_sha256
        )


        assert finalization_gate


        final_payload = (
            make_official_iic_checkpoint(
                kind="FINAL",
                run_id=run_id,
                mode_name=mode_name,
                mode_id=mode_id,
                last_completed_epoch=99,
                models=models,
                optimizers=optimizers,
                optimizer_steps=optimizer_steps,
                run_config_sha256=run_config_sha256,
            )
        )


        atomic_torch_save(
            final_payload,
            final_path,
        )


        del latest_for_final
        del final_payload


    # ========================================================
    # FINAL CHECK
    # ========================================================

    final_verify = torch.load(
        final_path,
        map_location="cpu",
        weights_only=True,
    )


    final_gate = (

        final_verify[
            "kind"
        ] == "FINAL"

        and final_verify[
            "last_completed_epoch"
        ] == 99

        and final_verify[
            "next_epoch"
        ] == 100

        and final_verify[
            "optimizer_steps_per_classifier"
        ] == [
            6800,
            6800,
            6800,
            6800,
            6800,
        ]

        and final_verify[
            "run_config_sha256"
        ] == run_config_sha256
    )


    assert final_gate


    # ========================================================
    # ENCODER STILL IMMUTABLE
    # ========================================================

    encoder_after_session = (
        canonical_tree_sha256(
            official_iic_encoder.state_dict()
        )
    )


    assert (
        encoder_before_session
        == encoder_after_session
        == ENCODER_FINAL_STATE_SHA256
    )


    assert all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )


    # ========================================================
    # SESSION COMPLETE
    # ========================================================

    append_runtime_session(
        sessions_path,
        {
            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "event":
                "session_complete",

            "last_completed_epoch":
                99,

            "optimizer_steps_per_classifier":
                optimizer_steps,

            "final_verified":
                True,
        },
    )


    print()

    print(
        "OFFICIAL_IIC_RUN_COMPLETE =",
        True,
    )

    print(
        "FINAL_PATH =",
        final_path,
    )


    return final_path


# ============================================================
# DEFINITION-ONLY PATCH GATE
# ============================================================

a5fr_gate = (

    callable(
        run_official_iic
    )

    and official_runner_definition_gate
    and a5e_gate
    and a5d_gate
    and benchmark_gate

    and not PRIMARY_RUN_DIR.exists()
)


assert a5fr_gate


print()
print(
    "=== NOTEBOOK 06 ROBUST RESUME PATCH ==="
)

print(
    "FIRST_EPOCH_CRASH_RECOVERY = True"
)

print(
    "NORMAL_EPOCH_RESUME = True"
)

print(
    "EPOCH99_FINALIZATION_RECOVERY = True"
)

print(
    "OFFICIAL_RUN_DIRECTORY_EXISTS =",
    PRIMARY_RUN_DIR.exists(),
)

print(
    "OFFICIAL_TRAINING_STARTED = False"
)

print(
    "DRIVE_MODIFIED = False"
)

print()

print(
    "NOTEBOOK_06_A5F_R_ROBUST_RESUME_PASS =",
    a5fr_gate,
)


=== NOTEBOOK 06 ROBUST RESUME PATCH ===
FIRST_EPOCH_CRASH_RECOVERY = True
NORMAL_EPOCH_RESUME = True
EPOCH99_FINALIZATION_RECOVERY = True
OFFICIAL_RUN_DIRECTORY_EXISTS = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5F_R_ROBUST_RESUME_PASS = True


## STEP 06-A5F-T — Test reale degli edge case di resume

Il runner ufficiale viene testato in un ambiente temporaneo nella root runtime locale.

Sono verificati due casi limite.

### Caso 1 — interruzione prima del primo `latest.pt`

Si simula una run che possiede:

- directory;
- `run_config.json`;
- `metrics.csv`;

ma nessun checkpoint completato.

Viene inoltre simulata una riga metrics non ancora coperta da un
checkpoint.

Il runner deve:

1. riconoscere che nessuna epoca è stata committata;
2. eliminare la riga metrics non valida;
3. ricostruire modello e optimizer iniziali;
4. ripartire da epoch 0.

Il test arresta deliberatamente il runner immediatamente prima del primo
training batch, quindi nessuna epoca viene realmente eseguita.

### Caso 2 — epoch 99 completata ma `final.pt` assente

Si crea un `latest.pt` temporaneo semanticamente terminale:

$$
\mathrm{last\_completed\_epoch}=99,
$$

$$
\mathrm{next\_epoch}=100,
$$

$$
\mathrm{optimizer\_steps}=6800.
$$

Il runner deve:

1. caricare il checkpoint;
2. non eseguire alcuna nuova epoca;
3. creare `final.pt`;
4. preservare esattamente stato dei cinque classificatori e dei cinque Adam.

Il root PROJECT originale e gli stati RNG vengono ripristinati alla fine.

Nessuna label viene letta e Google Drive non viene modificato.

In [ ]:
# ============================================================
# NOTEBOOK 06 — REAL RESUME EDGE-CASE TESTS
# STEP 06-A5F-T
#
# Uses the ACTUAL patched run_official_iic().
#
# TEMP ROOT: /content only.
# NO Drive writes.
# NO official training.
# NO labels.
# ============================================================

from pathlib import Path

import csv
import gc
import hashlib
import json
import shutil

import numpy as np
import torch


assert a5fr_gate
assert official_runner_definition_gate
assert a5e_gate
assert a5d_gate


# ============================================================
# SAVE SCIENTIFIC RUNTIME STATE
# ============================================================

ORIGINAL_SAKAI224_ROOT = SAKAI224_ROOT

ORIGINAL_RUN_OFFICIAL_IIC_EPOCH = (
    run_official_iic_epoch
)

CPU_RNG_STATE_BEFORE_A5FT = (
    torch.random.get_rng_state()
)

CUDA_RNG_STATES_BEFORE_A5FT = (
    torch.cuda.get_rng_state_all()
)


# ============================================================
# TEMP ROOT
# ============================================================

A5FT_TEMP_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "notebook06_a5ft_resume_edgecases"
)


if A5FT_TEMP_ROOT.exists():

    shutil.rmtree(
        A5FT_TEMP_ROOT
    )


A5FT_TEMP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# TEMP CONFIG FACTORY
# ============================================================

def make_a5ft_run_config(
    run_id,
):

    config = json.loads(
        PRIMARY_RUN_CONFIG_JSON
    )


    config[
        "run_id"
    ] = run_id


    config[
        "role"
    ] = (
        "TEMP_RESUME_EDGECASE_TEST"
    )


    text = json.dumps(
        config,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    )


    sha256 = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


    return (
        config,
        text,
        sha256,
    )


# ============================================================
# READ METRICS
# ============================================================

def read_a5ft_metrics(
    path,
):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        return list(
            csv.DictReader(
                handle
            )
        )


# ============================================================
# READ JSONL
# ============================================================

def read_a5ft_jsonl(
    path,
):

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:

        return [
            json.loads(
                line
            )

            for line
            in handle

            if line.strip()
        ]


# ============================================================
# SYNTHETIC TERMINAL LATEST CHECKPOINT
#
# Allocate valid Adam state and then set its internal step
# counters consistently to 6800.
# ============================================================

def make_a5ft_terminal_latest(
    *,
    run_id,
    mode_name,
    mode_id,
    run_config_sha256,
):

    (
        models,
        optimizers,
    ) = build_k5_ensemble()


    # --------------------------------------------------------
    # Deterministic synthetic features.
    # No encoder and no dataset are modified.
    # --------------------------------------------------------

    base = torch.linspace(
        -1.0,
        1.0,
        steps=128 * 512,
        dtype=torch.float32,
        device=runtime_device,
    ).reshape(
        128,
        512,
    )


    perturbed = (
        torch.roll(
            base,
            shifts=1,
            dims=1,
        )
        .clone()
        .contiguous()
    )


    # One real update allocates all Adam state tensors.
    for model, optimizer in zip(
        models,
        optimizers,
    ):

        loss = train_one_iic_pair(
            model,
            optimizer,
            base,
            perturbed,
        )

        assert np.isfinite(
            loss
        )


    # Adam must own 10 parameter states per classifier.
    assert all(
        len(
            optimizer.state
        ) == 10

        for optimizer
        in optimizers
    )


    # --------------------------------------------------------
    # Make internal Adam counters consistent with the
    # terminal metadata: 6800 optimizer updates.
    # --------------------------------------------------------

    for optimizer in optimizers:

        for state in optimizer.state.values():

            assert "step" in state

            step_value = state[
                "step"
            ]


            if torch.is_tensor(
                step_value
            ):

                step_value.fill_(
                    6800
                )

            else:

                state[
                    "step"
                ] = 6800


    optimizer_step_internal_gate = all(

        all(

            (
                int(
                    state[
                        "step"
                    ].item()
                )
                if torch.is_tensor(
                    state[
                        "step"
                    ]
                )
                else int(
                    state[
                        "step"
                    ]
                )
            )
            == 6800

            for state
            in optimizer.state.values()
        )

        for optimizer
        in optimizers
    )


    assert optimizer_step_internal_gate


    payload = (
        make_official_iic_checkpoint(
            kind="LATEST",
            run_id=run_id,
            mode_name=mode_name,
            mode_id=mode_id,
            last_completed_epoch=99,
            models=models,
            optimizers=optimizers,
            optimizer_steps=[
                6800,
                6800,
                6800,
                6800,
                6800,
            ],
            run_config_sha256=run_config_sha256,
        )
    )


    del models
    del optimizers
    del base
    del perturbed

    gc.collect()
    torch.cuda.empty_cache()


    return (
        payload,
        optimizer_step_internal_gate,
    )


# ============================================================
# TEST RESULTS
# ============================================================

first_checkpoint_recovery_gate = False

epoch99_finalization_gate = False

finalization_no_training_gate = False

finalization_state_exact_gate = False

temp_root_redirect_gate = False


# ============================================================
# REDIRECT THE RUNNER TO /content
# ============================================================

try:

    SAKAI224_ROOT = (
        A5FT_TEMP_ROOT
    )


    temp_root_redirect_gate = (
        SAKAI224_ROOT.resolve(
            strict=False
        ).is_relative_to(
            LOCAL_RUNTIME_ROOT.resolve(
                strict=False
            )
        )

        and "drive" not in str(
            SAKAI224_ROOT
        ).lower()
    )


    assert temp_root_redirect_gate


    # ========================================================
    # CASE 1
    # CRASH BEFORE FIRST LATEST
    # ========================================================

    CASE1_RUN_ID = (
        "a5ft_before_first_latest"
    )


    (
        case1_config,
        case1_config_json,
        case1_config_sha256,
    ) = make_a5ft_run_config(
        CASE1_RUN_ID
    )


    case1_run_dir = (
        SAKAI224_ROOT
        / "iic"
        / CASE1_RUN_ID
    )


    case1_run_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    case1_config_path = (
        case1_run_dir
        / "run_config.json"
    )


    case1_metrics_path = (
        case1_run_dir
        / "metrics.csv"
    )


    case1_latest_path = (
        case1_run_dir
        / "latest.pt"
    )


    case1_final_path = (
        case1_run_dir
        / "final.pt"
    )


    case1_sessions_path = (
        case1_run_dir
        / "runtime_sessions.jsonl"
    )


    atomic_write_text(
        case1_config_path,
        case1_config_json,
    )


    initialize_metrics_file(
        case1_metrics_path
    )


    # Simulate:
    # metrics row committed, but crash before latest.pt.
    fake_row = {
        field:
            0

        for field
        in OFFICIAL_METRICS_FIELDS
    }


    fake_row[
        "epoch"
    ] = 0


    fake_row[
        "samples"
    ] = 8583


    fake_row[
        "batches"
    ] = 68


    append_metrics_row(
        case1_metrics_path,
        fake_row,
    )


    assert len(
        read_a5ft_metrics(
            case1_metrics_path
        )
    ) == 1


    assert not case1_latest_path.exists()


    # --------------------------------------------------------
    # Deliberately stop immediately when epoch 0 would start.
    # This lets us test the real resume branch without doing
    # any training.
    # --------------------------------------------------------

    class A5FTStopBeforeTraining(
        RuntimeError
    ):
        pass


    case1_observed = {}


    def a5ft_stop_epoch(
        *,
        models,
        optimizers,
        optimizer_steps,
        epoch,
        mode_id,
    ):

        case1_observed[
            "epoch"
        ] = int(
            epoch
        )


        case1_observed[
            "optimizer_steps"
        ] = list(
            optimizer_steps
        )


        raise A5FTStopBeforeTraining(
            "EXPECTED_A5FT_STOP"
        )


    run_official_iic_epoch = (
        a5ft_stop_epoch
    )


    expected_case1_stop = False


    try:

        run_official_iic(
            run_id=CASE1_RUN_ID,
            mode_name=PRIMARY_MODE_NAME,
            mode_id=PRIMARY_MODE_ID,
            run_config=case1_config,
            run_config_json=case1_config_json,
            run_config_sha256=case1_config_sha256,
        )


    except A5FTStopBeforeTraining as exc:

        assert str(
            exc
        ) == "EXPECTED_A5FT_STOP"

        expected_case1_stop = True


    assert expected_case1_stop


    # Restore the real epoch implementation.
    run_official_iic_epoch = (
        ORIGINAL_RUN_OFFICIAL_IIC_EPOCH
    )


    case1_rows_after = (
        read_a5ft_metrics(
            case1_metrics_path
        )
    )


    case1_sessions = (
        read_a5ft_jsonl(
            case1_sessions_path
        )
    )


    first_checkpoint_recovery_gate = (

        len(
            case1_rows_after
        ) == 0

        and case1_observed[
            "epoch"
        ] == 0

        and case1_observed[
            "optimizer_steps"
        ] == [
            0,
            0,
            0,
            0,
            0,
        ]

        and case1_sessions[
            -1
        ][
            "event"
        ] == "session_start"

        and case1_sessions[
            -1
        ][
            "mode"
        ] == (
            "RESTART_BEFORE_FIRST_CHECKPOINT"
        )

        and case1_sessions[
            -1
        ][
            "start_epoch"
        ] == 0

        and not case1_latest_path.exists()

        and not case1_final_path.exists()
    )


    assert first_checkpoint_recovery_gate


    # ========================================================
    # CASE 2
    # EPOCH 99 LATEST EXISTS, FINAL MISSING
    # ========================================================

    CASE2_RUN_ID = (
        "a5ft_epoch99_without_final"
    )


    (
        case2_config,
        case2_config_json,
        case2_config_sha256,
    ) = make_a5ft_run_config(
        CASE2_RUN_ID
    )


    case2_run_dir = (
        SAKAI224_ROOT
        / "iic"
        / CASE2_RUN_ID
    )


    case2_run_dir.mkdir(
        parents=True,
        exist_ok=False,
    )


    case2_config_path = (
        case2_run_dir
        / "run_config.json"
    )


    case2_metrics_path = (
        case2_run_dir
        / "metrics.csv"
    )


    case2_latest_path = (
        case2_run_dir
        / "latest.pt"
    )


    case2_final_path = (
        case2_run_dir
        / "final.pt"
    )


    atomic_write_text(
        case2_config_path,
        case2_config_json,
    )


    initialize_metrics_file(
        case2_metrics_path
    )


    # Write a complete synthetic 100-row metrics history.
    for epoch in range(
        100
    ):

        row = {
            field:
                0

            for field
            in OFFICIAL_METRICS_FIELDS
        }


        row[
            "epoch"
        ] = epoch


        row[
            "batches"
        ] = 68


        row[
            "samples"
        ] = 8583


        for classifier_index in range(
            5
        ):

            row[
                f"optimizer_step_c{classifier_index}"
            ] = (
                (epoch + 1)
                * 68
            )


        append_metrics_row(
            case2_metrics_path,
            row,
        )


    assert len(
        read_a5ft_metrics(
            case2_metrics_path
        )
    ) == 100


    (
        case2_latest_payload,
        optimizer_step_internal_gate,
    ) = make_a5ft_terminal_latest(
        run_id=CASE2_RUN_ID,
        mode_name=PRIMARY_MODE_NAME,
        mode_id=PRIMARY_MODE_ID,
        run_config_sha256=case2_config_sha256,
    )


    assert optimizer_step_internal_gate


    atomic_torch_save(
        case2_latest_payload,
        case2_latest_path,
    )


    assert case2_latest_path.is_file()
    assert not case2_final_path.exists()


    latest_loaded_before = torch.load(
        case2_latest_path,
        map_location="cpu",
        weights_only=True,
    )


    latest_classifier_hash_before = (
        canonical_tree_sha256(
            latest_loaded_before[
                "classifier_state_dicts"
            ]
        )
    )


    latest_optimizer_hash_before = (
        canonical_tree_sha256(
            latest_loaded_before[
                "optimizer_state_dicts"
            ]
        )
    )


    # --------------------------------------------------------
    # No epoch is allowed to run.
    # start_epoch must be 100.
    # --------------------------------------------------------

    case2_epoch_called = False


    def a5ft_forbid_epoch(
        **kwargs,
    ):

        nonlocal_placeholder = None

        raise AssertionError(
            "A5FT ERROR: training epoch was called "
            "during epoch99 finalization recovery."
        )


    run_official_iic_epoch = (
        a5ft_forbid_epoch
    )


    case2_returned_final = (
        run_official_iic(
            run_id=CASE2_RUN_ID,
            mode_name=PRIMARY_MODE_NAME,
            mode_id=PRIMARY_MODE_ID,
            run_config=case2_config,
            run_config_json=case2_config_json,
            run_config_sha256=case2_config_sha256,
        )
    )


    # Restore actual epoch implementation immediately.
    run_official_iic_epoch = (
        ORIGINAL_RUN_OFFICIAL_IIC_EPOCH
    )


    assert (
        case2_returned_final
        == case2_final_path
    )


    assert case2_final_path.is_file()


    case2_final_payload = torch.load(
        case2_final_path,
        map_location="cpu",
        weights_only=True,
    )


    final_classifier_hash = (
        canonical_tree_sha256(
            case2_final_payload[
                "classifier_state_dicts"
            ]
        )
    )


    final_optimizer_hash = (
        canonical_tree_sha256(
            case2_final_payload[
                "optimizer_state_dicts"
            ]
        )
    )


    finalization_no_training_gate = (

        case2_final_payload[
            "kind"
        ] == "FINAL"

        and case2_final_payload[
            "last_completed_epoch"
        ] == 99

        and case2_final_payload[
            "next_epoch"
        ] == 100

        and case2_final_payload[
            "optimizer_steps_per_classifier"
        ] == [
            6800,
            6800,
            6800,
            6800,
            6800,
        ]
    )


    finalization_state_exact_gate = (

        final_classifier_hash
        == latest_classifier_hash_before

        and final_optimizer_hash
        == latest_optimizer_hash_before
    )


    # metrics must remain exactly 100 rows.
    case2_metrics_after = (
        read_a5ft_metrics(
            case2_metrics_path
        )
    )


    epoch99_finalization_gate = (

        finalization_no_training_gate

        and finalization_state_exact_gate

        and len(
            case2_metrics_after
        ) == 100

        and int(
            case2_metrics_after[
                -1
            ][
                "epoch"
            ]
        ) == 99
    )


    assert epoch99_finalization_gate


finally:

    # ========================================================
    # ALWAYS RESTORE PROJECT GLOBALS / RNG
    # ========================================================

    SAKAI224_ROOT = (
        ORIGINAL_SAKAI224_ROOT
    )


    run_official_iic_epoch = (
        ORIGINAL_RUN_OFFICIAL_IIC_EPOCH
    )


    torch.random.set_rng_state(
        CPU_RNG_STATE_BEFORE_A5FT
    )


    torch.cuda.set_rng_state_all(
        CUDA_RNG_STATES_BEFORE_A5FT
    )


# ============================================================
# POST-RESTORE GATES
# ============================================================

project_root_restored_gate = (

    SAKAI224_ROOT
    == ORIGINAL_SAKAI224_ROOT
)


rng_restored_gate = (

    torch.equal(
        torch.random.get_rng_state(),
        CPU_RNG_STATE_BEFORE_A5FT,
    )

    and all(
        torch.equal(
            after,
            before,
        )

        for after, before
        in zip(
            torch.cuda.get_rng_state_all(),
            CUDA_RNG_STATES_BEFORE_A5FT,
        )
    )
)


official_primary_still_absent_gate = (
    not PRIMARY_RUN_DIR.exists()
)


encoder_after_a5ft = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_a5ft_gate = (

    encoder_after_a5ft
    == ENCODER_FINAL_STATE_SHA256

    and all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )
)


assert project_root_restored_gate
assert rng_restored_gate
assert official_primary_still_absent_gate
assert encoder_a5ft_gate


# ============================================================
# CONTRACT / HASH
# ============================================================

IIC_RESUME_EDGECASE_TEST_CONTRACT = {

    "contract_version":
        1,

    "actual_runner_tested":
        True,

    "before_first_latest_recovery":
        first_checkpoint_recovery_gate,

    "epoch99_finalization_recovery":
        epoch99_finalization_gate,

    "epoch99_no_training":
        finalization_no_training_gate,

    "epoch99_state_exact":
        finalization_state_exact_gate,

    "temp_root_only":
        temp_root_redirect_gate,

    "project_root_restored":
        project_root_restored_gate,

    "rng_restored":
        rng_restored_gate,

    "encoder_immutable":
        encoder_a5ft_gate,

    "official_primary_directory_absent":
        official_primary_still_absent_gate,

    "labels_used":
        False,

    "parent_checkpoint_resume_sha256":
        IIC_CHECKPOINT_RESUME_SHA256,
}


IIC_RESUME_EDGECASE_TEST_JSON = json.dumps(
    IIC_RESUME_EDGECASE_TEST_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_RESUME_EDGECASE_TEST_SHA256 = hashlib.sha256(
    IIC_RESUME_EDGECASE_TEST_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# FINAL GATE
# ============================================================

a5ft_gate = all([

    first_checkpoint_recovery_gate,

    epoch99_finalization_gate,

    finalization_no_training_gate,

    finalization_state_exact_gate,

    temp_root_redirect_gate,

    project_root_restored_gate,

    rng_restored_gate,

    official_primary_still_absent_gate,

    encoder_a5ft_gate,
])


assert a5ft_gate


# ============================================================
# CLEANUP TEMP FILES
# ============================================================

shutil.rmtree(
    A5FT_TEMP_ROOT
)


temporary_cleanup_gate = (
    not A5FT_TEMP_ROOT.exists()
)


assert temporary_cleanup_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 REAL RESUME EDGE-CASE AUDIT ==="
)

print()

print(
    "FIRST_CHECKPOINT_RECOVERY_GATE =",
    first_checkpoint_recovery_gate,
)

print(
    "EPOCH99_FINALIZATION_GATE =",
    epoch99_finalization_gate,
)

print(
    "FINALIZATION_NO_TRAINING_GATE =",
    finalization_no_training_gate,
)

print(
    "FINALIZATION_STATE_EXACT_GATE =",
    finalization_state_exact_gate,
)

print()

print(
    "TEMP_ROOT_REDIRECT_GATE =",
    temp_root_redirect_gate,
)

print(
    "PROJECT_ROOT_RESTORED_GATE =",
    project_root_restored_gate,
)

print(
    "RNG_RESTORED_GATE =",
    rng_restored_gate,
)

print(
    "ENCODER_A5FT_GATE =",
    encoder_a5ft_gate,
)

print(
    "OFFICIAL_PRIMARY_STILL_ABSENT_GATE =",
    official_primary_still_absent_gate,
)

print(
    "TEMPORARY_CLEANUP_GATE =",
    temporary_cleanup_gate,
)

print()

print(
    "IIC_RESUME_EDGECASE_TEST_SHA256 =",
    IIC_RESUME_EDGECASE_TEST_SHA256,
)

print()

print(
    "LABELS_READ = False"
)

print(
    "OFFICIAL_TRAINING_STARTED = False"
)

print(
    "DRIVE_MODIFIED = False"
)

print()

print(
    "NOTEBOOK_06_A5F_T_REAL_RESUME_EDGECASE_PASS =",
    a5ft_gate,
)


OFFICIAL_IIC_RUN_COMPLETE = True
FINAL_PATH = /content/notebook06_a5ft_resume_edgecases/iic/a5ft_epoch99_without_final/final.pt

=== NOTEBOOK 06 REAL RESUME EDGE-CASE AUDIT ===

FIRST_CHECKPOINT_RECOVERY_GATE = True
EPOCH99_FINALIZATION_GATE = True
FINALIZATION_NO_TRAINING_GATE = True
FINALIZATION_STATE_EXACT_GATE = True

TEMP_ROOT_REDIRECT_GATE = True
PROJECT_ROOT_RESTORED_GATE = True
RNG_RESTORED_GATE = True
ENCODER_A5FT_GATE = True
OFFICIAL_PRIMARY_STILL_ABSENT_GATE = True
TEMPORARY_CLEANUP_GATE = True

IIC_RESUME_EDGECASE_TEST_SHA256 = f15655c117fb6a9ca76bd414332b1a3491eb70c5b59de67da3a0e1fed946eca6

LABELS_READ = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5F_T_REAL_RESUME_EDGECASE_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 06 — SAFE CHECKPOINT SERIALIZATION PATCH
# STEP 06-A5F-S
#
# Fix:
# torch.__version__ can be torch.torch_version.TorchVersion.
#
# Official checkpoints must remain loadable with:
# torch.load(..., weights_only=True)
#
# NO official training.
# NO Drive writes.
# ============================================================

from pathlib import Path

import gc
import hashlib
import json

import numpy as np
import torch


assert a5fr_gate
assert a5e_gate
assert not PRIMARY_RUN_DIR.exists()


# ============================================================
# SAFE-SERIALIZATION CONTRACT
# ============================================================

IIC_SAFE_SERIALIZATION_CONTRACT = {

    "contract_version":
        1,

    "checkpoint_load_policy":
        "weights_only_true",

    "runtime_metadata":
        "builtin_primitives_only",

    "torch_version_storage":
        "builtin_str",

    "custom_TorchVersion_object_allowed":
        False,

    "weights_only_false_allowed":
        False,

    "safe_global_allowlist_required":
        False,

    "parent_official_run_sha256":
        IIC_OFFICIAL_RUN_SHA256,

    "parent_checkpoint_resume_sha256":
        IIC_CHECKPOINT_RESUME_SHA256,
}


IIC_SAFE_SERIALIZATION_JSON = json.dumps(
    IIC_SAFE_SERIALIZATION_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


IIC_SAFE_SERIALIZATION_SHA256 = hashlib.sha256(
    IIC_SAFE_SERIALIZATION_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# KEEP THE ORIGINAL CHECKPOINT FACTORY ONCE
# ============================================================

if (
    "_make_official_iic_checkpoint_base_v1"
    not in globals()
):

    _make_official_iic_checkpoint_base_v1 = (
        make_official_iic_checkpoint
    )


# ============================================================
# PATCHED CHECKPOINT FACTORY
# ============================================================

def make_official_iic_checkpoint(
    *,
    kind,
    run_id,
    mode_name,
    mode_id,
    last_completed_epoch,
    models,
    optimizers,
    optimizer_steps,
    run_config_sha256,
):

    payload = (
        _make_official_iic_checkpoint_base_v1(
            kind=kind,
            run_id=run_id,
            mode_name=mode_name,
            mode_id=mode_id,
            last_completed_epoch=last_completed_epoch,
            models=models,
            optimizers=optimizers,
            optimizer_steps=optimizer_steps,
            run_config_sha256=run_config_sha256,
        )
    )


    # --------------------------------------------------------
    # CRITICAL FIX:
    # force runtime metadata to true builtin primitives.
    # --------------------------------------------------------

    runtime = payload[
        "runtime"
    ]


    payload[
        "runtime"
    ] = {

        key: (
            None
            if value is None
            else str(value)
        )

        for key, value
        in runtime.items()
    }


    # Exact built-in str, not a subclass such as TorchVersion.
    assert all(

        value is None
        or type(value) is str

        for value
        in payload[
            "runtime"
        ].values()
    )


    payload[
        "contracts"
    ][
        "safe_serialization"
    ] = (
        IIC_SAFE_SERIALIZATION_SHA256
    )


    return payload


# ============================================================
# KEEP ORIGINAL RESTORE ONCE
# ============================================================

if (
    "_restore_official_iic_checkpoint_base_v1"
    not in globals()
):

    _restore_official_iic_checkpoint_base_v1 = (
        restore_official_iic_checkpoint
    )


# ============================================================
# PATCHED RESTORE
# ============================================================

def restore_official_iic_checkpoint(
    checkpoint_path,
    *,
    expected_run_id,
    expected_mode_name,
    expected_run_config_sha256,
):

    (
        payload,
        models,
        optimizers,
    ) = (
        _restore_official_iic_checkpoint_base_v1(
            checkpoint_path,
            expected_run_id=expected_run_id,
            expected_mode_name=expected_mode_name,
            expected_run_config_sha256=expected_run_config_sha256,
        )
    )


    safe_serialization_identity_gate = (

        payload[
            "contracts"
        ].get(
            "safe_serialization"
        )
        == IIC_SAFE_SERIALIZATION_SHA256

        and all(

            value is None
            or type(value) is str

            for value
            in payload[
                "runtime"
            ].values()
        )
    )


    assert safe_serialization_identity_gate


    return (
        payload,
        models,
        optimizers,
    )


# ============================================================
# REAL FULL-PAYLOAD SERIALIZATION PROBE
#
# Includes allocated Adam states, not merely model weights.
# ============================================================

PROBE_PATH = (
    LOCAL_RUNTIME_ROOT
    / "notebook06_a5fs_checkpoint_probe.pt"
)

PROBE_PARTIAL_PATH = Path(
    str(PROBE_PATH)
    + ".partial"
)


for path in [
    PROBE_PATH,
    PROBE_PARTIAL_PATH,
]:

    if path.exists():
        path.unlink()


CPU_RNG_BEFORE_A5FS = (
    torch.random.get_rng_state()
)

CUDA_RNG_BEFORE_A5FS = (
    torch.cuda.get_rng_state_all()
)


probe_models = None
probe_optimizers = None
probe_base = None
probe_perturbed = None


try:

    (
        probe_models,
        probe_optimizers,
    ) = build_k5_ensemble()


    # --------------------------------------------------------
    # Allocate realistic Adam state with one genuine update.
    # --------------------------------------------------------

    probe_base = torch.linspace(
        -1.0,
        1.0,
        steps=128 * 512,
        dtype=torch.float32,
        device=runtime_device,
    ).reshape(
        128,
        512,
    )


    probe_perturbed = (
        torch.roll(
            probe_base,
            shifts=1,
            dims=1,
        )
        .clone()
        .contiguous()
    )


    for model, optimizer in zip(
        probe_models,
        probe_optimizers,
    ):

        probe_loss = train_one_iic_pair(
            model,
            optimizer,
            probe_base,
            probe_perturbed,
        )


        assert np.isfinite(
            probe_loss
        )


    assert all(
        len(
            optimizer.state
        ) == 10

        for optimizer
        in probe_optimizers
    )


    probe_payload = (
        make_official_iic_checkpoint(
            kind="TEMP_SERIALIZATION_PROBE",
            run_id="a5fs_safe_serialization_probe",
            mode_name=PRIMARY_MODE_NAME,
            mode_id=PRIMARY_MODE_ID,
            last_completed_epoch=0,
            models=probe_models,
            optimizers=probe_optimizers,
            optimizer_steps=[
                1,
                1,
                1,
                1,
                1,
            ],
            run_config_sha256=PRIMARY_RUN_CONFIG_SHA256,
        )
    )


    runtime_builtin_gate = all(

        value is None
        or type(value) is str

        for value
        in probe_payload[
            "runtime"
        ].values()
    )


    torch_version_builtin_str_gate = (

        type(
            probe_payload[
                "runtime"
            ][
                "torch"
            ]
        )
        is str
    )


    assert runtime_builtin_gate
    assert torch_version_builtin_str_gate


    # --------------------------------------------------------
    # Actual atomic save.
    # --------------------------------------------------------

    atomic_torch_save(
        probe_payload,
        PROBE_PATH,
    )


    assert PROBE_PATH.is_file()
    assert not PROBE_PARTIAL_PATH.exists()


    # --------------------------------------------------------
    # CRITICAL TEST:
    # weights_only=True must load the COMPLETE checkpoint.
    # --------------------------------------------------------

    probe_loaded = torch.load(
        PROBE_PATH,
        map_location="cpu",
        weights_only=True,
    )


    weights_only_load_gate = (

        probe_loaded[
            "kind"
        ]
        == "TEMP_SERIALIZATION_PROBE"

        and probe_loaded[
            "contracts"
        ][
            "safe_serialization"
        ]
        == IIC_SAFE_SERIALIZATION_SHA256

        and type(
            probe_loaded[
                "runtime"
            ][
                "torch"
            ]
        )
        is str

        and len(
            probe_loaded[
                "classifier_state_dicts"
            ]
        ) == 5

        and len(
            probe_loaded[
                "optimizer_state_dicts"
            ]
        ) == 5
    )


    assert weights_only_load_gate


finally:

    # --------------------------------------------------------
    # Restore scientific RNG state.
    # --------------------------------------------------------

    torch.random.set_rng_state(
        CPU_RNG_BEFORE_A5FS
    )


    torch.cuda.set_rng_state_all(
        CUDA_RNG_BEFORE_A5FS
    )


    if PROBE_PATH.exists():
        PROBE_PATH.unlink()


    if PROBE_PARTIAL_PATH.exists():
        PROBE_PARTIAL_PATH.unlink()


    del probe_models
    del probe_optimizers
    del probe_base
    del probe_perturbed

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# POST-PROBE GATES
# ============================================================

rng_a5fs_restored_gate = (

    torch.equal(
        torch.random.get_rng_state(),
        CPU_RNG_BEFORE_A5FS,
    )

    and all(

        torch.equal(
            after,
            before,
        )

        for after, before
        in zip(
            torch.cuda.get_rng_state_all(),
            CUDA_RNG_BEFORE_A5FS,
        )
    )
)


probe_cleanup_gate = (

    not PROBE_PATH.exists()
    and not PROBE_PARTIAL_PATH.exists()
)


official_primary_absent_a5fs_gate = (
    not PRIMARY_RUN_DIR.exists()
)


encoder_a5fs_gate = (

    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256

    and all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )
)


a5fs_gate = all([

    runtime_builtin_gate,

    torch_version_builtin_str_gate,

    weights_only_load_gate,

    rng_a5fs_restored_gate,

    probe_cleanup_gate,

    official_primary_absent_a5fs_gate,

    encoder_a5fs_gate,
])


assert a5fs_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 SAFE CHECKPOINT SERIALIZATION ==="
)

print()

print(
    "RUNTIME_BUILTIN_GATE =",
    runtime_builtin_gate,
)

print(
    "TORCH_VERSION_BUILTIN_STR_GATE =",
    torch_version_builtin_str_gate,
)

print(
    "WEIGHTS_ONLY_LOAD_GATE =",
    weights_only_load_gate,
)

print(
    "RNG_RESTORED_GATE =",
    rng_a5fs_restored_gate,
)

print(
    "PROBE_CLEANUP_GATE =",
    probe_cleanup_gate,
)

print(
    "ENCODER_A5FS_GATE =",
    encoder_a5fs_gate,
)

print(
    "OFFICIAL_PRIMARY_ABSENT_GATE =",
    official_primary_absent_a5fs_gate,
)

print()

print(
    "IIC_SAFE_SERIALIZATION_SHA256 =",
    IIC_SAFE_SERIALIZATION_SHA256,
)

print()

print(
    "OFFICIAL_TRAINING_STARTED = False"
)

print(
    "DRIVE_MODIFIED = False"
)

print()

print(
    "NOTEBOOK_06_A5F_S_SAFE_SERIALIZATION_PASS =",
    a5fs_gate,
)


=== NOTEBOOK 06 SAFE CHECKPOINT SERIALIZATION ===

RUNTIME_BUILTIN_GATE = True
TORCH_VERSION_BUILTIN_STR_GATE = True
WEIGHTS_ONLY_LOAD_GATE = True
RNG_RESTORED_GATE = True
PROBE_CLEANUP_GATE = True
ENCODER_A5FS_GATE = True
OFFICIAL_PRIMARY_ABSENT_GATE = True

IIC_SAFE_SERIALIZATION_SHA256 = 57ccf6d4452ed649333e65d8f4281f314a42b2186d7d7ce7e9cded46a0933b78

OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5F_S_SAFE_SERIALIZATION_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 06 — ABSOLUTE FINAL START GATE
# STEP 06-A5G
#
# READ ONLY.
# NO directory creation.
# NO training.
# NO Drive writes.
# ============================================================

assert a5e_gate
assert official_runner_definition_gate
assert a5fr_gate
assert a5ft_gate

# Safe-serialization patch must also have passed.
assert "a5fs_gate" in globals()
assert a5fs_gate

assert (
    IIC_RESUME_EDGECASE_TEST_SHA256
    == "f15655c117fb6a9ca76bd414332b1a3491eb70c5b59de67da3a0e1fed946eca6"
)

assert (
    PRIMARY_RUN_CONFIG_SHA256
    == "6a34e279824042a12be112ba04ce0fb4c30f500c901921a8d2aee5ab7b817aca"
)

assert PRIMARY_RUN_ID == "sakai224_full_iic_symmetric_v1"
assert PRIMARY_MODE_NAME == "symmetric_figure"

assert not PRIMARY_RUN_DIR.exists()

assert (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256
)

assert official_iic_encoder.training is False

assert all(
    parameter.requires_grad is False
    for parameter in official_iic_encoder.parameters()
)

assert all(
    parameter.grad is None
    for parameter in official_iic_encoder.parameters()
)


a5g_gate = True


print()
print("=== NOTEBOOK 06 ABSOLUTE FINAL START GATE ===")
print()

print("RUN_ID =", PRIMARY_RUN_ID)
print("MODE =", PRIMARY_MODE_NAME)

print(
    "RUN_DIRECTORY =",
    PRIMARY_RUN_DIR,
)

print()

print(
    "SAFE_SERIALIZATION_GATE =",
    a5fs_gate,
)

print(
    "REAL_RESUME_EDGECASE_GATE =",
    a5ft_gate,
)

print(
    "RUN_DIRECTORY_ABSENT =",
    not PRIMARY_RUN_DIR.exists(),
)

print(
    "ENCODER_FROZEN =",
    True,
)

print()

print("LABELS_READ = False")
print("OFFICIAL_TRAINING_STARTED = False")
print("DRIVE_MODIFIED = False")

print()

print(
    "NOTEBOOK_06_A5G_FINAL_START_GATE_PASS =",
    a5g_gate,
)


=== NOTEBOOK 06 ABSOLUTE FINAL START GATE ===

RUN_ID = sakai224_full_iic_symmetric_v1
MODE = symmetric_figure
RUN_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_symmetric_v1

SAFE_SERIALIZATION_GATE = True
REAL_RESUME_EDGECASE_GATE = True
RUN_DIRECTORY_ABSENT = True
ENCODER_FROZEN = True

LABELS_READ = False
OFFICIAL_TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_A5G_FINAL_START_GATE_PASS = True


## STEP 06-A6A — Avvio della run IIC primaria ufficiale

Questa cella avvia realmente:

`sakai224_full_iic_symmetric_v1`.

La run utilizza il protocollo congelato e verificato nei gate precedenti.

Se il runtime viene interrotto, dopo aver ricostruito il notebook si deve
rieseguire la stessa chiamata: il runner userà `latest.pt` come autorità e
riprenderà dalla prima epoca non completata.

Non viene utilizzato alcun checkpoint BEST.

Il risultato scientifico della run sarà `final.pt` dopo 100 epoche complete.

In [ ]:
# ============================================================
# NOTEBOOK 06 — OFFICIAL PRIMARY IIC TRAINING
# STEP 06-A6A
#
# !!! THIS CELL STARTS THE OFFICIAL TRAINING !!!
#
# WRITES TO GOOGLE DRIVE.
# Creates/resumes:
# sakai224_full_iic_symmetric_v1
# ============================================================

assert a5g_gate
assert a5fs_gate
assert a5ft_gate
assert a5fr_gate

assert PRIMARY_RUN_ID == (
    "sakai224_full_iic_symmetric_v1"
)

assert PRIMARY_MODE_NAME == (
    "symmetric_figure"
)

assert PRIMARY_RUN_CONFIG_SHA256 == (
    "6a34e279824042a12be112ba04ce0fb4c"
    "30f500c901921a8d2aee5ab7b817aca"
)


print()
print(
    "=== STARTING / RESUMING OFFICIAL PRIMARY IIC RUN ==="
)

print(
    "RUN_ID =",
    PRIMARY_RUN_ID,
)

print(
    "MODE =",
    PRIMARY_MODE_NAME,
)

print(
    "RUN_DIRECTORY =",
    PRIMARY_RUN_DIR,
)

print(
    "TARGET_EPOCHS = 100"
)

print(
    "K = 5"
)

print(
    "THIS CELL WRITES TO DRIVE = True"
)

print()


official_primary_final_path = (
    run_official_iic(
        run_id=PRIMARY_RUN_ID,
        mode_name=PRIMARY_MODE_NAME,
        mode_id=PRIMARY_MODE_ID,
        run_config=PRIMARY_RUN_CONFIG,
        run_config_json=PRIMARY_RUN_CONFIG_JSON,
        run_config_sha256=PRIMARY_RUN_CONFIG_SHA256,
    )
)


print()
print(
    "=== PRIMARY IIC CALL RETURNED ==="
)

print(
    "FINAL_PATH =",
    official_primary_final_path,
)

print(
    "FINAL_EXISTS =",
    official_primary_final_path.is_file(),
)


=== STARTING / RESUMING OFFICIAL PRIMARY IIC RUN ===
RUN_ID = sakai224_full_iic_symmetric_v1
MODE = symmetric_figure
RUN_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_symmetric_v1
TARGET_EPOCHS = 100
K = 5
THIS CELL WRITES TO DRIVE = True

epoch 001/100 | 45.71s | main_MI=0.68779496 | over_MI=0.62329609 | loss=-1.31109105 | step=68
epoch 002/100 | 45.26s | main_MI=1.29863328 | over_MI=1.26441179 | loss=-2.56304507 | step=136
epoch 003/100 | 42.65s | main_MI=1.59255559 | over_MI=1.63711900 | loss=-3.22967460 | step=204
epoch 004/100 | 45.44s | main_MI=1.69716690 | over_MI=1.84183844 | loss=-3.53900534 | step=272
epoch 005/100 | 43.51s | main_MI=1.78482608 | over_MI=1.97071245 | loss=-3.75553852 | step=340
epoch 006/100 | 42.65s | main_MI=1.88440712 | over_MI=2.08645481 | loss=-3.97086193 | step=408
epoch 007/100 | 42.76s | main_MI=1.96794136 | over_MI=2.19716078 | loss=-4.16510213 | step=476
epoch 008/100 | 42.86s | main_MI=2.01447200 | over_MI=2

## STEP 06-A6B — Audit post-training della run IIC primaria

La run ufficiale primaria

`sakai224_full_iic_symmetric_v1`

ha completato 100 epoche.

Prima di utilizzare il risultato per gli step successivi viene eseguito un
audit read-only degli artefatti persistenti.

Vengono verificati:

- `final.pt`;
- `latest.pt`;
- `metrics.csv`;
- `run_config.json`;
- `runtime_sessions.jsonl`;
- identità dei contratti;
- identità del VAE e delle feature centrali;
- stato dei cinque classificatori;
- stato dei cinque optimizer Adam;
- contatori BatchNorm;
- caricamento con `weights_only=True`;
- equivalenza dello stato neurale tra `latest.pt` e `final.pt`;
- assenza di file `.partial`.

La presenza di un miglioramento della mutual information viene riportata
soltanto come diagnostica.

Non viene utilizzata per selezionare il checkpoint: il checkpoint scientifico
resta `FINAL`, come pre-registrato.

Nessuna label Gravity Spy viene letta.

In [ ]:
# ============================================================
# NOTEBOOK 06 — PRIMARY IIC POST-TRAINING AUDIT
# STEP 06-A6B
#
# READ ONLY.
# NO labels.
# NO training.
# NO Drive writes.
# ============================================================

from pathlib import Path

import csv
import json
import math
import statistics

import numpy as np
import torch


assert a5g_gate
assert a5fs_gate
assert a5ft_gate

assert PRIMARY_RUN_ID == (
    "sakai224_full_iic_symmetric_v1"
)


# ============================================================
# OFFICIAL PATHS
# ============================================================

A6B_RUN_DIR = (
    PRIMARY_RUN_DIR
)

A6B_LATEST_PATH = (
    A6B_RUN_DIR
    / "latest.pt"
)

A6B_FINAL_PATH = (
    A6B_RUN_DIR
    / "final.pt"
)

A6B_METRICS_PATH = (
    A6B_RUN_DIR
    / "metrics.csv"
)

A6B_CONFIG_PATH = (
    A6B_RUN_DIR
    / "run_config.json"
)

A6B_SESSIONS_PATH = (
    A6B_RUN_DIR
    / "runtime_sessions.jsonl"
)


required_files_gate = all(
    path.is_file()

    for path in [
        A6B_LATEST_PATH,
        A6B_FINAL_PATH,
        A6B_METRICS_PATH,
        A6B_CONFIG_PATH,
        A6B_SESSIONS_PATH,
    ]
)


assert required_files_gate


partial_files = list(
    A6B_RUN_DIR.rglob(
        "*.partial"
    )
)


no_partial_gate = (
    len(
        partial_files
    ) == 0
)


assert no_partial_gate


# ============================================================
# FILE HASHES
# ============================================================

A6B_LATEST_FILE_SHA256 = (
    sha256_stream(
        A6B_LATEST_PATH
    )
)

A6B_FINAL_FILE_SHA256 = (
    sha256_stream(
        A6B_FINAL_PATH
    )
)

A6B_METRICS_FILE_SHA256 = (
    sha256_stream(
        A6B_METRICS_PATH
    )
)

A6B_CONFIG_FILE_SHA256 = (
    sha256_stream(
        A6B_CONFIG_PATH
    )
)

A6B_SESSIONS_FILE_SHA256 = (
    sha256_stream(
        A6B_SESSIONS_PATH
    )
)


# ============================================================
# RUN CONFIG
# ============================================================

config_text = (
    A6B_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


config_identity_gate = (

    config_text
    == PRIMARY_RUN_CONFIG_JSON

    and A6B_CONFIG_FILE_SHA256
    == PRIMARY_RUN_CONFIG_SHA256
)


assert config_identity_gate


config_data = json.loads(
    config_text
)


config_semantic_gate = (

    config_data[
        "run_id"
    ] == PRIMARY_RUN_ID

    and config_data[
        "augmentation_mode"
    ] == "symmetric_figure"

    and config_data[
        "population"
    ][
        "event_count"
    ] == 8583

    and config_data[
        "iic"
    ][
        "C"
    ] == 36

    and config_data[
        "iic"
    ][
        "W"
    ] == 250

    and config_data[
        "iic"
    ][
        "K"
    ] == 5

    and config_data[
        "iic"
    ][
        "epochs"
    ] == 100

    and config_data[
        "labels_used"
    ] is False
)


assert config_semantic_gate


# ============================================================
# SAFE CHECKPOINT LOAD
# ============================================================

latest_payload = torch.load(
    A6B_LATEST_PATH,
    map_location="cpu",
    weights_only=True,
)


final_payload = torch.load(
    A6B_FINAL_PATH,
    map_location="cpu",
    weights_only=True,
)


weights_only_load_gate = (
    isinstance(
        latest_payload,
        dict,
    )

    and isinstance(
        final_payload,
        dict,
    )
)


assert weights_only_load_gate


# ============================================================
# COMMON CHECKPOINT VALIDATOR
# ============================================================

def a6b_validate_checkpoint(
    payload,
    *,
    expected_kind,
):

    identity_gate = (

        payload[
            "kind"
        ] == expected_kind

        and payload[
            "run_id"
        ] == PRIMARY_RUN_ID

        and payload[
            "augmentation_mode"
        ] == "symmetric_figure"

        and payload[
            "last_completed_epoch"
        ] == 99

        and payload[
            "next_epoch"
        ] == 100

        and payload[
            "classifier_seeds"
        ] == [
            123,
            124,
            125,
            126,
            127,
        ]

        and payload[
            "optimizer_steps_per_classifier"
        ] == [
            6800,
            6800,
            6800,
            6800,
            6800,
        ]

        and payload[
            "run_config_sha256"
        ] == PRIMARY_RUN_CONFIG_SHA256

        and payload[
            "labels_used"
        ] is False
    )


    contract_gate = (

        payload[
            "contracts"
        ][
            "loss"
        ] == IIC_LOSS_CONTRACT_SHA256

        and payload[
            "contracts"
        ][
            "architecture"
        ] == IIC_ARCHITECTURE_CONTRACT_SHA256

        and payload[
            "contracts"
        ][
            "training_core"
        ] == IIC_TRAINING_CORE_SHA256

        and payload[
            "contracts"
        ][
            "final_training"
        ] == IIC_FINAL_TRAINING_SHA256

        and payload[
            "contracts"
        ][
            "dual_experiment"
        ] == IIC_DUAL_EXPERIMENT_SHA256

        and payload[
            "contracts"
        ][
            "official_run"
        ] == IIC_OFFICIAL_RUN_SHA256

        and payload[
            "contracts"
        ][
            "checkpoint_resume"
        ] == IIC_CHECKPOINT_RESUME_SHA256

        and payload[
            "contracts"
        ][
            "safe_serialization"
        ] == IIC_SAFE_SERIALIZATION_SHA256
    )


    source_gate = (

        payload[
            "source_identity"
        ][
            "central_mu_sha256"
        ] == FULL_CENTRAL_MU_SHA256

        and payload[
            "source_identity"
        ][
            "encoder_state_sha256"
        ] == ENCODER_FINAL_STATE_SHA256
    )


    internal_hash_gate = (

        canonical_tree_sha256(
            payload[
                "classifier_state_dicts"
            ]
        )
        == payload[
            "classifier_states_sha256"
        ]

        and canonical_tree_sha256(
            payload[
                "optimizer_state_dicts"
            ]
        )
        == payload[
            "optimizer_states_sha256"
        ]
    )


    runtime_safe_gate = all(

        value is None
        or type(
            value
        ) is str

        for value
        in payload[
            "runtime"
        ].values()
    )


    count_gate = (

        len(
            payload[
                "classifier_state_dicts"
            ]
        ) == 5

        and len(
            payload[
                "optimizer_state_dicts"
            ]
        ) == 5
    )


    return all([
        identity_gate,
        contract_gate,
        source_gate,
        internal_hash_gate,
        runtime_safe_gate,
        count_gate,
    ])


latest_checkpoint_gate = (
    a6b_validate_checkpoint(
        latest_payload,
        expected_kind="LATEST",
    )
)


final_checkpoint_gate = (
    a6b_validate_checkpoint(
        final_payload,
        expected_kind="FINAL",
    )
)


assert latest_checkpoint_gate
assert final_checkpoint_gate


# ============================================================
# LATEST AND FINAL MUST REPRESENT THE SAME NEURAL STATE
# ============================================================

latest_classifier_tree_sha = (
    canonical_tree_sha256(
        latest_payload[
            "classifier_state_dicts"
        ]
    )
)

final_classifier_tree_sha = (
    canonical_tree_sha256(
        final_payload[
            "classifier_state_dicts"
        ]
    )
)


latest_optimizer_tree_sha = (
    canonical_tree_sha256(
        latest_payload[
            "optimizer_state_dicts"
        ]
    )
)

final_optimizer_tree_sha = (
    canonical_tree_sha256(
        final_payload[
            "optimizer_state_dicts"
        ]
    )
)


latest_final_state_gate = (

    latest_classifier_tree_sha
    == final_classifier_tree_sha

    and latest_optimizer_tree_sha
    == final_optimizer_tree_sha
)


assert latest_final_state_gate


# ============================================================
# ALL CHECKPOINT TENSORS MUST BE FINITE
# ============================================================

def tree_finite(
    value,
):

    if torch.is_tensor(
        value
    ):

        if (
            value.is_floating_point()
            or value.is_complex()
        ):

            return bool(
                torch.isfinite(
                    value
                ).all().item()
            )

        return True


    if isinstance(
        value,
        dict,
    ):

        return all(
            tree_finite(
                child
            )

            for child
            in value.values()
        )


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        return all(
            tree_finite(
                child
            )

            for child
            in value
        )


    return True


finite_checkpoint_gate = (

    tree_finite(
        final_payload[
            "classifier_state_dicts"
        ]
    )

    and tree_finite(
        final_payload[
            "optimizer_state_dicts"
        ]
    )
)


assert finite_checkpoint_gate


# ============================================================
# ADAM INTERNAL STEP AUDIT
# ============================================================

def optimizer_internal_steps(
    optimizer_state_dict,
):

    steps = []


    for state in (
        optimizer_state_dict[
            "state"
        ].values()
    ):

        assert "step" in state


        value = state[
            "step"
        ]


        if torch.is_tensor(
            value
        ):

            value = int(
                value.item()
            )

        else:

            value = int(
                value
            )


        steps.append(
            value
        )


    return steps


adam_internal_step_gate = True


for optimizer_state in final_payload[
    "optimizer_state_dicts"
]:

    internal_steps = (
        optimizer_internal_steps(
            optimizer_state
        )
    )


    adam_internal_step_gate &= (

        len(
            internal_steps
        ) == 10

        and set(
            internal_steps
        ) == {
            6800,
        }
    )


assert adam_internal_step_gate


# ============================================================
# BATCHNORM COUNTERS
#
# 6800 batches/classifier.
# Central + perturbed = two forwards per batch.
#
# Expected:
# 6800 * 2 = 13600
# ============================================================

EXPECTED_BN_BATCHES_TRACKED = (
    13600
)


bn_counter_gate = True

bn_counter_values = []


for state_dict in final_payload[
    "classifier_state_dicts"
]:

    main_bn_count = int(
        state_dict[
            "main_head.bn.num_batches_tracked"
        ].item()
    )


    over_bn_count = int(
        state_dict[
            "overcluster_head.bn.num_batches_tracked"
        ].item()
    )


    bn_counter_values.append(
        (
            main_bn_count,
            over_bn_count,
        )
    )


    bn_counter_gate &= (

        main_bn_count
        == EXPECTED_BN_BATCHES_TRACKED

        and over_bn_count
        == EXPECTED_BN_BATCHES_TRACKED
    )


assert bn_counter_gate


# ============================================================
# METRICS CSV
# ============================================================

with A6B_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    reader = csv.DictReader(
        handle
    )

    metrics_fields = list(
        reader.fieldnames
    )

    metrics_rows = list(
        reader
    )


metrics_schema_gate = (
    metrics_fields
    == OFFICIAL_METRICS_FIELDS
)


metrics_row_count_gate = (
    len(
        metrics_rows
    ) == 100
)


metrics_epoch_gate = (
    [
        int(
            row[
                "epoch"
            ]
        )

        for row
        in metrics_rows
    ]
    == list(
        range(
            100
        )
    )
)


assert metrics_schema_gate
assert metrics_row_count_gate
assert metrics_epoch_gate


metrics_numeric_gate = True
metrics_geometry_gate = True
metrics_loss_identity_gate = True


elapsed_seconds_values = []


for epoch, row in enumerate(
    metrics_rows
):

    elapsed_seconds = float(
        row[
            "elapsed_seconds"
        ]
    )


    elapsed_seconds_values.append(
        elapsed_seconds
    )


    metrics_numeric_gate &= (
        math.isfinite(
            elapsed_seconds
        )

        and elapsed_seconds
        > 0.0
    )


    metrics_geometry_gate &= (

        int(
            row[
                "batches"
            ]
        ) == 68

        and int(
            row[
                "samples"
            ]
        ) == 8583
    )


    expected_step = (
        (epoch + 1)
        * 68
    )


    for classifier_index in range(
        5
    ):

        metrics_geometry_gate &= (

            int(
                row[
                    f"optimizer_step_c{classifier_index}"
                ]
            )
            == expected_step
        )


        main_mi = float(
            row[
                f"mean_batch_main_mi_c{classifier_index}"
            ]
        )


        over_mi = float(
            row[
                f"mean_batch_overcluster_mi_c{classifier_index}"
            ]
        )


        total_loss = float(
            row[
                f"mean_batch_total_loss_c{classifier_index}"
            ]
        )


        metrics_numeric_gate &= all(
            math.isfinite(
                value
            )

            for value in [
                main_mi,
                over_mi,
                total_loss,
            ]
        )


        metrics_loss_identity_gate &= (
            np.isclose(
                total_loss,
                -(
                    main_mi
                    + over_mi
                ),
                rtol=1.0e-6,
                atol=5.0e-6,
            )
        )


assert metrics_numeric_gate
assert metrics_geometry_gate
assert metrics_loss_identity_gate


# ============================================================
# TRAINING SUMMARY — DIAGNOSTIC ONLY
# ============================================================

def mean_epoch_metric(
    row,
    prefix,
):

    return float(
        np.mean(
            [
                float(
                    row[
                        f"{prefix}_c{i}"
                    ]
                )

                for i
                in range(
                    5
                )
            ]
        )
    )


first_main_mi = (
    mean_epoch_metric(
        metrics_rows[0],
        "mean_batch_main_mi",
    )
)


final_main_mi = (
    mean_epoch_metric(
        metrics_rows[-1],
        "mean_batch_main_mi",
    )
)


first_over_mi = (
    mean_epoch_metric(
        metrics_rows[0],
        "mean_batch_overcluster_mi",
    )
)


final_over_mi = (
    mean_epoch_metric(
        metrics_rows[-1],
        "mean_batch_overcluster_mi",
    )
)


first_total_loss = (
    mean_epoch_metric(
        metrics_rows[0],
        "mean_batch_total_loss",
    )
)


final_total_loss = (
    mean_epoch_metric(
        metrics_rows[-1],
        "mean_batch_total_loss",
    )
)


total_training_seconds = float(
    sum(
        elapsed_seconds_values
    )
)


training_progress_observed = (

    final_main_mi
    > first_main_mi

    and final_over_mi
    > first_over_mi

    and final_total_loss
    < first_total_loss
)


# Diagnostic only — NOT checkpoint selection.
assert math.isfinite(
    total_training_seconds
)


# ============================================================
# RUNTIME SESSIONS
# ============================================================

with A6B_SESSIONS_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    session_records = [
        json.loads(
            line
        )

        for line
        in handle

        if line.strip()
    ]


session_start_records = [
    record
    for record
    in session_records
    if record.get(
        "event"
    ) == "session_start"
]


session_complete_records = [
    record
    for record
    in session_records
    if record.get(
        "event"
    ) == "session_complete"
]


runtime_session_gate = (

    len(
        session_start_records
    ) >= 1

    and len(
        session_complete_records
    ) >= 1

    and session_complete_records[
        -1
    ][
        "last_completed_epoch"
    ] == 99

    and session_complete_records[
        -1
    ][
        "optimizer_steps_per_classifier"
    ] == [
        6800,
        6800,
        6800,
        6800,
        6800,
    ]

    and session_complete_records[
        -1
    ].get(
        "final_verified"
    ) is True
)


assert runtime_session_gate


# ============================================================
# LOAD FINAL CLASSIFIERS AND RUN A SMALL REAL FEATURE PROBE
# ============================================================

CPU_RNG_BEFORE_A6B = (
    torch.random.get_rng_state()
)

CUDA_RNG_BEFORE_A6B = (
    torch.cuda.get_rng_state_all()
)


probe_models = []


try:

    for classifier_index, seed in enumerate(
        IIC_CLASSIFIER_SEEDS
    ):

        model = (
            build_iic_pair_from_seed(
                seed
            ).to(
                runtime_device
            )
        )


        load_result = (
            model.load_state_dict(
                final_payload[
                    "classifier_state_dicts"
                ][
                    classifier_index
                ],
                strict=True,
            )
        )


        assert (
            load_result.missing_keys == []
            and load_result.unexpected_keys == []
        )


        model.eval()

        probe_models.append(
            model
        )


    probe_indices = [
        0,
        2145,
        4291,
        6864,
        7663,
        8582,
    ]


    probe_mu = torch.from_numpy(
        np.asarray(
            full_central_mu[
                probe_indices
            ],
            dtype=np.float32,
        ).copy()
    ).to(
        runtime_device
    )


    classifier_forward_gate = True


    with torch.no_grad():

        for model in probe_models:

            outputs_a = model(
                probe_mu
            )


            outputs_b = model(
                probe_mu
            )


            main = outputs_a[
                "main"
            ]

            over = outputs_a[
                "overcluster"
            ]


            classifier_forward_gate &= (

                main.shape
                == (
                    6,
                    36,
                )

                and over.shape
                == (
                    6,
                    250,
                )

                and torch.isfinite(
                    main
                ).all().item()

                and torch.isfinite(
                    over
                ).all().item()

                and torch.allclose(
                    main.sum(
                        dim=1
                    ),
                    torch.ones(
                        6,
                        device=runtime_device,
                    ),
                    rtol=1.0e-6,
                    atol=1.0e-6,
                )

                and torch.allclose(
                    over.sum(
                        dim=1
                    ),
                    torch.ones(
                        6,
                        device=runtime_device,
                    ),
                    rtol=1.0e-6,
                    atol=1.0e-6,
                )

                and torch.allclose(
                    outputs_a[
                        "main"
                    ],
                    outputs_b[
                        "main"
                    ],
                    rtol=0.0,
                    atol=0.0,
                )

                and torch.allclose(
                    outputs_a[
                        "overcluster"
                    ],
                    outputs_b[
                        "overcluster"
                    ],
                    rtol=0.0,
                    atol=0.0,
                )
            )


    assert classifier_forward_gate


finally:

    probe_models.clear()


    torch.random.set_rng_state(
        CPU_RNG_BEFORE_A6B
    )


    torch.cuda.set_rng_state_all(
        CUDA_RNG_BEFORE_A6B
    )


rng_restored_gate = (

    torch.equal(
        torch.random.get_rng_state(),
        CPU_RNG_BEFORE_A6B,
    )

    and all(
        torch.equal(
            after,
            before,
        )

        for after, before
        in zip(
            torch.cuda.get_rng_state_all(),
            CUDA_RNG_BEFORE_A6B,
        )
    )
)


assert rng_restored_gate


# ============================================================
# ENCODER MUST STILL BE EXACTLY FROZEN
# ============================================================

encoder_posttraining_gate = (

    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256

    and official_iic_encoder.training
    is False

    and all(
        parameter.requires_grad
        is False

        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None

        for parameter
        in official_iic_encoder.parameters()
    )
)


assert encoder_posttraining_gate


# ============================================================
# FINAL AUDIT GATE
# ============================================================

a6b_gate = all([

    required_files_gate,
    no_partial_gate,

    config_identity_gate,
    config_semantic_gate,

    weights_only_load_gate,

    latest_checkpoint_gate,
    final_checkpoint_gate,

    latest_final_state_gate,
    finite_checkpoint_gate,

    adam_internal_step_gate,
    bn_counter_gate,

    metrics_schema_gate,
    metrics_row_count_gate,
    metrics_epoch_gate,
    metrics_numeric_gate,
    metrics_geometry_gate,
    metrics_loss_identity_gate,

    runtime_session_gate,

    classifier_forward_gate,
    rng_restored_gate,

    encoder_posttraining_gate,
])


assert a6b_gate


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 PRIMARY IIC POST-TRAINING AUDIT ==="
)

print()

print(
    "RUN_ID =",
    PRIMARY_RUN_ID,
)

print(
    "LATEST_FILE_SHA256 =",
    A6B_LATEST_FILE_SHA256,
)

print(
    "FINAL_FILE_SHA256 =",
    A6B_FINAL_FILE_SHA256,
)

print(
    "METRICS_FILE_SHA256 =",
    A6B_METRICS_FILE_SHA256,
)

print(
    "RUN_CONFIG_SHA256 =",
    A6B_CONFIG_FILE_SHA256,
)

print(
    "RUNTIME_SESSIONS_SHA256 =",
    A6B_SESSIONS_FILE_SHA256,
)

print()

print(
    "LATEST_CHECKPOINT_GATE =",
    latest_checkpoint_gate,
)

print(
    "FINAL_CHECKPOINT_GATE =",
    final_checkpoint_gate,
)

print(
    "LATEST_FINAL_NEURAL_STATE_EXACT =",
    latest_final_state_gate,
)

print(
    "ADAM_INTERNAL_STEP_GATE =",
    adam_internal_step_gate,
)

print(
    "BN_COUNTER_GATE =",
    bn_counter_gate,
)

print(
    "BN_BATCHES_TRACKED =",
    bn_counter_values,
)

print()

print(
    "METRICS_ROWS =",
    len(
        metrics_rows
    ),
)

print(
    "METRICS_SCHEMA_GATE =",
    metrics_schema_gate,
)

print(
    "METRICS_GEOMETRY_GATE =",
    metrics_geometry_gate,
)

print(
    "METRICS_LOSS_IDENTITY_GATE =",
    metrics_loss_identity_gate,
)

print()

print(
    "FIRST_MAIN_MI =",
    first_main_mi,
)

print(
    "FINAL_MAIN_MI =",
    final_main_mi,
)

print(
    "FIRST_OVERCLUSTER_MI =",
    first_over_mi,
)

print(
    "FINAL_OVERCLUSTER_MI =",
    final_over_mi,
)

print(
    "FIRST_TOTAL_LOSS =",
    first_total_loss,
)

print(
    "FINAL_TOTAL_LOSS =",
    final_total_loss,
)

print(
    "TRAINING_PROGRESS_OBSERVED =",
    training_progress_observed,
)

print()

print(
    "TOTAL_TRAINING_SECONDS =",
    total_training_seconds,
)

print(
    "TOTAL_TRAINING_MINUTES =",
    total_training_seconds
    / 60.0,
)

print(
    "MEDIAN_EPOCH_SECONDS =",
    statistics.median(
        elapsed_seconds_values
    ),
)

print()

print(
    "RUNTIME_SESSION_COUNT =",
    len(
        session_records
    ),
)

print(
    "RUNTIME_SESSION_GATE =",
    runtime_session_gate,
)

print(
    "CLASSIFIER_FORWARD_GATE =",
    classifier_forward_gate,
)

print(
    "ENCODER_POSTTRAINING_GATE =",
    encoder_posttraining_gate,
)

print(
    "NO_PARTIAL_GATE =",
    no_partial_gate,
)

print()

print(
    "LABELS_READ = False"
)

print(
    "DRIVE_MODIFIED_BY_AUDIT = False"
)

print()

print(
    "NOTEBOOK_06_A6B_PRIMARY_POSTTRAINING_AUDIT_PASS =",
    a6b_gate,
)


=== NOTEBOOK 06 PRIMARY IIC POST-TRAINING AUDIT ===

RUN_ID = sakai224_full_iic_symmetric_v1
LATEST_FILE_SHA256 = 354594985397fe3058fee78db859d96cd2e4eaeb482d7b47c3390b8a6da1f2f7
FINAL_FILE_SHA256 = 018008df8e6a1f97195cb1cc234e915036806b2d3c1d09efefbf4333fe7f1199
METRICS_FILE_SHA256 = 78492b88b7e860dda1f986d666eba8a3c67a44411985911c48b85a6931b53851
RUN_CONFIG_SHA256 = 6a34e279824042a12be112ba04ce0fb4c30f500c901921a8d2aee5ab7b817aca
RUNTIME_SESSIONS_SHA256 = db68f87ad38dfab6120af13f71eb047fac97992bdcf7f20a47e11e456aedc583

LATEST_CHECKPOINT_GATE = True
FINAL_CHECKPOINT_GATE = True
LATEST_FINAL_NEURAL_STATE_EXACT = True
ADAM_INTERNAL_STEP_GATE = True
BN_COUNTER_GATE = True
BN_BATCHES_TRACKED = [(13600, 13600), (13600, 13600), (13600, 13600), (13600, 13600), (13600, 13600)]

METRICS_ROWS = 100
METRICS_SCHEMA_GATE = True
METRICS_GEOMETRY_GATE = True
METRICS_LOSS_IDENTITY_GATE = True

FIRST_MAIN_MI = 0.6877949588498108
FINAL_MAIN_MI = 3.0927692024146807
FIRST_OVERCLUSTER_MI = 0.62329609093

In [ ]:
# ============================================================
# NOTEBOOK 06 — PRIMARY IIC SESSION HANDOFF
# STEP 06-A6C
#
# Writes ONE small audit/handoff JSON to Drive.
# NO training.
# NO model modification.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json


assert a6b_gate
assert final_checkpoint_gate
assert latest_checkpoint_gate
assert latest_final_state_gate
assert encoder_posttraining_gate


# Keep audit metadata outside the official run directory.
A6C_AUDIT_DIR = (
    SAKAI224_ROOT
    / "iic"
    / "audits"
)

A6C_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


A6C_HANDOFF_PATH = (
    A6C_AUDIT_DIR
    / "sakai224_full_iic_symmetric_v1_after_A6B.json"
)


# Refuse accidental overwrite.
assert not A6C_HANDOFF_PATH.exists(), (
    f"STOP: handoff already exists: {A6C_HANDOFF_PATH}"
)


A6C_HANDOFF = {

    "format_version":
        1,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project_phase":
        "Notebook06_IIC",

    "completed_step":
        "A6B_primary_posttraining_audit",

    "primary_run": {

        "run_id":
            PRIMARY_RUN_ID,

        "mode":
            PRIMARY_MODE_NAME,

        "epochs_completed":
            100,

        "optimizer_steps_per_classifier":
            6800,

        "classifier_count_K":
            5,

        "audit_pass":
            True,
    },

    "artifacts": {

        "latest_sha256":
            A6B_LATEST_FILE_SHA256,

        "final_sha256":
            A6B_FINAL_FILE_SHA256,

        "metrics_sha256":
            A6B_METRICS_FILE_SHA256,

        "run_config_sha256":
            A6B_CONFIG_FILE_SHA256,

        "runtime_sessions_sha256":
            A6B_SESSIONS_FILE_SHA256,

        "classifier_state_sha256":
            final_classifier_tree_sha,

        "optimizer_state_sha256":
            final_optimizer_tree_sha,
    },

    "scientific_state": {

        "latest_final_neural_state_exact":
            True,

        "adam_internal_step_gate":
            True,

        "bn_batches_tracked":
            13600,

        "metrics_rows":
            100,

        "encoder_immutable":
            True,

        "labels_read":
            False,
    },

    "next_required_step": {

        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "mode":
            "legacy_text",

        "role":
            "HTML_TEXT_FIDELITY_SENSITIVITY",

        "must_complete_before_spectral":
            True,

        "do_not_read_labels":
            True,
    },

    "status":
        "SAFE_TO_STOP_RUNTIME",
}


A6C_HANDOFF_JSON = json.dumps(
    A6C_HANDOFF,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


atomic_write_text(
    A6C_HANDOFF_PATH,
    A6C_HANDOFF_JSON,
)


A6C_HANDOFF_SHA256 = (
    sha256_stream(
        A6C_HANDOFF_PATH
    )
)


a6c_gate = (

    A6C_HANDOFF_PATH.is_file()

    and json.loads(
        A6C_HANDOFF_PATH.read_text(
            encoding="utf-8"
        )
    )[
        "status"
    ]
    == "SAFE_TO_STOP_RUNTIME"

    and canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256
)


assert a6c_gate


print()
print(
    "=== NOTEBOOK 06 END-OF-SESSION HANDOFF ==="
)

print(
    "PRIMARY_RUN_AUDITED = True"
)

print(
    "HANDOFF_PATH =",
    A6C_HANDOFF_PATH,
)

print(
    "HANDOFF_SHA256 =",
    A6C_HANDOFF_SHA256,
)

print(
    "NEXT_RUN = sakai224_full_iic_legacy_text_v1"
)

print(
    "LABELS_READ = False"
)

print(
    "SAFE_TO_STOP_RUNTIME = True"
)

print()

print(
    "NOTEBOOK_06_A6C_HANDOFF_PASS =",
    a6c_gate,
)


=== NOTEBOOK 06 END-OF-SESSION HANDOFF ===
PRIMARY_RUN_AUDITED = True
HANDOFF_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/audits/sakai224_full_iic_symmetric_v1_after_A6B.json
HANDOFF_SHA256 = 8154f74e4bd34d086d99c706c3e65cf2d67ba8da1b16b3308b360264bc2bd1db
NEXT_RUN = sakai224_full_iic_legacy_text_v1
LABELS_READ = False
SAFE_TO_STOP_RUNTIME = True

NOTEBOOK_06_A6C_HANDOFF_PASS = True


In [1]:
# ============================================================
# NOTEBOOK 06 — CONTROLLED RUNTIME RECOVERY
# STEP 06-R0
#
# Verifies the completed primary IIC run and handoff on Drive.
#
# READ ONLY.
# NO data extraction.
# NO model construction.
# NO training.
# ============================================================

from pathlib import Path

import hashlib
import json
import platform

import torch


# ============================================================
# RUNTIME POLICY
# ============================================================

assert torch.cuda.is_available(), (
    "STOP: selezionare un runtime GPU."
)

RUNTIME_GPU = torch.cuda.get_device_name(0)
RUNTIME_TORCH = str(torch.__version__)
RUNTIME_PYTHON = platform.python_version()


runtime_identity_gate = (

    RUNTIME_GPU
    == "NVIDIA A100-SXM4-40GB"

    and RUNTIME_TORCH
    == "2.11.0+cu128"

    and RUNTIME_PYTHON
    == "3.13.15"
)


assert runtime_identity_gate, (
    "STOP: runtime diverso da quello della run primaria."
)


torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ============================================================
# PATHS
# ============================================================

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

PRIMARY_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_symmetric_v1"
)

HANDOFF_PATH = (
    SAKAI224_ROOT
    / "iic"
    / "audits"
    / "sakai224_full_iic_symmetric_v1_after_A6B.json"
)

LEGACY_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)


PRIMARY_FILES = {

    "latest.pt":
        (
            "354594985397fe3058fee78db859d96cd"
            "2e4eaeb482d7b47c3390b8a6da1f2f7"
        ),

    "final.pt":
        (
            "018008df8e6a1f97195cb1cc234e91503"
            "6806b2d3c1d09efefbf4333fe7f1199"
        ),

    "metrics.csv":
        (
            "78492b88b7e860dda1f986d666eba8a3"
            "c67a44411985911c48b85a6931b53851"
        ),

    "run_config.json":
        (
            "6a34e279824042a12be112ba04ce0fb4c"
            "30f500c901921a8d2aee5ab7b817aca"
        ),

    "runtime_sessions.jsonl":
        (
            "db68f87ad38dfab6120af13f71eb047fa"
            "c97992bdcf7f20a47e11e456aedc583"
        ),
}


EXPECTED_HANDOFF_SHA256 = (
    "8154f74e4bd34d086d99c706c3e65cf2"
    "d67ba8da1b16b3308b360264bc2bd1db"
)


# ============================================================
# SHA-256
# ============================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


# ============================================================
# PERSISTENT FILE AUDIT
# ============================================================

primary_presence_gate = (

    PRIMARY_RUN_DIR.is_dir()

    and all(
        (
            PRIMARY_RUN_DIR
            / filename
        ).is_file()

        for filename
        in PRIMARY_FILES
    )

    and HANDOFF_PATH.is_file()
)


assert primary_presence_gate


primary_hashes = {

    filename:
        sha256_file(
            PRIMARY_RUN_DIR
            / filename
        )

    for filename
    in PRIMARY_FILES
}


primary_hash_gate = all(

    primary_hashes[
        filename
    ]
    == expected_hash

    for filename, expected_hash
    in PRIMARY_FILES.items()
)


handoff_hash_gate = (

    sha256_file(
        HANDOFF_PATH
    )
    == EXPECTED_HANDOFF_SHA256
)


no_partial_gate = (

    len(
        list(
            PRIMARY_RUN_DIR.rglob(
                "*.partial"
            )
        )
    )
    == 0
)


assert primary_hash_gate
assert handoff_hash_gate
assert no_partial_gate


# ============================================================
# HANDOFF SEMANTICS
# ============================================================

handoff = json.loads(
    HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)


handoff_gate = (

    handoff[
        "status"
    ] == "SAFE_TO_STOP_RUNTIME"

    and handoff[
        "primary_run"
    ][
        "run_id"
    ] == "sakai224_full_iic_symmetric_v1"

    and handoff[
        "primary_run"
    ][
        "epochs_completed"
    ] == 100

    and handoff[
        "primary_run"
    ][
        "optimizer_steps_per_classifier"
    ] == 6800

    and handoff[
        "primary_run"
    ][
        "audit_pass"
    ] is True

    and handoff[
        "next_required_step"
    ][
        "run_id"
    ] == "sakai224_full_iic_legacy_text_v1"

    and handoff[
        "next_required_step"
    ][
        "do_not_read_labels"
    ] is True
)


assert handoff_gate


# ============================================================
# CHECKPOINT SEMANTICS
# ============================================================

latest_payload = torch.load(
    PRIMARY_RUN_DIR
    / "latest.pt",
    map_location="cpu",
    weights_only=True,
)


final_payload = torch.load(
    PRIMARY_RUN_DIR
    / "final.pt",
    map_location="cpu",
    weights_only=True,
)


checkpoint_gate = all([

    latest_payload[
        "kind"
    ] == "LATEST",

    final_payload[
        "kind"
    ] == "FINAL",

    latest_payload[
        "run_id"
    ] == "sakai224_full_iic_symmetric_v1",

    final_payload[
        "run_id"
    ] == "sakai224_full_iic_symmetric_v1",

    latest_payload[
        "last_completed_epoch"
    ] == 99,

    final_payload[
        "last_completed_epoch"
    ] == 99,

    latest_payload[
        "optimizer_steps_per_classifier"
    ] == [
        6800,
        6800,
        6800,
        6800,
        6800,
    ],

    final_payload[
        "optimizer_steps_per_classifier"
    ] == [
        6800,
        6800,
        6800,
        6800,
        6800,
    ],

    latest_payload[
        "classifier_states_sha256"
    ]
    == final_payload[
        "classifier_states_sha256"
    ],

    latest_payload[
        "optimizer_states_sha256"
    ]
    == final_payload[
        "optimizer_states_sha256"
    ],

    final_payload[
        "labels_used"
    ] is False,
])


safe_serialization_gate = (

    isinstance(
        final_payload[
            "contracts"
        ].get(
            "safe_serialization"
        ),
        str,
    )

    and all(
        value is None
        or type(value) is str

        for value
        in final_payload[
            "runtime"
        ].values()
    )
)


legacy_directory_absent_gate = (
    not LEGACY_RUN_DIR.exists()
)


assert checkpoint_gate
assert safe_serialization_gate
assert legacy_directory_absent_gate


# ============================================================
# FINAL RECOVERY GATE
# ============================================================

r0_gate = all([

    runtime_identity_gate,
    primary_presence_gate,
    primary_hash_gate,
    handoff_hash_gate,
    handoff_gate,
    checkpoint_gate,
    safe_serialization_gate,
    no_partial_gate,
    legacy_directory_absent_gate,
])


assert r0_gate


print()
print(
    "=== NOTEBOOK 06 CONTROLLED RUNTIME RECOVERY — R0 ==="
)

print()

print(
    "PYTHON =",
    RUNTIME_PYTHON,
)

print(
    "PYTORCH =",
    RUNTIME_TORCH,
)

print(
    "GPU =",
    RUNTIME_GPU,
)

print()

print(
    "PRIMARY_RUN_VERIFIED =",
    primary_hash_gate,
)

print(
    "PRIMARY_FINAL_EPOCH =",
    final_payload[
        "last_completed_epoch"
    ],
)

print(
    "PRIMARY_OPTIMIZER_STEPS =",
    final_payload[
        "optimizer_steps_per_classifier"
    ],
)

print(
    "LATEST_FINAL_STATE_EXACT =",
    checkpoint_gate,
)

print()

print(
    "HANDOFF_SHA256 =",
    sha256_file(
        HANDOFF_PATH
    ),
)

print(
    "HANDOFF_GATE =",
    handoff_gate,
)

print(
    "LEGACY_RUN_DIRECTORY_ABSENT =",
    legacy_directory_absent_gate,
)

print()

print(
    "LABELS_READ =",
    False,
)

print(
    "TRAINING_STARTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print()

print(
    "NOTEBOOK_06_R0_CONTROLLED_RECOVERY_PASS =",
    r0_gate,
)

Mounted at /content/drive

=== NOTEBOOK 06 CONTROLLED RUNTIME RECOVERY — R0 ===

PYTHON = 3.13.15
PYTORCH = 2.11.0+cu128
GPU = NVIDIA A100-SXM4-40GB

PRIMARY_RUN_VERIFIED = True
PRIMARY_FINAL_EPOCH = 99
PRIMARY_OPTIMIZER_STEPS = [6800, 6800, 6800, 6800, 6800]
LATEST_FINAL_STATE_EXACT = True

HANDOFF_SHA256 = 8154f74e4bd34d086d99c706c3e65cf2d67ba8da1b16b3308b360264bc2bd1db
HANDOFF_GATE = True
LEGACY_RUN_DIRECTORY_ABSENT = True

LABELS_READ = False
TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_R0_CONTROLLED_RECOVERY_PASS = True


In [2]:
# ============================================================
# NOTEBOOK 06 — RECOVERY BRIDGE FOR A4A / A4B
# STEP 06-R1
#
# READ ONLY.
# NO extraction yet.
# NO model load.
# NO training.
# ============================================================

from pathlib import Path
import csv

import torch


assert r0_gate
assert torch.cuda.is_available()


runtime_device = torch.device(
    "cuda:0"
)


# ============================================================
# CANONICAL TRAINING CONTRACT IDENTITY
# ============================================================

IIC_FINAL_TRAINING_SHA256 = (
    "4f27a9bbf4290dff54767655d2025928"
    "c2eac751607d217925b0d5bba2b2f537"
)


# ============================================================
# PERSISTENT PATHS
# ============================================================

TRANSFER_V1 = (
    SAKAI224_ROOT
    / "transfer_v1"
)

OFFICIAL_RUN_DIR = (
    SAKAI224_ROOT
    / "runs"
    / "sakai224_full_v1"
)

FINAL_CHECKPOINT_PATH = (
    OFFICIAL_RUN_DIR
    / "final.pt"
)

EMBEDDING_DIR = (
    SAKAI224_ROOT
    / "embeddings"
    / "sakai224_full_mu_epoch100_v1"
)


OUT272_TAR_PATH = (
    TRANSFER_V1
    / "sakai224_full_out272_gray.tar"
)

EVENT_MANIFEST_PATH = (
    TRANSFER_V1
    / "manifest_events_272.csv"
)

TRAIN_SPLIT_PATH = (
    TRANSFER_V1
    / "split_train_indices_seed123.csv"
)

VALIDATION_SPLIT_PATH = (
    TRANSFER_V1
    / "split_val_indices_seed123.csv"
)

TRAIN_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_train_f32.bin"
)

VALIDATION_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_validation_f32.bin"
)

TRAIN_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_train.csv"
)

VALIDATION_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_validation.csv"
)


# ============================================================
# SOURCE HASHES
# ============================================================

RECOVERY_SOURCE_HASHES = {

    FINAL_CHECKPOINT_PATH:
        (
            "2bb6fd2ce2f73f8f9e479ed6969a52c1"
            "cdd41c1ae6e2cfc33da687ebba798f71"
        ),

    OUT272_TAR_PATH:
        (
            "70c766c3652f586c8da4c8cadf134a8c"
            "e83cfd07c8cd10545367f380017ddab2"
        ),

    EVENT_MANIFEST_PATH:
        (
            "620960e6b35da31bc4cf1a8a6da86790"
            "27d378e1d4b2ac73c5267c1c5311ae82"
        ),

    TRAIN_MU_PATH:
        (
            "b620fd9711719a7bd40293b6536ea6fe"
            "b235cbadb1d1e37dca0e46651332ad5e"
        ),

    VALIDATION_MU_PATH:
        (
            "c1692b3e7a4c86a7f4d6d7172117e110"
            "83d2bd2394947838a24d1e2b81f27cf9"
        ),

    TRAIN_ROW_MAP_PATH:
        (
            "4b25c7f41a945fc8195e93c11e60da3b"
            "2e5b8be8298f6919008fc57dc39cc9e8"
        ),

    VALIDATION_ROW_MAP_PATH:
        (
            "f5a0e48c79db4d466f834cbc54bdbf7e"
            "8c34d3fa45f545f58b37c71627c97b48"
        ),
}


source_presence_gate = all(
    path.is_file()
    for path in RECOVERY_SOURCE_HASHES
)


source_hash_gate = all(

    sha256_file(
        path
    )
    == expected

    for path, expected
    in RECOVERY_SOURCE_HASHES.items()
)


assert source_presence_gate
assert source_hash_gate


# ============================================================
# LABEL-FREE ROW MAPS
# ============================================================

EXPECTED_ROW_MAP_FIELDS = [
    "split",
    "split_pos",
    "manifest_index",
    "event_index",
    "event_id",
]


def read_recovery_row_map(
    path,
):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        assert list(
            reader.fieldnames
        ) == EXPECTED_ROW_MAP_FIELDS

        return list(
            reader
        )


train_rows = read_recovery_row_map(
    TRAIN_ROW_MAP_PATH
)

validation_rows = read_recovery_row_map(
    VALIDATION_ROW_MAP_PATH
)


train_manifest_indices = [
    int(
        row["manifest_index"]
    )
    for row in train_rows
]

validation_manifest_indices = [
    int(
        row["manifest_index"]
    )
    for row in validation_rows
]


row_map_gate = (

    len(
        train_rows
    ) == 6866

    and len(
        validation_rows
    ) == 1717

    and all(
        row["split"] == "train"
        and int(row["split_pos"]) == position
        and int(row["event_index"])
            == int(row["manifest_index"]) + 1

        for position, row
        in enumerate(train_rows)
    )

    and all(
        row["split"] == "validation"
        and int(row["split_pos"]) == position
        and int(row["event_index"])
            == int(row["manifest_index"]) + 1

        for position, row
        in enumerate(validation_rows)
    )

    and set(
        train_manifest_indices
    ).isdisjoint(
        validation_manifest_indices
    )

    and set(
        train_manifest_indices
        + validation_manifest_indices
    ) == set(
        range(8583)
    )
)


assert row_map_gate


r1_gate = all([
    source_presence_gate,
    source_hash_gate,
    row_map_gate,
    runtime_device.type == "cuda",
])


assert r1_gate


print()
print(
    "=== NOTEBOOK 06 RECOVERY BRIDGE — R1 ==="
)

print(
    "SOURCE_PRESENCE_GATE =",
    source_presence_gate,
)

print(
    "SOURCE_HASH_GATE =",
    source_hash_gate,
)

print(
    "TRAIN_ROWS =",
    len(train_rows),
)

print(
    "VALIDATION_ROWS =",
    len(validation_rows),
)

print(
    "ROW_MAP_GATE =",
    row_map_gate,
)

print(
    "RUNTIME_DEVICE =",
    runtime_device,
)

print(
    "TRAINING_STARTED = False"
)

print(
    "DRIVE_MODIFIED = False"
)

print()

print(
    "NOTEBOOK_06_R1_RECOVERY_BRIDGE_PASS =",
    r1_gate,
)


=== NOTEBOOK 06 RECOVERY BRIDGE — R1 ===
SOURCE_PRESENCE_GATE = True
SOURCE_HASH_GATE = True
TRAIN_ROWS = 6866
VALIDATION_ROWS = 1717
ROW_MAP_GATE = True
RUNTIME_DEVICE = cuda:0
TRAINING_STARTED = False
DRIVE_MODIFIED = False

NOTEBOOK_06_R1_RECOVERY_BRIDGE_PASS = True


In [1]:
# ============================================================
# NOTEBOOK 06 — R0 CONTROLLED RUNTIME RECOVERY
#
# READ ONLY.
#
# Scopo:
# - montare Google Drive;
# - verificare il runtime;
# - verificare byte-per-byte gli artefatti persistenti
#   della IIC primaria symmetric;
# - verificare l'handoff A6C;
# - controllare che il run legacy ufficiale non esista ancora.
#
# NO dataset extraction.
# NO labels.
# NO VAE/model creation.
# NO optimizer.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys

import torch


# ============================================================
# 2. PERSISTENT PATHS — APPROVED STATE
# ============================================================

ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

IIC_ROOT = ROOT / "iic"

PRIMARY_RUN_ID = (
    "sakai224_full_iic_symmetric_v1"
)

LEGACY_RUN_ID = (
    "sakai224_full_iic_legacy_text_v1"
)

PRIMARY_DIR = (
    IIC_ROOT
    / PRIMARY_RUN_ID
)

LEGACY_DIR = (
    IIC_ROOT
    / LEGACY_RUN_ID
)

AUDIT_DIR = (
    IIC_ROOT
    / "audits"
)

PRIMARY_HANDOFF = (
    AUDIT_DIR
    / "sakai224_full_iic_symmetric_v1_after_A6B.json"
)


# ============================================================
# 3. AUTHORITATIVE SHA-256 — RECAP 15 / A6B-A6C
# ============================================================

EXPECTED_SHA256 = {

    PRIMARY_DIR / "latest.pt":
        "354594985397fe3058fee78db859d96cd"
        "2e4eaeb482d7b47c3390b8a6da1f2f7",

    PRIMARY_DIR / "final.pt":
        "018008df8e6a1f97195cb1cc234e91503"
        "6806b2d3c1d09efefbf4333fe7f1199",

    PRIMARY_DIR / "metrics.csv":
        "78492b88b7e860dda1f986d666eba8a3c"
        "67a44411985911c48b85a6931b53851",

    PRIMARY_DIR / "run_config.json":
        "6a34e279824042a12be112ba04ce0fb4c"
        "30f500c901921a8d2aee5ab7b817aca",

    PRIMARY_DIR / "runtime_sessions.jsonl":
        "db68f87ad38dfab6120af13f71eb047fa"
        "c97992bdcf7f20a47e11e456aedc583",

    PRIMARY_HANDOFF:
        "8154f74e4bd34d086d99c706c3e65cf"
        "2d67ba8da1b16b3308b360264bc2bd1db",
}


# ============================================================
# 4. SHA HELPER
# ============================================================

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ============================================================
# 5. RUNTIME INVENTORY
# ============================================================

python_version = platform.python_version()
torch_version = torch.__version__
cuda_runtime = torch.version.cuda
cuda_available = torch.cuda.is_available()

gpu_name = None
gpu_total_memory = None

if cuda_available:

    gpu_name = torch.cuda.get_device_name(0)

    gpu_total_memory = (
        torch.cuda.get_device_properties(0).total_memory
    )


print()
print("=== NOTEBOOK 06 — R0 CONTROLLED RECOVERY ===")
print()

print("--- RUNTIME ---")

print(
    "PYTHON_VERSION =",
    python_version,
)

print(
    "PYTORCH_VERSION =",
    torch_version,
)

print(
    "TORCH_CUDA_RUNTIME =",
    cuda_runtime,
)

print(
    "CUDA_AVAILABLE =",
    cuda_available,
)

print(
    "GPU_NAME =",
    gpu_name,
)

print(
    "GPU_TOTAL_MEMORY_BYTES =",
    gpu_total_memory,
)


# Informazione NVIDIA aggiuntiva, diagnostica soltanto.
try:

    nvidia_smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        text=True,
    ).strip()

except Exception as exc:

    nvidia_smi = (
        f"<nvidia-smi unavailable: {exc}>"
    )

print(
    "NVIDIA_SMI =",
    nvidia_smi,
)


# ============================================================
# 6. RUNTIME GATES
#
# La primaria ufficiale è stata eseguita con:
# Python 3.13.15
# PyTorch 2.11.0+cu128
# NVIDIA A100-SXM4-40GB
# ============================================================

python_gate = (
    python_version == "3.13.15"
)

torch_gate = (
    torch_version == "2.11.0+cu128"
)

cuda_gate = (
    cuda_available
    and cuda_runtime == "12.8"
)

gpu_gate = (
    gpu_name == "NVIDIA A100-SXM4-40GB"
)


print()
print("--- RUNTIME GATES ---")

print(
    "PYTHON_GATE =",
    python_gate,
)

print(
    "PYTORCH_GATE =",
    torch_gate,
)

print(
    "CUDA_GATE =",
    cuda_gate,
)

print(
    "GPU_GATE =",
    gpu_gate,
)


# ============================================================
# 7. PRIMARY ARTIFACT HASH AUDIT
# ============================================================

print()
print("--- PRIMARY SYMMETRIC ARTIFACTS ---")

artifact_gates = {}

for path, expected_sha in EXPECTED_SHA256.items():

    exists = path.is_file()

    actual_sha = (
        sha256_file(path)
        if exists
        else None
    )

    gate = (
        exists
        and actual_sha == expected_sha
    )

    artifact_gates[str(path)] = gate

    print()
    print(
        "FILE =",
        path,
    )

    print(
        "EXISTS =",
        exists,
    )

    print(
        "EXPECTED_SHA256 =",
        expected_sha,
    )

    print(
        "ACTUAL_SHA256 =",
        actual_sha,
    )

    print(
        "SHA256_GATE =",
        gate,
    )


primary_artifacts_gate = all(
    artifact_gates.values()
)


# ============================================================
# 8. HANDOFF — READ ONLY
# ============================================================

handoff_json = None

if PRIMARY_HANDOFF.is_file():

    with open(
        PRIMARY_HANDOFF,
        "r",
        encoding="utf-8",
    ) as f:

        handoff_json = json.load(f)


print()
print("--- A6C HANDOFF ---")

print(
    "HANDOFF_EXISTS =",
    PRIMARY_HANDOFF.is_file(),
)

print(
    "HANDOFF_JSON_TYPE =",
    (
        None
        if handoff_json is None
        else type(handoff_json).__name__
    ),
)

print(
    "HANDOFF_TOP_LEVEL_KEYS =",
    (
        None
        if not isinstance(handoff_json, dict)
        else sorted(handoff_json.keys())
    ),
)


# ============================================================
# 9. LEGACY RUN COLLISION GUARD
# ============================================================

legacy_exists = LEGACY_DIR.exists()

legacy_entries = []

if legacy_exists:

    legacy_entries = sorted(
        p.name
        for p in LEGACY_DIR.iterdir()
    )


legacy_clean_gate = (
    not legacy_exists
)


print()
print("--- LEGACY RUN COLLISION GUARD ---")

print(
    "LEGACY_RUN_ID =",
    LEGACY_RUN_ID,
)

print(
    "LEGACY_RUN_DIR =",
    LEGACY_DIR,
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    legacy_exists,
)

print(
    "LEGACY_RUN_DIR_ENTRIES =",
    legacy_entries,
)

print(
    "LEGACY_CLEAN_GATE =",
    legacy_clean_gate,
)


# ============================================================
# 10. SAFETY FLAGS FOR THIS FRESH RUNTIME
# ============================================================

LABELS_READ = False

print()
print("--- SAFETY STATE ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_DATA_RESTORED =",
    False,
)

print(
    "VAE_ENCODER_RESTORED =",
    False,
)

print(
    "IIC_MODEL_CREATED =",
    False,
)

print(
    "OPTIMIZER_CREATED =",
    False,
)

print(
    "IIC_TRAINING_STARTED =",
    False,
)

print(
    "SPECTRAL_STARTED =",
    False,
)

print(
    "PRIMARY_SYMMETRIC_RERUN =",
    False,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    False,
)


# ============================================================
# 11. FINAL R0 GATE
# ============================================================

r0_gate = all([
    python_gate,
    torch_gate,
    cuda_gate,
    gpu_gate,
    primary_artifacts_gate,
    PRIMARY_HANDOFF.is_file(),
    legacy_clean_gate,
    LABELS_READ is False,
])


print()
print("--- FINAL GATE ---")

print(
    "PRIMARY_ARTIFACTS_GATE =",
    primary_artifacts_gate,
)

print(
    "R0_CONTROLLED_RECOVERY_PASS =",
    r0_gate,
)


assert r0_gate, (
    "R0 FAILED. Fermarsi qui. "
    "Non estrarre dati, non ricostruire il VAE "
    "e non creare il run legacy."
)


print()
print(
    "READY_FOR_A4A_DATA_RECOVERY =",
    True,
)

print(
    "READY_FOR_A4B_ENCODER_RECOVERY =",
    False,
)

print(
    "READY_FOR_A4C_IIC_DEFINITION_RECOVERY =",
    False,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    False,
)

Mounted at /content/drive

=== NOTEBOOK 06 — R0 CONTROLLED RECOVERY ===

--- RUNTIME ---
PYTHON_VERSION = 3.13.15
PYTORCH_VERSION = 2.11.0+cu128
TORCH_CUDA_RUNTIME = 12.8
CUDA_AVAILABLE = True
GPU_NAME = NVIDIA A100-SXM4-40GB
GPU_TOTAL_MEMORY_BYTES = 42405855232
NVIDIA_SMI = NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07

--- RUNTIME GATES ---
PYTHON_GATE = True
PYTORCH_GATE = True
CUDA_GATE = True
GPU_GATE = True

--- PRIMARY SYMMETRIC ARTIFACTS ---

FILE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_symmetric_v1/latest.pt
EXISTS = True
EXPECTED_SHA256 = 354594985397fe3058fee78db859d96cd2e4eaeb482d7b47c3390b8a6da1f2f7
ACTUAL_SHA256 = 354594985397fe3058fee78db859d96cd2e4eaeb482d7b47c3390b8a6da1f2f7
SHA256_GATE = True

FILE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_symmetric_v1/final.pt
EXISTS = True
EXPECTED_SHA256 = 018008df8e6a1f97195cb1cc234e915036806b2d3c1d09efefbf4333fe7f1199
ACTUAL_SHA256 = 018008df8e6a1f97195cb1cc234e9

In [4]:
# ============================================================
# NOTEBOOK 06 — A4A CONTROLLED DATA RECOVERY
#
# Ricostruisce SOLO:
#   - out272_gray nella root runtime locale
#   - popolazione IIC label-free in manifest order
#   - full_central_mu (8583, 512)
#
# NO model loading.
# NO label values accessed.
# NO IIC model.
# NO optimizer.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path, PurePosixPath
import csv
import hashlib
import shutil
import tarfile

import numpy as np


# ============================================================
# 0. PARENT GATE
# ============================================================

assert (
    "R0_CONTROLLED_RECOVERY_PASS" in globals()
    and R0_CONTROLLED_RECOVERY_PASS is True
), "R0 non disponibile o non PASS."


# ============================================================
# 1. PERSISTENT SOURCES
# ============================================================

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

TRANSFER_V1 = (
    SAKAI224_ROOT
    / "transfer_v1"
)

EMBEDDING_DIR = (
    SAKAI224_ROOT
    / "embeddings"
    / "sakai224_full_mu_epoch100_v1"
)

OUT272_TAR_PATH = (
    TRANSFER_V1
    / "sakai224_full_out272_gray.tar"
)

EVENT_MANIFEST_PATH = (
    TRANSFER_V1
    / "manifest_events_272.csv"
)

TRAIN_SPLIT_PATH = (
    TRANSFER_V1
    / "split_train_indices_seed123.csv"
)

VALIDATION_SPLIT_PATH = (
    TRANSFER_V1
    / "split_val_indices_seed123.csv"
)

TRAIN_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_train_f32.bin"
)

VALIDATION_MU_PATH = (
    EMBEDDING_DIR
    / "embeddings_mu_validation_f32.bin"
)

TRAIN_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_train.csv"
)

VALIDATION_ROW_MAP_PATH = (
    EMBEDDING_DIR
    / "row_map_validation.csv"
)


# ============================================================
# 2. LOCAL CACHE
#
# La root runtime locale è derivata e temporanea.
# ============================================================

LOCAL_DATA_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

OUT272_DIR = (
    LOCAL_DATA_ROOT
    / "out272_gray"
)


# ============================================================
# 3. FROZEN IDENTITIES
# ============================================================

EXPECTED_SOURCE_HASHES = {

    OUT272_TAR_PATH:
        (
            "70c766c3652f586c8da4c8cadf134a8c"
            "e83cfd07c8cd10545367f380017ddab2"
        ),

    EVENT_MANIFEST_PATH:
        (
            "620960e6b35da31bc4cf1a8a6da86790"
            "27d378e1d4b2ac73c5267c1c5311ae82"
        ),

    TRAIN_SPLIT_PATH:
        (
            "8c7a513b98dbe996261bc80965500ab15"
            "ab77f498ec9440858003152e42f926f"
        ),

    VALIDATION_SPLIT_PATH:
        (
            "e63109c88df7b3ee97317f94791f26df"
            "3fcaf8b937921cfb27967d01c2546467"
        ),

    TRAIN_MU_PATH:
        (
            "b620fd9711719a7bd40293b6536ea6fe"
            "b235cbadb1d1e37dca0e46651332ad5e"
        ),

    VALIDATION_MU_PATH:
        (
            "c1692b3e7a4c86a7f4d6d7172117e110"
            "83d2bd2394947838a24d1e2b81f27cf9"
        ),

    TRAIN_ROW_MAP_PATH:
        (
            "4b25c7f41a945fc8195e93c11e60da3b"
            "2e5b8be8298f6919008fc57dc39cc9e8"
        ),

    VALIDATION_ROW_MAP_PATH:
        (
            "f5a0e48c79db4d466f834cbc54bdbf7e"
            "8c34d3fa45f545f58b37c71627c97b48"
        ),
}


EXPECTED_OUT272_FILE_COUNT = 34332
EXPECTED_OUT272_BYTES = 1144867540

EXPECTED_POPULATION_SHA256 = (
    "bd95286ec6c70cb2b5c674a8a46f3fd"
    "450048367d91dc17fd3625977c538b8f4"
)

EXPECTED_FULL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)


# ============================================================
# 4. SHA HELPER
# ============================================================

def recovery_sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ============================================================
# 5. SOURCE HASH GATE
# ============================================================

source_presence_gate = all(
    path.is_file()
    for path in EXPECTED_SOURCE_HASHES
)

assert source_presence_gate


calculated_source_hashes = {
    path: recovery_sha256_file(path)
    for path in EXPECTED_SOURCE_HASHES
}


source_hash_gate = all(
    calculated_source_hashes[path]
    == expected
    for path, expected
    in EXPECTED_SOURCE_HASHES.items()
)

assert source_hash_gate


# ============================================================
# 6. TAR AUDIT BEFORE EXTRACTION
# ============================================================

with tarfile.open(
    OUT272_TAR_PATH,
    mode="r",
) as tar_handle:

    tar_members = tar_handle.getmembers()


tar_file_members = [
    member
    for member in tar_members
    if member.isfile()
]


def tar_member_safe(member):
    p = PurePosixPath(member.name)

    return (
        not p.is_absolute()
        and ".." not in p.parts
        and not member.issym()
        and not member.islnk()
    )


tar_path_gate = all(
    tar_member_safe(member)
    for member in tar_members
)


tar_layout_gate = all(
    (
        len(PurePosixPath(member.name).parts) >= 2
        and PurePosixPath(member.name).parts[0]
            == "out272_gray"
        and PurePosixPath(member.name).suffix.lower()
            == ".png"
    )
    for member in tar_file_members
)


tar_count_gate = (
    len(tar_file_members)
    == EXPECTED_OUT272_FILE_COUNT
)


tar_bytes = sum(
    member.size
    for member in tar_file_members
)

tar_bytes_gate = (
    tar_bytes
    == EXPECTED_OUT272_BYTES
)


tar_gate = all([
    tar_path_gate,
    tar_layout_gate,
    tar_count_gate,
    tar_bytes_gate,
])

assert tar_gate


# ============================================================
# 7. IDEMPOTENT LOCAL EXTRACTION
# ============================================================

def measure_out272(directory):
    if not directory.is_dir():
        return 0, 0, False

    files = [
        path
        for path in directory.rglob("*")
        if path.is_file()
    ]

    total_bytes = sum(
        path.stat().st_size
        for path in files
    )

    png_only = all(
        path.suffix.lower() == ".png"
        for path in files
    )

    return (
        len(files),
        total_bytes,
        png_only,
    )


local_count_before, local_bytes_before, local_png_before = (
    measure_out272(
        OUT272_DIR
    )
)


local_before_complete = (
    local_count_before
        == EXPECTED_OUT272_FILE_COUNT
    and local_bytes_before
        == EXPECTED_OUT272_BYTES
    and local_png_before
)


if not local_before_complete:

    if OUT272_DIR.exists():
        shutil.rmtree(
            OUT272_DIR
        )

    LOCAL_DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tarfile.open(
        OUT272_TAR_PATH,
        mode="r",
    ) as tar_handle:

        tar_handle.extractall(
            path=LOCAL_DATA_ROOT,
            filter="data",
        )


local_count, local_bytes, local_png_only = (
    measure_out272(
        OUT272_DIR
    )
)


out272_gate = (
    local_count
        == EXPECTED_OUT272_FILE_COUNT

    and local_bytes
        == EXPECTED_OUT272_BYTES

    and local_png_only
)


assert out272_gate


# ============================================================
# 8. LABEL-FREE ROW MAPS
# ============================================================

EXPECTED_ROW_MAP_FIELDS = [
    "split",
    "split_pos",
    "manifest_index",
    "event_index",
    "event_id",
]


def read_label_free_row_map(path):

    with Path(path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:

        reader = csv.DictReader(
            handle
        )

        assert list(
            reader.fieldnames
        ) == EXPECTED_ROW_MAP_FIELDS

        return list(reader)


train_rows = read_label_free_row_map(
    TRAIN_ROW_MAP_PATH
)

validation_rows = read_label_free_row_map(
    VALIDATION_ROW_MAP_PATH
)


assert len(train_rows) == 6866
assert len(validation_rows) == 1717


# ============================================================
# 9. RECONSTRUCT FULL IIC POPULATION
#    IN manifest_index ORDER
# ============================================================

full_iic_rows = [
    None
] * 8583


def insert_population_rows(
    rows,
    expected_split,
):

    for expected_split_pos, row in enumerate(rows):

        manifest_index = int(
            row["manifest_index"]
        )

        event_index = int(
            row["event_index"]
        )

        split_pos = int(
            row["split_pos"]
        )

        event_id = row[
            "event_id"
        ]

        assert (
            row["split"]
            == expected_split
        )

        assert (
            split_pos
            == expected_split_pos
        )

        assert (
            event_index
            == manifest_index + 1
        )

        assert (
            0 <= manifest_index < 8583
        )

        assert (
            full_iic_rows[
                manifest_index
            ]
            is None
        )

        assert event_id != ""

        full_iic_rows[
            manifest_index
        ] = {

            "manifest_index":
                manifest_index,

            "event_index":
                event_index,

            "event_id":
                event_id,

            "source_vae_split":
                expected_split,

            "source_vae_split_pos":
                split_pos,
        }


insert_population_rows(
    train_rows,
    "train",
)

insert_population_rows(
    validation_rows,
    "validation",
)


population_gate = (
    all(
        row is not None
        for row in full_iic_rows
    )

    and [
        row["manifest_index"]
        for row in full_iic_rows
    ] == list(range(8583))

    and len({
        row["event_id"]
        for row in full_iic_rows
    }) == 8583

    and sum(
        row["source_vae_split"] == "train"
        for row in full_iic_rows
    ) == 6866

    and sum(
        row["source_vae_split"] == "validation"
        for row in full_iic_rows
    ) == 1717
)

assert population_gate


population_identity_text = "\n".join(

    (
        f"{row['manifest_index']},"
        f"{row['event_index']},"
        f"{row['event_id']},"
        f"{row['source_vae_split']},"
        f"{row['source_vae_split_pos']}"
    )

    for row in full_iic_rows

) + "\n"


FULL_IIC_POPULATION_SHA256 = hashlib.sha256(
    population_identity_text.encode(
        "utf-8"
    )
).hexdigest()


population_sha_gate = (
    FULL_IIC_POPULATION_SHA256
    == EXPECTED_POPULATION_SHA256
)

assert population_sha_gate


# ============================================================
# 10. LOAD PUBLISHED CENTRAL MU
# ============================================================

LATENT_DIM = 512


def load_mu_matrix(
    path,
    row_count,
):

    values = np.fromfile(
        path,
        dtype="<f4",
    )

    expected_values = (
        row_count
        * LATENT_DIM
    )

    assert (
        values.size
        == expected_values
    )

    matrix = values.reshape(
        row_count,
        LATENT_DIM,
    )

    assert (
        matrix.dtype
        == np.float32
    )

    assert np.isfinite(
        matrix
    ).all()

    return matrix


train_mu = load_mu_matrix(
    TRAIN_MU_PATH,
    6866,
)

validation_mu = load_mu_matrix(
    VALIDATION_MU_PATH,
    1717,
)


# ============================================================
# 11. REORDER CENTRAL MU INTO MANIFEST ORDER
# ============================================================

full_central_mu = np.empty(
    (
        8583,
        LATENT_DIM,
    ),
    dtype=np.float32,
)

mu_seen = np.zeros(
    8583,
    dtype=bool,
)


for split_pos, row in enumerate(
    train_rows
):
    manifest_index = int(
        row["manifest_index"]
    )

    assert not mu_seen[
        manifest_index
    ]

    full_central_mu[
        manifest_index
    ] = train_mu[
        split_pos
    ]

    mu_seen[
        manifest_index
    ] = True


for split_pos, row in enumerate(
    validation_rows
):
    manifest_index = int(
        row["manifest_index"]
    )

    assert not mu_seen[
        manifest_index
    ]

    full_central_mu[
        manifest_index
    ] = validation_mu[
        split_pos
    ]

    mu_seen[
        manifest_index
    ] = True


mu_shape_gate = (
    mu_seen.all()

    and full_central_mu.shape
        == (8583, 512)

    and full_central_mu.dtype
        == np.float32

    and full_central_mu.flags[
        "C_CONTIGUOUS"
    ]

    and np.isfinite(
        full_central_mu
    ).all()
)

assert mu_shape_gate


FULL_CENTRAL_MU_SHA256 = hashlib.sha256(
    full_central_mu.tobytes(
        order="C"
    )
).hexdigest()


full_mu_sha_gate = (
    FULL_CENTRAL_MU_SHA256
    == EXPECTED_FULL_MU_SHA256
)

assert full_mu_sha_gate


# ============================================================
# 12. SAFETY STATE
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

DATA_EXTRACTED = True
GRAVITY_SPY_DATA_RESTORED = True

MODEL_LOADED = False
VAE_ENCODER_RESTORED = False

IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 13. FINAL A4A GATE
# ============================================================

a4a_gate = all([
    source_presence_gate,
    source_hash_gate,
    tar_gate,
    out272_gate,
    population_gate,
    population_sha_gate,
    mu_shape_gate,
    full_mu_sha_gate,
    LABELS_READ is False,
    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,
])


assert a4a_gate


# ============================================================
# 14. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4A CONTROLLED DATA RECOVERY ==="
)
print()

print("--- SOURCE GATES ---")
print(
    "SOURCE_PRESENCE_GATE =",
    source_presence_gate,
)
print(
    "SOURCE_HASH_GATE =",
    source_hash_gate,
)
print(
    "TAR_GATE =",
    tar_gate,
)

print()
print("--- OUT272 LOCAL CACHE ---")
print(
    "OUT272_DIR =",
    OUT272_DIR,
)
print(
    "OUT272_FILES =",
    local_count,
)
print(
    "OUT272_BYTES =",
    local_bytes,
)
print(
    "OUT272_GATE =",
    out272_gate,
)

print()
print("--- IIC POPULATION ---")
print(
    "IIC_EVENT_COUNT =",
    len(full_iic_rows),
)
print(
    "TRAIN_PROVENANCE_COUNT =",
    sum(
        row["source_vae_split"] == "train"
        for row in full_iic_rows
    ),
)
print(
    "VALIDATION_PROVENANCE_COUNT =",
    sum(
        row["source_vae_split"] == "validation"
        for row in full_iic_rows
    ),
)
print(
    "FULL_IIC_POPULATION_SHA256 =",
    FULL_IIC_POPULATION_SHA256,
)
print(
    "POPULATION_GATE =",
    population_gate,
)
print(
    "POPULATION_SHA_GATE =",
    population_sha_gate,
)

print()
print("--- CENTRAL MU ---")
print(
    "FULL_CENTRAL_MU_SHAPE =",
    full_central_mu.shape,
)
print(
    "FULL_CENTRAL_MU_DTYPE =",
    full_central_mu.dtype,
)
print(
    "FULL_CENTRAL_MU_FINITE =",
    np.isfinite(
        full_central_mu
    ).all(),
)
print(
    "FULL_CENTRAL_MU_SHA256 =",
    FULL_CENTRAL_MU_SHA256,
)
print(
    "FULL_MU_SHA_GATE =",
    full_mu_sha_gate,
)

print()
print("--- SAFETY ---")
print(
    "LABELS_READ =",
    LABELS_READ,
)
print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)
print(
    "MODEL_LOADED =",
    MODEL_LOADED,
)
print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)
print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)
print(
    "PRIMARY_SYMMETRIC_RERUN =",
    PRIMARY_SYMMETRIC_RERUN,
)
print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)

print()
print(
    "NOTEBOOK_06_A4A_DATA_RECOVERY_PASS =",
    a4a_gate,
)

print(
    "READY_FOR_A4B_ENCODER_RECOVERY =",
    True,
)

print(
    "READY_FOR_A4C_IIC_DEFINITION_RECOVERY =",
    False,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    False,
)


=== NOTEBOOK 06 — A4A CONTROLLED DATA RECOVERY ===

--- SOURCE GATES ---
SOURCE_PRESENCE_GATE = True
SOURCE_HASH_GATE = True
TAR_GATE = True

--- OUT272 LOCAL CACHE ---
OUT272_DIR = /content/sakai224_full_data/out272_gray
OUT272_FILES = 34332
OUT272_BYTES = 1144867540
OUT272_GATE = True

--- IIC POPULATION ---
IIC_EVENT_COUNT = 8583
TRAIN_PROVENANCE_COUNT = 6866
VALIDATION_PROVENANCE_COUNT = 1717
FULL_IIC_POPULATION_SHA256 = bd95286ec6c70cb2b5c674a8a46f3fd450048367d91dc17fd3625977c538b8f4
POPULATION_GATE = True
POPULATION_SHA_GATE = True

--- CENTRAL MU ---
FULL_CENTRAL_MU_SHAPE = (8583, 512)
FULL_CENTRAL_MU_DTYPE = float32
FULL_CENTRAL_MU_FINITE = True
FULL_CENTRAL_MU_SHA256 = fd55be588ef05df8afdf7ee048aa535e12eda37bd13ef8d59e5968fcfc9467f8
FULL_MU_SHA_GATE = True

--- SAFETY ---
LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False
MODEL_LOADED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False
PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = Fa

In [3]:
# ============================================================
# NOTEBOOK 06 — R0 VARIABLE ALIAS FIX
# Nessun I/O, nessun dato, nessun modello.
# ============================================================

print(
    "r0_gate present =",
    "r0_gate" in globals(),
)

print(
    "r0_gate value =",
    globals().get("r0_gate", None),
)

assert (
    "r0_gate" in globals()
    and r0_gate is True
), (
    "Il vero gate R0 non è presente nel runtime. "
    "In questo caso fermarsi e rieseguire R0."
)

R0_CONTROLLED_RECOVERY_PASS = bool(r0_gate)

assert R0_CONTROLLED_RECOVERY_PASS is True

print()
print(
    "R0_CONTROLLED_RECOVERY_PASS =",
    R0_CONTROLLED_RECOVERY_PASS,
)

print(
    "R0_ALIAS_FIX_PASS =",
    True,
)

r0_gate present = True
r0_gate value = True

R0_CONTROLLED_RECOVERY_PASS = True
R0_ALIAS_FIX_PASS = True


In [7]:
# ============================================================
# NOTEBOOK 06 — A4B-1
# CONTROLLED FINAL VAE / ENCODER RECOVERY
#
# SOURCE OF TRUTH:
#   Notebook 04 approved source on Drive.
#
# Executes verbatim:
#   STEP 04-R0
#   STEP 04-A2
#
# Then applies ONLY the Notebook-06 encoder policy:
#   - encoder alias
#   - frozen parameters
#   - eval mode
#   - encoder-state known-answer SHA
#
# NO Gravity Spy label access.
# NO IIC head creation.
# NO optimizer.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import torch


# ============================================================
# 0. PARENT GATE
# ============================================================

assert (
    "NOTEBOOK_06_A4A_DATA_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4A_DATA_RECOVERY_PASS is True
), (
    "A4A non disponibile o non PASS."
)


# ============================================================
# 1. LOCATE APPROVED NOTEBOOK 04 ON DRIVE
# ============================================================

NOTEBOOKS_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
)

preferred_notebook04 = (
    NOTEBOOKS_DIR
    / "04_sakai224_full_posttraining_audit_and_embeddings.ipynb"
)


if preferred_notebook04.is_file():

    NOTEBOOK04_PATH = preferred_notebook04

else:

    notebook04_candidates = sorted(
        NOTEBOOKS_DIR.glob(
            "04_sakai224_full_posttraining_audit_and_embeddings*.ipynb"
        )
    )

    if len(notebook04_candidates) != 1:

        raise RuntimeError(
            "Impossibile identificare univocamente Notebook 04. "
            f"Candidati trovati: {notebook04_candidates}"
        )

    NOTEBOOK04_PATH = notebook04_candidates[0]


assert NOTEBOOK04_PATH.is_file()


# ============================================================
# 2. LOAD NOTEBOOK SOURCE — NO EXECUTION YET
# ============================================================

with NOTEBOOK04_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    notebook04_json = json.load(
        handle
    )


def find_unique_code_cell(
    notebook,
    required_markers,
):

    matches = []

    for cell_index, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get(
            "cell_type"
        ) != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                [],
            )
        )

        if all(
            marker in source
            for marker in required_markers
        ):
            matches.append(
                (
                    cell_index,
                    source,
                )
            )

    if len(matches) != 1:

        raise RuntimeError(
            "Expected exactly one matching code cell for "
            f"{required_markers}; found {len(matches)}."
        )

    return matches[0]


r0_cell_index, r0_source = (
    find_unique_code_cell(
        notebook04_json,
        (
            "RUNTIME RECOVERY BOOTSTRAP",
            "STEP 04-R0",
        ),
    )
)


a2_cell_index, a2_source = (
    find_unique_code_cell(
        notebook04_json,
        (
            "REBUILD + LOAD OFFICIAL FINAL VAE",
            "STEP 04-A2",
        ),
    )
)


# ============================================================
# 3. SOURCE SHA GATE
#
# These SHA-256 values were computed from the exact source
# strings of the Notebook 04 reference copy used for this analysis.
#
# Execution stops if the Drive source differs.
# ============================================================

EXPECTED_R0_SOURCE_SHA256 = (
    "5660b2f12ff7c78b2f2bf043be061a77c"
    "a8ec3e90638671e33760ddf4089766b"
)

EXPECTED_A2_SOURCE_SHA256 = (
    "d3f6389bbf238deef0d69f8b3e1d7297"
    "877ddbe296e563f740ec4eb49a1f1908"
)


r0_source_sha256 = hashlib.sha256(
    r0_source.encode(
        "utf-8"
    )
).hexdigest()


a2_source_sha256 = hashlib.sha256(
    a2_source.encode(
        "utf-8"
    )
).hexdigest()


source_match_gate = (
    r0_source_sha256
        == EXPECTED_R0_SOURCE_SHA256

    and a2_source_sha256
        == EXPECTED_A2_SOURCE_SHA256
)


assert source_match_gate, (
    "Notebook 04 su Drive non coincide con il sorgente "
    "approvato allegato. Fermarsi senza eseguire le celle."
)


# ============================================================
# 4. EXECUTE APPROVED NOTEBOOK 04 STEP 04-R0 VERBATIM
# ============================================================

exec(
    compile(
        r0_source,
        (
            str(NOTEBOOK04_PATH)
            + "::STEP04-R0"
        ),
        "exec",
    ),
    globals(),
)


notebook04_r0_gate = all([
    directory_gate,
    final_size_gate,
    final_sha_gate,
])


assert notebook04_r0_gate


# ============================================================
# 5. EXECUTE APPROVED NOTEBOOK 04 STEP 04-A2 VERBATIM
#
# This reconstructs the exact Sakai224FullVAE,
# strictly loads FINAL and performs its synthetic EVAL audit.
# ============================================================

exec(
    compile(
        a2_source,
        (
            str(NOTEBOOK04_PATH)
            + "::STEP04-A2"
        ),
        "exec",
    ),
    globals(),
)


notebook04_a2_gate = all([
    structure_gate,
    checkpoint_gate,
    loaded_state_gate,
    forward_gate,
    state_preserved_gate,
])


assert notebook04_a2_gate


# ============================================================
# 6. NOTEBOOK 06 ENCODER POLICY
#
# Decoder remains present in official_final_model but will not
# be used by IIC.
# ============================================================

official_iic_encoder = (
    official_final_model.encoder
)


official_iic_encoder.eval()


for parameter in (
    official_iic_encoder.parameters()
):

    parameter.requires_grad_(
        False
    )


# ============================================================
# 7. FROZEN ENCODER IDENTITY
# ============================================================

EXPECTED_ENCODER_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)


encoder_state_sha256 = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_state_gate = (
    encoder_state_sha256
    == EXPECTED_ENCODER_STATE_SHA256
)


assert encoder_state_gate


# Full model numerical state must still be exactly FINAL.
full_model_state_after_freeze_sha256 = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)


full_model_state_gate = (
    full_model_state_after_freeze_sha256
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)


assert full_model_state_gate


# ============================================================
# 8. FROZEN / EVAL / NO-GRAD GATES
# ============================================================

encoder_eval_gate = (
    official_iic_encoder.training
    is False
)


encoder_frozen_gate = all(
    parameter.requires_grad
    is False

    for parameter
    in official_iic_encoder.parameters()
)


encoder_gradient_gate = all(
    parameter.grad
    is None

    for parameter
    in official_iic_encoder.parameters()
)


assert encoder_eval_gate
assert encoder_frozen_gate
assert encoder_gradient_gate


# ============================================================
# 9. ENCODER-ONLY SYNTHETIC EVAL PROBE
#
# No dataset read.
# No labels.
# ============================================================

encoder_probe_x = torch.zeros(
    (
        1,
        4,
        224,
        224,
    ),
    dtype=torch.float32,
    device=runtime_device,
)


encoder_state_before_probe = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


with torch.inference_mode():

    (
        encoder_probe_mu,
        encoder_probe_logvar,
    ) = official_iic_encoder(
        encoder_probe_x
    )


if runtime_device.type == "cuda":

    torch.cuda.synchronize()


encoder_state_after_probe = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_probe_gate = (
    tuple(
        encoder_probe_mu.shape
    ) == (
        1,
        512,
    )

    and tuple(
        encoder_probe_logvar.shape
    ) == (
        1,
        512,
    )

    and encoder_probe_mu.dtype
        == torch.float32

    and encoder_probe_logvar.dtype
        == torch.float32

    and torch.isfinite(
        encoder_probe_mu
    ).all().item()

    and torch.isfinite(
        encoder_probe_logvar
    ).all().item()

    and encoder_state_before_probe
        == encoder_state_after_probe
        == EXPECTED_ENCODER_STATE_SHA256
)


assert encoder_probe_gate


# ============================================================
# 10. SAFETY STATE
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

VAE_ENCODER_RESTORED = True

IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 11. FINAL A4B-1 GATE
# ============================================================

a4b1_gate = all([
    source_match_gate,
    notebook04_r0_gate,
    notebook04_a2_gate,
    encoder_state_gate,
    full_model_state_gate,
    encoder_eval_gate,
    encoder_frozen_gate,
    encoder_gradient_gate,
    encoder_probe_gate,
    LABELS_READ is False,
    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,
    IIC_MODEL_CREATED is False,
    IIC_TRAINING_STARTED is False,
    SPECTRAL_STARTED is False,
])


assert a4b1_gate


NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS = (
    a4b1_gate
)


# ============================================================
# 12. CLEAN ONLY TEMPORARY PROBE ACTIVATIONS
# ============================================================

del encoder_probe_x
del encoder_probe_mu
del encoder_probe_logvar

gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()


# ============================================================
# 13. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4B-1 ENCODER RECOVERY ==="
)
print()


print("--- APPROVED SOURCE ---")

print(
    "NOTEBOOK04_PATH =",
    NOTEBOOK04_PATH,
)

print(
    "STEP04_R0_CELL_INDEX =",
    r0_cell_index,
)

print(
    "STEP04_A2_CELL_INDEX =",
    a2_cell_index,
)

print(
    "STEP04_R0_SOURCE_SHA256 =",
    r0_source_sha256,
)

print(
    "STEP04_A2_SOURCE_SHA256 =",
    a2_source_sha256,
)

print(
    "SOURCE_MATCH_GATE =",
    source_match_gate,
)


print()
print("--- NOTEBOOK 04 RECOVERY ---")

print(
    "NOTEBOOK04_R0_GATE =",
    notebook04_r0_gate,
)

print(
    "NOTEBOOK04_A2_GATE =",
    notebook04_a2_gate,
)

print(
    "MODEL_CLASS =",
    official_final_model.__class__.__name__,
)

print(
    "FINAL_MODEL_STATE_SHA256 =",
    full_model_state_after_freeze_sha256,
)


print()
print("--- NOTEBOOK 06 ENCODER ---")

print(
    "ENCODER_CLASS =",
    official_iic_encoder.__class__.__name__,
)

print(
    "ENCODER_DEVICE =",
    next(
        official_iic_encoder.parameters()
    ).device,
)

print(
    "ENCODER_STATE_SHA256 =",
    encoder_state_sha256,
)

print(
    "ENCODER_STATE_GATE =",
    encoder_state_gate,
)

print(
    "ENCODER_EVAL_GATE =",
    encoder_eval_gate,
)

print(
    "ENCODER_FROZEN_GATE =",
    encoder_frozen_gate,
)

print(
    "ENCODER_GRADIENT_GATE =",
    encoder_gradient_gate,
)

print(
    "ENCODER_PROBE_GATE =",
    encoder_probe_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "IIC_MODEL_CREATED =",
    IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "PRIMARY_SYMMETRIC_RERUN =",
    PRIMARY_SYMMETRIC_RERUN,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS =",
    NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS,
)

print(
    "READY_FOR_A4B2_CENTRAL_MU_NUMERIC_AUDIT =",
    True,
)

print(
    "READY_FOR_A4C_IIC_DEFINITION_RECOVERY =",
    False,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    False,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verifying FINAL checkpoint SHA-256...

=== NOTEBOOK 04 RUNTIME RECOVERY ===
PYTHON = 3.13.15
PYTORCH = 2.11.0+cu128
GPU = NVIDIA A100-SXM4-40GB

CUDA_MATMUL_TF32 = False
CUDNN_TF32 = False
CUDNN_BENCHMARK = False
CUDNN_DETERMINISTIC = True

OFFICIAL_RUN_DIR = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1
FINAL_FILE_BYTES = 198545775
FINAL_FILE_SHA256 = 2bb6fd2ce2f73f8f9e479ed6969a52c1cdd41c1ae6e2cfc33da687ebba798f71

DIRECTORY_GATE = True
FINAL_SIZE_GATE = True
FINAL_SHA_GATE = True

OFFICIAL_FILES_MODIFIED = False

NOTEBOOK_04_RUNTIME_RECOVERY_PASS = True
Loading FINAL model state...

=== NOTEBOOK 04 OFFICIAL FINAL VAE LOAD AUDIT ===

MODEL_CLASS = Sakai224FullVAE
MODEL_DEVICE = cuda:0
MODEL_TRAINING_MODE = False

TRAINABLE_PARAMETERS = 16524548
PARAMETER_TENSORS = 28
STATE_TENSORS = 52

FINAL_CHECKPOINT_EPOCH = 99
COMPLETED_EPOCH

In [6]:
# ============================================================
# NOTEBOOK 06 — A4A VARIABLE ALIAS FIX
# Nessun I/O. Nessun modello. Nessun training.
# ============================================================

print(
    "a4a_gate present =",
    "a4a_gate" in globals(),
)

print(
    "a4a_gate value =",
    globals().get("a4a_gate", None),
)

print(
    "full_iic_rows present =",
    "full_iic_rows" in globals(),
)

print(
    "full_central_mu present =",
    "full_central_mu" in globals(),
)

assert (
    "a4a_gate" in globals()
    and a4a_gate is True
), (
    "Il vero gate A4A non è presente nel runtime. "
    "In quel caso fermarsi."
)

assert (
    "full_iic_rows" in globals()
    and len(full_iic_rows) == 8583
)

assert (
    "full_central_mu" in globals()
    and full_central_mu.shape == (8583, 512)
)

NOTEBOOK_06_A4A_DATA_RECOVERY_PASS = bool(
    a4a_gate
)

READY_FOR_A4B_ENCODER_RECOVERY = True

assert (
    NOTEBOOK_06_A4A_DATA_RECOVERY_PASS
    is True
)

print()
print(
    "NOTEBOOK_06_A4A_DATA_RECOVERY_PASS =",
    NOTEBOOK_06_A4A_DATA_RECOVERY_PASS,
)

print(
    "READY_FOR_A4B_ENCODER_RECOVERY =",
    READY_FOR_A4B_ENCODER_RECOVERY,
)

print(
    "A4A_ALIAS_FIX_PASS =",
    True,
)

a4a_gate present = True
a4a_gate value = True
full_iic_rows present = True
full_central_mu present = True

NOTEBOOK_06_A4A_DATA_RECOVERY_PASS = True
READY_FOR_A4B_ENCODER_RECOVERY = True
A4A_ALIAS_FIX_PASS = True


In [8]:
# ============================================================
# NOTEBOOK 06 — A4B-2
# FULL CENTRAL-MU NUMERIC RECOVERY AUDIT
#
# Scopo:
#   PNG centrali reali -> encoder FINAL -> mu
#   confronto con full_central_mu pubblicato.
#
# Tutti gli 8583 eventi.
#
# NO manifest_events_272.csv:
#   non vogliamo leggere neppure incidentalmente la colonna label.
#
# NO IIC heads.
# NO optimizer.
# NO training.
# NO spectral.
# NO Drive writes.
# ============================================================

from pathlib import Path
import hashlib
import gc

import numpy as np
import torch
from PIL import Image


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4A_DATA_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4A_DATA_RECOVERY_PASS is True
), "A4A non disponibile o non PASS."


assert (
    "NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS is True
), "A4B-1 non disponibile o non PASS."


assert (
    "full_iic_rows" in globals()
    and len(full_iic_rows) == 8583
)


assert (
    "full_central_mu" in globals()
    and full_central_mu.shape == (8583, 512)
    and full_central_mu.dtype == np.float32
    and np.isfinite(full_central_mu).all()
)


assert isinstance(
    official_iic_encoder,
    torch.nn.Module,
)


assert runtime_device.type == "cuda"


# ============================================================
# 1. FROZEN IDENTITIES
# ============================================================

EXPECTED_FULL_CENTRAL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)

EXPECTED_ENCODER_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

EXPECTED_FINAL_MODEL_STATE_SHA256 = (
    "3ac5a1cecdee2aa204419290b5cec55f3"
    "e6a4a846beebaa3f6b4e495f6fe6096"
)

# Known-answer ottenuto nel Notebook 06 originale.
EXPECTED_REFERENCE_MAX_ABS_DIFF = (
    1.1920928955078125e-06
)


# ============================================================
# 2. LOCAL PNG SOURCE
# ============================================================

A4B2_OUT272_DIR = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
    / "out272_gray"
)

EXPECTED_PNG_COUNT = 34332

assert A4B2_OUT272_DIR.is_dir()


# ============================================================
# 3. VERIFY PUBLISHED CENTRAL MU HAS NOT CHANGED
# ============================================================

current_full_mu_sha256 = hashlib.sha256(
    full_central_mu.tobytes(
        order="C"
    )
).hexdigest()


full_mu_identity_gate = (
    current_full_mu_sha256
    == EXPECTED_FULL_CENTRAL_MU_SHA256
)

assert full_mu_identity_gate


# ============================================================
# 4. PNG-NAME CONTRACT
#
# Preprocessing artifact:
#
#   img_000001.png
#   ...
#   img_034332.png
#
# Four consecutive PNGs form one event in channel order:
#   0.5, 1.0, 2.0, 4.0 s.
#
# This avoids reading manifest_events_272.csv and therefore
# avoids accessing its Gravity Spy label column.
# ============================================================

observed_png_names = {
    path.name
    for path in A4B2_OUT272_DIR.iterdir()
    if path.is_file()
}

expected_png_names = {
    f"img_{index:06d}.png"
    for index in range(
        1,
        EXPECTED_PNG_COUNT + 1,
    )
}


png_name_gate = (
    len(observed_png_names)
    == EXPECTED_PNG_COUNT

    and observed_png_names
    == expected_png_names
)

assert png_name_gate


# ============================================================
# 5. POPULATION / INDEX CONTRACT
# ============================================================

population_index_gate = all(

    row["manifest_index"] == manifest_index

    and row["event_index"]
        == manifest_index + 1

    for manifest_index, row
    in enumerate(full_iic_rows)
)

assert population_index_gate


# ============================================================
# 6. ENCODER PRE-AUDIT
# ============================================================

official_iic_encoder.eval()


encoder_state_before = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

full_model_state_before = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)


encoder_identity_before_gate = (
    encoder_state_before
    == EXPECTED_ENCODER_STATE_SHA256
)

full_model_identity_before_gate = (
    full_model_state_before
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)

encoder_eval_before_gate = (
    official_iic_encoder.training
    is False
)

encoder_frozen_before_gate = all(
    parameter.requires_grad is False
    for parameter
    in official_iic_encoder.parameters()
)

encoder_gradient_before_gate = all(
    parameter.grad is None
    for parameter
    in official_iic_encoder.parameters()
)


assert encoder_identity_before_gate
assert full_model_identity_before_gate
assert encoder_eval_before_gate
assert encoder_frozen_before_gate
assert encoder_gradient_before_gate


# ============================================================
# 7. FULL CENTRAL FORWARD
#
# manifest_index m:
#
#   event_index = m + 1
#   PNG indices  = 4*m + 1 ... 4*m + 4
#
# Central crop:
#   columns 24 ... 247
#   -> 224 columns.
#
# Pixel normalization:
#   uint8 / 255 -> float32.
# ============================================================

A4B2_BATCH_SIZE = 128

live_central_mu = np.empty(
    (
        8583,
        512,
    ),
    dtype=np.float32,
)


for start in range(
    0,
    8583,
    A4B2_BATCH_SIZE,
):

    stop = min(
        start + A4B2_BATCH_SIZE,
        8583,
    )

    batch_count = (
        stop - start
    )


    batch_u8 = np.empty(
        (
            batch_count,
            4,
            224,
            224,
        ),
        dtype=np.uint8,
    )


    for local_index, manifest_index in enumerate(
        range(
            start,
            stop,
        )
    ):

        row = full_iic_rows[
            manifest_index
        ]

        assert (
            row["manifest_index"]
            == manifest_index
        )

        assert (
            row["event_index"]
            == manifest_index + 1
        )


        first_png_index = (
            4 * manifest_index
            + 1
        )


        for channel in range(4):

            png_index = (
                first_png_index
                + channel
            )

            png_path = (
                A4B2_OUT272_DIR
                / f"img_{png_index:06d}.png"
            )

            assert png_path.is_file()


            with Image.open(
                png_path
            ) as image:

                # Preprocessed source must already be grayscale.
                assert image.mode == "L"

                # PIL convention = (width, height).
                assert image.size == (
                    272,
                    224,
                )

                image_u8 = np.asarray(
                    image,
                    dtype=np.uint8,
                )


            assert image_u8.shape == (
                224,
                272,
            )

            assert image_u8.dtype == np.uint8


            central_u8 = image_u8[
                :,
                24:248,
            ]


            assert central_u8.shape == (
                224,
                224,
            )


            batch_u8[
                local_index,
                channel,
                :,
                :,
            ] = central_u8


    # --------------------------------------------------------
    # uint8 -> float32 [0,1], CPU
    # --------------------------------------------------------

    batch_f32 = (
        batch_u8
        .astype(
            np.float32,
            copy=False,
        )
    )

    batch_f32 /= np.float32(
        255.0
    )


    assert batch_f32.dtype == np.float32

    assert np.isfinite(
        batch_f32
    ).all()

    assert float(
        batch_f32.min()
    ) >= 0.0

    assert float(
        batch_f32.max()
    ) <= 1.0


    batch_gpu = (
        torch.from_numpy(
            batch_f32
        )
        .to(
            runtime_device
        )
    )


    # --------------------------------------------------------
    # Frozen encoder only.
    # --------------------------------------------------------

    with torch.inference_mode():

        (
            batch_mu,
            batch_logvar,
        ) = official_iic_encoder(
            batch_gpu
        )


    assert batch_mu.shape == (
        batch_count,
        512,
    )

    assert batch_logvar.shape == (
        batch_count,
        512,
    )

    assert batch_mu.dtype == torch.float32

    assert torch.isfinite(
        batch_mu
    ).all().item()


    live_central_mu[
        start:stop,
        :,
    ] = (
        batch_mu
        .detach()
        .cpu()
        .numpy()
    )


    del batch_u8
    del batch_f32
    del batch_gpu
    del batch_mu
    del batch_logvar


    if (
        stop == 8583
        or stop % 1024 == 0
    ):
        print(
            "CENTRAL_FORWARD_PROGRESS =",
            f"{stop}/8583",
        )


if runtime_device.type == "cuda":
    torch.cuda.synchronize()


# ============================================================
# 8. NUMERIC COMPARISON
# ============================================================

live_shape_gate = (
    live_central_mu.shape
    == (8583, 512)

    and live_central_mu.dtype
    == np.float32

    and np.isfinite(
        live_central_mu
    ).all()
)

assert live_shape_gate


central_mu_abs_diff = np.abs(
    live_central_mu
    - full_central_mu
)


central_mu_max_abs_diff = float(
    np.max(
        central_mu_abs_diff
    )
)

central_mu_mean_abs_diff = float(
    np.mean(
        central_mu_abs_diff,
        dtype=np.float64,
    )
)

central_mu_nonzero_values = int(
    np.count_nonzero(
        central_mu_abs_diff
    )
)

central_mu_bitwise_equal = bool(
    np.array_equal(
        live_central_mu,
        full_central_mu,
    )
)


central_mu_numeric_gate = (
    central_mu_max_abs_diff
    <= EXPECTED_REFERENCE_MAX_ABS_DIFF
)

assert central_mu_numeric_gate, (
    "Central-mu numeric audit outside the approved "
    f"known-answer bound: {central_mu_max_abs_diff}"
)


# ============================================================
# 9. ENCODER / MODEL IMMUTABILITY AFTER 8583 EVENTS
# ============================================================

encoder_state_after = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

full_model_state_after = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)


encoder_state_preserved_gate = (
    encoder_state_before
    == encoder_state_after
    == EXPECTED_ENCODER_STATE_SHA256
)

full_model_state_preserved_gate = (
    full_model_state_before
    == full_model_state_after
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)

encoder_eval_after_gate = (
    official_iic_encoder.training
    is False
)

encoder_frozen_after_gate = all(
    parameter.requires_grad is False
    for parameter
    in official_iic_encoder.parameters()
)

encoder_gradient_after_gate = all(
    parameter.grad is None
    for parameter
    in official_iic_encoder.parameters()
)


assert encoder_state_preserved_gate
assert full_model_state_preserved_gate
assert encoder_eval_after_gate
assert encoder_frozen_after_gate
assert encoder_gradient_after_gate


# ============================================================
# 10. SAFETY STATE
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

VAE_ENCODER_RESTORED = True

IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 11. FINAL GATE
# ============================================================

a4b2_gate = all([
    full_mu_identity_gate,
    png_name_gate,
    population_index_gate,
    encoder_identity_before_gate,
    full_model_identity_before_gate,
    encoder_eval_before_gate,
    encoder_frozen_before_gate,
    encoder_gradient_before_gate,
    live_shape_gate,
    central_mu_numeric_gate,
    encoder_state_preserved_gate,
    full_model_state_preserved_gate,
    encoder_eval_after_gate,
    encoder_frozen_after_gate,
    encoder_gradient_after_gate,
    LABELS_READ is False,
    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,
    IIC_MODEL_CREATED is False,
    OPTIMIZER_CREATED is False,
    IIC_TRAINING_STARTED is False,
    SPECTRAL_STARTED is False,
    PRIMARY_SYMMETRIC_RERUN is False,
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])


assert a4b2_gate


# ============================================================
# 12. EXPLICIT PERSISTENT RUNTIME FLAGS
#
# Important: actual assignments, not print-only aliases.
# ============================================================

NOTEBOOK_06_A4B2_CENTRAL_MU_NUMERIC_AUDIT_PASS = bool(
    a4b2_gate
)

NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS = bool(
    NOTEBOOK_06_A4B1_ENCODER_RECOVERY_PASS
    and
    NOTEBOOK_06_A4B2_CENTRAL_MU_NUMERIC_AUDIT_PASS
)

READY_FOR_A4C_IIC_DEFINITION_RECOVERY = bool(
    NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS
)

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 13. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4B-2 "
    "FULL CENTRAL-MU NUMERIC AUDIT ==="
)
print()


print("--- INPUT / POPULATION ---")

print(
    "PNG_NAME_GATE =",
    png_name_gate,
)

print(
    "PNG_COUNT =",
    len(
        observed_png_names
    ),
)

print(
    "POPULATION_INDEX_GATE =",
    population_index_gate,
)

print(
    "CENTRAL_CROP_LEFT =",
    24,
)

print(
    "EVENTS_FORWARDED =",
    8583,
)

print(
    "FORWARD_BATCH_SIZE =",
    A4B2_BATCH_SIZE,
)


print()
print("--- PUBLISHED MU ---")

print(
    "FULL_CENTRAL_MU_SHA256 =",
    current_full_mu_sha256,
)

print(
    "FULL_MU_IDENTITY_GATE =",
    full_mu_identity_gate,
)


print()
print("--- NUMERIC AUDIT ---")

print(
    "LIVE_CENTRAL_MU_SHAPE =",
    live_central_mu.shape,
)

print(
    "LIVE_CENTRAL_MU_DTYPE =",
    live_central_mu.dtype,
)

print(
    "CENTRAL_MU_BITWISE_EQUAL =",
    central_mu_bitwise_equal,
)

print(
    "CENTRAL_MU_NONZERO_DIFF_VALUES =",
    central_mu_nonzero_values,
)

print(
    "CENTRAL_MU_MEAN_ABS_DIFF =",
    central_mu_mean_abs_diff,
)

print(
    "CENTRAL_MU_MAX_ABS_DIFF =",
    central_mu_max_abs_diff,
)

print(
    "REFERENCE_MAX_ABS_DIFF =",
    EXPECTED_REFERENCE_MAX_ABS_DIFF,
)

print(
    "CENTRAL_MU_NUMERIC_GATE =",
    central_mu_numeric_gate,
)


print()
print("--- ENCODER IMMUTABILITY ---")

print(
    "ENCODER_STATE_BEFORE =",
    encoder_state_before,
)

print(
    "ENCODER_STATE_AFTER =",
    encoder_state_after,
)

print(
    "ENCODER_STATE_PRESERVED_GATE =",
    encoder_state_preserved_gate,
)

print(
    "FULL_MODEL_STATE_PRESERVED_GATE =",
    full_model_state_preserved_gate,
)

print(
    "ENCODER_EVAL_AFTER_GATE =",
    encoder_eval_after_gate,
)

print(
    "ENCODER_FROZEN_AFTER_GATE =",
    encoder_frozen_after_gate,
)

print(
    "ENCODER_GRADIENT_AFTER_GATE =",
    encoder_gradient_after_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "IIC_MODEL_CREATED =",
    IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "PRIMARY_SYMMETRIC_RERUN =",
    PRIMARY_SYMMETRIC_RERUN,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4B2_CENTRAL_MU_NUMERIC_AUDIT_PASS =",
    NOTEBOOK_06_A4B2_CENTRAL_MU_NUMERIC_AUDIT_PASS,
)

print(
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS =",
    NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS,
)

print(
    "READY_FOR_A4C_IIC_DEFINITION_RECOVERY =",
    READY_FOR_A4C_IIC_DEFINITION_RECOVERY,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


# ============================================================
# 14. CLEAN TEMPORARY FULL LIVE MATRIX
# ============================================================

del central_mu_abs_diff
del live_central_mu
del observed_png_names
del expected_png_names

gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()

CENTRAL_FORWARD_PROGRESS = 1024/8583
CENTRAL_FORWARD_PROGRESS = 2048/8583
CENTRAL_FORWARD_PROGRESS = 3072/8583
CENTRAL_FORWARD_PROGRESS = 4096/8583
CENTRAL_FORWARD_PROGRESS = 5120/8583
CENTRAL_FORWARD_PROGRESS = 6144/8583
CENTRAL_FORWARD_PROGRESS = 7168/8583
CENTRAL_FORWARD_PROGRESS = 8192/8583
CENTRAL_FORWARD_PROGRESS = 8583/8583

=== NOTEBOOK 06 — A4B-2 FULL CENTRAL-MU NUMERIC AUDIT ===

--- INPUT / POPULATION ---
PNG_NAME_GATE = True
PNG_COUNT = 34332
POPULATION_INDEX_GATE = True
CENTRAL_CROP_LEFT = 24
EVENTS_FORWARDED = 8583
FORWARD_BATCH_SIZE = 128

--- PUBLISHED MU ---
FULL_CENTRAL_MU_SHA256 = fd55be588ef05df8afdf7ee048aa535e12eda37bd13ef8d59e5968fcfc9467f8
FULL_MU_IDENTITY_GATE = True

--- NUMERIC AUDIT ---
LIVE_CENTRAL_MU_SHAPE = (8583, 512)
LIVE_CENTRAL_MU_DTYPE = float32
CENTRAL_MU_BITWISE_EQUAL = False
CENTRAL_MU_NONZERO_DIFF_VALUES = 3352
CENTRAL_MU_MEAN_ABS_DIFF = 1.3922866603456585e-10
CENTRAL_MU_MAX_ABS_DIFF = 8.940696716308594e-07
REFERENCE_MAX_ABS_DIFF = 1.192092895507

In [9]:
# ============================================================
# NOTEBOOK 06 — A4C-0
# APPROVED NOTEBOOK-06 SOURCE INVENTORY
#
# READ ONLY.
#
# - individua il Notebook 06 reale su Drive;
# - legge il JSON del .ipynb;
# - NON esegue nessuna sua cella;
# - inventaria funzioni, classi e marker;
# - cerca gli 8 contract SHA congelati nel recap 15.
#
# NO labels.
# NO IIC model creation.
# NO optimizer.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import ast
import hashlib
import json


# ============================================================
# 0. PARENT GATE
# ============================================================

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."

assert (
    "READY_FOR_A4C_IIC_DEFINITION_RECOVERY" in globals()
    and READY_FOR_A4C_IIC_DEFINITION_RECOVERY is True
), "Runtime non autorizzato ad A4C."


# ============================================================
# 1. LOCATE REAL NOTEBOOK 06
# ============================================================

NOTEBOOKS_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
)

preferred_nb06 = (
    NOTEBOOKS_DIR
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

if preferred_nb06.is_file():

    NOTEBOOK06_PATH = preferred_nb06

else:

    candidates = sorted(
        NOTEBOOKS_DIR.glob(
            "06*sakai224*full*iic*.ipynb"
        )
    )

    if len(candidates) != 1:
        raise RuntimeError(
            "Notebook 06 non identificabile univocamente. "
            f"Candidati: {candidates}"
        )

    NOTEBOOK06_PATH = candidates[0]


assert NOTEBOOK06_PATH.is_file()


# ============================================================
# 2. LOAD NOTEBOOK JSON — ABSOLUTELY NO EXEC()
# ============================================================

notebook06_bytes = NOTEBOOK06_PATH.read_bytes()

NOTEBOOK06_FILE_SHA256 = hashlib.sha256(
    notebook06_bytes
).hexdigest()

notebook06 = json.loads(
    notebook06_bytes.decode("utf-8")
)

assert isinstance(notebook06, dict)
assert isinstance(notebook06.get("cells"), list)


# ============================================================
# 3. FROZEN CONTRACT HASHES FROM RECAP 15
# ============================================================

FROZEN_CONTRACT_SHA256 = {

    "LOSS_IIC":
        "2bf9cba7d23de553cbad1d876b1f1b14"
        "3ede63154ac20bbb29ca092223c5e7ac",

    "IIC_ARCHITECTURE":
        "12fd00c3dddc6474249a86899b14797c"
        "c891e815126cfb6989eb96e489d5e758",

    "TRAINING_CORE":
        "f1df0a78746739c7552668e0ed0cac29"
        "5b8d61d772088cb7893a85d6ae6bfe0a",

    "FINAL_TRAINING":
        "4f27a9bbf4290dff54767655d2025928"
        "c2eac751607d217925b0d5bba2b2f537",

    "DUAL_AUGMENTATION":
        "4fbae488ccbe6226c4c589c8bfcd712e"
        "d20e0ddf65ec95e4dfb80120d8432fbc",

    "OFFICIAL_RUNS":
        "010aeaa0aedc657c1b1214604e67992a"
        "ff4fffb9bf5bc0450a63758868ff783a",

    "CHECKPOINT_RESUME":
        "a2dc5e9135b69486494f6045b40cb55"
        "d96843cf6819cbce20738a098b66f7a39",

    "RESUME_EDGE_CASE":
        "f15655c117fb6a9ca76bd414332b1a34"
        "91eb70c5b59de67da3a0e1fed946eca6",
}


# ============================================================
# 4. SOURCE INSPECTION HELPERS
# ============================================================

def cell_source(cell):
    return "".join(
        cell.get("source", [])
    )


def first_nonempty_line(source):
    for line in source.splitlines():
        text = line.strip()
        if text:
            return text[:160]
    return "<empty>"


def top_level_symbols(source):
    """
    Parse only. Never executes source.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return [], [], []

    functions = []
    classes = []
    assigned_names = []

    for node in tree.body:

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            functions.append(node.name)

        elif isinstance(node, ast.ClassDef):
            classes.append(node.name)

        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    assigned_names.append(
                        target.id
                    )

        elif isinstance(node, ast.AnnAssign):
            if isinstance(node.target, ast.Name):
                assigned_names.append(
                    node.target.id
                )

    return (
        functions,
        classes,
        assigned_names,
    )


# ============================================================
# 5. INVENTORY ALL CODE CELLS
# ============================================================

code_cell_inventory = []

for cell_index, cell in enumerate(
    notebook06["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = cell_source(cell)

    source_sha = hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()

    (
        functions,
        classes,
        assigned_names,
    ) = top_level_symbols(source)

    found_contract_hashes = []

    for contract_name, contract_sha in (
        FROZEN_CONTRACT_SHA256.items()
    ):
        if contract_sha in source:
            found_contract_hashes.append(
                contract_name
            )

    code_cell_inventory.append({

        "cell_index":
            cell_index,

        "source_sha256":
            source_sha,

        "chars":
            len(source),

        "first_line":
            first_nonempty_line(source),

        "functions":
            functions,

        "classes":
            classes,

        "assigned_names":
            assigned_names,

        "embedded_contract_hashes":
            found_contract_hashes,
    })


# ============================================================
# 6. SEARCH IMPORTANT APPROVED SYMBOLS / MARKERS
# ============================================================

SEARCH_TOKENS = [

    # IIC loss / joint
    "joint",
    "mutual_information",
    "iic",

    # architecture
    "IICHead",
    "IICClassifier",

    # optimizer / training
    "optimizer",
    "train_one",
    "train_epoch",

    # augmentation
    "legacy_text",
    "symmetric_figure",
    "crop_left",

    # persistence
    "checkpoint",
    "resume",
    "latest.pt",
    "final.pt",

    # official run
    "sakai224_full_iic_symmetric_v1",
    "sakai224_full_iic_legacy_text_v1",
]


token_hits = {
    token: []
    for token in SEARCH_TOKENS
}


for item in code_cell_inventory:

    source = cell_source(
        notebook06["cells"][
            item["cell_index"]
        ]
    )

    for token in SEARCH_TOKENS:
        if token in source:
            token_hits[token].append(
                item["cell_index"]
            )


# ============================================================
# 7. SAFETY / COLLISION GUARDS
# ============================================================

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

legacy_collision_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_collision_gate


LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 8. FINAL DISCOVERY GATE
# ============================================================

a4c0_gate = all([

    NOTEBOOK06_PATH.is_file(),

    len(code_cell_inventory) > 0,

    legacy_collision_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c0_gate


NOTEBOOK_06_A4C0_SOURCE_INVENTORY_PASS = bool(
    a4c0_gate
)


# ============================================================
# 9. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-0 SOURCE INVENTORY ==="
)
print()


print("--- NOTEBOOK FILE ---")

print(
    "NOTEBOOK06_PATH =",
    NOTEBOOK06_PATH,
)

print(
    "NOTEBOOK06_FILE_BYTES =",
    len(notebook06_bytes),
)

print(
    "NOTEBOOK06_FILE_SHA256 =",
    NOTEBOOK06_FILE_SHA256,
)

print(
    "TOTAL_CELLS =",
    len(notebook06["cells"]),
)

print(
    "CODE_CELLS =",
    len(code_cell_inventory),
)


print()
print("--- CODE CELL INVENTORY ---")

for item in code_cell_inventory:

    print()
    print(
        "CELL_INDEX =",
        item["cell_index"],
    )

    print(
        "SOURCE_SHA256 =",
        item["source_sha256"],
    )

    print(
        "CHARS =",
        item["chars"],
    )

    print(
        "FIRST_LINE =",
        item["first_line"],
    )

    print(
        "FUNCTIONS =",
        item["functions"],
    )

    print(
        "CLASSES =",
        item["classes"],
    )

    # Limit output; names only, no values.
    print(
        "ASSIGNED_NAMES =",
        item["assigned_names"][:40],
    )

    print(
        "EMBEDDED_CONTRACT_HASHES =",
        item["embedded_contract_hashes"],
    )


print()
print("--- TOKEN → CELL INDEXES ---")

for token, cells in token_hits.items():
    print(
        repr(token),
        "=>",
        cells,
    )


print()
print("--- FROZEN CONTRACT SHA256 ---")

for name, value in (
    FROZEN_CONTRACT_SHA256.items()
):
    print(
        name,
        "=",
        value,
    )


print()
print("--- SAFETY ---")

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)

print(
    "LEGACY_COLLISION_GATE =",
    legacy_collision_gate,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "IIC_MODEL_CREATED =",
    IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C0_SOURCE_INVENTORY_PASS =",
    NOTEBOOK_06_A4C0_SOURCE_INVENTORY_PASS,
)

print(
    "READY_TO_EXECUTE_IIC_DEFINITION_CELLS =",
    False,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    False,
)


=== NOTEBOOK 06 — A4C-0 SOURCE INVENTORY ===

--- NOTEBOOK FILE ---
NOTEBOOK06_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/notebooks/06_sakai224_full_iic_and_spectral.ipynb
NOTEBOOK06_FILE_BYTES = 620322
NOTEBOOK06_FILE_SHA256 = 7b4490169617a2037f8038f037839151e5494b6ed33a09c6b64d5cbbc8cf57c5
TOTAL_CELLS = 60
CODE_CELLS = 34

--- CODE CELL INVENTORY ---

CELL_INDEX = 0
SOURCE_SHA256 = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
CHARS = 0
FIRST_LINE = <empty>
FUNCTIONS = []
CLASSES = []
ASSIGNED_NAMES = []
EMBEDDED_CONTRACT_HASHES = []

CELL_INDEX = 2
SOURCE_SHA256 = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
CHARS = 0
FIRST_LINE = <empty>
FUNCTIONS = []
CLASSES = []
ASSIGNED_NAMES = []
EMBEDDED_CONTRACT_HASHES = []

CELL_INDEX = 5
SOURCE_SHA256 = 2675f22cc92e439b457c65da02ffe25da0d23fd35dd72c90e19163b14e19708c
CHARS = 15100
FIRST_LINE = # ============================================================
FUNCTIONS = ['sha256_file', '

In [10]:
# ============================================================
# NOTEBOOK 06 — A4C-1
# CONTROLLED IIC CORE DEFINITION RECOVERY
#
# SOURCE OF TRUTH:
#   current approved Notebook 06 on Drive.
#
# Restores:
#   - canonical IIC data contract
#   - IIC loss implementation + numerical audit
#   - IICHead / IICClassifierPair
#   - initialization / loss-combination core
#   - final provenance realignment
#   - final 100-epoch training policy
#
# NO Gravity Spy labels.
# NO real perturbed batches.
# NO optimizer creation.
# NO optimizer step.
# NO official IIC run.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import copy
import gc
import hashlib
import json
import math

import numpy as np
import torch
import torch.nn as nn


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C0_SOURCE_INVENTORY_PASS" in globals()
    and NOTEBOOK_06_A4C0_SOURCE_INVENTORY_PASS is True
), "A4C-0 non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."

assert (
    "full_iic_rows" in globals()
    and len(full_iic_rows) == 8583
)

assert (
    "FULL_IIC_POPULATION_SHA256" in globals()
    and FULL_IIC_POPULATION_SHA256
        == (
            "bd95286ec6c70cb2b5c674a8a46f3fd"
            "450048367d91dc17fd3625977c538b8f4"
        )
)


# ============================================================
# 1. APPROVED NOTEBOOK 06
# ============================================================

NOTEBOOK06_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH.is_file()

with NOTEBOOK06_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    approved_nb06 = json.load(handle)


# ============================================================
# 2. HISTORICAL CELL SOURCE HASHES
#
# These are the exact source hashes observed by A4C-0.
# If any old scientific cell changed, STOP.
# ============================================================

EXPECTED_CELL_SHA256 = {

    5:
        "5bd723206d0c29268e4631470f5da105"
        "f784a1298f66a434b64e7956008ce800",


    8:
        "e41f632095b5a21e0b835915da9b29ef"
        "52203973a76391edae439d303f1d536e",

    10:
        "75207bf841c5f4d93883823db3bc2801"
        "c53c68f4fb5ebf409036b2b804f14da4",

    12:
        "11e5fe2c9b52b36c26397eb8f4fe62c"
        "803c347daf2072f3004ff0e3fad9ec7c3",

    15:
        "740da39e4fff1e7bcf56e6e5a5bd1c8f"
        "a6518bc48cf9a350dde900c1a6749747",

    17:
        "f1c93f8169a2b38b824ae56ecbfba5a1"
        "59efaa3780ef05c1693076ca0b66689d",

    19:
        "d2e41a4789c599929480dab26f9ec636b"
        "f1649ec7f5273467f8d3418e2c5d5c1",
}


def approved_cell_source(cell_index):

    cell = approved_nb06["cells"][
        cell_index
    ]

    assert cell.get(
        "cell_type"
    ) == "code"

    return "".join(
        cell.get(
            "source",
            [],
        )
    )


observed_cell_sha256 = {}

for cell_index, expected_sha in (
    EXPECTED_CELL_SHA256.items()
):

    source = approved_cell_source(
        cell_index
    )

    actual_sha = hashlib.sha256(
        source.encode(
            "utf-8"
        )
    ).hexdigest()

    observed_cell_sha256[
        cell_index
    ] = actual_sha

    assert actual_sha == expected_sha, (
        f"Approved Notebook 06 cell {cell_index} changed. "
        "STOP."
    )


historical_source_gate = all(
    observed_cell_sha256[index]
    == expected

    for index, expected
    in EXPECTED_CELL_SHA256.items()
)

assert historical_source_gate


# ============================================================
# 3. ENCODER / MODEL IMMUTABILITY BASELINE
# ============================================================

EXPECTED_ENCODER_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

EXPECTED_FINAL_MODEL_STATE_SHA256 = (
    "3ac5a1cecdee2aa204419290b5cec55f3"
    "e6a4a846beebaa3f6b4e495f6fe6096"
)


encoder_state_before_a4c1 = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

full_model_state_before_a4c1 = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)

assert (
    encoder_state_before_a4c1
    == EXPECTED_ENCODER_STATE_SHA256
)

assert (
    full_model_state_before_a4c1
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)


# ============================================================
# 4. RESTORE DATA-CONTRACT SECTION FROM HISTORICAL CELL 5
#
# We deliberately DO NOT execute the complete A1 cell:
# its large source hashes have already been verified in A4A.
#
# The scientific contract section itself is extracted
# directly from the approved cell source.
# ============================================================

cell5_source = approved_cell_source(
    5
)

contract_start_marker = (
    "# FROZEN DATA CONTRACT"
)

contract_end_marker = (
    "# REPORT"
)

contract_start = cell5_source.find(
    contract_start_marker
)

contract_end = cell5_source.find(
    contract_end_marker,
    contract_start,
)

assert contract_start >= 0
assert contract_end > contract_start


data_contract_source = (
    cell5_source[
        contract_start:
        contract_end
    ]
)


exec(
    compile(
        data_contract_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL5_DATA_CONTRACT"
        ),
        "exec",
    ),
    globals(),
)


EXPECTED_IIC_DATA_CONTRACT_SHA256 = (
    "03bfab28cd2f513dd3616aca9f394633"
    "3d12d85508e95bb860f6682941002e17"
)

data_contract_gate = (
    IIC_DATA_CONTRACT_SHA256
    == EXPECTED_IIC_DATA_CONTRACT_SHA256
)

assert data_contract_gate


# ============================================================
# 5. EXECUTE APPROVED CORE CELLS VERBATIM
#
# 8  = loss audit
# 10 = architecture contract
# 12 = structural synthetic probe + class definitions
# 15 = initialization / loss-combination contract
# 17 = final provenance / C11 realignment
# 19 = final training policy
#
# None of these cells reads real Gravity Spy data,
# creates an optimizer, trains, or writes to Drive.
# ============================================================

CORE_CELL_ORDER = [
    8,
    10,
    12,
    15,
    17,
    19,
]


for cell_index in CORE_CELL_ORDER:

    source = approved_cell_source(
        cell_index
    )

    print(
        "EXECUTING_APPROVED_CORE_CELL =",
        cell_index,
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + f"::CELL{cell_index}"
            ),
            "exec",
        ),
        globals(),
    )


# ============================================================
# 6. FINAL FROZEN CONTRACT KNOWN ANSWERS
# ============================================================

EXPECTED_FINAL_CONTRACTS = {

    "LOSS_IIC":
        (
            "2bf9cba7d23de553cbad1d876b1f1b14"
            "3ede63154ac20bbb29ca092223c5e7ac"
        ),

    "IIC_ARCHITECTURE":
        (
            "12fd00c3dddc6474249a86899b14797c"
            "c891e815126cfb6989eb96e489d5e758"
        ),

    "TRAINING_CORE":
        (
            "f1df0a78746739c7552668e0ed0cac29"
            "5b8d61d772088cb7893a85d6ae6bfe0a"
        ),

    "FINAL_TRAINING":
        (
            "4f27a9bbf4290dff54767655d2025928"
            "c2eac751607d217925b0d5bba2b2f537"
        ),
}


contract_hash_gate = all([

    IIC_LOSS_CONTRACT_SHA256
        == EXPECTED_FINAL_CONTRACTS[
            "LOSS_IIC"
        ],

    IIC_ARCHITECTURE_CONTRACT_SHA256
        == EXPECTED_FINAL_CONTRACTS[
            "IIC_ARCHITECTURE"
        ],

    IIC_TRAINING_CORE_SHA256
        == EXPECTED_FINAL_CONTRACTS[
            "TRAINING_CORE"
        ],

    IIC_FINAL_TRAINING_SHA256
        == EXPECTED_FINAL_CONTRACTS[
            "FINAL_TRAINING"
        ],
])

assert contract_hash_gate


# ============================================================
# 7. REQUIRED CORE DEFINITIONS
# ============================================================

required_core_symbols = [

    "validate_probability_matrix",

    "sakai_joint_probability",

    "sakai_iic_negative_mutual_information",

    "project_empirical_joint_probability",

    "project_iic_negative_mutual_information",

    "IICHead",

    "IICClassifierPair",

    "combine_iic_losses",

    "build_iic_pair_from_seed",

    "module_state_sha256",
]


core_symbol_gate = all(
    name in globals()
    for name in required_core_symbols
)

assert core_symbol_gate


# ============================================================
# 8. FINAL TRAINING POLICY KNOWN ANSWERS
# ============================================================

training_policy_gate = all([

    IIC_BATCH_SIZE == 128,

    IIC_LEARNING_RATE == 5.0e-4,

    IIC_ADAM_BETA1 == 0.9,

    IIC_ADAM_BETA2 == 0.999,

    IIC_ADAM_EPS == 1.0e-8,

    IIC_EPOCHS == 100,

    IIC_DROP_LAST is False,

    IIC_BATCHES_PER_EPOCH == 68,

    IIC_TAIL_BATCH_SIZE == 7,

    IIC_STEPS_PER_CLASSIFIER == 6800,

    IIC_CLASSIFIER_COUNT == 5,

    IIC_CLASSIFIER_SEEDS
        == [
            123,
            124,
            125,
            126,
            127,
        ],

    IIC_SHUFFLE_SEED == 101112,

    IIC_AUGMENTATION_SEED == 789,
])

assert training_policy_gate


# ============================================================
# 9. ENCODER MUST BE COMPLETELY UNCHANGED
# ============================================================

encoder_state_after_a4c1 = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

full_model_state_after_a4c1 = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)


encoder_immutability_gate = (
    encoder_state_before_a4c1
    == encoder_state_after_a4c1
    == EXPECTED_ENCODER_STATE_SHA256
)

full_model_immutability_gate = (
    full_model_state_before_a4c1
    == full_model_state_after_a4c1
    == EXPECTED_FINAL_MODEL_STATE_SHA256
)

encoder_policy_gate = (
    official_iic_encoder.training
    is False

    and all(
        parameter.requires_grad
        is False

        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad
        is None

        for parameter
        in official_iic_encoder.parameters()
    )
)


assert encoder_immutability_gate
assert full_model_immutability_gate
assert encoder_policy_gate


# ============================================================
# 10. LEGACY COLLISION / SAFETY
# ============================================================

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

legacy_collision_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_collision_gate


LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False
PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# Synthetic temporary IIC heads WERE created by the approved
# structural probes in cells 12 and 15.
TEMPORARY_SYNTHETIC_IIC_MODELS_CREATED = True
OFFICIAL_IIC_MODEL_CREATED = False


# ============================================================
# 11. CLEAN TEMPORARY SYNTHETIC MODELS/TENSORS
# ============================================================

temporary_names = [

    "main_train",
    "over_train",
    "reference_pair",

    "probe_input_train",
    "probe_input_eval",

    "main_running_mean_before",
    "over_running_mean_before",

    "main_eval_mean_before",
    "over_eval_mean_before",
]


for name in temporary_names:

    globals().pop(
        name,
        None,
    )


gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()


# ============================================================
# 12. FINAL GATE
# ============================================================

a4c1_gate = all([

    historical_source_gate,

    data_contract_gate,

    contract_hash_gate,

    core_symbol_gate,

    training_policy_gate,

    encoder_immutability_gate,

    full_model_immutability_gate,

    encoder_policy_gate,

    legacy_collision_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c1_gate


NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS = bool(
    a4c1_gate
)

READY_FOR_A4C2_RUNTIME_DEFINITION_RECOVERY = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 13. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-1 IIC CORE RECOVERY ==="
)
print()


print("--- APPROVED SOURCE ---")

print(
    "NOTEBOOK06_PATH =",
    NOTEBOOK06_PATH,
)

print(
    "HISTORICAL_SOURCE_GATE =",
    historical_source_gate,
)

for index in EXPECTED_CELL_SHA256:

    print(
        f"CELL_{index}_SHA256 =",
        observed_cell_sha256[
            index
        ],
    )


print()
print("--- CONTRACTS ---")

print(
    "IIC_DATA_CONTRACT_SHA256 =",
    IIC_DATA_CONTRACT_SHA256,
)

print(
    "IIC_LOSS_CONTRACT_SHA256 =",
    IIC_LOSS_CONTRACT_SHA256,
)

print(
    "IIC_ARCHITECTURE_CONTRACT_SHA256 =",
    IIC_ARCHITECTURE_CONTRACT_SHA256,
)

print(
    "IIC_TRAINING_CORE_SHA256 =",
    IIC_TRAINING_CORE_SHA256,
)

print(
    "IIC_FINAL_TRAINING_SHA256 =",
    IIC_FINAL_TRAINING_SHA256,
)

print(
    "CONTRACT_HASH_GATE =",
    contract_hash_gate,
)


print()
print("--- TRAINING POLICY ---")

print(
    "IIC_EPOCHS =",
    IIC_EPOCHS,
)

print(
    "IIC_BATCH_SIZE =",
    IIC_BATCH_SIZE,
)

print(
    "IIC_BATCHES_PER_EPOCH =",
    IIC_BATCHES_PER_EPOCH,
)

print(
    "IIC_TAIL_BATCH_SIZE =",
    IIC_TAIL_BATCH_SIZE,
)

print(
    "IIC_STEPS_PER_CLASSIFIER =",
    IIC_STEPS_PER_CLASSIFIER,
)

print(
    "IIC_CLASSIFIER_SEEDS =",
    IIC_CLASSIFIER_SEEDS,
)

print(
    "IIC_SHUFFLE_SEED =",
    IIC_SHUFFLE_SEED,
)

print(
    "IIC_AUGMENTATION_SEED =",
    IIC_AUGMENTATION_SEED,
)

print(
    "TRAINING_POLICY_GATE =",
    training_policy_gate,
)


print()
print("--- ENCODER IMMUTABILITY ---")

print(
    "ENCODER_STATE_BEFORE =",
    encoder_state_before_a4c1,
)

print(
    "ENCODER_STATE_AFTER =",
    encoder_state_after_a4c1,
)

print(
    "ENCODER_IMMUTABILITY_GATE =",
    encoder_immutability_gate,
)

print(
    "FULL_MODEL_IMMUTABILITY_GATE =",
    full_model_immutability_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "TEMPORARY_SYNTHETIC_IIC_MODELS_CREATED =",
    TEMPORARY_SYNTHETIC_IIC_MODELS_CREATED,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS =",
    NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS,
)

print(
    "READY_FOR_A4C2_RUNTIME_DEFINITION_RECOVERY =",
    READY_FOR_A4C2_RUNTIME_DEFINITION_RECOVERY,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)

EXECUTING_APPROVED_CORE_CELL = 8

=== NOTEBOOK 06 SAKAI Eq.4 LOSS AUDIT — CORRECTED ===

SOURCE = Sakai_Eq4
JOINT_SYMMETRIZED = False
EXTRA_JOINT_RENORMALIZATION = False
ARBITRARY_EPSILON_ADDED = False
ZERO_TERM_POLICY = Pij_gt_0_only

NUMPY_REFERENCE_LOSS = -0.02352875972182373
TORCH_REFERENCE_LOSS = -0.023528759721823736

BALANCED_MI = 1.3862943611198906
UNIFORM_MI = 0.0
COLLAPSED_MI = 0.0

NUMPY_REFERENCE_GATE = True
BALANCED_GATE = True
UNIFORM_GATE = True
COLLAPSED_GATE = True
NO_SYMMETRISATION_GATE = True
VIEW_SWAP_GATE = True
BATCH_ORDER_GATE = True
JOINT_SUM_GATE = True
AUTOGRAD_GATE = True
PAPER_DIMENSION_GATE = tensor(True)

SUPERSEDED_IIC_LOSS_CONTRACT_SHA256 = 5b76d50075a068c4db581da872ef3941356c9c2fd83d06488e3cd0615e9cf4c0
CORRECTED_IIC_LOSS_CONTRACT_SHA256 = b0b108727286c8202b4ebfcae987d6776a80eb3a8f993c992a5a328f785179f1

MAIN_OVERCLUSTER_RELATIONSHIP_FROZEN = False
LABELS_READ = False
MODEL_CREATED = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =

In [11]:
# ============================================================
# NOTEBOOK 06 — A4C-2A
# RUNTIME DATA / AUGMENTATION / SHUFFLE PRIMITIVES RECOVERY
#
# Restores ONLY:
#   - label-free event path map
#   - exact PNG crop loader
#   - exact stateless augmentation
#   - exact PROJECT shuffle
#   - make_real_perturbed_batch definition
#
# NO Gravity Spy label values.
# NO IIC official models.
# NO optimizer.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import ast
import hashlib
import json

import numpy as np
import torch
from PIL import Image


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS is True
), "A4C-1 non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."

assert len(full_iic_rows) == 8583

assert (
    FULL_IIC_POPULATION_SHA256
    ==
    "bd95286ec6c70cb2b5c674a8a46f3fd"
    "450048367d91dc17fd3625977c538b8f4"
)


# ============================================================
# 1. APPROVED NOTEBOOK 06 SOURCE
# ============================================================

NOTEBOOK06_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH.is_file()

with NOTEBOOK06_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    nb06_runtime_source = json.load(
        handle
    )


# ============================================================
# 2. HISTORICAL SOURCE HASH GATES
# ============================================================

EXPECTED_SOURCE_SHA256 = {

    21:
        "9bad1fe68de9beeb849ed48ef145fb04"
        "f7e1ffb9b2cd9a101258112b177c5f33",

    23:
        "ef608dabda3bb7d500f749783d51612d0"
        "3297c84025383508750170f8415e53c",

    25:
        "12955163342f2b37824798493f5b044a9"
        "0f314dea64a5e668a96eeb553cb15c5",

    29:
        "18aa232fd630423760a13cfb15e37637d"
        "6d6e69cd42438e749d9cfa869adf6f0",
}


def get_code_source(
    cell_index,
):

    cell = nb06_runtime_source[
        "cells"
    ][
        cell_index
    ]

    assert cell[
        "cell_type"
    ] == "code"

    return "".join(
        cell.get(
            "source",
            [],
        )
    )


observed_source_sha256 = {}

for cell_index, expected_sha in (
    EXPECTED_SOURCE_SHA256.items()
):

    source = get_code_source(
        cell_index
    )

    actual_sha = hashlib.sha256(
        source.encode(
            "utf-8"
        )
    ).hexdigest()

    observed_source_sha256[
        cell_index
    ] = actual_sha

    assert actual_sha == expected_sha, (
        f"Historical cell {cell_index} changed. STOP."
    )


historical_source_gate = all(
    observed_source_sha256[index]
    == expected

    for index, expected
    in EXPECTED_SOURCE_SHA256.items()
)

assert historical_source_gate


# ============================================================
# 3. SOURCE EXTRACTION HELPERS
#
# Extracts exact source nodes/regions from approved cells.
# Does not reconstruct implementations by hand.
# ============================================================

def extract_named_function(
    cell_index,
    function_name,
):

    source = get_code_source(
        cell_index
    )

    tree = ast.parse(
        source
    )

    matches = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and node.name == function_name
        )
    ]

    assert len(matches) == 1

    extracted = ast.get_source_segment(
        source,
        matches[0],
    )

    assert extracted is not None

    return extracted


def extract_named_assignment(
    cell_index,
    variable_name,
):

    source = get_code_source(
        cell_index
    )

    tree = ast.parse(
        source
    )

    matches = []

    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        names = [
            target.id
            for target in node.targets
            if isinstance(
                target,
                ast.Name,
            )
        ]

        if variable_name in names:
            matches.append(
                node
            )

    assert len(matches) == 1

    extracted = ast.get_source_segment(
        source,
        matches[0],
    )

    assert extracted is not None

    return extracted


def extract_between_markers(
    cell_index,
    start_marker,
    end_marker,
):

    source = get_code_source(
        cell_index
    )

    start = source.find(
        start_marker
    )

    end = source.find(
        end_marker,
        start,
    )

    assert start >= 0
    assert end > start

    return source[
        start:end
    ]


# ============================================================
# 4. EXACT HISTORICAL LOCAL ROOT + VIEW ORDER
# ============================================================

exec(
    compile(
        extract_named_assignment(
            21,
            "LOCAL_DATA_ROOT",
        ),
        str(NOTEBOOK06_PATH)
        + "::CELL21_LOCAL_DATA_ROOT",
        "exec",
    ),
    globals(),
)


exec(
    compile(
        extract_named_assignment(
            21,
            "VIEW_FIELDS",
        ),
        str(NOTEBOOK06_PATH)
        + "::CELL21_VIEW_FIELDS",
        "exec",
    ),
    globals(),
)


assert LOCAL_DATA_ROOT == (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

assert VIEW_FIELDS == [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


# ============================================================
# 5. LABEL-FREE RECOVERY EVENT PATH MAP
#
# We deliberately do NOT open manifest_events_272.csv.
#
# A4B-2 already validated the complete sequential mapping:
#   event m -> PNG 4m+1 ... 4m+4
# over all 8583 events.
#
# Relative path format matches the approved preprocessing:
#   out272_gray/img_XXXXXX.png
# ============================================================

iic_event_rows = []


for manifest_index, base_row in enumerate(
    full_iic_rows
):

    assert (
        base_row[
            "manifest_index"
        ]
        == manifest_index
    )

    assert (
        base_row[
            "event_index"
        ]
        == manifest_index + 1
    )


    first_png_index = (
        4 * manifest_index
        + 1
    )


    iic_event_rows.append({

        "manifest_index":
            manifest_index,

        "event_index":
            manifest_index + 1,

        "event_id":
            base_row[
                "event_id"
            ],

        "img_0_5_272":
            (
                "out272_gray/"
                f"img_{first_png_index:06d}.png"
            ),

        "img_1_0_272":
            (
                "out272_gray/"
                f"img_{first_png_index + 1:06d}.png"
            ),

        "img_2_0_272":
            (
                "out272_gray/"
                f"img_{first_png_index + 2:06d}.png"
            ),

        "img_4_0_272":
            (
                "out272_gray/"
                f"img_{first_png_index + 3:06d}.png"
            ),
    })


event_map_gate = (

    len(
        iic_event_rows
    ) == 8583

    and all(
        row[
            "manifest_index"
        ] == index

        for index, row
        in enumerate(
            iic_event_rows
        )
    )

    and all(
        row[
            "event_index"
        ] == index + 1

        for index, row
        in enumerate(
            iic_event_rows
        )
    )

    and len({
        row[
            "event_id"
        ]
        for row
        in iic_event_rows
    }) == 8583

    and all(
        "label" not in row
        for row
        in iic_event_rows
    )
)

assert event_map_gate


# Verify all 34,332 derived path references exist.
all_png_paths_exist = all(

    (
        LOCAL_DATA_ROOT
        / row[field]
    ).is_file()

    for row in iic_event_rows
    for field in VIEW_FIELDS
)

assert all_png_paths_exist


# ============================================================
# 6. EXACT PNG CROP LOADER FROM HISTORICAL CELL 23
# ============================================================

loader_source = extract_named_function(
    23,
    "load_png_event_crop_224",
)

loader_source_sha256 = hashlib.sha256(
    loader_source.encode(
        "utf-8"
    )
).hexdigest()


exec(
    compile(
        loader_source,
        str(NOTEBOOK06_PATH)
        + "::CELL23_load_png_event_crop_224",
        "exec",
    ),
    globals(),
)


assert (
    "load_png_event_crop_224"
    in globals()
)


# ============================================================
# 7. EXACT STATELESS AUGMENTATION + KNOWN ANSWERS
#
# Executes ONLY the historical source region from:
#   "EXACT STATELESS AUGMENTATION..."
# through crop-map hashes.
#
# Stops BEFORE the real-data audit.
# ============================================================

augmentation_source = extract_between_markers(
    25,
    "# EXACT STATELESS AUGMENTATION ALREADY VALIDATED IN NOTEBOOK 01",
    "# FIXED LABEL-BLIND REAL-DATA AUDIT SET",
)

augmentation_source_sha256 = hashlib.sha256(
    augmentation_source.encode(
        "utf-8"
    )
).hexdigest()


exec(
    compile(
        augmentation_source,
        str(NOTEBOOK06_PATH)
        + "::CELL25_AUGMENTATION_PRIMITIVES",
        "exec",
    ),
    globals(),
)


EXPECTED_CROP_HASHES = {

    "legacy_epoch0":
        "30dbe7e291812b3515999e729d57e6a4"
        "f5e12717762d3021f6601d8def952b30",

    "legacy_epoch1":
        "40c67f6e1ec6f4dbb5706efb7fe14687"
        "ccfb2bd47143d2fd21cfb17667c5d5de",

    "symmetric_epoch0":
        "bf9da0f12354ebb6843ce21e15f4831d"
        "f11849001741cbb40a7c3567bae59159",

    "symmetric_epoch1":
        "a118687a1ccb3b1062b57e697a05c2e4"
        "5eef121790fe0a3aa9d29dc6dbb9912d",
}


crop_hash_gate = all([

    LEGACY_CROP_EPOCH0_SHA256
        == EXPECTED_CROP_HASHES[
            "legacy_epoch0"
        ],

    LEGACY_CROP_EPOCH1_SHA256
        == EXPECTED_CROP_HASHES[
            "legacy_epoch1"
        ],

    SYMMETRIC_CROP_EPOCH0_SHA256
        == EXPECTED_CROP_HASHES[
            "symmetric_epoch0"
        ],

    SYMMETRIC_CROP_EPOCH1_SHA256
        == EXPECTED_CROP_HASHES[
            "symmetric_epoch1"
        ],
])

assert crop_hash_gate

assert augmentation_golden_gate
assert legacy_full_gate
assert symmetric_full_gate
assert dual_crop_distinct_gate


# ============================================================
# 8. EXACT REAL PERTURBED-BATCH BUILDER DEFINITION
#
# Definition only.
# It does NOT read any PNG until called.
# ============================================================

perturbed_batch_source = (
    extract_named_function(
        25,
        "make_real_perturbed_batch",
    )
)

perturbed_batch_source_sha256 = (
    hashlib.sha256(
        perturbed_batch_source.encode(
            "utf-8"
        )
    ).hexdigest()
)


exec(
    compile(
        perturbed_batch_source,
        str(NOTEBOOK06_PATH)
        + "::CELL25_make_real_perturbed_batch",
        "exec",
    ),
    globals(),
)


assert (
    "make_real_perturbed_batch"
    in globals()
)


# ============================================================
# 9. EXACT PROJECT SHUFFLE + KNOWN ANSWERS
#
# Executes historical source only through the full epoch-0
# shuffle known-answer.
#
# Stops before the real one-step IIC smoke.
# ============================================================

shuffle_source = extract_between_markers(
    29,
    "# EXACT PROJECT SHUFFLE FROM NOTEBOOK 01 / C",
    "SMOKE_MANIFEST_INDICES = (",
)

shuffle_source_sha256 = hashlib.sha256(
    shuffle_source.encode(
        "utf-8"
    )
).hexdigest()


exec(
    compile(
        shuffle_source,
        str(NOTEBOOK06_PATH)
        + "::CELL29_SHUFFLE_PRIMITIVES",
        "exec",
    ),
    globals(),
)


shuffle_gate = (
    shuffle_golden_gate
    and full_shuffle_gate
)

assert shuffle_gate


# ============================================================
# 10. ENCODER MUST STILL BE UNCHANGED
# ============================================================

encoder_state_a4c2a = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

full_model_state_a4c2a = (
    canonical_tree_sha256(
        official_final_model.state_dict()
    )
)


encoder_immutability_gate = (
    encoder_state_a4c2a
    ==
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

full_model_immutability_gate = (
    full_model_state_a4c2a
    ==
    "3ac5a1cecdee2aa204419290b5cec55f3"
    "e6a4a846beebaa3f6b4e495f6fe6096"
)

assert encoder_immutability_gate
assert full_model_immutability_gate


# ============================================================
# 11. SAFETY
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False
PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 12. FINAL GATE + EXPLICIT PASS VARIABLES
# ============================================================

a4c2a_gate = all([

    historical_source_gate,

    event_map_gate,

    all_png_paths_exist,

    crop_hash_gate,

    augmentation_golden_gate,

    legacy_full_gate,

    symmetric_full_gate,

    dual_crop_distinct_gate,

    shuffle_gate,

    encoder_immutability_gate,

    full_model_immutability_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    OFFICIAL_IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c2a_gate


NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS = bool(
    a4c2a_gate
)

READY_FOR_A4C2B_LEGACY_RUNTIME_AUDIT = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 13. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-2A "
    "RUNTIME PRIMITIVES RECOVERY ==="
)
print()


print("--- SOURCE ---")

print(
    "HISTORICAL_SOURCE_GATE =",
    historical_source_gate,
)

for index in [
    21,
    23,
    25,
    29,
]:

    print(
        f"CELL_{index}_SHA256 =",
        observed_source_sha256[
            index
        ],
    )


print()
print("--- LABEL-FREE EVENT MAP ---")

print(
    "EVENT_COUNT =",
    len(
        iic_event_rows
    ),
)

print(
    "EVENT_MAP_GATE =",
    event_map_gate,
)

print(
    "ALL_34332_PATHS_EXIST =",
    all_png_paths_exist,
)

print(
    "LABEL_FIELD_STORED =",
    any(
        "label" in row
        for row
        in iic_event_rows
    ),
)


print()
print("--- AUGMENTATION ---")

print(
    "AUGMENTATION_GOLDEN_GATE =",
    augmentation_golden_gate,
)

print(
    "LEGACY_FULL_GATE =",
    legacy_full_gate,
)

print(
    "SYMMETRIC_FULL_GATE =",
    symmetric_full_gate,
)

print(
    "DUAL_CROP_DISTINCT_GATE =",
    dual_crop_distinct_gate,
)

print(
    "LEGACY_CROP_EPOCH0_SHA256 =",
    LEGACY_CROP_EPOCH0_SHA256,
)

print(
    "LEGACY_CROP_EPOCH1_SHA256 =",
    LEGACY_CROP_EPOCH1_SHA256,
)

print(
    "SYMMETRIC_CROP_EPOCH0_SHA256 =",
    SYMMETRIC_CROP_EPOCH0_SHA256,
)

print(
    "SYMMETRIC_CROP_EPOCH1_SHA256 =",
    SYMMETRIC_CROP_EPOCH1_SHA256,
)

print(
    "CROP_HASH_GATE =",
    crop_hash_gate,
)


print()
print("--- SHUFFLE ---")

print(
    "SHUFFLE_GOLDEN_GATE =",
    shuffle_golden_gate,
)

print(
    "FULL_SHUFFLE_GATE =",
    full_shuffle_gate,
)

print(
    "FIRST_16_EPOCH0 =",
    full_permutation_epoch0[
        :16
    ],
)


print()
print("--- RECOVERED FUNCTIONS ---")

print(
    "load_png_event_crop_224 =",
    callable(
        load_png_event_crop_224
    ),
)

print(
    "make_real_perturbed_batch =",
    callable(
        make_real_perturbed_batch
    ),
)

print(
    "gs_augment_crop_left_for_event_mode =",
    callable(
        gs_augment_crop_left_for_event_mode
    ),
)

print(
    "vae_c_shuffle_make_permutation_py =",
    callable(
        vae_c_shuffle_make_permutation_py
    ),
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "PRIMARY_SYMMETRIC_RERUN =",
    PRIMARY_SYMMETRIC_RERUN,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS =",
    NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS,
)

print(
    "READY_FOR_A4C2B_LEGACY_RUNTIME_AUDIT =",
    READY_FOR_A4C2B_LEGACY_RUNTIME_AUDIT,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


=== NOTEBOOK 06 — A4C-2A RUNTIME PRIMITIVES RECOVERY ===

--- SOURCE ---
HISTORICAL_SOURCE_GATE = True
CELL_21_SHA256 = 0eefcc33a15c9380b40cc27567ac48f9988ce588cf528c85568fb4d3c719b7ac
CELL_23_SHA256 = ef608dabda3bb7d500f749783d51612d03297c84025383508750170f8415e53c
CELL_25_SHA256 = 12955163342f2b37824798493f5b044a90f314dea64a5e668a96eeb553cb15c5
CELL_29_SHA256 = 18aa232fd630423760a13cfb15e37637d6d6e69cd42438e749d9cfa869adf6f0

--- LABEL-FREE EVENT MAP ---
EVENT_COUNT = 8583
EVENT_MAP_GATE = True
ALL_34332_PATHS_EXIST = True
LABEL_FIELD_STORED = False

--- AUGMENTATION ---
AUGMENTATION_GOLDEN_GATE = True
LEGACY_FULL_GATE = True
SYMMETRIC_FULL_GATE = True
DUAL_CROP_DISTINCT_GATE = True
LEGACY_CROP_EPOCH0_SHA256 = 30dbe7e291812b3515999e729d57e6a4f5e12717762d3021f6601d8def952b30
LEGACY_CROP_EPOCH1_SHA256 = 40c67f6e1ec6f4dbb5706efb7fe14687ccfb2bd47143d2fd21cfb17667c5d5de
SYMMETRIC_CROP_EPOCH0_SHA256 = bf9da0f12354ebb6843ce21e15f4831df11849001741cbb40a7c3567bae59159
SYMMETRIC_CROP_EPOCH1_S

In [ ]:
# ============================================================
# NOTEBOOK 06 — A4C-2B
# LEGACY_TEXT REAL-DATA RUNTIME AUDIT
#
# Uses exact approved helpers from historical Notebook-06 cell 25.
#
# Tests:
#   - fixed legacy crop known answers
#   - real PNG batches at epoch 0 / epoch 1
#   - frozen FINAL encoder response
#   - response vs central mu
#   - epoch-to-epoch feature change
#   - encoder immutability
#
# NO labels.
# NO IIC official model.
# NO optimizer.
# NO optimizer step.
# NO training.
# NO spectral.
# NO Drive writes.
# ============================================================

from pathlib import Path
import gc
import hashlib

import numpy as np
import torch


# ============================================================
# 0. PARENT GATE
# ============================================================

assert (
    "NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS"
    in globals()
    and NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS is True
), "A4C-2A non disponibile o non PASS."


assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."


# ============================================================
# 1. HISTORICAL SOURCE IDENTITY
# ============================================================

EXPECTED_CELL25_SHA256 = (
    "12955163342f2b37824798493f5b044a"
    "90f314dea64a5e668a96eeb553cb15c5"
)

assert (
    observed_source_sha256[25]
    == EXPECTED_CELL25_SHA256
)


# ============================================================
# 2. RESTORE EXACT APPROVED AUDIT CONSTANTS
#    FROM HISTORICAL CELL 25
# ============================================================

for variable_name in [

    "AUDIT_MANIFEST_INDICES",

    "EXPECTED_LEGACY_EPOCH0",

    "EXPECTED_LEGACY_EPOCH1",

]:

    source = extract_named_assignment(
        25,
        variable_name,
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + f"::CELL25_{variable_name}"
            ),
            "exec",
        ),
        globals(),
    )


assert AUDIT_MANIFEST_INDICES == [
    0,
    2145,
    4291,
    6864,
    7663,
    8582,
]

assert EXPECTED_LEGACY_EPOCH0 == [
    7,
    21,
    1,
    20,
    6,
    21,
]

assert EXPECTED_LEGACY_EPOCH1 == [
    24,
    7,
    15,
    3,
    10,
    12,
]


# ============================================================
# 3. RESTORE EXACT APPROVED AUDIT HELPERS
#    FROM HISTORICAL CELL 25
# ============================================================

for function_name in [

    "real_batch_valid",

    "frozen_encoder_forward",

    "feature_matrix_valid",

    "central_response_gate",

    "float32_sha256",

]:

    source = extract_named_function(
        25,
        function_name,
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + f"::CELL25_{function_name}"
            ),
            "exec",
        ),
        globals(),
    )


# ============================================================
# 4. CENTRAL REFERENCE — EXACT HISTORICAL ASSIGNMENT
# ============================================================

central_reference_source = (
    extract_named_assignment(
        25,
        "central_mu_audit",
    )
)

exec(
    compile(
        central_reference_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL25_central_mu_audit"
        ),
        "exec",
    ),
    globals(),
)


assert central_mu_audit.shape == (
    6,
    512,
)

assert central_mu_audit.dtype == np.float32

assert np.isfinite(
    central_mu_audit
).all()


# ============================================================
# 5. LEGACY CROP KNOWN ANSWERS
# ============================================================

legacy_selected_epoch0 = [
    int(
        legacy_crop_epoch0[
            manifest_index
        ]
    )
    for manifest_index
    in AUDIT_MANIFEST_INDICES
]

legacy_selected_epoch1 = [
    int(
        legacy_crop_epoch1[
            manifest_index
        ]
    )
    for manifest_index
    in AUDIT_MANIFEST_INDICES
]


legacy_fixed_crop_gate = (
    legacy_selected_epoch0
    == EXPECTED_LEGACY_EPOCH0

    and legacy_selected_epoch1
    == EXPECTED_LEGACY_EPOCH1
)

assert legacy_fixed_crop_gate


# ============================================================
# 6. ENCODER PRE-AUDIT IDENTITY
# ============================================================

EXPECTED_ENCODER_FINAL_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)


encoder_state_before_legacy_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


assert (
    encoder_state_before_legacy_audit
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)

assert (
    official_iic_encoder.training
    is False
)

assert all(
    parameter.requires_grad is False
    for parameter
    in official_iic_encoder.parameters()
)

assert all(
    parameter.grad is None
    for parameter
    in official_iic_encoder.parameters()
)


# ============================================================
# 7. BUILD TWO REAL LEGACY BATCHES
# ============================================================

(
    legacy_batch_epoch0_cpu,
    legacy_batch_crops_epoch0,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=0,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


(
    legacy_batch_epoch1_cpu,
    legacy_batch_crops_epoch1,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=1,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


legacy_batch_crop_gate = (
    legacy_batch_crops_epoch0
    == EXPECTED_LEGACY_EPOCH0

    and legacy_batch_crops_epoch1
    == EXPECTED_LEGACY_EPOCH1
)

assert legacy_batch_crop_gate


legacy_input_gate = (
    real_batch_valid(
        legacy_batch_epoch0_cpu
    )
    and
    real_batch_valid(
        legacy_batch_epoch1_cpu
    )
)

assert legacy_input_gate


# ============================================================
# 8. FROZEN ENCODER FORWARD
# ============================================================

(
    legacy_mu_epoch0,
    legacy_logvar_epoch0,
) = frozen_encoder_forward(
    legacy_batch_epoch0_cpu
)


(
    legacy_mu_epoch1,
    legacy_logvar_epoch1,
) = frozen_encoder_forward(
    legacy_batch_epoch1_cpu
)


if runtime_device.type == "cuda":
    torch.cuda.synchronize()


legacy_feature_gate = (
    feature_matrix_valid(
        legacy_mu_epoch0,
        legacy_logvar_epoch0,
    )

    and

    feature_matrix_valid(
        legacy_mu_epoch1,
        legacy_logvar_epoch1,
    )
)

assert legacy_feature_gate


# ============================================================
# 9. RESPONSE AGAINST CENTRAL MU
#
# Exact historical rule:
#
# crop_left == 24:
#   mu must reproduce central reference numerically.
#
# crop_left != 24:
#   latent feature must respond (L2 > 0).
# ============================================================

legacy_response_epoch0_gate = (
    central_response_gate(
        legacy_mu_epoch0,
        legacy_batch_crops_epoch0,
    )
)

legacy_response_epoch1_gate = (
    central_response_gate(
        legacy_mu_epoch1,
        legacy_batch_crops_epoch1,
    )
)


legacy_response_gate = (
    legacy_response_epoch0_gate
    and legacy_response_epoch1_gate
)

assert legacy_response_gate


# ============================================================
# 10. EPOCH-TO-EPOCH FEATURE RESPONSE
# ============================================================

legacy_epoch_l2 = np.linalg.norm(
    legacy_mu_epoch0
    - legacy_mu_epoch1,
    axis=1,
)


legacy_epoch_feature_change_gate = bool(
    np.all(
        legacy_epoch_l2 > 0.0
    )
)

assert legacy_epoch_feature_change_gate


# ============================================================
# 11. AUDIT FEATURE HASHES
#
# Historical hashes are diagnostic identities recorded by
# the original Notebook 06 cell 25.
#
# They are reported explicitly.
# The scientific blocking gates remain the exact original
# numerical/response checks above.
# ============================================================

LEGACY_MU_EPOCH0_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch0
    )
)

LEGACY_MU_EPOCH1_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch1
    )
)


HISTORICAL_LEGACY_MU_EPOCH0_SHA256 = (
    "e519fd3a97f9c9bdfe530106d04015b2"
    "7d4ea5b6f7810971f992c92a9b127b8f"
)

HISTORICAL_LEGACY_MU_EPOCH1_SHA256 = (
    "e2c0b06fff14a5f013ffe5705689f198"
    "25a17d5a71bc974a639442fdfb7b6e51"
)


legacy_epoch0_hash_match = (
    LEGACY_MU_EPOCH0_AUDIT_SHA256
    == HISTORICAL_LEGACY_MU_EPOCH0_SHA256
)

legacy_epoch1_hash_match = (
    LEGACY_MU_EPOCH1_AUDIT_SHA256
    == HISTORICAL_LEGACY_MU_EPOCH1_SHA256
)


# ============================================================
# 12. DESCRIPTIVE CENTRAL DISTANCES
# ============================================================

legacy_central_l2_epoch0 = np.linalg.norm(
    legacy_mu_epoch0
    - central_mu_audit,
    axis=1,
)

legacy_central_l2_epoch1 = np.linalg.norm(
    legacy_mu_epoch1
    - central_mu_audit,
    axis=1,
)


# ============================================================
# 13. ENCODER IMMUTABILITY AFTER REAL DATA
# ============================================================

encoder_state_after_legacy_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_immutability_gate = (
    encoder_state_before_legacy_audit
    == encoder_state_after_legacy_audit
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)

assert encoder_immutability_gate


encoder_policy_gate = (
    official_iic_encoder.training is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_policy_gate


# ============================================================
# 14. LEGACY RUN COLLISION + SAFETY
# ============================================================

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

legacy_collision_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_collision_gate


LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 15. FINAL A4C-2B GATE
# ============================================================

a4c2b_gate = all([

    legacy_fixed_crop_gate,

    legacy_batch_crop_gate,

    legacy_input_gate,

    legacy_feature_gate,

    legacy_response_gate,

    legacy_epoch_feature_change_gate,

    encoder_immutability_gate,

    encoder_policy_gate,

    legacy_collision_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    OFFICIAL_IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c2b_gate


NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS = bool(
    a4c2b_gate
)

READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 16. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-2B "
    "LEGACY_TEXT REAL-DATA AUDIT ==="
)
print()


print("--- AUDIT SET ---")

print(
    "AUDIT_MANIFEST_INDICES =",
    AUDIT_MANIFEST_INDICES,
)

print(
    "LEGACY_CROPS_EPOCH0 =",
    legacy_batch_crops_epoch0,
)

print(
    "LEGACY_CROPS_EPOCH1 =",
    legacy_batch_crops_epoch1,
)

print(
    "LEGACY_FIXED_CROP_GATE =",
    legacy_fixed_crop_gate,
)

print(
    "LEGACY_INPUT_GATE =",
    legacy_input_gate,
)


print()
print("--- FEATURE RESPONSE ---")

print(
    "LEGACY_FEATURE_GATE =",
    legacy_feature_gate,
)

print(
    "LEGACY_RESPONSE_EPOCH0_GATE =",
    legacy_response_epoch0_gate,
)

print(
    "LEGACY_RESPONSE_EPOCH1_GATE =",
    legacy_response_epoch1_gate,
)

print(
    "LEGACY_EPOCH_FEATURE_CHANGE_GATE =",
    legacy_epoch_feature_change_gate,
)

print(
    "LEGACY_EPOCH0_TO_1_L2 =",
    legacy_epoch_l2.tolist(),
)


print()
print("--- CENTRAL L2 ---")

print(
    "LEGACY_CENTRAL_L2_EPOCH0 =",
    legacy_central_l2_epoch0.tolist(),
)

print(
    "LEGACY_CENTRAL_L2_EPOCH1 =",
    legacy_central_l2_epoch1.tolist(),
)


print()
print("--- FEATURE HASHES ---")

print(
    "LEGACY_MU_EPOCH0_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH0_AUDIT_SHA256,
)

print(
    "LEGACY_MU_EPOCH1_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH1_AUDIT_SHA256,
)

print(
    "HISTORICAL_EPOCH0_HASH_MATCH =",
    legacy_epoch0_hash_match,
)

print(
    "HISTORICAL_EPOCH1_HASH_MATCH =",
    legacy_epoch1_hash_match,
)


print()
print("--- ENCODER ---")

print(
    "ENCODER_STATE_BEFORE =",
    encoder_state_before_legacy_audit,
)

print(
    "ENCODER_STATE_AFTER =",
    encoder_state_after_legacy_audit,
)

print(
    "ENCODER_IMMUTABILITY_GATE =",
    encoder_immutability_gate,
)

print(
    "ENCODER_POLICY_GATE =",
    encoder_policy_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS =",
    NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS,
)

print(
    "READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY =",
    READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


# ============================================================
# 17. CLEAN TEMPORARY REAL BATCHES / FEATURES
# ============================================================

for name in [

    "legacy_batch_epoch0_cpu",
    "legacy_batch_epoch1_cpu",

    "legacy_mu_epoch0",
    "legacy_mu_epoch1",

    "legacy_logvar_epoch0",
    "legacy_logvar_epoch1",

    "legacy_epoch_l2",

    "legacy_central_l2_epoch0",
    "legacy_central_l2_epoch1",

]:

    globals().pop(
        name,
        None,
    )


gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()

In [12]:
# ============================================================
# NOTEBOOK 06 — A4C-2B
# LEGACY_TEXT REAL-DATA RUNTIME AUDIT
#
# Uses exact approved helpers from historical Notebook-06 cell 25.
#
# Tests:
#   - fixed legacy crop known answers
#   - real PNG batches at epoch 0 / epoch 1
#   - frozen FINAL encoder response
#   - response vs central mu
#   - epoch-to-epoch feature change
#   - encoder immutability
#
# NO labels.
# NO IIC official model.
# NO optimizer.
# NO optimizer step.
# NO training.
# NO spectral.
# NO Drive writes.
# ============================================================

from pathlib import Path
import gc
import hashlib

import numpy as np
import torch


# ============================================================
# 0. PARENT GATE
# ============================================================

assert (
    "NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS"
    in globals()
    and NOTEBOOK_06_A4C2A_RUNTIME_PRIMITIVES_PASS is True
), "A4C-2A non disponibile o non PASS."


assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."


# ============================================================
# 1. HISTORICAL SOURCE IDENTITY
# ============================================================

EXPECTED_CELL25_SHA256 = (
    "12955163342f2b37824798493f5b044a"
    "90f314dea64a5e668a96eeb553cb15c5"
)

assert (
    observed_source_sha256[25]
    == EXPECTED_CELL25_SHA256
)


# ============================================================
# 2. RESTORE EXACT APPROVED AUDIT CONSTANTS
#    FROM HISTORICAL CELL 25
# ============================================================

for variable_name in [

    "AUDIT_MANIFEST_INDICES",

    "EXPECTED_LEGACY_EPOCH0",

    "EXPECTED_LEGACY_EPOCH1",

]:

    source = extract_named_assignment(
        25,
        variable_name,
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + f"::CELL25_{variable_name}"
            ),
            "exec",
        ),
        globals(),
    )


assert AUDIT_MANIFEST_INDICES == [
    0,
    2145,
    4291,
    6864,
    7663,
    8582,
]

assert EXPECTED_LEGACY_EPOCH0 == [
    7,
    21,
    1,
    20,
    6,
    21,
]

assert EXPECTED_LEGACY_EPOCH1 == [
    24,
    7,
    15,
    3,
    10,
    12,
]


# ============================================================
# 3. RESTORE EXACT APPROVED AUDIT HELPERS
#    FROM HISTORICAL CELL 25
# ============================================================

for function_name in [

    "real_batch_valid",

    "frozen_encoder_forward",

    "feature_matrix_valid",

    "central_response_gate",

    "float32_sha256",

]:

    source = extract_named_function(
        25,
        function_name,
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + f"::CELL25_{function_name}"
            ),
            "exec",
        ),
        globals(),
    )


# ============================================================
# 4. CENTRAL REFERENCE — EXACT HISTORICAL ASSIGNMENT
# ============================================================

central_reference_source = (
    extract_named_assignment(
        25,
        "central_mu_audit",
    )
)

exec(
    compile(
        central_reference_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL25_central_mu_audit"
        ),
        "exec",
    ),
    globals(),
)


assert central_mu_audit.shape == (
    6,
    512,
)

assert central_mu_audit.dtype == np.float32

assert np.isfinite(
    central_mu_audit
).all()


# ============================================================
# 5. LEGACY CROP KNOWN ANSWERS
# ============================================================

legacy_selected_epoch0 = [
    int(
        legacy_crop_epoch0[
            manifest_index
        ]
    )
    for manifest_index
    in AUDIT_MANIFEST_INDICES
]

legacy_selected_epoch1 = [
    int(
        legacy_crop_epoch1[
            manifest_index
        ]
    )
    for manifest_index
    in AUDIT_MANIFEST_INDICES
]


legacy_fixed_crop_gate = (
    legacy_selected_epoch0
    == EXPECTED_LEGACY_EPOCH0

    and legacy_selected_epoch1
    == EXPECTED_LEGACY_EPOCH1
)

assert legacy_fixed_crop_gate


# ============================================================
# 6. ENCODER PRE-AUDIT IDENTITY
# ============================================================

EXPECTED_ENCODER_FINAL_STATE_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)


encoder_state_before_legacy_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


assert (
    encoder_state_before_legacy_audit
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)

assert (
    official_iic_encoder.training
    is False
)

assert all(
    parameter.requires_grad is False
    for parameter
    in official_iic_encoder.parameters()
)

assert all(
    parameter.grad is None
    for parameter
    in official_iic_encoder.parameters()
)


# ============================================================
# 7. BUILD TWO REAL LEGACY BATCHES
# ============================================================

(
    legacy_batch_epoch0_cpu,
    legacy_batch_crops_epoch0,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=0,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


(
    legacy_batch_epoch1_cpu,
    legacy_batch_crops_epoch1,
) = make_real_perturbed_batch(
    AUDIT_MANIFEST_INDICES,
    epoch=1,
    mode=GS_AUGMENT_MODE_LEGACY_TEXT,
)


legacy_batch_crop_gate = (
    legacy_batch_crops_epoch0
    == EXPECTED_LEGACY_EPOCH0

    and legacy_batch_crops_epoch1
    == EXPECTED_LEGACY_EPOCH1
)

assert legacy_batch_crop_gate


legacy_input_gate = (
    real_batch_valid(
        legacy_batch_epoch0_cpu
    )
    and
    real_batch_valid(
        legacy_batch_epoch1_cpu
    )
)

assert legacy_input_gate


# ============================================================
# 8. FROZEN ENCODER FORWARD
# ============================================================

(
    legacy_mu_epoch0,
    legacy_logvar_epoch0,
) = frozen_encoder_forward(
    legacy_batch_epoch0_cpu
)


(
    legacy_mu_epoch1,
    legacy_logvar_epoch1,
) = frozen_encoder_forward(
    legacy_batch_epoch1_cpu
)


if runtime_device.type == "cuda":
    torch.cuda.synchronize()


legacy_feature_gate = (
    feature_matrix_valid(
        legacy_mu_epoch0,
        legacy_logvar_epoch0,
    )

    and

    feature_matrix_valid(
        legacy_mu_epoch1,
        legacy_logvar_epoch1,
    )
)

assert legacy_feature_gate


# ============================================================
# 9. RESPONSE AGAINST CENTRAL MU
#
# Exact historical rule:
#
# crop_left == 24:
#   mu must reproduce central reference numerically.
#
# crop_left != 24:
#   latent feature must respond (L2 > 0).
# ============================================================

legacy_response_epoch0_gate = (
    central_response_gate(
        legacy_mu_epoch0,
        legacy_batch_crops_epoch0,
    )
)

legacy_response_epoch1_gate = (
    central_response_gate(
        legacy_mu_epoch1,
        legacy_batch_crops_epoch1,
    )
)


legacy_response_gate = (
    legacy_response_epoch0_gate
    and legacy_response_epoch1_gate
)

assert legacy_response_gate


# ============================================================
# 10. EPOCH-TO-EPOCH FEATURE RESPONSE
# ============================================================

legacy_epoch_l2 = np.linalg.norm(
    legacy_mu_epoch0
    - legacy_mu_epoch1,
    axis=1,
)


legacy_epoch_feature_change_gate = bool(
    np.all(
        legacy_epoch_l2 > 0.0
    )
)

assert legacy_epoch_feature_change_gate


# ============================================================
# 11. AUDIT FEATURE HASHES
#
# Historical hashes are diagnostic identities recorded by
# the original Notebook 06 cell 25.
#
# They are reported explicitly.
# The scientific blocking gates remain the exact original
# numerical/response checks above.
# ============================================================

LEGACY_MU_EPOCH0_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch0
    )
)

LEGACY_MU_EPOCH1_AUDIT_SHA256 = (
    float32_sha256(
        legacy_mu_epoch1
    )
)


HISTORICAL_LEGACY_MU_EPOCH0_SHA256 = (
    "e519fd3a97f9c9bdfe530106d04015b2"
    "7d4ea5b6f7810971f992c92a9b127b8f"
)

HISTORICAL_LEGACY_MU_EPOCH1_SHA256 = (
    "e2c0b06fff14a5f013ffe5705689f198"
    "25a17d5a71bc974a639442fdfb7b6e51"
)


legacy_epoch0_hash_match = (
    LEGACY_MU_EPOCH0_AUDIT_SHA256
    == HISTORICAL_LEGACY_MU_EPOCH0_SHA256
)

legacy_epoch1_hash_match = (
    LEGACY_MU_EPOCH1_AUDIT_SHA256
    == HISTORICAL_LEGACY_MU_EPOCH1_SHA256
)


# ============================================================
# 12. DESCRIPTIVE CENTRAL DISTANCES
# ============================================================

legacy_central_l2_epoch0 = np.linalg.norm(
    legacy_mu_epoch0
    - central_mu_audit,
    axis=1,
)

legacy_central_l2_epoch1 = np.linalg.norm(
    legacy_mu_epoch1
    - central_mu_audit,
    axis=1,
)


# ============================================================
# 13. ENCODER IMMUTABILITY AFTER REAL DATA
# ============================================================

encoder_state_after_legacy_audit = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)


encoder_immutability_gate = (
    encoder_state_before_legacy_audit
    == encoder_state_after_legacy_audit
    == EXPECTED_ENCODER_FINAL_STATE_SHA256
)

assert encoder_immutability_gate


encoder_policy_gate = (
    official_iic_encoder.training is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_policy_gate


# ============================================================
# 14. LEGACY RUN COLLISION + SAFETY
# ============================================================

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

legacy_collision_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_collision_gate


LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 15. FINAL A4C-2B GATE
# ============================================================

a4c2b_gate = all([

    legacy_fixed_crop_gate,

    legacy_batch_crop_gate,

    legacy_input_gate,

    legacy_feature_gate,

    legacy_response_gate,

    legacy_epoch_feature_change_gate,

    encoder_immutability_gate,

    encoder_policy_gate,

    legacy_collision_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    OFFICIAL_IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c2b_gate


NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS = bool(
    a4c2b_gate
)

READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 16. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-2B "
    "LEGACY_TEXT REAL-DATA AUDIT ==="
)
print()


print("--- AUDIT SET ---")

print(
    "AUDIT_MANIFEST_INDICES =",
    AUDIT_MANIFEST_INDICES,
)

print(
    "LEGACY_CROPS_EPOCH0 =",
    legacy_batch_crops_epoch0,
)

print(
    "LEGACY_CROPS_EPOCH1 =",
    legacy_batch_crops_epoch1,
)

print(
    "LEGACY_FIXED_CROP_GATE =",
    legacy_fixed_crop_gate,
)

print(
    "LEGACY_INPUT_GATE =",
    legacy_input_gate,
)


print()
print("--- FEATURE RESPONSE ---")

print(
    "LEGACY_FEATURE_GATE =",
    legacy_feature_gate,
)

print(
    "LEGACY_RESPONSE_EPOCH0_GATE =",
    legacy_response_epoch0_gate,
)

print(
    "LEGACY_RESPONSE_EPOCH1_GATE =",
    legacy_response_epoch1_gate,
)

print(
    "LEGACY_EPOCH_FEATURE_CHANGE_GATE =",
    legacy_epoch_feature_change_gate,
)

print(
    "LEGACY_EPOCH0_TO_1_L2 =",
    legacy_epoch_l2.tolist(),
)


print()
print("--- CENTRAL L2 ---")

print(
    "LEGACY_CENTRAL_L2_EPOCH0 =",
    legacy_central_l2_epoch0.tolist(),
)

print(
    "LEGACY_CENTRAL_L2_EPOCH1 =",
    legacy_central_l2_epoch1.tolist(),
)


print()
print("--- FEATURE HASHES ---")

print(
    "LEGACY_MU_EPOCH0_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH0_AUDIT_SHA256,
)

print(
    "LEGACY_MU_EPOCH1_AUDIT_SHA256 =",
    LEGACY_MU_EPOCH1_AUDIT_SHA256,
)

print(
    "HISTORICAL_EPOCH0_HASH_MATCH =",
    legacy_epoch0_hash_match,
)

print(
    "HISTORICAL_EPOCH1_HASH_MATCH =",
    legacy_epoch1_hash_match,
)


print()
print("--- ENCODER ---")

print(
    "ENCODER_STATE_BEFORE =",
    encoder_state_before_legacy_audit,
)

print(
    "ENCODER_STATE_AFTER =",
    encoder_state_after_legacy_audit,
)

print(
    "ENCODER_IMMUTABILITY_GATE =",
    encoder_immutability_gate,
)

print(
    "ENCODER_POLICY_GATE =",
    encoder_policy_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS =",
    NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS,
)

print(
    "READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY =",
    READY_FOR_A4C3_TRAINING_RUNTIME_RECOVERY,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


# ============================================================
# 17. CLEAN TEMPORARY REAL BATCHES / FEATURES
# ============================================================

for name in [

    "legacy_batch_epoch0_cpu",
    "legacy_batch_epoch1_cpu",

    "legacy_mu_epoch0",
    "legacy_mu_epoch1",

    "legacy_logvar_epoch0",
    "legacy_logvar_epoch1",

    "legacy_epoch_l2",

    "legacy_central_l2_epoch0",
    "legacy_central_l2_epoch1",

]:

    globals().pop(
        name,
        None,
    )


gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()


=== NOTEBOOK 06 — A4C-2B LEGACY_TEXT REAL-DATA AUDIT ===

--- AUDIT SET ---
AUDIT_MANIFEST_INDICES = [0, 2145, 4291, 6864, 7663, 8582]
LEGACY_CROPS_EPOCH0 = [7, 21, 1, 20, 6, 21]
LEGACY_CROPS_EPOCH1 = [24, 7, 15, 3, 10, 12]
LEGACY_FIXED_CROP_GATE = True
LEGACY_INPUT_GATE = True

--- FEATURE RESPONSE ---
LEGACY_FEATURE_GATE = True
LEGACY_RESPONSE_EPOCH0_GATE = True
LEGACY_RESPONSE_EPOCH1_GATE = True
LEGACY_EPOCH_FEATURE_CHANGE_GATE = True
LEGACY_EPOCH0_TO_1_L2 = [1.5108063220977783, 16.80643653869629, 5.967932224273682, 1.9433543682098389, 4.161664962768555, 1.2213338613510132]

--- CENTRAL L2 ---
LEGACY_CENTRAL_L2_EPOCH0 = [1.5108062028884888, 5.057406425476074, 7.1506829261779785, 0.583527684211731, 12.403892517089844, 0.687053382396698]
LEGACY_CENTRAL_L2_EPOCH1 = [3.376922222741996e-06, 16.646215438842773, 3.6756720542907715, 2.0584983825683594, 12.035293579101562, 1.6039574146270752]

--- FEATURE HASHES ---
LEGACY_MU_EPOCH0_AUDIT_SHA256 = e519fd3a97f9c9bdfe530106d04015b27d4ea5b6f78

In [15]:
# ============================================================
# NOTEBOOK 06 — A4C-3A
# DUAL-EXPERIMENT + OFFICIAL-RUN CONTRACT RECOVERY
#
# CONTRACTS ONLY.
#
# NO model creation.
# NO optimizer.
# NO training.
# NO spectral.
# NO labels.
# NO Drive writes.
# ============================================================

from pathlib import Path
import hashlib
import json
import re


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS"
    in globals()
    and NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS is True
), "A4C-2B non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4C1_IIC_CORE_RECOVERY_PASS is True
), "A4C-1 non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."


# ============================================================
# 1. APPROVED NOTEBOOK 06
# ============================================================

NOTEBOOK06_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH.is_file()

with NOTEBOOK06_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    approved_nb06 = json.load(
        handle
    )


def cell_source(index):

    cell = approved_nb06[
        "cells"
    ][index]

    assert cell[
        "cell_type"
    ] == "code"

    return "".join(
        cell.get(
            "source",
            [],
        )
    )


# ============================================================
# 2. HISTORICAL SOURCE IDENTITIES
# ============================================================

EXPECTED_CELL25_SHA256 = (
    "12955163342f2b37824798493f5b044a"
    "90f314dea64a5e668a96eeb553cb15c5"
)

EXPECTED_CELL27_SHA256 = (
    "5e16fdee93ee0d48d281ea77f1895b7"
    "4f32ed56c9b8e3d924fa10a9f541255b2"
)

EXPECTED_CELL34_SHA256 = (
    "25ee1dcfbc530e5d103f19d37b8b164b"
    "f342217db402b7bf46738052163a3d85"
)


observed_cell25_sha256 = hashlib.sha256(
    cell_source(25).encode(
        "utf-8"
    )
).hexdigest()

observed_cell27_sha256 = hashlib.sha256(
    cell_source(27).encode(
        "utf-8"
    )
).hexdigest()

observed_cell34_sha256 = hashlib.sha256(
    cell_source(34).encode(
        "utf-8"
    )
).hexdigest()


historical_source_gate = all([

    observed_cell25_sha256
        == EXPECTED_CELL25_SHA256,

    observed_cell27_sha256
        == EXPECTED_CELL27_SHA256,

    observed_cell34_sha256
        == EXPECTED_CELL34_SHA256,
])

assert historical_source_gate


# ============================================================
# 3. RECOVER HISTORICAL SYMMETRIC FEATURE-AUDIT HASHES
#
# We do NOT rerun symmetric.
#
# We recover the identities from the already stored output
# of historical approved cell 25.
# ============================================================

cell25_output_text = ""


for output in approved_nb06[
    "cells"
][25].get(
    "outputs",
    [],
):

    if "text" in output:

        cell25_output_text += "".join(
            output["text"]
        )

    elif (
        "data" in output
        and "text/plain" in output["data"]
    ):

        cell25_output_text += "".join(
            output["data"][
                "text/plain"
            ]
        )


def extract_historical_sha(
    variable_name,
):

    match = re.search(
        rf"{re.escape(variable_name)}\s*=\s*([0-9a-f]{{64}})",
        cell25_output_text,
    )

    assert match is not None, (
        f"Historical output missing {variable_name}"
    )

    return match.group(1)


HISTORICAL_LEGACY_MU_EPOCH0_SHA256 = (
    extract_historical_sha(
        "LEGACY_MU_EPOCH0_AUDIT_SHA256"
    )
)

HISTORICAL_LEGACY_MU_EPOCH1_SHA256 = (
    extract_historical_sha(
        "LEGACY_MU_EPOCH1_AUDIT_SHA256"
    )
)

SYMMETRIC_MU_EPOCH0_AUDIT_SHA256 = (
    extract_historical_sha(
        "SYMMETRIC_MU_EPOCH0_AUDIT_SHA256"
    )
)

SYMMETRIC_MU_EPOCH1_AUDIT_SHA256 = (
    extract_historical_sha(
        "SYMMETRIC_MU_EPOCH1_AUDIT_SHA256"
    )
)


historical_legacy_identity_gate = (

    LEGACY_MU_EPOCH0_AUDIT_SHA256
        == HISTORICAL_LEGACY_MU_EPOCH0_SHA256

    and

    LEGACY_MU_EPOCH1_AUDIT_SHA256
        == HISTORICAL_LEGACY_MU_EPOCH1_SHA256
)

assert historical_legacy_identity_gate


EXPECTED_SYMMETRIC_MU_EPOCH0_SHA256 = (
    "0b7eb867dfee1820f877c76df534b48a"
    "f6b0c14e3a00fe137db72d8133d17aab"
)

EXPECTED_SYMMETRIC_MU_EPOCH1_SHA256 = (
    "574eb15f3336d881bb0cd9733034734d"
    "35a806c03a86781cbea4e1eec06caebc"
)


historical_symmetric_identity_gate = (

    SYMMETRIC_MU_EPOCH0_AUDIT_SHA256
        == EXPECTED_SYMMETRIC_MU_EPOCH0_SHA256

    and

    SYMMETRIC_MU_EPOCH1_AUDIT_SHA256
        == EXPECTED_SYMMETRIC_MU_EPOCH1_SHA256
)

assert historical_symmetric_identity_gate


# ============================================================
# 4. RECOVER DUAL EXPERIMENT CONTRACT
#    EXACTLY FROM HISTORICAL CELL 27
#
# Skip only its obsolete parent-gate preamble and report.
# ============================================================

cell27 = cell_source(
    27
)

dual_start_marker = (
    "# IDENTIFY THE EXACT A4C AUDIT"
)

dual_end_marker = (
    "# REPORT"
)

dual_start = cell27.find(
    dual_start_marker
)

dual_end = cell27.find(
    dual_end_marker,
    dual_start,
)

assert dual_start >= 0
assert dual_end > dual_start


dual_contract_source = cell27[
    dual_start:
    dual_end
]


exec(
    compile(
        dual_contract_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL27_DUAL_CONTRACT"
        ),
        "exec",
    ),
    globals(),
)


EXPECTED_DUAL_EXPERIMENT_SHA256 = (
    "4fbae488ccbe6226c4c589c8bfcd712e"
    "d20e0ddf65ec95e4dfb80120d8432fbc"
)


dual_contract_hash_gate = (
    IIC_DUAL_EXPERIMENT_SHA256
    == EXPECTED_DUAL_EXPERIMENT_SHA256
)

assert dual_contract_hash_gate


# ============================================================
# 5. VERIFY SINGLE SCIENTIFIC DIFFERENCE
# ============================================================

shared_protocol = (
    IIC_DUAL_EXPERIMENT_CONTRACT[
        "shared_protocol"
    ]
)

legacy_contract = (
    IIC_EXPERIMENTS[
        "legacy_text"
    ]
)

symmetric_contract = (
    IIC_EXPERIMENTS[
        "symmetric_figure"
    ]
)


single_difference_gate = all([

    symmetric_contract[
        "run_id"
    ]
        == "sakai224_full_iic_symmetric_v1",

    legacy_contract[
        "run_id"
    ]
        == "sakai224_full_iic_legacy_text_v1",

    symmetric_contract[
        "augmentation_mode"
    ]
        == "symmetric_figure",

    legacy_contract[
        "augmentation_mode"
    ]
        == "legacy_text",

    symmetric_contract[
        "crop_left_range"
    ] == [0, 48],

    legacy_contract[
        "crop_left_range"
    ] == [0, 24],

    shared_protocol[
        "C"
    ] == 36,

    shared_protocol[
        "W"
    ] == 250,

    shared_protocol[
        "K"
    ] == 5,

    shared_protocol[
        "batch_size"
    ] == 128,

    shared_protocol[
        "epochs"
    ] == 100,

    shared_protocol[
        "learning_rate"
    ] == 5.0e-4,

    shared_protocol[
        "classifier_seeds"
    ] == [
        123,
        124,
        125,
        126,
        127,
    ],

    shared_protocol[
        "shuffle_seed"
    ] == 101112,

    shared_protocol[
        "augmentation_seed"
    ] == 789,

    shared_protocol[
        "drop_last"
    ] is False,
])

assert single_difference_gate


# ============================================================
# 6. RECOVER OFFICIAL-RUN / CHECKPOINT CONTRACT
#    EXACTLY FROM HISTORICAL CELL 34
# ============================================================

cell34 = cell_source(
    34
)

official_start_marker = (
    "# OFFICIAL RUN IDS"
)

official_end_marker = (
    "# REPORT"
)

official_start = cell34.find(
    official_start_marker
)

official_end = cell34.find(
    official_end_marker,
    official_start,
)

assert official_start >= 0
assert official_end > official_start


official_contract_source = cell34[
    official_start:
    official_end
]


exec(
    compile(
        official_contract_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL34_OFFICIAL_RUN_CONTRACT"
        ),
        "exec",
    ),
    globals(),
)


EXPECTED_OFFICIAL_RUN_SHA256 = (
    "010aeaa0aedc657c1b1214604e67992a"
    "ff4fffb9bf5bc0450a63758868ff783a"
)


official_run_hash_gate = (
    IIC_OFFICIAL_RUN_SHA256
    == EXPECTED_OFFICIAL_RUN_SHA256
)

assert official_run_hash_gate


# ============================================================
# 7. CHECKPOINT SEMANTICS
# ============================================================

official_checkpoint_gate = all([

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "epochs"
    ] == 100,

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "batches_per_epoch"
    ] == 68,

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "optimizer_steps_per_classifier"
    ] == 6800,

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "scientific_checkpoint"
    ] == "FINAL_epoch100",

    IIC_OFFICIAL_RUN_CONTRACT[
        "training"
    ][
        "best_checkpoint"
    ] is False,

    IIC_OFFICIAL_RUN_CONTRACT[
        "checkpoint"
    ][
        "latest"
    ] == "after_each_fully_completed_epoch",

    IIC_OFFICIAL_RUN_CONTRACT[
        "checkpoint"
    ][
        "resume"
    ] == "last_completed_epoch_plus_1",

    IIC_OFFICIAL_RUN_CONTRACT[
        "checkpoint"
    ][
        "atomic_publish"
    ] is True,

    IIC_OFFICIAL_RUN_CONTRACT[
        "labels_used"
    ] is False,
])

assert official_checkpoint_gate


# ============================================================
# 8. PRIMARY ARTIFACT REMAINS PRESENT /
#    LEGACY REMAINS ABSENT
# ============================================================

PRIMARY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_symmetric_v1"
)

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)


run_directory_gate = (
    PRIMARY_RUN_DIR.is_dir()
    and not LEGACY_RUN_DIR.exists()
)

assert run_directory_gate


# ============================================================
# 9. SAFETY
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 10. FINAL GATE
# ============================================================

a4c3a_gate = all([

    historical_source_gate,

    historical_legacy_identity_gate,

    historical_symmetric_identity_gate,

    dual_contract_hash_gate,

    single_difference_gate,

    official_run_hash_gate,

    official_checkpoint_gate,

    run_directory_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,

    OFFICIAL_IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c3a_gate


NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS = bool(
    a4c3a_gate
)

READY_FOR_A4C3B_TRAINING_DEFINITION_RECOVERY = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 11. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-3A "
    "DUAL/OFFICIAL CONTRACT RECOVERY ==="
)
print()


print("--- SOURCE ---")

print(
    "CELL25_SHA256 =",
    observed_cell25_sha256,
)

print(
    "CELL27_SHA256 =",
    observed_cell27_sha256,
)

print(
    "CELL34_SHA256 =",
    observed_cell34_sha256,
)

print(
    "HISTORICAL_SOURCE_GATE =",
    historical_source_gate,
)


print()
print("--- HISTORICAL FEATURE IDENTITIES ---")

print(
    "LEGACY_MU_EPOCH0_SHA256 =",
    HISTORICAL_LEGACY_MU_EPOCH0_SHA256,
)

print(
    "LEGACY_MU_EPOCH1_SHA256 =",
    HISTORICAL_LEGACY_MU_EPOCH1_SHA256,
)

print(
    "SYMMETRIC_MU_EPOCH0_SHA256 =",
    SYMMETRIC_MU_EPOCH0_AUDIT_SHA256,
)

print(
    "SYMMETRIC_MU_EPOCH1_SHA256 =",
    SYMMETRIC_MU_EPOCH1_AUDIT_SHA256,
)

print(
    "HISTORICAL_LEGACY_IDENTITY_GATE =",
    historical_legacy_identity_gate,
)

print(
    "HISTORICAL_SYMMETRIC_IDENTITY_GATE =",
    historical_symmetric_identity_gate,
)


print()
print("--- DUAL EXPERIMENT ---")

print(
    "IIC_DUAL_EXPERIMENT_SHA256 =",
    IIC_DUAL_EXPERIMENT_SHA256,
)

print(
    "DUAL_CONTRACT_HASH_GATE =",
    dual_contract_hash_gate,
)

print(
    "PRIMARY_MODE =",
    symmetric_contract[
        "augmentation_mode"
    ],
)

print(
    "LEGACY_MODE =",
    legacy_contract[
        "augmentation_mode"
    ],
)

print(
    "PRIMARY_CROP_RANGE =",
    symmetric_contract[
        "crop_left_range"
    ],
)

print(
    "LEGACY_CROP_RANGE =",
    legacy_contract[
        "crop_left_range"
    ],
)

print(
    "SINGLE_DIFFERENCE_GATE =",
    single_difference_gate,
)


print()
print("--- OFFICIAL RUN CONTRACT ---")

print(
    "IIC_OFFICIAL_RUN_SHA256 =",
    IIC_OFFICIAL_RUN_SHA256,
)

print(
    "OFFICIAL_RUN_HASH_GATE =",
    official_run_hash_gate,
)

print(
    "OFFICIAL_CHECKPOINT_GATE =",
    official_checkpoint_gate,
)

print(
    "PRIMARY_RUN_DIR_EXISTS =",
    PRIMARY_RUN_DIR.is_dir(),
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS =",
    NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS,
)

print(
    "READY_FOR_A4C3B_TRAINING_DEFINITION_RECOVERY =",
    READY_FOR_A4C3B_TRAINING_DEFINITION_RECOVERY,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


=== NOTEBOOK 06 — A4C-3A DUAL/OFFICIAL CONTRACT RECOVERY ===

--- SOURCE ---
CELL25_SHA256 = 12955163342f2b37824798493f5b044a90f314dea64a5e668a96eeb553cb15c5
CELL27_SHA256 = 5e16fdee93ee0d48d281ea77f1895b74f32ed56c9b8e3d924fa10a9f541255b2
CELL34_SHA256 = 25ee1dcfbc530e5d103f19d37b8b164bf342217db402b7bf46738052163a3d85
HISTORICAL_SOURCE_GATE = True

--- HISTORICAL FEATURE IDENTITIES ---
LEGACY_MU_EPOCH0_SHA256 = e519fd3a97f9c9bdfe530106d04015b27d4ea5b6f7810971f992c92a9b127b8f
LEGACY_MU_EPOCH1_SHA256 = e2c0b06fff14a5f013ffe5705689f19825a17d5a71bc974a639442fdfb7b6e51
SYMMETRIC_MU_EPOCH0_SHA256 = 0b7eb867dfee1820f877c76df534b48af6b0c14e3a00fe137db72d8133d17aab
SYMMETRIC_MU_EPOCH1_SHA256 = 574eb15f3336d881bb0cd9733034734d35a806c03a86781cbea4e1eec06caebc
HISTORICAL_LEGACY_IDENTITY_GATE = True
HISTORICAL_SYMMETRIC_IDENTITY_GATE = True

--- DUAL EXPERIMENT ---
IIC_DUAL_EXPERIMENT_SHA256 = 4fbae488ccbe6226c4c589c8bfcd712ed20e0ddf65ec95e4dfb80120d8432fbc
DUAL_CONTRACT_HASH_GATE = True
PRIMARY_M

In [14]:
# ============================================================
# NOTEBOOK 06 — A4C-3A HISTORICAL ALIAS FIX
#
# Ripristina il nome storico usato dalle celle 27/34.
# Nessun I/O, nessun optimizer, nessun training.
# ============================================================

EXPECTED_HISTORICAL_ENCODER_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

# Ricalcolo dallo stato reale attualmente in RAM.
current_encoder_sha256 = canonical_tree_sha256(
    official_iic_encoder.state_dict()
)

assert (
    current_encoder_sha256
    == EXPECTED_HISTORICAL_ENCODER_SHA256
), (
    "Encoder identity mismatch: fermarsi."
)

# Nome storico richiesto dal Notebook 06 originale.
ENCODER_FINAL_STATE_SHA256 = current_encoder_sha256

assert (
    ENCODER_FINAL_STATE_SHA256
    == EXPECTED_HISTORICAL_ENCODER_SHA256
)

# Anche la mu centrale deve essere ancora quella canonica.
assert (
    "FULL_CENTRAL_MU_SHA256" in globals()
)

assert (
    FULL_CENTRAL_MU_SHA256
    ==
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)

print(
    "ENCODER_FINAL_STATE_SHA256 =",
    ENCODER_FINAL_STATE_SHA256,
)

print(
    "FULL_CENTRAL_MU_SHA256 =",
    FULL_CENTRAL_MU_SHA256,
)

print(
    "A4C3A_HISTORICAL_ALIAS_FIX_PASS =",
    True,
)

ENCODER_FINAL_STATE_SHA256 = c9c1fe2c2ae0d27d7f5b8c8bef81b727f0da957744027f150e2af10d3a315dfd
FULL_CENTRAL_MU_SHA256 = fd55be588ef05df8afdf7ee048aa535e12eda37bd13ef8d59e5968fcfc9467f8
A4C3A_HISTORICAL_ALIAS_FIX_PASS = True


In [16]:
# ============================================================
# NOTEBOOK 06 — A4C-3B
# TRAINING / CHECKPOINT / RUNNER DEFINITION RECOVERY
#
# DEFINITIONS ONLY.
#
# Restores exact approved definitions from Notebook 06:
#   - Adam factory
#   - one IIC optimization step
#   - K=5 ensemble helpers
#   - atomic checkpoint helpers
#   - metrics helpers
#   - official runner
#   - robust resume patch A5F-R
#   - safe serialization patch A5F-S
#
# Also recovers frozen checkpoint/resume contract identities.
#
# NO model creation.
# NO optimizer creation.
# NO training.
# NO legacy directory creation.
# NO spectral.
# NO labels.
# NO Drive writes.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import ast
import csv
import hashlib
import json
import os
import platform
import time

import numpy as np
import torch


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS is True
), "A4C-3A non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS"
    in globals()
    and NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS is True
), "A4C-2B non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."


# ============================================================
# 1. APPROVED NOTEBOOK 06
# ============================================================

NOTEBOOK06_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH.is_file()

with NOTEBOOK06_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    nb06_training_source = json.load(
        handle
    )


def a4c3b_cell_source(
    index,
):

    cell = nb06_training_source[
        "cells"
    ][index]

    assert cell[
        "cell_type"
    ] == "code"

    return "".join(
        cell.get(
            "source",
            [],
        )
    )


# ============================================================
# 2. HISTORICAL SOURCE HASHES
# ============================================================

EXPECTED_SOURCE_SHA256 = {

    21:
        "9bad1fe68de9beeb849ed48ef145fb04"
        "f7e1ffb9b2cd9a101258112b177c5f33",

    32:
        "891ff54630a6acec62651b02c3b09cf0"
        "074fbcfb7f08908e154749e577c26432",

    36:
        "416554bee7a0c19f1b55fd85fa614a06"
        "24c12e8b81dba9931c064d51ef9f7b76",

    40:
        "5daa9a2bf0529442d554533c0e37a6c2"
        "bab7fe12637773bef11c782e648f15c3",

    41:
        "9201fbbb6ad71a924c51562c9c2d54d0"
        "ba65ce5180231d350d3d63aaf99d9b00",

    43:
        "62322fc6e75c3c1ee5dc62fa7154aab6"
        "1bb37dda8c4a617f80e48a96b30b14a6",

    44:
        "1d43f654a55867387e7e7f2e5ee921cc3"
        "687f07f5cdbe74ced89f1dd2d31cf3c",
}


observed_source_sha256 = {}

for index, expected_sha in (
    EXPECTED_SOURCE_SHA256.items()
):

    source = a4c3b_cell_source(
        index
    )

    actual_sha = hashlib.sha256(
        source.encode(
            "utf-8"
        )
    ).hexdigest()

    observed_source_sha256[
        index
    ] = actual_sha

    assert actual_sha == expected_sha, (
        f"Historical Notebook 06 cell {index} changed. STOP."
    )


historical_source_gate = all(
    observed_source_sha256[index]
    == expected

    for index, expected
    in EXPECTED_SOURCE_SHA256.items()
)

assert historical_source_gate


# ============================================================
# 3. EXACT AST EXTRACTION HELPERS
# ============================================================

def extract_exact_function(
    cell_index,
    function_name,
):

    source = a4c3b_cell_source(
        cell_index
    )

    tree = ast.parse(
        source
    )

    matches = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and node.name == function_name
        )
    ]

    assert len(matches) == 1

    segment = ast.get_source_segment(
        source,
        matches[0],
    )

    assert segment is not None

    return segment


def extract_exact_assignment(
    cell_index,
    variable_name,
):

    source = a4c3b_cell_source(
        cell_index
    )

    tree = ast.parse(
        source
    )

    matches = []

    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        targets = [
            target.id
            for target in node.targets
            if isinstance(
                target,
                ast.Name,
            )
        ]

        if variable_name in targets:
            matches.append(
                node
            )

    assert len(matches) == 1

    segment = ast.get_source_segment(
        source,
        matches[0],
    )

    assert segment is not None

    return segment


def execute_exact_source(
    source,
    tag,
):

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH)
                + "::"
                + tag
            ),
            "exec",
        ),
        globals(),
    )


# ============================================================
# 4. sha256_stream FROM APPROVED CELL 21
# ============================================================

execute_exact_source(
    extract_exact_function(
        21,
        "sha256_stream",
    ),
    "CELL21_sha256_stream",
)


# ============================================================
# 5. OPTIMIZER + BASIC TRAIN STEP FROM CELL 32
#
# Definitions only.
# ============================================================

for function_name in [

    "make_iic_optimizer",

    "train_one_iic_pair",

]:

    execute_exact_source(
        extract_exact_function(
            32,
            function_name,
        ),
        (
            "CELL32_"
            + function_name
        ),
    )


# ============================================================
# 6. CHECKPOINT / ENSEMBLE HELPERS FROM CELL 36
#
# Definitions only.
# ============================================================

for function_name in [

    "tree_to_cpu",

    "build_k5_ensemble",

    "ensemble_model_hashes",

    "ensemble_optimizer_hashes",

    "atomic_torch_save",

    "move_optimizer_state_to_device",

    "reconcile_metrics_to_checkpoint",

]:

    execute_exact_source(
        extract_exact_function(
            36,
            function_name,
        ),
        (
            "CELL36_"
            + function_name
        ),
    )


# ============================================================
# 7. CHECKPOINT/RESUME CONTRACT — EXACT HISTORICAL SOURCE
# ============================================================

for variable_name in [

    "IIC_CHECKPOINT_RESUME_CONTRACT",

    "IIC_CHECKPOINT_RESUME_JSON",

    "IIC_CHECKPOINT_RESUME_SHA256",

]:

    execute_exact_source(
        extract_exact_assignment(
            36,
            variable_name,
        ),
        (
            "CELL36_"
            + variable_name
        ),
    )


EXPECTED_CHECKPOINT_RESUME_SHA256 = (
    "a2dc5e9135b69486494f6045b40cb55"
    "d96843cf6819cbce20738a098b66f7a39"
)

checkpoint_resume_gate = (
    IIC_CHECKPOINT_RESUME_SHA256
    == EXPECTED_CHECKPOINT_RESUME_SHA256
)

assert checkpoint_resume_gate


# ============================================================
# 8. OFFICIAL METRICS SCHEMA FROM CELL 40
# ============================================================

execute_exact_source(
    extract_exact_assignment(
        40,
        "OFFICIAL_METRICS_FIELDS",
    ),
    "CELL40_OFFICIAL_METRICS_FIELDS",
)

assert len(
    OFFICIAL_METRICS_FIELDS
) == 24


# ============================================================
# 9. OFFICIAL RUNNER DEFINITIONS FROM CELL 40
#
# No function is called here.
# ============================================================

OFFICIAL_FUNCTIONS_CELL40 = [

    "atomic_write_text",

    "append_runtime_session",

    "initialize_metrics_file",

    "append_metrics_row",

    "make_official_iic_checkpoint",

    "restore_official_iic_checkpoint",

    "train_one_iic_pair_with_metrics",

    "run_official_iic_epoch",

    "make_official_metrics_row",

    "run_official_iic",
]


for function_name in (
    OFFICIAL_FUNCTIONS_CELL40
):

    execute_exact_source(
        extract_exact_function(
            40,
            function_name,
        ),
        (
            "CELL40_"
            + function_name
        ),
    )


# ============================================================
# 10. ROBUST RESUME PATCH A5F-R — CELL 41
#
# Exact patched run_official_iic replaces the base definition.
# No call.
# ============================================================

patched_runner_source = (
    extract_exact_function(
        41,
        "run_official_iic",
    )
)

PATCHED_RUNNER_SOURCE_SHA256 = (
    hashlib.sha256(
        patched_runner_source.encode(
            "utf-8"
        )
    ).hexdigest()
)


execute_exact_source(
    patched_runner_source,
    "CELL41_run_official_iic_PATCHED",
)


# ============================================================
# 11. SAFE SERIALIZATION CONTRACT — CELL 44
# ============================================================

for variable_name in [

    "IIC_SAFE_SERIALIZATION_CONTRACT",

    "IIC_SAFE_SERIALIZATION_JSON",

    "IIC_SAFE_SERIALIZATION_SHA256",

]:

    execute_exact_source(
        extract_exact_assignment(
            44,
            variable_name,
        ),
        (
            "CELL44_"
            + variable_name
        ),
    )


EXPECTED_SAFE_SERIALIZATION_SHA256 = (
    "57ccf6d4452ed649333e65d8f4281f3"
    "14a42b2186d7d7ce7e9cded46a0933b78"
)

safe_serialization_contract_gate = (
    IIC_SAFE_SERIALIZATION_SHA256
    == EXPECTED_SAFE_SERIALIZATION_SHA256
)

assert safe_serialization_contract_gate


# ============================================================
# 12. EXACT "KEEP BASE FUNCTION" PATCH REGIONS FROM CELL 44
# ============================================================

cell44_source = a4c3b_cell_source(
    44
)


checkpoint_alias_start = cell44_source.find(
    "# KEEP THE ORIGINAL CHECKPOINT FACTORY ONCE"
)

checkpoint_alias_end = cell44_source.find(
    "# PATCHED CHECKPOINT FACTORY",
    checkpoint_alias_start,
)

assert checkpoint_alias_start >= 0
assert checkpoint_alias_end > checkpoint_alias_start


execute_exact_source(
    cell44_source[
        checkpoint_alias_start:
        checkpoint_alias_end
    ],
    "CELL44_KEEP_CHECKPOINT_BASE",
)


restore_alias_start = cell44_source.find(
    "# KEEP ORIGINAL RESTORE ONCE"
)

restore_alias_end = cell44_source.find(
    "# PATCHED RESTORE",
    restore_alias_start,
)

assert restore_alias_start >= 0
assert restore_alias_end > restore_alias_start


execute_exact_source(
    cell44_source[
        restore_alias_start:
        restore_alias_end
    ],
    "CELL44_KEEP_RESTORE_BASE",
)


# ============================================================
# 13. SAFE SERIALIZATION PATCHED FUNCTIONS — CELL 44
# ============================================================

for function_name in [

    "make_official_iic_checkpoint",

    "restore_official_iic_checkpoint",

]:

    execute_exact_source(
        extract_exact_function(
            44,
            function_name,
        ),
        (
            "CELL44_PATCHED_"
            + function_name
        ),
    )


# ============================================================
# 14. RESUME EDGE-CASE CONTRACT IDENTITY
#
# We DO NOT rerun A5F-T here.
#
# Its historical source cell is hash-gated above, and the
# contract identity was already frozen after the validated
# primary run.
# ============================================================

EXPECTED_RESUME_EDGECASE_SHA256 = (
    "f15655c117fb6a9ca76bd414332b1a34"
    "91eb70c5b59de67da3a0e1fed946eca6"
)

IIC_RESUME_EDGECASE_TEST_SHA256 = (
    EXPECTED_RESUME_EDGECASE_SHA256
)

RESUME_EDGECASE_TEST_REEXECUTED = False

resume_edgecase_identity_gate = (
    observed_source_sha256[43]
    == EXPECTED_SOURCE_SHA256[43]

    and

    IIC_RESUME_EDGECASE_TEST_SHA256
    == EXPECTED_RESUME_EDGECASE_SHA256
)

assert resume_edgecase_identity_gate


# ============================================================
# 15. VERIFY PERSISTED PRIMARY FINAL CHECKPOINT
#     CARRIES CURRENT CHECKPOINT CONTRACTS
#
# Read only.
# ============================================================

PRIMARY_FINAL_PATH = (
    PRIMARY_RUN_DIR
    / "final.pt"
)

assert PRIMARY_FINAL_PATH.is_file()


primary_final_payload = torch.load(
    PRIMARY_FINAL_PATH,
    map_location="cpu",
    weights_only=True,
)


primary_checkpoint_contract_gate = (

    primary_final_payload[
        "run_id"
    ]
    == "sakai224_full_iic_symmetric_v1"

    and

    primary_final_payload[
        "kind"
    ]
    == "FINAL"

    and

    primary_final_payload[
        "contracts"
    ][
        "checkpoint_resume"
    ]
    == IIC_CHECKPOINT_RESUME_SHA256

    and

    primary_final_payload[
        "contracts"
    ][
        "safe_serialization"
    ]
    == IIC_SAFE_SERIALIZATION_SHA256

    and all(

        value is None
        or type(value) is str

        for value
        in primary_final_payload[
            "runtime"
        ].values()
    )
)

assert primary_checkpoint_contract_gate


del primary_final_payload


# ============================================================
# 16. DEFINITION AVAILABILITY GATE
# ============================================================

required_functions = [

    "make_iic_optimizer",

    "train_one_iic_pair",

    "build_k5_ensemble",

    "atomic_torch_save",

    "reconcile_metrics_to_checkpoint",

    "atomic_write_text",

    "append_runtime_session",

    "initialize_metrics_file",

    "append_metrics_row",

    "make_official_iic_checkpoint",

    "restore_official_iic_checkpoint",

    "train_one_iic_pair_with_metrics",

    "run_official_iic_epoch",

    "make_official_metrics_row",

    "run_official_iic",
]


definition_gate = all(
    callable(
        globals().get(
            name
        )
    )
    for name in required_functions
)

assert definition_gate


# ============================================================
# 17. ENCODER MUST STILL BE IMMUTABLE
# ============================================================

encoder_state_after_a4c3b = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

encoder_gate = (
    encoder_state_after_a4c3b
    == ENCODER_FINAL_STATE_SHA256

    and official_iic_encoder.training
        is False

    and all(
        parameter.requires_grad
        is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad
        is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_gate


# ============================================================
# 18. LEGACY COLLISION / SAFETY
# ============================================================

LEGACY_RUN_DIR = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

legacy_collision_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_collision_gate


LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 19. FINAL GATE
# ============================================================

a4c3b_gate = all([

    historical_source_gate,

    checkpoint_resume_gate,

    safe_serialization_contract_gate,

    resume_edgecase_identity_gate,

    primary_checkpoint_contract_gate,

    definition_gate,

    encoder_gate,

    legacy_collision_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED
        is False,

    OFFICIAL_IIC_MODEL_CREATED is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    PRIMARY_SYMMETRIC_RERUN is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED
        is False,
])

assert a4c3b_gate


NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS = bool(
    a4c3b_gate
)

READY_FOR_A4C4_LEGACY_PREFLIGHT = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 20. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-3B "
    "TRAINING DEFINITIONS RECOVERY ==="
)
print()


print("--- SOURCE ---")

print(
    "HISTORICAL_SOURCE_GATE =",
    historical_source_gate,
)

for index in [
    21,
    32,
    36,
    40,
    41,
    43,
    44,
]:

    print(
        f"CELL_{index}_SHA256 =",
        observed_source_sha256[
            index
        ],
    )


print()
print("--- CONTRACTS ---")

print(
    "IIC_CHECKPOINT_RESUME_SHA256 =",
    IIC_CHECKPOINT_RESUME_SHA256,
)

print(
    "CHECKPOINT_RESUME_GATE =",
    checkpoint_resume_gate,
)

print(
    "IIC_SAFE_SERIALIZATION_SHA256 =",
    IIC_SAFE_SERIALIZATION_SHA256,
)

print(
    "SAFE_SERIALIZATION_GATE =",
    safe_serialization_contract_gate,
)

print(
    "IIC_RESUME_EDGECASE_TEST_SHA256 =",
    IIC_RESUME_EDGECASE_TEST_SHA256,
)

print(
    "RESUME_EDGECASE_TEST_REEXECUTED =",
    RESUME_EDGECASE_TEST_REEXECUTED,
)

print(
    "RESUME_EDGECASE_IDENTITY_GATE =",
    resume_edgecase_identity_gate,
)


print()
print("--- PRIMARY PERSISTED CHECKPOINT ---")

print(
    "PRIMARY_CHECKPOINT_CONTRACT_GATE =",
    primary_checkpoint_contract_gate,
)


print()
print("--- DEFINITIONS ---")

for name in required_functions:

    print(
        name,
        "=",
        callable(
            globals().get(
                name
            )
        ),
    )


print()
print("--- SAFETY ---")

print(
    "ENCODER_GATE =",
    encoder_gate,
)

print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS =",
    NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS,
)

print(
    "READY_FOR_A4C4_LEGACY_PREFLIGHT =",
    READY_FOR_A4C4_LEGACY_PREFLIGHT,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


=== NOTEBOOK 06 — A4C-3B TRAINING DEFINITIONS RECOVERY ===

--- SOURCE ---
HISTORICAL_SOURCE_GATE = True
CELL_21_SHA256 = 0eefcc33a15c9380b40cc27567ac48f9988ce588cf528c85568fb4d3c719b7ac
CELL_32_SHA256 = 891ff54630a6acec62651b02c3b09cf0074fbcfb7f08908e154749e577c26432
CELL_36_SHA256 = ddf74fd8cd207f729a8d4fca1d16cf605b90b58ed54116a6ba8c71fd64c77a4c
CELL_40_SHA256 = 5daa9a2bf0529442d554533c0e37a6c2bab7fe12637773bef11c782e648f15c3
CELL_41_SHA256 = 9201fbbb6ad71a924c51562c9c2d54d0ba65ce5180231d350d3d63aaf99d9b00
CELL_43_SHA256 = 96f53b02d973005cb94cb8a4a1ce8ece5ef8a147213948af9a84c25b8247d7cc
CELL_44_SHA256 = 09af19eb2024c9d424aaf0ab3bb3268441479697498da9f00fc5a1959db739ca

--- CONTRACTS ---
IIC_CHECKPOINT_RESUME_SHA256 = a2dc5e9135b69486494f6045b40cb55d96843cf6819cbce20738a098b66f7a39
CHECKPOINT_RESUME_GATE = True
IIC_SAFE_SERIALIZATION_SHA256 = 57ccf6d4452ed649333e65d8f4281f314a42b2186d7d7ce7e9cded46a0933b78
SAFE_SERIALIZATION_GATE = True
IIC_RESUME_EDGECASE_TEST_SHA256 = f15655c117fb6

In [20]:
# ============================================================
# NOTEBOOK 06 — A4C-4
# FINAL LEGACY_TEXT OFFICIAL-RUN PREFLIGHT
#
# READ ONLY.
#
# Reconstructs the exact historical primary run-config template
# from approved Notebook-06 cell 38, then derives legacy_text
# by changing ONLY:
#
#   run_id
#   role
#   augmentation_mode
#   augmentation_mode_id
#
# NO models.
# NO optimizers.
# NO optimizer steps.
# NO run directory creation.
# NO labels.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import ast
import copy
import hashlib
import json
import math

import numpy as np
import torch


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS"
    in globals()
    and NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS is True
), "A4C-3B non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4C3A_CONTRACT_RECOVERY_PASS is True
), "A4C-3A non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS"
    in globals()
    and NOTEBOOK_06_A4C2B_LEGACY_RUNTIME_AUDIT_PASS is True
), "A4C-2B non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS"
    in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."


# ============================================================
# 1. APPROVED HISTORICAL PREFLIGHT SOURCE — CELL 38
# ============================================================

NOTEBOOK06_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH.is_file()

with NOTEBOOK06_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    nb06_preflight_source = json.load(
        handle
    )


cell38_source = "".join(
    nb06_preflight_source[
        "cells"
    ][38].get(
        "source",
        [],
    )
)


EXPECTED_CELL38_SHA256 = (
    "b3f2b23259fc4e433339a403b56fdabd"
    "ad4efafaa54197a065b91bd424aee465"
)


cell38_sha256 = hashlib.sha256(
    cell38_source.encode(
        "utf-8"
    )
).hexdigest()


historical_preflight_source_gate = (
    cell38_sha256
    == EXPECTED_CELL38_SHA256
)

assert historical_preflight_source_gate


# ============================================================
# 2. CONTRACT IDENTITIES
# ============================================================

EXPECTED_CONTRACT_HASHES = {

    "data":
        (
            "03bfab28cd2f513dd3616aca9f394633"
            "3d12d85508e95bb860f6682941002e17"
        ),

    "loss":
        (
            "2bf9cba7d23de553cbad1d876b1f1b14"
            "3ede63154ac20bbb29ca092223c5e7ac"
        ),

    "architecture":
        (
            "12fd00c3dddc6474249a86899b14797c"
            "c891e815126cfb6989eb96e489d5e758"
        ),

    "training_core":
        (
            "f1df0a78746739c7552668e0ed0cac29"
            "5b8d61d772088cb7893a85d6ae6bfe0a"
        ),

    "final_training":
        (
            "4f27a9bbf4290dff54767655d2025928"
            "c2eac751607d217925b0d5bba2b2f537"
        ),

    "dual_experiment":
        (
            "4fbae488ccbe6226c4c589c8bfcd712e"
            "d20e0ddf65ec95e4dfb80120d8432fbc"
        ),

    "official_run":
        (
            "010aeaa0aedc657c1b1214604e67992a"
            "ff4fffb9bf5bc0450a63758868ff783a"
        ),

    "checkpoint_resume":
        (
            "a2dc5e9135b69486494f6045b40cb55"
            "d96843cf6819cbce20738a098b66f7a39"
        ),

    "safe_serialization":
        (
            "57ccf6d4452ed649333e65d8f4281f3"
            "14a42b2186d7d7ce7e9cded46a0933b78"
        ),
}


contract_identity_gate = all([

    IIC_DATA_CONTRACT_SHA256
        == EXPECTED_CONTRACT_HASHES["data"],

    IIC_LOSS_CONTRACT_SHA256
        == EXPECTED_CONTRACT_HASHES["loss"],

    IIC_ARCHITECTURE_CONTRACT_SHA256
        == EXPECTED_CONTRACT_HASHES["architecture"],

    IIC_TRAINING_CORE_SHA256
        == EXPECTED_CONTRACT_HASHES["training_core"],

    IIC_FINAL_TRAINING_SHA256
        == EXPECTED_CONTRACT_HASHES["final_training"],

    IIC_DUAL_EXPERIMENT_SHA256
        == EXPECTED_CONTRACT_HASHES["dual_experiment"],

    IIC_OFFICIAL_RUN_SHA256
        == EXPECTED_CONTRACT_HASHES["official_run"],

    IIC_CHECKPOINT_RESUME_SHA256
        == EXPECTED_CONTRACT_HASHES["checkpoint_resume"],

    IIC_SAFE_SERIALIZATION_SHA256
        == EXPECTED_CONTRACT_HASHES["safe_serialization"],
])

assert contract_identity_gate


# ============================================================
# 3. DATA + ENCODER IDENTITIES
# ============================================================

EXPECTED_FULL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)

EXPECTED_ENCODER_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

EXPECTED_VAE_FINAL_SHA256 = (
    "2bb6fd2ce2f73f8f9e479ed6969a52c1"
    "cdd41c1ae6e2cfc33da687ebba798f71"
)


data_identity_gate = (
    FULL_CENTRAL_MU_SHA256
        == EXPECTED_FULL_MU_SHA256

    and full_central_mu.shape
        == (8583, 512)

    and full_central_mu.dtype
        == np.float32

    and np.isfinite(
        full_central_mu
    ).all()

    and len(
        iic_event_rows
    ) == 8583

    and all_png_paths_exist
)


encoder_identity_gate = (
    ENCODER_FINAL_STATE_SHA256
        == EXPECTED_ENCODER_SHA256

    and canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
        == EXPECTED_ENCODER_SHA256

    and official_iic_encoder.training
        is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )

    and sha256_stream(
        FINAL_CHECKPOINT_PATH
    )
        == EXPECTED_VAE_FINAL_SHA256
)


assert data_identity_gate
assert encoder_identity_gate


# ============================================================
# 4. LEGACY OFFICIAL IDENTITY
# ============================================================

LEGACY_MODE_NAME = (
    "legacy_text"
)

LEGACY_MODE_ID = (
    GS_AUGMENT_MODE_LEGACY_TEXT
)

LEGACY_RUN_ID = (
    "sakai224_full_iic_legacy_text_v1"
)

LEGACY_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / LEGACY_RUN_ID
)


legacy_identity_gate = (

    IIC_EXPERIMENTS[
        LEGACY_MODE_NAME
    ][
        "run_id"
    ]
        == LEGACY_RUN_ID

    and IIC_EXPERIMENTS[
        LEGACY_MODE_NAME
    ][
        "role"
    ]
        == "HTML_TEXT_FIDELITY_SENSITIVITY"

    and IIC_EXPERIMENTS[
        LEGACY_MODE_NAME
    ][
        "crop_left_range"
    ]
        == [0, 24]

    and LEGACY_MODE_ID
        == GS_AUGMENT_MODE_LEGACY_TEXT
)

assert legacy_identity_gate


# ============================================================
# 5. REFUSE ACCIDENTAL OVERWRITE
# ============================================================

legacy_directory_absent_gate = (
    not LEGACY_RUN_DIR.exists()
)

assert legacy_directory_absent_gate, (
    "STOP: legacy official run directory already exists."
)


# ============================================================
# 6. FULL TRAINING GEOMETRY
# ============================================================

EVENT_COUNT = 8583
BATCH_SIZE = 128
EPOCHS = 100
K = 5

BATCHES_PER_EPOCH = math.ceil(
    EVENT_COUNT
    / BATCH_SIZE
)

TAIL_BATCH_SIZE = (
    EVENT_COUNT
    % BATCH_SIZE
)

STEPS_PER_CLASSIFIER = (
    BATCHES_PER_EPOCH
    * EPOCHS
)


training_geometry_gate = all([

    EVENT_COUNT == 8583,

    BATCH_SIZE == 128,

    BATCHES_PER_EPOCH == 68,

    TAIL_BATCH_SIZE == 7,

    EPOCHS == 100,

    K == 5,

    STEPS_PER_CLASSIFIER == 6800,

    IIC_CLASSIFIER_SEEDS
        == [123, 124, 125, 126, 127],

    IIC_SHUFFLE_SEED == 101112,

    IIC_AUGMENTATION_SEED == 789,

    IIC_DROP_LAST is False,
])

assert training_geometry_gate


# ============================================================
# 7. SHUFFLE — FIRST + LAST EPOCH
# ============================================================

legacy_epoch0_permutation = (
    vae_c_shuffle_make_permutation_py(
        EVENT_COUNT,
        IIC_SHUFFLE_SEED,
        0,
    )
)

legacy_epoch99_permutation = (
    vae_c_shuffle_make_permutation_py(
        EVENT_COUNT,
        IIC_SHUFFLE_SEED,
        99,
    )
)


shuffle_gate = (

    len(
        legacy_epoch0_permutation
    ) == EVENT_COUNT

    and len(
        legacy_epoch99_permutation
    ) == EVENT_COUNT

    and len(
        set(
            legacy_epoch0_permutation
        )
    ) == EVENT_COUNT

    and len(
        set(
            legacy_epoch99_permutation
        )
    ) == EVENT_COUNT

    and legacy_epoch0_permutation
        != legacy_epoch99_permutation
)

assert shuffle_gate


# ============================================================
# 8. LEGACY AUGMENTATION — FIRST + LAST EPOCH
# ============================================================

legacy_epoch0_crop_map = np.asarray(
    [
        gs_augment_crop_left_for_event_mode(
            IIC_AUGMENTATION_SEED,
            0,
            manifest_index,
            LEGACY_MODE_ID,
        )

        for manifest_index
        in range(
            EVENT_COUNT
        )
    ],
    dtype=np.int16,
)


legacy_epoch99_crop_map = np.asarray(
    [
        gs_augment_crop_left_for_event_mode(
            IIC_AUGMENTATION_SEED,
            99,
            manifest_index,
            LEGACY_MODE_ID,
        )

        for manifest_index
        in range(
            EVENT_COUNT
        )
    ],
    dtype=np.int16,
)


legacy_augmentation_gate = (

    int(
        legacy_epoch0_crop_map.min()
    ) == 0

    and int(
        legacy_epoch0_crop_map.max()
    ) == 24

    and int(
        legacy_epoch99_crop_map.min()
    ) == 0

    and int(
        legacy_epoch99_crop_map.max()
    ) == 24

    and len(
        np.unique(
            legacy_epoch0_crop_map
        )
    ) == 25

    and len(
        np.unique(
            legacy_epoch99_crop_map
        )
    ) == 25

    and not np.array_equal(
        legacy_epoch0_crop_map,
        legacy_epoch99_crop_map,
    )
)

assert legacy_augmentation_gate


legacy_epoch0_crop_sha256 = (
    int16_sha256(
        legacy_epoch0_crop_map
    )
)

legacy_epoch99_crop_sha256 = (
    int16_sha256(
        legacy_epoch99_crop_map
    )
)


legacy_epoch0_known_answer_gate = (
    legacy_epoch0_crop_sha256
    == LEGACY_CROP_EPOCH0_SHA256
)

assert legacy_epoch0_known_answer_gate


# ============================================================
# 9. EXTRACT EXACT HISTORICAL PRIMARY RUN-CONFIG ASSIGNMENT
# ============================================================

def extract_cell38_assignment(
    variable_name,
):

    tree = ast.parse(
        cell38_source
    )

    matches = []

    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        target_names = [
            target.id
            for target in node.targets
            if isinstance(
                target,
                ast.Name,
            )
        ]

        if variable_name in target_names:
            matches.append(
                node
            )

    assert len(matches) == 1

    segment = ast.get_source_segment(
        cell38_source,
        matches[0],
    )

    assert segment is not None

    return segment


primary_config_assignment_source = (
    extract_cell38_assignment(
        "PRIMARY_RUN_CONFIG"
    )
)


# ============================================================
# 10. REBUILD HISTORICAL PRIMARY TEMPLATE IN ISOLATED NAMESPACE
#
# This does not alter runtime globals.
# ============================================================

template_namespace = {

    "PRIMARY_RUN_ID":
        "sakai224_full_iic_symmetric_v1",

    "PRIMARY_MODE_NAME":
        "symmetric_figure",

    "PRIMARY_MODE_ID":
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,

    "EVENT_COUNT":
        EVENT_COUNT,

    "K":
        K,

    "EPOCHS":
        EPOCHS,

    "BATCH_SIZE":
        BATCH_SIZE,

    "EXPECTED_CLASSIFIER_SEEDS":
        [
            123,
            124,
            125,
            126,
            127,
        ],

    "ENCODER_FINAL_STATE_SHA256":
        ENCODER_FINAL_STATE_SHA256,

    "IIC_LOSS_CONTRACT_SHA256":
        IIC_LOSS_CONTRACT_SHA256,

    "IIC_ARCHITECTURE_CONTRACT_SHA256":
        IIC_ARCHITECTURE_CONTRACT_SHA256,

    "IIC_TRAINING_CORE_SHA256":
        IIC_TRAINING_CORE_SHA256,

    "IIC_FINAL_TRAINING_SHA256":
        IIC_FINAL_TRAINING_SHA256,

    "IIC_DUAL_EXPERIMENT_SHA256":
        IIC_DUAL_EXPERIMENT_SHA256,

    "IIC_OFFICIAL_RUN_SHA256":
        IIC_OFFICIAL_RUN_SHA256,

    "IIC_CHECKPOINT_RESUME_SHA256":
        IIC_CHECKPOINT_RESUME_SHA256,

    "FULL_CENTRAL_MU_SHA256":
        FULL_CENTRAL_MU_SHA256,

    "EXPECTED_FINAL_CHECKPOINT_SHA256":
        EXPECTED_VAE_FINAL_SHA256,
}


exec(
    compile(
        primary_config_assignment_source,
        (
            str(NOTEBOOK06_PATH)
            + "::CELL38_PRIMARY_RUN_CONFIG_TEMPLATE"
        ),
        "exec",
    ),
    template_namespace,
)


historical_primary_template = (
    template_namespace[
        "PRIMARY_RUN_CONFIG"
    ]
)


historical_primary_template_json = json.dumps(
    historical_primary_template,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


historical_primary_template_sha256 = (
    hashlib.sha256(
        historical_primary_template_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


# Persisted primary config must reproduce exactly.
EXPECTED_PRIMARY_RUN_CONFIG_SHA256 = (
    "6a34e279824042a12be112ba04ce0fb4"
    "c30f500c901921a8d2aee5ab7b817aca"
)


historical_primary_template_gate = (
    historical_primary_template_sha256
        == EXPECTED_PRIMARY_RUN_CONFIG_SHA256

    and sha256_stream(
        PRIMARY_RUN_DIR
        / "run_config.json"
    )
        == EXPECTED_PRIMARY_RUN_CONFIG_SHA256
)

assert historical_primary_template_gate


# ============================================================
# 11. DERIVE LEGACY CONFIG
#
# Only the experiment identity / augmentation descriptors
# are changed.
# ============================================================

LEGACY_RUN_CONFIG = copy.deepcopy(
    historical_primary_template
)


LEGACY_RUN_CONFIG[
    "run_id"
] = LEGACY_RUN_ID


LEGACY_RUN_CONFIG[
    "role"
] = (
    "HTML_TEXT_FIDELITY_SENSITIVITY"
)


LEGACY_RUN_CONFIG[
    "augmentation_mode"
] = LEGACY_MODE_NAME


LEGACY_RUN_CONFIG[
    "augmentation_mode_id"
] = int(
    LEGACY_MODE_ID
)


LEGACY_RUN_CONFIG_JSON = json.dumps(
    LEGACY_RUN_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


LEGACY_RUN_CONFIG_SHA256 = hashlib.sha256(
    LEGACY_RUN_CONFIG_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 12. EXACT CONFIG-DIFF AUDIT
# ============================================================

def nested_diff_paths(
    left,
    right,
    prefix=(),
):

    differences = []

    if (
        isinstance(left, dict)
        and isinstance(right, dict)
    ):

        all_keys = (
            set(left.keys())
            | set(right.keys())
        )

        for key in sorted(
            all_keys
        ):

            if (
                key not in left
                or key not in right
            ):
                differences.append(
                    prefix
                    + (key,)
                )

            else:
                differences.extend(
                    nested_diff_paths(
                        left[key],
                        right[key],
                        prefix
                        + (key,),
                    )
                )

        return differences


    if left != right:

        differences.append(
            prefix
        )


    return differences


config_diff_paths = (
    nested_diff_paths(
        historical_primary_template,
        LEGACY_RUN_CONFIG,
    )
)


EXPECTED_CONFIG_DIFF_PATHS = {

    ("run_id",),

    ("role",),

    ("augmentation_mode",),

    ("augmentation_mode_id",),
}


config_single_difference_gate = (
    set(
        config_diff_paths
    )
    == EXPECTED_CONFIG_DIFF_PATHS
)

assert config_single_difference_gate


# Scientific subtrees must be identical.
scientific_config_identity_gate = all([

    LEGACY_RUN_CONFIG[
        "population"
    ]
        == historical_primary_template[
            "population"
        ],

    LEGACY_RUN_CONFIG[
        "encoder"
    ]
        == historical_primary_template[
            "encoder"
        ],

    LEGACY_RUN_CONFIG[
        "iic"
    ]
        == historical_primary_template[
            "iic"
        ],

    LEGACY_RUN_CONFIG[
        "optimizer"
    ]
        == historical_primary_template[
            "optimizer"
        ],

    LEGACY_RUN_CONFIG[
        "rng"
    ]
        == historical_primary_template[
            "rng"
        ],

    LEGACY_RUN_CONFIG[
        "checkpoint"
    ]
        == historical_primary_template[
            "checkpoint"
        ],

    LEGACY_RUN_CONFIG[
        "contracts"
    ]
        == historical_primary_template[
            "contracts"
        ],

    LEGACY_RUN_CONFIG[
        "source_identity"
    ]
        == historical_primary_template[
            "source_identity"
        ],

    LEGACY_RUN_CONFIG[
        "labels_used"
    ] is False,
])

assert scientific_config_identity_gate


# ============================================================
# 13. RUNNER / CHECKPOINT DEFINITIONS
# ============================================================

required_runtime_functions = [

    "build_k5_ensemble",

    "make_iic_optimizer",

    "train_one_iic_pair_with_metrics",

    "run_official_iic_epoch",

    "make_official_iic_checkpoint",

    "restore_official_iic_checkpoint",

    "atomic_torch_save",

    "run_official_iic",
]


runtime_definition_gate = all(
    callable(
        globals().get(
            name
        )
    )

    for name
    in required_runtime_functions
)

assert runtime_definition_gate


# Patched robust runner must be the recovered A5F-R source.
cell41_source = "".join(
    nb06_preflight_source[
        "cells"
    ][41].get(
        "source",
        [],
    )
)

cell41_tree = ast.parse(
    cell41_source
)

cell41_runner_nodes = [
    node
    for node in cell41_tree.body
    if (
        isinstance(
            node,
            ast.FunctionDef,
        )
        and node.name
        == "run_official_iic"
    )
]

assert len(
    cell41_runner_nodes
) == 1


expected_patched_runner_source = (
    ast.get_source_segment(
        cell41_source,
        cell41_runner_nodes[0],
    )
)


expected_patched_runner_sha256 = (
    hashlib.sha256(
        expected_patched_runner_source.encode(
            "utf-8"
        )
    ).hexdigest()
)


patched_runner_identity_gate = (
    PATCHED_RUNNER_SOURCE_SHA256
        == expected_patched_runner_sha256
)

assert patched_runner_identity_gate


# ============================================================
# 14. PRIMARY RUN REMAINS FROZEN
# ============================================================

PRIMARY_FINAL_IIC_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)


primary_frozen_gate = (
    PRIMARY_RUN_DIR.is_dir()

    and (
        PRIMARY_RUN_DIR
        / "final.pt"
    ).is_file()

    and sha256_stream(
        PRIMARY_RUN_DIR
        / "final.pt"
    )
        == PRIMARY_FINAL_IIC_SHA256

    and not LEGACY_RUN_DIR.exists()
)

assert primary_frozen_gate


# ============================================================
# 15. SAFETY
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

OFFICIAL_IIC_MODEL_CREATED = False
OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
SPECTRAL_STARTED = False

LEGACY_RUN_DIRECTORY_CREATED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 16. FINAL PREFLIGHT GATE
# ============================================================

a4c4_gate = all([

    historical_preflight_source_gate,

    contract_identity_gate,

    data_identity_gate,

    encoder_identity_gate,

    legacy_identity_gate,

    legacy_directory_absent_gate,

    training_geometry_gate,

    shuffle_gate,

    legacy_augmentation_gate,

    legacy_epoch0_known_answer_gate,

    historical_primary_template_gate,

    config_single_difference_gate,

    scientific_config_identity_gate,

    runtime_definition_gate,

    patched_runner_identity_gate,

    primary_frozen_gate,

    LABELS_READ is False,

    GRAVITY_SPY_LABEL_VALUES_ACCESSED
        is False,

    OFFICIAL_IIC_MODEL_CREATED
        is False,

    OPTIMIZER_CREATED is False,

    IIC_TRAINING_STARTED is False,

    SPECTRAL_STARTED is False,

    LEGACY_RUN_DIRECTORY_CREATED
        is False,

    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED
        is False,
])

assert a4c4_gate


NOTEBOOK_06_A4C4_LEGACY_PREFLIGHT_PASS = bool(
    a4c4_gate
)

READY_FOR_A4C5_LEGACY_ONE_STEP_SMOKE = True

READY_FOR_LEGACY_TRAINING = False


# ============================================================
# 17. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-4 "
    "LEGACY OFFICIAL-RUN PREFLIGHT ==="
)
print()


print("--- IDENTITY ---")

print(
    "RUN_ID =",
    LEGACY_RUN_ID,
)

print(
    "MODE =",
    LEGACY_MODE_NAME,
)

print(
    "MODE_ID =",
    LEGACY_MODE_ID,
)

print(
    "RUN_DIRECTORY =",
    LEGACY_RUN_DIR,
)

print(
    "RUN_DIRECTORY_EXISTS =",
    LEGACY_RUN_DIR.exists(),
)


print()
print("--- CONTRACTS ---")

print(
    "CONTRACT_IDENTITY_GATE =",
    contract_identity_gate,
)

print(
    "DATA_IDENTITY_GATE =",
    data_identity_gate,
)

print(
    "ENCODER_IDENTITY_GATE =",
    encoder_identity_gate,
)


print()
print("--- TRAINING GEOMETRY ---")

print(
    "EVENT_COUNT =",
    EVENT_COUNT,
)

print(
    "BATCH_SIZE =",
    BATCH_SIZE,
)

print(
    "BATCHES_PER_EPOCH =",
    BATCHES_PER_EPOCH,
)

print(
    "TAIL_BATCH_SIZE =",
    TAIL_BATCH_SIZE,
)

print(
    "EPOCHS =",
    EPOCHS,
)

print(
    "K =",
    K,
)

print(
    "STEPS_PER_CLASSIFIER =",
    STEPS_PER_CLASSIFIER,
)

print(
    "TRAINING_GEOMETRY_GATE =",
    training_geometry_gate,
)


print()
print("--- LEGACY AUGMENTATION ---")

print(
    "EPOCH0_MIN_MAX =",
    (
        int(
            legacy_epoch0_crop_map.min()
        ),
        int(
            legacy_epoch0_crop_map.max()
        ),
    ),
)

print(
    "EPOCH99_MIN_MAX =",
    (
        int(
            legacy_epoch99_crop_map.min()
        ),
        int(
            legacy_epoch99_crop_map.max()
        ),
    ),
)

print(
    "EPOCH0_UNIQUE_CROPS =",
    len(
        np.unique(
            legacy_epoch0_crop_map
        )
    ),
)

print(
    "EPOCH99_UNIQUE_CROPS =",
    len(
        np.unique(
            legacy_epoch99_crop_map
        )
    ),
)

print(
    "LEGACY_EPOCH0_CROP_SHA256 =",
    legacy_epoch0_crop_sha256,
)

print(
    "LEGACY_EPOCH99_CROP_SHA256 =",
    legacy_epoch99_crop_sha256,
)

print(
    "LEGACY_AUGMENTATION_GATE =",
    legacy_augmentation_gate,
)


print()
print("--- RUN CONFIG ---")

print(
    "HISTORICAL_PRIMARY_TEMPLATE_SHA256 =",
    historical_primary_template_sha256,
)

print(
    "PRIMARY_TEMPLATE_GATE =",
    historical_primary_template_gate,
)

print(
    "CONFIG_DIFF_PATHS =",
    [
        ".".join(path)
        for path
        in config_diff_paths
    ],
)

print(
    "CONFIG_SINGLE_DIFFERENCE_GATE =",
    config_single_difference_gate,
)

print(
    "SCIENTIFIC_CONFIG_IDENTITY_GATE =",
    scientific_config_identity_gate,
)

print(
    "LEGACY_RUN_CONFIG_SHA256 =",
    LEGACY_RUN_CONFIG_SHA256,
)


print()
print("--- RUNTIME ---")

print(
    "RUNTIME_DEFINITION_GATE =",
    runtime_definition_gate,
)

print(
    "PATCHED_RUNNER_IDENTITY_GATE =",
    patched_runner_identity_gate,
)

print(
    "PRIMARY_FROZEN_GATE =",
    primary_frozen_gate,
)


print()
print("--- SAFETY ---")

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)

print(
    "OPTIMIZER_CREATED =",
    OPTIMIZER_CREATED,
)

print(
    "IIC_TRAINING_STARTED =",
    IIC_TRAINING_STARTED,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "LEGACY_RUN_DIRECTORY_CREATED =",
    LEGACY_RUN_DIRECTORY_CREATED,
)

print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)


print()
print(
    "NOTEBOOK_06_A4C4_LEGACY_PREFLIGHT_PASS =",
    NOTEBOOK_06_A4C4_LEGACY_PREFLIGHT_PASS,
)

print(
    "READY_FOR_A4C5_LEGACY_ONE_STEP_SMOKE =",
    READY_FOR_A4C5_LEGACY_ONE_STEP_SMOKE,
)

print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


=== NOTEBOOK 06 — A4C-4 LEGACY OFFICIAL-RUN PREFLIGHT ===

--- IDENTITY ---
RUN_ID = sakai224_full_iic_legacy_text_v1
MODE = legacy_text
MODE_ID = 0
RUN_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_legacy_text_v1
RUN_DIRECTORY_EXISTS = False

--- CONTRACTS ---
CONTRACT_IDENTITY_GATE = True
DATA_IDENTITY_GATE = True
ENCODER_IDENTITY_GATE = True

--- TRAINING GEOMETRY ---
EVENT_COUNT = 8583
BATCH_SIZE = 128
BATCHES_PER_EPOCH = 68
TAIL_BATCH_SIZE = 7
EPOCHS = 100
K = 5
STEPS_PER_CLASSIFIER = 6800
TRAINING_GEOMETRY_GATE = True

--- LEGACY AUGMENTATION ---
EPOCH0_MIN_MAX = (0, 24)
EPOCH99_MIN_MAX = (0, 24)
EPOCH0_UNIQUE_CROPS = 25
EPOCH99_UNIQUE_CROPS = 25
LEGACY_EPOCH0_CROP_SHA256 = 30dbe7e291812b3515999e729d57e6a4f5e12717762d3021f6601d8def952b30
LEGACY_EPOCH99_CROP_SHA256 = 45a1b33c43c30cc315e5f0b52e60501c48e64859dde7ba7371eb081af7760a17
LEGACY_AUGMENTATION_GATE = True

--- RUN CONFIG ---
HISTORICAL_PRIMARY_TEMPLATE_SHA256 = 6a34e279824042a12be112

In [19]:
# ============================================================
# NOTEBOOK 06 — HISTORICAL PATH ALIAS RECOVERY
# Exact aliases from approved historical cell 5.
#
# READ ONLY.
# ============================================================

from pathlib import Path

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

OFFICIAL_RUN_DIR = (
    SAKAI224_ROOT
    / "runs"
    / "sakai224_full_v1"
)

FINAL_CHECKPOINT_PATH = (
    OFFICIAL_RUN_DIR
    / "final.pt"
)


EXPECTED_FINAL_CHECKPOINT_SHA256 = (
    "2bb6fd2ce2f73f8f9e479ed6969a52c1"
    "cdd41c1ae6e2cfc33da687ebba798f71"
)

assert FINAL_CHECKPOINT_PATH.is_file()

actual_final_sha256 = sha256_stream(
    FINAL_CHECKPOINT_PATH
)

assert (
    actual_final_sha256
    == EXPECTED_FINAL_CHECKPOINT_SHA256
)

print(
    "OFFICIAL_RUN_DIR =",
    OFFICIAL_RUN_DIR,
)

print(
    "FINAL_CHECKPOINT_PATH =",
    FINAL_CHECKPOINT_PATH,
)

print(
    "FINAL_CHECKPOINT_SHA256 =",
    actual_final_sha256,
)

print(
    "HISTORICAL_PATH_ALIAS_RECOVERY_PASS =",
    True,
)

OFFICIAL_RUN_DIR = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1
FINAL_CHECKPOINT_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1/final.pt
FINAL_CHECKPOINT_SHA256 = 2bb6fd2ce2f73f8f9e479ed6969a52c1cdd41c1ae6e2cfc33da687ebba798f71
HISTORICAL_PATH_ALIAS_RECOVERY_PASS = True


In [23]:
# ============================================================
# NOTEBOOK 06 — A4C-5
# LEGACY_TEXT TEMPORARY ONE-STEP SMOKE
#
# FINAL PRE-TRAINING GATE.
#
# Source authority:
#   historical Notebook-06 cell 29 (STEP 06-A5A)
#
# This recovery smoke executes ONLY legacy_text.
# symmetric_figure is NOT rerun.
#
# It performs:
#   - exact PROJECT epoch-0 shuffle
#   - 128 real events
#   - legacy_text perturbation
#   - frozen encoder forward
#   - one temporary IIC classifier pair
#   - one temporary Adam step
#   - comparison with historical saved cell-29 legacy output
#
# It does NOT:
#   - create the official legacy run directory
#   - write checkpoints / metrics / config
#   - modify Drive
#   - read Gravity Spy labels
#   - start spectral clustering
# ============================================================

from pathlib import Path
import ast
import gc
import hashlib
import json
import re

import numpy as np
import torch


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C4_LEGACY_PREFLIGHT_PASS" in globals()
    and NOTEBOOK_06_A4C4_LEGACY_PREFLIGHT_PASS is True
), "A4C-4 non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS" in globals()
    and NOTEBOOK_06_A4C3B_TRAINING_DEFINITIONS_PASS is True
), "A4C-3B non disponibile o non PASS."

assert (
    "NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS" in globals()
    and NOTEBOOK_06_A4B_ENCODER_RECOVERY_PASS is True
), "A4B non disponibile o non PASS."

assert runtime_device.type == "cuda"


# ============================================================
# 1. FROZEN LEGACY CONFIG IDENTITY
# ============================================================

EXPECTED_LEGACY_RUN_CONFIG_SHA256 = (
    "0be4c46a68b7571cb399883e75783a7c"
    "9ce3653d7d4506377ae796423459118f"
)

assert "LEGACY_RUN_CONFIG_SHA256" in globals()

legacy_config_gate = (
    LEGACY_RUN_CONFIG_SHA256
    == EXPECTED_LEGACY_RUN_CONFIG_SHA256
)

assert legacy_config_gate


# ============================================================
# 2. OFFICIAL RUN PATH SAFETY
# ============================================================

PRIMARY_RUN_DIR_A4C5 = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_symmetric_v1"
)

LEGACY_RUN_DIR_A4C5 = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_legacy_text_v1"
)

PRIMARY_FINAL_IIC_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)

primary_pre_smoke_gate = (
    PRIMARY_RUN_DIR_A4C5.is_dir()
    and (
        PRIMARY_RUN_DIR_A4C5
        / "final.pt"
    ).is_file()
    and sha256_stream(
        PRIMARY_RUN_DIR_A4C5
        / "final.pt"
    ) == PRIMARY_FINAL_IIC_SHA256
)

legacy_directory_absent_before = (
    not LEGACY_RUN_DIR_A4C5.exists()
)

assert primary_pre_smoke_gate
assert legacy_directory_absent_before


# ============================================================
# 3. APPROVED HISTORICAL NOTEBOOK-06 CELL 29
# ============================================================

NOTEBOOK06_PATH_A4C5 = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH_A4C5.is_file()

with NOTEBOOK06_PATH_A4C5.open(
    "r",
    encoding="utf-8",
) as handle:
    nb06_a4c5 = json.load(handle)

cell29 = nb06_a4c5["cells"][29]

assert cell29["cell_type"] == "code"

cell29_source = "".join(
    cell29.get(
        "source",
        [],
    )
)

EXPECTED_CELL29_SHA256 = (
    "18aa232fd630423760a13cfb15e37637"
    "d6d6e69cd42438e749d9cfa869adf6f0"
)

cell29_sha256 = hashlib.sha256(
    cell29_source.encode("utf-8")
).hexdigest()

historical_source_gate = (
    cell29_sha256
    == EXPECTED_CELL29_SHA256
)

assert historical_source_gate


# ============================================================
# 4. EXACT FUNCTION EXTRACTION FROM HISTORICAL CELL 29
#
# We extract definitions only.
# We do NOT execute the historical cell body, because that
# would also rerun symmetric_figure.
# ============================================================

cell29_tree = ast.parse(cell29_source)


def extract_cell29_function(function_name):
    matches = [
        node
        for node in cell29_tree.body
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == function_name
        )
    ]

    assert len(matches) == 1, (
        f"Expected exactly one {function_name} in cell 29."
    )

    source = ast.get_source_segment(
        cell29_source,
        matches[0],
    )

    assert source is not None

    return source


def execute_cell29_function(function_name):
    source = extract_cell29_function(
        function_name
    )

    exec(
        compile(
            source,
            (
                str(NOTEBOOK06_PATH_A4C5)
                + "::CELL29_"
                + function_name
            ),
            "exec",
        ),
        globals(),
    )


for function_name in [
    "build_perturbed_mu_smoke",
    "parameter_sha256",
    "run_one_step_smoke",
]:
    execute_cell29_function(
        function_name
    )


historical_function_gate = all(
    callable(
        globals().get(name)
    )
    for name in [
        "build_perturbed_mu_smoke",
        "parameter_sha256",
        "run_one_step_smoke",
    ]
)

assert historical_function_gate


# ============================================================
# 5. REQUIRED RECOVERED RUNTIME DEFINITIONS
# ============================================================

required_runtime_symbols = [
    "vae_c_shuffle_make_permutation_py",
    "make_real_perturbed_batch",
    "gs_augment_crop_left_for_event_mode",
    "build_iic_pair_from_seed",
    "module_state_sha256",
    "project_iic_negative_mutual_information",
    "combine_iic_losses",
]

runtime_symbol_gate = all(
    callable(
        globals().get(name)
    )
    for name in required_runtime_symbols
)

assert runtime_symbol_gate


# ============================================================
# 6. EXACT PROJECT SHUFFLE — EPOCH 0
# ============================================================

full_permutation_epoch0_a4c5 = (
    vae_c_shuffle_make_permutation_py(
        8583,
        101112,
        0,
    )
)

EXPECTED_FIRST_16 = [
    6926,
    5642,
    4410,
    8192,
    953,
    7325,
    8221,
    8226,
    5494,
    962,
    1495,
    8456,
    3759,
    7567,
    4863,
    7695,
]

full_shuffle_gate_a4c5 = (
    len(
        full_permutation_epoch0_a4c5
    ) == 8583

    and len(
        set(
            full_permutation_epoch0_a4c5
        )
    ) == 8583

    and full_permutation_epoch0_a4c5[:16]
        == EXPECTED_FIRST_16
)

assert full_shuffle_gate_a4c5


SMOKE_MANIFEST_INDICES = (
    full_permutation_epoch0_a4c5[:128]
)

assert len(
    SMOKE_MANIFEST_INDICES
) == 128


# ============================================================
# 7. CENTRAL FEATURES — SAME HISTORICAL BATCH
# ============================================================

central_mu_smoke_cpu = torch.from_numpy(
    np.asarray(
        full_central_mu[
            SMOKE_MANIFEST_INDICES
        ],
        dtype=np.float32,
    ).copy()
)

central_feature_gate = (
    central_mu_smoke_cpu.shape
        == (128, 512)

    and central_mu_smoke_cpu.dtype
        == torch.float32

    and torch.isfinite(
        central_mu_smoke_cpu
    ).all().item()
)

assert central_feature_gate


# ============================================================
# 8. ENCODER PRE-SMOKE IDENTITY
# ============================================================

EXPECTED_ENCODER_SHA256 = (
    "c9c1fe2c2ae0d27d7f5b8c8bef81b72"
    "7f0da957744027f150e2af10d3a315dfd"
)

encoder_state_before_smoke = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

encoder_pre_gate = (
    encoder_state_before_smoke
        == EXPECTED_ENCODER_SHA256

    and ENCODER_FINAL_STATE_SHA256
        == EXPECTED_ENCODER_SHA256

    and official_iic_encoder.training
        is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_pre_gate


# ============================================================
# 9. BUILD ONLY LEGACY_TEXT PERTURBED FEATURES
# ============================================================

print(
    "Encoding legacy_text smoke batch..."
)

(
    legacy_mu_smoke_cpu,
    legacy_smoke_crops,
) = build_perturbed_mu_smoke(
    GS_AUGMENT_MODE_LEGACY_TEXT
)


expected_legacy_smoke_crops = [
    int(
        gs_augment_crop_left_for_event_mode(
            IIC_AUGMENTATION_SEED,
            0,
            manifest_index,
            GS_AUGMENT_MODE_LEGACY_TEXT,
        )
    )
    for manifest_index
    in SMOKE_MANIFEST_INDICES
]


legacy_crop_gate = (
    legacy_smoke_crops
        == expected_legacy_smoke_crops

    and min(
        legacy_smoke_crops
    ) >= 0

    and max(
        legacy_smoke_crops
    ) <= 24
)

assert legacy_crop_gate


legacy_feature_gate = (
    legacy_mu_smoke_cpu.shape
        == (128, 512)

    and legacy_mu_smoke_cpu.dtype
        == torch.float32

    and torch.isfinite(
        legacy_mu_smoke_cpu
    ).all().item()
)

assert legacy_feature_gate


legacy_view_difference_gate = (
    not torch.equal(
        central_mu_smoke_cpu,
        legacy_mu_smoke_cpu,
    )
)

assert legacy_view_difference_gate


# ============================================================
# 10. HISTORICAL LEGACY ONE-STEP NUMERIC REFERENCE
#
# Read the already-saved output of historical cell 29.
# symmetric_figure is NOT executed.
# ============================================================

cell29_output_text = ""

for output in cell29.get(
    "outputs",
    [],
):
    if "text" in output:
        cell29_output_text += "".join(
            output["text"]
        )

    elif (
        "data" in output
        and "text/plain" in output["data"]
    ):
        cell29_output_text += "".join(
            output["data"]["text/plain"]
        )


legacy_block_match = re.search(
    (
        r"---\s*legacy_text\s*---"
        r"(.*?)"
        r"---\s*symmetric_figure\s*---"
    ),
    cell29_output_text,
    flags=re.DOTALL,
)

assert legacy_block_match is not None, (
    "Historical legacy_text output block not found."
)

historical_legacy_block = (
    legacy_block_match.group(1)
)


def extract_historical_float(label):
    match = re.search(
        (
            rf"{re.escape(label)}"
            r"\s*=\s*"
            r"([-+0-9.eE]+)"
        ),
        historical_legacy_block,
    )

    assert match is not None, (
        f"Historical value missing: {label}"
    )

    return float(
        match.group(1)
    )


HISTORICAL_LEGACY_ONE_STEP = {
    "main_mi":
        extract_historical_float(
            "MAIN_MI"
        ),

    "overcluster_mi":
        extract_historical_float(
            "OVERCLUSTER_MI"
        ),

    "main_loss":
        extract_historical_float(
            "MAIN_LOSS"
        ),

    "overcluster_loss":
        extract_historical_float(
            "OVERCLUSTER_LOSS"
        ),

    "total_loss":
        extract_historical_float(
            "TOTAL_LOSS"
        ),

    "main_grad_norm":
        extract_historical_float(
            "MAIN_GRAD_NORM"
        ),

    "over_grad_norm":
        extract_historical_float(
            "OVERCLUSTER_GRAD_NORM"
        ),
}


historical_reference_gate = all(
    np.isfinite(value)
    for value
    in HISTORICAL_LEGACY_ONE_STEP.values()
)

assert historical_reference_gate


# ============================================================
# 11. ONE TEMPORARY REAL ADAM STEP — LEGACY ONLY
# ============================================================

print(
    "Running legacy_text temporary one-step smoke..."
)

legacy_smoke = run_one_step_smoke(
    mode_name="legacy_text",
    perturbed_mu_cpu=legacy_mu_smoke_cpu,
)


# ============================================================
# 12. HISTORICAL INTERNAL SMOKE GATES
# ============================================================

legacy_internal_gate = all(
    bool(
        legacy_smoke[key]
    )
    for key in [
        "loss_gate",
        "gradient_gate",
        "encoder_gradient_gate",
        "parameter_update_gate",
        "optimizer_state_gate",
        "bn_tracking_gate",
    ]
)

assert legacy_internal_gate


# ============================================================
# 13. HISTORICAL NUMERIC REPRODUCTION
# ============================================================

CURRENT_TO_HISTORICAL_KEYS = {
    "main_mi":
        "main_mi",

    "overcluster_mi":
        "overcluster_mi",

    "main_loss":
        "main_loss",

    "overcluster_loss":
        "overcluster_loss",

    "total_loss":
        "total_loss",

    "main_grad_norm":
        "main_grad_norm",

    "over_grad_norm":
        "over_grad_norm",
}


historical_numeric_deltas = {
    current_key:
        abs(
            float(
                legacy_smoke[
                    current_key
                ]
            )
            -
            HISTORICAL_LEGACY_ONE_STEP[
                historical_key
            ]
        )

    for current_key, historical_key
    in CURRENT_TO_HISTORICAL_KEYS.items()
}


historical_numeric_gate = all([
    np.isclose(
        float(
            legacy_smoke[
                current_key
            ]
        ),
        HISTORICAL_LEGACY_ONE_STEP[
            historical_key
        ],
        rtol=1.0e-6,
        atol=1.0e-9,
    )
    for current_key, historical_key
    in CURRENT_TO_HISTORICAL_KEYS.items()
])

assert historical_numeric_gate


# ============================================================
# 14. ENCODER IMMUTABILITY AFTER TEMPORARY STEP
# ============================================================

encoder_state_after_smoke = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
)

encoder_immutability_gate = (
    encoder_state_before_smoke
        == encoder_state_after_smoke
        == EXPECTED_ENCODER_SHA256

    and official_iic_encoder.training
        is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_immutability_gate


# ============================================================
# 15. OFFICIAL ARTIFACT SAFETY AFTER TEMPORARY STEP
# ============================================================

primary_post_smoke_gate = (
    PRIMARY_RUN_DIR_A4C5.is_dir()
    and sha256_stream(
        PRIMARY_RUN_DIR_A4C5
        / "final.pt"
    ) == PRIMARY_FINAL_IIC_SHA256
)

legacy_directory_absent_after = (
    not LEGACY_RUN_DIR_A4C5.exists()
)

assert primary_post_smoke_gate
assert legacy_directory_absent_after


# ============================================================
# 16. SAFETY STATE
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

TEMPORARY_IIC_MODEL_CREATED = True
TEMPORARY_OPTIMIZER_CREATED = True
TEMPORARY_IIC_TRAIN_STEP_EXECUTED = True

OFFICIAL_IIC_MODEL_CREATED = False
OFFICIAL_IIC_OPTIMIZER_CREATED = False
OFFICIAL_IIC_TRAINING_STARTED = False

OFFICIAL_RUN_DIRECTORY_CREATED = False
OFFICIAL_CHECKPOINT_WRITTEN = False

SPECTRAL_STARTED = False
PRIMARY_SYMMETRIC_RERUN = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False


# ============================================================
# 17. FINAL GATE
# ============================================================

a4c5_gate = all([
    legacy_config_gate,
    primary_pre_smoke_gate,
    legacy_directory_absent_before,
    historical_source_gate,
    historical_function_gate,
    runtime_symbol_gate,
    full_shuffle_gate_a4c5,
    central_feature_gate,
    encoder_pre_gate,
    legacy_crop_gate,
    legacy_feature_gate,
    legacy_view_difference_gate,
    historical_reference_gate,
    legacy_internal_gate,
    historical_numeric_gate,
    encoder_immutability_gate,
    primary_post_smoke_gate,
    legacy_directory_absent_after,
    LABELS_READ is False,
    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,
    OFFICIAL_IIC_MODEL_CREATED is False,
    OFFICIAL_IIC_OPTIMIZER_CREATED is False,
    OFFICIAL_IIC_TRAINING_STARTED is False,
    OFFICIAL_RUN_DIRECTORY_CREATED is False,
    OFFICIAL_CHECKPOINT_WRITTEN is False,
    SPECTRAL_STARTED is False,
    PRIMARY_SYMMETRIC_RERUN is False,
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert a4c5_gate


NOTEBOOK_06_A4C5_LEGACY_ONE_STEP_SMOKE_PASS = bool(
    a4c5_gate
)

READY_FOR_LEGACY_TRAINING = True


# ============================================================
# 18. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A4C-5 "
    "LEGACY ONE-STEP SMOKE ==="
)
print()

print("--- SOURCE / CONFIG ---")
print(
    "CELL29_SHA256 =",
    cell29_sha256,
)
print(
    "HISTORICAL_SOURCE_GATE =",
    historical_source_gate,
)
print(
    "LEGACY_RUN_CONFIG_SHA256 =",
    LEGACY_RUN_CONFIG_SHA256,
)
print(
    "LEGACY_CONFIG_GATE =",
    legacy_config_gate,
)

print()
print("--- BATCH ---")
print(
    "FIRST_16_MANIFEST_INDICES =",
    SMOKE_MANIFEST_INDICES[:16],
)
print(
    "FULL_SHUFFLE_GATE =",
    full_shuffle_gate_a4c5,
)
print(
    "CENTRAL_FEATURE_GATE =",
    central_feature_gate,
)
print(
    "LEGACY_CROP_MIN_MAX =",
    (
        min(
            legacy_smoke_crops
        ),
        max(
            legacy_smoke_crops
        ),
    ),
)
print(
    "LEGACY_CROP_GATE =",
    legacy_crop_gate,
)
print(
    "LEGACY_FEATURE_GATE =",
    legacy_feature_gate,
)
print(
    "LEGACY_VIEW_DIFFERENCE_GATE =",
    legacy_view_difference_gate,
)

print()
print("--- ONE TEMPORARY ADAM STEP ---")
print(
    "MAIN_MI =",
    legacy_smoke[
        "main_mi"
    ],
)
print(
    "OVERCLUSTER_MI =",
    legacy_smoke[
        "overcluster_mi"
    ],
)
print(
    "MAIN_LOSS =",
    legacy_smoke[
        "main_loss"
    ],
)
print(
    "OVERCLUSTER_LOSS =",
    legacy_smoke[
        "overcluster_loss"
    ],
)
print(
    "TOTAL_LOSS =",
    legacy_smoke[
        "total_loss"
    ],
)
print(
    "MAIN_GRAD_NORM =",
    legacy_smoke[
        "main_grad_norm"
    ],
)
print(
    "OVERCLUSTER_GRAD_NORM =",
    legacy_smoke[
        "over_grad_norm"
    ],
)
print(
    "PARAMETERS_CHANGED =",
    legacy_smoke[
        "parameter_update_gate"
    ],
)
print(
    "OPTIMIZER_STATE_GATE =",
    legacy_smoke[
        "optimizer_state_gate"
    ],
)
print(
    "BN_TRACKING_GATE =",
    legacy_smoke[
        "bn_tracking_gate"
    ],
)
print(
    "LEGACY_INTERNAL_GATE =",
    legacy_internal_gate,
)

print()
print("--- HISTORICAL REPRODUCTION ---")

for key in CURRENT_TO_HISTORICAL_KEYS:
    print(
        key.upper()
        + "_HISTORICAL =",
        HISTORICAL_LEGACY_ONE_STEP[
            key
        ],
    )

    print(
        key.upper()
        + "_DELTA =",
        historical_numeric_deltas[
            key
        ],
    )

print(
    "HISTORICAL_NUMERIC_GATE =",
    historical_numeric_gate,
)

print()
print("--- ENCODER / OFFICIAL ARTIFACTS ---")
print(
    "ENCODER_STATE_BEFORE =",
    encoder_state_before_smoke,
)
print(
    "ENCODER_STATE_AFTER =",
    encoder_state_after_smoke,
)
print(
    "ENCODER_IMMUTABILITY_GATE =",
    encoder_immutability_gate,
)
print(
    "PRIMARY_PRE_SMOKE_GATE =",
    primary_pre_smoke_gate,
)
print(
    "PRIMARY_POST_SMOKE_GATE =",
    primary_post_smoke_gate,
)
print(
    "LEGACY_RUN_DIR_EXISTS =",
    LEGACY_RUN_DIR_A4C5.exists(),
)

print()
print("--- SAFETY ---")
print(
    "LABELS_READ =",
    LABELS_READ,
)
print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)
print(
    "TEMPORARY_IIC_MODEL_CREATED =",
    TEMPORARY_IIC_MODEL_CREATED,
)
print(
    "TEMPORARY_OPTIMIZER_CREATED =",
    TEMPORARY_OPTIMIZER_CREATED,
)
print(
    "TEMPORARY_IIC_TRAIN_STEP_EXECUTED =",
    TEMPORARY_IIC_TRAIN_STEP_EXECUTED,
)
print(
    "OFFICIAL_IIC_MODEL_CREATED =",
    OFFICIAL_IIC_MODEL_CREATED,
)
print(
    "OFFICIAL_IIC_TRAINING_STARTED =",
    OFFICIAL_IIC_TRAINING_STARTED,
)
print(
    "OFFICIAL_RUN_DIRECTORY_CREATED =",
    OFFICIAL_RUN_DIRECTORY_CREATED,
)
print(
    "OFFICIAL_CHECKPOINT_WRITTEN =",
    OFFICIAL_CHECKPOINT_WRITTEN,
)
print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)
print(
    "PRIMARY_SYMMETRIC_RERUN =",
    PRIMARY_SYMMETRIC_RERUN,
)
print(
    "OFFICIAL_DRIVE_ARTIFACTS_MODIFIED =",
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED,
)

print()
print(
    "NOTEBOOK_06_A4C5_LEGACY_ONE_STEP_SMOKE_PASS =",
    NOTEBOOK_06_A4C5_LEGACY_ONE_STEP_SMOKE_PASS,
)
print(
    "READY_FOR_LEGACY_TRAINING =",
    READY_FOR_LEGACY_TRAINING,
)


# ============================================================
# 19. CLEAN TEMPORARY SMOKE ARTIFACTS
# ============================================================

for name in [
    "central_mu_smoke_cpu",
    "legacy_mu_smoke_cpu",
    "legacy_smoke_crops",
    "expected_legacy_smoke_crops",
    "legacy_smoke",
]:
    globals().pop(
        name,
        None,
    )


gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()

Encoding legacy_text smoke batch...
Running legacy_text temporary one-step smoke...

=== NOTEBOOK 06 — A4C-5 LEGACY ONE-STEP SMOKE ===

--- SOURCE / CONFIG ---
CELL29_SHA256 = 18aa232fd630423760a13cfb15e37637d6d6e69cd42438e749d9cfa869adf6f0
HISTORICAL_SOURCE_GATE = True
LEGACY_RUN_CONFIG_SHA256 = 0be4c46a68b7571cb399883e75783a7c9ce3653d7d4506377ae796423459118f
LEGACY_CONFIG_GATE = True

--- BATCH ---
FIRST_16_MANIFEST_INDICES = [6926, 5642, 4410, 8192, 953, 7325, 8221, 8226, 5494, 962, 1495, 8456, 3759, 7567, 4863, 7695]
FULL_SHUFFLE_GATE = True
CENTRAL_FEATURE_GATE = True
LEGACY_CROP_MIN_MAX = (0, 24)
LEGACY_CROP_GATE = True
LEGACY_FEATURE_GATE = True
LEGACY_VIEW_DIFFERENCE_GATE = True

--- ONE TEMPORARY ADAM STEP ---
MAIN_MI = 0.0006363919237628579
OVERCLUSTER_MI = 0.00037637626519426703
MAIN_LOSS = -0.0006363919237628579
OVERCLUSTER_LOSS = -0.00037637626519426703
TOTAL_LOSS = -0.001012768130749464
MAIN_GRAD_NORM = 0.022840456708540516
OVERCLUSTER_GRAD_NORM = 0.0035355902585864555
PA

In [24]:
# ============================================================
# NOTEBOOK 06 — A6D
# OFFICIAL LEGACY_TEXT IIC TRAINING
#
# STARTS / RESUMES THE OFFICIAL RUN.
# WRITES TO GOOGLE DRIVE.
#
# Run:
#   sakai224_full_iic_legacy_text_v1
#
# No labels.
# No spectral clustering.
# ============================================================

from pathlib import Path
import torch


# ============================================================
# 0. FINAL AUTHORIZATION GATES
# ============================================================

assert (
    "NOTEBOOK_06_A4C5_LEGACY_ONE_STEP_SMOKE_PASS" in globals()
    and NOTEBOOK_06_A4C5_LEGACY_ONE_STEP_SMOKE_PASS is True
)

assert (
    "READY_FOR_LEGACY_TRAINING" in globals()
    and READY_FOR_LEGACY_TRAINING is True
)

assert LEGACY_RUN_ID == "sakai224_full_iic_legacy_text_v1"
assert LEGACY_MODE_NAME == "legacy_text"
assert LEGACY_MODE_ID == GS_AUGMENT_MODE_LEGACY_TEXT

EXPECTED_LEGACY_RUN_CONFIG_SHA256 = (
    "0be4c46a68b7571cb399883e75783a7c"
    "9ce3653d7d4506377ae796423459118f"
)

assert (
    LEGACY_RUN_CONFIG_SHA256
    == EXPECTED_LEGACY_RUN_CONFIG_SHA256
)

assert (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256
)

assert official_iic_encoder.training is False

assert all(
    parameter.requires_grad is False
    for parameter in official_iic_encoder.parameters()
)

assert all(
    parameter.grad is None
    for parameter in official_iic_encoder.parameters()
)


# ============================================================
# 1. PRIMARY MUST REMAIN FROZEN
# ============================================================

PRIMARY_OFFICIAL_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_symmetric_v1"
)

PRIMARY_FINAL_PATH = (
    PRIMARY_OFFICIAL_RUN_DIR
    / "final.pt"
)

EXPECTED_PRIMARY_FINAL_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)

assert PRIMARY_FINAL_PATH.is_file()

assert (
    sha256_stream(
        PRIMARY_FINAL_PATH
    )
    == EXPECTED_PRIMARY_FINAL_SHA256
)


# ============================================================
# 2. LEGACY NEW / RESUME SAFETY
# ============================================================

LEGACY_OFFICIAL_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / LEGACY_RUN_ID
)

LEGACY_LATEST_PATH = (
    LEGACY_OFFICIAL_RUN_DIR
    / "latest.pt"
)

LEGACY_FINAL_PATH = (
    LEGACY_OFFICIAL_RUN_DIR
    / "final.pt"
)

LEGACY_CONFIG_PATH = (
    LEGACY_OFFICIAL_RUN_DIR
    / "run_config.json"
)


# Never overwrite an already completed scientific run.
assert not LEGACY_FINAL_PATH.exists(), (
    "STOP: legacy FINAL already exists."
)


if LEGACY_OFFICIAL_RUN_DIR.exists():

    assert LEGACY_CONFIG_PATH.is_file(), (
        "STOP: existing legacy directory has no run_config.json."
    )

    assert (
        sha256_stream(
            LEGACY_CONFIG_PATH
        )
        == LEGACY_RUN_CONFIG_SHA256
    ), (
        "STOP: legacy run_config hash mismatch."
    )

    if LEGACY_LATEST_PATH.is_file():
        launch_mode = "RESUME_FROM_LATEST"
    else:
        launch_mode = "RESTART_BEFORE_FIRST_CHECKPOINT"

else:
    launch_mode = "NEW"


# ============================================================
# 3. SAFETY STATE BEFORE OFFICIAL CALL
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False
SPECTRAL_STARTED = False
PRIMARY_SYMMETRIC_RERUN = False

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False
assert PRIMARY_SYMMETRIC_RERUN is False


# ============================================================
# 4. START / RESUME OFFICIAL LEGACY RUN
# ============================================================

print()
print(
    "=== STARTING / RESUMING OFFICIAL LEGACY_TEXT IIC RUN ==="
)
print("RUN_ID =", LEGACY_RUN_ID)
print("MODE =", LEGACY_MODE_NAME)
print("MODE_ID =", LEGACY_MODE_ID)
print("RUN_DIRECTORY =", LEGACY_OFFICIAL_RUN_DIR)
print("LAUNCH_MODE =", launch_mode)
print("RUN_CONFIG_SHA256 =", LEGACY_RUN_CONFIG_SHA256)
print("TARGET_EPOCHS = 100")
print("BATCHES_PER_EPOCH = 68")
print("STEPS_PER_CLASSIFIER = 6800")
print("K = 5")
print("LABELS_READ = False")
print("SPECTRAL_STARTED = False")
print("THIS CELL WRITES TO DRIVE = True")
print()


OFFICIAL_LEGACY_TRAINING_STARTED = True


official_legacy_final_path = run_official_iic(
    run_id=LEGACY_RUN_ID,
    mode_name=LEGACY_MODE_NAME,
    mode_id=LEGACY_MODE_ID,
    run_config=LEGACY_RUN_CONFIG,
    run_config_json=LEGACY_RUN_CONFIG_JSON,
    run_config_sha256=LEGACY_RUN_CONFIG_SHA256,
)


# ============================================================
# 5. QUICK FINAL GATE
#
# Reached only after the official runner returns.
# ============================================================

assert (
    official_legacy_final_path
    == LEGACY_FINAL_PATH
)

assert official_legacy_final_path.is_file()


legacy_final_payload_quick = torch.load(
    official_legacy_final_path,
    map_location="cpu",
    weights_only=True,
)


legacy_quick_final_gate = (
    legacy_final_payload_quick["kind"] == "FINAL"
    and legacy_final_payload_quick["run_id"] == LEGACY_RUN_ID
    and legacy_final_payload_quick["augmentation_mode"] == LEGACY_MODE_NAME
    and legacy_final_payload_quick["last_completed_epoch"] == 99
    and legacy_final_payload_quick["next_epoch"] == 100
    and legacy_final_payload_quick[
        "optimizer_steps_per_classifier"
    ] == [
        6800,
        6800,
        6800,
        6800,
        6800,
    ]
    and legacy_final_payload_quick[
        "run_config_sha256"
    ] == LEGACY_RUN_CONFIG_SHA256
    and legacy_final_payload_quick["labels_used"] is False
)

assert legacy_quick_final_gate

del legacy_final_payload_quick


legacy_encoder_postrun_gate = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    )
    == ENCODER_FINAL_STATE_SHA256
    and official_iic_encoder.training is False
    and all(
        parameter.requires_grad is False
        for parameter in official_iic_encoder.parameters()
    )
    and all(
        parameter.grad is None
        for parameter in official_iic_encoder.parameters()
    )
)

assert legacy_encoder_postrun_gate


NOTEBOOK_06_A6D_LEGACY_TRAINING_CALL_RETURNED = True
READY_FOR_A6E_LEGACY_POSTTRAINING_AUDIT = True

READY_FOR_SPECTRAL = False
READY_FOR_LABEL_EVALUATION = False


# ============================================================
# 6. REPORT
# ============================================================

print()
print(
    "=== OFFICIAL LEGACY_TEXT IIC CALL RETURNED ==="
)
print("FINAL_PATH =", official_legacy_final_path)
print("FINAL_EXISTS =", official_legacy_final_path.is_file())
print("QUICK_FINAL_GATE =", legacy_quick_final_gate)
print("ENCODER_POSTRUN_GATE =", legacy_encoder_postrun_gate)
print("LABELS_READ = False")
print("SPECTRAL_STARTED = False")
print(
    "NOTEBOOK_06_A6D_LEGACY_TRAINING_CALL_RETURNED =",
    NOTEBOOK_06_A6D_LEGACY_TRAINING_CALL_RETURNED,
)
print(
    "READY_FOR_A6E_LEGACY_POSTTRAINING_AUDIT =",
    READY_FOR_A6E_LEGACY_POSTTRAINING_AUDIT,
)
print("READY_FOR_SPECTRAL =", READY_FOR_SPECTRAL)
print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


=== STARTING / RESUMING OFFICIAL LEGACY_TEXT IIC RUN ===
RUN_ID = sakai224_full_iic_legacy_text_v1
MODE = legacy_text
MODE_ID = 0
RUN_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/sakai224_full_iic_legacy_text_v1
LAUNCH_MODE = NEW
RUN_CONFIG_SHA256 = 0be4c46a68b7571cb399883e75783a7c9ce3653d7d4506377ae796423459118f
TARGET_EPOCHS = 100
BATCHES_PER_EPOCH = 68
STEPS_PER_CLASSIFIER = 6800
K = 5
LABELS_READ = False
SPECTRAL_STARTED = False
THIS CELL WRITES TO DRIVE = True

epoch 001/100 | 45.22s | main_MI=0.68175979 | over_MI=0.63019817 | loss=-1.31195797 | step=68
epoch 002/100 | 45.39s | main_MI=1.28173392 | over_MI=1.23503589 | loss=-2.51676982 | step=136
epoch 003/100 | 43.78s | main_MI=1.62088377 | over_MI=1.64193001 | loss=-3.26281378 | step=204
epoch 004/100 | 42.49s | main_MI=1.77586153 | over_MI=1.87689566 | loss=-3.65275718 | step=272
epoch 005/100 | 44.33s | main_MI=1.86565971 | over_MI=2.01763014 | loss=-3.88328984 | step=340
epoch 006/100 | 44.18s | main_M

In [25]:
# ============================================================
# NOTEBOOK 06 — A6E
# LEGACY_TEXT POST-TRAINING AUDIT
#
# READ ONLY.
#
# Reuses the complete approved audit in historical
# Notebook-06 cell 50. It runs in an isolated namespace with
# the legacy run aliases and changes only the literal run/mode
# names plus the report title.
#
# NO labels.
# NO training.
# NO spectral clustering.
# NO Drive writes.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import statistics

import torch


# ============================================================
# 0. PARENT GATES
# ============================================================

assert (
    "NOTEBOOK_06_A6D_LEGACY_TRAINING_CALL_RETURNED" in globals()
    and NOTEBOOK_06_A6D_LEGACY_TRAINING_CALL_RETURNED is True
), "A6D non disponibile o non completato."

assert (
    "READY_FOR_A6E_LEGACY_POSTTRAINING_AUDIT" in globals()
    and READY_FOR_A6E_LEGACY_POSTTRAINING_AUDIT is True
), "Runtime non autorizzato ad A6E."

assert LEGACY_RUN_ID == "sakai224_full_iic_legacy_text_v1"
assert LEGACY_MODE_NAME == "legacy_text"
assert LEGACY_RUN_CONFIG_SHA256 == (
    "0be4c46a68b7571cb399883e75783a7c"
    "9ce3653d7d4506377ae796423459118f"
)

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ============================================================
# 1. PRIMARY MUST REMAIN FROZEN
# ============================================================

A6E_PRIMARY_FINAL_PATH = (
    SAKAI224_ROOT
    / "iic"
    / "sakai224_full_iic_symmetric_v1"
    / "final.pt"
)

EXPECTED_PRIMARY_FINAL_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)

assert A6E_PRIMARY_FINAL_PATH.is_file()

primary_sha_before_a6e = sha256_stream(
    A6E_PRIMARY_FINAL_PATH
)

assert (
    primary_sha_before_a6e
    == EXPECTED_PRIMARY_FINAL_SHA256
)


# ============================================================
# 2. LOAD APPROVED HISTORICAL AUDIT SOURCE
# ============================================================

NOTEBOOK06_PATH_A6E = (
    PROJECT_ROOT
    / "sakai224_full"
    / "notebooks"
    / "06_sakai224_full_iic_and_spectral.ipynb"
)

assert NOTEBOOK06_PATH_A6E.is_file()

with NOTEBOOK06_PATH_A6E.open(
    "r",
    encoding="utf-8",
) as handle:
    nb06_a6e = json.load(handle)

cell50 = nb06_a6e["cells"][50]

assert cell50["cell_type"] == "code"

cell50_source = "".join(
    cell50.get(
        "source",
        [],
    )
)

EXPECTED_CELL50_SHA256 = (
    "d228f30a6e50b5bfacc976e8a225ba76"
    "6d8aaa8cb2c6ffff0a454cfe63421093"
)

cell50_sha256 = hashlib.sha256(
    cell50_source.encode("utf-8")
).hexdigest()

assert (
    cell50_sha256
    == EXPECTED_CELL50_SHA256
)


# ============================================================
# 3. CONTROLLED LEGACY ADAPTATION
#
# Exact occurrence counts prevent unintended replacements.
# ============================================================

replacements = [
    (
        '"sakai224_full_iic_symmetric_v1"',
        '"sakai224_full_iic_legacy_text_v1"',
        1,
    ),
    (
        '"symmetric_figure"',
        '"legacy_text"',
        2,
    ),
    (
        '"=== NOTEBOOK 06 PRIMARY IIC POST-TRAINING AUDIT ==="',
        '"=== NOTEBOOK 06 LEGACY_TEXT IIC POST-TRAINING AUDIT ==="',
        1,
    ),
    (
        '"NOTEBOOK_06_A6B_PRIMARY_POSTTRAINING_AUDIT_PASS ="',
        '"NOTEBOOK_06_A6E_LEGACY_POSTTRAINING_AUDIT_PASS ="',
        1,
    ),
]

legacy_audit_source = cell50_source

for old, new, expected_count in replacements:
    assert (
        legacy_audit_source.count(old)
        == expected_count
    )

    legacy_audit_source = (
        legacy_audit_source.replace(
            old,
            new,
        )
    )


EXPECTED_LEGACY_AUDIT_SOURCE_SHA256 = (
    "f1a84c8e0525a313edc3a5197c186bae"
    "2515cd9b52ceb6f62e9d6fadfb4bd63d"
)

legacy_audit_source_sha256 = hashlib.sha256(
    legacy_audit_source.encode("utf-8")
).hexdigest()

assert (
    legacy_audit_source_sha256
    == EXPECTED_LEGACY_AUDIT_SOURCE_SHA256
)


# ============================================================
# 4. ISOLATED LEGACY AUDIT NAMESPACE
# ============================================================

A6E_LEGACY_RUN_DIR = (
    SAKAI224_ROOT
    / "iic"
    / LEGACY_RUN_ID
)

assert A6E_LEGACY_RUN_DIR.is_dir()

audit_namespace = dict(globals())

audit_namespace.update({
    "a5g_gate": True,
    "a5fs_gate": True,
    "a5ft_gate": True,

    "PRIMARY_RUN_ID":
        LEGACY_RUN_ID,

    "PRIMARY_RUN_DIR":
        A6E_LEGACY_RUN_DIR,

    "PRIMARY_RUN_CONFIG_JSON":
        LEGACY_RUN_CONFIG_JSON,

    "PRIMARY_RUN_CONFIG_SHA256":
        LEGACY_RUN_CONFIG_SHA256,
})


# ============================================================
# 5. EXECUTE COMPLETE APPROVED AUDIT
# ============================================================

exec(
    compile(
        legacy_audit_source,
        (
            str(NOTEBOOK06_PATH_A6E)
            + "::CELL50_LEGACY_ADAPTATION"
        ),
        "exec",
    ),
    audit_namespace,
    audit_namespace,
)


assert (
    audit_namespace.get(
        "a6b_gate"
    ) is True
)


# ============================================================
# 6. LEGACY-SPECIFIC EXTRA CHECKS
# ============================================================

audited_config = audit_namespace[
    "config_data"
]

legacy_extra_gate = (
    audited_config["run_id"]
        == LEGACY_RUN_ID
    and audited_config["role"]
        == "HTML_TEXT_FIDELITY_SENSITIVITY"
    and audited_config["augmentation_mode"]
        == "legacy_text"
    and audited_config["augmentation_mode_id"]
        == int(GS_AUGMENT_MODE_LEGACY_TEXT)
    and audited_config["labels_used"] is False
)

assert legacy_extra_gate


A6E_LATEST_FILE_SHA256 = audit_namespace[
    "A6B_LATEST_FILE_SHA256"
]

A6E_FINAL_FILE_SHA256 = audit_namespace[
    "A6B_FINAL_FILE_SHA256"
]

A6E_METRICS_FILE_SHA256 = audit_namespace[
    "A6B_METRICS_FILE_SHA256"
]

A6E_CONFIG_FILE_SHA256 = audit_namespace[
    "A6B_CONFIG_FILE_SHA256"
]

A6E_SESSIONS_FILE_SHA256 = audit_namespace[
    "A6B_SESSIONS_FILE_SHA256"
]

A6E_FINAL_CLASSIFIER_TREE_SHA256 = audit_namespace[
    "final_classifier_tree_sha"
]

A6E_FINAL_OPTIMIZER_TREE_SHA256 = audit_namespace[
    "final_optimizer_tree_sha"
]

A6E_FIRST_MAIN_MI = audit_namespace[
    "first_main_mi"
]

A6E_FINAL_MAIN_MI = audit_namespace[
    "final_main_mi"
]

A6E_FIRST_OVERCLUSTER_MI = audit_namespace[
    "first_over_mi"
]

A6E_FINAL_OVERCLUSTER_MI = audit_namespace[
    "final_over_mi"
]

A6E_FIRST_TOTAL_LOSS = audit_namespace[
    "first_total_loss"
]

A6E_FINAL_TOTAL_LOSS = audit_namespace[
    "final_total_loss"
]

A6E_TOTAL_TRAINING_SECONDS = audit_namespace[
    "total_training_seconds"
]

A6E_MEDIAN_EPOCH_SECONDS = statistics.median(
    audit_namespace[
        "elapsed_seconds_values"
    ]
)


# ============================================================
# 7. POST-AUDIT IMMUTABILITY
# ============================================================

primary_sha_after_a6e = sha256_stream(
    A6E_PRIMARY_FINAL_PATH
)

primary_frozen_gate = (
    primary_sha_before_a6e
    == primary_sha_after_a6e
    == EXPECTED_PRIMARY_FINAL_SHA256
)

encoder_frozen_gate = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    ) == ENCODER_FINAL_STATE_SHA256
    and official_iic_encoder.training is False
    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )
    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert primary_frozen_gate
assert encoder_frozen_gate


# ============================================================
# 8. FINAL FLAGS
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False
SPECTRAL_STARTED = False
DRIVE_MODIFIED_BY_AUDIT = False

NOTEBOOK_06_A6E_LEGACY_POSTTRAINING_AUDIT_PASS = True
READY_FOR_A6F_LEGACY_HANDOFF = True
READY_FOR_SPECTRAL = False
READY_FOR_LABEL_EVALUATION = False


# ============================================================
# 9. FINAL LEGACY-SPECIFIC REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — A6E LEGACY AUDIT FINAL GATE ==="
)
print(
    "CELL50_SOURCE_SHA256 =",
    cell50_sha256,
)
print(
    "LEGACY_AUDIT_SOURCE_SHA256 =",
    legacy_audit_source_sha256,
)
print(
    "LEGACY_EXTRA_GATE =",
    legacy_extra_gate,
)
print(
    "LATEST_FILE_SHA256 =",
    A6E_LATEST_FILE_SHA256,
)
print(
    "FINAL_FILE_SHA256 =",
    A6E_FINAL_FILE_SHA256,
)
print(
    "METRICS_FILE_SHA256 =",
    A6E_METRICS_FILE_SHA256,
)
print(
    "RUN_CONFIG_SHA256 =",
    A6E_CONFIG_FILE_SHA256,
)
print(
    "RUNTIME_SESSIONS_SHA256 =",
    A6E_SESSIONS_FILE_SHA256,
)
print(
    "CLASSIFIER_STATE_SHA256 =",
    A6E_FINAL_CLASSIFIER_TREE_SHA256,
)
print(
    "OPTIMIZER_STATE_SHA256 =",
    A6E_FINAL_OPTIMIZER_TREE_SHA256,
)
print(
    "FIRST_MAIN_MI =",
    A6E_FIRST_MAIN_MI,
)
print(
    "FINAL_MAIN_MI =",
    A6E_FINAL_MAIN_MI,
)
print(
    "FIRST_OVERCLUSTER_MI =",
    A6E_FIRST_OVERCLUSTER_MI,
)
print(
    "FINAL_OVERCLUSTER_MI =",
    A6E_FINAL_OVERCLUSTER_MI,
)
print(
    "FIRST_TOTAL_LOSS =",
    A6E_FIRST_TOTAL_LOSS,
)
print(
    "FINAL_TOTAL_LOSS =",
    A6E_FINAL_TOTAL_LOSS,
)
print(
    "TOTAL_TRAINING_SECONDS =",
    A6E_TOTAL_TRAINING_SECONDS,
)
print(
    "MEDIAN_EPOCH_SECONDS =",
    A6E_MEDIAN_EPOCH_SECONDS,
)
print(
    "PRIMARY_FROZEN_GATE =",
    primary_frozen_gate,
)
print(
    "ENCODER_FROZEN_GATE =",
    encoder_frozen_gate,
)
print(
    "LABELS_READ =",
    LABELS_READ,
)
print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)
print(
    "DRIVE_MODIFIED_BY_AUDIT =",
    DRIVE_MODIFIED_BY_AUDIT,
)
print(
    "NOTEBOOK_06_A6E_LEGACY_POSTTRAINING_AUDIT_PASS =",
    NOTEBOOK_06_A6E_LEGACY_POSTTRAINING_AUDIT_PASS,
)
print(
    "READY_FOR_A6F_LEGACY_HANDOFF =",
    READY_FOR_A6F_LEGACY_HANDOFF,
)
print(
    "READY_FOR_SPECTRAL =",
    READY_FOR_SPECTRAL,
)
print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


# ============================================================
# 10. CLEAN ISOLATED NAMESPACE
# ============================================================

audit_namespace.clear()

gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()


=== NOTEBOOK 06 LEGACY_TEXT IIC POST-TRAINING AUDIT ===

RUN_ID = sakai224_full_iic_legacy_text_v1
LATEST_FILE_SHA256 = 9cbe4b0b2dd61c687eaef4f6fd79317f8b99ef3aa046ce76f96b5c0187680573
FINAL_FILE_SHA256 = c6ba5cf4272afdc22335eadfaf46c0f0a63a6d33fccebf58884063a4db896a37
METRICS_FILE_SHA256 = 08a02ad6030594110c8ed24be5069c752f0d58b5b2e4a5f431426b44ee3aa952
RUN_CONFIG_SHA256 = 0be4c46a68b7571cb399883e75783a7c9ce3653d7d4506377ae796423459118f
RUNTIME_SESSIONS_SHA256 = 6a837385d59a32cc7ba622ba1de30f98953639cfb0bdec47d29d833125275957

LATEST_CHECKPOINT_GATE = True
FINAL_CHECKPOINT_GATE = True
LATEST_FINAL_NEURAL_STATE_EXACT = True
ADAM_INTERNAL_STEP_GATE = True
BN_COUNTER_GATE = True
BN_BATCHES_TRACKED = [(13600, 13600), (13600, 13600), (13600, 13600), (13600, 13600), (13600, 13600)]

METRICS_ROWS = 100
METRICS_SCHEMA_GATE = True
METRICS_GEOMETRY_GATE = True
METRICS_LOSS_IDENTITY_GATE = True

FIRST_MAIN_MI = 0.6817597937991036
FINAL_MAIN_MI = 3.100201808354434
FIRST_OVERCLUSTER_MI = 0.630198

In [26]:
# ============================================================
# NOTEBOOK 06 — A6F
# DUAL-IIC PERSISTENT HANDOFF TO LABEL-BLIND SPECTRAL
#
# Scrive un solo JSON di handoff su Drive.
# Non esegue training, spectral clustering o lettura label.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import torch


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert (
    globals().get(
        "NOTEBOOK_06_A6E_LEGACY_POSTTRAINING_AUDIT_PASS"
    ) is True
), "A6E non disponibile o non PASS."

assert (
    globals().get(
        "READY_FOR_A6F_LEGACY_HANDOFF"
    ) is True
), "Runtime non autorizzato ad A6F."

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

A6F_IIC_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
)

A6F_AUDIT_DIR = (
    A6F_IIC_ROOT
    / "audits"
)

A6F_PRIMARY_RUN_DIR = (
    A6F_IIC_ROOT
    / "sakai224_full_iic_symmetric_v1"
)

A6F_LEGACY_RUN_DIR = (
    A6F_IIC_ROOT
    / "sakai224_full_iic_legacy_text_v1"
)

A6F_PRIMARY_HANDOFF_PATH = (
    A6F_AUDIT_DIR
    / "sakai224_full_iic_symmetric_v1_after_A6B.json"
)

A6F_DUAL_HANDOFF_PATH = (
    A6F_AUDIT_DIR
    / "sakai224_full_iic_legacy_text_v1_after_A6E.json"
)


# ------------------------------------------------------------
# 2. Frozen identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_HANDOFF_SHA256 = (
    "8154f74e4bd34d086d99c706c3e65cf"
    "2d67ba8da1b16b3308b360264bc2bd1db"
)

EXPECTED_PRIMARY_FINAL_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)

EXPECTED_LEGACY_SHA256 = {
    "latest.pt":
        "9cbe4b0b2dd61c687eaef4f6fd79317f"
        "8b99ef3aa046ce76f96b5c0187680573",

    "final.pt":
        "c6ba5cf4272afdc22335eadfaf46c0f0"
        "a63a6d33fccebf58884063a4db896a37",

    "metrics.csv":
        "08a02ad6030594110c8ed24be5069c752"
        "f0d58b5b2e4a5f431426b44ee3aa952",

    "run_config.json":
        "0be4c46a68b7571cb399883e75783a7c"
        "9ce3653d7d4506377ae796423459118f",

    "runtime_sessions.jsonl":
        "6a837385d59a32cc7ba622ba1de30f989"
        "53639cfb0bdec47d29d833125275957",
}

EXPECTED_LEGACY_CLASSIFIER_STATE_SHA256 = (
    "5e0b0de8d1e9654c7556d4501c6856ad"
    "c634c0f859ce5ac4c1cb172bc378d734"
)

EXPECTED_LEGACY_OPTIMIZER_STATE_SHA256 = (
    "468a14951de8235429f57716729de1f4b"
    "a7198aaedd7c57765378f9f20c621ae"
)


# ------------------------------------------------------------
# 3. Primary A6B/A6C verification
# ------------------------------------------------------------

primary_final_path = (
    A6F_PRIMARY_RUN_DIR
    / "final.pt"
)

assert primary_final_path.is_file()
assert A6F_PRIMARY_HANDOFF_PATH.is_file()

primary_final_sha256 = sha256_stream(
    primary_final_path
)

primary_handoff_sha256 = sha256_stream(
    A6F_PRIMARY_HANDOFF_PATH
)

primary_identity_gate = (
    primary_final_sha256
        == EXPECTED_PRIMARY_FINAL_SHA256
    and primary_handoff_sha256
        == EXPECTED_PRIMARY_HANDOFF_SHA256
)

assert primary_identity_gate

with A6F_PRIMARY_HANDOFF_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    primary_handoff = json.load(handle)

primary_handoff_gate = (
    primary_handoff["status"]
        == "SAFE_TO_STOP_RUNTIME"

    and primary_handoff["primary_run"]["run_id"]
        == "sakai224_full_iic_symmetric_v1"

    and primary_handoff["primary_run"]["mode"]
        == "symmetric_figure"

    and primary_handoff["primary_run"]["epochs_completed"]
        == 100

    and primary_handoff["primary_run"][
        "optimizer_steps_per_classifier"
    ] == 6800

    and primary_handoff["primary_run"]["classifier_count_K"]
        == 5

    and primary_handoff["primary_run"]["audit_pass"]
        is True

    and primary_handoff["scientific_state"]["labels_read"]
        is False
)

assert primary_handoff_gate


# ------------------------------------------------------------
# 4. Legacy A6E verification
# ------------------------------------------------------------

assert A6F_LEGACY_RUN_DIR.is_dir()

legacy_artifact_hashes = {}

for file_name, expected_sha256 in (
    EXPECTED_LEGACY_SHA256.items()
):
    path = (
        A6F_LEGACY_RUN_DIR
        / file_name
    )

    assert path.is_file()

    actual_sha256 = sha256_stream(
        path
    )

    legacy_artifact_hashes[
        file_name
    ] = actual_sha256

    assert actual_sha256 == expected_sha256


legacy_artifact_gate = (
    legacy_artifact_hashes
    == EXPECTED_LEGACY_SHA256
)

legacy_partial_files = list(
    A6F_LEGACY_RUN_DIR.rglob(
        "*.partial"
    )
)

legacy_no_partial_gate = (
    len(
        legacy_partial_files
    ) == 0
)

assert legacy_artifact_gate
assert legacy_no_partial_gate


legacy_final_payload = torch.load(
    A6F_LEGACY_RUN_DIR
    / "final.pt",
    map_location="cpu",
    weights_only=True,
)

legacy_final_semantic_gate = (
    legacy_final_payload["kind"]
        == "FINAL"

    and legacy_final_payload["run_id"]
        == "sakai224_full_iic_legacy_text_v1"

    and legacy_final_payload["augmentation_mode"]
        == "legacy_text"

    and legacy_final_payload["last_completed_epoch"]
        == 99

    and legacy_final_payload["next_epoch"]
        == 100

    and legacy_final_payload[
        "optimizer_steps_per_classifier"
    ] == [
        6800,
        6800,
        6800,
        6800,
        6800,
    ]

    and legacy_final_payload["run_config_sha256"]
        == EXPECTED_LEGACY_SHA256[
            "run_config.json"
        ]

    and legacy_final_payload["labels_used"]
        is False
)

assert legacy_final_semantic_gate

del legacy_final_payload


legacy_a6e_identity_gate = (
    A6E_LATEST_FILE_SHA256
        == EXPECTED_LEGACY_SHA256[
            "latest.pt"
        ]

    and A6E_FINAL_FILE_SHA256
        == EXPECTED_LEGACY_SHA256[
            "final.pt"
        ]

    and A6E_METRICS_FILE_SHA256
        == EXPECTED_LEGACY_SHA256[
            "metrics.csv"
        ]

    and A6E_CONFIG_FILE_SHA256
        == EXPECTED_LEGACY_SHA256[
            "run_config.json"
        ]

    and A6E_SESSIONS_FILE_SHA256
        == EXPECTED_LEGACY_SHA256[
            "runtime_sessions.jsonl"
        ]

    and A6E_FINAL_CLASSIFIER_TREE_SHA256
        == EXPECTED_LEGACY_CLASSIFIER_STATE_SHA256

    and A6E_FINAL_OPTIMIZER_TREE_SHA256
        == EXPECTED_LEGACY_OPTIMIZER_STATE_SHA256
)

assert legacy_a6e_identity_gate


# ------------------------------------------------------------
# 5. Encoder immutability
# ------------------------------------------------------------

encoder_handoff_gate = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    ) == ENCODER_FINAL_STATE_SHA256

    and official_iic_encoder.training
        is False

    and all(
        parameter.requires_grad is False
        for parameter
        in official_iic_encoder.parameters()
    )

    and all(
        parameter.grad is None
        for parameter
        in official_iic_encoder.parameters()
    )
)

assert encoder_handoff_gate


# ------------------------------------------------------------
# 6. Refuse overwrite
# ------------------------------------------------------------

A6F_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert not A6F_DUAL_HANDOFF_PATH.exists(), (
    f"STOP: handoff already exists: "
    f"{A6F_DUAL_HANDOFF_PATH}"
)


# ------------------------------------------------------------
# 7. Build handoff
# ------------------------------------------------------------

A6F_DUAL_HANDOFF = {
    "format_version":
        1,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project_phase":
        "Notebook06_dual_IIC_complete",

    "completed_step":
        "A6E_legacy_posttraining_audit",

    "primary_run":
        primary_handoff[
            "primary_run"
        ],

    "primary_artifacts":
        primary_handoff[
            "artifacts"
        ],

    "primary_handoff": {
        "path":
            (
                "/content/drive/MyDrive/TESI_SAKAI2022/"
                "sakai224_full/iic/audits/"
                "sakai224_full_iic_symmetric_v1_after_A6B.json"
            ),

        "sha256":
            primary_handoff_sha256,
    },

    "legacy_run": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "mode":
            "legacy_text",

        "role":
            "HTML_TEXT_FIDELITY_SENSITIVITY",

        "epochs_completed":
            100,

        "optimizer_steps_per_classifier":
            6800,

        "classifier_count_K":
            5,

        "audit_pass":
            True,
    },

    "legacy_artifacts": {
        "latest_sha256":
            legacy_artifact_hashes[
                "latest.pt"
            ],

        "final_sha256":
            legacy_artifact_hashes[
                "final.pt"
            ],

        "metrics_sha256":
            legacy_artifact_hashes[
                "metrics.csv"
            ],

        "run_config_sha256":
            legacy_artifact_hashes[
                "run_config.json"
            ],

        "runtime_sessions_sha256":
            legacy_artifact_hashes[
                "runtime_sessions.jsonl"
            ],

        "classifier_state_sha256":
            A6E_FINAL_CLASSIFIER_TREE_SHA256,

        "optimizer_state_sha256":
            A6E_FINAL_OPTIMIZER_TREE_SHA256,
    },

    "shared_scientific_contract": {
        "event_count":
            8583,

        "central_feature":
            "mu_phi_x",

        "latent_dim":
            512,

        "main_classes_C":
            36,

        "overcluster_classes_W":
            250,

        "classifier_count_K":
            5,

        "central_view":
            (
                "precomputed_audited_mu_"
                "from_FINAL_encoder"
            ),

        "perturbed_view":
            (
                "on_the_fly_mu_from_"
                "same_FINAL_encoder"
            ),

        "encoder_shared_frozen_eval_no_grad":
            True,

        "labels_used":
            False,
    },

    "scientific_state": {
        "primary_mode":
            "symmetric_figure",

        "sensitivity_mode":
            "legacy_text",

        "both_iic_runs_complete":
            True,

        "both_iic_runs_audited":
            True,

        "encoder_immutable":
            True,

        "labels_read":
            False,

        "spectral_started":
            False,
    },

    "next_required_step": {
        "phase":
            "label_blind_spectral_clustering",

        "primary_run_id":
            "sakai224_full_iic_symmetric_v1",

        "sensitivity_run_id":
            "sakai224_full_iic_legacy_text_v1",

        "required_order": [
            "construct_classifier_outputs",
            "construct_H",
            "construct_affinity",
            "run_spectral_clustering",
            (
                "freeze_primary_and_"
                "sensitivity_partitions"
            ),
        ],

        "primary_interpretation":
            "symmetric_figure",

        "sensitivity_interpretation":
            "legacy_text",

        "do_not_read_labels":
            True,
    },

    "status":
        "READY_FOR_LABEL_BLIND_SPECTRAL",
}


A6F_DUAL_HANDOFF_JSON = json.dumps(
    A6F_DUAL_HANDOFF,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


# ------------------------------------------------------------
# 8. Atomic write and read-back
# ------------------------------------------------------------

atomic_write_text(
    A6F_DUAL_HANDOFF_PATH,
    A6F_DUAL_HANDOFF_JSON,
)

A6F_DUAL_HANDOFF_SHA256 = sha256_stream(
    A6F_DUAL_HANDOFF_PATH
)

with A6F_DUAL_HANDOFF_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    a6f_readback = json.load(handle)


a6f_readback_gate = (
    a6f_readback["status"]
        == "READY_FOR_LABEL_BLIND_SPECTRAL"

    and a6f_readback["scientific_state"][
        "both_iic_runs_complete"
    ] is True

    and a6f_readback["scientific_state"][
        "both_iic_runs_audited"
    ] is True

    and a6f_readback["scientific_state"][
        "labels_read"
    ] is False

    and a6f_readback["scientific_state"][
        "spectral_started"
    ] is False

    and a6f_readback["next_required_step"][
        "do_not_read_labels"
    ] is True

    and a6f_readback["primary_run"]["run_id"]
        == "sakai224_full_iic_symmetric_v1"

    and a6f_readback["legacy_run"]["run_id"]
        == "sakai224_full_iic_legacy_text_v1"
)

assert a6f_readback_gate


# ------------------------------------------------------------
# 9. Post-write immutability
# ------------------------------------------------------------

primary_post_handoff_gate = (
    sha256_stream(
        primary_final_path
    ) == EXPECTED_PRIMARY_FINAL_SHA256
)

legacy_post_handoff_gate = all(
    sha256_stream(
        A6F_LEGACY_RUN_DIR
        / file_name
    ) == expected_sha256

    for file_name, expected_sha256
    in EXPECTED_LEGACY_SHA256.items()
)

encoder_post_handoff_gate = (
    canonical_tree_sha256(
        official_iic_encoder.state_dict()
    ) == ENCODER_FINAL_STATE_SHA256
)

assert primary_post_handoff_gate
assert legacy_post_handoff_gate
assert encoder_post_handoff_gate


# ------------------------------------------------------------
# 10. Final flags
# ------------------------------------------------------------

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False
SPECTRAL_STARTED = False

NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS = True
READY_FOR_SPECTRAL = True
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — A6F DUAL-IIC HANDOFF ==="
)

print(
    "PRIMARY_RUN_ID =",
    a6f_readback[
        "primary_run"
    ][
        "run_id"
    ],
)

print(
    "PRIMARY_FINAL_SHA256 =",
    primary_final_sha256,
)

print(
    "PRIMARY_HANDOFF_SHA256 =",
    primary_handoff_sha256,
)

print(
    "LEGACY_RUN_ID =",
    a6f_readback[
        "legacy_run"
    ][
        "run_id"
    ],
)

print(
    "LEGACY_FINAL_SHA256 =",
    legacy_artifact_hashes[
        "final.pt"
    ],
)

print(
    "LEGACY_CLASSIFIER_STATE_SHA256 =",
    A6E_FINAL_CLASSIFIER_TREE_SHA256,
)

print(
    "LEGACY_OPTIMIZER_STATE_SHA256 =",
    A6E_FINAL_OPTIMIZER_TREE_SHA256,
)

print(
    "HANDOFF_PATH =",
    A6F_DUAL_HANDOFF_PATH,
)

print(
    "HANDOFF_SHA256 =",
    A6F_DUAL_HANDOFF_SHA256,
)

print(
    "BOTH_IIC_RUNS_COMPLETE =",
    a6f_readback[
        "scientific_state"
    ][
        "both_iic_runs_complete"
    ],
)

print(
    "BOTH_IIC_RUNS_AUDITED =",
    a6f_readback[
        "scientific_state"
    ][
        "both_iic_runs_audited"
    ],
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS =",
    NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS,
)

print(
    "READY_FOR_SPECTRAL =",
    READY_FOR_SPECTRAL,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


=== NOTEBOOK 06 — A6F DUAL-IIC HANDOFF ===
PRIMARY_RUN_ID = sakai224_full_iic_symmetric_v1
PRIMARY_FINAL_SHA256 = 018008df8e6a1f97195cb1cc234e915036806b2d3c1d09efefbf4333fe7f1199
PRIMARY_HANDOFF_SHA256 = 8154f74e4bd34d086d99c706c3e65cf2d67ba8da1b16b3308b360264bc2bd1db
LEGACY_RUN_ID = sakai224_full_iic_legacy_text_v1
LEGACY_FINAL_SHA256 = c6ba5cf4272afdc22335eadfaf46c0f0a63a6d33fccebf58884063a4db896a37
LEGACY_CLASSIFIER_STATE_SHA256 = 5e0b0de8d1e9654c7556d4501c6856adc634c0f859ce5ac4c1cb172bc378d734
LEGACY_OPTIMIZER_STATE_SHA256 = 468a14951de8235429f57716729de1f4ba7198aaedd7c57765378f9f20c621ae
HANDOFF_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/iic/audits/sakai224_full_iic_legacy_text_v1_after_A6E.json
HANDOFF_SHA256 = e2212af6a6791e45f0bef3a5ea1771aeb976b2130a355f692f2bda3654400564
BOTH_IIC_RUNS_COMPLETE = True
BOTH_IIC_RUNS_AUDITED = True
LABELS_READ = False
SPECTRAL_STARTED = False
NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS = True
READY_FOR_SPECTRAL = True
READY_FOR_LABEL_

In [27]:
# ============================================================
# NOTEBOOK 06 — S0
# LABEL-BLIND H-CONSTRUCTION PREFLIGHT
#
# PAPER:
#   H has shape D x (K*C), built from K classifier results.
#
# PROJECT frozen here:
#   - central mu in manifest_index order
#   - main-head SoftMax probabilities only
#   - classifier order = seeds 123..127
#   - eval() + inference_mode(), float32
#
# NO labels.
# NO affinity.
# NO spectral clustering.
# NO optimizer.
# NO training.
# NO Drive writes.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json

import numpy as np
import torch


# ============================================================
# 0. PARENT GATES
# ============================================================

assert globals().get(
    "NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS"
) is True, "A6F non disponibile o non PASS."

assert globals().get(
    "READY_FOR_SPECTRAL"
) is True, "Runtime non autorizzato allo spectral label-blind."

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ============================================================
# 1. AUTHORITATIVE PATHS / HASHES
# ============================================================

SPECTRAL_IIC_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
    / "iic"
)

SPECTRAL_PRIMARY_FINAL = (
    SPECTRAL_IIC_ROOT
    / "sakai224_full_iic_symmetric_v1"
    / "final.pt"
)

SPECTRAL_LEGACY_FINAL = (
    SPECTRAL_IIC_ROOT
    / "sakai224_full_iic_legacy_text_v1"
    / "final.pt"
)

SPECTRAL_DUAL_HANDOFF = (
    SPECTRAL_IIC_ROOT
    / "audits"
    / "sakai224_full_iic_legacy_text_v1_after_A6E.json"
)

EXPECTED_PRIMARY_FINAL_SHA256 = (
    "018008df8e6a1f97195cb1cc234e91503"
    "6806b2d3c1d09efefbf4333fe7f1199"
)

EXPECTED_LEGACY_FINAL_SHA256 = (
    "c6ba5cf4272afdc22335eadfaf46c0f0"
    "a63a6d33fccebf58884063a4db896a37"
)

EXPECTED_DUAL_HANDOFF_SHA256 = (
    "e2212af6a6791e45f0bef3a5ea1771ae"
    "b976b2130a355f692f2bda3654400564"
)

EXPECTED_PRIMARY_CONFIG_SHA256 = (
    "6a34e279824042a12be112ba04ce0fb4"
    "c30f500c901921a8d2aee5ab7b817aca"
)

EXPECTED_LEGACY_CONFIG_SHA256 = (
    "0be4c46a68b7571cb399883e75783a7c"
    "9ce3653d7d4506377ae796423459118f"
)

assert SPECTRAL_PRIMARY_FINAL.is_file()
assert SPECTRAL_LEGACY_FINAL.is_file()
assert SPECTRAL_DUAL_HANDOFF.is_file()

artifact_identity_gate = (
    sha256_stream(SPECTRAL_PRIMARY_FINAL)
        == EXPECTED_PRIMARY_FINAL_SHA256
    and sha256_stream(SPECTRAL_LEGACY_FINAL)
        == EXPECTED_LEGACY_FINAL_SHA256
    and sha256_stream(SPECTRAL_DUAL_HANDOFF)
        == EXPECTED_DUAL_HANDOFF_SHA256
)

assert artifact_identity_gate

with SPECTRAL_DUAL_HANDOFF.open(
    "r",
    encoding="utf-8",
) as handle:
    spectral_handoff = json.load(handle)

handoff_gate = (
    spectral_handoff["status"]
        == "READY_FOR_LABEL_BLIND_SPECTRAL"
    and spectral_handoff["scientific_state"][
        "both_iic_runs_complete"
    ] is True
    and spectral_handoff["scientific_state"][
        "both_iic_runs_audited"
    ] is True
    and spectral_handoff["scientific_state"][
        "labels_read"
    ] is False
    and spectral_handoff["scientific_state"][
        "spectral_started"
    ] is False
)

assert handoff_gate


# ============================================================
# 2. LOAD FINAL CHECKPOINTS — READ ONLY
# ============================================================

spectral_primary_payload = torch.load(
    SPECTRAL_PRIMARY_FINAL,
    map_location="cpu",
    weights_only=True,
)

spectral_legacy_payload = torch.load(
    SPECTRAL_LEGACY_FINAL,
    map_location="cpu",
    weights_only=True,
)


def validate_spectral_source_payload(
    payload,
    *,
    expected_run_id,
    expected_mode,
    expected_config_sha256,
):
    return all([
        payload["kind"] == "FINAL",
        payload["run_id"] == expected_run_id,
        payload["augmentation_mode"] == expected_mode,
        payload["last_completed_epoch"] == 99,
        payload["next_epoch"] == 100,
        payload["classifier_seeds"]
            == [123, 124, 125, 126, 127],
        payload["optimizer_steps_per_classifier"]
            == [6800] * 5,
        payload["run_config_sha256"]
            == expected_config_sha256,
        payload["labels_used"] is False,
        len(payload["classifier_state_dicts"]) == 5,
        canonical_tree_sha256(
            payload["classifier_state_dicts"]
        ) == payload["classifier_states_sha256"],
        payload["source_identity"]["central_mu_sha256"]
            == FULL_CENTRAL_MU_SHA256,
        payload["source_identity"]["encoder_state_sha256"]
            == ENCODER_FINAL_STATE_SHA256,
        payload["contracts"]["loss"]
            == IIC_LOSS_CONTRACT_SHA256,
        payload["contracts"]["architecture"]
            == IIC_ARCHITECTURE_CONTRACT_SHA256,
        payload["contracts"]["training_core"]
            == IIC_TRAINING_CORE_SHA256,
        payload["contracts"]["final_training"]
            == IIC_FINAL_TRAINING_SHA256,
        payload["contracts"]["dual_experiment"]
            == IIC_DUAL_EXPERIMENT_SHA256,
        payload["contracts"]["official_run"]
            == IIC_OFFICIAL_RUN_SHA256,
        payload["contracts"]["checkpoint_resume"]
            == IIC_CHECKPOINT_RESUME_SHA256,
        payload["contracts"]["safe_serialization"]
            == IIC_SAFE_SERIALIZATION_SHA256,
    ])


primary_payload_gate = validate_spectral_source_payload(
    spectral_primary_payload,
    expected_run_id="sakai224_full_iic_symmetric_v1",
    expected_mode="symmetric_figure",
    expected_config_sha256=EXPECTED_PRIMARY_CONFIG_SHA256,
)

legacy_payload_gate = validate_spectral_source_payload(
    spectral_legacy_payload,
    expected_run_id="sakai224_full_iic_legacy_text_v1",
    expected_mode="legacy_text",
    expected_config_sha256=EXPECTED_LEGACY_CONFIG_SHA256,
)

assert primary_payload_gate
assert legacy_payload_gate


# ============================================================
# 3. RESTORE CLASSIFIERS ONLY — NO OPTIMIZERS
# ============================================================

def restore_frozen_eval_ensemble(payload):
    models = []
    expected_hashes = []

    for seed, state_dict in zip(
        payload["classifier_seeds"],
        payload["classifier_state_dicts"],
        strict=True,
    ):
        model = build_iic_pair_from_seed(seed)

        load_result = model.load_state_dict(
            state_dict,
            strict=True,
        )

        assert load_result.missing_keys == []
        assert load_result.unexpected_keys == []

        expected_hash = canonical_tree_sha256(
            state_dict
        )

        model = model.to(runtime_device)
        model.eval()

        for parameter in model.parameters():
            parameter.requires_grad_(False)

        assert canonical_tree_sha256(
            model.state_dict()
        ) == expected_hash

        assert model.training is False
        assert all(
            parameter.requires_grad is False
            for parameter in model.parameters()
        )
        assert all(
            parameter.grad is None
            for parameter in model.parameters()
        )

        models.append(model)
        expected_hashes.append(expected_hash)

    return models, expected_hashes


(
    spectral_primary_models,
    spectral_primary_model_hashes,
) = restore_frozen_eval_ensemble(
    spectral_primary_payload
)

(
    spectral_legacy_models,
    spectral_legacy_model_hashes,
) = restore_frozen_eval_ensemble(
    spectral_legacy_payload
)

ensemble_restore_gate = (
    len(spectral_primary_models) == 5
    and len(spectral_legacy_models) == 5
    and len(set(spectral_primary_model_hashes)) == 5
    and len(set(spectral_legacy_model_hashes)) == 5
)

assert ensemble_restore_gate

del spectral_primary_payload
del spectral_legacy_payload
gc.collect()


# ============================================================
# 4. FIXED LABEL-BLIND PROBE
# ============================================================

SPECTRAL_PROBE_MANIFEST_INDICES = [
    0,
    1,
    2,
    2145,
    4291,
    6864,
    7663,
    8582,
]

spectral_probe_cpu = torch.from_numpy(
    np.asarray(
        full_central_mu[
            SPECTRAL_PROBE_MANIFEST_INDICES
        ],
        dtype=np.float32,
    ).copy()
)

spectral_probe_gpu = spectral_probe_cpu.to(
    runtime_device
)


def build_main_probability_blocks(
    models,
    features_gpu,
):
    blocks = []

    with torch.inference_mode():
        for model in models:
            blocks.append(
                model.main_head(
                    features_gpu
                )
            )

    return blocks


primary_probe_blocks = build_main_probability_blocks(
    spectral_primary_models,
    spectral_probe_gpu,
)

legacy_probe_blocks = build_main_probability_blocks(
    spectral_legacy_models,
    spectral_probe_gpu,
)

primary_h_probe = torch.cat(
    primary_probe_blocks,
    dim=1,
)

legacy_h_probe = torch.cat(
    legacy_probe_blocks,
    dim=1,
)


# ============================================================
# 5. H SEMANTIC GATES
# ============================================================

def probability_blocks_valid(blocks):
    return (
        len(blocks) == 5
        and all(
            block.shape == (8, 36)
            and block.dtype == torch.float32
            and torch.isfinite(block).all().item()
            and float(block.min().item()) >= 0.0
            and float(block.max().item()) <= 1.0
            and torch.allclose(
                block.sum(dim=1),
                torch.ones(
                    8,
                    dtype=torch.float32,
                    device=block.device,
                ),
                rtol=0.0,
                atol=2.0e-6,
            )
            for block in blocks
        )
    )


primary_probability_gate = probability_blocks_valid(
    primary_probe_blocks
)

legacy_probability_gate = probability_blocks_valid(
    legacy_probe_blocks
)

h_probe_gate = (
    primary_h_probe.shape == (8, 180)
    and legacy_h_probe.shape == (8, 180)
    and primary_h_probe.dtype == torch.float32
    and legacy_h_probe.dtype == torch.float32
    and torch.isfinite(primary_h_probe).all().item()
    and torch.isfinite(legacy_h_probe).all().item()
    and torch.allclose(
        primary_h_probe.sum(dim=1),
        torch.full(
            (8,),
            5.0,
            dtype=torch.float32,
            device=runtime_device,
        ),
        rtol=0.0,
        atol=1.0e-5,
    )
    and torch.allclose(
        legacy_h_probe.sum(dim=1),
        torch.full(
            (8,),
            5.0,
            dtype=torch.float32,
            device=runtime_device,
        ),
        rtol=0.0,
        atol=1.0e-5,
    )
)

assert primary_probability_gate
assert legacy_probability_gate
assert h_probe_gate

primary_legacy_probe_distinct = (
    not torch.equal(
        primary_h_probe,
        legacy_h_probe,
    )
)

assert primary_legacy_probe_distinct


# ============================================================
# 6. CLASSIFIER IMMUTABILITY
# ============================================================

classifier_immutability_gate = (
    [
        canonical_tree_sha256(
            model.state_dict()
        )
        for model in spectral_primary_models
    ] == spectral_primary_model_hashes

    and

    [
        canonical_tree_sha256(
            model.state_dict()
        )
        for model in spectral_legacy_models
    ] == spectral_legacy_model_hashes
)

assert classifier_immutability_gate


# ============================================================
# 7. FREEZE H-CONSTRUCTION CONTRACT
# ============================================================

SPECTRAL_H_CONTRACT = {
    "contract_version": 1,

    "paper": {
        "D": 8583,
        "K": 5,
        "C": 36,
        "H_shape": [8583, 180],
        "construction":
            "concatenate_K_classifier_results",
    },

    "project": {
        "source_feature":
            "full_central_mu",

        "source_feature_sha256":
            FULL_CENTRAL_MU_SHA256,

        "row_order":
            "manifest_index_0_to_8582",

        "classifier_branch":
            "main_head_only",

        "classifier_output":
            "softmax_probabilities",

        "classifier_block_order":
            [123, 124, 125, 126, 127],

        "model_mode":
            "eval",

        "gradient_mode":
            "torch.inference_mode",

        "dtype":
            "float32",

        "overcluster_output_used":
            False,

        "labels_used":
            False,
    },

    "runs": {
        "primary":
            "sakai224_full_iic_symmetric_v1",

        "sensitivity":
            "sakai224_full_iic_legacy_text_v1",
    },

    "not_yet_frozen_here": [
        "affinity_storage_dtype",
        "spectral_laplacian_normalization",
        "eigensolver",
        "eigenvector_postprocessing",
        "final_kmeans_seed_and_n_init",
    ],
}

SPECTRAL_H_CONTRACT_JSON = json.dumps(
    SPECTRAL_H_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

SPECTRAL_H_CONTRACT_SHA256 = hashlib.sha256(
    SPECTRAL_H_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 8. FINAL FLAGS
# ============================================================

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

FULL_H_BUILT = False
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
SPECTRAL_PARTITIONS_FROZEN = False

OPTIMIZER_CREATED = False
IIC_TRAINING_STARTED = False
OFFICIAL_DRIVE_ARTIFACTS_MODIFIED = False

s0_gate = all([
    artifact_identity_gate,
    handoff_gate,
    primary_payload_gate,
    legacy_payload_gate,
    ensemble_restore_gate,
    primary_probability_gate,
    legacy_probability_gate,
    h_probe_gate,
    primary_legacy_probe_distinct,
    classifier_immutability_gate,
    LABELS_READ is False,
    GRAVITY_SPY_LABEL_VALUES_ACCESSED is False,
    FULL_H_BUILT is False,
    AFFINITY_BUILT is False,
    SPECTRAL_STARTED is False,
    SPECTRAL_PARTITIONS_FROZEN is False,
    OPTIMIZER_CREATED is False,
    IIC_TRAINING_STARTED is False,
    OFFICIAL_DRIVE_ARTIFACTS_MODIFIED is False,
])

assert s0_gate

NOTEBOOK_06_S0_H_PREFLIGHT_PASS = True
READY_FOR_S1_FULL_H_BUILD = True
READY_FOR_AFFINITY = False
READY_FOR_LABEL_EVALUATION = False


# ============================================================
# 9. REPORT
# ============================================================

print()
print(
    "=== NOTEBOOK 06 — S0 H-CONSTRUCTION PREFLIGHT ==="
)
print(
    "PRIMARY_PAYLOAD_GATE =",
    primary_payload_gate,
)
print(
    "LEGACY_PAYLOAD_GATE =",
    legacy_payload_gate,
)
print(
    "ENSEMBLE_RESTORE_GATE =",
    ensemble_restore_gate,
)
print(
    "PRIMARY_PROBABILITY_GATE =",
    primary_probability_gate,
)
print(
    "LEGACY_PROBABILITY_GATE =",
    legacy_probability_gate,
)
print(
    "H_PROBE_SHAPE =",
    tuple(
        primary_h_probe.shape
    ),
)
print(
    "H_BLOCKS = 5 x 36"
)
print(
    "H_ROW_SUM_EXPECTED = 5.0"
)
print(
    "H_PROBE_GATE =",
    h_probe_gate,
)
print(
    "PRIMARY_LEGACY_PROBE_DISTINCT =",
    primary_legacy_probe_distinct,
)
print(
    "CLASSIFIER_IMMUTABILITY_GATE =",
    classifier_immutability_gate,
)
print(
    "SPECTRAL_H_CONTRACT_SHA256 =",
    SPECTRAL_H_CONTRACT_SHA256,
)
print(
    "LABELS_READ =",
    LABELS_READ,
)
print(
    "AFFINITY_BUILT =",
    AFFINITY_BUILT,
)
print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)
print(
    "NOTEBOOK_06_S0_H_PREFLIGHT_PASS =",
    NOTEBOOK_06_S0_H_PREFLIGHT_PASS,
)
print(
    "READY_FOR_S1_FULL_H_BUILD =",
    READY_FOR_S1_FULL_H_BUILD,
)
print(
    "READY_FOR_AFFINITY =",
    READY_FOR_AFFINITY,
)
print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


# ============================================================
# 10. CLEAN PROBE ONLY
# KEEP THE TWO RESTORED ENSEMBLES FOR S1
# ============================================================

del spectral_probe_cpu
del spectral_probe_gpu
del primary_probe_blocks
del legacy_probe_blocks
del primary_h_probe
del legacy_h_probe

gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()


=== NOTEBOOK 06 — S0 H-CONSTRUCTION PREFLIGHT ===
PRIMARY_PAYLOAD_GATE = True
LEGACY_PAYLOAD_GATE = True
ENSEMBLE_RESTORE_GATE = True
PRIMARY_PROBABILITY_GATE = True
LEGACY_PROBABILITY_GATE = True
H_PROBE_SHAPE = (8, 180)
H_BLOCKS = 5 x 36
H_ROW_SUM_EXPECTED = 5.0
H_PROBE_GATE = True
PRIMARY_LEGACY_PROBE_DISTINCT = True
CLASSIFIER_IMMUTABILITY_GATE = True
SPECTRAL_H_CONTRACT_SHA256 = 4aa38fd5c51e0fd1f6b91f299c99dfc16ebe0b8c4d7146484f6ea291ebb6c9e5
LABELS_READ = False
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
NOTEBOOK_06_S0_H_PREFLIGHT_PASS = True
READY_FOR_S1_FULL_H_BUILD = True
READY_FOR_AFFINITY = False
READY_FOR_LABEL_EVALUATION = False


In [28]:
# ============================================================
# NOTEBOOK 06 — S1
# FULL H BUILD — PRIMARY + LEGACY
#
# PAPER:
#   H dimension = (D, K*C)
#   D=8583, K=5, C=36 -> H=(8583,180)
#
# PROJECT already frozen in S0:
#   central mu, main-head SoftMax, seeds 123..127,
#   eval/inference_mode, float32, manifest order.
#
# LABEL-BLIND.
# NO affinity.
# NO spectral clustering.
# ============================================================

from pathlib import Path
import hashlib
import json
import os
import gc

import numpy as np
import torch


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S0_H_PREFLIGHT_PASS"
) is True

assert globals().get(
    "READY_FOR_S1_FULL_H_BUILD"
) is True

assert (
    SPECTRAL_H_CONTRACT_SHA256
    == "4aa38fd5c51e0fd1f6b91f299c99dfc1"
       "6ebe0b8c4d7146484f6ea291ebb6c9e5"
)

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SPECTRAL_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
    / "spectral"
)

SPECTRAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PRIMARY_H_PATH = (
    SPECTRAL_ROOT
    / "sakai224_full_iic_symmetric_v1_H.npy"
)

LEGACY_H_PATH = (
    SPECTRAL_ROOT
    / "sakai224_full_iic_legacy_text_v1_H.npy"
)

H_CONTRACT_PATH = (
    SPECTRAL_ROOT
    / "H_construction_contract.json"
)

H_METADATA_PATH = (
    SPECTRAL_ROOT
    / "H_build_metadata.json"
)


# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def array_sha256_f32(array):
    array = np.ascontiguousarray(
        array,
        dtype=np.float32,
    )

    return hashlib.sha256(
        array.tobytes(order="C")
    ).hexdigest()


def atomic_save_npy(
    path,
    array,
):
    partial = Path(
        str(path)
        + ".partial"
    )

    assert not partial.exists()

    with partial.open("wb") as handle:
        np.save(
            handle,
            array,
            allow_pickle=False,
        )
        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        partial,
        path,
    )


def build_full_h(
    models,
    *,
    batch_size=512,
):
    D = 8583
    K = 5
    C = 36

    h = np.empty(
        (D, K * C),
        dtype=np.float32,
    )

    for start in range(
        0,
        D,
        batch_size,
    ):
        end = min(
            start + batch_size,
            D,
        )

        batch_cpu = torch.from_numpy(
            np.asarray(
                full_central_mu[
                    start:end
                ],
                dtype=np.float32,
            ).copy()
        )

        batch_gpu = batch_cpu.to(
            runtime_device
        )

        with torch.inference_mode():
            blocks = [
                model.main_head(
                    batch_gpu
                ).detach().cpu().numpy()
                for model in models
            ]

        h[
            start:end
        ] = np.concatenate(
            blocks,
            axis=1,
        )

        del batch_cpu
        del batch_gpu
        del blocks

    return h


def validate_full_h(h):
    if (
        h.shape != (8583, 180)
        or h.dtype != np.float32
        or not np.isfinite(h).all()
    ):
        return False

    blocks = h.reshape(
        8583,
        5,
        36,
    )

    if (
        float(h.min()) < 0.0
        or float(h.max()) > 1.0
    ):
        return False

    return bool(
        np.allclose(
            blocks.sum(axis=2),
            1.0,
            rtol=0.0,
            atol=2.0e-6,
        )
        and np.allclose(
            h.sum(axis=1),
            5.0,
            rtol=0.0,
            atol=1.0e-5,
        )
    )


def build_or_reuse_h(
    path,
    models,
):
    if path.exists():
        h = np.load(
            path,
            allow_pickle=False,
        )

        reused = True

    else:
        h = build_full_h(
            models
        )

        assert validate_full_h(
            h
        )

        atomic_save_npy(
            path,
            h,
        )

        reused = False

    assert validate_full_h(
        h
    )

    return h, reused


# ------------------------------------------------------------
# 3. Classifier state before H
# ------------------------------------------------------------

primary_hashes_before = [
    canonical_tree_sha256(
        model.state_dict()
    )
    for model
    in spectral_primary_models
]

legacy_hashes_before = [
    canonical_tree_sha256(
        model.state_dict()
    )
    for model
    in spectral_legacy_models
]

assert (
    primary_hashes_before
    == spectral_primary_model_hashes
)

assert (
    legacy_hashes_before
    == spectral_legacy_model_hashes
)


# ------------------------------------------------------------
# 4. Build / recover full H
# ------------------------------------------------------------

print(
    "Building/recovering primary H..."
)

primary_H, primary_reused = (
    build_or_reuse_h(
        PRIMARY_H_PATH,
        spectral_primary_models,
    )
)

print(
    "Building/recovering legacy H..."
)

legacy_H, legacy_reused = (
    build_or_reuse_h(
        LEGACY_H_PATH,
        spectral_legacy_models,
    )
)


# ------------------------------------------------------------
# 5. Full scientific gates
# ------------------------------------------------------------

primary_h_gate = validate_full_h(
    primary_H
)

legacy_h_gate = validate_full_h(
    legacy_H
)

primary_legacy_distinct_gate = (
    not np.array_equal(
        primary_H,
        legacy_H,
    )
)

assert primary_h_gate
assert legacy_h_gate
assert primary_legacy_distinct_gate


PRIMARY_H_ARRAY_SHA256 = (
    array_sha256_f32(
        primary_H
    )
)

LEGACY_H_ARRAY_SHA256 = (
    array_sha256_f32(
        legacy_H
    )
)

PRIMARY_H_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_H_PATH
    )
)

LEGACY_H_FILE_SHA256 = (
    sha256_stream(
        LEGACY_H_PATH
    )
)


# ------------------------------------------------------------
# 6. Re-load persistence audit
# ------------------------------------------------------------

primary_reload = np.load(
    PRIMARY_H_PATH,
    allow_pickle=False,
)

legacy_reload = np.load(
    LEGACY_H_PATH,
    allow_pickle=False,
)

persistence_gate = (
    np.array_equal(
        primary_H,
        primary_reload,
    )
    and np.array_equal(
        legacy_H,
        legacy_reload,
    )
    and array_sha256_f32(
        primary_reload
    ) == PRIMARY_H_ARRAY_SHA256
    and array_sha256_f32(
        legacy_reload
    ) == LEGACY_H_ARRAY_SHA256
)

assert persistence_gate

del primary_reload
del legacy_reload


# ------------------------------------------------------------
# 7. Classifier immutability
# ------------------------------------------------------------

classifier_immutability_gate = (
    [
        canonical_tree_sha256(
            model.state_dict()
        )
        for model
        in spectral_primary_models
    ] == primary_hashes_before

    and

    [
        canonical_tree_sha256(
            model.state_dict()
        )
        for model
        in spectral_legacy_models
    ] == legacy_hashes_before
)

assert classifier_immutability_gate


# ------------------------------------------------------------
# 8. Persist frozen H contract
# ------------------------------------------------------------

if H_CONTRACT_PATH.exists():
    assert (
        sha256_stream(
            H_CONTRACT_PATH
        )
        == SPECTRAL_H_CONTRACT_SHA256
    )

else:
    atomic_write_text(
        H_CONTRACT_PATH,
        SPECTRAL_H_CONTRACT_JSON,
    )

    assert (
        sha256_stream(
            H_CONTRACT_PATH
        )
        == SPECTRAL_H_CONTRACT_SHA256
    )


# ------------------------------------------------------------
# 9. Metadata
# ------------------------------------------------------------

H_BUILD_METADATA = {
    "format_version":
        1,

    "status":
        "FULL_H_BUILT_LABEL_BLIND",

    "paper_contract": {
        "D": 8583,
        "K": 5,
        "C": 36,
        "H_shape": [
            8583,
            180,
        ],
    },

    "project_contract_sha256":
        SPECTRAL_H_CONTRACT_SHA256,

    "row_order":
        "manifest_index_0_to_8582",

    "classifier_block_order":
        [123, 124, 125, 126, 127],

    "classifier_output":
        "main_head_softmax_probabilities",

    "dtype":
        "float32",

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "H_array_sha256":
            PRIMARY_H_ARRAY_SHA256,

        "H_file_sha256":
            PRIMARY_H_FILE_SHA256,

        "source_final_sha256":
            EXPECTED_PRIMARY_FINAL_SHA256,
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "H_array_sha256":
            LEGACY_H_ARRAY_SHA256,

        "H_file_sha256":
            LEGACY_H_FILE_SHA256,

        "source_final_sha256":
            EXPECTED_LEGACY_FINAL_SHA256,
    },

    "labels_used":
        False,

    "affinity_built":
        False,

    "spectral_started":
        False,
}


H_BUILD_METADATA_JSON = json.dumps(
    H_BUILD_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


if H_METADATA_PATH.exists():
    with H_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        existing_metadata = json.load(
            handle
        )

    assert (
        existing_metadata
        == H_BUILD_METADATA
    )

else:
    atomic_write_text(
        H_METADATA_PATH,
        H_BUILD_METADATA_JSON,
    )


H_BUILD_METADATA_SHA256 = (
    sha256_stream(
        H_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 10. Final state
# ------------------------------------------------------------

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

FULL_H_BUILT = True
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S1_FULL_H_BUILD_PASS = True
READY_FOR_S2_AFFINITY_PREFLIGHT = True
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S1 FULL H BUILD ==="
)

print(
    "PRIMARY_H_SHAPE =",
    primary_H.shape,
)

print(
    "LEGACY_H_SHAPE =",
    legacy_H.shape,
)

print(
    "PRIMARY_H_REUSED =",
    primary_reused,
)

print(
    "LEGACY_H_REUSED =",
    legacy_reused,
)

print(
    "PRIMARY_H_GATE =",
    primary_h_gate,
)

print(
    "LEGACY_H_GATE =",
    legacy_h_gate,
)

print(
    "PRIMARY_LEGACY_DISTINCT =",
    primary_legacy_distinct_gate,
)

print(
    "PRIMARY_H_ARRAY_SHA256 =",
    PRIMARY_H_ARRAY_SHA256,
)

print(
    "LEGACY_H_ARRAY_SHA256 =",
    LEGACY_H_ARRAY_SHA256,
)

print(
    "PRIMARY_H_FILE_SHA256 =",
    PRIMARY_H_FILE_SHA256,
)

print(
    "LEGACY_H_FILE_SHA256 =",
    LEGACY_H_FILE_SHA256,
)

print(
    "PERSISTENCE_GATE =",
    persistence_gate,
)

print(
    "CLASSIFIER_IMMUTABILITY_GATE =",
    classifier_immutability_gate,
)

print(
    "H_BUILD_METADATA_SHA256 =",
    H_BUILD_METADATA_SHA256,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "AFFINITY_BUILT =",
    AFFINITY_BUILT,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "NOTEBOOK_06_S1_FULL_H_BUILD_PASS =",
    NOTEBOOK_06_S1_FULL_H_BUILD_PASS,
)

print(
    "READY_FOR_S2_AFFINITY_PREFLIGHT =",
    READY_FOR_S2_AFFINITY_PREFLIGHT,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


gc.collect()

if runtime_device.type == "cuda":
    torch.cuda.empty_cache()

Building/recovering primary H...
Building/recovering legacy H...

=== NOTEBOOK 06 — S1 FULL H BUILD ===
PRIMARY_H_SHAPE = (8583, 180)
LEGACY_H_SHAPE = (8583, 180)
PRIMARY_H_REUSED = False
LEGACY_H_REUSED = False
PRIMARY_H_GATE = True
LEGACY_H_GATE = True
PRIMARY_LEGACY_DISTINCT = True
PRIMARY_H_ARRAY_SHA256 = f8e49254a5dc0f3fa89bba845c66a71bb84cd45297c539770e33f0556738a657
LEGACY_H_ARRAY_SHA256 = d8670367a19176d9b977f4d6f222f0545c2215a65546ed162dd9125277e3786d
PRIMARY_H_FILE_SHA256 = 1f42ed4b9f8ec29886a7a4c4932f1de741d7b1f2d76f3d0f6c713657060f1638
LEGACY_H_FILE_SHA256 = edadd83506b0c850779b4e8af779e25f737289ea424478e50dfdb947168f1322
PERSISTENCE_GATE = True
CLASSIFIER_IMMUTABILITY_GATE = True
H_BUILD_METADATA_SHA256 = dcb074dec9848b9578dc6aa1c08e650238ab016b22622ab122bcceca6f424946
LABELS_READ = False
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
NOTEBOOK_06_S1_FULL_H_BUILD_PASS = True
READY_FOR_S2_AFFINITY_PREFLIGHT = True
READY_FOR_LABEL_EVALUATION = False


In [29]:
# ============================================================
# NOTEBOOK 06 — S2
# AFFINITY FORMULA PREFLIGHT — LABEL BLIND
#
# PAPER (snapshot HTML):
#   A_ij = exp(-||h^(i) - h^(j)||_2)
#
# PROJECT frozen here:
#   - full dense pairwise affinity
#   - no squared distance
#   - no gamma / bandwidth
#   - diagonal retained at 1
#   - reference computation in float64
#   - persisted affinity dtype target = float64
#
# NO full affinity build yet.
# NO spectral clustering.
# NO labels.
# NO Drive writes.
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S1_FULL_H_BUILD_PASS"
) is True

assert globals().get(
    "READY_FOR_S2_AFFINITY_PREFLIGHT"
) is True

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ------------------------------------------------------------
# 1. Frozen H identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_H_ARRAY_SHA256 = (
    "f8e49254a5dc0f3fa89bba845c66a71b"
    "b84cd45297c539770e33f0556738a657"
)

EXPECTED_LEGACY_H_ARRAY_SHA256 = (
    "d8670367a19176d9b977f4d6f222f054"
    "5c2215a65546ed162dd9125277e3786d"
)

EXPECTED_PRIMARY_H_FILE_SHA256 = (
    "1f42ed4b9f8ec29886a7a4c4932f1de7"
    "41d7b1f2d76f3d0f6c713657060f1638"
)

EXPECTED_LEGACY_H_FILE_SHA256 = (
    "edadd83506b0c850779b4e8af779e25f7"
    "37289ea424478e50dfdb947168f1322"
)

EXPECTED_H_METADATA_SHA256 = (
    "dcb074dec9848b9578dc6aa1c08e65023"
    "8ab016b22622ab122bcceca6f424946"
)

assert PRIMARY_H_ARRAY_SHA256 == EXPECTED_PRIMARY_H_ARRAY_SHA256
assert LEGACY_H_ARRAY_SHA256 == EXPECTED_LEGACY_H_ARRAY_SHA256

assert sha256_stream(
    PRIMARY_H_PATH
) == EXPECTED_PRIMARY_H_FILE_SHA256

assert sha256_stream(
    LEGACY_H_PATH
) == EXPECTED_LEGACY_H_FILE_SHA256

assert sha256_stream(
    H_METADATA_PATH
) == EXPECTED_H_METADATA_SHA256


# ------------------------------------------------------------
# 2. Reload canonical H
# ------------------------------------------------------------

s2_primary_H = np.load(
    PRIMARY_H_PATH,
    allow_pickle=False,
)

s2_legacy_H = np.load(
    LEGACY_H_PATH,
    allow_pickle=False,
)

assert s2_primary_H.shape == (8583, 180)
assert s2_legacy_H.shape == (8583, 180)

assert s2_primary_H.dtype == np.float32
assert s2_legacy_H.dtype == np.float32

assert array_sha256_f32(
    s2_primary_H
) == EXPECTED_PRIMARY_H_ARRAY_SHA256

assert array_sha256_f32(
    s2_legacy_H
) == EXPECTED_LEGACY_H_ARRAY_SHA256


# ------------------------------------------------------------
# 3. Exact PAPER affinity reference
# ------------------------------------------------------------

def paper_affinity_reference(
    h_rows,
):
    x = np.asarray(
        h_rows,
        dtype=np.float64,
    )

    diff = (
        x[:, None, :]
        - x[None, :, :]
    )

    distances = np.linalg.norm(
        diff,
        axis=2,
    )

    affinity = np.exp(
        -distances
    )

    return distances, affinity


# ------------------------------------------------------------
# 4. Fixed label-blind probe
# ------------------------------------------------------------

S2_PROBE_INDICES = [
    0,
    1,
    2,
    2145,
    4291,
    6864,
    7663,
    8582,
]

primary_probe_H = s2_primary_H[
    S2_PROBE_INDICES
]

legacy_probe_H = s2_legacy_H[
    S2_PROBE_INDICES
]

(
    primary_probe_distances,
    primary_probe_affinity,
) = paper_affinity_reference(
    primary_probe_H
)

(
    legacy_probe_distances,
    legacy_probe_affinity,
) = paper_affinity_reference(
    legacy_probe_H
)


# ------------------------------------------------------------
# 5. Mathematical gates
# ------------------------------------------------------------

def affinity_probe_valid(
    distances,
    affinity,
):
    return bool(
        distances.shape == (8, 8)
        and affinity.shape == (8, 8)

        and distances.dtype == np.float64
        and affinity.dtype == np.float64

        and np.isfinite(distances).all()
        and np.isfinite(affinity).all()

        and np.all(
            distances >= 0.0
        )

        and np.all(
            affinity > 0.0
        )

        and np.all(
            affinity <= 1.0
        )

        and np.allclose(
            distances,
            distances.T,
            rtol=0.0,
            atol=1.0e-14,
        )

        and np.allclose(
            affinity,
            affinity.T,
            rtol=0.0,
            atol=1.0e-14,
        )

        and np.array_equal(
            np.diag(distances),
            np.zeros(
                8,
                dtype=np.float64,
            ),
        )

        and np.array_equal(
            np.diag(affinity),
            np.ones(
                8,
                dtype=np.float64,
            ),
        )

        and np.any(
            affinity[
                ~np.eye(
                    8,
                    dtype=bool,
                )
            ] < 1.0
        )
    )


primary_affinity_probe_gate = (
    affinity_probe_valid(
        primary_probe_distances,
        primary_probe_affinity,
    )
)

legacy_affinity_probe_gate = (
    affinity_probe_valid(
        legacy_probe_distances,
        legacy_probe_affinity,
    )
)

assert primary_affinity_probe_gate
assert legacy_affinity_probe_gate


# ------------------------------------------------------------
# 6. Direct pair known-answer
# ------------------------------------------------------------

manual_primary_distance_0_1 = float(
    np.linalg.norm(
        primary_probe_H[0].astype(
            np.float64
        )
        - primary_probe_H[1].astype(
            np.float64
        )
    )
)

manual_primary_affinity_0_1 = float(
    np.exp(
        -manual_primary_distance_0_1
    )
)

pair_formula_gate = (
    primary_probe_distances[0, 1]
        == manual_primary_distance_0_1
    and primary_probe_affinity[0, 1]
        == manual_primary_affinity_0_1
)

assert pair_formula_gate


# ------------------------------------------------------------
# 7. Guard against wrong squared-distance kernel
# ------------------------------------------------------------

primary_wrong_squared_affinity = np.exp(
    -np.square(
        primary_probe_distances
    )
)

legacy_wrong_squared_affinity = np.exp(
    -np.square(
        legacy_probe_distances
    )
)

no_squared_distance_gate = (
    not np.allclose(
        primary_probe_affinity,
        primary_wrong_squared_affinity,
        rtol=0.0,
        atol=1.0e-12,
    )
    and not np.allclose(
        legacy_probe_affinity,
        legacy_wrong_squared_affinity,
        rtol=0.0,
        atol=1.0e-12,
    )
)

assert no_squared_distance_gate


# ------------------------------------------------------------
# 8. Primary and sensitivity remain distinct
# ------------------------------------------------------------

primary_legacy_affinity_distinct_gate = (
    not np.array_equal(
        primary_probe_affinity,
        legacy_probe_affinity,
    )
)

assert primary_legacy_affinity_distinct_gate


# ------------------------------------------------------------
# 9. Freeze affinity-construction contract
# ------------------------------------------------------------

SPECTRAL_AFFINITY_CONTRACT = {
    "contract_version": 1,

    "source_authority":
        "Scientific_Reports_version_of_record_HTML",

    "paper": {
        "input":
            "rows_h_i_of_H",

        "formula":
            "A_ij=exp(-||h_i-h_j||_2)",

        "distance":
            "euclidean_L2_norm",

        "distance_squared":
            False,

        "gamma_or_bandwidth":
            None,

        "D":
            8583,

        "matrix_shape":
            [8583, 8583],

        "diagonal_from_formula":
            1.0,
    },

    "project": {
        "pairing":
            "all_pairs_dense",

        "knn_sparsification":
            False,

        "thresholding":
            False,

        "diagonal_override_to_zero":
            False,

        "reference_compute_dtype":
            "float64",

        "persisted_affinity_dtype":
            "float64",

        "row_order":
            "manifest_index_0_to_8582",

        "primary_H_array_sha256":
            PRIMARY_H_ARRAY_SHA256,

        "legacy_H_array_sha256":
            LEGACY_H_ARRAY_SHA256,

        "labels_used":
            False,
    },

    "not_frozen_here": [
        "spectral_laplacian_variant",
        "eigensolver",
        "number_of_eigenvectors",
        "eigenvector_normalization",
        "final_assignment_method",
        "final_assignment_seed",
        "final_assignment_n_init",
    ],
}

SPECTRAL_AFFINITY_CONTRACT_JSON = json.dumps(
    SPECTRAL_AFFINITY_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

SPECTRAL_AFFINITY_CONTRACT_SHA256 = hashlib.sha256(
    SPECTRAL_AFFINITY_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# ------------------------------------------------------------
# 10. Memory estimate
# ------------------------------------------------------------

AFFINITY_D = 8583

AFFINITY_FLOAT64_BYTES = (
    AFFINITY_D
    * AFFINITY_D
    * np.dtype(
        np.float64
    ).itemsize
)

AFFINITY_FLOAT64_MIB = (
    AFFINITY_FLOAT64_BYTES
    / (1024 ** 2)
)


# ------------------------------------------------------------
# 11. Final state
# ------------------------------------------------------------

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

AFFINITY_PROBE_BUILT = True
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S2_AFFINITY_PREFLIGHT_PASS = True
READY_FOR_S3_FULL_AFFINITY_BUILD = True
READY_FOR_SPECTRAL_SOLVER_PREFLIGHT = False
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S2 AFFINITY PREFLIGHT ==="
)

print(
    "PRIMARY_AFFINITY_PROBE_GATE =",
    primary_affinity_probe_gate,
)

print(
    "LEGACY_AFFINITY_PROBE_GATE =",
    legacy_affinity_probe_gate,
)

print(
    "PAIR_FORMULA_GATE =",
    pair_formula_gate,
)

print(
    "NO_SQUARED_DISTANCE_GATE =",
    no_squared_distance_gate,
)

print(
    "PRIMARY_LEGACY_AFFINITY_DISTINCT =",
    primary_legacy_affinity_distinct_gate,
)

print(
    "PRIMARY_DISTANCE_0_1 =",
    manual_primary_distance_0_1,
)

print(
    "PRIMARY_AFFINITY_0_1 =",
    manual_primary_affinity_0_1,
)

print(
    "AFFINITY_SHAPE_TARGET =",
    (8583, 8583),
)

print(
    "AFFINITY_FLOAT64_MIB_PER_RUN =",
    AFFINITY_FLOAT64_MIB,
)

print(
    "SPECTRAL_AFFINITY_CONTRACT_SHA256 =",
    SPECTRAL_AFFINITY_CONTRACT_SHA256,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "AFFINITY_BUILT =",
    AFFINITY_BUILT,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "NOTEBOOK_06_S2_AFFINITY_PREFLIGHT_PASS =",
    NOTEBOOK_06_S2_AFFINITY_PREFLIGHT_PASS,
)

print(
    "READY_FOR_S3_FULL_AFFINITY_BUILD =",
    READY_FOR_S3_FULL_AFFINITY_BUILD,
)

print(
    "READY_FOR_SPECTRAL_SOLVER_PREFLIGHT =",
    READY_FOR_SPECTRAL_SOLVER_PREFLIGHT,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


=== NOTEBOOK 06 — S2 AFFINITY PREFLIGHT ===
PRIMARY_AFFINITY_PROBE_GATE = True
LEGACY_AFFINITY_PROBE_GATE = True
PAIR_FORMULA_GATE = True
NO_SQUARED_DISTANCE_GATE = True
PRIMARY_LEGACY_AFFINITY_DISTINCT = True
PRIMARY_DISTANCE_0_1 = 1.9110080579927884
PRIMARY_AFFINITY_0_1 = 0.14793118819113607
AFFINITY_SHAPE_TARGET = (8583, 8583)
AFFINITY_FLOAT64_MIB_PER_RUN = 562.041389465332
SPECTRAL_AFFINITY_CONTRACT_SHA256 = 85cc4a3defe14abb3290057d176567ec0940a98574a0b02bf1ec2a5bbfd425ce
LABELS_READ = False
AFFINITY_BUILT = False
SPECTRAL_STARTED = False
NOTEBOOK_06_S2_AFFINITY_PREFLIGHT_PASS = True
READY_FOR_S3_FULL_AFFINITY_BUILD = True
READY_FOR_SPECTRAL_SOLVER_PREFLIGHT = False
READY_FOR_LABEL_EVALUATION = False


In [30]:
# ============================================================
# NOTEBOOK 06 — S3
# FULL AFFINITY BUILD — PRIMARY + LEGACY
#
# PAPER:
#   A_ij = exp(-||h_i - h_j||_2)
#
# PROJECT already frozen in S2:
#   dense all-pairs matrix, float64,
#   diagonal retained at 1,
#   no gamma, no squared distance.
#
# RESUMABLE blockwise build.
# NO labels.
# NO spectral eigensolver yet.
# ============================================================

from pathlib import Path
import hashlib
import json
import os
import gc

import numpy as np
from scipy.spatial.distance import cdist


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S2_AFFINITY_PREFLIGHT_PASS"
) is True

assert globals().get(
    "READY_FOR_S3_FULL_AFFINITY_BUILD"
) is True

assert (
    SPECTRAL_AFFINITY_CONTRACT_SHA256
    == "85cc4a3defe14abb3290057d176567ec"
       "0940a98574a0b02bf1ec2a5bbfd425ce"
)

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ------------------------------------------------------------
# 1. Frozen H identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_H_ARRAY_SHA256 = (
    "f8e49254a5dc0f3fa89bba845c66a71b"
    "b84cd45297c539770e33f0556738a657"
)

EXPECTED_LEGACY_H_ARRAY_SHA256 = (
    "d8670367a19176d9b977f4d6f222f054"
    "5c2215a65546ed162dd9125277e3786d"
)

assert (
    PRIMARY_H_ARRAY_SHA256
    == EXPECTED_PRIMARY_H_ARRAY_SHA256
)

assert (
    LEGACY_H_ARRAY_SHA256
    == EXPECTED_LEGACY_H_ARRAY_SHA256
)


# ------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------

AFFINITY_ROOT = (
    SPECTRAL_ROOT
    / "affinity"
)

AFFINITY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PRIMARY_A_PATH = (
    AFFINITY_ROOT
    / "sakai224_full_iic_symmetric_v1_affinity.npy"
)

LEGACY_A_PATH = (
    AFFINITY_ROOT
    / "sakai224_full_iic_legacy_text_v1_affinity.npy"
)

PRIMARY_PROGRESS_PATH = (
    AFFINITY_ROOT
    / "sakai224_full_iic_symmetric_v1_affinity_progress.json"
)

LEGACY_PROGRESS_PATH = (
    AFFINITY_ROOT
    / "sakai224_full_iic_legacy_text_v1_affinity_progress.json"
)

AFFINITY_CONTRACT_PATH = (
    AFFINITY_ROOT
    / "affinity_construction_contract.json"
)

AFFINITY_METADATA_PATH = (
    AFFINITY_ROOT
    / "affinity_build_metadata.json"
)


# ------------------------------------------------------------
# 3. Persist frozen S2 contract
# ------------------------------------------------------------

if AFFINITY_CONTRACT_PATH.exists():

    assert (
        sha256_stream(
            AFFINITY_CONTRACT_PATH
        )
        == SPECTRAL_AFFINITY_CONTRACT_SHA256
    )

else:

    atomic_write_text(
        AFFINITY_CONTRACT_PATH,
        SPECTRAL_AFFINITY_CONTRACT_JSON,
    )

    assert (
        sha256_stream(
            AFFINITY_CONTRACT_PATH
        )
        == SPECTRAL_AFFINITY_CONTRACT_SHA256
    )


# ------------------------------------------------------------
# 4. Array hash helper
# ------------------------------------------------------------

def affinity_array_sha256(
    path,
    block_rows=256,
):

    array = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    assert array.shape == (8583, 8583)
    assert array.dtype == np.float64

    digest = hashlib.sha256()

    for start in range(
        0,
        8583,
        block_rows,
    ):

        end = min(
            start + block_rows,
            8583,
        )

        block = np.ascontiguousarray(
            array[start:end, :]
        )

        digest.update(
            block.tobytes(
                order="C"
            )
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# 5. Full persistence validator
# ------------------------------------------------------------

def validate_affinity_file(
    path,
    H_path,
    block_rows=512,
):

    A = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    H = np.load(
        H_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    if (
        A.shape != (8583, 8583)
        or A.dtype != np.float64
        or H.shape != (8583, 180)
    ):
        return False

    # Complete blockwise validation.
    for start in range(
        0,
        8583,
        block_rows,
    ):

        end = min(
            start + block_rows,
            8583,
        )

        rows = np.asarray(
            A[start:end, :]
        )

        cols_t = np.asarray(
            A[:, start:end]
        ).T

        if not np.isfinite(
            rows
        ).all():
            return False

        if (
            float(rows.min()) <= 0.0
            or float(rows.max()) > 1.0
        ):
            return False

        # Exact symmetry because blocks are mirrored.
        if not np.array_equal(
            rows,
            cols_t,
        ):
            return False

    if not np.array_equal(
        np.diag(A),
        np.ones(
            8583,
            dtype=np.float64,
        ),
    ):
        return False

    # Fixed PAPER-formula known-answer on S2 probe.
    idx = np.asarray(
        S2_PROBE_INDICES,
        dtype=np.int64,
    )

    H_probe = np.asarray(
        H[idx],
        dtype=np.float64,
    )

    expected = np.exp(
        -cdist(
            H_probe,
            H_probe,
            metric="euclidean",
        )
    )

    observed = np.asarray(
        A[
            np.ix_(
                idx,
                idx,
            )
        ]
    )

    if not np.array_equal(
        observed,
        expected,
    ):
        return False

    return True


# ------------------------------------------------------------
# 6. Resumable dense affinity builder
# ------------------------------------------------------------

AFFINITY_BLOCK_SIZE = 512


def build_affinity_resumable(
    *,
    run_id,
    H_path,
    H_array_sha256,
    output_path,
    progress_path,
):

    D = 8583
    block_size = AFFINITY_BLOCK_SIZE

    H = np.load(
        H_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    assert H.shape == (D, 180)
    assert H.dtype == np.float32

    partial_path = Path(
        str(output_path)
        + ".partial"
    )

    block_starts = list(
        range(
            0,
            D,
            block_size,
        )
    )

    block_count = len(
        block_starts
    )

    # Already completed.
    if output_path.exists():

        assert not partial_path.exists()

        assert validate_affinity_file(
            output_path,
            H_path,
        )

        return True

    # Fresh build.
    if not partial_path.exists():

        assert not progress_path.exists()

        A = np.lib.format.open_memmap(
            partial_path,
            mode="w+",
            dtype=np.float64,
            shape=(D, D),
        )

        progress = {
            "format_version":
                1,

            "run_id":
                run_id,

            "H_array_sha256":
                H_array_sha256,

            "affinity_contract_sha256":
                SPECTRAL_AFFINITY_CONTRACT_SHA256,

            "D":
                D,

            "block_size":
                block_size,

            "block_count":
                block_count,

            "next_i_block":
                0,

            "status":
                "IN_PROGRESS",

            "labels_used":
                False,
        }

        atomic_write_text(
            progress_path,
            json.dumps(
                progress,
                sort_keys=True,
                indent=2,
                ensure_ascii=True,
            ),
        )

        start_block = 0

    # Resume.
    else:

        assert progress_path.is_file()

        with progress_path.open(
            "r",
            encoding="utf-8",
        ) as handle:

            progress = json.load(
                handle
            )

        assert progress["run_id"] == run_id
        assert progress["H_array_sha256"] == H_array_sha256
        assert (
            progress["affinity_contract_sha256"]
            == SPECTRAL_AFFINITY_CONTRACT_SHA256
        )
        assert progress["D"] == D
        assert progress["block_size"] == block_size
        assert progress["block_count"] == block_count
        assert progress["labels_used"] is False
        assert progress["status"] == "IN_PROGRESS"

        start_block = int(
            progress[
                "next_i_block"
            ]
        )

        assert (
            0
            <= start_block
            <= block_count
        )

        A = np.lib.format.open_memmap(
            partial_path,
            mode="r+",
        )

        assert A.shape == (D, D)
        assert A.dtype == np.float64

    # Build upper blocks and mirror them exactly.
    for bi in range(
        start_block,
        block_count,
    ):

        i0 = block_starts[bi]

        i1 = min(
            i0 + block_size,
            D,
        )

        Xi = np.asarray(
            H[i0:i1],
            dtype=np.float64,
        )

        for bj in range(
            bi,
            block_count,
        ):

            j0 = block_starts[bj]

            j1 = min(
                j0 + block_size,
                D,
            )

            Xj = np.asarray(
                H[j0:j1],
                dtype=np.float64,
            )

            distances = cdist(
                Xi,
                Xj,
                metric="euclidean",
            )

            block = np.exp(
                -distances
            )

            if bi == bj:

                np.fill_diagonal(
                    block,
                    1.0,
                )

            A[
                i0:i1,
                j0:j1,
            ] = block

            if bj != bi:

                A[
                    j0:j1,
                    i0:i1,
                ] = block.T

        A.flush()

        progress[
            "next_i_block"
        ] = bi + 1

        atomic_write_text(
            progress_path,
            json.dumps(
                progress,
                sort_keys=True,
                indent=2,
                ensure_ascii=True,
            ),
        )

        print(
            run_id,
            "block",
            bi + 1,
            "/",
            block_count,
        )

    A.flush()

    del A

    gc.collect()

    # Every row-block is complete before publication.
    assert (
        progress[
            "next_i_block"
        ] == block_count
    )

    os.replace(
        partial_path,
        output_path,
    )

    progress[
        "status"
    ] = "COMPLETE"

    atomic_write_text(
        progress_path,
        json.dumps(
            progress,
            sort_keys=True,
            indent=2,
            ensure_ascii=True,
        ),
    )

    assert validate_affinity_file(
        output_path,
        H_path,
    )

    return False


# ------------------------------------------------------------
# 7. Build primary
# ------------------------------------------------------------

print()
print(
    "Building/resuming PRIMARY affinity..."
)

primary_affinity_reused = (
    build_affinity_resumable(
        run_id=
            "sakai224_full_iic_symmetric_v1",

        H_path=
            PRIMARY_H_PATH,

        H_array_sha256=
            PRIMARY_H_ARRAY_SHA256,

        output_path=
            PRIMARY_A_PATH,

        progress_path=
            PRIMARY_PROGRESS_PATH,
    )
)


# ------------------------------------------------------------
# 8. Build legacy
# ------------------------------------------------------------

print()
print(
    "Building/resuming LEGACY affinity..."
)

legacy_affinity_reused = (
    build_affinity_resumable(
        run_id=
            "sakai224_full_iic_legacy_text_v1",

        H_path=
            LEGACY_H_PATH,

        H_array_sha256=
            LEGACY_H_ARRAY_SHA256,

        output_path=
            LEGACY_A_PATH,

        progress_path=
            LEGACY_PROGRESS_PATH,
    )
)


# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

primary_affinity_gate = (
    validate_affinity_file(
        PRIMARY_A_PATH,
        PRIMARY_H_PATH,
    )
)

legacy_affinity_gate = (
    validate_affinity_file(
        LEGACY_A_PATH,
        LEGACY_H_PATH,
    )
)

assert primary_affinity_gate
assert legacy_affinity_gate


# ------------------------------------------------------------
# 10. Frozen hashes
# ------------------------------------------------------------

PRIMARY_A_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_A_PATH
    )
)

LEGACY_A_FILE_SHA256 = (
    sha256_stream(
        LEGACY_A_PATH
    )
)

PRIMARY_A_ARRAY_SHA256 = (
    affinity_array_sha256(
        PRIMARY_A_PATH
    )
)

LEGACY_A_ARRAY_SHA256 = (
    affinity_array_sha256(
        LEGACY_A_PATH
    )
)


# ------------------------------------------------------------
# 11. Distinctness
# ------------------------------------------------------------

primary_A_mm = np.load(
    PRIMARY_A_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

legacy_A_mm = np.load(
    LEGACY_A_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

AFFINITY_DISTINCT_PROBE_INDICES = [
    0,
    1,
    2145,
    4291,
    6864,
    8582,
]

idx = np.asarray(
    AFFINITY_DISTINCT_PROBE_INDICES,
    dtype=np.int64,
)

primary_affinity_probe = np.asarray(
    primary_A_mm[
        np.ix_(
            idx,
            idx,
        )
    ]
)

legacy_affinity_probe = np.asarray(
    legacy_A_mm[
        np.ix_(
            idx,
            idx,
        )
    ]
)

primary_legacy_affinity_distinct_gate = (
    not np.array_equal(
        primary_affinity_probe,
        legacy_affinity_probe,
    )
)

assert primary_legacy_affinity_distinct_gate

del primary_A_mm
del legacy_A_mm
del primary_affinity_probe
del legacy_affinity_probe


# ------------------------------------------------------------
# 12. Metadata
# ------------------------------------------------------------

AFFINITY_BUILD_METADATA = {
    "format_version":
        1,

    "status":
        "FULL_AFFINITY_BUILT_LABEL_BLIND",

    "source_authority":
        "Scientific_Reports_version_of_record_HTML",

    "formula":
        "A_ij=exp(-||h_i-h_j||_2)",

    "shape":
        [8583, 8583],

    "dtype":
        "float64",

    "build_backend":
        "scipy.spatial.distance.cdist",

    "build_mode":
        "blockwise_resumable_upper_triangle_exact_mirror",

    "block_size":
        AFFINITY_BLOCK_SIZE,

    "affinity_contract_sha256":
        SPECTRAL_AFFINITY_CONTRACT_SHA256,

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "H_array_sha256":
            PRIMARY_H_ARRAY_SHA256,

        "affinity_array_sha256":
            PRIMARY_A_ARRAY_SHA256,

        "affinity_file_sha256":
            PRIMARY_A_FILE_SHA256,
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "H_array_sha256":
            LEGACY_H_ARRAY_SHA256,

        "affinity_array_sha256":
            LEGACY_A_ARRAY_SHA256,

        "affinity_file_sha256":
            LEGACY_A_FILE_SHA256,
    },

    "labels_used":
        False,

    "spectral_solver_started":
        False,
}

AFFINITY_BUILD_METADATA_JSON = json.dumps(
    AFFINITY_BUILD_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

if AFFINITY_METADATA_PATH.exists():

    with AFFINITY_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:

        existing_metadata = json.load(
            handle
        )

    assert (
        existing_metadata
        == AFFINITY_BUILD_METADATA
    )

else:

    atomic_write_text(
        AFFINITY_METADATA_PATH,
        AFFINITY_BUILD_METADATA_JSON,
    )


AFFINITY_BUILD_METADATA_SHA256 = (
    sha256_stream(
        AFFINITY_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 13. Final state
# ------------------------------------------------------------

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

AFFINITY_BUILT = True
SPECTRAL_STARTED = False
SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S3_FULL_AFFINITY_BUILD_PASS = True
READY_FOR_S4_SPECTRAL_SOLVER_PREFLIGHT = True
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S3 FULL AFFINITY BUILD ==="
)

print(
    "PRIMARY_REUSED =",
    primary_affinity_reused,
)

print(
    "LEGACY_REUSED =",
    legacy_affinity_reused,
)

print(
    "PRIMARY_AFFINITY_GATE =",
    primary_affinity_gate,
)

print(
    "LEGACY_AFFINITY_GATE =",
    legacy_affinity_gate,
)

print(
    "PRIMARY_LEGACY_AFFINITY_DISTINCT =",
    primary_legacy_affinity_distinct_gate,
)

print(
    "PRIMARY_A_ARRAY_SHA256 =",
    PRIMARY_A_ARRAY_SHA256,
)

print(
    "LEGACY_A_ARRAY_SHA256 =",
    LEGACY_A_ARRAY_SHA256,
)

print(
    "PRIMARY_A_FILE_SHA256 =",
    PRIMARY_A_FILE_SHA256,
)

print(
    "LEGACY_A_FILE_SHA256 =",
    LEGACY_A_FILE_SHA256,
)

print(
    "AFFINITY_BUILD_METADATA_SHA256 =",
    AFFINITY_BUILD_METADATA_SHA256,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "NOTEBOOK_06_S3_FULL_AFFINITY_BUILD_PASS =",
    NOTEBOOK_06_S3_FULL_AFFINITY_BUILD_PASS,
)

print(
    "READY_FOR_S4_SPECTRAL_SOLVER_PREFLIGHT =",
    READY_FOR_S4_SPECTRAL_SOLVER_PREFLIGHT,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


Building/resuming PRIMARY affinity...
sakai224_full_iic_symmetric_v1 block 1 / 17
sakai224_full_iic_symmetric_v1 block 2 / 17
sakai224_full_iic_symmetric_v1 block 3 / 17
sakai224_full_iic_symmetric_v1 block 4 / 17
sakai224_full_iic_symmetric_v1 block 5 / 17
sakai224_full_iic_symmetric_v1 block 6 / 17
sakai224_full_iic_symmetric_v1 block 7 / 17
sakai224_full_iic_symmetric_v1 block 8 / 17
sakai224_full_iic_symmetric_v1 block 9 / 17
sakai224_full_iic_symmetric_v1 block 10 / 17
sakai224_full_iic_symmetric_v1 block 11 / 17
sakai224_full_iic_symmetric_v1 block 12 / 17
sakai224_full_iic_symmetric_v1 block 13 / 17
sakai224_full_iic_symmetric_v1 block 14 / 17
sakai224_full_iic_symmetric_v1 block 15 / 17
sakai224_full_iic_symmetric_v1 block 16 / 17
sakai224_full_iic_symmetric_v1 block 17 / 17

Building/resuming LEGACY affinity...
sakai224_full_iic_legacy_text_v1 block 1 / 17
sakai224_full_iic_legacy_text_v1 block 2 / 17
sakai224_full_iic_legacy_text_v1 block 3 / 17
sakai224_full_iic_legacy_text

In [31]:
# ============================================================
# NOTEBOOK 06 — S4
# SPECTRAL SOLVER PREFLIGHT + PROJECT CONTRACT FREEZE
#
# PAPER (HTML):
#   spectral clustering on the affinity matrix, C=36.
#
# PROJECT:
#   normalized symmetric Laplacian; self-loops ignored in the
#   Laplacian; ARPACK/eigsh; row-normalized embedding;
#   final k-means++ with fixed seed.
#
# LABEL-BLIND. No full spectral partition yet.
# ============================================================

from pathlib import Path
import hashlib
import json
import time

import numpy as np
import scipy
from scipy.sparse.csgraph import laplacian as csgraph_laplacian
from scipy.sparse.linalg import eigsh
import sklearn
from sklearn.cluster import KMeans


# 0. Parent gates
assert globals().get(
    "NOTEBOOK_06_S3_FULL_AFFINITY_BUILD_PASS"
) is True

assert globals().get(
    "READY_FOR_S4_SPECTRAL_SOLVER_PREFLIGHT"
) is True

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# 1. Frozen affinity identities
assert PRIMARY_A_ARRAY_SHA256 == (
    "9d0f2d6b3108eccc18762feb85d7d52e44e0113974a84aabe75d187d9b90d777"
)

assert LEGACY_A_ARRAY_SHA256 == (
    "93fb893b6e028b65c09430c5f3f80798d23e62b27349ada19f2069f1a2f97958"
)

assert PRIMARY_A_FILE_SHA256 == (
    "8e6d46c9ca7e51848a8d768c3f925f46b6457e148b64d50ccd36a59504669b98"
)

assert LEGACY_A_FILE_SHA256 == (
    "2f1e4d236a6287ee059ddedee6aa2873e3a65d6ec37c8a96a831bd8d23cd7d7d"
)

assert PRIMARY_A_PATH.is_file()
assert LEGACY_A_PATH.is_file()


# 2. PROJECT parameters — frozen before full spectral run
SPECTRAL_N_CLUSTERS = 36

SPECTRAL_EIGEN_WHICH = "LA"
SPECTRAL_EIGEN_TOL = 1.0e-8
SPECTRAL_EIGEN_MAXITER = 100000
SPECTRAL_EIGEN_NCV = 145
SPECTRAL_EIGEN_V0_SEED = 123

SPECTRAL_KMEANS_SEED = 123
SPECTRAL_KMEANS_N_INIT = 50
SPECTRAL_KMEANS_MAX_ITER = 1000
SPECTRAL_KMEANS_TOL = 1.0e-6
SPECTRAL_KMEANS_ALGORITHM = "lloyd"


# 3. Helpers
def normalized_adjacency(
    affinity,
):
    A = np.asarray(
        affinity,
        dtype=np.float64,
    )

    assert A.ndim == 2
    assert A.shape[0] == A.shape[1]
    assert np.isfinite(A).all()

    # PROJECT:
    # stored A keeps A_ii=1 from the PAPER formula,
    # but self-loops are ignored by graph-Laplacian normalization.
    W = A.copy()
    np.fill_diagonal(
        W,
        0.0,
    )

    degree = W.sum(
        axis=1,
        dtype=np.float64,
    )

    assert np.isfinite(
        degree
    ).all()

    assert np.all(
        degree > 0.0
    )

    inv_sqrt = (
        1.0
        / np.sqrt(
            degree
        )
    )

    M = (
        inv_sqrt[:, None]
        * W
        * inv_sqrt[None, :]
    )

    return W, degree, M


def canonicalize_signs(
    U,
):
    U = np.array(
        U,
        dtype=np.float64,
        copy=True,
    )

    for j in range(
        U.shape[1]
    ):
        pivot = int(
            np.argmax(
                np.abs(
                    U[:, j]
                )
            )
        )

        if U[
            pivot,
            j
        ] < 0.0:
            U[:, j] *= -1.0

    return U


def row_normalize(
    U,
):
    norms = np.linalg.norm(
        U,
        axis=1,
    )

    assert np.isfinite(
        norms
    ).all()

    assert np.all(
        norms > 0.0
    )

    return (
        U
        / norms[:, None]
    )


def fixed_v0(
    n,
):
    rng = np.random.default_rng(
        SPECTRAL_EIGEN_V0_SEED
    )

    v = rng.standard_normal(
        n
    ).astype(
        np.float64
    )

    return (
        v
        / np.linalg.norm(
            v
        )
    )


def solver_probe(
    affinity,
):
    _, _, M = normalized_adjacency(
        affinity
    )

    n = M.shape[0]

    # Independent dense reference.
    ref_values, ref_vectors = (
        np.linalg.eigh(
            M
        )
    )

    order = np.argsort(
        ref_values
    )[
        ::-1
    ][
        :SPECTRAL_N_CLUSTERS
    ]

    ref_values = ref_values[
        order
    ]

    ref_vectors = ref_vectors[
        :,
        order
    ]

    # Production-style solver.
    values, vectors = eigsh(
        M,
        k=SPECTRAL_N_CLUSTERS,
        which=SPECTRAL_EIGEN_WHICH,
        tol=SPECTRAL_EIGEN_TOL,
        maxiter=SPECTRAL_EIGEN_MAXITER,
        ncv=min(
            SPECTRAL_EIGEN_NCV,
            n,
        ),
        v0=fixed_v0(
            n
        ),
    )

    order = np.argsort(
        values
    )[
        ::-1
    ]

    values = values[
        order
    ]

    vectors = canonicalize_signs(
        vectors[
            :,
            order
        ]
    )

    residuals = np.linalg.norm(
        (
            M
            @ vectors
        )
        - (
            vectors
            * values[
                None,
                :
            ]
        ),
        axis=0,
    )

    ref_projector = (
        ref_vectors
        @ ref_vectors.T
    )

    solver_projector = (
        vectors
        @ vectors.T
    )

    return {
        "values":
            values,

        "vectors":
            vectors,

        "embedding":
            row_normalize(
                vectors
            ),

        "eig_error":
            float(
                np.max(
                    np.abs(
                        ref_values
                        - values
                    )
                )
            ),

        "projector_error":
            float(
                np.max(
                    np.abs(
                        ref_projector
                        - solver_projector
                    )
                )
            ),

        "residual_max":
            float(
                residuals.max()
            ),
    }


def fixed_kmeans(
    embedding,
):
    km = KMeans(
        n_clusters=
            SPECTRAL_N_CLUSTERS,

        init=
            "k-means++",

        n_init=
            SPECTRAL_KMEANS_N_INIT,

        max_iter=
            SPECTRAL_KMEANS_MAX_ITER,

        tol=
            SPECTRAL_KMEANS_TOL,

        random_state=
            SPECTRAL_KMEANS_SEED,

        algorithm=
            SPECTRAL_KMEANS_ALGORITHM,
    )

    return km.fit_predict(
        embedding
    )


# 4. Fixed label-blind 512-event probe
S4_PROBE_SIZE = 512

S4_PROBE_INDICES = np.linspace(
    0,
    8582,
    S4_PROBE_SIZE,
    dtype=np.int64,
)

assert (
    np.unique(
        S4_PROBE_INDICES
    ).size
    == S4_PROBE_SIZE
)

primary_A_mm = np.load(
    PRIMARY_A_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

legacy_A_mm = np.load(
    LEGACY_A_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

primary_probe = np.asarray(
    primary_A_mm[
        np.ix_(
            S4_PROBE_INDICES,
            S4_PROBE_INDICES,
        )
    ],
    dtype=np.float64,
)

legacy_probe = np.asarray(
    legacy_A_mm[
        np.ix_(
            S4_PROBE_INDICES,
            S4_PROBE_INDICES,
        )
    ],
    dtype=np.float64,
)

del primary_A_mm
del legacy_A_mm


# 5. Verify Laplacian semantics against SciPy reference
scipy_L = csgraph_laplacian(
    primary_probe,
    normed=True,
)

_, _, manual_M = (
    normalized_adjacency(
        primary_probe
    )
)

manual_L = (
    np.eye(
        S4_PROBE_SIZE,
        dtype=np.float64,
    )
    - manual_M
)

laplacian_semantics_gate = (
    np.allclose(
        scipy_L,
        manual_L,
        rtol=0.0,
        atol=2.0e-14,
    )
)

assert laplacian_semantics_gate

del scipy_L
del manual_M
del manual_L


# 6. Eigensolver probes
t0 = time.perf_counter()

primary_s4 = solver_probe(
    primary_probe
)

primary_seconds = (
    time.perf_counter()
    - t0
)

t0 = time.perf_counter()

legacy_s4 = solver_probe(
    legacy_probe
)

legacy_seconds = (
    time.perf_counter()
    - t0
)


def solver_gate(
    result,
):
    return bool(
        result[
            "values"
        ].shape == (36,)

        and result[
            "vectors"
        ].shape == (512, 36)

        and result[
            "embedding"
        ].shape == (512, 36)

        and np.isfinite(
            result[
                "values"
            ]
        ).all()

        and np.isfinite(
            result[
                "vectors"
            ]
        ).all()

        and np.isfinite(
            result[
                "embedding"
            ]
        ).all()

        and result[
            "eig_error"
        ] <= 1.0e-9

        and result[
            "projector_error"
        ] <= 1.0e-7

        and result[
            "residual_max"
        ] <= 1.0e-7

        and np.allclose(
            np.linalg.norm(
                result[
                    "embedding"
                ],
                axis=1,
            ),
            1.0,
            rtol=0.0,
            atol=2.0e-12,
        )
    )


primary_solver_gate = (
    solver_gate(
        primary_s4
    )
)

legacy_solver_gate = (
    solver_gate(
        legacy_s4
    )
)

assert primary_solver_gate
assert legacy_solver_gate


# 7. Deterministic final-assignment probe
primary_labels_1 = fixed_kmeans(
    primary_s4[
        "embedding"
    ]
)

primary_labels_2 = fixed_kmeans(
    primary_s4[
        "embedding"
    ]
)

legacy_labels_1 = fixed_kmeans(
    legacy_s4[
        "embedding"
    ]
)

legacy_labels_2 = fixed_kmeans(
    legacy_s4[
        "embedding"
    ]
)

assignment_reproducibility_gate = (
    np.array_equal(
        primary_labels_1,
        primary_labels_2,
    )

    and np.array_equal(
        legacy_labels_1,
        legacy_labels_2,
    )

    and np.unique(
        primary_labels_1
    ).size == 36

    and np.unique(
        legacy_labels_1
    ).size == 36
)

assert assignment_reproducibility_gate

primary_legacy_probe_partition_distinct = (
    not np.array_equal(
        primary_labels_1,
        legacy_labels_1,
    )
)

assert primary_legacy_probe_partition_distinct


# 8. Freeze PROJECT solver contract
SPECTRAL_SOLVER_CONTRACT = {
    "contract_version":
        1,

    "source_authority": {
        "paper":
            "Scientific_Reports_version_of_record_HTML",

        "paper_reference_44":
            "Von_Luxburg_2007_tutorial_on_spectral_clustering",
    },

    "paper": {
        "input":
            "affinity_matrix",

        "final_cluster_count_C":
            36,

        "instruction":
            "compute_spectral_clustering_to_affinity_matrix",
    },

    "project": {
        "variant":
            "symmetric_normalized_spectral_clustering",

        "self_affinity_in_input":
            1.0,

        "self_affinity_in_laplacian":
            "ignored_as_self_loop",

        "W":
            "A_with_zero_diagonal",

        "degree":
            "d_i=sum_j_W_ij",

        "normalized_adjacency":
            "M=D^-1/2_W_D^-1/2",

        "equivalent_laplacian":
            "L_sym=I-M",

        "eigenproblem":
            "top_36_eigenvectors_of_M",

        "eigensolver":
            "scipy.sparse.linalg.eigsh_ARPACK",

        "which":
            SPECTRAL_EIGEN_WHICH,

        "tol":
            SPECTRAL_EIGEN_TOL,

        "maxiter":
            SPECTRAL_EIGEN_MAXITER,

        "ncv":
            SPECTRAL_EIGEN_NCV,

        "v0_seed":
            SPECTRAL_EIGEN_V0_SEED,

        "eigenvector_sign_rule":
            "largest_absolute_entry_positive",

        "row_normalize_embedding":
            True,

        "assignment":
            "kmeans++",

        "kmeans_seed":
            SPECTRAL_KMEANS_SEED,

        "kmeans_n_init":
            SPECTRAL_KMEANS_N_INIT,

        "kmeans_max_iter":
            SPECTRAL_KMEANS_MAX_ITER,

        "kmeans_tol":
            SPECTRAL_KMEANS_TOL,

        "kmeans_algorithm":
            SPECTRAL_KMEANS_ALGORITHM,

        "labels_used":
            False,
    },
}

SPECTRAL_SOLVER_CONTRACT_JSON = json.dumps(
    SPECTRAL_SOLVER_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

SPECTRAL_SOLVER_CONTRACT_SHA256 = hashlib.sha256(
    SPECTRAL_SOLVER_CONTRACT_JSON.encode(
        "utf-8"
    )
).hexdigest()


# 9. Persist contract before full run
SPECTRAL_SOLVER_ROOT = (
    SPECTRAL_ROOT
    / "solver"
)

SPECTRAL_SOLVER_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SPECTRAL_SOLVER_CONTRACT_PATH = (
    SPECTRAL_SOLVER_ROOT
    / "spectral_solver_contract.json"
)

if SPECTRAL_SOLVER_CONTRACT_PATH.exists():

    assert (
        sha256_stream(
            SPECTRAL_SOLVER_CONTRACT_PATH
        )
        == SPECTRAL_SOLVER_CONTRACT_SHA256
    )

else:

    atomic_write_text(
        SPECTRAL_SOLVER_CONTRACT_PATH,
        SPECTRAL_SOLVER_CONTRACT_JSON,
    )

    assert (
        sha256_stream(
            SPECTRAL_SOLVER_CONTRACT_PATH
        )
        == SPECTRAL_SOLVER_CONTRACT_SHA256
    )


# 10. Final state
LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

SPECTRAL_SOLVER_PROBE_EXECUTED = True
SPECTRAL_STARTED = False
SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS = True
READY_FOR_S5_PRIMARY_SPECTRAL = True
READY_FOR_LABEL_EVALUATION = False


# 11. Report
print()
print(
    "=== NOTEBOOK 06 — S4 SPECTRAL SOLVER PREFLIGHT ==="
)

print(
    "SCIPY_VERSION =",
    scipy.__version__,
)

print(
    "SKLEARN_VERSION =",
    sklearn.__version__,
)

print(
    "LAPLACIAN_SEMANTICS_GATE =",
    laplacian_semantics_gate,
)

print(
    "PRIMARY_SOLVER_GATE =",
    primary_solver_gate,
)

print(
    "LEGACY_SOLVER_GATE =",
    legacy_solver_gate,
)

print(
    "PRIMARY_EIGENVALUE_MAX_ABS_ERROR =",
    primary_s4[
        "eig_error"
    ],
)

print(
    "LEGACY_EIGENVALUE_MAX_ABS_ERROR =",
    legacy_s4[
        "eig_error"
    ],
)

print(
    "PRIMARY_PROJECTOR_MAX_ABS_ERROR =",
    primary_s4[
        "projector_error"
    ],
)

print(
    "LEGACY_PROJECTOR_MAX_ABS_ERROR =",
    legacy_s4[
        "projector_error"
    ],
)

print(
    "PRIMARY_RESIDUAL_MAX =",
    primary_s4[
        "residual_max"
    ],
)

print(
    "LEGACY_RESIDUAL_MAX =",
    legacy_s4[
        "residual_max"
    ],
)

print(
    "ASSIGNMENT_REPRODUCIBILITY_GATE =",
    assignment_reproducibility_gate,
)

print(
    "PRIMARY_LEGACY_PROBE_PARTITION_DISTINCT =",
    primary_legacy_probe_partition_distinct,
)

print(
    "PRIMARY_PROBE_SECONDS =",
    primary_seconds,
)

print(
    "LEGACY_PROBE_SECONDS =",
    legacy_seconds,
)

print(
    "SPECTRAL_SOLVER_CONTRACT_SHA256 =",
    SPECTRAL_SOLVER_CONTRACT_SHA256,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "SPECTRAL_STARTED =",
    SPECTRAL_STARTED,
)

print(
    "NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS =",
    NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS,
)

print(
    "READY_FOR_S5_PRIMARY_SPECTRAL =",
    READY_FOR_S5_PRIMARY_SPECTRAL,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


=== NOTEBOOK 06 — S4 SPECTRAL SOLVER PREFLIGHT ===
SCIPY_VERSION = 1.16.3
SKLEARN_VERSION = 1.6.1
LAPLACIAN_SEMANTICS_GATE = True
PRIMARY_SOLVER_GATE = True
LEGACY_SOLVER_GATE = True
PRIMARY_EIGENVALUE_MAX_ABS_ERROR = 4.996003610813204e-16
LEGACY_EIGENVALUE_MAX_ABS_ERROR = 4.996003610813204e-16
PRIMARY_PROJECTOR_MAX_ABS_ERROR = 1.1749498943225412e-15
LEGACY_PROJECTOR_MAX_ABS_ERROR = 3.2421981766006525e-15
PRIMARY_RESIDUAL_MAX = 9.015756945640409e-16
LEGACY_RESIDUAL_MAX = 1.0026881764630252e-15
ASSIGNMENT_REPRODUCIBILITY_GATE = True
PRIMARY_LEGACY_PROBE_PARTITION_DISTINCT = True
PRIMARY_PROBE_SECONDS = 0.21593762599877664
LEGACY_PROBE_SECONDS = 0.392021654999553
SPECTRAL_SOLVER_CONTRACT_SHA256 = 3b75e9d89673280284f9bcd735424d41d89a6b9f066176a560dc930d57192a0f
LABELS_READ = False
SPECTRAL_STARTED = False
NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS = True
READY_FOR_S5_PRIMARY_SPECTRAL = True
READY_FOR_LABEL_EVALUATION = False


In [32]:
# ============================================================
# NOTEBOOK 06 — S5
# PRIMARY FULL SPECTRAL — LABEL BLIND
#
# PAPER:
#   spectral clustering on affinity, C=36.
#
# PROJECT frozen in S4:
#   symmetric normalized spectral clustering,
#   ARPACK/eigsh, 36 eigenvectors, row normalization,
#   k-means++ assignment with fixed seed.
#
# PRIMARY ONLY: symmetric_figure.
# NO labels.
# NO sensitivity spectral yet.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import os
import time

import numpy as np
from scipy.sparse.linalg import LinearOperator, eigsh
from sklearn.cluster import KMeans


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS"
) is True

assert globals().get(
    "READY_FOR_S5_PRIMARY_SPECTRAL"
) is True

assert (
    SPECTRAL_SOLVER_CONTRACT_SHA256
    == "3b75e9d89673280284f9bcd735424d41"
       "d89a6b9f066176a560dc930d57192a0f"
)

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_STARTED is False


# ------------------------------------------------------------
# 1. Frozen primary affinity
# ------------------------------------------------------------

EXPECTED_PRIMARY_A_ARRAY_SHA256 = (
    "9d0f2d6b3108eccc18762feb85d7d52e"
    "44e0113974a84aabe75d187d9b90d777"
)

EXPECTED_PRIMARY_A_FILE_SHA256 = (
    "8e6d46c9ca7e51848a8d768c3f925f4"
    "6b6457e148b64d50ccd36a59504669b98"
)

assert (
    PRIMARY_A_ARRAY_SHA256
    == EXPECTED_PRIMARY_A_ARRAY_SHA256
)

assert (
    sha256_stream(
        PRIMARY_A_PATH
    )
    == EXPECTED_PRIMARY_A_FILE_SHA256
)

assert (
    sha256_stream(
        SPECTRAL_SOLVER_CONTRACT_PATH
    )
    == SPECTRAL_SOLVER_CONTRACT_SHA256
)


# ------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------

PRIMARY_SPECTRAL_DIR = (
    SPECTRAL_ROOT
    / "partitions"
    / "sakai224_full_iic_symmetric_v1"
)

PRIMARY_SPECTRAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PRIMARY_EIGENVALUES_PATH = (
    PRIMARY_SPECTRAL_DIR
    / "eigenvalues.npy"
)

PRIMARY_EIGENVECTORS_PATH = (
    PRIMARY_SPECTRAL_DIR
    / "eigenvectors.npy"
)

PRIMARY_EMBEDDING_PATH = (
    PRIMARY_SPECTRAL_DIR
    / "spectral_embedding.npy"
)

PRIMARY_PARTITION_PATH = (
    PRIMARY_SPECTRAL_DIR
    / "partition.npy"
)

PRIMARY_SPECTRAL_METADATA_PATH = (
    PRIMARY_SPECTRAL_DIR
    / "spectral_metadata.json"
)

assert not PRIMARY_SPECTRAL_METADATA_PATH.exists()

for path in [
    PRIMARY_EIGENVALUES_PATH,
    PRIMARY_EIGENVECTORS_PATH,
    PRIMARY_EMBEDDING_PATH,
    PRIMARY_PARTITION_PATH,
]:
    assert not path.exists(), (
        f"STOP: pre-existing spectral artifact: {path}"
    )


# ------------------------------------------------------------
# 3. Atomic helpers
# ------------------------------------------------------------

def s5_atomic_save_npy(
    path,
    array,
):
    partial = Path(
        str(path)
        + ".partial"
    )

    assert not partial.exists()

    with partial.open(
        "wb"
    ) as handle:

        np.save(
            handle,
            array,
            allow_pickle=False,
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        partial,
        path,
    )


def s5_array_sha256(
    array,
):
    contiguous = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        contiguous.tobytes(
            order="C"
        )
    ).hexdigest()


# ------------------------------------------------------------
# 4. Load primary affinity
# ------------------------------------------------------------

print()
print(
    "Loading PRIMARY affinity..."
)

t0 = time.perf_counter()

primary_A = np.load(
    PRIMARY_A_PATH,
    allow_pickle=False,
)

PRIMARY_A_LOAD_SECONDS = (
    time.perf_counter()
    - t0
)

assert primary_A.shape == (
    8583,
    8583,
)

assert primary_A.dtype == np.float64
assert np.isfinite(primary_A).all()

assert np.array_equal(
    np.diag(
        primary_A
    ),
    np.ones(
        8583,
        dtype=np.float64,
    ),
)


# ------------------------------------------------------------
# 5. Normalized adjacency operator
#
# W = A - I
# M = D^-1/2 W D^-1/2
# ------------------------------------------------------------

primary_degree = (
    primary_A.sum(
        axis=1,
        dtype=np.float64,
    )
    - 1.0
)

assert np.isfinite(
    primary_degree
).all()

assert np.all(
    primary_degree > 0.0
)

primary_inv_sqrt_degree = (
    1.0
    / np.sqrt(
        primary_degree
    )
)


def primary_m_matvec(
    x,
):
    z = (
        primary_inv_sqrt_degree
        * x
    )

    y = (
        primary_A
        @ z
        - z
    )

    return (
        primary_inv_sqrt_degree
        * y
    )


def primary_m_matmat(
    X,
):
    Z = (
        primary_inv_sqrt_degree[:, None]
        * X
    )

    Y = (
        primary_A
        @ Z
        - Z
    )

    return (
        primary_inv_sqrt_degree[:, None]
        * Y
    )


PRIMARY_M_OPERATOR = LinearOperator(
    shape=(8583, 8583),
    matvec=primary_m_matvec,
    matmat=primary_m_matmat,
    dtype=np.float64,
)


# ------------------------------------------------------------
# 6. Full primary eigensolve
# ------------------------------------------------------------

print(
    "Running PRIMARY ARPACK eigensolve..."
)

SPECTRAL_STARTED = True

t0 = time.perf_counter()

primary_eigenvalues, primary_eigenvectors = eigsh(
    PRIMARY_M_OPERATOR,
    k=SPECTRAL_N_CLUSTERS,
    which=SPECTRAL_EIGEN_WHICH,
    tol=SPECTRAL_EIGEN_TOL,
    maxiter=SPECTRAL_EIGEN_MAXITER,
    ncv=SPECTRAL_EIGEN_NCV,
    v0=fixed_v0(8583),
)

PRIMARY_EIGENSOLVE_SECONDS = (
    time.perf_counter()
    - t0
)


primary_order = np.argsort(
    primary_eigenvalues
)[::-1]

primary_eigenvalues = np.asarray(
    primary_eigenvalues[
        primary_order
    ],
    dtype=np.float64,
)

primary_eigenvectors = np.asarray(
    primary_eigenvectors[
        :,
        primary_order
    ],
    dtype=np.float64,
)

primary_eigenvectors = canonicalize_signs(
    primary_eigenvectors
)


# ------------------------------------------------------------
# 7. Eigensolver audit
# ------------------------------------------------------------

primary_residuals = np.linalg.norm(
    (
        PRIMARY_M_OPERATOR
        @ primary_eigenvectors
    )
    - (
        primary_eigenvectors
        * primary_eigenvalues[
            None,
            :
        ]
    ),
    axis=0,
)

PRIMARY_RESIDUAL_MAX = float(
    primary_residuals.max()
)

PRIMARY_ORTHOGONALITY_MAX_ERROR = float(
    np.max(
        np.abs(
            (
                primary_eigenvectors.T
                @ primary_eigenvectors
            )
            - np.eye(
                SPECTRAL_N_CLUSTERS,
                dtype=np.float64,
            )
        )
    )
)

PRIMARY_TOP_EIGENVALUE = float(
    primary_eigenvalues[0]
)

primary_eigensolver_gate = (
    primary_eigenvalues.shape == (36,)
    and primary_eigenvectors.shape == (8583, 36)

    and np.isfinite(
        primary_eigenvalues
    ).all()

    and np.isfinite(
        primary_eigenvectors
    ).all()

    and np.all(
        np.diff(
            primary_eigenvalues
        ) <= 1.0e-12
    )

    and abs(
        PRIMARY_TOP_EIGENVALUE
        - 1.0
    ) <= 1.0e-8

    and PRIMARY_RESIDUAL_MAX
        <= 1.0e-6

    and PRIMARY_ORTHOGONALITY_MAX_ERROR
        <= 1.0e-8
)

assert primary_eigensolver_gate


# ------------------------------------------------------------
# 8. Row-normalized spectral embedding
# ------------------------------------------------------------

primary_spectral_embedding = (
    row_normalize(
        primary_eigenvectors
    )
)

primary_embedding_gate = (
    primary_spectral_embedding.shape
        == (8583, 36)

    and primary_spectral_embedding.dtype
        == np.float64

    and np.isfinite(
        primary_spectral_embedding
    ).all()

    and np.allclose(
        np.linalg.norm(
            primary_spectral_embedding,
            axis=1,
        ),
        1.0,
        rtol=0.0,
        atol=2.0e-12,
    )
)

assert primary_embedding_gate


# ------------------------------------------------------------
# 9. Final PROJECT k-means++ assignment
# ------------------------------------------------------------

print(
    "Running PRIMARY final k-means++ assignment..."
)

t0 = time.perf_counter()

primary_kmeans_1 = KMeans(
    n_clusters=SPECTRAL_N_CLUSTERS,
    init="k-means++",
    n_init=SPECTRAL_KMEANS_N_INIT,
    max_iter=SPECTRAL_KMEANS_MAX_ITER,
    tol=SPECTRAL_KMEANS_TOL,
    random_state=SPECTRAL_KMEANS_SEED,
    algorithm=SPECTRAL_KMEANS_ALGORITHM,
)

primary_partition_1 = (
    primary_kmeans_1.fit_predict(
        primary_spectral_embedding
    )
)

PRIMARY_KMEANS_SECONDS = (
    time.perf_counter()
    - t0
)


primary_kmeans_2 = KMeans(
    n_clusters=SPECTRAL_N_CLUSTERS,
    init="k-means++",
    n_init=SPECTRAL_KMEANS_N_INIT,
    max_iter=SPECTRAL_KMEANS_MAX_ITER,
    tol=SPECTRAL_KMEANS_TOL,
    random_state=SPECTRAL_KMEANS_SEED,
    algorithm=SPECTRAL_KMEANS_ALGORITHM,
)

primary_partition_2 = (
    primary_kmeans_2.fit_predict(
        primary_spectral_embedding
    )
)


primary_assignment_reproducibility_gate = (
    np.array_equal(
        primary_partition_1,
        primary_partition_2,
    )
)

assert primary_assignment_reproducibility_gate


primary_partition = np.asarray(
    primary_partition_1,
    dtype=np.int32,
)

primary_cluster_counts = np.bincount(
    primary_partition,
    minlength=SPECTRAL_N_CLUSTERS,
)

primary_partition_gate = (
    primary_partition.shape == (8583,)
    and primary_partition.dtype == np.int32

    and int(
        primary_partition.min()
    ) == 0

    and int(
        primary_partition.max()
    ) == 35

    and np.unique(
        primary_partition
    ).size == 36

    and int(
        primary_cluster_counts.sum()
    ) == 8583

    and np.all(
        primary_cluster_counts > 0
    )
)

assert primary_partition_gate


PRIMARY_KMEANS_INERTIA = float(
    primary_kmeans_1.inertia_
)

PRIMARY_KMEANS_ITERATIONS = int(
    primary_kmeans_1.n_iter_
)

PRIMARY_CLUSTER_MIN_SIZE = int(
    primary_cluster_counts.min()
)

PRIMARY_CLUSTER_MAX_SIZE = int(
    primary_cluster_counts.max()
)


# ------------------------------------------------------------
# 10. Persist artifacts
# ------------------------------------------------------------

s5_atomic_save_npy(
    PRIMARY_EIGENVALUES_PATH,
    primary_eigenvalues,
)

s5_atomic_save_npy(
    PRIMARY_EIGENVECTORS_PATH,
    primary_eigenvectors,
)

s5_atomic_save_npy(
    PRIMARY_EMBEDDING_PATH,
    primary_spectral_embedding,
)

s5_atomic_save_npy(
    PRIMARY_PARTITION_PATH,
    primary_partition,
)


PRIMARY_EIGENVALUES_ARRAY_SHA256 = (
    s5_array_sha256(
        primary_eigenvalues
    )
)

PRIMARY_EIGENVECTORS_ARRAY_SHA256 = (
    s5_array_sha256(
        primary_eigenvectors
    )
)

PRIMARY_EMBEDDING_ARRAY_SHA256 = (
    s5_array_sha256(
        primary_spectral_embedding
    )
)

PRIMARY_PARTITION_ARRAY_SHA256 = (
    s5_array_sha256(
        primary_partition
    )
)


PRIMARY_EIGENVALUES_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_EIGENVALUES_PATH
    )
)

PRIMARY_EIGENVECTORS_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_EIGENVECTORS_PATH
    )
)

PRIMARY_EMBEDDING_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_EMBEDDING_PATH
    )
)

PRIMARY_PARTITION_FILE_SHA256 = (
    sha256_stream(
        PRIMARY_PARTITION_PATH
    )
)


# ------------------------------------------------------------
# 11. Persistence audit
# ------------------------------------------------------------

reload_eigenvalues = np.load(
    PRIMARY_EIGENVALUES_PATH,
    allow_pickle=False,
)

reload_eigenvectors = np.load(
    PRIMARY_EIGENVECTORS_PATH,
    allow_pickle=False,
)

reload_embedding = np.load(
    PRIMARY_EMBEDDING_PATH,
    allow_pickle=False,
)

reload_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)


primary_persistence_gate = (
    np.array_equal(
        reload_eigenvalues,
        primary_eigenvalues,
    )

    and np.array_equal(
        reload_eigenvectors,
        primary_eigenvectors,
    )

    and np.array_equal(
        reload_embedding,
        primary_spectral_embedding,
    )

    and np.array_equal(
        reload_partition,
        primary_partition,
    )
)

assert primary_persistence_gate

del reload_eigenvalues
del reload_eigenvectors
del reload_embedding
del reload_partition


# ------------------------------------------------------------
# 12. Metadata
# ------------------------------------------------------------

PRIMARY_SPECTRAL_METADATA = {
    "format_version":
        1,

    "status":
        "PRIMARY_SPECTRAL_BUILT_LABEL_BLIND",

    "run_id":
        "sakai224_full_iic_symmetric_v1",

    "role":
        "PRIMARY",

    "source_authority":
        "Scientific_Reports_version_of_record_HTML",

    "cluster_count_C":
        36,

    "affinity_array_sha256":
        PRIMARY_A_ARRAY_SHA256,

    "affinity_file_sha256":
        EXPECTED_PRIMARY_A_FILE_SHA256,

    "affinity_contract_sha256":
        SPECTRAL_AFFINITY_CONTRACT_SHA256,

    "spectral_solver_contract_sha256":
        SPECTRAL_SOLVER_CONTRACT_SHA256,

    "solver": {
        "backend":
            "scipy.sparse.linalg.eigsh_ARPACK",

        "which":
            SPECTRAL_EIGEN_WHICH,

        "tol":
            SPECTRAL_EIGEN_TOL,

        "maxiter":
            SPECTRAL_EIGEN_MAXITER,

        "ncv":
            SPECTRAL_EIGEN_NCV,

        "v0_seed":
            SPECTRAL_EIGEN_V0_SEED,

        "top_eigenvalue":
            PRIMARY_TOP_EIGENVALUE,

        "residual_max":
            PRIMARY_RESIDUAL_MAX,

        "orthogonality_max_error":
            PRIMARY_ORTHOGONALITY_MAX_ERROR,
    },

    "assignment": {
        "method":
            "kmeans++",

        "seed":
            SPECTRAL_KMEANS_SEED,

        "n_init":
            SPECTRAL_KMEANS_N_INIT,

        "max_iter":
            SPECTRAL_KMEANS_MAX_ITER,

        "tol":
            SPECTRAL_KMEANS_TOL,

        "algorithm":
            SPECTRAL_KMEANS_ALGORITHM,

        "inertia":
            PRIMARY_KMEANS_INERTIA,

        "iterations":
            PRIMARY_KMEANS_ITERATIONS,

        "cluster_min_size":
            PRIMARY_CLUSTER_MIN_SIZE,

        "cluster_max_size":
            PRIMARY_CLUSTER_MAX_SIZE,
    },

    "artifacts": {
        "eigenvalues_array_sha256":
            PRIMARY_EIGENVALUES_ARRAY_SHA256,

        "eigenvalues_file_sha256":
            PRIMARY_EIGENVALUES_FILE_SHA256,

        "eigenvectors_array_sha256":
            PRIMARY_EIGENVECTORS_ARRAY_SHA256,

        "eigenvectors_file_sha256":
            PRIMARY_EIGENVECTORS_FILE_SHA256,

        "embedding_array_sha256":
            PRIMARY_EMBEDDING_ARRAY_SHA256,

        "embedding_file_sha256":
            PRIMARY_EMBEDDING_FILE_SHA256,

        "partition_array_sha256":
            PRIMARY_PARTITION_ARRAY_SHA256,

        "partition_file_sha256":
            PRIMARY_PARTITION_FILE_SHA256,
    },

    "runtime_seconds": {
        "affinity_load":
            PRIMARY_A_LOAD_SECONDS,

        "eigensolve":
            PRIMARY_EIGENSOLVE_SECONDS,

        "first_kmeans":
            PRIMARY_KMEANS_SECONDS,
    },

    "labels_used":
        False,

    "sensitivity_spectral_built":
        False,

    "partitions_frozen_for_label_evaluation":
        False,
}


PRIMARY_SPECTRAL_METADATA_JSON = json.dumps(
    PRIMARY_SPECTRAL_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

atomic_write_text(
    PRIMARY_SPECTRAL_METADATA_PATH,
    PRIMARY_SPECTRAL_METADATA_JSON,
)

PRIMARY_SPECTRAL_METADATA_SHA256 = (
    sha256_stream(
        PRIMARY_SPECTRAL_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 13. Final state
# ------------------------------------------------------------

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False

PRIMARY_SPECTRAL_BUILT = True
SENSITIVITY_SPECTRAL_BUILT = False
SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS = True
READY_FOR_S6_LEGACY_SPECTRAL = True
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S5 PRIMARY FULL SPECTRAL ==="
)

print(
    "TOP_EIGENVALUE =",
    PRIMARY_TOP_EIGENVALUE,
)

print(
    "RESIDUAL_MAX =",
    PRIMARY_RESIDUAL_MAX,
)

print(
    "ORTHOGONALITY_MAX_ERROR =",
    PRIMARY_ORTHOGONALITY_MAX_ERROR,
)

print(
    "EIGENSOLVER_GATE =",
    primary_eigensolver_gate,
)

print(
    "EMBEDDING_GATE =",
    primary_embedding_gate,
)

print(
    "ASSIGNMENT_REPRODUCIBILITY_GATE =",
    primary_assignment_reproducibility_gate,
)

print(
    "PARTITION_GATE =",
    primary_partition_gate,
)

print(
    "CLUSTER_MIN_SIZE =",
    PRIMARY_CLUSTER_MIN_SIZE,
)

print(
    "CLUSTER_MAX_SIZE =",
    PRIMARY_CLUSTER_MAX_SIZE,
)

print(
    "KMEANS_INERTIA =",
    PRIMARY_KMEANS_INERTIA,
)

print(
    "KMEANS_ITERATIONS =",
    PRIMARY_KMEANS_ITERATIONS,
)

print(
    "EIGENSOLVE_SECONDS =",
    PRIMARY_EIGENSOLVE_SECONDS,
)

print(
    "PARTITION_ARRAY_SHA256 =",
    PRIMARY_PARTITION_ARRAY_SHA256,
)

print(
    "PARTITION_FILE_SHA256 =",
    PRIMARY_PARTITION_FILE_SHA256,
)

print(
    "METADATA_SHA256 =",
    PRIMARY_SPECTRAL_METADATA_SHA256,
)

print(
    "PERSISTENCE_GATE =",
    primary_persistence_gate,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS =",
    NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS,
)

print(
    "READY_FOR_S6_LEGACY_SPECTRAL =",
    READY_FOR_S6_LEGACY_SPECTRAL,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


# ------------------------------------------------------------
# 15. Clean large matrix
# ------------------------------------------------------------

del primary_A
del PRIMARY_M_OPERATOR

gc.collect()


Loading PRIMARY affinity...
Running PRIMARY ARPACK eigensolve...
Running PRIMARY final k-means++ assignment...

=== NOTEBOOK 06 — S5 PRIMARY FULL SPECTRAL ===
TOP_EIGENVALUE = 1.0
RESIDUAL_MAX = 1.0367813987250168e-15
ORTHOGONALITY_MAX_ERROR = 1.1102230246251565e-15
EIGENSOLVER_GATE = True
EMBEDDING_GATE = True
ASSIGNMENT_REPRODUCIBILITY_GATE = True
PARTITION_GATE = True
CLUSTER_MIN_SIZE = 83
CLUSTER_MAX_SIZE = 1435
KMEANS_INERTIA = 1026.660133419584
KMEANS_ITERATIONS = 17
EIGENSOLVE_SECONDS = 5.443925157000194
PARTITION_ARRAY_SHA256 = 812d329469fefbc705f146cb6310afdcee60c13d4d23aebdc4f56c6faa1afd87
PARTITION_FILE_SHA256 = 5a01318f703441bbf291f7b32478043f6d0376dd81265078d6dbeac43e306ac0
METADATA_SHA256 = 1662aa04e340bc94d538bf222a9316e9f37489b14470090124c93df6590713ed
PERSISTENCE_GATE = True
LABELS_READ = False
NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS = True
READY_FOR_S6_LEGACY_SPECTRAL = True
READY_FOR_LABEL_EVALUATION = False


55

In [33]:
# ============================================================
# NOTEBOOK 06 — S6
# LEGACY FULL SPECTRAL — LABEL BLIND
#
# Same frozen PROJECT spectral contract used for PRIMARY.
# Only the source affinity changes:
#   symmetric_figure -> legacy_text
#
# NO labels.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import os
import time

import numpy as np
from scipy.sparse.linalg import LinearOperator, eigsh
from sklearn.cluster import KMeans


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS"
) is True

assert globals().get(
    "READY_FOR_S6_LEGACY_SPECTRAL"
) is True

assert (
    SPECTRAL_SOLVER_CONTRACT_SHA256
    == "3b75e9d89673280284f9bcd735424d41"
       "d89a6b9f066176a560dc930d57192a0f"
)

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False
assert SPECTRAL_PARTITIONS_FROZEN is False


# ------------------------------------------------------------
# 1. Frozen legacy affinity
# ------------------------------------------------------------

EXPECTED_LEGACY_A_ARRAY_SHA256 = (
    "93fb893b6e028b65c09430c5f3f80798"
    "d23e62b27349ada19f2069f1a2f97958"
)

EXPECTED_LEGACY_A_FILE_SHA256 = (
    "2f1e4d236a6287ee059ddedee6aa2873"
    "e3a65d6ec37c8a96a831bd8d23cd7d7d"
)

assert (
    LEGACY_A_ARRAY_SHA256
    == EXPECTED_LEGACY_A_ARRAY_SHA256
)

assert (
    sha256_stream(
        LEGACY_A_PATH
    )
    == EXPECTED_LEGACY_A_FILE_SHA256
)

assert (
    sha256_stream(
        SPECTRAL_SOLVER_CONTRACT_PATH
    )
    == SPECTRAL_SOLVER_CONTRACT_SHA256
)


# ------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------

LEGACY_SPECTRAL_DIR = (
    SPECTRAL_ROOT
    / "partitions"
    / "sakai224_full_iic_legacy_text_v1"
)

LEGACY_SPECTRAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LEGACY_EIGENVALUES_PATH = (
    LEGACY_SPECTRAL_DIR
    / "eigenvalues.npy"
)

LEGACY_EIGENVECTORS_PATH = (
    LEGACY_SPECTRAL_DIR
    / "eigenvectors.npy"
)

LEGACY_EMBEDDING_PATH = (
    LEGACY_SPECTRAL_DIR
    / "spectral_embedding.npy"
)

LEGACY_PARTITION_PATH = (
    LEGACY_SPECTRAL_DIR
    / "partition.npy"
)

LEGACY_SPECTRAL_METADATA_PATH = (
    LEGACY_SPECTRAL_DIR
    / "spectral_metadata.json"
)

assert not LEGACY_SPECTRAL_METADATA_PATH.exists()

for path in [
    LEGACY_EIGENVALUES_PATH,
    LEGACY_EIGENVECTORS_PATH,
    LEGACY_EMBEDDING_PATH,
    LEGACY_PARTITION_PATH,
]:
    assert not path.exists(), (
        f"STOP: pre-existing spectral artifact: {path}"
    )


# ------------------------------------------------------------
# 3. Atomic helpers
# ------------------------------------------------------------

def s6_atomic_save_npy(
    path,
    array,
):
    partial = Path(
        str(path)
        + ".partial"
    )

    assert not partial.exists()

    with partial.open("wb") as handle:

        np.save(
            handle,
            array,
            allow_pickle=False,
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        partial,
        path,
    )


def s6_array_sha256(
    array,
):
    contiguous = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        contiguous.tobytes(
            order="C"
        )
    ).hexdigest()


# ------------------------------------------------------------
# 4. Load legacy affinity
# ------------------------------------------------------------

print()
print(
    "Loading LEGACY affinity..."
)

t0 = time.perf_counter()

legacy_A = np.load(
    LEGACY_A_PATH,
    allow_pickle=False,
)

LEGACY_A_LOAD_SECONDS = (
    time.perf_counter()
    - t0
)

assert legacy_A.shape == (
    8583,
    8583,
)

assert legacy_A.dtype == np.float64
assert np.isfinite(legacy_A).all()

assert np.array_equal(
    np.diag(
        legacy_A
    ),
    np.ones(
        8583,
        dtype=np.float64,
    ),
)


# ------------------------------------------------------------
# 5. Normalized adjacency operator
#
# W = A - I
# M = D^-1/2 W D^-1/2
# ------------------------------------------------------------

legacy_degree = (
    legacy_A.sum(
        axis=1,
        dtype=np.float64,
    )
    - 1.0
)

assert np.isfinite(
    legacy_degree
).all()

assert np.all(
    legacy_degree > 0.0
)

legacy_inv_sqrt_degree = (
    1.0
    / np.sqrt(
        legacy_degree
    )
)


def legacy_m_matvec(
    x,
):
    z = (
        legacy_inv_sqrt_degree
        * x
    )

    y = (
        legacy_A
        @ z
        - z
    )

    return (
        legacy_inv_sqrt_degree
        * y
    )


def legacy_m_matmat(
    X,
):
    Z = (
        legacy_inv_sqrt_degree[:, None]
        * X
    )

    Y = (
        legacy_A
        @ Z
        - Z
    )

    return (
        legacy_inv_sqrt_degree[:, None]
        * Y
    )


LEGACY_M_OPERATOR = LinearOperator(
    shape=(8583, 8583),
    matvec=legacy_m_matvec,
    matmat=legacy_m_matmat,
    dtype=np.float64,
)


# ------------------------------------------------------------
# 6. Full legacy eigensolve
# ------------------------------------------------------------

print(
    "Running LEGACY ARPACK eigensolve..."
)

SPECTRAL_STARTED = True

t0 = time.perf_counter()

legacy_eigenvalues, legacy_eigenvectors = eigsh(
    LEGACY_M_OPERATOR,
    k=SPECTRAL_N_CLUSTERS,
    which=SPECTRAL_EIGEN_WHICH,
    tol=SPECTRAL_EIGEN_TOL,
    maxiter=SPECTRAL_EIGEN_MAXITER,
    ncv=SPECTRAL_EIGEN_NCV,
    v0=fixed_v0(8583),
)

LEGACY_EIGENSOLVE_SECONDS = (
    time.perf_counter()
    - t0
)


legacy_order = np.argsort(
    legacy_eigenvalues
)[::-1]

legacy_eigenvalues = np.asarray(
    legacy_eigenvalues[
        legacy_order
    ],
    dtype=np.float64,
)

legacy_eigenvectors = np.asarray(
    legacy_eigenvectors[
        :,
        legacy_order
    ],
    dtype=np.float64,
)

legacy_eigenvectors = canonicalize_signs(
    legacy_eigenvectors
)


# ------------------------------------------------------------
# 7. Eigensolver audit
# ------------------------------------------------------------

legacy_residuals = np.linalg.norm(
    (
        LEGACY_M_OPERATOR
        @ legacy_eigenvectors
    )
    - (
        legacy_eigenvectors
        * legacy_eigenvalues[
            None,
            :
        ]
    ),
    axis=0,
)

LEGACY_RESIDUAL_MAX = float(
    legacy_residuals.max()
)

LEGACY_ORTHOGONALITY_MAX_ERROR = float(
    np.max(
        np.abs(
            (
                legacy_eigenvectors.T
                @ legacy_eigenvectors
            )
            - np.eye(
                SPECTRAL_N_CLUSTERS,
                dtype=np.float64,
            )
        )
    )
)

LEGACY_TOP_EIGENVALUE = float(
    legacy_eigenvalues[0]
)


legacy_eigensolver_gate = (
    legacy_eigenvalues.shape == (36,)
    and legacy_eigenvectors.shape == (8583, 36)

    and np.isfinite(
        legacy_eigenvalues
    ).all()

    and np.isfinite(
        legacy_eigenvectors
    ).all()

    and np.all(
        np.diff(
            legacy_eigenvalues
        ) <= 1.0e-12
    )

    and abs(
        LEGACY_TOP_EIGENVALUE
        - 1.0
    ) <= 1.0e-8

    and LEGACY_RESIDUAL_MAX
        <= 1.0e-6

    and LEGACY_ORTHOGONALITY_MAX_ERROR
        <= 1.0e-8
)

assert legacy_eigensolver_gate


# ------------------------------------------------------------
# 8. Row-normalized embedding
# ------------------------------------------------------------

legacy_spectral_embedding = (
    row_normalize(
        legacy_eigenvectors
    )
)

legacy_embedding_gate = (
    legacy_spectral_embedding.shape
        == (8583, 36)

    and legacy_spectral_embedding.dtype
        == np.float64

    and np.isfinite(
        legacy_spectral_embedding
    ).all()

    and np.allclose(
        np.linalg.norm(
            legacy_spectral_embedding,
            axis=1,
        ),
        1.0,
        rtol=0.0,
        atol=2.0e-12,
    )
)

assert legacy_embedding_gate


# ------------------------------------------------------------
# 9. Final PROJECT k-means++
# ------------------------------------------------------------

print(
    "Running LEGACY final k-means++ assignment..."
)

t0 = time.perf_counter()

legacy_kmeans_1 = KMeans(
    n_clusters=SPECTRAL_N_CLUSTERS,
    init="k-means++",
    n_init=SPECTRAL_KMEANS_N_INIT,
    max_iter=SPECTRAL_KMEANS_MAX_ITER,
    tol=SPECTRAL_KMEANS_TOL,
    random_state=SPECTRAL_KMEANS_SEED,
    algorithm=SPECTRAL_KMEANS_ALGORITHM,
)

legacy_partition_1 = (
    legacy_kmeans_1.fit_predict(
        legacy_spectral_embedding
    )
)

LEGACY_KMEANS_SECONDS = (
    time.perf_counter()
    - t0
)


legacy_kmeans_2 = KMeans(
    n_clusters=SPECTRAL_N_CLUSTERS,
    init="k-means++",
    n_init=SPECTRAL_KMEANS_N_INIT,
    max_iter=SPECTRAL_KMEANS_MAX_ITER,
    tol=SPECTRAL_KMEANS_TOL,
    random_state=SPECTRAL_KMEANS_SEED,
    algorithm=SPECTRAL_KMEANS_ALGORITHM,
)

legacy_partition_2 = (
    legacy_kmeans_2.fit_predict(
        legacy_spectral_embedding
    )
)


legacy_assignment_reproducibility_gate = (
    np.array_equal(
        legacy_partition_1,
        legacy_partition_2,
    )
)

assert legacy_assignment_reproducibility_gate


legacy_partition = np.asarray(
    legacy_partition_1,
    dtype=np.int32,
)

legacy_cluster_counts = np.bincount(
    legacy_partition,
    minlength=SPECTRAL_N_CLUSTERS,
)

legacy_partition_gate = (
    legacy_partition.shape == (8583,)
    and legacy_partition.dtype == np.int32

    and int(
        legacy_partition.min()
    ) == 0

    and int(
        legacy_partition.max()
    ) == 35

    and np.unique(
        legacy_partition
    ).size == 36

    and int(
        legacy_cluster_counts.sum()
    ) == 8583

    and np.all(
        legacy_cluster_counts > 0
    )
)

assert legacy_partition_gate


LEGACY_KMEANS_INERTIA = float(
    legacy_kmeans_1.inertia_
)

LEGACY_KMEANS_ITERATIONS = int(
    legacy_kmeans_1.n_iter_
)

LEGACY_CLUSTER_MIN_SIZE = int(
    legacy_cluster_counts.min()
)

LEGACY_CLUSTER_MAX_SIZE = int(
    legacy_cluster_counts.max()
)


# ------------------------------------------------------------
# 10. Persist artifacts
# ------------------------------------------------------------

s6_atomic_save_npy(
    LEGACY_EIGENVALUES_PATH,
    legacy_eigenvalues,
)

s6_atomic_save_npy(
    LEGACY_EIGENVECTORS_PATH,
    legacy_eigenvectors,
)

s6_atomic_save_npy(
    LEGACY_EMBEDDING_PATH,
    legacy_spectral_embedding,
)

s6_atomic_save_npy(
    LEGACY_PARTITION_PATH,
    legacy_partition,
)


LEGACY_EIGENVALUES_ARRAY_SHA256 = (
    s6_array_sha256(
        legacy_eigenvalues
    )
)

LEGACY_EIGENVECTORS_ARRAY_SHA256 = (
    s6_array_sha256(
        legacy_eigenvectors
    )
)

LEGACY_EMBEDDING_ARRAY_SHA256 = (
    s6_array_sha256(
        legacy_spectral_embedding
    )
)

LEGACY_PARTITION_ARRAY_SHA256 = (
    s6_array_sha256(
        legacy_partition
    )
)


LEGACY_EIGENVALUES_FILE_SHA256 = (
    sha256_stream(
        LEGACY_EIGENVALUES_PATH
    )
)

LEGACY_EIGENVECTORS_FILE_SHA256 = (
    sha256_stream(
        LEGACY_EIGENVECTORS_PATH
    )
)

LEGACY_EMBEDDING_FILE_SHA256 = (
    sha256_stream(
        LEGACY_EMBEDDING_PATH
    )
)

LEGACY_PARTITION_FILE_SHA256 = (
    sha256_stream(
        LEGACY_PARTITION_PATH
    )
)


# ------------------------------------------------------------
# 11. Persistence audit
# ------------------------------------------------------------

reload_eigenvalues = np.load(
    LEGACY_EIGENVALUES_PATH,
    allow_pickle=False,
)

reload_eigenvectors = np.load(
    LEGACY_EIGENVECTORS_PATH,
    allow_pickle=False,
)

reload_embedding = np.load(
    LEGACY_EMBEDDING_PATH,
    allow_pickle=False,
)

reload_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)


legacy_persistence_gate = (
    np.array_equal(
        reload_eigenvalues,
        legacy_eigenvalues,
    )

    and np.array_equal(
        reload_eigenvectors,
        legacy_eigenvectors,
    )

    and np.array_equal(
        reload_embedding,
        legacy_spectral_embedding,
    )

    and np.array_equal(
        reload_partition,
        legacy_partition,
    )
)

assert legacy_persistence_gate

del reload_eigenvalues
del reload_eigenvectors
del reload_embedding
del reload_partition


# ------------------------------------------------------------
# 12. Must differ from PRIMARY
# ------------------------------------------------------------

primary_partition_saved = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

primary_legacy_partition_distinct_gate = (
    not np.array_equal(
        primary_partition_saved,
        legacy_partition,
    )
)

assert primary_legacy_partition_distinct_gate

del primary_partition_saved


# ------------------------------------------------------------
# 13. Metadata
# ------------------------------------------------------------

LEGACY_SPECTRAL_METADATA = {
    "format_version":
        1,

    "status":
        "SENSITIVITY_SPECTRAL_BUILT_LABEL_BLIND",

    "run_id":
        "sakai224_full_iic_legacy_text_v1",

    "role":
        "SENSITIVITY",

    "source_authority":
        "Scientific_Reports_version_of_record_HTML",

    "cluster_count_C":
        36,

    "affinity_array_sha256":
        LEGACY_A_ARRAY_SHA256,

    "affinity_file_sha256":
        EXPECTED_LEGACY_A_FILE_SHA256,

    "affinity_contract_sha256":
        SPECTRAL_AFFINITY_CONTRACT_SHA256,

    "spectral_solver_contract_sha256":
        SPECTRAL_SOLVER_CONTRACT_SHA256,

    "solver": {
        "backend":
            "scipy.sparse.linalg.eigsh_ARPACK",

        "which":
            SPECTRAL_EIGEN_WHICH,

        "tol":
            SPECTRAL_EIGEN_TOL,

        "maxiter":
            SPECTRAL_EIGEN_MAXITER,

        "ncv":
            SPECTRAL_EIGEN_NCV,

        "v0_seed":
            SPECTRAL_EIGEN_V0_SEED,

        "top_eigenvalue":
            LEGACY_TOP_EIGENVALUE,

        "residual_max":
            LEGACY_RESIDUAL_MAX,

        "orthogonality_max_error":
            LEGACY_ORTHOGONALITY_MAX_ERROR,
    },

    "assignment": {
        "method":
            "kmeans++",

        "seed":
            SPECTRAL_KMEANS_SEED,

        "n_init":
            SPECTRAL_KMEANS_N_INIT,

        "max_iter":
            SPECTRAL_KMEANS_MAX_ITER,

        "tol":
            SPECTRAL_KMEANS_TOL,

        "algorithm":
            SPECTRAL_KMEANS_ALGORITHM,

        "inertia":
            LEGACY_KMEANS_INERTIA,

        "iterations":
            LEGACY_KMEANS_ITERATIONS,

        "cluster_min_size":
            LEGACY_CLUSTER_MIN_SIZE,

        "cluster_max_size":
            LEGACY_CLUSTER_MAX_SIZE,
    },

    "artifacts": {
        "eigenvalues_array_sha256":
            LEGACY_EIGENVALUES_ARRAY_SHA256,

        "eigenvalues_file_sha256":
            LEGACY_EIGENVALUES_FILE_SHA256,

        "eigenvectors_array_sha256":
            LEGACY_EIGENVECTORS_ARRAY_SHA256,

        "eigenvectors_file_sha256":
            LEGACY_EIGENVECTORS_FILE_SHA256,

        "embedding_array_sha256":
            LEGACY_EMBEDDING_ARRAY_SHA256,

        "embedding_file_sha256":
            LEGACY_EMBEDDING_FILE_SHA256,

        "partition_array_sha256":
            LEGACY_PARTITION_ARRAY_SHA256,

        "partition_file_sha256":
            LEGACY_PARTITION_FILE_SHA256,
    },

    "runtime_seconds": {
        "affinity_load":
            LEGACY_A_LOAD_SECONDS,

        "eigensolve":
            LEGACY_EIGENSOLVE_SECONDS,

        "first_kmeans":
            LEGACY_KMEANS_SECONDS,
    },

    "labels_used":
        False,

    "primary_spectral_built":
        True,

    "partitions_frozen_for_label_evaluation":
        False,
}


LEGACY_SPECTRAL_METADATA_JSON = json.dumps(
    LEGACY_SPECTRAL_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

atomic_write_text(
    LEGACY_SPECTRAL_METADATA_PATH,
    LEGACY_SPECTRAL_METADATA_JSON,
)

LEGACY_SPECTRAL_METADATA_SHA256 = (
    sha256_stream(
        LEGACY_SPECTRAL_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 14. Final state
# ------------------------------------------------------------

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False

PRIMARY_SPECTRAL_BUILT = True
SENSITIVITY_SPECTRAL_BUILT = True

SPECTRAL_PARTITIONS_FROZEN = False

NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS = True
READY_FOR_S7_DUAL_PARTITION_FREEZE = True
READY_FOR_LABEL_EVALUATION = False


# ------------------------------------------------------------
# 15. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S6 LEGACY FULL SPECTRAL ==="
)

print(
    "TOP_EIGENVALUE =",
    LEGACY_TOP_EIGENVALUE,
)

print(
    "RESIDUAL_MAX =",
    LEGACY_RESIDUAL_MAX,
)

print(
    "ORTHOGONALITY_MAX_ERROR =",
    LEGACY_ORTHOGONALITY_MAX_ERROR,
)

print(
    "EIGENSOLVER_GATE =",
    legacy_eigensolver_gate,
)

print(
    "EMBEDDING_GATE =",
    legacy_embedding_gate,
)

print(
    "ASSIGNMENT_REPRODUCIBILITY_GATE =",
    legacy_assignment_reproducibility_gate,
)

print(
    "PARTITION_GATE =",
    legacy_partition_gate,
)

print(
    "PRIMARY_LEGACY_PARTITION_DISTINCT =",
    primary_legacy_partition_distinct_gate,
)

print(
    "CLUSTER_MIN_SIZE =",
    LEGACY_CLUSTER_MIN_SIZE,
)

print(
    "CLUSTER_MAX_SIZE =",
    LEGACY_CLUSTER_MAX_SIZE,
)

print(
    "KMEANS_INERTIA =",
    LEGACY_KMEANS_INERTIA,
)

print(
    "KMEANS_ITERATIONS =",
    LEGACY_KMEANS_ITERATIONS,
)

print(
    "EIGENSOLVE_SECONDS =",
    LEGACY_EIGENSOLVE_SECONDS,
)

print(
    "PARTITION_ARRAY_SHA256 =",
    LEGACY_PARTITION_ARRAY_SHA256,
)

print(
    "PARTITION_FILE_SHA256 =",
    LEGACY_PARTITION_FILE_SHA256,
)

print(
    "METADATA_SHA256 =",
    LEGACY_SPECTRAL_METADATA_SHA256,
)

print(
    "PERSISTENCE_GATE =",
    legacy_persistence_gate,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS =",
    NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS,
)

print(
    "READY_FOR_S7_DUAL_PARTITION_FREEZE =",
    READY_FOR_S7_DUAL_PARTITION_FREEZE,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


# ------------------------------------------------------------
# 16. Clean large matrix
# ------------------------------------------------------------

del legacy_A
del LEGACY_M_OPERATOR

gc.collect()


Loading LEGACY affinity...
Running LEGACY ARPACK eigensolve...
Running LEGACY final k-means++ assignment...

=== NOTEBOOK 06 — S6 LEGACY FULL SPECTRAL ===
TOP_EIGENVALUE = 0.9999999999999998
RESIDUAL_MAX = 9.798961413836475e-16
ORTHOGONALITY_MAX_ERROR = 1.3322676295501878e-15
EIGENSOLVER_GATE = True
EMBEDDING_GATE = True
ASSIGNMENT_REPRODUCIBILITY_GATE = True
PARTITION_GATE = True
PRIMARY_LEGACY_PARTITION_DISTINCT = True
CLUSTER_MIN_SIZE = 83
CLUSTER_MAX_SIZE = 883
KMEANS_INERTIA = 1043.1404524249501
KMEANS_ITERATIONS = 10
EIGENSOLVE_SECONDS = 4.560480672000267
PARTITION_ARRAY_SHA256 = bc4d0f8a7b69dce58b59d785d354f17be320f3ef13688e4809e1c0b816f61a28
PARTITION_FILE_SHA256 = 69edcfaf5006214c992ce39fb359d29841c78ab5fc13fe1f58901d06915a3632
METADATA_SHA256 = 88271aedb7f00ecbb611d88633d74f63a11cd197f88e3f950e7c13009a9492f5
PERSISTENCE_GATE = True
LABELS_READ = False
NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS = True
READY_FOR_S7_DUAL_PARTITION_FREEZE = True
READY_FOR_LABEL_EVALUATION = False


66

In [34]:
# ============================================================
# NOTEBOOK 06 — S7
# DUAL SPECTRAL PARTITION FREEZE
#
# Verifica e congela:
#   - primary symmetric_figure
#   - sensitivity legacy_text
#
# NO label read.
# NO retraining.
# NO new clustering.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS"
) is True

assert globals().get(
    "READY_FOR_S7_DUAL_PARTITION_FREEZE"
) is True

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False


# ------------------------------------------------------------
# 1. Frozen identities from S5 / S6
# ------------------------------------------------------------

EXPECTED_PRIMARY_PARTITION_ARRAY_SHA256 = (
    "812d329469fefbc705f146cb6310afdcee"
    "60c13d4d23aebdc4f56c6faa1afd87"
)

EXPECTED_PRIMARY_PARTITION_FILE_SHA256 = (
    "5a01318f703441bbf291f7b32478043f"
    "6d0376dd81265078d6dbeac43e306ac0"
)

EXPECTED_PRIMARY_METADATA_SHA256 = (
    "1662aa04e340bc94d538bf222a9316e9"
    "f37489b14470090124c93df6590713ed"
)

EXPECTED_LEGACY_PARTITION_ARRAY_SHA256 = (
    "bc4d0f8a7b69dce58b59d785d354f17b"
    "e320f3ef13688e4809e1c0b816f61a28"
)

EXPECTED_LEGACY_PARTITION_FILE_SHA256 = (
    "69edcfaf5006214c992ce39fb359d298"
    "41c78ab5fc13fe1f58901d06915a3632"
)

EXPECTED_LEGACY_METADATA_SHA256 = (
    "88271aedb7f00ecbb611d88633d74f63"
    "a11cd197f88e3f950e7c13009a9492f5"
)


assert (
    PRIMARY_PARTITION_ARRAY_SHA256
    == EXPECTED_PRIMARY_PARTITION_ARRAY_SHA256
)

assert (
    PRIMARY_PARTITION_FILE_SHA256
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256
)

assert (
    PRIMARY_SPECTRAL_METADATA_SHA256
    == EXPECTED_PRIMARY_METADATA_SHA256
)

assert (
    LEGACY_PARTITION_ARRAY_SHA256
    == EXPECTED_LEGACY_PARTITION_ARRAY_SHA256
)

assert (
    LEGACY_PARTITION_FILE_SHA256
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert (
    LEGACY_SPECTRAL_METADATA_SHA256
    == EXPECTED_LEGACY_METADATA_SHA256
)


# ------------------------------------------------------------
# 2. Re-read persisted partitions
# ------------------------------------------------------------

s7_primary_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

s7_legacy_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)


def partition_gate(
    partition,
):
    counts = np.bincount(
        partition,
        minlength=36,
    )

    return bool(
        partition.shape == (8583,)
        and partition.dtype == np.int32
        and int(partition.min()) == 0
        and int(partition.max()) == 35
        and np.unique(partition).size == 36
        and int(counts.sum()) == 8583
        and np.all(counts > 0)
    )


primary_partition_gate_s7 = partition_gate(
    s7_primary_partition
)

legacy_partition_gate_s7 = partition_gate(
    s7_legacy_partition
)

assert primary_partition_gate_s7
assert legacy_partition_gate_s7


# ------------------------------------------------------------
# 3. File identities
# ------------------------------------------------------------

file_identity_gate = (
    sha256_stream(
        PRIMARY_PARTITION_PATH
    ) == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and sha256_stream(
        LEGACY_PARTITION_PATH
    ) == EXPECTED_LEGACY_PARTITION_FILE_SHA256

    and sha256_stream(
        PRIMARY_SPECTRAL_METADATA_PATH
    ) == EXPECTED_PRIMARY_METADATA_SHA256

    and sha256_stream(
        LEGACY_SPECTRAL_METADATA_PATH
    ) == EXPECTED_LEGACY_METADATA_SHA256
)

assert file_identity_gate


# ------------------------------------------------------------
# 4. The two preregistered results remain distinct
# ------------------------------------------------------------

dual_partition_distinct_gate = (
    not np.array_equal(
        s7_primary_partition,
        s7_legacy_partition,
    )
)

assert dual_partition_distinct_gate


primary_counts = np.bincount(
    s7_primary_partition,
    minlength=36,
)

legacy_counts = np.bincount(
    s7_legacy_partition,
    minlength=36,
)


# ------------------------------------------------------------
# 5. Freeze handoff
# ------------------------------------------------------------

SPECTRAL_AUDIT_DIR = (
    SPECTRAL_ROOT
    / "audits"
)

SPECTRAL_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

S7_FREEZE_PATH = (
    SPECTRAL_AUDIT_DIR
    / "sakai224_full_dual_spectral_after_S7.json"
)

assert not S7_FREEZE_PATH.exists(), (
    f"STOP: freeze already exists: {S7_FREEZE_PATH}"
)


S7_FREEZE = {
    "format_version":
        1,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "DUAL_SPECTRAL_PARTITIONS_FROZEN",

    "population": {
        "D": 8583,
        "row_order":
            "manifest_index_0_to_8582",
    },

    "paper_contract": {
        "K": 5,
        "C": 36,
        "H_shape":
            [8583, 180],
        "affinity_formula":
            "A_ij=exp(-||h_i-h_j||_2)",
        "spectral_cluster_count":
            36,
    },

    "project_contracts": {
        "H_sha256":
            SPECTRAL_H_CONTRACT_SHA256,

        "affinity_sha256":
            SPECTRAL_AFFINITY_CONTRACT_SHA256,

        "solver_sha256":
            SPECTRAL_SOLVER_CONTRACT_SHA256,
    },

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "role":
            "PRIMARY",

        "partition_array_sha256":
            PRIMARY_PARTITION_ARRAY_SHA256,

        "partition_file_sha256":
            PRIMARY_PARTITION_FILE_SHA256,

        "metadata_sha256":
            PRIMARY_SPECTRAL_METADATA_SHA256,

        "cluster_min_size":
            int(
                primary_counts.min()
            ),

        "cluster_max_size":
            int(
                primary_counts.max()
            ),
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "role":
            "SENSITIVITY",

        "partition_array_sha256":
            LEGACY_PARTITION_ARRAY_SHA256,

        "partition_file_sha256":
            LEGACY_PARTITION_FILE_SHA256,

        "metadata_sha256":
            LEGACY_SPECTRAL_METADATA_SHA256,

        "cluster_min_size":
            int(
                legacy_counts.min()
            ),

        "cluster_max_size":
            int(
                legacy_counts.max()
            ),
    },

    "scientific_state": {
        "both_IIC_runs_complete":
            True,

        "both_spectral_runs_complete":
            True,

        "both_partitions_frozen_before_labels":
            True,

        "primary_selected_before_labels":
            "symmetric_figure",

        "sensitivity_selected_before_labels":
            "legacy_text",

        "labels_read":
            False,
    },

    "next_step":
        "posterior_label_evaluation",
}


S7_FREEZE_JSON = json.dumps(
    S7_FREEZE,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

atomic_write_text(
    S7_FREEZE_PATH,
    S7_FREEZE_JSON,
)

S7_FREEZE_SHA256 = sha256_stream(
    S7_FREEZE_PATH
)


# ------------------------------------------------------------
# 6. Read-back
# ------------------------------------------------------------

with S7_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    s7_readback = json.load(
        handle
    )


freeze_readback_gate = (
    s7_readback["status"]
        == "DUAL_SPECTRAL_PARTITIONS_FROZEN"

    and s7_readback[
        "scientific_state"
    ][
        "both_partitions_frozen_before_labels"
    ] is True

    and s7_readback[
        "scientific_state"
    ][
        "labels_read"
    ] is False

    and s7_readback[
        "primary"
    ][
        "partition_array_sha256"
    ] == PRIMARY_PARTITION_ARRAY_SHA256

    and s7_readback[
        "sensitivity"
    ][
        "partition_array_sha256"
    ] == LEGACY_PARTITION_ARRAY_SHA256
)

assert freeze_readback_gate


# ------------------------------------------------------------
# 7. Final state
# ------------------------------------------------------------

LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False

PRIMARY_SPECTRAL_BUILT = True
SENSITIVITY_SPECTRAL_BUILT = True
SPECTRAL_PARTITIONS_FROZEN = True

NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS = True
READY_FOR_LABEL_EVALUATION = True


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — S7 DUAL PARTITION FREEZE ==="
)

print(
    "PRIMARY_PARTITION_GATE =",
    primary_partition_gate_s7,
)

print(
    "LEGACY_PARTITION_GATE =",
    legacy_partition_gate_s7,
)

print(
    "DUAL_PARTITION_DISTINCT =",
    dual_partition_distinct_gate,
)

print(
    "PRIMARY_CLUSTER_MIN_MAX =",
    (
        int(primary_counts.min()),
        int(primary_counts.max()),
    ),
)

print(
    "LEGACY_CLUSTER_MIN_MAX =",
    (
        int(legacy_counts.min()),
        int(legacy_counts.max()),
    ),
)

print(
    "FILE_IDENTITY_GATE =",
    file_identity_gate,
)

print(
    "FREEZE_READBACK_GATE =",
    freeze_readback_gate,
)

print(
    "S7_FREEZE_PATH =",
    S7_FREEZE_PATH,
)

print(
    "S7_FREEZE_SHA256 =",
    S7_FREEZE_SHA256,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "SPECTRAL_PARTITIONS_FROZEN =",
    SPECTRAL_PARTITIONS_FROZEN,
)

print(
    "NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS =",
    NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS,
)

print(
    "READY_FOR_LABEL_EVALUATION =",
    READY_FOR_LABEL_EVALUATION,
)


=== NOTEBOOK 06 — S7 DUAL PARTITION FREEZE ===
PRIMARY_PARTITION_GATE = True
LEGACY_PARTITION_GATE = True
DUAL_PARTITION_DISTINCT = True
PRIMARY_CLUSTER_MIN_MAX = (83, 1435)
LEGACY_CLUSTER_MIN_MAX = (83, 883)
FILE_IDENTITY_GATE = True
FREEZE_READBACK_GATE = True
S7_FREEZE_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/audits/sakai224_full_dual_spectral_after_S7.json
S7_FREEZE_SHA256 = 7fb80c7bc65ca2d6b98b5bc59985a2cb629f9189231fb759ce2ba674e2382fe1
LABELS_READ = False
SPECTRAL_PARTITIONS_FROZEN = True
NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS = True
READY_FOR_LABEL_EVALUATION = True


In [36]:
# ============================================================
# NOTEBOOK 06 — E0A
# LOCATE MANIFEST_EVENTS_272.CSV
#
# NO file contents read.
# NO labels accessed.
# ============================================================

from pathlib import Path
import os


TARGET_NAME = "manifest_events_272.csv"

found_paths = []


# 1. Check likely local roots first.
likely_paths = [
    (
        LOCAL_RUNTIME_ROOT
        / "sakai224_full_transfer_v1"
        / TARGET_NAME
    ),
    (
        LOCAL_RUNTIME_ROOT
        / "sakai224_full_data"
        / TARGET_NAME
    ),
]

for path in likely_paths:
    if path.is_file():
        found_paths.append(
            path.resolve()
        )


# 2. Check any already-existing runtime path variables.
for name, value in list(
    globals().items()
):
    if isinstance(
        value,
        (str, Path),
    ):
        try:
            path = Path(value)

            if (
                path.name == TARGET_NAME
                and path.is_file()
            ):
                found_paths.append(
                    path.resolve()
                )
        except Exception:
            pass


# 3. Search the local runtime root, excluding the project tree.
runtime_search_root = LOCAL_RUNTIME_ROOT.resolve(
    strict=False
)

project_search_root = PROJECT_ROOT.resolve(
    strict=False
)

for root, dirs, files in os.walk(
    runtime_search_root
):
    root_path = Path(root).resolve(
        strict=False
    )

    dirs[:] = [
        directory
        for directory in dirs
        if not (
            project_search_root.is_relative_to(
                (
                    root_path
                    / directory
                ).resolve(
                    strict=False
                )
            )
            or (
                root_path
                / directory
            ).resolve(
                strict=False
            ).is_relative_to(
                project_search_root
            )
        )
    ]

    if TARGET_NAME in files:
        found_paths.append(
            root_path
            / TARGET_NAME
        )


# 4. If still missing, search only the persistent project root.
if not found_paths:

    if PROJECT_ROOT.is_dir():

        for root, dirs, files in os.walk(
            PROJECT_ROOT
        ):
            if TARGET_NAME in files:
                found_paths.append(
                    Path(root)
                    / TARGET_NAME
                )


# Deduplicate.
found_paths = sorted(
    {
        str(
            Path(path)
        )
        for path in found_paths
    }
)


print(
    "MANIFEST_MATCH_COUNT =",
    len(found_paths),
)

for index, path in enumerate(
    found_paths
):
    print(
        f"MATCH_{index} =",
        path,
    )

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

MANIFEST_MATCH_COUNT = 1
MATCH_0 = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/transfer_v1/manifest_events_272.csv
LABELS_READ = False
GRAVITY_SPY_LABEL_VALUES_ACCESSED = False


In [37]:
# ============================================================
# NOTEBOOK 06 — E0
# CONTROLLED LABEL OPENING AFTER SPECTRAL FREEZE
#
# FIRST AUTHORIZED ACCESS TO GRAVITY SPY LABEL VALUES.
#
# Preconditions:
#   - both spectral partitions frozen in S7
#   - primary/sensitivity identities unchanged
#
# This cell:
#   - verifies S7 freeze
#   - verifies both frozen partition files
#   - reads labels for the first time
#   - verifies exact event alignment
#   - verifies 8583 labels / 22 classes
#
# NO metrics yet.
# NO repartitioning.
# NO model selection.
# NO changes to spectral artifacts.
# ============================================================

from pathlib import Path
import csv
import hashlib
import json

import numpy as np


# ------------------------------------------------------------
# 0. Runtime authorization gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS"
) is True, "S7 non disponibile o non PASS."

assert globals().get(
    "READY_FOR_LABEL_EVALUATION"
) is True, "Label evaluation non ancora autorizzata."

assert globals().get(
    "SPECTRAL_PARTITIONS_FROZEN"
) is True, "Partizioni spectral non congelate."

assert LABELS_READ is False
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is False


# ------------------------------------------------------------
# 1. Local hash helpers
# ------------------------------------------------------------

def e0_sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def e0_array_sha256(array):
    contiguous = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        contiguous.tobytes(order="C")
    ).hexdigest()


# ------------------------------------------------------------
# 2. Frozen S7 handoff
# ------------------------------------------------------------

E0_S7_FREEZE_PATH = (
    SPECTRAL_ROOT
    / "audits"
    / "sakai224_full_dual_spectral_after_S7.json"
)

EXPECTED_S7_FREEZE_SHA256 = (
    "7fb80c7bc65ca2d6b98b5bc59985a2cb"
    "629f9189231fb759ce2ba674e2382fe1"
)

assert E0_S7_FREEZE_PATH.is_file()

assert (
    e0_sha256_file(
        E0_S7_FREEZE_PATH
    )
    == EXPECTED_S7_FREEZE_SHA256
)

with E0_S7_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    e0_freeze = json.load(handle)


freeze_gate = (
    e0_freeze["status"]
        == "DUAL_SPECTRAL_PARTITIONS_FROZEN"

    and e0_freeze[
        "scientific_state"
    ][
        "both_partitions_frozen_before_labels"
    ] is True

    and e0_freeze[
        "scientific_state"
    ][
        "labels_read"
    ] is False

    and e0_freeze[
        "scientific_state"
    ][
        "primary_selected_before_labels"
    ] == "symmetric_figure"

    and e0_freeze[
        "scientific_state"
    ][
        "sensitivity_selected_before_labels"
    ] == "legacy_text"
)

assert freeze_gate


# ------------------------------------------------------------
# 3. Frozen partition identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_PARTITION_ARRAY_SHA256 = (
    "812d329469fefbc705f146cb6310afdcee"
    "60c13d4d23aebdc4f56c6faa1afd87"
)

EXPECTED_PRIMARY_PARTITION_FILE_SHA256 = (
    "5a01318f703441bbf291f7b32478043f"
    "6d0376dd81265078d6dbeac43e306ac0"
)

EXPECTED_LEGACY_PARTITION_ARRAY_SHA256 = (
    "bc4d0f8a7b69dce58b59d785d354f17b"
    "e320f3ef13688e4809e1c0b816f61a28"
)

EXPECTED_LEGACY_PARTITION_FILE_SHA256 = (
    "69edcfaf5006214c992ce39fb359d298"
    "41c78ab5fc13fe1f58901d06915a3632"
)


assert (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256
)

assert (
    e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)


e0_primary_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

e0_legacy_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)


partition_identity_gate = (
    e0_primary_partition.shape == (8583,)
    and e0_legacy_partition.shape == (8583,)

    and e0_primary_partition.dtype == np.int32
    and e0_legacy_partition.dtype == np.int32

    and e0_array_sha256(
        e0_primary_partition
    ) == EXPECTED_PRIMARY_PARTITION_ARRAY_SHA256

    and e0_array_sha256(
        e0_legacy_partition
    ) == EXPECTED_LEGACY_PARTITION_ARRAY_SHA256
)

assert partition_identity_gate


# ------------------------------------------------------------
# 4. Correct authoritative manifest path
#
# Until this point label VALUES still have not been accessed.
# ------------------------------------------------------------

E0_MANIFEST_PATH = (
    PROJECT_ROOT
    / "sakai224_full"
    / "transfer_v1"
    / "manifest_events_272.csv"
)

assert E0_MANIFEST_PATH.is_file()


# ------------------------------------------------------------
# 5. Frozen label-blind population already in RAM
# ------------------------------------------------------------

assert "full_iic_rows" in globals()
assert len(full_iic_rows) == 8583

population_gate = (
    all(
        full_iic_rows[i][
            "manifest_index"
        ] == i

        for i in range(8583)
    )

    and len({
        row["event_id"]
        for row in full_iic_rows
    }) == 8583
)

assert population_gate


# ------------------------------------------------------------
# 6. FIRST AUTHORIZED LABEL READ
# ------------------------------------------------------------

with E0_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as handle:

    reader = csv.DictReader(
        handle
    )

    assert reader.fieldnames is not None
    assert "event_id" in reader.fieldnames
    assert "label" in reader.fieldnames

    manifest_rows_with_labels = list(
        reader
    )


assert len(
    manifest_rows_with_labels
) == 8583


gravity_spy_labels = np.asarray(
    [
        row["label"].strip()
        for row
        in manifest_rows_with_labels
    ],
    dtype=object,
)


# ------------------------------------------------------------
# 7. Exact alignment with frozen spectral population
# ------------------------------------------------------------

label_alignment_gate = all(
    manifest_rows_with_labels[i][
        "event_id"
    ]
    == full_iic_rows[i][
        "event_id"
    ]

    for i in range(8583)
)

assert label_alignment_gate


# ------------------------------------------------------------
# 8. Label integrity
# ------------------------------------------------------------

nonempty_label_gate = all(
    isinstance(
        label,
        str,
    )
    and len(label) > 0

    for label
    in gravity_spy_labels
)

assert nonempty_label_gate


unique_labels = sorted(
    set(
        gravity_spy_labels.tolist()
    )
)

class_count_gate = (
    len(unique_labels) == 22
)

assert class_count_gate


label_counts = {
    label:
        int(
            np.sum(
                gravity_spy_labels
                == label
            )
        )

    for label
    in unique_labels
}

label_total_gate = (
    sum(
        label_counts.values()
    ) == 8583
)

assert label_total_gate


# ------------------------------------------------------------
# 9. Spectral partitions MUST remain unchanged after label read
# ------------------------------------------------------------

post_label_partition_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256

    and e0_array_sha256(
        e0_primary_partition
    )
    == EXPECTED_PRIMARY_PARTITION_ARRAY_SHA256

    and e0_array_sha256(
        e0_legacy_partition
    )
    == EXPECTED_LEGACY_PARTITION_ARRAY_SHA256
)

assert post_label_partition_gate


# ------------------------------------------------------------
# 10. Final state
# ------------------------------------------------------------

LABELS_READ = True
GRAVITY_SPY_LABEL_VALUES_ACCESSED = True

PARTITIONS_CHANGED_AFTER_LABELS = (
    not post_label_partition_gate
)

assert PARTITIONS_CHANGED_AFTER_LABELS is False

SPECTRAL_PARTITIONS_FROZEN = True

NOTEBOOK_06_E0_LABEL_OPENING_PASS = True
READY_FOR_E1_CONTINGENCY_AND_METRICS = True


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E0 CONTROLLED LABEL OPENING ==="
)

print(
    "S7_FREEZE_GATE =",
    freeze_gate,
)

print(
    "PARTITION_IDENTITY_GATE =",
    partition_identity_gate,
)

print(
    "POPULATION_GATE =",
    population_gate,
)

print(
    "LABEL_ALIGNMENT_GATE =",
    label_alignment_gate,
)

print(
    "MANIFEST_PATH =",
    E0_MANIFEST_PATH,
)

print(
    "MANIFEST_ROWS =",
    len(
        manifest_rows_with_labels
    ),
)

print(
    "LABEL_COUNT =",
    gravity_spy_labels.size,
)

print(
    "UNIQUE_CLASS_COUNT =",
    len(
        unique_labels
    ),
)

print(
    "NONEMPTY_LABEL_GATE =",
    nonempty_label_gate,
)

print(
    "LABEL_TOTAL_GATE =",
    label_total_gate,
)

print(
    "POST_LABEL_PARTITION_GATE =",
    post_label_partition_gate,
)

print(
    "LABELS_READ =",
    LABELS_READ,
)

print(
    "GRAVITY_SPY_LABEL_VALUES_ACCESSED =",
    GRAVITY_SPY_LABEL_VALUES_ACCESSED,
)

print(
    "PARTITIONS_CHANGED_AFTER_LABELS =",
    PARTITIONS_CHANGED_AFTER_LABELS,
)

print(
    "NOTEBOOK_06_E0_LABEL_OPENING_PASS =",
    NOTEBOOK_06_E0_LABEL_OPENING_PASS,
)

print(
    "READY_FOR_E1_CONTINGENCY_AND_METRICS =",
    READY_FOR_E1_CONTINGENCY_AND_METRICS,
)


=== NOTEBOOK 06 — E0 CONTROLLED LABEL OPENING ===
S7_FREEZE_GATE = True
PARTITION_IDENTITY_GATE = True
POPULATION_GATE = True
LABEL_ALIGNMENT_GATE = True
MANIFEST_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/transfer_v1/manifest_events_272.csv
MANIFEST_ROWS = 8583
LABEL_COUNT = 8583
UNIQUE_CLASS_COUNT = 22
NONEMPTY_LABEL_GATE = True
LABEL_TOTAL_GATE = True
POST_LABEL_PARTITION_GATE = True
LABELS_READ = True
GRAVITY_SPY_LABEL_VALUES_ACCESSED = True
PARTITIONS_CHANGED_AFTER_LABELS = False
NOTEBOOK_06_E0_LABEL_OPENING_PASS = True
READY_FOR_E1_CONTINGENCY_AND_METRICS = True


In [38]:
# ============================================================
# NOTEBOOK 06 — E1
# CONTINGENCY MATRICES + STANDARD EXTERNAL METRICS
#
# Label-based evaluation AFTER frozen spectral partitions.
#
# Metrics:
#   - Purity
#   - NMI arithmetic
#   - ARI
#   - Homogeneity
#   - Completeness
#
# PRIMARY and LEGACY evaluated independently.
#
# NO repartitioning.
# NO model selection.
# NO Sakai-like accuracy yet.
# ============================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
)


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E0_LABEL_OPENING_PASS"
) is True

assert globals().get(
    "READY_FOR_E1_CONTINGENCY_AND_METRICS"
) is True

assert LABELS_READ is True
assert GRAVITY_SPY_LABEL_VALUES_ACCESSED is True
assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False


# ------------------------------------------------------------
# 1. Frozen partition identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_PARTITION_FILE_SHA256 = (
    "5a01318f703441bbf291f7b32478043f"
    "6d0376dd81265078d6dbeac43e306ac0"
)

EXPECTED_LEGACY_PARTITION_FILE_SHA256 = (
    "69edcfaf5006214c992ce39fb359d298"
    "41c78ab5fc13fe1f58901d06915a3632"
)

assert (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256
)

assert (
    e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)


primary_eval_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

legacy_eval_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)

assert primary_eval_partition.shape == (8583,)
assert legacy_eval_partition.shape == (8583,)
assert gravity_spy_labels.shape == (8583,)
assert len(unique_labels) == 22


# ------------------------------------------------------------
# 2. Fixed contingency-matrix convention
#
# rows    = 22 Gravity Spy labels, alphabetical order
# columns = unsupervised cluster IDs 0..35
# ------------------------------------------------------------

E1_LABEL_ORDER = list(
    unique_labels
)

E1_CLUSTER_ORDER = list(
    range(36)
)

label_to_row = {
    label: index
    for index, label
    in enumerate(
        E1_LABEL_ORDER
    )
}


def build_contingency(
    labels,
    partition,
):
    matrix = np.zeros(
        (22, 36),
        dtype=np.int64,
    )

    for label, cluster in zip(
        labels,
        partition,
        strict=True,
    ):
        matrix[
            label_to_row[
                label
            ],
            int(cluster),
        ] += 1

    return matrix


primary_contingency = build_contingency(
    gravity_spy_labels,
    primary_eval_partition,
)

legacy_contingency = build_contingency(
    gravity_spy_labels,
    legacy_eval_partition,
)


contingency_gate = (
    primary_contingency.shape == (22, 36)
    and legacy_contingency.shape == (22, 36)

    and primary_contingency.dtype == np.int64
    and legacy_contingency.dtype == np.int64

    and int(
        primary_contingency.sum()
    ) == 8583

    and int(
        legacy_contingency.sum()
    ) == 8583

    and np.array_equal(
        primary_contingency.sum(
            axis=1
        ),
        legacy_contingency.sum(
            axis=1
        ),
    )

    and np.array_equal(
        primary_contingency.sum(
            axis=1
        ),
        np.asarray(
            [
                label_counts[label]
                for label
                in E1_LABEL_ORDER
            ],
            dtype=np.int64,
        ),
    )
)

assert contingency_gate


# ------------------------------------------------------------
# 3. Standard Purity
# ------------------------------------------------------------

def standard_purity(
    contingency,
):
    return float(
        contingency.max(
            axis=0
        ).sum()
        / contingency.sum()
    )


# ------------------------------------------------------------
# 4. Standard external metrics
# ------------------------------------------------------------

def external_metrics(
    labels,
    partition,
    contingency,
):
    return {
        "purity":
            standard_purity(
                contingency
            ),

        "nmi_arithmetic":
            float(
                normalized_mutual_info_score(
                    labels,
                    partition,
                    average_method="arithmetic",
                )
            ),

        "ari":
            float(
                adjusted_rand_score(
                    labels,
                    partition,
                )
            ),

        "homogeneity":
            float(
                homogeneity_score(
                    labels,
                    partition,
                )
            ),

        "completeness":
            float(
                completeness_score(
                    labels,
                    partition,
                )
            ),
    }


PRIMARY_EXTERNAL_METRICS = external_metrics(
    gravity_spy_labels,
    primary_eval_partition,
    primary_contingency,
)

LEGACY_EXTERNAL_METRICS = external_metrics(
    gravity_spy_labels,
    legacy_eval_partition,
    legacy_contingency,
)


metric_finite_gate = all(
    np.isfinite(value)

    for value
    in (
        list(
            PRIMARY_EXTERNAL_METRICS.values()
        )
        + list(
            LEGACY_EXTERNAL_METRICS.values()
        )
    )
)

assert metric_finite_gate


# ------------------------------------------------------------
# 5. Do NOT select winner / alter protocol
# ------------------------------------------------------------

PRIMARY_ROLE = "PRIMARY_PRESELECTED_BEFORE_LABELS"
LEGACY_ROLE = "SENSITIVITY_PRESELECTED_BEFORE_LABELS"

MODEL_SELECTION_PERFORMED = False
PARTITIONS_CHANGED_AFTER_LABELS = False


# ------------------------------------------------------------
# 6. Output directory
# ------------------------------------------------------------

EVALUATION_ROOT = (
    SPECTRAL_ROOT
    / "evaluation"
)

EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PRIMARY_CONTINGENCY_PATH = (
    EVALUATION_ROOT
    / "primary_symmetric_contingency_22x36.npy"
)

LEGACY_CONTINGENCY_PATH = (
    EVALUATION_ROOT
    / "legacy_text_contingency_22x36.npy"
)

E1_RESULTS_PATH = (
    EVALUATION_ROOT
    / "standard_external_metrics.json"
)


# ------------------------------------------------------------
# 7. Atomic NPY helper
# ------------------------------------------------------------

def e1_atomic_save_npy(
    path,
    array,
):
    partial = Path(
        str(path)
        + ".partial"
    )

    assert not partial.exists()

    with partial.open(
        "wb"
    ) as handle:

        np.save(
            handle,
            array,
            allow_pickle=False,
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        partial,
        path,
    )


# ------------------------------------------------------------
# 8. Persist / verify contingency matrices
# ------------------------------------------------------------

for path, array in [
    (
        PRIMARY_CONTINGENCY_PATH,
        primary_contingency,
    ),
    (
        LEGACY_CONTINGENCY_PATH,
        legacy_contingency,
    ),
]:

    if path.exists():

        existing = np.load(
            path,
            allow_pickle=False,
        )

        assert np.array_equal(
            existing,
            array,
        )

    else:

        e1_atomic_save_npy(
            path,
            array,
        )


PRIMARY_CONTINGENCY_FILE_SHA256 = (
    e0_sha256_file(
        PRIMARY_CONTINGENCY_PATH
    )
)

LEGACY_CONTINGENCY_FILE_SHA256 = (
    e0_sha256_file(
        LEGACY_CONTINGENCY_PATH
    )
)


# ------------------------------------------------------------
# 9. Results metadata
# ------------------------------------------------------------

E1_RESULTS = {
    "format_version":
        1,

    "status":
        "STANDARD_EXTERNAL_METRICS_COMPLETE",

    "population_size":
        8583,

    "gravity_spy_class_count":
        22,

    "cluster_count":
        36,

    "contingency_convention": {
        "rows":
            "Gravity_Spy_labels",

        "row_order":
            E1_LABEL_ORDER,

        "columns":
            "unsupervised_cluster_id_0_to_35",
    },

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "role":
            PRIMARY_ROLE,

        "partition_file_sha256":
            EXPECTED_PRIMARY_PARTITION_FILE_SHA256,

        "contingency_file_sha256":
            PRIMARY_CONTINGENCY_FILE_SHA256,

        "metrics":
            PRIMARY_EXTERNAL_METRICS,
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "role":
            LEGACY_ROLE,

        "partition_file_sha256":
            EXPECTED_LEGACY_PARTITION_FILE_SHA256,

        "contingency_file_sha256":
            LEGACY_CONTINGENCY_FILE_SHA256,

        "metrics":
            LEGACY_EXTERNAL_METRICS,
    },

    "metric_definitions": {
        "purity":
            "sum_c max_label N[label,c] / D",

        "nmi":
            "sklearn_normalized_mutual_info_score_average_method_arithmetic",

        "ari":
            "sklearn_adjusted_rand_score",

        "homogeneity":
            "sklearn_homogeneity_score",

        "completeness":
            "sklearn_completeness_score",
    },

    "sakai_like_accuracy_computed":
        False,

    "model_selection_performed":
        False,

    "partitions_changed_after_labels":
        False,
}


E1_RESULTS_JSON = json.dumps(
    E1_RESULTS,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


if E1_RESULTS_PATH.exists():

    with E1_RESULTS_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:

        existing_results = json.load(
            handle
        )

    assert (
        existing_results
        == E1_RESULTS
    )

else:

    atomic_write_text(
        E1_RESULTS_PATH,
        E1_RESULTS_JSON,
    )


E1_RESULTS_SHA256 = (
    e0_sha256_file(
        E1_RESULTS_PATH
    )
)


# ------------------------------------------------------------
# 10. Final immutability check
# ------------------------------------------------------------

partition_post_metrics_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_metrics_gate


# ------------------------------------------------------------
# 11. Final state
# ------------------------------------------------------------

LABELS_READ = True
GRAVITY_SPY_LABEL_VALUES_ACCESSED = True

NOTEBOOK_06_E1_STANDARD_METRICS_PASS = True
READY_FOR_E2_SAKAI_LIKE_ACCURACY_AUDIT = True


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E1 STANDARD EXTERNAL METRICS ==="
)

print()
print("--- PRIMARY symmetric_figure ---")

for key, value in (
    PRIMARY_EXTERNAL_METRICS.items()
):
    print(
        key.upper(),
        "=",
        value,
    )

print()
print("--- LEGACY sensitivity ---")

for key, value in (
    LEGACY_EXTERNAL_METRICS.items()
):
    print(
        key.upper(),
        "=",
        value,
    )

print()
print(
    "CONTINGENCY_GATE =",
    contingency_gate,
)

print(
    "METRIC_FINITE_GATE =",
    metric_finite_gate,
)

print(
    "MODEL_SELECTION_PERFORMED =",
    MODEL_SELECTION_PERFORMED,
)

print(
    "PARTITION_POST_METRICS_GATE =",
    partition_post_metrics_gate,
)

print(
    "PRIMARY_CONTINGENCY_SHA256 =",
    PRIMARY_CONTINGENCY_FILE_SHA256,
)

print(
    "LEGACY_CONTINGENCY_SHA256 =",
    LEGACY_CONTINGENCY_FILE_SHA256,
)

print(
    "E1_RESULTS_SHA256 =",
    E1_RESULTS_SHA256,
)

print(
    "NOTEBOOK_06_E1_STANDARD_METRICS_PASS =",
    NOTEBOOK_06_E1_STANDARD_METRICS_PASS,
)

print(
    "READY_FOR_E2_SAKAI_LIKE_ACCURACY_AUDIT =",
    READY_FOR_E2_SAKAI_LIKE_ACCURACY_AUDIT,
)


=== NOTEBOOK 06 — E1 STANDARD EXTERNAL METRICS ===

--- PRIMARY symmetric_figure ---
PURITY = 0.8264010252825352
NMI_ARITHMETIC = 0.7018325449722438
ARI = 0.4509773588198464
HOMOGENEITY = 0.7836699716771438
COMPLETENESS = 0.6354712757726401

--- LEGACY sensitivity ---
PURITY = 0.7980892461843179
NMI_ARITHMETIC = 0.6821419109736958
ARI = 0.33427224443938447
HOMOGENEITY = 0.7729849474767357
COMPLETENESS = 0.6104055888493044

CONTINGENCY_GATE = True
METRIC_FINITE_GATE = True
MODEL_SELECTION_PERFORMED = False
PARTITION_POST_METRICS_GATE = True
PRIMARY_CONTINGENCY_SHA256 = 3a892d4f73c30d214728d5d1bf50e8f57a7e12926d4ac4b8562ecbd8d0722511
LEGACY_CONTINGENCY_SHA256 = 7f2c2c00602857db7038ec2ea0624a0d4c14c60b462c6a27b020aecfcadf5469
E1_RESULTS_SHA256 = 5353d0d8e24fb1ac6d2e7cbe31f4adacfc3f628abd53cd52fe2c1017bbb75023
NOTEBOOK_06_E1_STANDARD_METRICS_PASS = True
READY_FOR_E2_SAKAI_LIKE_ACCURACY_AUDIT = True


In [39]:
# ============================================================
# NOTEBOOK 06 — E2
# SAKAI Eq.(1) ACCURACY AUDIT
#
# HTML PAPER:
#
#   A = sum_i max(v^(i)) / ||v^(i)||_1
#
# with C=36 and reported accuracy = 90.9%.
#
# Because the printed sum has range [0,C], while 90.9% is
# percentage-like, we report separately:
#
#   1. literal Eq.(1) sum
#   2. reconstructed Sakai-like macro mean = Eq.(1)/C
#   3. standard global purity already computed in E1
#
# No reinterpretation is hidden.
# ============================================================

from pathlib import Path
import json
import numpy as np


# ------------------------------------------------------------
# 0. Parent gate
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E1_STANDARD_METRICS_PASS"
) is True

assert globals().get(
    "READY_FOR_E2_SAKAI_LIKE_ACCURACY_AUDIT"
) is True

assert LABELS_READ is True
assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False


# ------------------------------------------------------------
# 1. PAPER reference
# ------------------------------------------------------------

SAKAI_REPORTED_C = 36
SAKAI_REPORTED_ACCURACY = 0.909

SAKAI_EQ1_PRINTED = (
    "sum_i max(v_i) / ||v_i||_1"
)

SAKAI_EQ1_NORMALIZATION_PRINTED = False


# ------------------------------------------------------------
# 2. Compute cluster-wise majority fractions
# ------------------------------------------------------------

def sakai_eq1_audit(
    contingency,
):
    contingency = np.asarray(
        contingency,
        dtype=np.int64,
    )

    assert contingency.shape == (22, 36)

    cluster_sizes = contingency.sum(
        axis=0
    )

    assert np.all(
        cluster_sizes > 0
    )

    majority_counts = contingency.max(
        axis=0
    )

    majority_fractions = (
        majority_counts.astype(
            np.float64
        )
        / cluster_sizes.astype(
            np.float64
        )
    )

    literal_sum = float(
        majority_fractions.sum()
    )

    macro_mean = float(
        majority_fractions.mean()
    )

    global_purity = float(
        majority_counts.sum()
        / contingency.sum()
    )

    return {
        "cluster_sizes":
            cluster_sizes,

        "majority_counts":
            majority_counts,

        "majority_fractions":
            majority_fractions,

        "literal_eq1_sum":
            literal_sum,

        "macro_mean_divide_by_C":
            macro_mean,

        "global_purity":
            global_purity,
    }


primary_eq1 = sakai_eq1_audit(
    primary_contingency
)

legacy_eq1 = sakai_eq1_audit(
    legacy_contingency
)


# ------------------------------------------------------------
# 3. Mathematical identity gates
# ------------------------------------------------------------

primary_identity_gate = (
    np.isclose(
        primary_eq1[
            "literal_eq1_sum"
        ] / 36.0,
        primary_eq1[
            "macro_mean_divide_by_C"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )

    and np.isclose(
        primary_eq1[
            "global_purity"
        ],
        PRIMARY_EXTERNAL_METRICS[
            "purity"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )
)

legacy_identity_gate = (
    np.isclose(
        legacy_eq1[
            "literal_eq1_sum"
        ] / 36.0,
        legacy_eq1[
            "macro_mean_divide_by_C"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )

    and np.isclose(
        legacy_eq1[
            "global_purity"
        ],
        LEGACY_EXTERNAL_METRICS[
            "purity"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )
)

assert primary_identity_gate
assert legacy_identity_gate


# ------------------------------------------------------------
# 4. Range / interpretation audit
# ------------------------------------------------------------

literal_formula_range_gate = (
    0.0
    <= primary_eq1[
        "literal_eq1_sum"
    ]
    <= 36.0

    and 0.0
    <= legacy_eq1[
        "literal_eq1_sum"
    ]
    <= 36.0
)

macro_percentage_range_gate = (
    0.0
    <= primary_eq1[
        "macro_mean_divide_by_C"
    ]
    <= 1.0

    and 0.0
    <= legacy_eq1[
        "macro_mean_divide_by_C"
    ]
    <= 1.0
)

assert literal_formula_range_gate
assert macro_percentage_range_gate


# ------------------------------------------------------------
# 5. Comparison with reported 90.9%
#
# Descriptive only.
# It is NOT used for model selection.
# ------------------------------------------------------------

PRIMARY_SAKAI_LIKE_MACRO = (
    primary_eq1[
        "macro_mean_divide_by_C"
    ]
)

LEGACY_SAKAI_LIKE_MACRO = (
    legacy_eq1[
        "macro_mean_divide_by_C"
    ]
)

PRIMARY_SAKAI_REPORTED_GAP = float(
    PRIMARY_SAKAI_LIKE_MACRO
    - SAKAI_REPORTED_ACCURACY
)

LEGACY_SAKAI_REPORTED_GAP = float(
    LEGACY_SAKAI_LIKE_MACRO
    - SAKAI_REPORTED_ACCURACY
)


# ------------------------------------------------------------
# 6. Persist audit
# ------------------------------------------------------------

E2_RESULTS_PATH = (
    EVALUATION_ROOT
    / "sakai_eq1_accuracy_audit.json"
)

E2_RESULTS = {
    "format_version":
        1,

    "status":
        "SAKAI_EQ1_AUDIT_COMPLETE",

    "source_authority":
        "Scientific_Reports_version_of_record_HTML",

    "paper": {
        "C":
            36,

        "gravity_spy_label_count":
            22,

        "printed_equation":
            SAKAI_EQ1_PRINTED,

        "explicit_1_over_C_in_printed_equation":
            False,

        "reported_accuracy":
            SAKAI_REPORTED_ACCURACY,
    },

    "interpretation": {
        "literal_eq1":
            "sum_of_cluster_majority_fractions",

        "literal_range":
            "[0,C]",

        "sakai_like_macro":
            "literal_eq1_divided_by_C",

        "reason":
            (
                "reported_90.9_percent_is_percentage_scaled_"
                "while_printed_sum_has_range_0_to_C"
            ),

        "claim":
            (
                "reconstructed_normalization_not_explicitly_"
                "printed_in_HTML"
            ),
    },

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "literal_eq1_sum":
            primary_eq1[
                "literal_eq1_sum"
            ],

        "sakai_like_macro":
            PRIMARY_SAKAI_LIKE_MACRO,

        "standard_global_purity":
            primary_eq1[
                "global_purity"
            ],

        "gap_vs_reported_0_909":
            PRIMARY_SAKAI_REPORTED_GAP,
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "literal_eq1_sum":
            legacy_eq1[
                "literal_eq1_sum"
            ],

        "sakai_like_macro":
            LEGACY_SAKAI_LIKE_MACRO,

        "standard_global_purity":
            legacy_eq1[
                "global_purity"
            ],

        "gap_vs_reported_0_909":
            LEGACY_SAKAI_REPORTED_GAP,
    },

    "model_selection_performed":
        False,

    "partitions_changed_after_labels":
        False,
}


E2_RESULTS_JSON = json.dumps(
    E2_RESULTS,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


if E2_RESULTS_PATH.exists():

    with E2_RESULTS_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:

        existing_e2 = json.load(
            handle
        )

    assert existing_e2 == E2_RESULTS

else:

    atomic_write_text(
        E2_RESULTS_PATH,
        E2_RESULTS_JSON,
    )


E2_RESULTS_SHA256 = e0_sha256_file(
    E2_RESULTS_PATH
)


# ------------------------------------------------------------
# 7. Partition immutability
# ------------------------------------------------------------

partition_post_e2_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e2_gate


# ------------------------------------------------------------
# 8. Final state
# ------------------------------------------------------------

MODEL_SELECTION_PERFORMED = False
PARTITIONS_CHANGED_AFTER_LABELS = False

NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS = True
READY_FOR_E3_CONTINGENCY_INTERPRETATION = True


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E2 SAKAI Eq.(1) AUDIT ==="
)

print()
print("--- PAPER HTML ---")
print(
    "PRINTED_EQ1 =",
    SAKAI_EQ1_PRINTED,
)
print(
    "EXPLICIT_1_OVER_C_PRINTED =",
    SAKAI_EQ1_NORMALIZATION_PRINTED,
)
print(
    "REPORTED_ACCURACY =",
    SAKAI_REPORTED_ACCURACY,
)

print()
print("--- PRIMARY symmetric_figure ---")
print(
    "LITERAL_EQ1_SUM =",
    primary_eq1[
        "literal_eq1_sum"
    ],
)
print(
    "SAKAI_LIKE_MACRO =",
    PRIMARY_SAKAI_LIKE_MACRO,
)
print(
    "STANDARD_GLOBAL_PURITY =",
    primary_eq1[
        "global_purity"
    ],
)
print(
    "GAP_VS_REPORTED_0_909 =",
    PRIMARY_SAKAI_REPORTED_GAP,
)

print()
print("--- LEGACY sensitivity ---")
print(
    "LITERAL_EQ1_SUM =",
    legacy_eq1[
        "literal_eq1_sum"
    ],
)
print(
    "SAKAI_LIKE_MACRO =",
    LEGACY_SAKAI_LIKE_MACRO,
)
print(
    "STANDARD_GLOBAL_PURITY =",
    legacy_eq1[
        "global_purity"
    ],
)
print(
    "GAP_VS_REPORTED_0_909 =",
    LEGACY_SAKAI_REPORTED_GAP,
)

print()
print(
    "PRIMARY_IDENTITY_GATE =",
    primary_identity_gate,
)
print(
    "LEGACY_IDENTITY_GATE =",
    legacy_identity_gate,
)
print(
    "PARTITION_POST_E2_GATE =",
    partition_post_e2_gate,
)
print(
    "MODEL_SELECTION_PERFORMED =",
    MODEL_SELECTION_PERFORMED,
)
print(
    "E2_RESULTS_SHA256 =",
    E2_RESULTS_SHA256,
)
print(
    "NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS =",
    NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS,
)
print(
    "READY_FOR_E3_CONTINGENCY_INTERPRETATION =",
    READY_FOR_E3_CONTINGENCY_INTERPRETATION,
)


=== NOTEBOOK 06 — E2 SAKAI Eq.(1) AUDIT ===

--- PAPER HTML ---
PRINTED_EQ1 = sum_i max(v_i) / ||v_i||_1
EXPLICIT_1_OVER_C_PRINTED = False
REPORTED_ACCURACY = 0.909

--- PRIMARY symmetric_figure ---
LITERAL_EQ1_SUM = 30.649094333132975
SAKAI_LIKE_MACRO = 0.8513637314759159
STANDARD_GLOBAL_PURITY = 0.8264010252825352
GAP_VS_REPORTED_0_909 = -0.0576362685240841

--- LEGACY sensitivity ---
LITERAL_EQ1_SUM = 30.07559626864015
SAKAI_LIKE_MACRO = 0.8354332296844487
STANDARD_GLOBAL_PURITY = 0.7980892461843179
GAP_VS_REPORTED_0_909 = -0.07356677031555137

PRIMARY_IDENTITY_GATE = True
LEGACY_IDENTITY_GATE = True
PARTITION_POST_E2_GATE = True
MODEL_SELECTION_PERFORMED = False
E2_RESULTS_SHA256 = ab07670afcaeb3b9827873d203ee18b3d72b3ec68fd32d2dbe395ed20ab1fc9e
NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS = True
READY_FOR_E3_CONTINGENCY_INTERPRETATION = True


In [40]:
# ============================================================
# NOTEBOOK 06 — E3
# CONTINGENCY INTERPRETATION TABLES
#
# Produces deterministic cluster- and label-level summaries for
# PRIMARY and LEGACY, plus permutation-invariant agreement.
#
# NO repartitioning.
# NO model selection.
# NO image selection yet.
# ============================================================

from pathlib import Path
import csv
import io
import json

import numpy as np
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS"
) is True

assert globals().get(
    "READY_FOR_E3_CONTINGENCY_INTERPRETATION"
) is True

assert LABELS_READ is True
assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False
assert MODEL_SELECTION_PERFORMED is False


# ------------------------------------------------------------
# 1. Frozen inputs
# ------------------------------------------------------------

EXPECTED_PRIMARY_PARTITION_FILE_SHA256 = (
    "5a01318f703441bbf291f7b32478043f"
    "6d0376dd81265078d6dbeac43e306ac0"
)

EXPECTED_LEGACY_PARTITION_FILE_SHA256 = (
    "69edcfaf5006214c992ce39fb359d298"
    "41c78ab5fc13fe1f58901d06915a3632"
)

EXPECTED_PRIMARY_CONTINGENCY_FILE_SHA256 = (
    "3a892d4f73c30d214728d5d1bf50e8f"
    "57a7e12926d4ac4b8562ecbd8d0722511"
)

EXPECTED_LEGACY_CONTINGENCY_FILE_SHA256 = (
    "7f2c2c00602857db7038ec2ea0624a0d4"
    "c14c60b462c6a27b020aecfcadf5469"
)


assert e0_sha256_file(
    PRIMARY_PARTITION_PATH
) == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

assert e0_sha256_file(
    LEGACY_PARTITION_PATH
) == EXPECTED_LEGACY_PARTITION_FILE_SHA256

assert e0_sha256_file(
    PRIMARY_CONTINGENCY_PATH
) == EXPECTED_PRIMARY_CONTINGENCY_FILE_SHA256

assert e0_sha256_file(
    LEGACY_CONTINGENCY_PATH
) == EXPECTED_LEGACY_CONTINGENCY_FILE_SHA256


primary_part = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

legacy_part = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)

primary_cont = np.load(
    PRIMARY_CONTINGENCY_PATH,
    allow_pickle=False,
)

legacy_cont = np.load(
    LEGACY_CONTINGENCY_PATH,
    allow_pickle=False,
)


assert primary_part.shape == legacy_part.shape == (8583,)
assert primary_cont.shape == legacy_cont.shape == (22, 36)
assert len(E1_LABEL_ORDER) == 22


# ------------------------------------------------------------
# 2. Deterministic summaries
#
# Tie-breaks:
#   dominant label   -> alphabetical
#   dominant cluster -> smallest cluster ID
# ------------------------------------------------------------

def cluster_summary(
    contingency,
):
    rows = []

    for cluster_id in range(36):

        counts = contingency[
            :,
            cluster_id,
        ]

        size = int(
            counts.sum()
        )

        assert size > 0

        order = sorted(
            range(22),
            key=lambda index: (
                -int(
                    counts[index]
                ),
                E1_LABEL_ORDER[index],
            ),
        )

        first, second = order[:2]

        first_count = int(
            counts[first]
        )

        second_count = int(
            counts[second]
        )

        rows.append({
            "cluster_id":
                cluster_id,

            "cluster_size":
                size,

            "dominant_label":
                E1_LABEL_ORDER[first],

            "dominant_count":
                first_count,

            "dominant_fraction":
                float(
                    first_count
                    / size
                ),

            "second_label":
                E1_LABEL_ORDER[second],

            "second_count":
                second_count,

            "second_fraction":
                float(
                    second_count
                    / size
                ),

            "occupied_label_count":
                int(
                    np.count_nonzero(
                        counts
                    )
                ),
        })

    return rows


def label_summary(
    contingency,
):
    rows = []

    for label_index, label in enumerate(
        E1_LABEL_ORDER
    ):

        counts = contingency[
            label_index,
            :,
        ]

        total = int(
            counts.sum()
        )

        assert total > 0

        order = sorted(
            range(36),
            key=lambda cluster_id: (
                -int(
                    counts[
                        cluster_id
                    ]
                ),
                cluster_id,
            ),
        )

        first, second = order[:2]

        first_count = int(
            counts[first]
        )

        second_count = int(
            counts[second]
        )

        rows.append({
            "label":
                label,

            "label_count":
                total,

            "dominant_cluster_id":
                first,

            "dominant_cluster_count":
                first_count,

            "dominant_cluster_fraction":
                float(
                    first_count
                    / total
                ),

            "second_cluster_id":
                second,

            "second_cluster_count":
                second_count,

            "second_cluster_fraction":
                float(
                    second_count
                    / total
                ),

            "occupied_cluster_count":
                int(
                    np.count_nonzero(
                        counts
                    )
                ),
        })

    return rows


PRIMARY_CLUSTER_SUMMARY = (
    cluster_summary(
        primary_cont
    )
)

LEGACY_CLUSTER_SUMMARY = (
    cluster_summary(
        legacy_cont
    )
)

PRIMARY_LABEL_SUMMARY = (
    label_summary(
        primary_cont
    )
)

LEGACY_LABEL_SUMMARY = (
    label_summary(
        legacy_cont
    )
)


PRIMARY_SUMMARY_PURITY = float(
    sum(
        row[
            "dominant_count"
        ]
        for row
        in PRIMARY_CLUSTER_SUMMARY
    )
    / 8583
)

LEGACY_SUMMARY_PURITY = float(
    sum(
        row[
            "dominant_count"
        ]
        for row
        in LEGACY_CLUSTER_SUMMARY
    )
    / 8583
)


summary_gate = (
    len(
        PRIMARY_CLUSTER_SUMMARY
    ) == 36

    and len(
        LEGACY_CLUSTER_SUMMARY
    ) == 36

    and len(
        PRIMARY_LABEL_SUMMARY
    ) == 22

    and len(
        LEGACY_LABEL_SUMMARY
    ) == 22

    and np.isclose(
        PRIMARY_SUMMARY_PURITY,
        PRIMARY_EXTERNAL_METRICS[
            "purity"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )

    and np.isclose(
        LEGACY_SUMMARY_PURITY,
        LEGACY_EXTERNAL_METRICS[
            "purity"
        ],
        rtol=0.0,
        atol=1.0e-15,
    )
)

assert summary_gate


# ------------------------------------------------------------
# 3. Primary-vs-legacy partition agreement
#
# Cluster IDs are nominal, therefore permutation-invariant
# metrics are used.
# ------------------------------------------------------------

PRIMARY_LEGACY_PARTITION_ARI = float(
    adjusted_rand_score(
        primary_part,
        legacy_part,
    )
)

PRIMARY_LEGACY_PARTITION_NMI = float(
    normalized_mutual_info_score(
        primary_part,
        legacy_part,
        average_method="arithmetic",
    )
)


partition_agreement_gate = (
    np.isfinite(
        PRIMARY_LEGACY_PARTITION_ARI
    )

    and np.isfinite(
        PRIMARY_LEGACY_PARTITION_NMI
    )

    and -1.0
        <= PRIMARY_LEGACY_PARTITION_ARI
        <= 1.0

    and 0.0
        <= PRIMARY_LEGACY_PARTITION_NMI
        <= 1.0
)

assert partition_agreement_gate


# ------------------------------------------------------------
# 4. Stable CSV serialization
# ------------------------------------------------------------

def csv_text(
    records,
    fields,
):
    stream = io.StringIO(
        newline=""
    )

    writer = csv.DictWriter(
        stream,
        fieldnames=fields,
        lineterminator="\n",
    )

    writer.writeheader()

    for record in records:

        writer.writerow({
            key: (
                format(
                    value,
                    ".17g",
                )
                if isinstance(
                    value,
                    float,
                )
                else value
            )
            for key, value
            in record.items()
        })

    return stream.getvalue()


def write_or_verify(
    path,
    text,
):
    if path.exists():

        assert path.read_text(
            encoding="utf-8"
        ) == text

    else:

        atomic_write_text(
            path,
            text,
        )


E3_ROOT = (
    EVALUATION_ROOT
    / "interpretation"
)

E3_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


cluster_fields = [
    "cluster_id",
    "cluster_size",
    "dominant_label",
    "dominant_count",
    "dominant_fraction",
    "second_label",
    "second_count",
    "second_fraction",
    "occupied_label_count",
]

label_fields = [
    "label",
    "label_count",
    "dominant_cluster_id",
    "dominant_cluster_count",
    "dominant_cluster_fraction",
    "second_cluster_id",
    "second_cluster_count",
    "second_cluster_fraction",
    "occupied_cluster_count",
]


paths = {
    "primary_cluster":
        E3_ROOT
        / "primary_cluster_summary.csv",

    "legacy_cluster":
        E3_ROOT
        / "legacy_cluster_summary.csv",

    "primary_label":
        E3_ROOT
        / "primary_label_summary.csv",

    "legacy_label":
        E3_ROOT
        / "legacy_label_summary.csv",

    "metadata":
        E3_ROOT
        / "cluster_label_interpretation.json",
}


write_or_verify(
    paths[
        "primary_cluster"
    ],
    csv_text(
        PRIMARY_CLUSTER_SUMMARY,
        cluster_fields,
    ),
)

write_or_verify(
    paths[
        "legacy_cluster"
    ],
    csv_text(
        LEGACY_CLUSTER_SUMMARY,
        cluster_fields,
    ),
)

write_or_verify(
    paths[
        "primary_label"
    ],
    csv_text(
        PRIMARY_LABEL_SUMMARY,
        label_fields,
    ),
)

write_or_verify(
    paths[
        "legacy_label"
    ],
    csv_text(
        LEGACY_LABEL_SUMMARY,
        label_fields,
    ),
)


# ------------------------------------------------------------
# 5. Persist metadata
# ------------------------------------------------------------

E3_RESULTS = {
    "format_version":
        1,

    "status":
        "CLUSTER_LABEL_INTERPRETATION_COMPLETE",

    "population_size":
        8583,

    "label_count":
        22,

    "cluster_count":
        36,

    "cluster_ids_are_nominal":
        True,

    "tie_breaks": {
        "dominant_label":
            "alphabetical",

        "dominant_cluster":
            "smallest_cluster_id",
    },

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "role":
            "PRIMARY_PRESELECTED_BEFORE_LABELS",

        "purity":
            PRIMARY_SUMMARY_PURITY,

        "cluster_summary_sha256":
            e0_sha256_file(
                paths[
                    "primary_cluster"
                ]
            ),

        "label_summary_sha256":
            e0_sha256_file(
                paths[
                    "primary_label"
                ]
            ),
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "role":
            "SENSITIVITY_PRESELECTED_BEFORE_LABELS",

        "purity":
            LEGACY_SUMMARY_PURITY,

        "cluster_summary_sha256":
            e0_sha256_file(
                paths[
                    "legacy_cluster"
                ]
            ),

        "label_summary_sha256":
            e0_sha256_file(
                paths[
                    "legacy_label"
                ]
            ),
    },

    "partition_agreement": {
        "ari":
            PRIMARY_LEGACY_PARTITION_ARI,

        "nmi_arithmetic":
            PRIMARY_LEGACY_PARTITION_NMI,
    },

    "model_selection_performed":
        False,

    "partitions_changed_after_labels":
        False,
}


E3_RESULTS_JSON = json.dumps(
    E3_RESULTS,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

write_or_verify(
    paths[
        "metadata"
    ],
    E3_RESULTS_JSON,
)

E3_RESULTS_SHA256 = e0_sha256_file(
    paths[
        "metadata"
    ]
)


# ------------------------------------------------------------
# 6. Concise report views
# ------------------------------------------------------------

def largest_five(
    records,
):
    ordered = sorted(
        records,
        key=lambda row: (
            -row[
                "cluster_size"
            ],
            row[
                "cluster_id"
            ],
        ),
    )

    return [
        (
            row[
                "cluster_id"
            ],
            row[
                "cluster_size"
            ],
            row[
                "dominant_label"
            ],
            row[
                "dominant_fraction"
            ],
        )
        for row
        in ordered[:5]
    ]


def most_fragmented_five(
    records,
):
    ordered = sorted(
        records,
        key=lambda row: (
            row[
                "dominant_cluster_fraction"
            ],
            row[
                "label"
            ],
        ),
    )

    return [
        (
            row[
                "label"
            ],
            row[
                "label_count"
            ],
            row[
                "dominant_cluster_id"
            ],
            row[
                "dominant_cluster_fraction"
            ],
            row[
                "occupied_cluster_count"
            ],
        )
        for row
        in ordered[:5]
    ]


# ------------------------------------------------------------
# 7. Final immutability
# ------------------------------------------------------------

partition_post_e3_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e3_gate


MODEL_SELECTION_PERFORMED = False
PARTITIONS_CHANGED_AFTER_LABELS = False

NOTEBOOK_06_E3_INTERPRETATION_TABLES_PASS = True
READY_FOR_E4_FIGURES_AND_REPRESENTATIVES = True


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E3 INTERPRETATION TABLES ==="
)

print(
    "SUMMARY_GATE =",
    summary_gate,
)

print(
    "PRIMARY_SUMMARY_PURITY =",
    PRIMARY_SUMMARY_PURITY,
)

print(
    "LEGACY_SUMMARY_PURITY =",
    LEGACY_SUMMARY_PURITY,
)

print(
    "PRIMARY_LEGACY_PARTITION_ARI =",
    PRIMARY_LEGACY_PARTITION_ARI,
)

print(
    "PRIMARY_LEGACY_PARTITION_NMI =",
    PRIMARY_LEGACY_PARTITION_NMI,
)

print(
    "PRIMARY_LARGEST_CLUSTERS =",
    largest_five(
        PRIMARY_CLUSTER_SUMMARY
    ),
)

print(
    "LEGACY_LARGEST_CLUSTERS =",
    largest_five(
        LEGACY_CLUSTER_SUMMARY
    ),
)

print(
    "PRIMARY_MOST_FRAGMENTED_LABELS =",
    most_fragmented_five(
        PRIMARY_LABEL_SUMMARY
    ),
)

print(
    "LEGACY_MOST_FRAGMENTED_LABELS =",
    most_fragmented_five(
        LEGACY_LABEL_SUMMARY
    ),
)

print(
    "PARTITION_POST_E3_GATE =",
    partition_post_e3_gate,
)

print(
    "MODEL_SELECTION_PERFORMED =",
    MODEL_SELECTION_PERFORMED,
)

print(
    "E3_RESULTS_SHA256 =",
    E3_RESULTS_SHA256,
)

print(
    "NOTEBOOK_06_E3_INTERPRETATION_TABLES_PASS =",
    NOTEBOOK_06_E3_INTERPRETATION_TABLES_PASS,
)

print(
    "READY_FOR_E4_FIGURES_AND_REPRESENTATIVES =",
    READY_FOR_E4_FIGURES_AND_REPRESENTATIVES,
)


=== NOTEBOOK 06 — E3 INTERPRETATION TABLES ===
SUMMARY_GATE = True
PRIMARY_SUMMARY_PURITY = 0.8264010252825352
LEGACY_SUMMARY_PURITY = 0.7980892461843179
PRIMARY_LEGACY_PARTITION_ARI = 0.4725572203674846
PRIMARY_LEGACY_PARTITION_NMI = 0.7519546514674184
PRIMARY_LARGEST_CLUSTERS = [(26, 1435, 'Blip', 0.7763066202090593), (5, 412, 'Blip', 0.9466019417475728), (11, 338, 'Low_Frequency_Burst', 1.0), (3, 334, 'Violin_Mode', 0.6017964071856288), (9, 326, 'Koi_Fish', 0.9877300613496932)]
LEGACY_LARGEST_CLUSTERS = [(1, 883, 'Blip', 0.5889014722536806), (34, 436, 'Blip', 0.6055045871559633), (20, 402, 'Blip', 0.9502487562189055), (3, 384, 'Low_Frequency_Burst', 1.0), (7, 380, '1080Lines', 0.45263157894736844)]
PRIMARY_MOST_FRAGMENTED_LABELS = [('Wandering_Line', 44, 2, 0.2727272727272727, 9), ('Light_Modulation', 573, 0, 0.3036649214659686, 10), ('Extremely_Loud', 454, 14, 0.33480176211453744, 12), ('None_of_the_Above', 88, 19, 0.3409090909090909, 11), ('Whistle', 305, 18, 0.3639344262295082, 

In [41]:
# ============================================================
# NOTEBOOK 06 — E4A
# CONTINGENCY FIGURES
#
# Produces:
#   - raw-count heatmap PRIMARY
#   - row-normalized heatmap PRIMARY
#   - raw-count heatmap LEGACY
#   - row-normalized heatmap LEGACY
#
# NO repartitioning.
# NO representative-image selection yet.
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E3_INTERPRETATION_TABLES_PASS"
) is True

assert globals().get(
    "READY_FOR_E4_FIGURES_AND_REPRESENTATIVES"
) is True

assert LABELS_READ is True
assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False
assert MODEL_SELECTION_PERFORMED is False


# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

E4_FIGURE_ROOT = (
    EVALUATION_ROOT
    / "figures"
)

E4_FIGURE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Normalized contingency
# ------------------------------------------------------------

def row_normalized_contingency(
    contingency,
):
    matrix = np.asarray(
        contingency,
        dtype=np.float64,
    )

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    assert np.all(
        row_sums > 0
    )

    return (
        matrix
        / row_sums
    )


primary_contingency_row_norm = (
    row_normalized_contingency(
        primary_contingency
    )
)

legacy_contingency_row_norm = (
    row_normalized_contingency(
        legacy_contingency
    )
)


normalization_gate = (
    np.allclose(
        primary_contingency_row_norm.sum(
            axis=1
        ),
        1.0,
        rtol=0.0,
        atol=1.0e-14,
    )

    and np.allclose(
        legacy_contingency_row_norm.sum(
            axis=1
        ),
        1.0,
        rtol=0.0,
        atol=1.0e-14,
    )
)

assert normalization_gate


# ------------------------------------------------------------
# 3. Plot helper
# ------------------------------------------------------------

def save_contingency_heatmap(
    matrix,
    *,
    title,
    colorbar_label,
    output_path,
    value_format,
):
    fig, ax = plt.subplots(
        figsize=(15, 8)
    )

    image = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
    )

    ax.set_title(
        title
    )

    ax.set_xlabel(
        "Cluster spectral non supervisionato"
    )

    ax.set_ylabel(
        "Label Gravity Spy"
    )

    ax.set_xticks(
        np.arange(36)
    )

    ax.set_xticklabels(
        [
            str(i)
            for i in range(36)
        ],
        rotation=90,
        fontsize=7,
    )

    ax.set_yticks(
        np.arange(22)
    )

    ax.set_yticklabels(
        E1_LABEL_ORDER,
        fontsize=8,
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
    )

    colorbar.set_label(
        colorbar_label
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    assert output_path.is_file()


# ------------------------------------------------------------
# 4. Paths
# ------------------------------------------------------------

PRIMARY_RAW_FIGURE_PATH = (
    E4_FIGURE_ROOT
    / "primary_symmetric_contingency_counts.png"
)

PRIMARY_NORM_FIGURE_PATH = (
    E4_FIGURE_ROOT
    / "primary_symmetric_contingency_row_normalized.png"
)

LEGACY_RAW_FIGURE_PATH = (
    E4_FIGURE_ROOT
    / "legacy_text_contingency_counts.png"
)

LEGACY_NORM_FIGURE_PATH = (
    E4_FIGURE_ROOT
    / "legacy_text_contingency_row_normalized.png"
)


# ------------------------------------------------------------
# 5. Produce figures
# ------------------------------------------------------------

save_contingency_heatmap(
    primary_contingency,
    title=(
        "PRIMARY symmetric_figure — "
        "matrice di contingenza 22×36"
    ),
    colorbar_label="Numero di eventi",
    output_path=PRIMARY_RAW_FIGURE_PATH,
    value_format="count",
)

save_contingency_heatmap(
    primary_contingency_row_norm,
    title=(
        "PRIMARY symmetric_figure — "
        "contingenza normalizzata per label"
    ),
    colorbar_label="Frazione della label",
    output_path=PRIMARY_NORM_FIGURE_PATH,
    value_format="fraction",
)

save_contingency_heatmap(
    legacy_contingency,
    title=(
        "SENSITIVITY legacy_text — "
        "matrice di contingenza 22×36"
    ),
    colorbar_label="Numero di eventi",
    output_path=LEGACY_RAW_FIGURE_PATH,
    value_format="count",
)

save_contingency_heatmap(
    legacy_contingency_row_norm,
    title=(
        "SENSITIVITY legacy_text — "
        "contingenza normalizzata per label"
    ),
    colorbar_label="Frazione della label",
    output_path=LEGACY_NORM_FIGURE_PATH,
    value_format="fraction",
)


# ------------------------------------------------------------
# 6. Hash figures
# ------------------------------------------------------------

E4A_FIGURE_HASHES = {
    "primary_counts":
        e0_sha256_file(
            PRIMARY_RAW_FIGURE_PATH
        ),

    "primary_row_normalized":
        e0_sha256_file(
            PRIMARY_NORM_FIGURE_PATH
        ),

    "legacy_counts":
        e0_sha256_file(
            LEGACY_RAW_FIGURE_PATH
        ),

    "legacy_row_normalized":
        e0_sha256_file(
            LEGACY_NORM_FIGURE_PATH
        ),
}


# ------------------------------------------------------------
# 7. Persist metadata
# ------------------------------------------------------------

E4A_METADATA_PATH = (
    E4_FIGURE_ROOT
    / "contingency_figures_metadata.json"
)

E4A_METADATA = {
    "format_version":
        1,

    "status":
        "CONTINGENCY_FIGURES_COMPLETE",

    "row_order":
        E1_LABEL_ORDER,

    "column_order":
        list(
            range(36)
        ),

    "figures_sha256":
        E4A_FIGURE_HASHES,

    "primary_partition_sha256":
        EXPECTED_PRIMARY_PARTITION_FILE_SHA256,

    "legacy_partition_sha256":
        EXPECTED_LEGACY_PARTITION_FILE_SHA256,

    "model_selection_performed":
        False,

    "partitions_changed_after_labels":
        False,
}


E4A_METADATA_JSON = json.dumps(
    E4A_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


if E4A_METADATA_PATH.exists():

    with E4A_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:

        existing_e4a = json.load(
            handle
        )

    assert existing_e4a == E4A_METADATA

else:

    atomic_write_text(
        E4A_METADATA_PATH,
        E4A_METADATA_JSON,
    )


E4A_METADATA_SHA256 = (
    e0_sha256_file(
        E4A_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 8. Partition immutability
# ------------------------------------------------------------

partition_post_e4a_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e4a_gate


# ------------------------------------------------------------
# 9. Final state
# ------------------------------------------------------------

NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS = True
READY_FOR_E4B_REPRESENTATIVE_SELECTION_PREFLIGHT = True


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E4A CONTINGENCY FIGURES ==="
)

print(
    "NORMALIZATION_GATE =",
    normalization_gate,
)

print(
    "PRIMARY_COUNTS_FIGURE =",
    PRIMARY_RAW_FIGURE_PATH,
)

print(
    "PRIMARY_NORMALIZED_FIGURE =",
    PRIMARY_NORM_FIGURE_PATH,
)

print(
    "LEGACY_COUNTS_FIGURE =",
    LEGACY_RAW_FIGURE_PATH,
)

print(
    "LEGACY_NORMALIZED_FIGURE =",
    LEGACY_NORM_FIGURE_PATH,
)

print(
    "PARTITION_POST_E4A_GATE =",
    partition_post_e4a_gate,
)

print(
    "E4A_METADATA_SHA256 =",
    E4A_METADATA_SHA256,
)

print(
    "NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS =",
    NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS,
)

print(
    "READY_FOR_E4B_REPRESENTATIVE_SELECTION_PREFLIGHT =",
    READY_FOR_E4B_REPRESENTATIVE_SELECTION_PREFLIGHT,
)


=== NOTEBOOK 06 — E4A CONTINGENCY FIGURES ===
NORMALIZATION_GATE = True
PRIMARY_COUNTS_FIGURE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/figures/primary_symmetric_contingency_counts.png
PRIMARY_NORMALIZED_FIGURE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/figures/primary_symmetric_contingency_row_normalized.png
LEGACY_COUNTS_FIGURE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/figures/legacy_text_contingency_counts.png
LEGACY_NORMALIZED_FIGURE = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/figures/legacy_text_contingency_row_normalized.png
PARTITION_POST_E4A_GATE = True
E4A_METADATA_SHA256 = 881cfd76e7c1d649ec5b729d3def55f3d5bc59a10fa4214bd4524a3dc5e70fd7
NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS = True
READY_FOR_E4B_REPRESENTATIVE_SELECTION_PREFLIGHT = True


In [42]:
# ============================================================
# NOTEBOOK 06 — E4B
# REPRESENTATIVE-SELECTION PREFLIGHT
#
# PROJECT rule:
#   frozen central mu512
#   -> L2-normalize each event vector
#   -> mean direction within spectral cluster
#   -> normalize prototype
#   -> cosine similarity event <-> prototype
#   -> descending ranking
#   -> tie-break: smaller manifest_index
#
# Labels are NOT used for ranking.
# ============================================================

import hashlib
import json
import numpy as np


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS"
) is True

assert globals().get(
    "READY_FOR_E4B_REPRESENTATIVE_SELECTION_PREFLIGHT"
) is True

assert LABELS_READ is True
assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False
assert MODEL_SELECTION_PERFORMED is False


# ------------------------------------------------------------
# 1. Frozen inputs
# ------------------------------------------------------------

EXPECTED_FULL_CENTRAL_MU_SHA256 = (
    "fd55be588ef05df8afdf7ee048aa535e"
    "12eda37bd13ef8d59e5968fcfc9467f8"
)

assert (
    FULL_CENTRAL_MU_SHA256
    == EXPECTED_FULL_CENTRAL_MU_SHA256
)

assert full_central_mu.shape == (8583, 512)
assert full_central_mu.dtype == np.float32
assert np.isfinite(full_central_mu).all()


e4b_primary_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

e4b_legacy_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)

assert e4b_primary_partition.shape == (8583,)
assert e4b_legacy_partition.shape == (8583,)


# ------------------------------------------------------------
# 2. PROJECT selection rule
# ------------------------------------------------------------

REPRESENTATIVE_TOP_N = 5


def rank_cluster_representatives(
    mu,
    partition,
    cluster_id,
):
    members = np.flatnonzero(
        partition == cluster_id
    ).astype(
        np.int64
    )

    assert members.size > 0

    X = np.asarray(
        mu[members],
        dtype=np.float64,
    )

    norms = np.linalg.norm(
        X,
        axis=1,
    )

    assert np.isfinite(norms).all()
    assert np.all(norms > 0.0)

    # Unit vectors: cosine geometry.
    X_unit = (
        X
        / norms[:, None]
    )

    # PROJECT prototype = normalized mean direction.
    prototype = X_unit.mean(
        axis=0
    )

    prototype_norm = float(
        np.linalg.norm(
            prototype
        )
    )

    assert (
        np.isfinite(
            prototype_norm
        )
        and prototype_norm > 0.0
    )

    prototype = (
        prototype
        / prototype_norm
    )

    similarities = (
        X_unit
        @ prototype
    )

    assert np.isfinite(
        similarities
    ).all()

    assert np.all(
        similarities >= -1.0 - 1e-12
    )

    assert np.all(
        similarities <= 1.0 + 1e-12
    )

    # Primary key: descending cosine.
    # Tie-break: ascending manifest_index.
    order = np.lexsort(
        (
            members,
            -similarities,
        )
    )

    return (
        members[order],
        similarities[order],
        prototype,
    )


# ------------------------------------------------------------
# 3. Fixed label-independent probe
# ------------------------------------------------------------

E4B_PROBE_CLUSTERS = [
    0,
    17,
    35,
]


def probe_partition(
    partition,
):
    result = {}

    for cluster_id in (
        E4B_PROBE_CLUSTERS
    ):
        (
            ranked,
            similarities,
            prototype,
        ) = rank_cluster_representatives(
            full_central_mu,
            partition,
            cluster_id,
        )

        result[
            cluster_id
        ] = {
            "member_count":
                int(
                    ranked.size
                ),

            "top_indices":
                ranked[
                    :REPRESENTATIVE_TOP_N
                ].tolist(),

            "top_similarities":
                similarities[
                    :REPRESENTATIVE_TOP_N
                ].tolist(),

            "prototype_norm":
                float(
                    np.linalg.norm(
                        prototype
                    )
                ),
        }

    return result


primary_probe_1 = probe_partition(
    e4b_primary_partition
)

primary_probe_2 = probe_partition(
    e4b_primary_partition
)

legacy_probe_1 = probe_partition(
    e4b_legacy_partition
)

legacy_probe_2 = probe_partition(
    e4b_legacy_partition
)


# ------------------------------------------------------------
# 4. Reproducibility / mathematical gates
# ------------------------------------------------------------

reproducibility_gate = (
    primary_probe_1
        == primary_probe_2

    and legacy_probe_1
        == legacy_probe_2
)

assert reproducibility_gate


prototype_norm_gate = all(
    np.isclose(
        result[
            "prototype_norm"
        ],
        1.0,
        rtol=0.0,
        atol=2.0e-14,
    )

    for probe in [
        primary_probe_1,
        legacy_probe_1,
    ]

    for result in (
        probe.values()
    )
)

assert prototype_norm_gate


ranking_gate = all(
    all(
        result[
            "top_similarities"
        ][i]
        >= result[
            "top_similarities"
        ][i + 1]
        - 1.0e-15

        for i in range(
            len(
                result[
                    "top_similarities"
                ]
            ) - 1
        )
    )

    for probe in [
        primary_probe_1,
        legacy_probe_1,
    ]

    for result in (
        probe.values()
    )
)

assert ranking_gate


primary_legacy_probe_distinct = (
    primary_probe_1
    != legacy_probe_1
)

assert primary_legacy_probe_distinct


# ------------------------------------------------------------
# 5. Freeze PROJECT contract
# ------------------------------------------------------------

REPRESENTATIVE_SELECTION_CONTRACT = {
    "contract_version":
        1,

    "basis":
        "thesis_roadmap_requires_cosine_similarity_ordering",

    "project": {
        "source_feature":
            "full_central_mu",

        "source_feature_sha256":
            FULL_CENTRAL_MU_SHA256,

        "feature_dimension":
            512,

        "partition_source":
            "frozen_spectral_partition",

        "event_vector_preprocessing":
            "L2_normalize",

        "cluster_prototype":
            "L2_normalized_mean_of_member_unit_vectors",

        "similarity":
            "cosine_similarity",

        "ranking":
            "descending_similarity",

        "tie_break":
            "ascending_manifest_index",

        "top_n_saved":
            REPRESENTATIVE_TOP_N,

        "labels_used_for_ranking":
            False,
    },

    "primary":
        "symmetric_figure",

    "sensitivity":
        "legacy_text",
}


REPRESENTATIVE_SELECTION_CONTRACT_JSON = json.dumps(
    REPRESENTATIVE_SELECTION_CONTRACT,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

REPRESENTATIVE_SELECTION_CONTRACT_SHA256 = (
    hashlib.sha256(
        REPRESENTATIVE_SELECTION_CONTRACT_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ------------------------------------------------------------
# 6. Persist contract
# ------------------------------------------------------------

E4B_CONTRACT_PATH = (
    EVALUATION_ROOT
    / "representative_selection_contract.json"
)


if E4B_CONTRACT_PATH.exists():

    assert (
        e0_sha256_file(
            E4B_CONTRACT_PATH
        )
        == REPRESENTATIVE_SELECTION_CONTRACT_SHA256
    )

else:

    atomic_write_text(
        E4B_CONTRACT_PATH,
        REPRESENTATIVE_SELECTION_CONTRACT_JSON,
    )

    assert (
        e0_sha256_file(
            E4B_CONTRACT_PATH
        )
        == REPRESENTATIVE_SELECTION_CONTRACT_SHA256
    )


# ------------------------------------------------------------
# 7. Partition immutability
# ------------------------------------------------------------

partition_post_e4b_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e4b_gate


# ------------------------------------------------------------
# 8. Final state
# ------------------------------------------------------------

REPRESENTATIVE_RULE_FROZEN = True

NOTEBOOK_06_E4B_REPRESENTATIVE_PREFLIGHT_PASS = True
READY_FOR_E4C_REPRESENTATIVE_SELECTION = True


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E4B REPRESENTATIVE PREFLIGHT ==="
)

print(
    "REPRESENTATIVE_TOP_N =",
    REPRESENTATIVE_TOP_N,
)

print(
    "REPRODUCIBILITY_GATE =",
    reproducibility_gate,
)

print(
    "PROTOTYPE_NORM_GATE =",
    prototype_norm_gate,
)

print(
    "RANKING_GATE =",
    ranking_gate,
)

print(
    "PRIMARY_LEGACY_PROBE_DISTINCT =",
    primary_legacy_probe_distinct,
)

print(
    "PRIMARY_PROBE =",
    primary_probe_1,
)

print(
    "LEGACY_PROBE =",
    legacy_probe_1,
)

print(
    "LABELS_USED_FOR_RANKING =",
    False,
)

print(
    "REPRESENTATIVE_SELECTION_CONTRACT_SHA256 =",
    REPRESENTATIVE_SELECTION_CONTRACT_SHA256,
)

print(
    "PARTITION_POST_E4B_GATE =",
    partition_post_e4b_gate,
)

print(
    "NOTEBOOK_06_E4B_REPRESENTATIVE_PREFLIGHT_PASS =",
    NOTEBOOK_06_E4B_REPRESENTATIVE_PREFLIGHT_PASS,
)

print(
    "READY_FOR_E4C_REPRESENTATIVE_SELECTION =",
    READY_FOR_E4C_REPRESENTATIVE_SELECTION,
)


=== NOTEBOOK 06 — E4B REPRESENTATIVE PREFLIGHT ===
REPRESENTATIVE_TOP_N = 5
REPRODUCIBILITY_GATE = True
PROTOTYPE_NORM_GATE = True
RANKING_GATE = True
PRIMARY_LEGACY_PROBE_DISTINCT = True
PRIMARY_PROBE = {0: {'member_count': 186, 'top_indices': [4492, 4633, 4621, 4608, 4592], 'top_similarities': [0.9718678935514862, 0.9621130685251966, 0.9616178836330016, 0.9577827108345462, 0.9510339922010028], 'prototype_norm': 1.0}, 17: {'member_count': 159, 'top_indices': [5571, 5648, 5708, 5624, 5461], 'top_similarities': [0.9735928933635691, 0.9707341838657219, 0.9681562411314184, 0.9645940324342724, 0.9644021154421865], 'prototype_norm': 0.9999999999999999}, 35: {'member_count': 126, 'top_indices': [4273, 4475, 4256, 4453, 4654], 'top_similarities': [0.9822745498716932, 0.9703852446482182, 0.9689951979456609, 0.9676349186138009, 0.9671129500989126], 'prototype_norm': 0.9999999999999999}}
LEGACY_PROBE = {0: {'member_count': 311, 'top_indices': [8230, 8015, 8203, 8223, 8190], 'top_similarities': 

In [43]:
# ============================================================
# NOTEBOOK 06 — E4C
# FULL REPRESENTATIVE RANKING
#
# Applies frozen E4B cosine rule to all 36 clusters.
#
# Saves:
#   - complete ranking for PRIMARY
#   - complete ranking for LEGACY
#   - top-5 representatives per cluster
#
# Labels may be reported as annotation only.
# They are NOT used for ranking.
# ============================================================

from pathlib import Path
import csv
import io
import json

import numpy as np


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E4B_REPRESENTATIVE_PREFLIGHT_PASS"
) is True

assert globals().get(
    "READY_FOR_E4C_REPRESENTATIVE_SELECTION"
) is True

assert globals().get(
    "REPRESENTATIVE_RULE_FROZEN"
) is True

assert (
    REPRESENTATIVE_SELECTION_CONTRACT_SHA256
    == "c1a3fd7edaefb9e70b8a5664a55ccd2b"
       "d41a2e95f858fdacb2aeb5c7f64857d1"
)

assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False
assert MODEL_SELECTION_PERFORMED is False


# ------------------------------------------------------------
# 1. Frozen partitions
# ------------------------------------------------------------

primary_rep_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

legacy_rep_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)

assert primary_rep_partition.shape == (8583,)
assert legacy_rep_partition.shape == (8583,)


# ------------------------------------------------------------
# 2. Build complete ranking
# ------------------------------------------------------------

def build_full_representative_ranking(
    partition,
):
    records = []

    for cluster_id in range(36):

        (
            ranked_indices,
            similarities,
            prototype,
        ) = rank_cluster_representatives(
            full_central_mu,
            partition,
            cluster_id,
        )

        assert ranked_indices.size > 0

        for rank_position, (
            manifest_index,
            similarity,
        ) in enumerate(
            zip(
                ranked_indices,
                similarities,
                strict=True,
            ),
            start=1,
        ):

            manifest_index = int(
                manifest_index
            )

            records.append({
                "cluster_id":
                    int(cluster_id),

                "rank_in_cluster":
                    int(rank_position),

                "is_top5":
                    bool(
                        rank_position
                        <= REPRESENTATIVE_TOP_N
                    ),

                "manifest_index":
                    manifest_index,

                "event_id":
                    full_iic_rows[
                        manifest_index
                    ][
                        "event_id"
                    ],

                "cosine_similarity":
                    float(
                        similarity
                    ),

                # Annotation only.
                "gravity_spy_label":
                    str(
                        gravity_spy_labels[
                            manifest_index
                        ]
                    ),
            })

    return records


PRIMARY_REPRESENTATIVE_RANKING = (
    build_full_representative_ranking(
        primary_rep_partition
    )
)

LEGACY_REPRESENTATIVE_RANKING = (
    build_full_representative_ranking(
        legacy_rep_partition
    )
)


# ------------------------------------------------------------
# 3. Structural gates
# ------------------------------------------------------------

def ranking_gate(
    records,
    partition,
):
    if len(records) != 8583:
        return False

    indices = [
        row["manifest_index"]
        for row in records
    ]

    if sorted(indices) != list(
        range(8583)
    ):
        return False

    for cluster_id in range(36):

        cluster_rows = [
            row
            for row in records
            if row[
                "cluster_id"
            ] == cluster_id
        ]

        expected_count = int(
            np.sum(
                partition
                == cluster_id
            )
        )

        if len(cluster_rows) != expected_count:
            return False

        ranks = [
            row[
                "rank_in_cluster"
            ]
            for row
            in cluster_rows
        ]

        if ranks != list(
            range(
                1,
                expected_count + 1,
            )
        ):
            return False

        sims = np.asarray(
            [
                row[
                    "cosine_similarity"
                ]
                for row
                in cluster_rows
            ],
            dtype=np.float64,
        )

        if np.any(
            np.diff(
                sims
            ) > 1.0e-15
        ):
            return False

        if sum(
            row[
                "is_top5"
            ]
            for row
            in cluster_rows
        ) != min(
            REPRESENTATIVE_TOP_N,
            expected_count,
        ):
            return False

    return True


primary_ranking_gate = ranking_gate(
    PRIMARY_REPRESENTATIVE_RANKING,
    primary_rep_partition,
)

legacy_ranking_gate = ranking_gate(
    LEGACY_REPRESENTATIVE_RANKING,
    legacy_rep_partition,
)

assert primary_ranking_gate
assert legacy_ranking_gate


# ------------------------------------------------------------
# 4. Top-5 subsets
# ------------------------------------------------------------

PRIMARY_TOP5_REPRESENTATIVES = [
    row
    for row
    in PRIMARY_REPRESENTATIVE_RANKING
    if row["is_top5"]
]

LEGACY_TOP5_REPRESENTATIVES = [
    row
    for row
    in LEGACY_REPRESENTATIVE_RANKING
    if row["is_top5"]
]

top5_gate = (
    len(
        PRIMARY_TOP5_REPRESENTATIVES
    ) == 36 * 5

    and len(
        LEGACY_TOP5_REPRESENTATIVES
    ) == 36 * 5
)

assert top5_gate


# ------------------------------------------------------------
# 5. Stable CSV serialization
# ------------------------------------------------------------

REPRESENTATIVE_FIELDS = [
    "cluster_id",
    "rank_in_cluster",
    "is_top5",
    "manifest_index",
    "event_id",
    "cosine_similarity",
    "gravity_spy_label",
]


def representative_csv_text(
    records,
):
    stream = io.StringIO(
        newline=""
    )

    writer = csv.DictWriter(
        stream,
        fieldnames=REPRESENTATIVE_FIELDS,
        lineterminator="\n",
    )

    writer.writeheader()

    for row in records:

        writer.writerow({
            **row,

            "is_top5":
                int(
                    row[
                        "is_top5"
                    ]
                ),

            "cosine_similarity":
                format(
                    row[
                        "cosine_similarity"
                    ],
                    ".17g",
                ),
        })

    return stream.getvalue()


# ------------------------------------------------------------
# 6. Output paths
# ------------------------------------------------------------

E4C_ROOT = (
    EVALUATION_ROOT
    / "representatives"
)

E4C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PRIMARY_FULL_RANKING_PATH = (
    E4C_ROOT
    / "primary_full_cosine_ranking.csv"
)

LEGACY_FULL_RANKING_PATH = (
    E4C_ROOT
    / "legacy_full_cosine_ranking.csv"
)

PRIMARY_TOP5_PATH = (
    E4C_ROOT
    / "primary_top5_representatives.csv"
)

LEGACY_TOP5_PATH = (
    E4C_ROOT
    / "legacy_top5_representatives.csv"
)

E4C_METADATA_PATH = (
    E4C_ROOT
    / "representative_ranking_metadata.json"
)


# ------------------------------------------------------------
# 7. Persist / verify
# ------------------------------------------------------------

def e4c_write_or_verify(
    path,
    text,
):
    if path.exists():

        assert path.read_text(
            encoding="utf-8"
        ) == text

    else:

        atomic_write_text(
            path,
            text,
        )


e4c_write_or_verify(
    PRIMARY_FULL_RANKING_PATH,
    representative_csv_text(
        PRIMARY_REPRESENTATIVE_RANKING
    ),
)

e4c_write_or_verify(
    LEGACY_FULL_RANKING_PATH,
    representative_csv_text(
        LEGACY_REPRESENTATIVE_RANKING
    ),
)

e4c_write_or_verify(
    PRIMARY_TOP5_PATH,
    representative_csv_text(
        PRIMARY_TOP5_REPRESENTATIVES
    ),
)

e4c_write_or_verify(
    LEGACY_TOP5_PATH,
    representative_csv_text(
        LEGACY_TOP5_REPRESENTATIVES
    ),
)


# ------------------------------------------------------------
# 8. Metadata
# ------------------------------------------------------------

E4C_METADATA = {
    "format_version":
        1,

    "status":
        "FULL_REPRESENTATIVE_RANKING_COMPLETE",

    "selection_contract_sha256":
        REPRESENTATIVE_SELECTION_CONTRACT_SHA256,

    "ranking_feature":
        "full_central_mu",

    "ranking_feature_sha256":
        FULL_CENTRAL_MU_SHA256,

    "similarity":
        "cosine",

    "top_n":
        REPRESENTATIVE_TOP_N,

    "labels_used_for_ranking":
        False,

    "labels_included_as_annotation":
        True,

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "full_ranking_rows":
            len(
                PRIMARY_REPRESENTATIVE_RANKING
            ),

        "top5_rows":
            len(
                PRIMARY_TOP5_REPRESENTATIVES
            ),

        "full_ranking_sha256":
            e0_sha256_file(
                PRIMARY_FULL_RANKING_PATH
            ),

        "top5_sha256":
            e0_sha256_file(
                PRIMARY_TOP5_PATH
            ),
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "full_ranking_rows":
            len(
                LEGACY_REPRESENTATIVE_RANKING
            ),

        "top5_rows":
            len(
                LEGACY_TOP5_REPRESENTATIVES
            ),

        "full_ranking_sha256":
            e0_sha256_file(
                LEGACY_FULL_RANKING_PATH
            ),

        "top5_sha256":
            e0_sha256_file(
                LEGACY_TOP5_PATH
            ),
    },

    "model_selection_performed":
        False,

    "partitions_changed_after_labels":
        False,
}


E4C_METADATA_JSON = json.dumps(
    E4C_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

e4c_write_or_verify(
    E4C_METADATA_PATH,
    E4C_METADATA_JSON,
)

E4C_METADATA_SHA256 = (
    e0_sha256_file(
        E4C_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 9. Partition immutability
# ------------------------------------------------------------

partition_post_e4c_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e4c_gate


# ------------------------------------------------------------
# 10. Final state
# ------------------------------------------------------------

NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS = True
READY_FOR_E4D_REPRESENTATIVE_IMAGES = True


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E4C REPRESENTATIVE SELECTION ==="
)

print(
    "PRIMARY_RANKING_GATE =",
    primary_ranking_gate,
)

print(
    "LEGACY_RANKING_GATE =",
    legacy_ranking_gate,
)

print(
    "TOP5_GATE =",
    top5_gate,
)

print(
    "PRIMARY_FULL_RANKING_ROWS =",
    len(
        PRIMARY_REPRESENTATIVE_RANKING
    ),
)

print(
    "LEGACY_FULL_RANKING_ROWS =",
    len(
        LEGACY_REPRESENTATIVE_RANKING
    ),
)

print(
    "PRIMARY_TOP5_ROWS =",
    len(
        PRIMARY_TOP5_REPRESENTATIVES
    ),
)

print(
    "LEGACY_TOP5_ROWS =",
    len(
        LEGACY_TOP5_REPRESENTATIVES
    ),
)

print(
    "PRIMARY_TOP5_SHA256 =",
    e0_sha256_file(
        PRIMARY_TOP5_PATH
    ),
)

print(
    "LEGACY_TOP5_SHA256 =",
    e0_sha256_file(
        LEGACY_TOP5_PATH
    ),
)

print(
    "LABELS_USED_FOR_RANKING =",
    False,
)

print(
    "PARTITION_POST_E4C_GATE =",
    partition_post_e4c_gate,
)

print(
    "E4C_METADATA_SHA256 =",
    E4C_METADATA_SHA256,
)

print(
    "NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS =",
    NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS,
)

print(
    "READY_FOR_E4D_REPRESENTATIVE_IMAGES =",
    READY_FOR_E4D_REPRESENTATIVE_IMAGES,
)


=== NOTEBOOK 06 — E4C REPRESENTATIVE SELECTION ===
PRIMARY_RANKING_GATE = True
LEGACY_RANKING_GATE = True
TOP5_GATE = True
PRIMARY_FULL_RANKING_ROWS = 8583
LEGACY_FULL_RANKING_ROWS = 8583
PRIMARY_TOP5_ROWS = 180
LEGACY_TOP5_ROWS = 180
PRIMARY_TOP5_SHA256 = 90682a7a353c6a7bbf970ddf273eef70f4f21739fa1bd459c774b9ee1601cef5
LEGACY_TOP5_SHA256 = 1df4b839a8e82e8e71c097bae502836df3413e95e57fd2e8ac6d5f8681cf520c
LABELS_USED_FOR_RANKING = False
PARTITION_POST_E4C_GATE = True
E4C_METADATA_SHA256 = 4c2065dbb4bb552b600fb5bd4956d30a4900acc44e6212b775f326db02b7ad97
NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS = True
READY_FOR_E4D_REPRESENTATIVE_IMAGES = True


In [44]:
# ============================================================
# NOTEBOOK 06 — E4D
# REPRESENTATIVE IMAGE CONTACT SHEETS
#
# 36 PRIMARY + 36 LEGACY sheets.
# Each sheet = top-5 representatives x 4 temporal views.
#
# Ranking is already frozen and label-independent.
# Images use central crop_left=24.
# ============================================================

from pathlib import Path
import csv
import hashlib
import json
import os

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 0. Parent gates
# ------------------------------------------------------------

assert globals().get(
    "NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS"
) is True

assert globals().get(
    "READY_FOR_E4D_REPRESENTATIVE_IMAGES"
) is True

assert (
    REPRESENTATIVE_SELECTION_CONTRACT_SHA256
    == "c1a3fd7edaefb9e70b8a5664a55ccd2b"
       "d41a2e95f858fdacb2aeb5c7f64857d1"
)

assert SPECTRAL_PARTITIONS_FROZEN is True
assert PARTITIONS_CHANGED_AFTER_LABELS is False


# ------------------------------------------------------------
# 1. Frozen top-5 ranking identities
# ------------------------------------------------------------

EXPECTED_PRIMARY_TOP5_SHA256 = (
    "90682a7a353c6a7bbf970ddf273eef70"
    "f4f21739fa1bd459c774b9ee1601cef5"
)

EXPECTED_LEGACY_TOP5_SHA256 = (
    "1df4b839a8e82e8e71c097bae502836d"
    "f3413e95e57fd2e8ac6d5f8681cf520c"
)

assert (
    e0_sha256_file(
        PRIMARY_TOP5_PATH
    )
    == EXPECTED_PRIMARY_TOP5_SHA256
)

assert (
    e0_sha256_file(
        LEGACY_TOP5_PATH
    )
    == EXPECTED_LEGACY_TOP5_SHA256
)


# ------------------------------------------------------------
# 2. Read persisted rankings
# ------------------------------------------------------------

def read_top5_csv(path):
    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:

        rows = list(
            csv.DictReader(
                handle
            )
        )

    assert len(rows) == 180

    parsed = []

    for row in rows:
        parsed.append({
            "cluster_id":
                int(
                    row[
                        "cluster_id"
                    ]
                ),

            "rank_in_cluster":
                int(
                    row[
                        "rank_in_cluster"
                    ]
                ),

            "manifest_index":
                int(
                    row[
                        "manifest_index"
                    ]
                ),

            "event_id":
                row[
                    "event_id"
                ],

            "cosine_similarity":
                float(
                    row[
                        "cosine_similarity"
                    ]
                ),

            "gravity_spy_label":
                row[
                    "gravity_spy_label"
                ],
        })

    return parsed


primary_top5_rows = read_top5_csv(
    PRIMARY_TOP5_PATH
)

legacy_top5_rows = read_top5_csv(
    LEGACY_TOP5_PATH
)


def group_by_cluster(rows):
    grouped = {
        cluster_id: []
        for cluster_id in range(36)
    }

    for row in rows:
        grouped[
            row[
                "cluster_id"
            ]
        ].append(
            row
        )

    for cluster_id in range(36):
        grouped[
            cluster_id
        ] = sorted(
            grouped[
                cluster_id
            ],
            key=lambda row:
                row[
                    "rank_in_cluster"
                ],
        )

        assert len(
            grouped[
                cluster_id
            ]
        ) == 5

        assert [
            row[
                "rank_in_cluster"
            ]
            for row
            in grouped[
                cluster_id
            ]
        ] == [
            1,
            2,
            3,
            4,
            5,
        ]

    return grouped


primary_grouped = group_by_cluster(
    primary_top5_rows
)

legacy_grouped = group_by_cluster(
    legacy_top5_rows
)


# ------------------------------------------------------------
# 3. Image source gate
# ------------------------------------------------------------

assert callable(
    load_png_event_crop_224
)

assert LOCAL_DATA_ROOT == (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

assert VIEW_FIELDS == [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]

VIEW_TITLES = [
    "0.5 s",
    "1.0 s",
    "2.0 s",
    "4.0 s",
]

CENTRAL_CROP_LEFT = 24


# ------------------------------------------------------------
# 4. Output directories
# ------------------------------------------------------------

E4D_ROOT = (
    EVALUATION_ROOT
    / "representatives"
    / "images"
)

PRIMARY_IMAGE_DIR = (
    E4D_ROOT
    / "primary_symmetric"
)

LEGACY_IMAGE_DIR = (
    E4D_ROOT
    / "legacy_text"
)

PRIMARY_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LEGACY_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 5. Deterministic contact-sheet builder
# ------------------------------------------------------------

def make_cluster_sheet(
    *,
    cluster_id,
    records,
    role_title,
    output_path,
):

    assert len(records) == 5

    fig, axes = plt.subplots(
        5,
        4,
        figsize=(10, 12),
    )

    for row_index, record in enumerate(
        records
    ):

        manifest_index = record[
            "manifest_index"
        ]

        tensor = load_png_event_crop_224(
            manifest_index,
            CENTRAL_CROP_LEFT,
        )

        array = tensor.numpy()

        assert array.shape == (
            4,
            224,
            224,
        )

        assert np.isfinite(
            array
        ).all()

        for channel_index in range(4):

            ax = axes[
                row_index,
                channel_index,
            ]

            ax.imshow(
                array[
                    channel_index
                ],
                cmap="gray",
                vmin=0.0,
                vmax=1.0,
                origin="upper",
            )

            ax.set_xticks([])
            ax.set_yticks([])

            if row_index == 0:
                ax.set_title(
                    VIEW_TITLES[
                        channel_index
                    ],
                    fontsize=10,
                )

        axes[
            row_index,
            0
        ].set_ylabel(
            (
                f"rank {record['rank_in_cluster']}\n"
                f"m={manifest_index}\n"
                f"{record['gravity_spy_label']}\n"
                f"cos={record['cosine_similarity']:.4f}"
            ),
            fontsize=8,
            rotation=0,
            ha="right",
            va="center",
            labelpad=8,
        )

    fig.suptitle(
        (
            f"{role_title} — cluster {cluster_id:02d}\n"
            "Top-5 per cosine similarity su μ centrale"
        ),
        fontsize=12,
    )

    fig.tight_layout(
        rect=[
            0,
            0,
            1,
            0.96,
        ]
    )

    partial_path = Path(
        str(
            output_path
        )
        + ".partial.png"
    )

    assert not partial_path.exists()

    fig.savefig(
        partial_path,
        dpi=160,
        bbox_inches="tight",
        format="png",
    )

    plt.close(
        fig
    )

    os.replace(
        partial_path,
        output_path,
    )

    assert output_path.is_file()
    assert output_path.stat().st_size > 0


# ------------------------------------------------------------
# 6. Build / reuse all 72 sheets
# ------------------------------------------------------------

def build_all_sheets(
    *,
    grouped,
    output_dir,
    role_title,
):

    hashes = {}

    for cluster_id in range(36):

        output_path = (
            output_dir
            / f"cluster_{cluster_id:02d}_top5.png"
        )

        if not output_path.exists():

            make_cluster_sheet(
                cluster_id=
                    cluster_id,

                records=
                    grouped[
                        cluster_id
                    ],

                role_title=
                    role_title,

                output_path=
                    output_path,
            )

        assert output_path.is_file()
        assert output_path.stat().st_size > 0

        hashes[
            f"cluster_{cluster_id:02d}"
        ] = e0_sha256_file(
            output_path
        )

    return hashes


print(
    "Building/recovering PRIMARY representative sheets..."
)

PRIMARY_REPRESENTATIVE_IMAGE_HASHES = (
    build_all_sheets(
        grouped=
            primary_grouped,

        output_dir=
            PRIMARY_IMAGE_DIR,

        role_title=
            "PRIMARY symmetric_figure",
    )
)


print(
    "Building/recovering LEGACY representative sheets..."
)

LEGACY_REPRESENTATIVE_IMAGE_HASHES = (
    build_all_sheets(
        grouped=
            legacy_grouped,

        output_dir=
            LEGACY_IMAGE_DIR,

        role_title=
            "SENSITIVITY legacy_text",
    )
)


# ------------------------------------------------------------
# 7. Count gate
# ------------------------------------------------------------

primary_pngs = sorted(
    PRIMARY_IMAGE_DIR.glob(
        "cluster_*_top5.png"
    )
)

legacy_pngs = sorted(
    LEGACY_IMAGE_DIR.glob(
        "cluster_*_top5.png"
    )
)

image_count_gate = (
    len(
        primary_pngs
    ) == 36

    and len(
        legacy_pngs
    ) == 36

    and len(
        PRIMARY_REPRESENTATIVE_IMAGE_HASHES
    ) == 36

    and len(
        LEGACY_REPRESENTATIVE_IMAGE_HASHES
    ) == 36
)

assert image_count_gate


# ------------------------------------------------------------
# 8. Metadata
# ------------------------------------------------------------

E4D_METADATA_PATH = (
    E4D_ROOT
    / "representative_images_metadata.json"
)

E4D_METADATA = {
    "format_version":
        1,

    "status":
        "REPRESENTATIVE_IMAGES_COMPLETE",

    "selection_contract_sha256":
        REPRESENTATIVE_SELECTION_CONTRACT_SHA256,

    "central_crop_left":
        CENTRAL_CROP_LEFT,

    "views":
        VIEW_TITLES,

    "top_n":
        5,

    "sheet_layout":
        "5_events_x_4_temporal_views",

    "labels_used_for_selection":
        False,

    "labels_used_as_annotation":
        True,

    "primary": {
        "run_id":
            "sakai224_full_iic_symmetric_v1",

        "top5_csv_sha256":
            EXPECTED_PRIMARY_TOP5_SHA256,

        "sheet_count":
            36,

        "sheet_sha256":
            PRIMARY_REPRESENTATIVE_IMAGE_HASHES,
    },

    "sensitivity": {
        "run_id":
            "sakai224_full_iic_legacy_text_v1",

        "top5_csv_sha256":
            EXPECTED_LEGACY_TOP5_SHA256,

        "sheet_count":
            36,

        "sheet_sha256":
            LEGACY_REPRESENTATIVE_IMAGE_HASHES,
    },

    "partitions_changed_after_labels":
        False,
}


E4D_METADATA_JSON = json.dumps(
    E4D_METADATA,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)


if E4D_METADATA_PATH.exists():

    with E4D_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:

        existing_e4d = json.load(
            handle
        )

    assert existing_e4d == E4D_METADATA

else:

    atomic_write_text(
        E4D_METADATA_PATH,
        E4D_METADATA_JSON,
    )


E4D_METADATA_SHA256 = (
    e0_sha256_file(
        E4D_METADATA_PATH
    )
)


# ------------------------------------------------------------
# 9. Partition immutability
# ------------------------------------------------------------

partition_post_e4d_gate = (
    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    )
    == EXPECTED_PRIMARY_PARTITION_FILE_SHA256

    and e0_sha256_file(
        LEGACY_PARTITION_PATH
    )
    == EXPECTED_LEGACY_PARTITION_FILE_SHA256
)

assert partition_post_e4d_gate


# ------------------------------------------------------------
# 10. Final state
# ------------------------------------------------------------

NOTEBOOK_06_E4D_REPRESENTATIVE_IMAGES_PASS = True
READY_FOR_NOTEBOOK06_FINAL_AUDIT = True


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — E4D REPRESENTATIVE IMAGES ==="
)

print(
    "PRIMARY_SHEET_COUNT =",
    len(
        primary_pngs
    ),
)

print(
    "LEGACY_SHEET_COUNT =",
    len(
        legacy_pngs
    ),
)

print(
    "IMAGE_COUNT_GATE =",
    image_count_gate,
)

print(
    "PRIMARY_IMAGE_DIR =",
    PRIMARY_IMAGE_DIR,
)

print(
    "LEGACY_IMAGE_DIR =",
    LEGACY_IMAGE_DIR,
)

print(
    "LABELS_USED_FOR_SELECTION =",
    False,
)

print(
    "PARTITION_POST_E4D_GATE =",
    partition_post_e4d_gate,
)

print(
    "E4D_METADATA_SHA256 =",
    E4D_METADATA_SHA256,
)

print(
    "NOTEBOOK_06_E4D_REPRESENTATIVE_IMAGES_PASS =",
    NOTEBOOK_06_E4D_REPRESENTATIVE_IMAGES_PASS,
)

print(
    "READY_FOR_NOTEBOOK06_FINAL_AUDIT =",
    READY_FOR_NOTEBOOK06_FINAL_AUDIT,
)

Building/recovering PRIMARY representative sheets...
Building/recovering LEGACY representative sheets...

=== NOTEBOOK 06 — E4D REPRESENTATIVE IMAGES ===
PRIMARY_SHEET_COUNT = 36
LEGACY_SHEET_COUNT = 36
IMAGE_COUNT_GATE = True
PRIMARY_IMAGE_DIR = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/representatives/images/primary_symmetric
LEGACY_IMAGE_DIR = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/evaluation/representatives/images/legacy_text
LABELS_USED_FOR_SELECTION = False
PARTITION_POST_E4D_GATE = True
E4D_METADATA_SHA256 = 39f511550c726c6799879abe41acc24a44273af422b2193105b657397b19ae2d
NOTEBOOK_06_E4D_REPRESENTATIVE_IMAGES_PASS = True
READY_FOR_NOTEBOOK06_FINAL_AUDIT = True


In [45]:
# ============================================================
# NOTEBOOK 06 — FINAL AUDIT
# FULL224 IIC + SPECTRAL + LABEL EVALUATION
#
# READ-ONLY audit + one final persistent handoff JSON.
# No clustering, no retraining, no model selection.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np


# ------------------------------------------------------------
# 0. Required completed gates
# ------------------------------------------------------------

required_gates = [
    "NOTEBOOK_06_A6F_DUAL_IIC_HANDOFF_PASS",
    "NOTEBOOK_06_S1_FULL_H_BUILD_PASS",
    "NOTEBOOK_06_S3_FULL_AFFINITY_BUILD_PASS",
    "NOTEBOOK_06_S4_SPECTRAL_SOLVER_PREFLIGHT_PASS",
    "NOTEBOOK_06_S5_PRIMARY_SPECTRAL_PASS",
    "NOTEBOOK_06_S6_LEGACY_SPECTRAL_PASS",
    "NOTEBOOK_06_S7_DUAL_PARTITION_FREEZE_PASS",
    "NOTEBOOK_06_E0_LABEL_OPENING_PASS",
    "NOTEBOOK_06_E1_STANDARD_METRICS_PASS",
    "NOTEBOOK_06_E2_SAKAI_EQ1_AUDIT_PASS",
    "NOTEBOOK_06_E3_INTERPRETATION_TABLES_PASS",
    "NOTEBOOK_06_E4A_CONTINGENCY_FIGURES_PASS",
    "NOTEBOOK_06_E4B_REPRESENTATIVE_PREFLIGHT_PASS",
    "NOTEBOOK_06_E4C_REPRESENTATIVE_SELECTION_PASS",
    "NOTEBOOK_06_E4D_REPRESENTATIVE_IMAGES_PASS",
]

gate_map = {
    name: globals().get(name) is True
    for name in required_gates
}

all_step_gates = all(
    gate_map.values()
)

assert all_step_gates


# ------------------------------------------------------------
# 1. Frozen core identities
# ------------------------------------------------------------

EXPECTED = {
    "dual_iic_handoff":
        "e2212af6a6791e45f0bef3a5ea1771ae"
        "b976b2130a355f692f2bda3654400564",

    "primary_partition":
        "5a01318f703441bbf291f7b32478043f"
        "6d0376dd81265078d6dbeac43e306ac0",

    "legacy_partition":
        "69edcfaf5006214c992ce39fb359d298"
        "41c78ab5fc13fe1f58901d06915a3632",

    "s7_freeze":
        "7fb80c7bc65ca2d6b98b5bc59985a2cb"
        "629f9189231fb759ce2ba674e2382fe1",

    "e1_metrics":
        "5353d0d8e24fb1ac6d2e7cbe31f4ad"
        "acfc3f628abd53cd52fe2c1017bbb75023",

    "e2_sakai":
        "ab07670afcaeb3b9827873d203ee18b3"
        "d72b3ec68fd32d2dbe395ed20ab1fc9e",

    "e3_interpretation":
        "c19d7a6ab9ad39e7dfae40fbaf43f062"
        "da750124dda123e6c5a67a92ff29f58a",

    "e4a_figures":
        "881cfd76e7c1d649ec5b729d3def55f3"
        "d5bc59a10fa4214bd4524a3dc5e70fd7",

    "e4b_rule":
        "c1a3fd7edaefb9e70b8a5664a55ccd2b"
        "d41a2e95f858fdacb2aeb5c7f64857d1",

    "e4c_ranking":
        "4c2065dbb4bb552b600fb5bd4956d30a"
        "4900acc44e6212b775f326db02b7ad97",

    "e4d_images":
        "39f511550c726c6799879abe41acc24a4"
        "4273af422b2193105b657397b19ae2d",
}


identity_gate = all([
    e0_sha256_file(
        SPECTRAL_DUAL_HANDOFF
    ) == EXPECTED["dual_iic_handoff"],

    e0_sha256_file(
        PRIMARY_PARTITION_PATH
    ) == EXPECTED["primary_partition"],

    e0_sha256_file(
        LEGACY_PARTITION_PATH
    ) == EXPECTED["legacy_partition"],

    e0_sha256_file(
        S7_FREEZE_PATH_E0
    ) == EXPECTED["s7_freeze"],

    e0_sha256_file(
        E1_RESULTS_PATH
    ) == EXPECTED["e1_metrics"],

    e0_sha256_file(
        E2_RESULTS_PATH
    ) == EXPECTED["e2_sakai"],

    e0_sha256_file(
        paths["metadata"]
    ) == EXPECTED["e3_interpretation"],

    e0_sha256_file(
        E4A_METADATA_PATH
    ) == EXPECTED["e4a_figures"],

    e0_sha256_file(
        E4B_CONTRACT_PATH
    ) == EXPECTED["e4b_rule"],

    e0_sha256_file(
        E4C_METADATA_PATH
    ) == EXPECTED["e4c_ranking"],

    e0_sha256_file(
        E4D_METADATA_PATH
    ) == EXPECTED["e4d_images"],
])

assert identity_gate


# ------------------------------------------------------------
# 2. Frozen partitions still valid
# ------------------------------------------------------------

final_primary_partition = np.load(
    PRIMARY_PARTITION_PATH,
    allow_pickle=False,
)

final_legacy_partition = np.load(
    LEGACY_PARTITION_PATH,
    allow_pickle=False,
)

partition_gate = (
    final_primary_partition.shape == (8583,)
    and final_legacy_partition.shape == (8583,)

    and np.unique(
        final_primary_partition
    ).size == 36

    and np.unique(
        final_legacy_partition
    ).size == 36

    and not np.array_equal(
        final_primary_partition,
        final_legacy_partition,
    )
)

assert partition_gate


# ------------------------------------------------------------
# 3. Scientific results gate
# ------------------------------------------------------------

EXPECTED_PRIMARY_METRICS = {
    "purity":
        0.8264010252825352,

    "nmi_arithmetic":
        0.7018325449722438,

    "ari":
        0.4509773588198464,

    "homogeneity":
        0.7836699716771438,

    "completeness":
        0.6354712757726401,
}

EXPECTED_LEGACY_METRICS = {
    "purity":
        0.7980892461843179,

    "nmi_arithmetic":
        0.6821419109736958,

    "ari":
        0.33427224443938447,

    "homogeneity":
        0.7729849474767357,

    "completeness":
        0.6104055888493044,
}


metrics_gate = all(
    np.isclose(
        PRIMARY_EXTERNAL_METRICS[key],
        value,
        rtol=0.0,
        atol=1e-15,
    )
    for key, value
    in EXPECTED_PRIMARY_METRICS.items()
) and all(
    np.isclose(
        LEGACY_EXTERNAL_METRICS[key],
        value,
        rtol=0.0,
        atol=1e-15,
    )
    for key, value
    in EXPECTED_LEGACY_METRICS.items()
)

assert metrics_gate


sakai_like_gate = (
    np.isclose(
        PRIMARY_SAKAI_LIKE_MACRO,
        0.8513637314759159,
        rtol=0.0,
        atol=1e-15,
    )

    and np.isclose(
        LEGACY_SAKAI_LIKE_MACRO,
        0.8354332296844487,
        rtol=0.0,
        atol=1e-15,
    )
)

assert sakai_like_gate


# ------------------------------------------------------------
# 4. Representative artifacts
# ------------------------------------------------------------

primary_sheets = sorted(
    PRIMARY_IMAGE_DIR.glob(
        "cluster_*_top5.png"
    )
)

legacy_sheets = sorted(
    LEGACY_IMAGE_DIR.glob(
        "cluster_*_top5.png"
    )
)

representative_gate = (
    len(primary_sheets) == 36
    and len(legacy_sheets) == 36
    and len(PRIMARY_TOP5_REPRESENTATIVES) == 180
    and len(LEGACY_TOP5_REPRESENTATIVES) == 180
)

assert representative_gate


# ------------------------------------------------------------
# 5. Methodological integrity
# ------------------------------------------------------------

methodology_gate = (
    SPECTRAL_PARTITIONS_FROZEN is True

    and PARTITIONS_CHANGED_AFTER_LABELS is False

    and MODEL_SELECTION_PERFORMED is False

    and REPRESENTATIVE_RULE_FROZEN is True
)

assert methodology_gate


# ------------------------------------------------------------
# 6. Final handoff
# ------------------------------------------------------------

FINAL_AUDIT_PATH = (
    SPECTRAL_ROOT
    / "audits"
    / "sakai224_full_notebook06_final_audit.json"
)

assert not FINAL_AUDIT_PATH.exists(), (
    f"STOP: final audit already exists: {FINAL_AUDIT_PATH}"
)


FINAL_AUDIT = {
    "format_version":
        1,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "NOTEBOOK06_FULL224_IIC_SPECTRAL_EVALUATION_COMPLETE",

    "source_policy": {
        "scientific_primary":
            "Scientific_Reports_version_of_record_HTML",

        "supplemental":
            "Feature_Visualization_of_Transient_Noise_using_tSNE",

        "paper_vs_project_separation":
            True,
    },

    "population": {
        "events":
            8583,

        "gravity_spy_labels":
            22,

        "spectral_clusters":
            36,
    },

    "primary": {
        "mode":
            "symmetric_figure",

        "role":
            "PRIMARY_PRESELECTED_BEFORE_LABELS",

        "metrics":
            PRIMARY_EXTERNAL_METRICS,

        "sakai_like_macro":
            PRIMARY_SAKAI_LIKE_MACRO,

        "partition_sha256":
            EXPECTED["primary_partition"],
    },

    "sensitivity": {
        "mode":
            "legacy_text",

        "role":
            "SENSITIVITY_PRESELECTED_BEFORE_LABELS",

        "metrics":
            LEGACY_EXTERNAL_METRICS,

        "sakai_like_macro":
            LEGACY_SAKAI_LIKE_MACRO,

        "partition_sha256":
            EXPECTED["legacy_partition"],
    },

    "primary_vs_sensitivity": {
        "partition_ARI":
            PRIMARY_LEGACY_PARTITION_ARI,

        "partition_NMI_arithmetic":
            PRIMARY_LEGACY_PARTITION_NMI,
    },

    "methodological_integrity": {
        "partitions_frozen_before_labels":
            True,

        "partitions_changed_after_labels":
            False,

        "model_selection_after_labels":
            False,

        "representative_ranking_used_labels":
            False,
    },

    "representatives": {
        "top_n_per_cluster":
            5,

        "primary_contact_sheets":
            36,

        "legacy_contact_sheets":
            36,

        "selection_contract_sha256":
            REPRESENTATIVE_SELECTION_CONTRACT_SHA256,
    },

    "next_phase": {
        "full224_computational_chain_closed":
            True,

        "mini_C11_ablation_phase":
            True,

        "remaining_full_interpretation":
            (
                "final_morphological_subclass_and_"
                "tSNE_discussion_for_thesis"
            ),
    },
}


FINAL_AUDIT_JSON = json.dumps(
    FINAL_AUDIT,
    sort_keys=True,
    indent=2,
    ensure_ascii=True,
)

atomic_write_text(
    FINAL_AUDIT_PATH,
    FINAL_AUDIT_JSON,
)

FINAL_AUDIT_SHA256 = e0_sha256_file(
    FINAL_AUDIT_PATH
)


# ------------------------------------------------------------
# 7. Read-back gate
# ------------------------------------------------------------

with FINAL_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    final_readback = json.load(handle)


readback_gate = (
    final_readback["status"]
        == "NOTEBOOK06_FULL224_IIC_SPECTRAL_EVALUATION_COMPLETE"

    and final_readback[
        "methodological_integrity"
    ][
        "partitions_frozen_before_labels"
    ] is True

    and final_readback[
        "methodological_integrity"
    ][
        "partitions_changed_after_labels"
    ] is False

    and final_readback[
        "methodological_integrity"
    ][
        "model_selection_after_labels"
    ] is False
)

assert readback_gate


# ------------------------------------------------------------
# 8. Final flags
# ------------------------------------------------------------

NOTEBOOK_06_FINAL_AUDIT_PASS = True
FULL224_IIC_SPECTRAL_EVALUATION_CLOSED = True

READY_FOR_MINI_C11_PHASE = True


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 06 — FINAL AUDIT ==="
)

print(
    "ALL_STEP_GATES =",
    all_step_gates,
)

print(
    "IDENTITY_GATE =",
    identity_gate,
)

print(
    "PARTITION_GATE =",
    partition_gate,
)

print(
    "METRICS_GATE =",
    metrics_gate,
)

print(
    "SAKAI_LIKE_GATE =",
    sakai_like_gate,
)

print(
    "REPRESENTATIVE_GATE =",
    representative_gate,
)

print(
    "METHODOLOGY_GATE =",
    methodology_gate,
)

print()
print(
    "PRIMARY_PURITY =",
    PRIMARY_EXTERNAL_METRICS[
        "purity"
    ],
)

print(
    "PRIMARY_NMI =",
    PRIMARY_EXTERNAL_METRICS[
        "nmi_arithmetic"
    ],
)

print(
    "PRIMARY_ARI =",
    PRIMARY_EXTERNAL_METRICS[
        "ari"
    ],
)

print(
    "PRIMARY_SAKAI_LIKE =",
    PRIMARY_SAKAI_LIKE_MACRO,
)

print()
print(
    "LEGACY_PURITY =",
    LEGACY_EXTERNAL_METRICS[
        "purity"
    ],
)

print(
    "LEGACY_NMI =",
    LEGACY_EXTERNAL_METRICS[
        "nmi_arithmetic"
    ],
)

print(
    "LEGACY_ARI =",
    LEGACY_EXTERNAL_METRICS[
        "ari"
    ],
)

print(
    "LEGACY_SAKAI_LIKE =",
    LEGACY_SAKAI_LIKE_MACRO,
)

print()
print(
    "FINAL_AUDIT_PATH =",
    FINAL_AUDIT_PATH,
)

print(
    "FINAL_AUDIT_SHA256 =",
    FINAL_AUDIT_SHA256,
)

print(
    "NOTEBOOK_06_FINAL_AUDIT_PASS =",
    NOTEBOOK_06_FINAL_AUDIT_PASS,
)

print(
    "FULL224_IIC_SPECTRAL_EVALUATION_CLOSED =",
    FULL224_IIC_SPECTRAL_EVALUATION_CLOSED,
)

print(
    "READY_FOR_MINI_C11_PHASE =",
    READY_FOR_MINI_C11_PHASE,
)


=== NOTEBOOK 06 — FINAL AUDIT ===
ALL_STEP_GATES = True
IDENTITY_GATE = True
PARTITION_GATE = True
METRICS_GATE = True
SAKAI_LIKE_GATE = True
REPRESENTATIVE_GATE = True
METHODOLOGY_GATE = True

PRIMARY_PURITY = 0.8264010252825352
PRIMARY_NMI = 0.7018325449722438
PRIMARY_ARI = 0.4509773588198464
PRIMARY_SAKAI_LIKE = 0.8513637314759159

LEGACY_PURITY = 0.7980892461843179
LEGACY_NMI = 0.6821419109736958
LEGACY_ARI = 0.33427224443938447
LEGACY_SAKAI_LIKE = 0.8354332296844487

FINAL_AUDIT_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/spectral/audits/sakai224_full_notebook06_final_audit.json
FINAL_AUDIT_SHA256 = 835c6fe2a2e775b865992841c731a0b5e2f7ee62265f1af69a4668b615942086
NOTEBOOK_06_FINAL_AUDIT_PASS = True
FULL224_IIC_SPECTRAL_EVALUATION_CLOSED = True
READY_FOR_MINI_C11_PHASE = True
